# Helpful functions

In [ ]:
%pip install python-docx

In [ ]:
import pandas as pd
import re
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from docx import Document
import numpy as np

In [ ]:
def extract_sections(df):
    sections_dict = {}
    current_section = "GENERAL"
    current_columns = []

    for col in df.columns:
        if str(col).isupper():
            if current_columns:
                sections_dict[current_section] = df[current_columns].copy()

            current_section = col
            current_columns = [col]
        else:
            current_columns.append(col)

    if current_columns:
        sections_dict[current_section] = df[current_columns].copy()

    summary_data = []
    for section_name, section_df in sections_dict.items():
        if section_name != 'GENERAL' and section_name in section_df.columns:
            section_df.drop(columns=[section_name], inplace=True)

        summary_data.append({
            "Section Name": section_name,
            "Columns Count": section_df.shape[1]
        })

    summary_df = pd.DataFrame(summary_data)

    display(summary_df)

    return sections_dict

In [ ]:
def mapping_func(section_df,base_df,col,col_map):
  section_df[col] = section_df[col].map(col_map)
  section_df[col] = section_df[col].astype('category')
  base_df[col] = base_df[col].map(col_map)
  base_df[col] = base_df[col].astype('category')
  extract_information(section_df,col)

In [ ]:
def analyze_any_column(df, col_name):
    total = len(df)
    missing = df[col_name].isna().sum()
    unique_count = df[col_name].nunique(dropna=True)

    print(f"Column: {col_name}")

    print(f"type :{df[col_name].dtype}")
    print(f"missed values count {missing} from {total} ({missing/total*100:.2f}%)")
    print(f" uniqe values count {unique_count}")
    print("-------------------------------------------------")

    print("first 10 rows")
    print(df[col_name].head(10).to_string())
    print("-------------------------------------------------")

    valid_data = df[col_name].dropna()


    if pd.api.types.is_numeric_dtype(df[col_name]) and unique_count > 10:
        print("dtype = (Numeric)")
        print(f" (Min): {valid_data.min()}")
        print(f" (Max): {valid_data.max()}")
        print(f" (Mean): {valid_data.mean():.2f}")
        print(f" (Median): {valid_data.median()}")

    else:
        if unique_count <= 10:
            print("dtype = (Categorical / Flag)")
        else:
            print("dtype = (Text / IDs / Dates)")

        print("\n top 5 frequent values")
        counts = valid_data.astype(str).value_counts().head(5)
        percentages = (counts / len(valid_data) * 100).round(2)

        summary_df = pd.DataFrame({
            'Count': counts,
            ' %': percentages
        })
        print(summary_df)


In [ ]:
def extract_information(df,column):
    print(df[[column]].dtypes)
    print(pd.Series(df[column].unique()))
    print(df[column].isnull().sum())
    print((df[column].isnull().sum()) / (df.shape[0]) * 100)
    print(df[column].value_counts())
    print(df[[column]].describe())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
layout_options = {
    'paper_bgcolor':"#383838",
    'plot_bgcolor':'#383838',
    'title_font': dict(color='white'),
    'legend_font': dict(color='white'),
    'yaxis':dict(color="white"),
    'xaxis':dict(color="white")
    }

In [ ]:
columns_to_drop_parent = []
columns_to_drop_children = []

In [ ]:
def process_numeric_cols(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype('float64')

    print(f"Processed {len(columns)} numeric columns.")
    return df

def process_binary_cols(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype('Int64')

    print(f"Processed {len(columns)} binary columns.")
    return df

def process_text_cols(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype('string')

    print(f"Processed {len(columns)} text columns.")
    return df

def process_categorical_cols(df, columns, mapping_dicts=None):
    for col in columns:
        if col in df.columns:
            if mapping_dicts and col in mapping_dicts:
                df[col] = df[col].map(mapping_dicts[col]).astype('category')
            # else:
            #     try:
            #         # محاولة الحماية بـ Int64 إذا كانت أرقاماً
            #         df[col] = df[col].astype('Int64').astype('category')
            #     except (ValueError, TypeError):
            #         # إذا فشل بسبب وجود نصوص، التحويل مباشرة لفئة
            #         df[col] = df[col].astype('category')

    print(f"Processed {len(columns)} categorical columns.")
    return df

def process_date_cols(df, columns, time_unit=None):
    for col in columns:
        if col in df.columns:
            if time_unit:
                df[col] = pd.to_datetime(df[col], unit=time_unit, errors='coerce')
            else:
                df[col] = pd.to_datetime(df[col], errors='coerce')

    print(f"Processed {len(columns)} date columns.")
    return df

In [ ]:
def check_null_ratio(my_df):
  info_df = pd.DataFrame()
  info_df['missing_val'] = my_df.isnull().sum()
  info_df['missing_val_ratio'] = (info_df['missing_val'] / my_df.shape[0] * 100).round().astype(int)
  print(info_df)


# Reading data

## Reading parent data

In [ ]:
FILE_PATH="/content/drive/MyDrive/Data mining homework 2/womens_mens_mcm.tab"
parent_df = pd.read_csv(FILE_PATH, sep='\t')
parent_df.head()

/tmp/ipykernel_1035/3942432061.py:2: DtypeWarning:

Columns (186,200,602,1318) have mixed types. Specify dtype option on import or set low_memory=False.



,id,wemp,whn,mhn,treat,control,stratum,villageid,district_name,districtid,subcountyid,sample_women,sample_married,sample_couple,sample_pair,sample_anthro,BL_WOMENS,w_date,w_date_code,w_enum_id,w_time,w_eligibility_kids,DEMOGRAPHICS_ROSTER,w_age,w_educ,w_inc,w_marital,w_num_adults,w_numkids18,w_numkids18_bio,w_numkids18_non,w_biokids_out,w_othkids12,w_numkids0to5,FOOD__RECALL,w_milk,w_water,w_tea,w_juice,w_otherbev,w_matooke,w_roots,w_grains,w_beans,w_organ,w_meat,w_egg,w_fish,w_green,w_pump,w_otherfru,w_oil,w_sweet,w_othersolid,w_juiceplus,w_meats,w_otherfv,REPRODUCTIVE_HISTORY,w_biokids_hh,w_biokids_out2,w_biokids_num,w_biokids_num_alive,w_numkids_husb,w_pregnow,w_preg_mo,w_preg_learn,w_preg_want,w_preg_ac,w_preg_whynoac,w_preg_whynoac_oth,w_preg_ac_num,w_preg_ac_mo,w_hus_go_ac,w_preg_ac_eat,w_preg_ac_comp,w_hus_ac_advice,w_hus_ac_where,w_hus_ac_nutr,w_hus_ac_deliver,w_hus_ac_kids,w_hus_ac_whenkid,w_preg_mal,w_preg_mal_num,w_preg_comp,w_preg_comp_list,w_hus_aware_comp,w_hus_assist_comp,w_hus_assist_how,w_hus_assist_how_oth,w_preg_foods,w_preg_foods_list,w_pregpast2,w_preg_ac2,w_preg_whynoac2,w_preg_whynoac_oth2,w_preg_ac_num2,w_preg_ac_mo2,w_hus_go_ac2,w_preg_ac_eat2,w_preg_ac_comp2,w_hus_ac_advice2,w_hus_ac_where2,w_hus_ac_nutr2,w_hus_ac_deliver2,w_hus_ac_kids2,w_hus_ac_whenkid2,w_preg_mal2,w_preg_mal_num2,w_preg_comp2,w_preg_comp_list2,w_birth_comp2,w_birth_comp_list2,w_hus_aware_comp2,w_hus_assist_comp2,w_hus_assist_how2,w_hus_assist_how_oth2,w_hus_aware_bcomp2,w_hus_assist_bcomp2,w_hus_assist_bhow2,w_hus_assist_bhow_oth2,w_preg_foods2,w_preg_foods_list2,w_where_birth,w_hus_assist_birth,w_hus_assist_birth_how,w_hus_assist_birth_oth,LAST_PREGNANCY,w_lastpreg_ac,w_lastpreg_foods,w_bfeed_atemore,w_time_breastfeed_weeks,w_breastfed_weeks_bi,INFANT_CARE,w_hus_disc_care,w_birth_weighed,w_birth_weight,w_weight_immcard,w_birth_size,w_check2mo,w_mom_health2mo,w_mom_healthnow,w_first_check,w_first_check_unit,w_first_check_weeks,w_first_check_weeks_bi,w_weight_1mo,w_weight_1mo_card,w_weight_3mo,w_weight_3mo_card,w_breastfeed,w_time_breastfeed,w_time_bfeed_unit,w_colostrum,w_time_milk,w_time_milk_units,w_othliq_1week,w_othliq_3mo,w_othliq_what,w_othliq_other,w_food_3mo,w_bfeed_eat,iron,vita_6wk,vita_6mo,total_vacc,w_hus_ac,w_hus_feed,w_hus_preg,w_hus_hyg,w_fp_decide,w_fp_talk,w_ideal_fert_t,w_ideal_fert_b,w_ideal_fert_g,w_ideal_fert_e,w_nextpreg_when,w_hus_ideal_fert,w_hus_ideal_fert_t,w_hus_ideal_fert_b,w_hus_ideal_fert_g,w_hus_ideal_fert_e,w_smallkids,w_earlykids,w_pregcomp,SANITATION_PRACTICE,w_water_source,w_water_source_oth,w_treat_water,w_treat_water_how,w_treat_water_oth,w_soap_day,w_wash_bath,w_soap_toilet,w_soap_toilet_bi,w_wash_meal,w_soap_meal,w_soap_meal_bi,w_refuse_dispose,w_refuse_dispose_bi,w_toilet_type,w_toilet_oth,w_toilet_sweep,w_toilet_sweep_bi,HEALTH_KNOWLEDGE,w_birth,w_birth_bi,w_birth_oth,w_mal_cause,w_mal_cause_bi,w_mal_cause_bi2,w_mal_cause_oth,w_diar_cause,w_diar_cause_bi,w_diar_cause_bi2,w_cstrum_feed,w_cstrum_feed_bi,w_cstrum_no_why,w_grow_lackfood,w_grow_lackfood_sw,w_grow_lackfood_vm,w_grow_lackdiet,w_grow_lackdiet_sw,w_grow_lackdiet_vm,w_grow_witch,w_grow_witch_lt,w_grow_witch_vl,w_grow_sick,w_grow_sick_sw,w_grow_sick_vm,w_kwash_cause,w_kwash_cause_bi,w_kwash_cause_bi2,w_kwash_treat,w_kwash_treat_bi,w_kwash_treat_bi2,w_ugx_spend,w_ugx_oth,w_ugx_matooke,w_ugx_matooke_bi,w_ugx_maize,w_ugx_maize_bi,w_ugx_potato,w_ugx_potato_bi,w_ugx_rice,w_ugx_rice_bi,w_ugx_fish,w_ugx_fish_bi,w_ugx_gnuts,w_ugx_gnuts_bi,w_ugx_veg,w_ugx_veg_bi,w_ugx_fruit,w_ugx_fruit_bi,w_ugx_meat,w_ugx_meat_bi,w_ugx_bean,w_ugx_bean_bi,w_ugx_egg,w_ugx_egg_bi,w_ugx_milk,w_ugx_milk_bi,w_ugx_sweet,w_ugx_sweet_bi,w_ugx_other,w_ugx_other_bi,w_ageliquid,w_ageliquid_6,w_agefood,w_agefood_6,w_bfeed,w_bfeed24,w_starchmilk,w_starchmilk_bi,w_starchmillet,w_starchmillet_bi,w_starchoil,w_starchoil_bi,w_starchgnuts,w_starchgnuts_bi,w_starchrice,w_starchrice_bi,w_starchpineapple,w_starchpineapple_bi,w_starchmatooke,w_starchmatooke_bi,w_proteinchicken,w_prot

In [ ]:
parent_df.shape

(5512, 1510)

In [ ]:
parent_all_sections = extract_sections(parent_df)

,Section Name,Columns Count
0,GENERAL,16
1,BL_WOMENS,5
2,DEMOGRAPHICS_ROSTER,11
3,FOOD__RECALL,22
4,REPRODUCTIVE_HISTORY,68
5,LAST_PREGNANCY,5
6,INFANT_CARE,51
7,SANITATION_PRACTICE,18
8,HEALTH_KNOWLEDGE,138
9,DECISION_MAKING,184


## Reading Children data

In [ ]:
FILE_PATH="/content/drive/MyDrive/Data mining homework 2/child_level_mcm.tab"
pd.set_option('display.max_columns', None)
children_df = pd.read_csv(FILE_PATH, sep='\t')
children_df.head()

,id,villageid,stratum,districtid,district_name,wemp,whn,mhn,control,treat,sample,sample_married,sample_moved,sample_alive,sample_2yr,sample_1yr,onlyin,survey_status,BASELINE_SURVEY,age,gender,survey_date,kidsunder18,under18bio,under18nonbio,biooutside,otherkids,FOOD_RECALL,milk,water,tea,juice,otherbev,solidfood,matooke,roots,grains,beans,organ,meat,egg,fish,green,pump,otherfru,oil,sweet,othersolid,OTHER_HEALTH,immcard_w,biokid,breastfed,time_bf,bednet,diarrhea,diar_days,diar_drink,diar_medpro,malaria,mal_days,mal_drink,mal_medpro,cough,cough_days,cough_medpro,worm_drugs,worm_medpro,semi_solidfood,num_meal,daysprotein,bcg,polio0,polio1,polio2,polio3,dpt1,dpt2,dpt3,measles,vita_6wk,vita_6mo,iron,elapsed_time_mo,BASELINE_ANTHRO,anthro_date,c_weight,wfa,wfa_neg2,wfa_neg1,wfl,wfl_neg2,wfl_neg1,c_weight_flag,c_weight_flagwhy,weigh_behave,weigh_how,c_height,lhfa,lhfa_neg2,lhfa_neg1,c_height_flag,c_height_flagwhy,height_how,c_muac,mfa,mfa_neg3,mfa_neg23,mfa_neg2,mfa_neg1,c_muac_flag,c_muac_flagwhy,press_skin,anthro_status,c_hblevel,hb_11_child,hb_10_child,hb_7_child,c_hblevel_flag,c_hblevel_flagwhy,ENDLINE_SURVEY,Ew_a_complete,Estatus_w,Eage,Eage_flag,Eimmcard_age,Eincomplete_w,Eincomplete_a,Edistrictid,Esubcountyid,Eparishid,Evillageid,Emom_marital,Emoved,Emom_married,Esurvey_date,Ebl_father,FOOD__RECALL,Emilk,Ewater,Etea,Ejuice,Eotherbev,Esolidfood,Ematooke,Eroots,Egrains,Ebeans,Eorgan,Emeat,Eegg,Efish,Egreen,Epump,Eotherfru,Eothdairy,Eoil,Esweet,Eothersolid,Enum_meal,Edaysmilk,Edaysbeans,Edaysmeat,Ebednet,Ediarrhea,Ediar_days,Ediar_drink,Ediar_medpro,Emalaria,Emal_days,Emal_drink,Emal_medpro,Ecough,Ecough_days,Ecough_medpro,Eworm_drugs,Ebcg,Epolio0,Epolio1,Epolio2,Epolio3,Edpt1,Edpt2,Edpt3,Emeasles,Eimmcard_vacc,Evita_6wk,Evita_6mo,Eiron,Ebiokid,Echild_alive,Edeath_date,Ein_hh_now,Ewhere_living,ENDLINE_ANTHRO,Ebirth_weighed,Ebirth_weight,Ebweight_units,Eweight_immcard,Estatus_a,Eanthro_status,Eanthro_date,Eelapsed_time_mo,Ec_hblevel,Ec_hblevel_flag,Ehb_11_child,Ehb_10_child,Ehb_7_child,Ec_weight,Ec_weight_flag,Ewfa,Ewfa_neg2,Ewfa_neg1,Ewfl,Ewfl_neg2,Ewfl_neg1,Eweigh_behave,Eweigh_how,Ec_height,Ec_height_flag,Elhfa,Elhfa_neg2,Elhfa_neg1,Eheight_how,Ec_muac,Ec_muac_flag,Emfa,Emfa_neg3,Emfa_neg23,Emfa_neg2,Emfa_neg1,Eswelling,Epress_skin,Esource,Emerge,Elhfa2,Ewfa2,Ewfl2,lhfa2,wfa2,wfl2,female,juiceplus,meats,otherfv,Ejuiceplus,Emeats,Eotherfv,sick_days,Esick_days,flag_mfa_imp,flag_wfa_imp,flag_lhfa_imp,flag_c_hblevel_imp,flag_sick_days_imp
0,4.011010e+09,11010102,3.0,1.0,MITOOMA,1.0,0.0,0.0,0.0,2,0.0,0.0,0.0,1.0,0.0,0.0,3,1.0,NaN,69.0,2.0,2012-10-30,2.0,2.0,NaN,0.0,0.0,=== FOOD RECALL ===,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,=== Baseline Anthro ===,19296.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,=== ENDLINE SURVEY ===,1.0,1.0,89.0,NaN,1.0,NaN,NaN,MITOOMA,BITEREKO,BUGONGO,BUGONGO CENTRAL,NaN,0.0,1.0,2014-07-07,1.0,=== FOOD RECALL ===,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,=== ENDLINE ANTHRO ===,NaN,NaN,NaN,NaN,1.0,NaN,2014-07-07,0.001142,NaN,1,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
1,6.110101e+08,11010102,3.0,1.0,MITOOMA,1.0,0.0,0.0,0.0,2,1.0,1.0,0.0,1.0,1.0,0.0,2,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,=== FOOD RECALL ===,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [ ]:
children_df.shape

(22308, 254)

In [ ]:
children_all_sections = extract_sections(children_df)

,Section Name,Columns Count
0,GENERAL,18
1,BASELINE_SURVEY,8
2,FOOD_RECALL,20
3,OTHER_HEALTH,34
4,BASELINE_ANTHRO,35
5,ENDLINE_SURVEY,16
6,FOOD__RECALL,56
7,ENDLINE_ANTHRO,60


# 2- Exploring the data

## Parent section

### 1- GENERAL Section

In [ ]:
general = parent_all_sections['GENERAL']
general.columns

Index(['id', 'wemp', 'whn', 'mhn', 'treat', 'control', 'stratum', 'villageid',
       'district_name', 'districtid', 'subcountyid', 'sample_women',
       'sample_married', 'sample_couple', 'sample_pair', 'sample_anthro'],
      dtype='object')

#### 1.1- id

In [ ]:
extract_information(general,'id')

id    float64
dtype: object
0       1.110101e+08
1       1.110104e+08
2       1.110301e+08
3       1.110302e+08
4       1.110403e+08
            ...     
5507    2.552104e+10
5508    2.552202e+10
5509    2.552202e+10
5510    2.552202e+10
5511    2.552301e+10
Length: 5512, dtype: float64
0
0.0
id
2.552301e+10    1
1.110101e+08    1
1.110104e+08    1
1.110301e+08    1
1.110302e+08    1
               ..
1.110907e+08    1
1.110906e+08    1
1.110805e+08    1
1.110804e+08    1
1.110802e+08    1
Name: count, Length: 5512, dtype: int64
                 id
count  5.512000e+03
mean   5.234752e+09
std    4.580704e+09
min    1.110101e+08
25%    2.131768e+09
50%    4.227025e+09
75%    6.823013e+09
max    2.552301e+10


#### 1.2- wemp

In [ ]:
extract_information(general,'wemp')

wemp    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
wemp
0    4182
1    1330
Name: count, dtype: int64
              wemp
count  5512.000000
mean      0.241292
std       0.427906
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 1.3- whn

In [ ]:
extract_information(general,'whn')

whn    int64
dtype: object
0    0
1    1
dtype: int64
0
0.0
whn
0    4077
1    1435
Name: count, dtype: int64
               whn
count  5512.000000
mean      0.260341
std       0.438861
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 1.4- mhn

In [ ]:
extract_information(general,'mhn')

mhn    int64
dtype: object
0    0
1    1
dtype: int64
0
0.0
mhn
0    4144
1    1368
Name: count, dtype: int64
               mhn
count  5512.000000
mean      0.248186
std       0.431999
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 1.5- treat

In [ ]:
extract_information(general,'treat')

treat    int64
dtype: object
0    2
1    1
2    4
3    3
dtype: int64
0
0.0
treat
3    1435
1    1379
4    1368
2    1330
Name: count, dtype: int64
             treat
count  5512.000000
mean      2.506531
std       1.116655
min       1.000000
25%       1.000000
50%       3.000000
75%       3.000000
max       4.000000


#### 1.6- control

In [ ]:
extract_information(general,'control')

control    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
control
0.0    4133
1.0    1379
Name: count, dtype: int64
           control
count  5512.000000
mean      0.250181
std       0.433157
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 1.7- stratum

In [ ]:
extract_information(general,'stratum')

stratum    int64
dtype: object
0    3
1    4
2    1
3    2
dtype: int64
0
0.0
stratum
4    1477
1    1456
2    1292
3    1287
Name: count, dtype: int64
           stratum
count  5512.000000
mean      2.505261
std       1.146488
min       1.000000
25%       1.000000
50%       3.000000
75%       4.000000
max       4.000000


#### 1.8- villageid

In [ ]:
extract_information(general,'villageid')

villageid    int64
dtype: object
0      11010102
1      11010408
2      11030106
3      11030207
4      11040310
         ...   
407    11020403
408    21020208
409    41050606
410    32050104
411    32050303
Length: 412, dtype: int64
0
0.0
villageid
22040101    17
23010101    17
22040511    17
11030207    16
11010102    16
            ..
41060303     6
11100110     5
11100306     5
11100403     5
21020208     4
Name: count, Length: 412, dtype: int64
          villageid
count  5.512000e+03
mean   2.501011e+07
std    9.492509e+06
min    1.101010e+07
25%    2.102051e+07
50%    2.208040e+07
75%    3.201050e+07
max    4.107041e+07


#### 1.9- district_name

In [ ]:
extract_information(general,'district_name')

district_name    object
dtype: object
0      MITOOMA
1     NTUNGAMO
2    RUKUNGIRI
3       SHEEMA
dtype: object
0
0.0
district_name
NTUNGAMO     2266
RUKUNGIRI    1361
MITOOMA      1078
SHEEMA        807
Name: count, dtype: int64
       district_name
count           5512
unique             4
top         NTUNGAMO
freq            2266


#### 1.10- districtid

In [ ]:
extract_information(general,'districtid')

districtid    int64
dtype: object
0    1
1    2
2    3
3    4
dtype: int64
0
0.0
districtid
2    2266
3    1361
1    1078
4     807
Name: count, dtype: int64
        districtid
count  5512.000000
mean      2.344158
std       0.953856
min       1.000000
25%       2.000000
50%       2.000000
75%       3.000000
max       4.000000


#### 1.11- subcountyid

In [ ]:
extract_information(general,'subcountyid')

subcountyid    int64
dtype: object
0     1
1     3
2     4
3     5
4     6
5     8
6     9
7    10
8     2
9     7
dtype: int64
0
0.0
subcountyid
1     1072
3     1032
4      887
2      872
5      556
6      374
8      249
7      243
9      174
10      53
Name: count, dtype: int64
       subcountyid
count  5512.000000
mean      3.677975
std       2.257165
min       1.000000
25%       2.000000
50%       3.000000
75%       5.000000
max      10.000000


#### 1.12- sample_women

In [ ]:
extract_information(general,'sample_women')

sample_women    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
sample_women
1    5288
0     224
Name: count, dtype: int64
       sample_women
count   5512.000000
mean       0.959361
std        0.197469
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


#### 1.13- sample_married

In [ ]:
extract_information(general,'sample_married')

sample_married    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
sample_married
1    5155
0     357
Name: count, dtype: int64
       sample_married
count     5512.000000
mean         0.935232
std          0.246138
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 1.14- sample_couple

In [ ]:
extract_information(general,'sample_couple')

sample_couple    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
sample_couple
1    4980
0     532
Name: count, dtype: int64
       sample_couple
count    5512.000000
mean        0.903483
std         0.295325
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000


#### 1.15- sample_pair

In [ ]:
extract_information(general,'sample_pair')

sample_pair    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
sample_pair
1    5080
0     432
Name: count, dtype: int64
       sample_pair
count  5512.000000
mean      0.921626
std       0.268784
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 1.16- sample_anthro

In [ ]:
extract_information(general,'sample_anthro')

sample_anthro    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
sample_anthro
1    5202
0     310
Name: count, dtype: int64
       sample_anthro
count    5512.000000
mean        0.943759
std         0.230407
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000


### 2- BL_WOMENS Section

In [ ]:
bl_womens = parent_all_sections['BL_WOMENS']
bl_womens.columns

Index(['w_date', 'w_date_code', 'w_enum_id', 'w_time', 'w_eligibility_kids'], dtype='object')

#### 2.1- w_date

In [ ]:
extract_information(bl_womens,'w_date')

w_date    object
dtype: object
0     10/30/2012 0:00:00
1     10/29/2012 0:00:00
2      11/1/2012 0:00:00
3      11/7/2012 0:00:00
4      11/9/2012 0:00:00
             ...        
78    11/18/2012 0:00:00
79     11/5/2012 0:00:00
80     11/6/2012 0:00:00
81     12/7/2013 0:00:00
82     1/15/2013 0:00:00
Length: 83, dtype: object
21
0.38098693759071117
w_date
12/11/2012 0:00:00    101
12/14/2012 0:00:00     99
12/17/2012 0:00:00     98
12/13/2012 0:00:00     97
1/7/2013 0:00:00       95
                     ... 
11/5/2012 0:00:00       5
10/24/2012 0:00:00      3
11/6/2012 0:00:00       2
12/7/2013 0:00:00       1
1/15/2013 0:00:00       1
Name: count, Length: 82, dtype: int64
                    w_date
count                 5491
unique                  82
top     12/11/2012 0:00:00
freq                   101


#### 2.2- w_date_code

In [ ]:
extract_information(bl_womens,'w_date_code')

w_date_code    int64
dtype: object
0     19296
1     19295
2     19298
3     19304
4     19306
      ...  
76    19369
77    19315
78    19302
79    19303
80    19373
Length: 81, dtype: int64
0
0.0
w_date_code
19338    101
19341     99
19344     98
19340     97
19365     96
        ... 
19370      6
19302      5
19290      3
19303      2
19373      1
Name: count, Length: 81, dtype: int64
        w_date_code
count   5512.000000
mean   19314.962808
std       26.607844
min    19268.000000
25%    19293.000000
50%    19316.000000
75%    19336.000000
max    19373.000000


#### 2.3- w_enum_id

In [ ]:
extract_information(bl_womens,'w_enum_id')

w_enum_id    float64
dtype: object
0     49.0
1     30.0
2     12.0
3     26.0
4     15.0
5     45.0
6     20.0
7      8.0
8     25.0
9     44.0
10    51.0
11    35.0
12    48.0
13    37.0
14    19.0
15    27.0
16    40.0
17    21.0
18    28.0
19    23.0
20    39.0
21    43.0
22    36.0
23    22.0
24    41.0
25    34.0
26     NaN
27    31.0
28    24.0
29    13.0
30    29.0
31    50.0
32    16.0
33    33.0
34    42.0
35    18.0
36    32.0
37    47.0
38     7.0
39    46.0
40     9.0
41    61.0
42    17.0
43     5.0
dtype: float64
18
0.3265602322206096
w_enum_id
41.0    171
40.0    167
35.0    165
29.0    165
32.0    165
42.0    163
34.0    162
31.0    160
26.0    157
28.0    157
13.0    157
12.0    156
24.0    154
18.0    153
19.0    152
39.0    151
27.0    151
44.0    150
36.0    148
21.0    148
37.0    148
30.0    146
47.0    145
15.0    143
45.0    142
51.0    141
16.0    140
48.0    139
33.0    138
20.0    136
22.0    133
25.0    128
49.0    125
50.0    125
23.0    125
43.0    115
46

#### 2.4- w_time

In [ ]:
extract_information(bl_womens,'w_time')

w_time    object
dtype: object
0      12/30/1899 10:12:00
1      12/30/1899 10:07:00
2      12/30/1899 13:43:00
3      12/30/1899 14:03:00
4      12/30/1899 12:11:00
              ...         
569     12/30/1899 1:01:00
570    12/30/1899 17:27:00
571    12/30/1899 17:08:00
572     12/30/1899 8:30:00
573    12/30/1899 17:55:00
Length: 574, dtype: object
23
0.41727140783744554
w_time
12/30/1899 9:47:00     33
12/30/1899 9:50:00     29
12/30/1899 10:34:00    26
12/30/1899 11:17:00    24
12/30/1899 11:43:00    23
                       ..
12/30/1899 1:01:00      1
12/30/1899 17:27:00     1
12/30/1899 17:08:00     1
12/30/1899 8:30:00      1
12/30/1899 17:55:00     1
Name: count, Length: 573, dtype: int64
                    w_time
count                 5489
unique                 573
top     12/30/1899 9:47:00
freq                    33


#### 2.5- w_eligibility_kids

In [ ]:
extract_information(bl_womens,'w_eligibility_kids')

w_eligibility_kids    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
27
0.48984034833091433
w_eligibility_kids
1.0    4884
0.0     601
Name: count, dtype: int64
       w_eligibility_kids
count         5485.000000
mean             0.890428
std              0.312383
min              0.000000
25%              1.000000
50%              1.000000
75%              1.000000
max              1.000000


### 3- DEMOGRAPHICS_ROSTER Section

In [ ]:
demographic_rosted = parent_all_sections['DEMOGRAPHICS_ROSTER']
demographic_rosted.columns

Index(['w_age', 'w_educ', 'w_inc', 'w_marital', 'w_num_adults', 'w_numkids18',
       'w_numkids18_bio', 'w_numkids18_non', 'w_biokids_out', 'w_othkids12',
       'w_numkids0to5'],
      dtype='object')

#### 3.1- w_age

In [ ]:
extract_information(demographic_rosted,'w_age')

w_age    float64
dtype: object
0     39.0
1     34.0
2     38.0
3     27.0
4     35.0
5     42.0
6     31.0
7     37.0
8     36.0
9     26.0
10    20.0
11    21.0
12    30.0
13    40.0
14    23.0
15    29.0
16    24.0
17    25.0
18    28.0
19    33.0
20    32.0
21    41.0
22    46.0
23    43.0
24    22.0
25    19.0
26    18.0
27    48.0
28    44.0
29    45.0
30     NaN
31    17.0
32    50.0
33    49.0
34    47.0
35    63.0
36    59.0
37    13.0
38    16.0
dtype: float64
7
0.1269956458635704
w_age
30.0    419
22.0    367
23.0    363
25.0    363
20.0    355
28.0    352
24.0    344
26.0    307
21.0    260
32.0    249
27.0    241
35.0    207
29.0    201
19.0    195
31.0    154
33.0    144
36.0    142
34.0    135
38.0    132
37.0     99
18.0     97
40.0     94
39.0     72
42.0     52
43.0     45
41.0     32
44.0     20
17.0     19
45.0     18
46.0      7
48.0      6
47.0      5
50.0      3
49.0      2
63.0      1
59.0      1
13.0      1
16.0      1
Name: count, dtype: int64
             w_a

#### 3.2- w_educ

In [ ]:
extract_information(demographic_rosted,'w_educ')

w_educ    float64
dtype: object
0      4.0
1      1.0
2      NaN
3      8.0
4      7.0
5      6.0
6      5.0
7     11.0
8      0.0
9      3.0
10    10.0
11     2.0
12     9.0
13    14.0
14    13.0
15    12.0
dtype: float64
173
3.138606676342525
w_educ
7.0     1184
6.0      703
5.0      618
0.0      584
4.0      493
3.0      420
2.0      334
11.0     282
9.0      207
1.0      181
8.0      154
10.0     146
13.0      18
14.0      11
12.0       4
Name: count, dtype: int64
            w_educ
count  5339.000000
mean      5.201349
std       3.018506
min       0.000000
25%       3.000000
50%       6.000000
75%       7.000000
max      14.000000


#### 3.3- w_inc

In [ ]:
extract_information(demographic_rosted,'w_inc')

w_inc    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
45
0.816400580551524
w_inc
1.0    4316
0.0    1151
Name: count, dtype: int64
             w_inc
count  5467.000000
mean      0.789464
std       0.407727
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 3.4- w_marital

In [ ]:
extract_information(demographic_rosted,'w_marital')

w_marital    float64
dtype: object
0    1.0
1    NaN
2    4.0
3    2.0
4    3.0
dtype: float64
4
0.07256894049346879
w_marital
1.0    5412
4.0      57
2.0      34
3.0       5
Name: count, dtype: int64
         w_marital
count  5508.000000
mean      1.039034
std       0.318490
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       4.000000


#### 3.5- w_num_adults

In [ ]:
extract_information(demographic_rosted,'w_num_adults')

w_num_adults    float64
dtype: object
0     2.0
1     3.0
2     5.0
3     4.0
4     7.0
5     6.0
6     NaN
7     8.0
8    11.0
dtype: float64
6
0.10885341074020319
w_num_adults
2.0     4881
3.0      409
4.0      149
5.0       51
6.0       10
7.0        3
8.0        2
11.0       1
Name: count, dtype: int64
       w_num_adults
count   5506.000000
mean       2.169996
std        0.554543
min        2.000000
25%        2.000000
50%        2.000000
75%        2.000000
max       11.000000


#### 3.6- w_numkids18

In [ ]:
extract_information(demographic_rosted,'w_numkids18')

w_numkids18    float64
dtype: object
0      6.0
1      5.0
2      4.0
3     10.0
4      1.0
5      3.0
6      7.0
7     11.0
8      2.0
9      9.0
10     8.0
11     0.0
12     NaN
13    12.0
dtype: float64
1
0.018142235123367198
w_numkids18
2.0     1053
1.0     1010
3.0      987
4.0      859
5.0      645
6.0      421
7.0      215
0.0      192
8.0       95
9.0       23
10.0       8
11.0       2
12.0       1
Name: count, dtype: int64
       w_numkids18
count  5511.000000
mean      3.238977
std       1.951232
min       0.000000
25%       2.000000
50%       3.000000
75%       5.000000
max      12.000000


#### 3.7- w_numkids18_bio

In [ ]:
extract_information(demographic_rosted,'w_numkids18_bio')

w_numkids18_bio    float64
dtype: object
0      5.0
1      6.0
2      4.0
3      2.0
4      1.0
5      3.0
6      0.0
7      9.0
8      7.0
9      8.0
10     NaN
11    10.0
dtype: float64
1
0.018142235123367198
w_numkids18_bio
1.0     1213
2.0     1090
3.0      935
4.0      814
5.0      584
6.0      371
0.0      234
7.0      198
8.0       57
9.0       11
10.0       4
Name: count, dtype: int64
       w_numkids18_bio
count      5511.000000
mean          3.008710
std           1.891886
min           0.000000
25%           1.000000
50%           3.000000
75%           4.000000
max          10.000000


#### 3.8- w_numkids18_non

In [ ]:
extract_information(demographic_rosted,'w_numkids18_non')

w_numkids18_non    float64
dtype: object
0    1.0
1    NaN
2    8.0
3    0.0
4    5.0
5    2.0
6    3.0
7    4.0
8    6.0
9    7.0
dtype: float64
4330
78.55587808417998
w_numkids18_non
1.0    624
0.0    296
2.0    161
3.0     50
4.0     26
5.0     15
6.0      5
7.0      3
8.0      2
Name: count, dtype: int64
       w_numkids18_non
count      1182.000000
mean          1.135364
std           1.104353
min           0.000000
25%           0.250000
50%           1.000000
75%           1.000000
max           8.000000


#### 3.9- w_biokids_out

In [ ]:
extract_information(demographic_rosted,'w_biokids_out')

w_biokids_out    float64
dtype: object
0    0.0
1    1.0
2    2.0
3    3.0
4    4.0
5    5.0
6    NaN
7    6.0
8    8.0
9    7.0
dtype: float64
1
0.018142235123367198
w_biokids_out
0.0    4061
1.0     916
2.0     339
3.0     127
4.0      41
5.0      16
6.0       7
7.0       3
8.0       1
Name: count, dtype: int64
       w_biokids_out
count    5511.000000
mean        0.415533
std         0.850155
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         8.000000


#### 3.10- w_othkids12

In [ ]:
extract_information(demographic_rosted,'w_othkids12')

w_othkids12    float64
dtype: object
0      0.0
1      1.0
2      9.0
3      4.0
4      6.0
5      7.0
6      2.0
7      3.0
8      5.0
9      NaN
10     8.0
11    10.0
12    14.0
13    21.0
14    11.0
15    12.0
16    13.0
17    20.0
18    17.0
19    15.0
dtype: float64
190
3.447024673439768
w_othkids12
0.0     3900
1.0      631
2.0      274
3.0      143
4.0      127
5.0       76
6.0       64
7.0       37
8.0       21
10.0      16
9.0       11
11.0       8
12.0       3
20.0       3
14.0       2
15.0       2
13.0       2
21.0       1
17.0       1
Name: count, dtype: int64
       w_othkids12
count  5322.000000
mean      0.727546
std       1.765789
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max      21.000000


#### 3.11- w_numkids0to5

In [ ]:
extract_information(demographic_rosted,'w_numkids0to5')

w_numkids0to5    float64
dtype: object
0    2.0
1    1.0
2    0.0
3    3.0
4    4.0
5    NaN
6    5.0
dtype: float64
180
3.265602322206096
w_numkids0to5
2.0    2605
1.0    2165
3.0     426
0.0     114
4.0      20
5.0       2
Name: count, dtype: int64
       w_numkids0to5
count    5332.000000
mean        1.639722
std         0.678322
min         0.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         5.000000


### 4- FOOD__RECALL Section

In [ ]:
food__recall = parent_all_sections['FOOD__RECALL']
food__recall.columns

Index(['w_milk', 'w_water', 'w_tea', 'w_juice', 'w_otherbev', 'w_matooke',
       'w_roots', 'w_grains', 'w_beans', 'w_organ', 'w_meat', 'w_egg',
       'w_fish', 'w_green', 'w_pump', 'w_otherfru', 'w_oil', 'w_sweet',
       'w_othersolid', 'w_juiceplus', 'w_meats', 'w_otherfv'],
      dtype='object')

#### 4.1- w_milk

In [ ]:
extract_information(food__recall,'w_milk')

w_milk    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
5
0.09071117561683599
w_milk
0.0    3408
1.0    2099
Name: count, dtype: int64
            w_milk
count  5507.000000
mean      0.381151
std       0.485714
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 4.2- w_water

In [ ]:
extract_information(food__recall,'w_water')

w_water    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_water
1.0    4873
0.0     636
Name: count, dtype: int64
           w_water
count  5509.000000
mean      0.884553
std       0.319590
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.3- w_tea

In [ ]:
extract_information(food__recall,'w_tea')

w_tea    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_tea
0.0    4608
1.0     901
Name: count, dtype: int64
             w_tea
count  5509.000000
mean      0.163551
std       0.369901
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.4- w_juice

In [ ]:
extract_information(food__recall,'w_juice')

w_juice    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
9
0.1632801161103048
w_juice
0.0    5246
1.0     257
Name: count, dtype: int64
           w_juice
count  5503.000000
mean      0.046702
std       0.211019
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.5- w_otherbev

In [ ]:
extract_information(food__recall,'w_otherbev')

w_otherbev    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
4
0.07256894049346879
w_otherbev
1.0    3619
0.0    1889
Name: count, dtype: int64
        w_otherbev
count  5508.000000
mean      0.657044
std       0.474740
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.6- w_matooke

In [ ]:
extract_information(food__recall,'w_matooke')

w_matooke    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
4
0.07256894049346879
w_matooke
1.0    4674
0.0     834
Name: count, dtype: int64
         w_matooke
count  5508.000000
mean      0.848584
std       0.358487
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.7- w_roots

In [ ]:
extract_information(food__recall,'w_roots')

w_roots    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
4
0.07256894049346879
w_roots
1.0    4373
0.0    1135
Name: count, dtype: int64
           w_roots
count  5508.000000
mean      0.793936
std       0.404514
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.8- w_grains

In [ ]:
extract_information(food__recall,'w_grains')

w_grains    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_grains
0.0    2794
1.0    2711
Name: count, dtype: int64
          w_grains
count  5505.000000
mean      0.492461
std       0.499989
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 4.9- w_beans

In [ ]:
extract_information(food__recall,'w_beans')

w_beans    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_beans
1.0    5224
0.0     285
Name: count, dtype: int64
           w_beans
count  5509.000000
mean      0.948266
std       0.221509
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.10- w_organ

In [ ]:
extract_information(food__recall,'w_organ')

w_organ    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_organ
0.0    5372
1.0     133
Name: count, dtype: int64
           w_organ
count  5505.000000
mean      0.024160
std       0.153559
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.11- w_meat

In [ ]:
extract_information(food__recall,'w_meat')

w_meat    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
5
0.09071117561683599
w_meat
0.0    4911
1.0     596
Name: count, dtype: int64
            w_meat
count  5507.000000
mean      0.108226
std       0.310694
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.12- w_egg

In [ ]:
extract_information(food__recall,'w_egg')

w_egg    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_egg
0.0    5231
1.0     274
Name: count, dtype: int64
             w_egg
count  5505.000000
mean      0.049773
std       0.217495
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.13- w_fish

In [ ]:
extract_information(food__recall,'w_fish')

w_fish    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
8
0.14513788098693758
w_fish
0.0    5358
1.0     146
Name: count, dtype: int64
            w_fish
count  5504.000000
mean      0.026526
std       0.160708
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.14- w_green

In [ ]:
extract_information(food__recall,'w_green')

w_green    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
5
0.09071117561683599
w_green
1.0    3217
0.0    2290
Name: count, dtype: int64
           w_green
count  5507.000000
mean      0.584166
std       0.492910
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.15- w_pump

In [ ]:
extract_information(food__recall,'w_pump')

w_pump    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
11
0.19956458635703916
w_pump
0.0    4538
1.0     963
Name: count, dtype: int64
            w_pump
count  5501.000000
mean      0.175059
std       0.380052
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.16- w_otherfru

In [ ]:
extract_information(food__recall,'w_otherfru')

w_otherfru    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
6
0.10885341074020319
w_otherfru
1.0    2976
0.0    2530
Name: count, dtype: int64
        w_otherfru
count  5506.000000
mean      0.540501
std       0.498402
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.17- w_oil

In [ ]:
extract_information(food__recall,'w_oil')

w_oil    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
14
0.2539912917271408
w_oil
0.0    3973
1.0    1525
Name: count, dtype: int64
             w_oil
count  5498.000000
mean      0.277374
std       0.447743
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 4.18- w_sweet

In [ ]:
extract_information(food__recall,'w_sweet')

w_sweet    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_sweet
0.0    4755
1.0     750
Name: count, dtype: int64
           w_sweet
count  5505.000000
mean      0.136240
std       0.343074
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.19- w_othersolid

In [ ]:
extract_information(food__recall,'w_othersolid')

w_othersolid    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
7
0.1269956458635704
w_othersolid
0.0    3758
1.0    1747
Name: count, dtype: int64
       w_othersolid
count   5505.000000
mean       0.317348
std        0.465486
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000


#### 4.20- w_juiceplus

In [ ]:
extract_information(food__recall,'w_juiceplus')

w_juiceplus    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_juiceplus
1.0    3690
0.0    1819
Name: count, dtype: int64
       w_juiceplus
count  5509.000000
mean      0.669813
std       0.470323
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.21- w_meats

In [ ]:
extract_information(food__recall,'w_meats')

w_meats    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_meats
0.0    4880
1.0     629
Name: count, dtype: int64
           w_meats
count  5509.000000
mean      0.114177
std       0.318055
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.22- w_otherfv

In [ ]:
extract_information(food__recall,'w_otherfv')

w_otherfv    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_otherfv
1.0    3269
0.0    2240
Name: count, dtype: int64
         w_otherfv
count  5509.000000
mean      0.593393
std       0.491245
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


### 5- REPRODUCTIVE_HISTORY Section

In [ ]:
reproductive_history = parent_all_sections['REPRODUCTIVE_HISTORY']
reproductive_history.columns

Index(['w_biokids_hh', 'w_biokids_out2', 'w_biokids_num',
       'w_biokids_num_alive', 'w_numkids_husb', 'w_pregnow', 'w_preg_mo',
       'w_preg_learn', 'w_preg_want', 'w_preg_ac', 'w_preg_whynoac',
       'w_preg_whynoac_oth', 'w_preg_ac_num', 'w_preg_ac_mo', 'w_hus_go_ac',
       'w_preg_ac_eat', 'w_preg_ac_comp', 'w_hus_ac_advice', 'w_hus_ac_where',
       'w_hus_ac_nutr', 'w_hus_ac_deliver', 'w_hus_ac_kids',
       'w_hus_ac_whenkid', 'w_preg_mal', 'w_preg_mal_num', 'w_preg_comp',
       'w_preg_comp_list', 'w_hus_aware_comp', 'w_hus_assist_comp',
       'w_hus_assist_how', 'w_hus_assist_how_oth', 'w_preg_foods',
       'w_preg_foods_list', 'w_pregpast2', 'w_preg_ac2', 'w_preg_whynoac2',
       'w_preg_whynoac_oth2', 'w_preg_ac_num2', 'w_preg_ac_mo2',
       'w_hus_go_ac2', 'w_preg_ac_eat2', 'w_preg_ac_comp2', 'w_hus_ac_advice2',
       'w_hus_ac_where2', 'w_hus_ac_nutr2', 'w_hus_ac_deliver2',
       'w_hus_ac_kids2', 'w_hus_ac_whenkid2', 'w_preg_mal2', 'w_preg_mal_num2',
       

#### 5.1- w_biokids_hh

In [ ]:
extract_information(reproductive_history,'w_biokids_hh')

w_biokids_hh    float64
dtype: object
0      5.0
1      6.0
2      4.0
3      2.0
4      1.0
5      3.0
6      0.0
7      9.0
8      7.0
9      8.0
10     NaN
11    10.0
dtype: float64
1
0.018142235123367198
w_biokids_hh
1.0     1221
2.0     1089
3.0      937
4.0      811
5.0      587
6.0      373
0.0      224
7.0      197
8.0       57
9.0       11
10.0       4
Name: count, dtype: int64
       w_biokids_hh
count   5511.000000
mean       3.012339
std        1.889563
min        0.000000
25%        1.000000
50%        3.000000
75%        4.000000
max       10.000000


#### 5.2- w_biokids_out2

In [ ]:
extract_information(reproductive_history,'w_biokids_out2')

w_biokids_out2    float64
dtype: object
0    0.0
1    1.0
2    2.0
3    3.0
4    4.0
5    5.0
6    6.0
7    8.0
8    7.0
9    NaN
dtype: float64
1
0.018142235123367198
w_biokids_out2
0.0    4063
1.0     917
2.0     338
3.0     125
4.0      40
5.0      17
6.0       7
7.0       3
8.0       1
Name: count, dtype: int64
       w_biokids_out2
count     5511.000000
mean         0.414444
std          0.849406
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          8.000000


#### 5.3- w_biokids_num

In [ ]:
extract_information(reproductive_history,'w_biokids_num')

w_biokids_num    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_biokids_num
1.0    5388
0.0     122
Name: count, dtype: int64
       w_biokids_num
count    5510.000000
mean        0.977858
std         0.147157
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000


#### 5.4- w_biokids_num_alive

In [ ]:
extract_information(reproductive_history,'w_biokids_num_alive')

w_biokids_num_alive    float64
dtype: object
0      NaN
1      6.0
2      1.0
3      5.0
4      8.0
5      7.0
6      9.0
7     11.0
8      2.0
9      3.0
10     0.0
11     4.0
12    10.0
13    12.0
dtype: float64
5375
97.51451378809868
w_biokids_num_alive
8.0     24
1.0     22
7.0     21
6.0     19
5.0     13
9.0     10
3.0      9
4.0      7
10.0     4
0.0      3
11.0     2
2.0      2
12.0     1
Name: count, dtype: int64
       w_biokids_num_alive
count           137.000000
mean              5.569343
std               2.899786
min               0.000000
25%               3.000000
50%               6.000000
75%               8.000000
max              12.000000


#### 5.5- w_numkids_husb

In [ ]:
extract_information(reproductive_history,'w_numkids_husb')

w_numkids_husb    float64
dtype: object
0      NaN
1      6.0
2      1.0
3      9.0
4      2.0
5      5.0
6      4.0
7      3.0
8      7.0
9      0.0
10    10.0
11     8.0
dtype: float64
4994
90.60232220609579
w_numkids_husb
1.0     185
2.0     100
3.0      73
4.0      42
5.0      39
6.0      32
7.0      19
0.0      12
8.0       9
9.0       5
10.0      2
Name: count, dtype: int64
       w_numkids_husb
count      518.000000
mean         2.758687
std          2.067916
min          0.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         10.000000


#### 5.6- w_pregnow

In [ ]:
extract_information(reproductive_history,'w_pregnow')

w_pregnow    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
w_pregnow
0    4352
1    1160
Name: count, dtype: int64
         w_pregnow
count  5512.000000
mean      0.210450
std       0.407665
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 5.7- w_preg_mo

In [ ]:
extract_information(reproductive_history,'w_preg_mo')

w_preg_mo    float64
dtype: object
0      5.0
1      NaN
2      6.0
3      3.0
4      8.0
5      7.0
6      4.0
7      9.0
8      2.0
9      1.0
10     8.5
11     6.5
12    10.0
13     1.5
dtype: float64
4352
78.95500725689405
w_preg_mo
8.0     197
7.0     170
4.0     139
3.0     134
5.0     128
6.0     123
2.0     118
9.0      96
1.0      49
8.5       2
10.0      2
6.5       1
1.5       1
Name: count, dtype: int64
         w_preg_mo
count  1160.000000
mean      5.427586
std       2.369987
min       1.000000
25%       3.000000
50%       6.000000
75%       8.000000
max      10.000000


#### 5.8- w_preg_learn

In [ ]:
extract_information(reproductive_history,'w_preg_learn')

w_preg_learn    float64
dtype: object
0    1.0
1    NaN
2    3.0
3    2.0
4    0.0
5    6.0
6    4.0
7    5.0
8    8.0
dtype: float64
4352
78.95500725689405
w_preg_learn
1.0    613
2.0    341
3.0    137
4.0     46
0.0     11
5.0      7
6.0      4
8.0      1
Name: count, dtype: int64
       w_preg_learn
count   1160.000000
mean       1.687069
std        0.941520
min        0.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        8.000000


#### 5.9- w_preg_want

In [ ]:
extract_information(reproductive_history,'w_preg_want')

w_preg_want    float64
dtype: object
0    2.0
1    NaN
2    1.0
3    3.0
dtype: float64
4352
78.95500725689405
w_preg_want
2.0    538
1.0    483
3.0    139
Name: count, dtype: int64
       w_preg_want
count  1160.000000
mean      1.703448
std       0.669814
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       3.000000


#### 5.10- w_preg_ac

In [ ]:
extract_information(reproductive_history,'w_preg_ac')

w_preg_ac    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
4352
78.95500725689405
w_preg_ac
1.0    748
0.0    412
Name: count, dtype: int64
         w_preg_ac
count  1160.000000
mean      0.644828
std       0.478772
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 5.11- w_preg_whynoac

In [ ]:
extract_information(reproductive_history,'w_preg_whynoac')

w_preg_whynoac    object
dtype: object
0                                            NaN
1                                          Other
2                                   Did not need
3                         Too early in pregnancy
4                     Needed advanced care;Other
5                               Could not afford
6                             Referred elsewhere
7                       Referred elsewhere;Other
8                         Could not afford;Other
9        Could not afford;Too early in pregnancy
10                  Too early in pregnancy;Other
11                                    Don't Know
12                            Did not need;Other
13    Too early in pregnancy;Husband was opposed
14               Could not afford;Feared midwife
15           Did not need;Too early in pregnancy
16       Staff are not polite;Referred elsewhere
17                    Staff are not polite;Other
18                          Feared midwife;Other
19                          Ne

#### 5.12- w_preg_whynoac_oth

In [ ]:
extract_information(reproductive_history,'w_preg_whynoac_oth')

w_preg_whynoac_oth    object
dtype: object
0                                                   NaN
1     she didn't have anyone to leave with the young...
2                                     no maternity wear
3                                 going there next week
4                                      haven't got time
                            ...                        
88                                    she is still fine
89                       was Still busy with house work
90                      services are far, has been sick
91    waiting for the midwives/nurses who go village...
92                               was down with sickness
Length: 93, dtype: object
5419
98.31277213352685
w_preg_whynoac_oth
no maternity wear                                                     2
she didn't have anyone to leave with the youngest child               1
going there next week                                                 1
haven't got time                                         

#### 5.13- w_preg_ac_num

In [ ]:
extract_information(reproductive_history,'w_preg_ac_num')

w_preg_ac_num    float64
dtype: object
0      1.0
1      NaN
2      3.0
3      2.0
4      4.0
5      5.0
6      8.0
7      6.0
8      0.0
9     10.0
10     9.0
11     7.0
dtype: float64
4760
86.35703918722787
w_preg_ac_num
1.0     227
2.0     220
3.0     156
4.0      95
5.0      31
6.0      13
8.0       4
0.0       3
10.0      1
9.0       1
7.0       1
Name: count, dtype: int64
       w_preg_ac_num
count     752.000000
mean        2.401596
std         1.376383
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        10.000000


#### 5.14- w_preg_ac_mo

In [ ]:
extract_information(reproductive_history,'w_preg_ac_mo')

w_preg_ac_mo    float64
dtype: object
0     4.0
1     NaN
2     3.0
3     7.0
4     2.0
5     5.0
6     6.0
7     8.0
8     1.0
9     9.0
10    4.5
11    0.0
12    2.5
dtype: float64
4762
86.3933236574746
w_preg_ac_mo
3.0    230
4.0    175
5.0    101
6.0     80
2.0     79
1.0     44
7.0     25
8.0      9
9.0      3
0.0      2
4.5      1
2.5      1
Name: count, dtype: int64
       w_preg_ac_mo
count    750.000000
mean       3.810667
std        1.556508
min        0.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        9.000000


#### 5.15- w_hus_go_ac

In [ ]:
extract_information(reproductive_history,'w_hus_go_ac')

w_hus_go_ac    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4763
86.41146589259797
w_hus_go_ac
0.0    555
1.0    194
Name: count, dtype: int64
       w_hus_go_ac
count   749.000000
mean      0.259012
std       0.438385
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 5.16- w_preg_ac_eat

In [ ]:
extract_information(reproductive_history,'w_preg_ac_eat')

w_preg_ac_eat    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4763
86.41146589259797
w_preg_ac_eat
0.0    392
1.0    357
Name: count, dtype: int64
       w_preg_ac_eat
count     749.000000
mean        0.476636
std         0.499788
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000


#### 5.17- w_preg_ac_comp

In [ ]:
extract_information(reproductive_history,'w_preg_ac_comp')

w_preg_ac_comp    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4764
86.42960812772134
w_preg_ac_comp
0.0    534
1.0    214
Name: count, dtype: int64
       w_preg_ac_comp
count      748.000000
mean         0.286096
std          0.452237
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


#### 5.18- w_hus_ac_advice

In [ ]:
extract_information(reproductive_history,'w_hus_ac_advice')

w_hus_ac_advice    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4764
86.42960812772134
w_hus_ac_advice
1.0    533
0.0    215
Name: count, dtype: int64
       w_hus_ac_advice
count       748.000000
mean          0.712567
std           0.452868
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


#### 5.19- w_hus_ac_where

In [ ]:
extract_information(reproductive_history,'w_hus_ac_where')

w_hus_ac_where    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
4671
84.74238026124819
w_hus_ac_where
1.0    650
0.0    191
Name: count, dtype: int64
       w_hus_ac_where
count      841.000000
mean         0.772889
std          0.419214
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 5.20- w_hus_ac_nutr

In [ ]:
extract_information(reproductive_history,'w_hus_ac_nutr')

w_hus_ac_nutr    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4667
84.66981132075472
w_hus_ac_nutr
1.0    470
0.0    375
Name: count, dtype: int64
       w_hus_ac_nutr
count     845.000000
mean        0.556213
std         0.497124
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000


#### 5.21- w_hus_ac_deliver

In [ ]:
extract_information(reproductive_history,'w_hus_ac_deliver')

w_hus_ac_deliver    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4668
84.68795355587808
w_hus_ac_deliver
1.0    544
0.0    300
Name: count, dtype: int64
       w_hus_ac_deliver
count        844.000000
mean           0.644550
std            0.478933
min            0.000000
25%            0.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 5.22- w_hus_ac_kids

In [ ]:
extract_information(reproductive_history,'w_hus_ac_kids')

w_hus_ac_kids    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
4667
84.66981132075472
w_hus_ac_kids
0.0    447
1.0    398
Name: count, dtype: int64
       w_hus_ac_kids
count     845.000000
mean        0.471006
std         0.499454
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000


#### 5.23- w_hus_ac_whenkid

In [ ]:
extract_information(reproductive_history,'w_hus_ac_whenkid')

w_hus_ac_whenkid    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4667
84.66981132075472
w_hus_ac_whenkid
0.0    570
1.0    275
Name: count, dtype: int64
       w_hus_ac_whenkid
count        845.000000
mean           0.325444
std            0.468818
min            0.000000
25%            0.000000
50%            0.000000
75%            1.000000
max            1.000000


#### 5.24- w_preg_mal

In [ ]:
extract_information(reproductive_history,'w_preg_mal')

w_preg_mal    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4352
78.95500725689405
w_preg_mal
0.0    814
1.0    346
Name: count, dtype: int64
        w_preg_mal
count  1160.000000
mean      0.298276
std       0.457699
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 5.25- w_preg_mal_num

In [ ]:
extract_information(reproductive_history,'w_preg_mal_num')

w_preg_mal_num    float64
dtype: object
0     NaN
1     2.0
2     1.0
3     3.0
4     4.0
5     7.0
6    10.0
7     0.0
8     5.0
dtype: float64
5164
93.6865021770682
w_preg_mal_num
1.0     231
2.0      75
3.0      35
4.0       3
7.0       1
10.0      1
0.0       1
5.0       1
Name: count, dtype: int64
       w_preg_mal_num
count      348.000000
mean         1.494253
std          0.915742
min          0.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         10.000000


#### 5.26- w_preg_comp

In [ ]:
extract_information(reproductive_history,'w_preg_comp')

w_preg_comp    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
4352
78.95500725689405
w_preg_comp
0.0    911
1.0    249
Name: count, dtype: int64
       w_preg_comp
count  1160.000000
mean      0.214655
std       0.410760
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 5.27- w_preg_comp_list

In [ ]:
extract_information(reproductive_history,'w_preg_comp_list')

w_preg_comp_list    object
dtype: object
0                                               Other
1                                                 NaN
2                                    Vaginal Bleeding
3                                            Swelling
4                                 Positioning of baby
5                                         Convulsions
6                                      Swelling;Other
7                               Night blindness;Other
8                         Trouble w vision during day
9                                Convulsions;Swelling
10    Convulsions;Night blindness;High blood pressure
11                          High blood pressure;Other
12                          Positioning of baby;Other
13                                  Convulsions;Other
14                  Trouble w vision during day;Other
15       Vaginal Bleeding;Trouble w vision during day
16                       Convulsions;Vaginal Bleeding
17            Convulsions;Trouble w visio

#### 5.28- w_hus_aware_comp

In [ ]:
extract_information(reproductive_history,'w_hus_aware_comp')

w_hus_aware_comp    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
5260
95.42815674891146
w_hus_aware_comp
1.0    241
0.0     11
Name: count, dtype: int64
       w_hus_aware_comp
count        252.000000
mean           0.956349
std            0.204724
min            0.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 5.29- w_hus_assist_comp

In [ ]:
extract_information(reproductive_history,'w_hus_assist_comp')

w_hus_assist_comp    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
5279
95.77285921625544
w_hus_assist_comp
1.0    126
0.0    107
Name: count, dtype: int64
       w_hus_assist_comp
count         233.000000
mean            0.540773
std             0.499408
min             0.000000
25%             0.000000
50%             1.000000
75%             1.000000
max             1.000000


#### 5.30- w_hus_assist_how

In [ ]:
extract_information(reproductive_history,'w_hus_assist_how')

w_hus_assist_how    object
dtype: object
0                                                   NaN
1                        Took you to clinic;Home chores
2     Took you to clinic;Arranged transport to clini...
3                                     Home chores;Other
4       Took you to clinic;Arranged transport to clinic
5                                           Home chores
6                                                 Other
7     Gave $ to go to health clini;Arranged transpor...
8                                    Took you to clinic
9     Arranged transport to clinic;Was present at cl...
10                         Gave $ to go to health clini
11      Gave $ to go to health clini;Took you to clinic
12    Gave $ to go to health clini;Took you to clini...
13    Arranged transport to clinic;Was present at cl...
14                         Arranged transport to clinic
15    Took you to clinic;Arranged transport to clini...
16             Gave $ to go to health clini;Home chores
17     

#### 5.31- w_hus_assist_how_oth

In [ ]:
extract_information(reproductive_history,'w_hus_assist_how_oth')

w_hus_assist_how_oth    float64
dtype: object
0   NaN
dtype: float64
5512
100.0
Series([], Name: count, dtype: int64)
       w_hus_assist_how_oth
count                   0.0
mean                    NaN
std                     NaN
min                     NaN
25%                     NaN
50%                     NaN
75%                     NaN
max                     NaN


#### 5.32- w_preg_foods

In [ ]:
extract_information(reproductive_history,'w_preg_foods')

w_preg_foods    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
4353
78.97314949201741
w_preg_foods
1.0    615
0.0    544
Name: count, dtype: int64
       w_preg_foods
count   1159.000000
mean       0.530630
std        0.499276
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000


#### 5.33- w_preg_foods_list

In [ ]:
extract_information(reproductive_history,'w_preg_foods_list')

w_preg_foods_list    object
dtype: object
0                  Fruit;Meat;Greens or other vegetables
1                                                    NaN
2                        Meat;Greens or other vegetables
3                                   Posho/maize porridge
4                     Matooke;Greens or other vegetables
                             ...                        
153    Matooke;Sweet Potatoes;Rice;Posho/maize porrid...
154                Rice;Greens or other vegetables;Other
155                      Rice;Greens or other vegetables
156       Sweet Potatoes;Rice;Posho/maize porridge;Other
157    Matooke;Irish Potatoes;Greens or other vegetab...
Length: 158, dtype: object
4897
88.84252539912917
w_preg_foods_list
Matooke                                                                        44
Matooke;Other                                                                  28
Other                                                                          27
Meat             

#### 5.34- w_pregpast2

In [ ]:
extract_information(reproductive_history,'w_pregpast2')

w_pregpast2    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
24
0.43541364296081275
w_pregpast2
1.0    4618
0.0     870
Name: count, dtype: int64
       w_pregpast2
count  5488.000000
mean      0.841472
std       0.365268
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 5.35- w_preg_ac2

In [ ]:
extract_information(reproductive_history,'w_preg_ac2')

w_preg_ac2    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
897
16.27358490566038
w_preg_ac2
1.0    4532
0.0      83
Name: count, dtype: int64
        w_preg_ac2
count  4615.000000
mean      0.982015
std       0.132911
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 5.36- w_preg_whynoac2

In [ ]:
extract_information(reproductive_history,'w_preg_whynoac2')

w_preg_whynoac2    object
dtype: object
0                                           NaN
1                                      10 Other
2                               03 Did not need
3                                             .
4                           02 Could not afford
5                     04 Too early in pregnancy
6                             07 Feared midwife
7                         08 Referred elsewhere
8              04 Needed advanced care;10 Other
9               09 Husband was opposed;10 Other
10    03 Did not need;04 Too early in pregnancy
11                                05 Don't Know
12          02 Could not afford;03 Did not need
13                      01 Staff are not polite
14                 06 Was sent away by AC staff
15      04 Too early in pregnancy;05 Don't Know
dtype: object
5375
97.51451378809868
w_preg_whynoac2
.                                            56
04 Too early in pregnancy                    22
10 Other                                   

#### 5.37- w_preg_whynoac_oth2

In [ ]:
extract_information(reproductive_history,'w_preg_whynoac_oth2')

w_preg_whynoac_oth2    object
dtype: object
0                                                   NaN
1                       no midwife at the health centre
2     she didn't have anyone to leave with the young...
3                                     no maternity wear
4        peopl wl laugh at me cz the baby z stl yououng
                            ...                        
71                 she is weak and the distance is long
72                                    she is still fine
73                       was Still busy with house work
74                      services are far, has been sick
75                               was down with sickness
Length: 76, dtype: object
5436
98.62119013062409
w_preg_whynoac_oth2
no maternity wear                                          2
no midwife at the health centre                            1
she didn't have anyone to leave with the youngest child    1
peopl wl laugh at me cz the baby z stl yououng             1
waiting 3 months           

#### 5.38- w_preg_ac_num2

In [ ]:
extract_information(reproductive_history,'w_preg_ac_num2')

w_preg_ac_num2    float64
dtype: object
0      3.0
1      NaN
2      5.0
3      7.0
4      4.0
5      2.0
6      1.0
7     10.0
8      6.0
9      9.0
10     8.0
11    20.0
12     0.0
13    12.0
14    15.0
dtype: float64
979
17.761248185776488
w_preg_ac_num2
4.0     1692
3.0     1187
5.0      600
2.0      350
6.0      338
1.0      156
7.0       85
8.0       56
10.0      32
9.0       28
20.0       4
0.0        2
12.0       2
15.0       1
Name: count, dtype: int64
       w_preg_ac_num2
count     4533.000000
mean         3.959188
std          1.540384
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max         20.000000


#### 5.39- w_preg_ac_mo2

In [ ]:
extract_information(reproductive_history,'w_preg_ac_mo2')

w_preg_ac_mo2    float64
dtype: object
0     4.0
1     6.0
2     NaN
3     3.0
4     5.0
5     1.0
6     2.0
7     7.0
8     8.0
9     9.0
10    2.5
11    3.5
dtype: float64
983
17.833817126269956
w_preg_ac_mo2
3.0    1385
4.0    1272
5.0     684
6.0     416
2.0     404
7.0     161
1.0     136
8.0      55
9.0      13
2.5       2
3.5       1
Name: count, dtype: int64
       w_preg_ac_mo2
count    4529.000000
mean        3.929234
std         1.426115
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         9.000000


#### 5.40- w_hus_go_ac2

In [ ]:
extract_information(reproductive_history,'w_hus_go_ac2')

w_hus_go_ac2    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
985
17.87010159651669
w_hus_go_ac2
0.0    3429
1.0    1098
Name: count, dtype: int64
       w_hus_go_ac2
count   4527.000000
mean       0.242545
std        0.428669
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


#### 5.41- w_preg_ac_eat2

In [ ]:
extract_information(reproductive_history,'w_preg_ac_eat2')

w_preg_ac_eat2    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
984
17.851959361393323
w_preg_ac_eat2
1.0    2954
0.0    1574
Name: count, dtype: int64
       w_preg_ac_eat2
count     4528.000000
mean         0.652385
std          0.476266
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 5.42- w_preg_ac_comp2

In [ ]:
extract_information(reproductive_history,'w_preg_ac_comp2')

w_preg_ac_comp2    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
985
17.87010159651669
w_preg_ac_comp2
0.0    2768
1.0    1759
Name: count, dtype: int64
       w_preg_ac_comp2
count      4527.000000
mean          0.388558
std           0.487476
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000


#### 5.43- w_hus_ac_advice2

In [ ]:
extract_information(reproductive_history,'w_hus_ac_advice2')

w_hus_ac_advice2    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
985
17.87010159651669
w_hus_ac_advice2
1.0    3291
0.0    1236
Name: count, dtype: int64
       w_hus_ac_advice2
count       4527.000000
mean           0.726972
std            0.445565
min            0.000000
25%            0.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 5.44- w_hus_ac_where2

In [ ]:
extract_information(reproductive_history,'w_hus_ac_where2')

w_hus_ac_where2    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
902
16.364296081277214
w_hus_ac_where2
1.0    3497
0.0    1113
Name: count, dtype: int64
       w_hus_ac_where2
count      4610.000000
mean          0.758568
std           0.427998
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           1.000000


#### 5.45- w_hus_ac_nutr2

In [ ]:
extract_information(reproductive_history,'w_hus_ac_nutr2')

w_hus_ac_nutr2    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
902
16.364296081277214
w_hus_ac_nutr2
1.0    2910
0.0    1700
Name: count, dtype: int64
       w_hus_ac_nutr2
count     4610.000000
mean         0.631236
std          0.482522
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 5.46- w_hus_ac_deliver2

In [ ]:
extract_information(reproductive_history,'w_hus_ac_deliver2')

w_hus_ac_deliver2    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
901
16.346153846153847
w_hus_ac_deliver2
1.0    3599
0.0    1012
Name: count, dtype: int64
       w_hus_ac_deliver2
count        4611.000000
mean            0.780525
std             0.413936
min             0.000000
25%             1.000000
50%             1.000000
75%             1.000000
max             1.000000


#### 5.47- w_hus_ac_kids2

In [ ]:
extract_information(reproductive_history,'w_hus_ac_kids2')

w_hus_ac_kids2    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
902
16.364296081277214
w_hus_ac_kids2
1.0    3089
0.0    1521
Name: count, dtype: int64
       w_hus_ac_kids2
count     4610.000000
mean         0.670065
std          0.470240
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 5.48- w_hus_ac_whenkid2

In [ ]:
extract_information(reproductive_history,'w_hus_ac_whenkid2')

w_hus_ac_whenkid2    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
902
16.364296081277214
w_hus_ac_whenkid2
1.0    2311
0.0    2299
Name: count, dtype: int64
       w_hus_ac_whenkid2
count        4610.000000
mean            0.501302
std             0.500053
min             0.000000
25%             0.000000
50%             1.000000
75%             1.000000
max             1.000000


#### 5.49- w_preg_mal2

In [ ]:
extract_information(reproductive_history,'w_preg_mal2')

w_preg_mal2    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
903
16.38243831640058
w_preg_mal2
0.0    2799
1.0    1810
Name: count, dtype: int64
       w_preg_mal2
count  4609.000000
mean      0.392710
std       0.488406
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 5.50- w_preg_mal_num2

In [ ]:
extract_information(reproductive_history,'w_preg_mal_num2')

w_preg_mal_num2    float64
dtype: object
0      NaN
1      9.0
2      1.0
3      2.0
4      0.0
5      3.0
6      5.0
7      6.0
8      4.0
9      7.0
10    14.0
11     8.0
12    15.0
13    10.0
dtype: float64
3694
67.01741654571843
w_preg_mal_num2
1.0     973
2.0     487
3.0     228
4.0      61
5.0      31
6.0      12
0.0       9
9.0       5
7.0       5
8.0       3
14.0      2
15.0      1
10.0      1
Name: count, dtype: int64
       w_preg_mal_num2
count      1818.000000
mean          1.792629
std           1.257410
min           0.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          15.000000


#### 5.51- w_preg_comp2

In [ ]:
extract_information(reproductive_history,'w_preg_comp2')

w_preg_comp2    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
901
16.346153846153847
w_preg_comp2
0.0    3671
1.0     940
Name: count, dtype: int64
       w_preg_comp2
count    4611.00000
mean        0.20386
std         0.40291
min         0.00000
25%         0.00000
50%         0.00000
75%         0.00000
max         1.00000


#### 5.52- w_preg_comp_list2

In [ ]:
extract_information(reproductive_history,'w_preg_comp_list2')

w_preg_comp_list2    object
dtype: object
0                                   03 Vaginal Bleeding
1                                                   NaN
2                                              08 Other
3                                07 Positioning of baby
4                                        01 Convulsions
5                                           02 Swelling
6        01 Convulsions;02 Swelling;03 Vaginal Bleeding
7     01 Convulsions;05 Night blindness;06 High bloo...
8                    02 Swelling;07 Positioning of baby
9                        04 Trouble w vision during day
10                                 02 Swelling;08 Other
11                               06 High blood pressure
12    03 Vaginal Bleeding;07 Positioning of baby;08 ...
13                      06 High blood pressure;08 Other
14        06 High blood pressure;07 Positioning of baby
15           03 Vaginal Bleeding;07 Positioning of baby
16                                   05 Night blindness
17    

#### 5.53- w_birth_comp2

In [ ]:
extract_information(reproductive_history,'w_birth_comp2')

w_birth_comp2    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
902
16.364296081277214
w_birth_comp2
0.0    3896
1.0     714
Name: count, dtype: int64
       w_birth_comp2
count    4610.000000
mean        0.154881
std         0.361830
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000


#### 5.54- w_birth_comp_list2

In [ ]:
extract_information(reproductive_history,'w_birth_comp_list2')

w_birth_comp_list2    object
dtype: object
0                                              08 Other
1                                                   NaN
2                                07 Positioning of baby
3                                 03 Placenta ejections
4     04 Trouble w vision during day;06 High blood p...
5                                06 High blood pressure
6                       07 Positioning of baby;08 Other
7                                        01 Convulsions
8     02 Swelling;05 Night blindness;06 High blood p...
9                   01 Convulsions;02 Swelling;08 Other
10         03 Placenta ejections;07 Positioning of baby
11                                          02 Swelling
12                           01 Convulsions;02 Swelling
13                       03 Placenta ejections;08 Other
14                       04 Trouble w vision during day
15          02 Swelling;07 Positioning of baby;08 Other
16                01 Convulsions;07 Positioning of baby
17   

#### 5.55- w_hus_aware_comp2

In [ ]:
extract_information(reproductive_history,'w_hus_aware_comp2')

w_hus_aware_comp2    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
4783
86.7743105950653
w_hus_aware_comp2
1.0    596
0.0    133
Name: count, dtype: int64
       w_hus_aware_comp2
count         729.000000
mean            0.817558
std             0.386473
min             0.000000
25%             1.000000
50%             1.000000
75%             1.000000
max             1.000000


#### 5.56- w_hus_assist_comp2

In [ ]:
extract_information(reproductive_history,'w_hus_assist_comp2')

w_hus_assist_comp2    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4888
88.67924528301887
w_hus_assist_comp2
1.0    534
0.0     90
Name: count, dtype: int64
       w_hus_assist_comp2
count          624.000000
mean             0.855769
std              0.351605
min              0.000000
25%              1.000000
50%              1.000000
75%              1.000000
max              1.000000


#### 5.57- w_hus_assist_how2

In [ ]:
extract_information(reproductive_history,'w_hus_assist_how2')

w_hus_assist_how2    object
dtype: object
0                                                   NaN
1                                        06 Home chores
2                               06 Home chores;07 Other
3         01 Gave $ to go to health clin;06 Home chores
4         02 Took you to clinic;06 Home chores;07 Other
                            ...                        
66    01 Gave $ to go to health clin;03 Got someone ...
67    02 Took you to clinic;03 Got someone else to g...
68    01 Gave $ to go to health clin;02 Took you to ...
69    01 Gave $ to go to health clin;02 Took you to ...
70              03 Got someone else to go w yo;07 Other
Length: 71, dtype: object
4969
90.1487663280116
w_hus_assist_how2
02 Took you to clinic                                                                                                                43
01 Gave $ to go to health clin                                                                                                       40
06 H

#### 5.58- w_hus_assist_how_oth2

In [ ]:
extract_information(reproductive_history,'w_hus_assist_how_oth2')

w_hus_assist_how_oth2    float64
dtype: object
0   NaN
dtype: float64
5512
100.0
Series([], Name: count, dtype: int64)
       w_hus_assist_how_oth2
count                    0.0
mean                     NaN
std                      NaN
min                      NaN
25%                      NaN
50%                      NaN
75%                      NaN
max                      NaN


#### 5.59- w_hus_aware_bcomp2

In [ ]:
extract_information(reproductive_history,'w_hus_aware_bcomp2')

w_hus_aware_bcomp2    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
4788
86.86502177068215
w_hus_aware_bcomp2
1.0    656
0.0     68
Name: count, dtype: int64
       w_hus_aware_bcomp2
count          724.000000
mean             0.906077
std              0.291923
min              0.000000
25%              1.000000
50%              1.000000
75%              1.000000
max              1.000000


#### 5.60- w_hus_assist_bcomp2

In [ ]:
extract_information(reproductive_history,'w_hus_assist_bcomp2')

w_hus_assist_bcomp2    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4856
88.09869375907112
w_hus_assist_bcomp2
1.0    595
0.0     61
Name: count, dtype: int64
       w_hus_assist_bcomp2
count           656.000000
mean              0.907012
std               0.290637
min               0.000000
25%               1.000000
50%               1.000000
75%               1.000000
max               1.000000


#### 5.61- w_hus_assist_bhow2

In [ ]:
extract_information(reproductive_history,'w_hus_assist_bhow2')

w_hus_assist_bhow2    object
dtype: object
0                                                   NaN
1     01 Gave $ to go to health clin;04 Arranged tra...
2                                        06 Home chores
3     04 Arranged transport to clini;06 Home chores;...
4         02 Took you to clinic;06 Home chores;07 Other
                            ...                        
66    01 Gave $ to go to health clin;02 Took you to ...
67    01 Gave $ to go to health clin;03 Got someone ...
68    05 Was present at clicinc;06 Home chores;07 Other
69    02 Took you to clinic;04 Arranged transport to...
70    01 Gave $ to go to health clin;02 Took you to ...
Length: 71, dtype: object
4919
89.24165457184326
w_hus_assist_bhow2
02 Took you to clinic                                                                                                                37
06 Home chores                                                                                                                       33
0

#### 5.62- w_hus_assist_bhow_oth2

In [ ]:
extract_information(reproductive_history,'w_hus_assist_bhow_oth2')

w_hus_assist_bhow_oth2    float64
dtype: object
0   NaN
dtype: float64
5512
100.0
Series([], Name: count, dtype: int64)
       w_hus_assist_bhow_oth2
count                     0.0
mean                      NaN
std                       NaN
min                       NaN
25%                       NaN
50%                       NaN
75%                       NaN
max                       NaN


#### 5.63- w_preg_foods2

In [ ]:
extract_information(reproductive_history,'w_preg_foods2')

w_preg_foods2    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
902
16.364296081277214
w_preg_foods2
1.0    2782
0.0    1828
Name: count, dtype: int64
       w_preg_foods2
count    4610.000000
mean        0.603471
std         0.489230
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000


#### 5.64- w_preg_foods_list2

In [ ]:
extract_information(reproductive_history,'w_preg_foods_list2')

w_preg_foods_list2    object
dtype: object
0                                                    NaN
1                                              Fish;Meat
2               06 Fruit;Meat;Greens or other vegetables
3      03 Sweet Potatoes;Posho/maize porridge;Greens ...
4                 03 Sweet Potatoes;Posho/maize porridge
                             ...                        
323    02 Irish Potatoes;Fish;Meat;Greens or other ve...
324     01 Matooke;Fish;Greens or other vegetables;Other
325              01 Matooke;03 Sweet Potatoes;Fish;Other
326              01 Matooke;03 Sweet Potatoes;Meat;Other
327                          01 Matooke;05 Rice;06 Fruit
Length: 328, dtype: object
2730
49.528301886792455
w_preg_foods_list2
Meat                                                      165
Other                                                     164
01 Matooke                                                147
06 Fruit                                                  142
01 Matooke;O

#### 5.65- w_where_birth

In [ ]:
extract_information(reproductive_history,'w_where_birth')

w_where_birth    float64
dtype: object
0    4.0
1    1.0
2    5.0
3    NaN
4    2.0
5    7.0
6    6.0
7    3.0
8    8.0
9    9.0
dtype: float64
901
16.346153846153847
w_where_birth
1.0    1336
4.0    1102
5.0    1057
7.0     718
2.0     267
3.0      80
6.0      36
8.0      12
9.0       3
Name: count, dtype: int64
       w_where_birth
count    4611.000000
mean        3.723270
std         2.121519
min         1.000000
25%         1.000000
50%         4.000000
75%         5.000000
max         9.000000


#### 5.66- w_hus_assist_birth

In [ ]:
extract_information(reproductive_history,'w_hus_assist_birth')

w_hus_assist_birth    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
901
16.346153846153847
w_hus_assist_birth
1.0    3391
0.0    1220
Name: count, dtype: int64
       w_hus_assist_birth
count         4611.000000
mean             0.735415
std              0.441160
min              0.000000
25%              0.000000
50%              1.000000
75%              1.000000
max              1.000000


#### 5.67- w_hus_assist_birth_how

In [ ]:
extract_information(reproductive_history,'w_hus_assist_birth_how')

w_hus_assist_birth_how    object
dtype: object
0                                                    NaN
1               06 Present during delivery;08 Did chores
2      01 Gave you money/bought thin;04 Arranged for ...
3      06 Present during delivery;07 Got family membe...
4      01 Gave you money/bought thin;07 Got family me...
                             ...                        
176    01 Gave you money/bought thin;03 Got someone e...
177    05 Arranged transport to clini;06 Present duri...
178    02 Took you to health facility;07 Got family m...
179    01 Gave you money/bought thin;03 Got someone e...
180    01 Gave you money/bought thin;03 Got someone e...
Length: 181, dtype: object
2118
38.425253991291726
w_hus_assist_birth_how
08 Did chores                                                                                                                                                                           265
02 Took you to health facility                                  

#### 5.68- w_hus_assist_birth_oth

In [ ]:
extract_information(reproductive_history,'w_hus_assist_birth_oth')

w_hus_assist_birth_oth    object
dtype: object
0                                     NaN
1                              paid bills
2                             broght food
3                              paid Bills
4               held the back of his wife
                      ...                
110                           Buying food
111                 brought a TBA to help
112                 helped her to deliver
113    arranged for a doctor to.come home
114          arranged for midwife to come
Length: 115, dtype: object
5352
97.09724238026125
w_hus_assist_birth_oth
paid bills                            18
nothing                               11
paid Bills                             7
cut the umbrical cord                  3
bought her a soda                      2
                                      ..
Buying food                            1
brought a TBA to help                  1
helped her to deliver                  1
arranged for a doctor to.come home     1
arranged

### 6- LAST_PREGNANCY Section

In [ ]:
last_pregnancy = parent_all_sections['LAST_PREGNANCY']
last_pregnancy.columns

Index(['w_lastpreg_ac', 'w_lastpreg_foods', 'w_bfeed_atemore',
       'w_time_breastfeed_weeks', 'w_breastfed_weeks_bi'],
      dtype='object')

#### 6.1- w_lastpreg_ac

In [ ]:
extract_information(last_pregnancy,'w_lastpreg_ac')

w_lastpreg_ac    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
321
5.8236574746008705
w_lastpreg_ac
1.0    4745
0.0     446
Name: count, dtype: int64
       w_lastpreg_ac
count    5191.000000
mean        0.914082
std         0.280270
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000


#### 6.2- w_lastpreg_foods

In [ ]:
extract_information(last_pregnancy,'w_lastpreg_foods')

w_lastpreg_foods    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
155
2.812046444121916
w_lastpreg_foods
1.0    3191
0.0    2166
Name: count, dtype: int64
       w_lastpreg_foods
count       5357.000000
mean           0.595669
std            0.490808
min            0.000000
25%            0.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 6.3- w_bfeed_atemore

In [ ]:
extract_information(last_pregnancy,'w_bfeed_atemore')

w_bfeed_atemore    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
927
16.817851959361395
w_bfeed_atemore
0.0    2736
1.0    1849
Name: count, dtype: int64
       w_bfeed_atemore
count      4585.000000
mean          0.403272
std           0.490608
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000


#### 6.4- w_time_breastfeed_weeks

In [ ]:
extract_information(last_pregnancy,'w_time_breastfeed_weeks')

w_time_breastfeed_weeks    float64
dtype: object
0     0.005952
1     0.002976
2     0.011905
3     0.000496
4          NaN
        ...   
62    1.571429
63    0.002778
64    0.004762
65    0.001290
66    0.714286
Length: 67, dtype: float64
926
16.799709724238028
w_time_breastfeed_weeks
0.005952    1071
0.002976     765
0.011905     397
0.000496     335
0.000992     310
            ... 
1.571429       1
0.002778       1
0.004762       1
0.001290       1
0.714286       1
Name: count, Length: 66, dtype: int64
       w_time_breastfeed_weeks
count              4586.000000
mean                  0.040747
std                   0.432824
min                   0.000000
25%                   0.001488
50%                   0.005456
75%                   0.011905
max                  20.000000


#### 6.5- w_breastfed_weeks_bi

In [ ]:
extract_information(last_pregnancy,'w_breastfed_weeks_bi')

w_breastfed_weeks_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
926
16.799709724238028
w_breastfed_weeks_bi
0.0    2308
1.0    2278
Name: count, dtype: int64
       w_breastfed_weeks_bi
count           4586.000000
mean               0.496729
std                0.500044
min                0.000000
25%                0.000000
50%                0.000000
75%                1.000000
max                1.000000


### 7- INFANT_CARE Section

In [ ]:
infant_care = parent_all_sections['INFANT_CARE']
infant_care.columns

Index(['w_hus_disc_care', 'w_birth_weighed', 'w_birth_weight',
       'w_weight_immcard', 'w_birth_size', 'w_check2mo', 'w_mom_health2mo',
       'w_mom_healthnow', 'w_first_check', 'w_first_check_unit',
       'w_first_check_weeks', 'w_first_check_weeks_bi', 'w_weight_1mo',
       'w_weight_1mo_card', 'w_weight_3mo', 'w_weight_3mo_card',
       'w_breastfeed', 'w_time_breastfeed', 'w_time_bfeed_unit', 'w_colostrum',
       'w_time_milk', 'w_time_milk_units', 'w_othliq_1week', 'w_othliq_3mo',
       'w_othliq_what', 'w_othliq_other', 'w_food_3mo', 'w_bfeed_eat', 'iron',
       'vita_6wk', 'vita_6mo', 'total_vacc', 'w_hus_ac', 'w_hus_feed',
       'w_hus_preg', 'w_hus_hyg', 'w_fp_decide', 'w_fp_talk', 'w_ideal_fert_t',
       'w_ideal_fert_b', 'w_ideal_fert_g', 'w_ideal_fert_e', 'w_nextpreg_when',
       'w_hus_ideal_fert', 'w_hus_ideal_fert_t', 'w_hus_ideal_fert_b',
       'w_hus_ideal_fert_g', 'w_hus_ideal_fert_e', 'w_smallkids',
       'w_earlykids', 'w_pregcomp'],
      dtype='objec

#### 7.1- w_hus_disc_care

In [ ]:
extract_information(infant_care,'w_hus_disc_care')

w_hus_disc_care    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
901
16.346153846153847
w_hus_disc_care
1.0    2778
0.0    1833
Name: count, dtype: int64
       w_hus_disc_care
count      4611.000000
mean          0.602472
std           0.489440
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


#### 7.2- w_birth_weighed

In [ ]:
extract_information(infant_care,'w_birth_weighed')

w_birth_weighed    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
913
16.563860667634252
w_birth_weighed
1.0    2327
0.0    2272
Name: count, dtype: int64
       w_birth_weighed
count      4599.000000
mean          0.505980
std           0.500019
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


#### 7.3- w_birth_weight

In [ ]:
extract_information(infant_care,'w_birth_weight')

w_birth_weight    float64
dtype: object
0     4.40
1      NaN
2     4.00
3     3.00
4     3.30
      ... 
93    3.17
94    4.67
95    6.70
96    3.37
97    6.10
Length: 98, dtype: float64
3205
58.145863570391874
w_birth_weight
3.00    450
3.50    277
4.00    199
2.50    116
3.20    112
       ... 
3.17      1
4.67      1
6.70      1
3.37      1
6.10      1
Name: count, Length: 97, dtype: int64
       w_birth_weight
count     2307.000000
mean         3.384556
std          0.832370
min          0.000000
25%          3.000000
50%          3.300000
75%          3.800000
max          8.000000


#### 7.4- w_weight_immcard

In [ ]:
extract_information(infant_care,'w_weight_immcard')

w_weight_immcard    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
3175
57.601596516690854
w_weight_immcard
1.0    1174
0.0    1163
Name: count, dtype: int64
       w_weight_immcard
count       2337.000000
mean           0.502353
std            0.500101
min            0.000000
25%            0.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 7.5- w_birth_size

In [ ]:
extract_information(infant_care,'w_birth_size')

w_birth_size    float64
dtype: object
0    1.0
1    3.0
2    5.0
3    4.0
4    2.0
5    NaN
dtype: float64
901
16.346153846153847
w_birth_size
3.0    2227
2.0    1293
4.0     621
1.0     331
5.0     139
Name: count, dtype: int64
       w_birth_size
count   4611.000000
mean       2.770982
std        0.877800
min        1.000000
25%        2.000000
50%        3.000000
75%        3.000000
max        5.000000


#### 7.6- w_check2mo

In [ ]:
extract_information(infant_care,'w_check2mo')

w_check2mo    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
904
16.400580551523948
w_check2mo
1.0    3588
0.0    1020
Name: count, dtype: int64
        w_check2mo
count  4608.000000
mean      0.778646
std       0.415203
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 7.7- w_mom_health2mo

In [ ]:
extract_information(infant_care,'w_mom_health2mo')

w_mom_health2mo    float64
dtype: object
0    3.0
1    2.0
2    NaN
3    1.0
4    4.0
dtype: float64
913
16.563860667634252
w_mom_health2mo
2.0    2478
3.0    1136
1.0     760
4.0     225
Name: count, dtype: int64
       w_mom_health2mo
count      4599.000000
mean          2.179604
std           0.758832
min           1.000000
25%           2.000000
50%           2.000000
75%           3.000000
max           4.000000


#### 7.8- w_mom_healthnow

In [ ]:
extract_information(infant_care,'w_mom_healthnow')

w_mom_healthnow    float64
dtype: object
0    NaN
1    1.0
2    2.0
3    3.0
4    4.0
dtype: float64
3432
62.264150943396224
w_mom_healthnow
2.0    1096
3.0     555
1.0     393
4.0      36
Name: count, dtype: int64
       w_mom_healthnow
count      2080.000000
mean          2.112500
std           0.715954
min           1.000000
25%           2.000000
50%           2.000000
75%           3.000000
max           4.000000


#### 7.9- w_first_check

In [ ]:
extract_information(infant_care,'w_first_check')

w_first_check    float64
dtype: object
0      1.0
1      0.0
2      8.0
3      3.0
4      NaN
5      2.0
6     10.0
7      4.0
8     30.0
9      6.0
10    16.0
11     5.0
12    12.0
13     7.0
14    21.0
15    20.0
16    99.0
17    28.0
18    11.0
19    15.0
20     9.0
21     1.5
22    13.0
23    25.0
24    14.0
25    72.0
26    17.0
27    88.0
28    32.0
29    24.0
30    45.0
31    98.0
32    40.0
33    23.0
34    33.0
35    18.0
36    54.0
37    48.0
38    38.0
39    36.0
40    44.0
41    77.0
42    90.0
43    53.0
44    60.0
45    35.0
46    27.0
47    89.0
48    19.0
49    96.0
50    78.0
51    66.0
52     4.5
53    50.0
dtype: float64
902
16.364296081277214
w_first_check
4.0     807
1.0     682
2.0     519
8.0     433
0.0     392
3.0     311
5.0     244
12.0    227
6.0     181
30.0    159
10.0    128
16.0     83
7.0      82
20.0     75
9.0      39
99.0     36
15.0     32
24.0     29
14.0     16
11.0     14
40.0     13
77.0     11
13.0     11
36.0      9
28.0      7
18.0      7
48.

#### 7.10- w_first_check_unit

In [ ]:
extract_information(infant_care,'w_first_check_unit')

w_first_check_unit    float64
dtype: object
0    2.0
1    4.0
2    1.0
3    NaN
4    3.0
5    5.0
dtype: float64
901
16.346153846153847
w_first_check_unit
4.0    2490
1.0     835
3.0     728
2.0     555
5.0       3
Name: count, dtype: int64
       w_first_check_unit
count         4611.000000
mean             3.058773
std              1.176512
min              1.000000
25%              2.000000
50%              4.000000
75%              4.000000
max              5.000000


#### 7.11- w_first_check_weeks

In [ ]:
extract_information(infant_care,'w_first_check_weeks')

w_first_check_weeks    float64
dtype: object
0       0.005952
1       0.000000
2       8.000000
3       3.000000
4       0.000298
         ...    
111     0.006548
112     0.008730
113    28.000000
114     0.026786
115     0.004960
Length: 116, dtype: float64
902
16.364296081277214
w_first_check_weeks
4.000000    756
8.000000    407
0.000000    390
0.142857    350
2.000000    228
           ... 
0.007738      1
0.006548      1
0.008730      1
0.026786      1
0.004960      1
Name: count, Length: 115, dtype: int64
       w_first_check_weeks
count          4610.000000
mean              4.479749
std              26.426263
min               0.000000
25%               0.011905
50%               2.000000
75%               4.285714
max             999.000000


#### 7.12- w_first_check_weeks_bi

In [ ]:
extract_information(infant_care,'w_first_check_weeks_bi')

w_first_check_weeks_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
902
16.364296081277214
w_first_check_weeks_bi
0.0    2361
1.0    2249
Name: count, dtype: int64
       w_first_check_weeks_bi
count             4610.000000
mean                 0.487852
std                  0.499907
min                  0.000000
25%                  0.000000
50%                  0.000000
75%                  1.000000
max                  1.000000


#### 7.13- w_weight_1mo

In [ ]:
extract_information(infant_care,'w_weight_1mo')

w_weight_1mo    float64
dtype: object
0     6.00
1     5.10
2     2.00
3     2.50
4     3.00
      ... 
78    3.20
79    4.75
80    7.40
81    5.25
82    5.35
Length: 83, dtype: float64
1968
35.703918722786646
w_weight_1mo
4.00    762
5.00    495
0.00    448
3.00    354
4.50    200
       ... 
3.20      1
4.75      1
7.40      1
5.25      1
5.35      1
Name: count, Length: 82, dtype: int64
       w_weight_1mo
count   3544.000000
mean       3.793885
std        1.762710
min        0.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        9.300000


#### 7.14- w_weight_1mo_card

In [ ]:
extract_information(infant_care,'w_weight_1mo_card')

w_weight_1mo_card    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
1041
18.886066763425255
w_weight_1mo_card
0.0    3294
1.0    1177
Name: count, dtype: int64
       w_weight_1mo_card
count        4471.000000
mean            0.263252
std             0.440447
min             0.000000
25%             0.000000
50%             0.000000
75%             1.000000
max             1.000000


#### 7.15- w_weight_3mo

In [ ]:
extract_information(infant_care,'w_weight_3mo')

w_weight_3mo    float64
dtype: object
0      6.90
1      6.55
2      5.00
3      4.00
4      9.00
      ...  
87    15.00
88     9.40
89     3.80
90     4.90
91     9.80
Length: 92, dtype: float64
2111
38.29825834542815
w_weight_3mo
6.0     565
5.0     488
0.0     359
7.0     271
4.0     203
       ... 
5.3       1
15.0      1
3.8       1
4.9       1
9.8       1
Name: count, Length: 91, dtype: int64
       w_weight_3mo
count   3401.000000
mean       5.168806
std        2.143030
min        0.000000
25%        4.500000
50%        5.500000
75%        6.400000
max       15.000000


#### 7.16- w_weight_3mo_card

In [ ]:
extract_information(infant_care,'w_weight_3mo_card')

w_weight_3mo_card    object
dtype: object
0    01 yes
1     02 no
2       NaN
3     99 DK
dtype: object
1270
23.040638606676342
w_weight_3mo_card
02 no     2735
01 yes    1502
99 DK        5
Name: count, dtype: int64
       w_weight_3mo_card
count               4242
unique                 3
top                02 no
freq                2735


#### 7.17- w_breastfeed

In [ ]:
extract_information(infant_care,'w_breastfeed')

w_breastfeed    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
901
16.346153846153847
w_breastfeed
1.0    4590
0.0      21
Name: count, dtype: int64
       w_breastfeed
count   4611.000000
mean       0.995446
std        0.067339
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


#### 7.18- w_time_breastfeed

In [ ]:
extract_information(infant_care,'w_time_breastfeed')

w_time_breastfeed    float64
dtype: object
0     60.0
1     30.0
2      2.0
3      1.0
4      5.0
5      NaN
6     25.0
7     10.0
8      3.0
9     15.0
10    40.0
11    20.0
12     4.0
13    50.0
14     8.0
15    12.0
16    90.0
17    45.0
18     6.0
19    24.0
20     9.0
21     0.0
22     7.0
23    14.0
24     1.5
25    11.0
26    99.0
27    18.0
28    38.0
29    13.0
30    23.0
31    17.0
32    16.0
33    21.0
34    35.0
35    28.0
36    48.0
dtype: float64
925
16.78156748911466
w_time_breastfeed
1.0     1241
30.0     773
2.0      591
5.0      395
3.0      355
10.0     331
20.0     222
15.0     133
4.0      132
6.0      104
40.0      68
8.0       46
7.0       31
12.0      31
60.0      26
50.0      15
0.0       14
25.0      14
90.0      10
9.0        9
45.0       8
24.0       7
14.0       5
11.0       4
35.0       4
38.0       3
13.0       3
21.0       2
48.0       2
17.0       2
1.5        1
99.0       1
18.0       1
23.0       1
16.0       1
28.0       1
Name: count, dtype: int64
 

#### 7.19- w_time_bfeed_unit

In [ ]:
extract_information(infant_care,'w_time_bfeed_unit')

w_time_bfeed_unit    float64
dtype: object
0    1.0
1    2.0
2    NaN
3    3.0
4    4.0
dtype: float64
926
16.799709724238028
w_time_bfeed_unit
1.0    2322
2.0    2004
3.0     237
4.0      23
Name: count, dtype: int64
       w_time_bfeed_unit
count        4586.000000
mean            1.555386
std             0.616818
min             1.000000
25%             1.000000
50%             1.000000
75%             2.000000
max             4.000000


#### 7.20- w_colostrum

In [ ]:
extract_information(infant_care,'w_colostrum')

w_colostrum    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
926
16.799709724238028
w_colostrum
1.0    4522
0.0      64
Name: count, dtype: int64
       w_colostrum
count  4586.000000
mean      0.986044
std       0.117319
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 7.21- w_time_milk

In [ ]:
extract_information(infant_care,'w_colostrum')

w_colostrum    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
926
16.799709724238028
w_colostrum
1.0    4522
0.0      64
Name: count, dtype: int64
       w_colostrum
count  4586.000000
mean      0.986044
std       0.117319
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 7.22- w_time_milk_units

In [ ]:
extract_information(infant_care,'w_time_milk_units')

w_time_milk_units    object
dtype: object
0       03 days
1      02 hours
2      04 weeks
3           NaN
4    01 minutes
dtype: object
925
16.78156748911466
w_time_milk_units
03 days       3514
02 hours       631
04 weeks       298
01 minutes     144
Name: count, dtype: int64
       w_time_milk_units
count               4587
unique                 4
top              03 days
freq                3514


#### 7.23- w_othliq_1week

In [ ]:
extract_information(infant_care,'w_othliq_1week')

w_othliq_1week    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
926
16.799709724238028
w_othliq_1week
0.0    3129
1.0    1457
Name: count, dtype: int64
       w_othliq_1week
count     4586.000000
mean         0.317706
std          0.465635
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


#### 7.24- w_othliq_3mo

In [ ]:
extract_information(infant_care,'w_othliq_3mo')

w_othliq_3mo    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
929
16.854136429608126
w_othliq_3mo
0.0    3447
1.0    1136
Name: count, dtype: int64
       w_othliq_3mo
count   4583.000000
mean       0.247873
std        0.431825
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


#### 7.25- w_othliq_what

In [ ]:
extract_information(infant_care,'w_othliq_what')

w_othliq_what    object
dtype: object
0               01 Milk (not breastmilk);02 Plain water
1                                        02 Plain water
2                                                   NaN
3                              03 Sugar/glucose w water
4                               02 Plain water;12 Other
5                              01 Milk (not breastmilk)
6                                              12 Other
7                                           11 Porridge
8                         02 Plain water;06 Fruit juice
9             01 Milk (not breastmilk);08 Tea/infusions
10                                    07 Infant formula
11                                     08 Tea/infusions
12                 01 Milk (not breastmilk);11 Porridge
13                      02 Plain water;08 Tea/infusions
14                                       04 Gripe water
15                           02 Plain water;11 Porridge
16              02 Plain water;03 Sugar/glucose w water
17    01 M

#### 7.26- w_othliq_other

In [ ]:
extract_information(infant_care,'w_othliq_other')

w_othliq_other    object
dtype: object
0                                   NaN
1                         mushroom soup
2                                   tea
3                                     .
4                          waterly Milk
5                               yoghurt
6                            Irish,rice
7                         Mushroom soup
8             Water With Some sore Herb
9                       Water And Herbs
10                      Water Mushrooms
11               raw egg, mushroom soup
12               Water And Millet Floor
13                      Soghum porredge
14                    traditional harbs
15          water from boiled mashrooms
16                       Mushroom Water
17                  Water And Mushrooms
18                                herbs
19                         herbal water
20                 boiled mushroom soup
21                   water from obutuzi
22                        Mushroom Soup
23                    soup for mushroom
2

#### 7.27- w_food_3mo

In [ ]:
extract_information(infant_care,'w_food_3mo')

w_food_3mo    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
934
16.944847605224965
w_food_3mo
1.0    4344
0.0     234
Name: count, dtype: int64
        w_food_3mo
count  4578.000000
mean      0.948886
std       0.220254
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 7.28- w_bfeed_eat

In [ ]:
extract_information(infant_care,'w_bfeed_eat')

w_bfeed_eat    float64
dtype: object
0    3.0
1    1.0
2    2.0
3    NaN
dtype: float64
927
16.817851959361395
w_bfeed_eat
2.0    2478
1.0    1849
3.0     258
Name: count, dtype: int64
       w_bfeed_eat
count  4585.000000
mean      1.652999
std       0.582414
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       3.000000


#### 7.29- iron

In [ ]:
extract_information(infant_care,'iron')

iron    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2248
40.78374455732946
iron
1.0    2353
0.0     911
Name: count, dtype: int64
              iron
count  3264.000000
mean      0.720895
std       0.448628
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 7.30- vita_6wk

In [ ]:
extract_information(infant_care,'vita_6wk')

vita_6wk    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
2247
40.76560232220609
vita_6wk
1.0    1771
0.0    1494
Name: count, dtype: int64
          vita_6wk
count  3265.000000
mean      0.542420
std       0.498274
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 7.31- vita_6mo

In [ ]:
extract_information(infant_care,'vita_6mo')

vita_6mo    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
2246
40.74746008708273
vita_6mo
0.0    1869
1.0    1397
Name: count, dtype: int64
          vita_6mo
count  3266.000000
mean      0.427740
std       0.494827
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 7.32- total_vacc

In [ ]:
extract_information(infant_care,'total_vacc')

total_vacc    float64
dtype: object
0     9.0
1     6.0
2     NaN
3     8.0
4     7.0
5     0.0
6     1.0
7     2.0
8     4.0
9     3.0
10    5.0
dtype: float64
2251
40.83817126269956
total_vacc
9.0    951
8.0    763
0.0    315
7.0    238
4.0    210
6.0    205
2.0    200
3.0    158
5.0    123
1.0     98
Name: count, dtype: int64
        total_vacc
count  3261.000000
mean      6.128795
std       3.092139
min       0.000000
25%       4.000000
50%       8.000000
75%       9.000000
max       9.000000


#### 7.33- w_hus_ac

In [ ]:
extract_information(infant_care,'w_hus_ac')

w_hus_ac    float64
dtype: object
0    1.0
1    2.0
2    4.0
3    3.0
4    NaN
5    5.0
dtype: float64
30
0.5442670537010159
w_hus_ac
1.0    1785
3.0    1481
2.0    1329
4.0     886
5.0       1
Name: count, dtype: int64
          w_hus_ac
count  5482.000000
mean      2.268333
std       1.082636
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       5.000000


#### 7.34- w_hus_feed

In [ ]:
extract_information(infant_care,'w_hus_feed')

w_hus_feed    float64
dtype: object
0    1.0
1    4.0
2    3.0
3    2.0
4    NaN
5    5.0
dtype: float64
32
0.5805515239477503
w_hus_feed
1.0    1855
3.0    1545
2.0    1482
4.0     597
5.0       1
Name: count, dtype: int64
        w_hus_feed
count  5480.000000
mean      2.161861
std       1.015793
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       5.000000


#### 7.35- w_hus_preg

In [ ]:
extract_information(infant_care,'w_hus_preg')

w_hus_preg    float64
dtype: object
0    3.0
1    1.0
2    4.0
3    2.0
4    NaN
5    5.0
dtype: float64
19
0.3447024673439768
w_hus_preg
1.0    1781
3.0    1627
2.0    1392
4.0     692
5.0       1
Name: count, dtype: int64
        w_hus_preg
count  5493.000000
mean      2.224468
std       1.037202
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       5.000000


#### 7.36- w_hus_hyg

In [ ]:
extract_information(infant_care,'w_hus_hyg')

w_hus_hyg    float64
dtype: object
0    1.0
1    3.0
2    2.0
3    4.0
4    NaN
dtype: float64
14
0.2539912917271408
w_hus_hyg
1.0    2719
2.0    1789
3.0     831
4.0     159
Name: count, dtype: int64
         w_hus_hyg
count  5498.000000
mean      1.714442
std       0.824590
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       4.000000


#### 7.37- w_fp_decide

In [ ]:
extract_information(infant_care,'w_fp_decide')

w_fp_decide    float64
dtype: object
0    3.0
1    2.0
2    1.0
3    NaN
dtype: float64
46
0.8345428156748911
w_fp_decide
3.0    3829
1.0    1097
2.0     540
Name: count, dtype: int64
       w_fp_decide
count  5466.000000
mean      2.499817
std       0.807161
min       1.000000
25%       2.000000
50%       3.000000
75%       3.000000
max       3.000000


#### 7.38- w_fp_talk

In [ ]:
extract_information(infant_care,'w_fp_talk')

w_fp_talk    float64
dtype: object
0    2.0
1    3.0
2    1.0
3    NaN
dtype: float64
7
0.1269956458635704
w_fp_talk
3.0    3385
2.0    1440
1.0     680
Name: count, dtype: int64
         w_fp_talk
count  5505.000000
mean      2.491371
std       0.705028
min       1.000000
25%       2.000000
50%       3.000000
75%       3.000000
max       3.000000


#### 7.39- w_ideal_fert_t

In [ ]:
extract_information(infant_care,'w_ideal_fert_t')

w_ideal_fert_t    float64
dtype: object
0      4.0
1      2.0
2      6.0
3      3.0
4      1.0
5      8.0
6      5.0
7     12.0
8      7.0
9      0.0
10     NaN
11    15.0
12    10.0
13    11.0
14     9.0
dtype: float64
7
0.1269956458635704
w_ideal_fert_t
4.0     2867
3.0      732
6.0      701
5.0      498
2.0      458
8.0       72
1.0       59
7.0       55
10.0      38
12.0       8
0.0        7
9.0        5
15.0       4
11.0       1
Name: count, dtype: int64
       w_ideal_fert_t
count     5505.000000
mean         4.157675
std          1.384949
min          0.000000
25%          4.000000
50%          4.000000
75%          5.000000
max         15.000000


#### 7.40- w_ideal_fert_b

In [ ]:
extract_information(infant_care,'w_ideal_fert_b')

w_ideal_fert_b    float64
dtype: object
0     3.0
1     0.0
2     2.0
3     1.0
4     4.0
5     7.0
6     NaN
7    10.0
8     5.0
9     6.0
dtype: float64
51
0.9252539912917271
w_ideal_fert_b
2.0     3092
3.0      933
1.0      662
0.0      577
4.0      163
5.0       21
6.0        8
7.0        4
10.0       1
Name: count, dtype: int64
       w_ideal_fert_b
count     5461.000000
mean         1.920527
std          0.957093
min          0.000000
25%          2.000000
50%          2.000000
75%          2.000000
max         10.000000


#### 7.41- w_ideal_fert_g

In [ ]:
extract_information(infant_care,'w_ideal_fert_g')

w_ideal_fert_g    float64
dtype: object
0     1.0
1     0.0
2     3.0
3     4.0
4     2.0
5     5.0
6     NaN
7     6.0
8    10.0
9     8.0
dtype: float64
51
0.9252539912917271
w_ideal_fert_g
2.0     3005
1.0      917
3.0      780
0.0      613
4.0      119
5.0       16
6.0        9
10.0       1
8.0        1
Name: count, dtype: int64
       w_ideal_fert_g
count     5461.000000
mean         1.811939
std          0.939552
min          0.000000
25%          1.000000
50%          2.000000
75%          2.000000
max         10.000000


#### 7.42- w_ideal_fert_e

In [ ]:
extract_information(infant_care,'w_ideal_fert_e')

w_ideal_fert_e    float64
dtype: object
0      0.0
1      2.0
2      3.0
3      4.0
4      5.0
5      NaN
6      1.0
7      6.0
8     15.0
9      8.0
10     7.0
11     9.0
12    12.0
dtype: float64
51
0.9252539912917271
w_ideal_fert_e
0.0     4885
4.0      220
3.0      112
2.0       80
5.0       56
6.0       50
1.0       42
8.0       10
7.0        3
15.0       1
9.0        1
12.0       1
Name: count, dtype: int64
       w_ideal_fert_e
count     5461.000000
mean         0.390954
std          1.247144
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         15.000000


#### 7.43- w_nextpreg_when

In [ ]:
extract_information(infant_care,'w_nextpreg_when')

w_nextpreg_when    float64
dtype: object
0     666.0
1      36.0
2      48.0
3      24.0
4      50.0
      ...  
60     68.0
61    108.0
62     49.0
63      2.5
64     20.0
Length: 65, dtype: float64
7
0.1269956458635704
w_nextpreg_when
666.0    1764
36.0      958
60.0      713
24.0      657
48.0      514
         ... 
59.0        1
68.0        1
53.0        1
49.0        1
20.0        1
Name: count, Length: 64, dtype: int64
       w_nextpreg_when
count      5505.000000
mean        242.259401
std         291.672411
min           0.000000
25%          36.000000
50%          60.000000
75%         666.000000
max         666.000000


#### 7.44- w_hus_ideal_fert

In [ ]:
extract_information(infant_care,'w_hus_ideal_fert')

w_hus_ideal_fert    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
9
0.1632801161103048
w_hus_ideal_fert
1.0    3522
0.0    1981
Name: count, dtype: int64
       w_hus_ideal_fert
count       5503.000000
mean           0.640015
std            0.480039
min            0.000000
25%            0.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 7.45- w_hus_ideal_fert_t

In [ ]:
extract_information(infant_care,'w_hus_ideal_fert_t')

w_hus_ideal_fert_t    float64
dtype: object
0      6.0
1      5.0
2      7.0
3      9.0
4      8.0
5      4.0
6      2.0
7      3.0
8     10.0
9      0.0
10     NaN
11     1.0
12    15.0
13    12.0
14    11.0
15    14.0
16    13.0
dtype: float64
8
0.14513788098693758
w_hus_ideal_fert_t
4.0     2062
6.0      933
5.0      653
3.0      586
2.0      421
10.0     259
8.0      235
7.0      196
1.0       39
12.0      36
15.0      32
9.0       25
0.0       15
11.0       5
13.0       5
14.0       2
Name: count, dtype: int64
       w_hus_ideal_fert_t
count         5504.000000
mean             4.883176
std              2.150603
min              0.000000
25%              4.000000
50%              4.000000
75%              6.000000
max             15.000000


#### 7.46- w_hus_ideal_fert_b

In [ ]:
extract_information(infant_care,'w_hus_ideal_fert_b')

w_hus_ideal_fert_b    float64
dtype: object
0      4.0
1      3.0
2      2.0
3      0.0
4      6.0
5      8.0
6      1.0
7      NaN
8      5.0
9     10.0
10    12.0
11     7.0
12    14.0
13     9.0
14    13.0
15    11.0
dtype: float64
279
5.061683599419449
w_hus_ideal_fert_b
2.0     2082
3.0     1141
0.0      812
1.0      483
4.0      480
5.0      115
6.0       73
8.0       16
10.0      12
7.0       11
9.0        4
12.0       1
14.0       1
13.0       1
11.0       1
Name: count, dtype: int64
       w_hus_ideal_fert_b
count         5233.000000
mean             2.181158
std              1.455525
min              0.000000
25%              2.000000
50%              2.000000
75%              3.000000
max             14.000000


#### 7.47- w_hus_ideal_fert_g

In [ ]:
extract_information(infant_care,'w_hus_ideal_fert_g')

w_hus_ideal_fert_g    float64
dtype: object
0      2.0
1      4.0
2      0.0
3      3.0
4      1.0
5      6.0
6      NaN
7      5.0
8      8.0
9      9.0
10    10.0
11     7.0
dtype: float64
279
5.061683599419449
w_hus_ideal_fert_g
2.0     2179
0.0     1055
1.0      854
3.0      808
4.0      236
5.0       49
6.0       39
8.0        5
7.0        5
9.0        2
10.0       1
Name: count, dtype: int64
       w_hus_ideal_fert_g
count         5233.000000
mean             1.750812
std              1.239101
min              0.000000
25%              1.000000
50%              2.000000
75%              2.000000
max             10.000000


#### 7.48- w_hus_ideal_fert_e

In [ ]:
extract_information(infant_care,'w_hus_ideal_fert_e')

w_hus_ideal_fert_e    float64
dtype: object
0      0.0
1      2.0
2      9.0
3      6.0
4      4.0
5      8.0
6      3.0
7      NaN
8      5.0
9      7.0
10     1.0
11    10.0
12    11.0
13    15.0
14    12.0
15    13.0
dtype: float64
279
5.061683599419449
w_hus_ideal_fert_e
0.0     4388
4.0      265
6.0      143
3.0      127
2.0       89
5.0       80
1.0       52
7.0       31
8.0       29
15.0      12
9.0        9
12.0       3
11.0       3
10.0       1
13.0       1
Name: count, dtype: int64
       w_hus_ideal_fert_e
count         5233.000000
mean             0.712975
std              1.862375
min              0.000000
25%              0.000000
50%              0.000000
75%              0.000000
max             15.000000


#### 7.49- w_smallkids

In [ ]:
extract_information(infant_care,'w_smallkids')

w_smallkids    float64
dtype: object
0    1.0
1    4.0
2    3.0
3    2.0
4    NaN
dtype: float64
57
1.0341074020319303
w_smallkids
1.0    2565
2.0    1102
4.0     983
3.0     805
Name: count, dtype: int64
       w_smallkids
count  5455.000000
mean      2.037764
std       1.156463
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       4.000000


#### 7.50- w_earlykids

In [ ]:
extract_information(infant_care,'w_earlykids')

w_earlykids    float64
dtype: object
0    4.0
1    3.0
2    1.0
3    2.0
4    NaN
dtype: float64
102
1.8505079825834543
w_earlykids
1.0    1599
4.0    1486
3.0    1186
2.0    1139
Name: count, dtype: int64
       w_earlykids
count  5410.000000
mean      2.473013
std       1.178987
min       1.000000
25%       1.000000
50%       2.000000
75%       4.000000
max       4.000000


#### 7.51- w_pregcomp

In [ ]:
extract_information(infant_care,'w_pregcomp')

w_pregcomp    float64
dtype: object
0    1.0
1    4.0
2    2.0
3    3.0
4    NaN
dtype: float64
36
0.6531204644412192
w_pregcomp
1.0    3991
2.0     873
4.0     362
3.0     250
Name: count, dtype: int64
        w_pregcomp
count  5476.000000
mean      1.449050
std       0.857605
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       4.000000


### 8- SANITATION_PRACTICE Section

In [ ]:
sanitation_practice = parent_all_sections['SANITATION_PRACTICE']
sanitation_practice.columns

Index(['w_water_source', 'w_water_source_oth', 'w_treat_water',
       'w_treat_water_how', 'w_treat_water_oth', 'w_soap_day', 'w_wash_bath',
       'w_soap_toilet', 'w_soap_toilet_bi', 'w_wash_meal', 'w_soap_meal',
       'w_soap_meal_bi', 'w_refuse_dispose', 'w_refuse_dispose_bi',
       'w_toilet_type', 'w_toilet_oth', 'w_toilet_sweep', 'w_toilet_sweep_bi'],
      dtype='object')

#### 8.1- w_water_source

In [ ]:
extract_information(sanitation_practice,'w_water_source')

w_water_source    float64
dtype: object
0    4.0
1    5.0
2    3.0
3    2.0
4    6.0
5    8.0
6    7.0
7    NaN
8    1.0
dtype: float64
2
0.036284470246734396
w_water_source
2.0    2012
5.0    1537
4.0     607
3.0     557
6.0     484
8.0     243
1.0      48
7.0      22
Name: count, dtype: int64
       w_water_source
count     5510.000000
mean         3.785481
std          1.722353
min          1.000000
25%          2.000000
50%          4.000000
75%          5.000000
max          8.000000


#### 8.2- w_water_source_oth

In [ ]:
extract_information(sanitation_practice,'w_water_source_oth')

w_water_source_oth    object
dtype: object
0                 NaN
1                pond
2            borehole
3           deep hole
4              stream
5       mineral water
6               swamp
7              swamps
8     Water reservers
9             omukura
10     stagnent water
11          Reservers
12          tsp water
dtype: object
5489
99.58272859216255
w_water_source_oth
swamp              6
stream             5
swamps             3
pond               1
deep hole          1
borehole           1
mineral water      1
Water reservers    1
omukura            1
stagnent water     1
Reservers          1
tsp water          1
Name: count, dtype: int64
       w_water_source_oth
count                  23
unique                 12
top                 swamp
freq                    6


#### 8.3- w_treat_water

In [ ]:
extract_information(sanitation_practice,'w_treat_water')

w_treat_water    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_treat_water
1.0    4634
0.0     876
Name: count, dtype: int64
       w_treat_water
count    5510.000000
mean        0.841016
std         0.365694
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000


#### 8.4- w_treat_water_how

In [ ]:
extract_information(sanitation_practice,'w_treat_water_how')

w_treat_water_how    object
dtype: object
0                                                1 Boil
1                        1 Boil;2 Filter/strain w cloth
2                            1 Boil;4 Use water filter 
3                                                   NaN
4     1 Boil;4 Use water filter ;5 Let it stand and ...
5                                        1 Boil;6 Other
6                                               6 Other
7                      1 Boil;5 Let it stand and settle
8                               2 Filter/strain w cloth
9                              1 Boil;3 Bleach/cholrine
10                            5 Let it stand and settle
11    1 Boil;2 Filter/strain w cloth;5 Let it stand ...
12                                  4 Use water filter 
13                                    3 Bleach/cholrine
14                3 Bleach/cholrine;4 Use water filter 
dtype: object
862
15.638606676342526
w_treat_water_how
1 Boil                                                      431

#### 8.5- w_treat_water_oth

In [ ]:
extract_information(sanitation_practice,'w_treat_water_oth')

w_treat_water_oth    object
dtype: object
0                                              NaN
1                                                .
2                             put in clean jerycan
3                                       Waterguard
4                        put  it in clean jerrycan
5     sometimes they boil but most times she boils
6                         put it in clean jerrycan
7                        Put it in a clean jerican
8                       put it in a good container
9                            nothing ie just drink
10                                     water guard
11                                  use waterguard
12                                  just like that
13                                 use water guard
14                        kip it in clean jerrycan
15                        puy it in clean jerrycan
16                          Put in a clean jerican
17                                         nothing
18                         keep it in ck

#### 8.6- w_soap_day

In [ ]:
extract_information(sanitation_practice,'w_soap_day')

w_soap_day    float64
dtype: object
0      3.0
1      6.0
2      2.0
3      5.0
4      4.0
5     10.0
6      1.0
7      7.0
8      0.0
9      NaN
10     8.0
11    12.0
12     9.0
13    20.0
14    14.0
15    13.0
dtype: float64
4
0.07256894049346879
w_soap_day
3.0     1831
2.0     1344
4.0      910
1.0      455
5.0      431
6.0      236
0.0      137
10.0      87
8.0       38
7.0       30
12.0       3
9.0        3
20.0       1
14.0       1
13.0       1
Name: count, dtype: int64
        w_soap_day
count  5508.000000
mean      3.148330
std       1.682423
min       0.000000
25%       2.000000
50%       3.000000
75%       4.000000
max      20.000000


#### 8.7- w_wash_bath

In [ ]:
extract_information(sanitation_practice,'w_wash_bath')

w_wash_bath    float64
dtype: object
0    1.0
1    3.0
2    4.0
3    2.0
4    5.0
5    NaN
dtype: float64
2
0.036284470246734396
w_wash_bath
1.0    3879
2.0     709
3.0     492
4.0     270
5.0     160
Name: count, dtype: int64
       w_wash_bath
count  5510.000000
mean      1.570417
std       1.032612
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       5.000000


#### 8.8- w_soap_toilet

In [ ]:
extract_information(sanitation_practice,'w_soap_toilet')

w_soap_toilet    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64
158
2.8664731494920175
w_soap_toilet
1.0    3049
2.0    1379
3.0     926
Name: count, dtype: int64
       w_soap_toilet
count    5354.000000
mean        1.603474
std         0.765057
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         3.000000


#### 8.9- w_soap_toilet_bi

In [ ]:
extract_information(sanitation_practice,'w_soap_toilet_bi')

w_soap_toilet_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
158
2.8664731494920175
w_soap_toilet_bi
1.0    3049
0.0    2305
Name: count, dtype: int64
       w_soap_toilet_bi
count       5354.000000
mean           0.569481
std            0.495195
min            0.000000
25%            0.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 8.10- w_wash_meal

In [ ]:
extract_information(sanitation_practice,'w_wash_meal')

w_wash_meal    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    NaN
4    5.0
5    4.0
dtype: float64
2
0.036284470246734396
w_wash_meal
1.0    5141
2.0     259
3.0     101
5.0       6
4.0       3
Name: count, dtype: int64
       w_wash_meal
count  5510.000000
mean      1.089655
std       0.366928
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       5.000000


#### 8.11- w_soap_meal

In [ ]:
extract_information(sanitation_practice,'w_soap_meal')

w_soap_meal    float64
dtype: object
0    3.0
1    2.0
2    1.0
3    NaN
dtype: float64
9
0.1632801161103048
w_soap_meal
3.0    2393
1.0    1586
2.0    1524
Name: count, dtype: int64
       w_soap_meal
count  5503.000000
mean      2.146647
std       0.837665
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       3.000000


#### 8.12- w_soap_meal_bi

In [ ]:
extract_information(sanitation_practice,'w_soap_meal_bi')

w_soap_meal_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
9
0.1632801161103048
w_soap_meal_bi
1.0    3110
0.0    2393
Name: count, dtype: int64
       w_soap_meal_bi
count     5503.000000
mean         0.565146
std          0.495783
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 8.13- w_refuse_dispose

In [ ]:
extract_information(sanitation_practice,'w_refuse_dispose')

w_refuse_dispose    float64
dtype: object
0    1.0
1    2.0
2    4.0
3    5.0
4    3.0
5    6.0
6    7.0
7    NaN
dtype: float64
2
0.036284470246734396
w_refuse_dispose
1.0    3838
2.0    1208
7.0     219
5.0     139
6.0      71
4.0      20
3.0      15
Name: count, dtype: int64
       w_refuse_dispose
count       5510.000000
mean           1.639383
std            1.418088
min            1.000000
25%            1.000000
50%            1.000000
75%            2.000000
max            7.000000


#### 8.14- w_refuse_dispose_bi

In [ ]:
extract_information(sanitation_practice,'w_refuse_dispose_bi')

w_refuse_dispose_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
2
0.036284470246734396
w_refuse_dispose_bi
0.0    5419
1.0      91
Name: count, dtype: int64
       w_refuse_dispose_bi
count          5510.000000
mean              0.016515
std               0.127458
min               0.000000
25%               0.000000
50%               0.000000
75%               0.000000
max               1.000000


#### 8.15- w_toilet_type

In [ ]:
extract_information(sanitation_practice,'w_toilet_type')

w_toilet_type    float64
dtype: object
0    1.0
1    5.0
2    2.0
3    9.0
4    3.0
5    NaN
6    8.0
7    4.0
8    6.0
dtype: float64
2
0.036284470246734396
w_toilet_type
5.0    2568
1.0    1678
2.0    1144
3.0      63
9.0      26
4.0      21
8.0       8
6.0       2
Name: count, dtype: int64
       w_toilet_type
count    5510.000000
mean        3.155898
std         1.865367
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max         9.000000


#### 8.16- w_toilet_oth

In [ ]:
extract_information(sanitation_practice,'w_toilet_oth')

w_toilet_oth    object
dtype: object
0                                                   NaN
1                                        they dont have
2                                                  none
3                                               coveres
4                                             No toilet
5                                         does not have
6                                        Have no toilet
7                              they use the neighbours'
8                                                dnt ve
9                           UN coveredshared Pitlatrine
10                                            no toilet
11    had the toilet but fall down now building anot...
12                        UN covered Shared pit latrine
13                                        no own toilet
14                                use for mother in law
15                             None...use for neighbors
16                         uncovrted Shared pit latrine
17         

#### 8.17- w_toilet_sweep

In [ ]:
extract_information(sanitation_practice,'w_toilet_sweep')

w_toilet_sweep    float64
dtype: object
0      3.0
1      1.0
2      2.0
3      4.0
4      5.0
5      7.0
6      0.0
7     20.0
8      NaN
9      6.0
10    14.0
dtype: float64
112
2.0319303338171264
w_toilet_sweep
2.0     1516
3.0     1210
1.0     1107
7.0      799
0.0      359
4.0      266
5.0       95
6.0       46
20.0       1
14.0       1
Name: count, dtype: int64
       w_toilet_sweep
count     5400.000000
mean         2.816852
std          2.075080
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         20.000000


#### 8.18- w_toilet_sweep_bi

In [ ]:
extract_information(sanitation_practice,'w_toilet_sweep_bi')

w_toilet_sweep_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
112
2.0319303338171264
w_toilet_sweep_bi
0.0    2982
1.0    2418
Name: count, dtype: int64
       w_toilet_sweep_bi
count        5400.000000
mean            0.447778
std             0.497311
min             0.000000
25%             0.000000
50%             0.000000
75%             1.000000
max             1.000000


### 9- HEALTH_KNOWLEDGE Section

In [ ]:
health_knowledge = parent_all_sections['HEALTH_KNOWLEDGE']
health_knowledge.columns

Index(['w_birth', 'w_birth_bi', 'w_birth_oth', 'w_mal_cause', 'w_mal_cause_bi',
       'w_mal_cause_bi2', 'w_mal_cause_oth', 'w_diar_cause', 'w_diar_cause_bi',
       'w_diar_cause_bi2',
       ...
       'w_timedeworm_3', 'w_timedeworm_4', 'w_timedeworm_5', 'w_timedeworm_6',
       'w_timedeworm_7', 'w_timedeworm_oth', 'w_hygint', 'w_hygint_bi',
       'w_childtoilet', 'w_childtoilet_bi'],
      dtype='object', length=138)

#### 9.1- w_birth

In [ ]:
extract_information(health_knowledge,'w_birth')

w_birth    float64
dtype: object
0     6.0
1     5.0
2    99.0
3     NaN
4     3.0
5     7.0
dtype: float64
1
0.018142235123367198
w_birth
6.0     5190
5.0      318
99.0       1
3.0        1
7.0        1
Name: count, dtype: int64
           w_birth
count  5511.000000
mean      5.958810
std       1.275744
min       3.000000
25%       6.000000
50%       6.000000
75%       6.000000
max      99.000000


#### 9.2- w_birth_bi

In [ ]:
extract_information(health_knowledge,'w_birth_bi')

w_birth_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
1
0.018142235123367198
w_birth_bi
1.0    5508
0.0       3
Name: count, dtype: int64
        w_birth_bi
count  5511.000000
mean      0.999456
std       0.023327
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 9.3- w_birth_oth

In [ ]:
extract_information(health_knowledge,'w_birth_oth')

w_birth_oth    object
dtype: object
0               NaN
1                 .
2    private clinic
dtype: object
5306
96.26269956458636
w_birth_oth
.                 205
private clinic      1
Name: count, dtype: int64
       w_birth_oth
count          206
unique           2
top              .
freq           205


#### 9.4- w_mal_cause

In [ ]:
extract_information(health_knowledge,'w_mal_cause')

w_mal_cause    object
dtype: object
0                     5=Poor sanitation;7=Mosquito bite
1                                       7=Mosquito bite
2     3=Improper behavior/vice;5=Poor sanitation;6=P...
3            5=Poor sanitation;7=Mosquito bite;10=Other
4                            5=Poor sanitation;10=Other
5     5=Poor sanitation;6=Poor health care;7=Mosquit...
6                                              10=Other
7     5=Poor sanitation;6=Poor health care;7=Mosquit...
8                              7=Mosquito bite;10=Other
9                                    6=Poor health care
10          6=Poor health care;7=Mosquito bite;10=Other
11                                                98=DK
12                                    5=Poor sanitation
13                             8=Eating mangos;10=Other
14                   6=Poor health care;7=Mosquito bite
15             3=Improper behavior/vice;7=Mosquito bite
16                 5=Poor sanitation;6=Poor health care
17          

#### 9.5- w_mal_cause_bi

In [ ]:
extract_information(health_knowledge,'w_mal_cause_bi')

w_mal_cause_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_mal_cause_bi
1.0    4959
0.0     551
Name: count, dtype: int64
       w_mal_cause_bi
count     5510.000000
mean         0.900000
std          0.300027
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 9.6- w_mal_cause_bi2

In [ ]:
extract_information(health_knowledge,'w_mal_cause_bi2')

w_mal_cause_bi2    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
2
0.036284470246734396
w_mal_cause_bi2
0.0    3335
1.0    2175
Name: count, dtype: int64
       w_mal_cause_bi2
count      5510.000000
mean          0.394737
std           0.488838
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000


#### 9.7- w_mal_cause_oth

In [ ]:
extract_information(health_knowledge,'w_mal_cause_oth')

w_mal_cause_oth    object
dtype: object
0                                                NaN
1                                     stagnate water
2                  Not Sleeping Under A mosquito Net
3                    not sleeping under mosquito net
4      not sleeping under the net and stagnant water
                           ...                      
373               dirty water n NT slping under nets
374            sleeping without nets, unbilled water
375         Unboiled Water, Not Using A Mosquito Net
376                                    having no net
377                                 its part of life
Length: 378, dtype: object
4517
81.94847605224965
w_mal_cause_oth
unboiled water                        72
not sleeping under mosquito net       61
Stagnant Water                        55
dirty water                           54
drinking unboiled water               52
                                      ..
overworkong                            1
not using the net and po

#### 9.8- w_diar_cause

In [ ]:
extract_information(health_knowledge,'w_diar_cause')

w_diar_cause    object
dtype: object
0               03 Rubbish;05 Dirty Water;06 Dirty Food
1               01 Human Feces;03 Rubbish;06 Dirty Food
2                                         06 Dirty Food
3                 05 Dirty Water;06 Dirty Food;07 Other
4                                06 Dirty Food;07 Other
                            ...                        
64                                     03 Rubbish;98 DK
65                04 Mosquitoes;05 Dirty Water;07 Other
66    01 Human Feces;04 Mosquitoes;05 Dirty Water;06...
67           01 Human Feces;04 Mosquitoes;06 Dirty Food
68                       02 Animal Feces;05 Dirty Water
Length: 69, dtype: object
7
0.1269956458635704
w_diar_cause
05 Dirty Water;06 Dirty Food                                            1088
06 Dirty Food                                                            863
06 Dirty Food;07 Other                                                   558
07 Other                                                

#### 9.9- w_diar_cause_bi

In [ ]:
extract_information(health_knowledge,'w_diar_cause_bi')

w_diar_cause_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
7
0.1269956458635704
w_diar_cause_bi
1.0    4508
0.0     997
Name: count, dtype: int64
       w_diar_cause_bi
count      5505.000000
mean          0.818892
std           0.385143
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           1.000000


#### 9.10- w_diar_cause_bi2

In [ ]:
extract_information(health_knowledge,'w_diar_cause_bi2')

w_diar_cause_bi2    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_diar_cause_bi2
0.0    3403
1.0    2102
Name: count, dtype: int64
       w_diar_cause_bi2
count       5505.000000
mean           0.381835
std            0.485880
min            0.000000
25%            0.000000
50%            0.000000
75%            1.000000
max            1.000000


#### 9.11- w_cstrum_feed

In [ ]:
extract_information(health_knowledge,'w_cstrum_feed')

w_cstrum_feed    float64
dtype: object
0     1.0
1     2.0
2    99.0
3     NaN
dtype: float64
1
0.018142235123367198
w_cstrum_feed
1.0     4922
2.0      485
99.0     104
Name: count, dtype: int64
       w_cstrum_feed
count    5511.000000
mean        2.937398
std        13.326933
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        99.000000


#### 9.12- w_cstrum_feed_bi

In [ ]:
extract_information(health_knowledge,'w_cstrum_feed_bi')

w_cstrum_feed_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
1
0.018142235123367198
w_cstrum_feed_bi
1.0    4922
0.0     589
Name: count, dtype: int64
       w_cstrum_feed_bi
count       5511.000000
mean           0.893123
std            0.308985
min            0.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 9.13- w_cstrum_no_why

In [ ]:
extract_information(health_knowledge,'w_cstrum_no_why')

w_cstrum_no_why    float64
dtype: object
0     NaN
1     1.0
2     4.0
3     2.0
4     3.0
5    99.0
dtype: float64
5021
91.09216255442671
w_cstrum_no_why
1.0     342
4.0      84
2.0      38
99.0     22
3.0       5
Name: count, dtype: int64
       w_cstrum_no_why
count       491.000000
mean          6.002037
std          20.193806
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          99.000000


#### 9.14- w_grow_lackfood

In [ ]:
extract_information(health_knowledge,'w_grow_lackfood')

w_grow_lackfood    float64
dtype: object
0     1.0
1     3.0
2     2.0
3     4.0
4    99.0
5     NaN
dtype: float64
1
0.018142235123367198
w_grow_lackfood
1.0     5035
2.0      211
3.0      138
4.0      118
99.0       9
Name: count, dtype: int64
       w_grow_lackfood
count      5511.000000
mean          1.312647
std           3.990058
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          99.000000


#### 9.15- w_grow_lackfood_sw

In [ ]:
extract_information(health_knowledge,'w_grow_lackfood_sw')

w_grow_lackfood_sw    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
1
0.018142235123367198
w_grow_lackfood_sw
1.0    5246
0.0     265
Name: count, dtype: int64
       w_grow_lackfood_sw
count         5511.000000
mean             0.951914
std              0.213967
min              0.000000
25%              1.000000
50%              1.000000
75%              1.000000
max              1.000000


#### 9.16- w_grow_lackfood_vm

In [ ]:
extract_information(health_knowledge,'w_grow_lackfood_vm')

w_grow_lackfood_vm    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
1
0.018142235123367198
w_grow_lackfood_vm
1.0    5035
0.0     476
Name: count, dtype: int64
       w_grow_lackfood_vm
count         5511.000000
mean             0.913627
std              0.280939
min              0.000000
25%              1.000000
50%              1.000000
75%              1.000000
max              1.000000


#### 9.17- w_grow_lackdiet

In [ ]:
extract_information(health_knowledge,'w_grow_lackdiet')

w_grow_lackdiet    float64
dtype: object
0     3.0
1     2.0
2     1.0
3     4.0
4    99.0
5     NaN
dtype: float64
1
0.018142235123367198
w_grow_lackdiet
1.0     4494
2.0      522
3.0      308
4.0      173
99.0      14
Name: count, dtype: int64
       w_grow_lackdiet
count      5511.000000
mean          1.549628
std           4.970004
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          99.000000


#### 9.18- w_grow_lackdiet_sw

In [ ]:
extract_information(health_knowledge,'w_grow_lackdiet_sw')

w_grow_lackdiet_sw    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
1
0.018142235123367198
w_grow_lackdiet_sw
1.0    5016
0.0     495
Name: count, dtype: int64
       w_grow_lackdiet_sw
count          5511.00000
mean              0.91018
std               0.28595
min               0.00000
25%               1.00000
50%               1.00000
75%               1.00000
max               1.00000


#### 9.19- w_grow_lackdiet_vm

In [ ]:
extract_information(health_knowledge,'w_grow_lackdiet_vm')

w_grow_lackdiet_vm    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
1
0.018142235123367198
w_grow_lackdiet_vm
1.0    4494
0.0    1017
Name: count, dtype: int64
       w_grow_lackdiet_vm
count         5511.000000
mean             0.815460
std              0.387959
min              0.000000
25%              1.000000
50%              1.000000
75%              1.000000
max              1.000000


#### 9.20- w_grow_witch

In [ ]:
extract_information(health_knowledge,'w_grow_witch')

w_grow_witch    float64
dtype: object
0     1.0
1     2.0
2     4.0
3     3.0
4    99.0
5     NaN
dtype: float64
1
0.018142235123367198
w_grow_witch
1.0     4670
2.0      350
4.0      302
3.0      131
99.0      58
Name: count, dtype: int64
       w_grow_witch
count   5511.000000
mean       2.306841
std       10.001925
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       99.000000


#### 9.21- w_grow_witch_lt

In [ ]:
extract_information(health_knowledge,'w_grow_witch_lt')

w_grow_witch_lt    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
1
0.018142235123367198
w_grow_witch_lt
0.0    5078
1.0     433
Name: count, dtype: int64
       w_grow_witch_lt
count      5511.000000
mean          0.078570
std           0.269091
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000


#### 9.22- w_grow_witch_vl

In [ ]:
extract_information(health_knowledge,'w_grow_witch_vl')

w_grow_witch_vl    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
1
0.018142235123367198
w_grow_witch_vl
0.0    5209
1.0     302
Name: count, dtype: int64
       w_grow_witch_vl
count      5511.000000
mean          0.054799
std           0.227609
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000


#### 9.23- w_grow_sick

In [ ]:
extract_information(health_knowledge,'w_grow_sick')

w_grow_sick    float64
dtype: object
0     2.0
1     1.0
2     4.0
3     3.0
4     NaN
5    99.0
dtype: float64
2
0.036284470246734396
w_grow_sick
1.0     5073
2.0      316
3.0       87
4.0       33
99.0       1
Name: count, dtype: int64
       w_grow_sick
count  5510.000000
mean      1.124682
std       1.379211
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max      99.000000


#### 9.24- w_grow_sick_sw

In [ ]:
extract_information(health_knowledge,'w_grow_sick_sw')

w_grow_sick_sw    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_grow_sick_sw
1.0    5389
0.0     121
Name: count, dtype: int64
       w_grow_sick_sw
count     5510.000000
mean         0.978040
std          0.146566
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 9.25- w_grow_sick_vm

In [ ]:
extract_information(health_knowledge,'w_grow_sick_vm')

w_grow_sick_vm    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
2
0.036284470246734396
w_grow_sick_vm
1.0    5073
0.0     437
Name: count, dtype: int64
       w_grow_sick_vm
count     5510.000000
mean         0.920690
std          0.270247
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 9.26- w_kwash_cause

In [ ]:
extract_information(health_knowledge,'w_kwash_cause')

w_kwash_cause    object
dtype: object
0      05 Lack of balanced diet;06 Births too close t...
1      04 Worms;05 Lack of balanced diet;06 Births to...
2               01 Lack of food;05 Lack of balanced diet
3      03 Lack of breastfeeding;05 Lack of balanced diet
4      03 Lack of breastfeeding;04 Worms;06 Births to...
                             ...                        
164                01 Lack of food;02 Cold food;04 Worms
165    04 Worms;06 Births too close together;07 Poor ...
166               03 Lack of breastfeeding;08 Witchcraft
167             01 Lack of food;07 Poor hygiene;09 Other
168    01 Lack of food;06 Births too close together;0...
Length: 169, dtype: object
5
0.09071117561683599
w_kwash_cause
05 Lack of balanced diet                                 1524
05 Lack of balanced diet;06 Births too close together     765
06 Births too close together                              313
04 Worms;05 Lack of balanced diet                         255
05 Lack of balanced diet

#### 9.27- w_kwash_cause_bi

In [ ]:
extract_information(health_knowledge,'w_kwash_cause_bi')

w_kwash_cause_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
5
0.09071117561683599
w_kwash_cause_bi
1.0    4478
0.0    1029
Name: count, dtype: int64
       w_kwash_cause_bi
count       5507.000000
mean           0.813147
std            0.389829
min            0.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 9.28- w_kwash_cause_bi2

In [ ]:
extract_information(health_knowledge,'w_kwash_cause_bi2')

w_kwash_cause_bi2    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
5
0.09071117561683599
w_kwash_cause_bi2
0.0    3980
1.0    1527
Name: count, dtype: int64
       w_kwash_cause_bi2
count        5507.000000
mean            0.277283
std             0.447698
min             0.000000
25%             0.000000
50%             0.000000
75%             1.000000
max             1.000000


#### 9.29- w_kwash_treat

In [ ]:
extract_information(health_knowledge,'w_kwash_treat')

w_kwash_treat    object
dtype: object
0       01 Change of diet;02 Medicine (deworming/other)
1                                     01 Change of diet
2     02 Medicine (deworming/other);03 There is no t...
3                            01 Change of diet;05 Other
4                                                   NaN
5                         02 Medicine (deworming/other)
6     01 Change of diet;02 Medicine (deworming/other...
7                    01 Change of diet;04 Sleeping well
8                02 Medicine (deworming/other);05 Other
9                              03 There is no treatment
10    01 Change of diet;02 Medicine (deworming/other...
11                                             05 Other
12                                                98 DK
13                            04 Sleeping well;05 Other
14            03 There is no treatment;04 Sleeping well
15    01 Change of diet;02 Medicine (deworming/other...
16           01 Change of diet;03 There is no treatment
17       0

#### 9.30- w_kwash_treat_bi

In [ ]:
extract_information(health_knowledge,'w_kwash_treat_bi')

w_kwash_treat_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
124
2.2496371552975325
w_kwash_treat_bi
1.0    4835
0.0     553
Name: count, dtype: int64
       w_kwash_treat_bi
count       5388.000000
mean           0.897365
std            0.303510
min            0.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 9.31- w_kwash_treat_bi2

In [ ]:
extract_information(health_knowledge,'w_kwash_treat_bi2')

w_kwash_treat_bi2    int64
dtype: object
0    0
1    1
dtype: int64
0
0.0
w_kwash_treat_bi2
0    2787
1    2725
Name: count, dtype: int64
       w_kwash_treat_bi2
count        5512.000000
mean            0.494376
std             0.500014
min             0.000000
25%             0.000000
50%             0.000000
75%             1.000000
max             1.000000


#### 9.32- w_ugx_spend

In [ ]:
extract_information(health_knowledge,'w_ugx_spend')

w_ugx_spend    object
dtype: object
0       02 Maize, Posho;05 Fish;08 Fruits, Juice;14 Other
1                  05 Fish;07 Vegetables;08 Fruits, Juice
2             01 Matooke;04 Rice;08 Fruits, Juice;11 Eggs
3                     01 Matooke;08 Fruits, Juice;11 Eggs
4               08 Fruits, Juice;11 Eggs;12 Milk;14 Other
                              ...                        
1670    04 Rice;06 Ground nuts;07 Vegetables;10 Beans;...
1671    02 Maize, Posho;03 Irish Potato;07 Vegetables;...
1672    01 Matooke;03 Irish Potato;07 Vegetables;11 Eg...
1673           01 Matooke;03 Irish Potato;04 Rice;05 Fish
1674    03 Irish Potato;07 Vegetables;10 Beans;11 Eggs...
Length: 1675, dtype: object
6
0.10885341074020319
w_ugx_spend
12 Milk;14 Other                                                                     163
11 Eggs;12 Milk;14 Other                                                             115
05 Fish;12 Milk;14 Other                                                             1

#### 9.33- w_ugx_oth

In [ ]:
extract_information(health_knowledge,'w_ugx_oth')

w_ugx_oth    object
dtype: object
0                    soybeans
1                         NaN
2                        soya
3                   Soyabeans
4                     chapati
                ...          
690    sweet potatoes,cassava
691           ghee and casava
692             soya and Ghee
693                   ensonzi
694          soyabean,Mandazi
Length: 695, dtype: object
2547
46.20827285921625
w_ugx_oth
soya                      951
ghee                      230
millet flour              100
Soya                       89
millet                     66
                         ... 
soya flour , mukene         1
dodo,1000                   1
pumpkin                     1
butter and Silver fish      1
bread and soya              1
Name: count, Length: 694, dtype: int64
       w_ugx_oth
count       2965
unique       694
top         soya
freq         951


#### 9.34- w_ugx_matooke

In [ ]:
extract_information(health_knowledge,'w_ugx_matooke')

w_ugx_matooke    float64
dtype: object
0         0.0
1      3000.0
2      8000.0
3      4000.0
4      2000.0
5      1000.0
6      2500.0
7      5000.0
8      5500.0
9      6000.0
10     1500.0
11        NaN
12     2800.0
13     2300.0
14     2600.0
15      200.0
16      500.0
17     7000.0
18      700.0
19    10000.0
20      800.0
21     3500.0
22     3200.0
23      600.0
24      400.0
25     1400.0
26     2200.0
27     1200.0
28     4500.0
dtype: float64
3
0.054426705370101594
w_ugx_matooke
0.0        4179
2000.0      397
3000.0      335
5000.0      153
1000.0      132
4000.0      125
2500.0       46
1500.0       42
6000.0       33
500.0        16
7000.0       14
8000.0        9
3500.0        7
10000.0       4
5500.0        3
2300.0        2
2800.0        1
200.0         1
700.0         1
2600.0        1
800.0         1
3200.0        1
600.0         1
400.0         1
1400.0        1
2200.0        1
1200.0        1
4500.0        1
Name: count, dtype: int64
       w_ugx_matooke
count   

#### 9.35- w_ugx_matooke_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_matooke_bi')

w_ugx_matooke_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_matooke_bi
0.0    4179
1.0    1330
Name: count, dtype: int64
       w_ugx_matooke_bi
count       5509.000000
mean           0.241423
std            0.427985
min            0.000000
25%            0.000000
50%            0.000000
75%            0.000000
max            1.000000


#### 9.36- w_ugx_maize

In [ ]:
extract_information(health_knowledge,'w_ugx_maize')

w_ugx_maize    float64
dtype: object
0      2000.0
1         0.0
2      4000.0
3      3000.0
4      2500.0
5      1000.0
6      2800.0
7      1500.0
8      5000.0
9      1600.0
10     1400.0
11     1800.0
12     3600.0
13      750.0
14      800.0
15     4500.0
16     1300.0
17     3500.0
18        NaN
19     3200.0
20      500.0
21     8000.0
22     1100.0
23     2200.0
24     5400.0
25     2400.0
26     1200.0
27     6000.0
28     3100.0
29     2250.0
30     2300.0
31     3300.0
32     1700.0
33      700.0
34      600.0
35     1900.0
36    10000.0
37     2100.0
38     4600.0
39     3700.0
dtype: float64
3
0.054426705370101594
w_ugx_maize
0.0        4392
2000.0      451
3000.0      213
1000.0      102
4000.0       83
1500.0       78
2500.0       45
5000.0       43
3500.0       14
1600.0        9
6000.0        8
1200.0        7
500.0         7
1800.0        7
1400.0        6
4500.0        6
2800.0        4
3600.0        3
2200.0        3
700.0         3
8000.0        2
3200.0        2
1

#### 9.37- w_ugx_maize_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_maize_bi')

w_ugx_maize_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_maize_bi
0.0    4392
1.0    1117
Name: count, dtype: int64
       w_ugx_maize_bi
count     5509.000000
mean         0.202759
std          0.402091
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


#### 9.38- w_ugx_potato

In [ ]:
extract_information(health_knowledge,'w_ugx_potato')

w_ugx_potato    float64
dtype: object
0        0.0
1     3000.0
2     1500.0
3     2500.0
4     2000.0
5     1000.0
6      500.0
7     4000.0
8     3300.0
9      200.0
10     700.0
11    5000.0
12    3500.0
13     300.0
14       NaN
15    6000.0
16    2400.0
17     900.0
18    1200.0
19    1900.0
20     600.0
21     400.0
22    3600.0
23     800.0
24    7000.0
dtype: float64
3
0.054426705370101594
w_ugx_potato
0.0       4325
2000.0     441
3000.0     272
1000.0     238
4000.0      73
2500.0      38
1500.0      35
5000.0      32
500.0       28
3500.0       6
300.0        4
200.0        2
6000.0       2
700.0        2
1200.0       2
3300.0       1
2400.0       1
900.0        1
1900.0       1
600.0        1
400.0        1
3600.0       1
800.0        1
7000.0       1
Name: count, dtype: int64
       w_ugx_potato
count   5509.000000
mean     473.552369
std     1020.828527
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max     7000.000000


#### 9.39- w_ugx_potato_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_potato_bi')

w_ugx_potato_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_potato_bi
0.0    4325
1.0    1184
Name: count, dtype: int64
       w_ugx_potato_bi
count      5509.000000
mean          0.214921
std           0.410805
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000


#### 9.40- w_ugx_rice

In [ ]:
extract_information(health_knowledge,'w_ugx_rice')

w_ugx_rice    float64
dtype: object
0         0.0
1      2000.0
2      2500.0
3      3000.0
4      4000.0
5      3500.0
6      1500.0
7       500.0
8      1000.0
9      6000.0
10     1800.0
11     3400.0
12     5600.0
13     1300.0
14     1700.0
15     4500.0
16     2800.0
17     5000.0
18     3200.0
19     1250.0
20     1200.0
21     3700.0
22      800.0
23     2600.0
24     3800.0
25     2300.0
26     2400.0
27        NaN
28     3600.0
29     6200.0
30     1600.0
31     4200.0
32      600.0
33     1400.0
34      750.0
35    10000.0
36     9000.0
37     8000.0
38     1100.0
39      700.0
40     7000.0
41     6400.0
42      900.0
43     4800.0
44     1750.0
dtype: float64
3
0.054426705370101594
w_ugx_rice
0.0        3850
3000.0      656
2000.0      353
1500.0      151
3500.0      103
2500.0       79
4000.0       78
1000.0       68
5000.0       31
6000.0       27
1800.0       14
3200.0       12
1200.0       11
1600.0        8
3400.0        7
3600.0        6
2800.0        6
7000.0       

#### 9.41- w_ugx_rice_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_rice_bi')

w_ugx_rice_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_rice_bi
0.0    3850
1.0    1659
Name: count, dtype: int64
       w_ugx_rice_bi
count    5509.000000
mean        0.301144
std         0.458797
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000


#### 9.42- w_ugx_fish

In [ ]:
extract_information(health_knowledge,'w_ugx_fish')

w_ugx_fish    float64
dtype: object
0      5000.0
1         0.0
2      2000.0
3      6000.0
4      1500.0
5      1000.0
6      3000.0
7      3500.0
8      4000.0
9       500.0
10     1200.0
11     2500.0
12      800.0
13     5500.0
14     8000.0
15    10000.0
16        NaN
17      300.0
18     1700.0
19     7500.0
20     1800.0
21     2700.0
22      700.0
23     4500.0
24      600.0
25     1900.0
26     7000.0
27     3400.0
28     2300.0
29     2200.0
30     1600.0
31     1400.0
32     2800.0
33     3300.0
34     3800.0
35     7200.0
dtype: float64
3
0.054426705370101594
w_ugx_fish
0.0        3621
2000.0      604
3000.0      562
1000.0      185
5000.0      141
4000.0      135
2500.0       82
1500.0       75
6000.0       25
3500.0       20
500.0        16
1200.0        4
4500.0        4
7000.0        4
8000.0        3
700.0         3
1800.0        2
5500.0        2
10000.0       2
600.0         2
2200.0        2
1600.0        2
800.0         1
7500.0        1
1700.0        1
300.0      

#### 9.43- w_ugx_fish_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_fish_bi')

w_ugx_fish_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_fish_bi
0.0    3621
1.0    1888
Name: count, dtype: int64
       w_ugx_fish_bi
count    5509.000000
mean        0.342712
std         0.474659
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000


#### 9.44- w_ugx_gnuts

In [ ]:
extract_information(health_knowledge,'w_ugx_gnuts')

w_ugx_gnuts    float64
dtype: object
0        0.0
1     1000.0
2     3000.0
3     2000.0
4     2500.0
5     5000.0
6     1500.0
7     3500.0
8     3400.0
9     4000.0
10     500.0
11    2400.0
12     600.0
13       NaN
14    5200.0
15    1300.0
16     200.0
17     900.0
18    1200.0
19     700.0
20    1400.0
21    1600.0
22    3200.0
23     300.0
24    4500.0
25    4800.0
dtype: float64
3
0.054426705370101594
w_ugx_gnuts
0.0       5095
2000.0     136
1000.0     117
3000.0      44
1500.0      38
4000.0      15
500.0       14
2500.0      12
600.0        5
700.0        4
1200.0       4
3500.0       4
5000.0       3
2400.0       3
1400.0       2
1300.0       2
900.0        2
1600.0       2
3400.0       1
200.0        1
5200.0       1
3200.0       1
300.0        1
4500.0       1
4800.0       1
Name: count, dtype: int64
       w_ugx_gnuts
count  5509.000000
mean    136.830641
std     541.481377
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max    5200.000000


#### 9.45- w_ugx_gnuts_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_gnuts_bi')

w_ugx_gnuts_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_gnuts_bi
0.0    5095
1.0     414
Name: count, dtype: int64
       w_ugx_gnuts_bi
count     5509.000000
mean         0.075150
std          0.263657
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


#### 9.46- w_ugx_veg

In [ ]:
extract_information(health_knowledge,'w_ugx_veg')

w_ugx_veg    float64
dtype: object
0        0.0
1     3000.0
2     1000.0
3     2000.0
4     2500.0
5     1500.0
6     1200.0
7      500.0
8      200.0
9     4000.0
10    6000.0
11     300.0
12     600.0
13     400.0
14    1700.0
15    1400.0
16    5000.0
17       NaN
18    2600.0
19     900.0
20     700.0
21     800.0
22    4500.0
23    2400.0
24    3500.0
25    1100.0
26    1300.0
27     100.0
28    2800.0
dtype: float64
3
0.054426705370101594
w_ugx_veg
0.0       4411
1000.0     440
2000.0     299
500.0      130
1500.0      64
3000.0      54
2500.0      28
5000.0      14
300.0       10
4000.0      10
600.0        8
200.0        7
800.0        6
1200.0       6
3500.0       4
400.0        4
700.0        2
1300.0       2
6000.0       1
1400.0       1
900.0        1
2600.0       1
1700.0       1
4500.0       1
2400.0       1
1100.0       1
100.0        1
2800.0       1
Name: count, dtype: int64
         w_ugx_veg
count  5509.000000
mean    291.395898
std     696.207196
min       0.000000

#### 9.47- w_ugx_veg_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_veg_bi')

w_ugx_veg_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_veg_bi
0.0    4411
1.0    1098
Name: count, dtype: int64
       w_ugx_veg_bi
count   5509.000000
mean       0.199310
std        0.399518
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


#### 9.48- w_ugx_fruit

In [ ]:
extract_information(health_knowledge,'w_ugx_fruit')

w_ugx_fruit    float64
dtype: object
0      1000.0
1      2000.0
2      1700.0
3         0.0
4       500.0
5      6000.0
6      2500.0
7      4000.0
8      3000.0
9      4200.0
10      400.0
11     1500.0
12      700.0
13     5000.0
14     3500.0
15     2200.0
16      300.0
17     1800.0
18     7000.0
19     2800.0
20     1300.0
21     1200.0
22     8000.0
23        NaN
24      600.0
25      900.0
26     2900.0
27      800.0
28     1600.0
29      100.0
30      200.0
31     7200.0
32     1400.0
33     2400.0
34     1550.0
35     6500.0
36     2100.0
37     4100.0
38    10000.0
39     5500.0
40     4500.0
41     5200.0
42     4900.0
43     5800.0
44     8700.0
45     3800.0
46     2600.0
47     4300.0
48     1100.0
49     4400.0
50     1900.0
dtype: float64
3
0.054426705370101594
w_ugx_fruit
0.0        3527
2000.0      640
1000.0      561
3000.0      212
500.0       130
1500.0       97
4000.0       79
2500.0       62
5000.0       47
6000.0       19
3500.0       15
400.0        13
300.0  

#### 9.49- w_ugx_fruit_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_fruit_bi')

w_ugx_fruit_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_fruit_bi
0.0    3527
1.0    1982
Name: count, dtype: int64
       w_ugx_fruit_bi
count     5509.000000
mean         0.359775
std          0.479978
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


#### 9.50- w_ugx_meat

In [ ]:
extract_information(health_knowledge,'w_ugx_meat')

w_ugx_meat    float64
dtype: object
0        0.0
1     3000.0
2     2000.0
3     5000.0
4     3500.0
5     7000.0
6     2500.0
7     3100.0
8     4000.0
9     1500.0
10    4500.0
11    6000.0
12    1000.0
13    4200.0
14       NaN
15    8000.0
16     500.0
17    1700.0
18    1200.0
19    2200.0
20    6200.0
21    3200.0
22    1400.0
23     600.0
dtype: float64
3
0.054426705370101594
w_ugx_meat
0.0       5001
3000.0     159
2000.0      85
4000.0      45
3500.0      41
2500.0      40
5000.0      39
6000.0      36
1500.0      26
7000.0      10
1000.0       8
4500.0       5
8000.0       2
2200.0       2
1700.0       2
4200.0       1
3100.0       1
500.0        1
1200.0       1
6200.0       1
3200.0       1
1400.0       1
600.0        1
Name: count, dtype: int64
        w_ugx_meat
count  5509.000000
mean    302.269014
std    1034.276542
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max    8000.000000


#### 9.51- w_ugx_meat_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_meat_bi')

w_ugx_meat_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_meat_bi
0.0    5001
1.0     508
Name: count, dtype: int64
       w_ugx_meat_bi
count    5509.000000
mean        0.092213
std         0.289352
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000


#### 9.52- w_ugx_bean

In [ ]:
extract_information(health_knowledge,'w_ugx_bean')

w_ugx_bean    float64
dtype: object
0        0.0
1     2000.0
2     3000.0
3     1000.0
4     5000.0
5      500.0
6     2100.0
7     1500.0
8     4000.0
9     1400.0
10     700.0
11    2500.0
12    1200.0
13    6000.0
14    4500.0
15    1600.0
16     800.0
17       NaN
18    4400.0
19    1800.0
20     900.0
21    1700.0
22    3600.0
23    4800.0
24     200.0
25    1100.0
26    3500.0
27    3700.0
28    6500.0
29    7000.0
30    3400.0
31    2800.0
32    2300.0
33    1300.0
34    1900.0
35    4600.0
36    2600.0
37    2400.0
38    2200.0
39    3200.0
dtype: float64
3
0.054426705370101594
w_ugx_bean
0.0       4213
2000.0     532
1000.0     255
3000.0     189
1500.0     115
2500.0      41
5000.0      38
4000.0      34
500.0       16
1200.0       8
6000.0       7
4500.0       6
1400.0       6
1600.0       6
700.0        5
1700.0       5
2400.0       3
1800.0       3
800.0        3
2600.0       2
4400.0       2
3400.0       2
1900.0       2
2100.0       1
4800.0       1
900.0        1
3600.

#### 9.53- w_ugx_bean_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_bean_bi')

w_ugx_bean_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_bean_bi
0.0    4213
1.0    1296
Name: count, dtype: int64
       w_ugx_bean_bi
count    5509.000000
mean        0.235251
std         0.424194
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000


#### 9.54- w_ugx_egg

In [ ]:
extract_information(health_knowledge,'w_ugx_egg')

w_ugx_egg    float64
dtype: object
0        0.0
1     3000.0
2      300.0
3     5000.0
4     1000.0
5     2000.0
6     4000.0
7     3500.0
8     1500.0
9     1200.0
10    1600.0
11    2500.0
12    2100.0
13    7000.0
14    1400.0
15     400.0
16    2300.0
17    1900.0
18     500.0
19    1250.0
20     800.0
21    1800.0
22    4500.0
23     600.0
24    6000.0
25    2400.0
26       NaN
27    6300.0
28     200.0
29     900.0
30    2600.0
31    4800.0
32     700.0
33     750.0
34    8500.0
35    3200.0
36    4300.0
37    6500.0
38    9000.0
39    5500.0
40    1300.0
41    3400.0
42    2800.0
43    7500.0
44    8000.0
45    2200.0
46    3600.0
47    3700.0
48    2700.0
dtype: float64
3
0.054426705370101594
w_ugx_egg
0.0       2959
2000.0     917
1000.0     549
3000.0     447
5000.0     127
4000.0     124
1500.0      85
2500.0      82
500.0       46
3500.0      19
6000.0      18
600.0       15
1200.0      13
1600.0      12
900.0       10
400.0        9
800.0        9
1800.0       7
7000.0    

#### 9.55- w_ugx_egg_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_egg_bi')

w_ugx_egg_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_egg_bi
0.0    2959
1.0    2550
Name: count, dtype: int64
       w_ugx_egg_bi
count   5509.000000
mean       0.462879
std        0.498665
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000


#### 9.56- w_ugx_milk

In [ ]:
extract_information(health_knowledge,'w_ugx_milk')

w_ugx_milk    float64
dtype: object
0         0.0
1      1000.0
2      2000.0
3      3000.0
4      3500.0
5      6000.0
6      5000.0
7      2500.0
8     10000.0
9       500.0
10     1500.0
11     4000.0
12     4500.0
13      600.0
14     7000.0
15     1200.0
16      300.0
17     8000.0
18      700.0
19      800.0
20     2400.0
21      400.0
22     9000.0
23      350.0
24     1600.0
25     1400.0
26      900.0
27     2700.0
28        NaN
29     5700.0
30     2100.0
31     2600.0
32     7500.0
33     1800.0
34     4300.0
35     4200.0
36     5500.0
37     1300.0
38     6100.0
39     2800.0
40      200.0
41     3600.0
42     4800.0
43     2300.0
44     4600.0
45     1250.0
46     6200.0
47     6800.0
48     3400.0
49     6500.0
50      250.0
51     3800.0
52     2200.0
53     3900.0
54     4700.0
55     2250.0
dtype: float64
3
0.054426705370101594
w_ugx_milk
0.0        1950
2000.0      907
1000.0      759
3000.0      473
5000.0      303
4000.0      216
500.0       135
1500.0      133
250

#### 9.57- w_ugx_milk_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_milk_bi')

w_ugx_milk_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_milk_bi
1.0    3559
0.0    1950
Name: count, dtype: int64
       w_ugx_milk_bi
count    5509.000000
mean        0.646034
std         0.478242
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000


#### 9.58- w_ugx_sweet

In [ ]:
extract_information(health_knowledge,'w_ugx_sweet')

w_ugx_sweet    float64
dtype: object
0        0.0
1     1000.0
2     2000.0
3     6000.0
4      400.0
5     1800.0
6      500.0
7     3000.0
8        NaN
9      300.0
10    1500.0
11    2500.0
12    3500.0
13     800.0
14      50.0
15    1200.0
16     600.0
17     200.0
18    4000.0
19    5000.0
20    1600.0
21     700.0
dtype: float64
3
0.054426705370101594
w_ugx_sweet
0.0       5373
1000.0      46
2000.0      27
500.0       25
3000.0       8
1500.0       5
400.0        3
200.0        3
300.0        3
5000.0       2
1800.0       2
4000.0       2
2500.0       2
6000.0       1
3500.0       1
800.0        1
50.0         1
600.0        1
1200.0       1
1600.0       1
700.0        1
Name: count, dtype: int64
       w_ugx_sweet
count  5509.000000
mean     34.080595
std     267.011578
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max    6000.000000


#### 9.59- w_ugx_sweet_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_sweet_bi')

w_ugx_sweet_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_sweet_bi
0.0    5373
1.0     136
Name: count, dtype: int64
       w_ugx_sweet_bi
count     5509.000000
mean         0.024687
std          0.155183
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


#### 9.60- w_ugx_other

In [ ]:
extract_information(health_knowledge,'w_ugx_other')

w_ugx_other    float64
dtype: object
0     2000.0
1        0.0
2     3000.0
3     5000.0
4      500.0
       ...  
76    8200.0
77    5900.0
78    6400.0
79    4700.0
80    4600.0
Length: 81, dtype: float64
3
0.054426705370101594
w_ugx_other
0.0       2342
3000.0     654
2000.0     559
5000.0     398
4000.0     388
          ... 
8200.0       1
5900.0       1
6400.0       1
4700.0       1
4600.0       1
Name: count, Length: 80, dtype: int64
        w_ugx_other
count   5509.000000
mean    1993.619532
std     2206.231817
min        0.000000
25%        0.000000
50%     1600.000000
75%     3000.000000
max    10000.000000


#### 9.61- w_ugx_other_bi

In [ ]:
extract_information(health_knowledge,'w_ugx_other_bi')

w_ugx_other_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_ugx_other_bi
1.0    3167
0.0    2342
Name: count, dtype: int64
       w_ugx_other_bi
count     5509.000000
mean         0.574877
std          0.494406
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 9.62- w_ageliquid

In [ ]:
extract_information(health_knowledge,'w_ageliquid')

w_ageliquid    float64
dtype: object
0      6.0
1      5.0
2      3.0
3      4.0
4      7.0
5      2.0
6      1.0
7      0.0
8      9.0
9      NaN
10    12.0
11     8.0
12    18.0
13    36.0
14    14.0
15    24.0
16    48.0
17    10.0
dtype: float64
10
0.18142235123367198
w_ageliquid
6.0     3225
4.0      894
3.0      530
5.0      475
2.0      148
1.0       93
7.0       64
8.0       30
9.0       13
0.0       10
12.0       9
24.0       4
18.0       2
48.0       2
36.0       1
14.0       1
10.0       1
Name: count, dtype: int64
       w_ageliquid
count  5502.000000
mean      5.176481
std       1.750666
min       0.000000
25%       4.000000
50%       6.000000
75%       6.000000
max      48.000000


#### 9.63- w_ageliquid_6

In [ ]:
extract_information(health_knowledge,'w_ageliquid_6')

w_ageliquid_6    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
10
0.18142235123367198
w_ageliquid_6
1.0    3225
0.0    2277
Name: count, dtype: int64
       w_ageliquid_6
count    5502.000000
mean        0.586150
std         0.492567
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000


#### 9.64- w_agefood

In [ ]:
extract_information(health_knowledge,'w_agefood')

w_agefood    float64
dtype: object
0      6.0
1      5.0
2      8.0
3      7.0
4     12.0
5      NaN
6      4.0
7      9.0
8      3.0
9      0.0
10    18.0
11    24.0
12    10.0
13    16.0
14    14.0
15    28.0
16     2.0
17    20.0
dtype: float64
19
0.3447024673439768
w_agefood
6.0     4197
7.0      427
8.0      290
5.0      248
9.0      103
4.0      101
12.0      77
3.0       19
10.0      10
0.0        7
18.0       4
24.0       3
2.0        3
16.0       1
14.0       1
28.0       1
20.0       1
Name: count, dtype: int64
         w_agefood
count  5493.000000
mean      6.257236
std       1.251185
min       0.000000
25%       6.000000
50%       6.000000
75%       6.000000
max      28.000000


#### 9.65- w_agefood_6

In [ ]:
extract_information(health_knowledge,'w_agefood_6')

w_agefood_6    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
19
0.3447024673439768
w_agefood_6
1.0    4197
0.0    1296
Name: count, dtype: int64
       w_agefood_6
count  5493.000000
mean      0.764063
std       0.424621
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 9.66- w_bfeed

In [ ]:
extract_information(health_knowledge,'w_bfeed')

w_bfeed    float64
dtype: object
0     24.0
1     48.0
2      2.0
3      NaN
4     36.0
5     12.0
6     30.0
7     18.0
8     10.0
9     60.0
10    20.0
11    29.0
12     6.0
13    26.0
14    15.0
15    28.0
16    17.0
17    19.0
18     9.0
19    32.0
20    42.0
21     7.0
22    22.0
23    27.0
24    31.0
25    14.0
26     8.0
27    21.0
28    39.0
29    25.0
30     0.0
31    23.0
32     5.0
33    16.0
34     3.0
35    34.0
dtype: float64
145
2.630624092888244
w_bfeed
24.0    3901
36.0     454
30.0     290
18.0     282
12.0     197
48.0      38
20.0      25
29.0      16
6.0       15
2.0       14
60.0      14
28.0      13
14.0      11
17.0       9
19.0       8
25.0       8
15.0       8
32.0       7
8.0        7
27.0       7
42.0       6
26.0       5
21.0       5
0.0        5
9.0        4
10.0       3
22.0       3
16.0       3
23.0       2
5.0        2
7.0        1
31.0       1
39.0       1
3.0        1
34.0       1
Name: count, dtype: int64
           w_bfeed
count  5367.000000
mean   

#### 9.67- w_bfeed24

In [ ]:
extract_information(health_knowledge,'w_bfeed24')

w_bfeed24    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
145
2.630624092888244
w_bfeed24
1.0    3901
0.0    1466
Name: count, dtype: int64
         w_bfeed24
count  5367.000000
mean      0.726849
std       0.445619
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 9.68- w_starchmilk

In [ ]:
extract_information(health_knowledge,'w_starchmilk')

w_starchmilk    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64
2
0.036284470246734396
w_starchmilk
1.0    5223
2.0     256
3.0      31
Name: count, dtype: int64
       w_starchmilk
count   5510.000000
mean       1.057713
std        0.256216
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        3.000000


#### 9.69- w_starchmilk_bi

In [ ]:
extract_information(health_knowledge,'w_starchmilk_bi')

w_starchmilk_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
2
0.036284470246734396
w_starchmilk_bi
0.0    5254
1.0     256
Name: count, dtype: int64
       w_starchmilk_bi
count      5510.000000
mean          0.046461
std           0.210500
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000


#### 9.70- w_starchmillet

In [ ]:
extract_information(health_knowledge,'w_starchmillet')

w_starchmillet    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64
2
0.036284470246734396
w_starchmillet
1.0    5080
2.0     310
3.0     120
Name: count, dtype: int64
       w_starchmillet
count     5510.000000
mean         1.099819
std          0.365289
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000


#### 9.71- w_starchmillet_bi

In [ ]:
extract_information(health_knowledge,'w_starchmillet_bi')

w_starchmillet_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_starchmillet_bi
1.0    5080
0.0     430
Name: count, dtype: int64
       w_starchmillet_bi
count        5510.000000
mean            0.921960
std             0.268259
min             0.000000
25%             1.000000
50%             1.000000
75%             1.000000
max             1.000000


#### 9.72- w_starchoil

In [ ]:
extract_information(health_knowledge,'w_starchoil')

w_starchoil    float64
dtype: object
0    2.0
1    1.0
2    3.0
3    NaN
dtype: float64
2
0.036284470246734396
w_starchoil
2.0    3496
1.0    1524
3.0     490
Name: count, dtype: int64
       w_starchoil
count  5510.000000
mean      1.812341
std       0.574771
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       3.000000


#### 9.73- w_starchoil_bi

In [ ]:
extract_information(health_knowledge,'w_starchoil_bi')

w_starchoil_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_starchoil_bi
1.0    3496
0.0    2014
Name: count, dtype: int64
       w_starchoil_bi
count     5510.000000
mean         0.634483
std          0.481619
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 9.74- w_starchgnuts

In [ ]:
extract_information(health_knowledge,'w_starchgnuts')

w_starchgnuts    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64
2
0.036284470246734396
w_starchgnuts
1.0    4999
2.0     423
3.0      88
Name: count, dtype: int64
       w_starchgnuts
count    5510.000000
mean        1.108711
std         0.358969
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000


#### 9.75- w_starchgnuts_bi

In [ ]:
extract_information(health_knowledge,'w_starchgnuts_bi')

w_starchgnuts_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_starchgnuts_bi
1.0    4999
0.0     511
Name: count, dtype: int64
       w_starchgnuts_bi
count       5510.000000
mean           0.907260
std            0.290095
min            0.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 9.76- w_starchrice

In [ ]:
extract_information(health_knowledge,'w_starchrice')

w_starchrice    float64
dtype: object
0    2.0
1    1.0
2    3.0
3    NaN
dtype: float64
1
0.018142235123367198
w_starchrice
1.0    4703
2.0     560
3.0     248
Name: count, dtype: int64
       w_starchrice
count   5511.000000
mean       1.191617
std        0.494920
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        3.000000


#### 9.77- w_starchrice_bi

In [ ]:
extract_information(health_knowledge,'w_starchrice_bi')

w_starchrice_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
1
0.018142235123367198
w_starchrice_bi
1.0    4703
0.0     808
Name: count, dtype: int64
       w_starchrice_bi
count      5511.000000
mean          0.853384
std           0.353755
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           1.000000


#### 9.78- w_starchpineapple

In [ ]:
extract_information(health_knowledge,'w_starchpineapple')

w_starchpineapple    float64
dtype: object
0    1.0
1    3.0
2    2.0
3    NaN
dtype: float64
1
0.018142235123367198
w_starchpineapple
1.0    4836
2.0     448
3.0     227
Name: count, dtype: int64
       w_starchpineapple
count        5511.000000
mean            1.163673
std             0.468299
min             1.000000
25%             1.000000
50%             1.000000
75%             1.000000
max             3.000000


#### 9.79- w_starchpineapple_bi

In [ ]:
extract_information(health_knowledge,'w_starchpineapple_bi')

w_starchpineapple_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
1
0.018142235123367198
w_starchpineapple_bi
0.0    5063
1.0     448
Name: count, dtype: int64
       w_starchpineapple_bi
count           5511.000000
mean               0.081292
std                0.273308
min                0.000000
25%                0.000000
50%                0.000000
75%                0.000000
max                1.000000


#### 9.80- w_starchmatooke

In [ ]:
extract_information(health_knowledge,'w_starchmatooke')

w_starchmatooke    float64
dtype: object
0    2.0
1    1.0
2    3.0
3    NaN
dtype: float64
1
0.018142235123367198
w_starchmatooke
1.0    3630
2.0    1687
3.0     194
Name: count, dtype: int64
       w_starchmatooke
count      5511.000000
mean          1.376520
std           0.552461
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           3.000000


#### 9.81- w_starchmatooke_bi

In [ ]:
extract_information(health_knowledge,'w_starchmatooke_bi')

w_starchmatooke_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
1
0.018142235123367198
w_starchmatooke_bi
1.0    3630
0.0    1881
Name: count, dtype: int64
       w_starchmatooke_bi
count         5511.000000
mean             0.658683
std              0.474195
min              0.000000
25%              0.000000
50%              1.000000
75%              1.000000
max              1.000000


#### 9.82- w_proteinchicken

In [ ]:
extract_information(health_knowledge,'w_proteinchicken')

w_proteinchicken    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64
2
0.036284470246734396
w_proteinchicken
1.0    5289
2.0     181
3.0      40
Name: count, dtype: int64
       w_proteinchicken
count       5510.000000
mean           1.047368
std            0.244243
min            1.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            3.000000


#### 9.83- w_proteinchicken_bi

In [ ]:
extract_information(health_knowledge,'w_proteinchicken_bi')

w_proteinchicken_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_proteinchicken_bi
1.0    5289
0.0     221
Name: count, dtype: int64
       w_proteinchicken_bi
count          5510.000000
mean              0.959891
std               0.196232
min               0.000000
25%               1.000000
50%               1.000000
75%               1.000000
max               1.000000


#### 9.84- w_proteinmatooke

In [ ]:
extract_information(health_knowledge,'w_proteinmatooke')

w_proteinmatooke    float64
dtype: object
0    2.0
1    1.0
2    3.0
3    NaN
dtype: float64
2
0.036284470246734396
w_proteinmatooke
2.0    4124
1.0    1261
3.0     125
Name: count, dtype: int64
       w_proteinmatooke
count       5510.000000
mean           1.793829
std            0.457246
min            1.000000
25%            2.000000
50%            2.000000
75%            2.000000
max            3.000000


#### 9.85- w_proteinmatooke_bi

In [ ]:
extract_information(health_knowledge,'w_proteinmatooke_bi')

w_proteinmatooke_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_proteinmatooke_bi
1.0    4124
0.0    1386
Name: count, dtype: int64
       w_proteinmatooke_bi
count          5510.000000
mean              0.748457
std               0.433939
min               0.000000
25%               0.000000
50%               1.000000
75%               1.000000
max               1.000000


#### 9.86- w_proteinfruit

In [ ]:
extract_information(health_knowledge,'w_proteinfruit')

w_proteinfruit    float64
dtype: object
0    1.0
1    3.0
2    2.0
3    NaN
dtype: float64
2
0.036284470246734396
w_proteinfruit
1.0    5276
2.0     217
3.0      17
Name: count, dtype: int64
       w_proteinfruit
count     5510.000000
mean         1.045554
std          0.222841
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000


#### 9.87- w_proteinfruit_bi

In [ ]:
extract_information(health_knowledge,'w_proteinfruit_bi')

w_proteinfruit_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
2
0.036284470246734396
w_proteinfruit_bi
0.0    5293
1.0     217
Name: count, dtype: int64
       w_proteinfruit_bi
count        5510.000000
mean            0.039383
std             0.194522
min             0.000000
25%             0.000000
50%             0.000000
75%             0.000000
max             1.000000


#### 9.88- w_proteinbeans

In [ ]:
extract_information(health_knowledge,'w_proteinbeans')

w_proteinbeans    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64
2
0.036284470246734396
w_proteinbeans
1.0    5298
2.0     202
3.0      10
Name: count, dtype: int64
       w_proteinbeans
count     5510.000000
mean         1.040290
std          0.205681
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000


#### 9.89- w_proteinbeans_bi

In [ ]:
extract_information(health_knowledge,'w_proteinbeans_bi')

w_proteinbeans_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_proteinbeans_bi
1.0    5298
0.0     212
Name: count, dtype: int64
       w_proteinbeans_bi
count        5510.000000
mean            0.961525
std             0.192359
min             0.000000
25%             1.000000
50%             1.000000
75%             1.000000
max             1.000000


#### 9.90- w_proteinposho

In [ ]:
extract_information(health_knowledge,'w_proteinposho')

w_proteinposho    float64
dtype: object
0    2.0
1    1.0
2    3.0
3    NaN
dtype: float64
2
0.036284470246734396
w_proteinposho
1.0    4762
2.0     673
3.0      75
Name: count, dtype: int64
       w_proteinposho
count     5510.000000
mean         1.149365
std          0.392818
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000


#### 9.91- w_proteinposho_bi

In [ ]:
extract_information(health_knowledge,'w_proteinposho_bi')

w_proteinposho_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_proteinposho_bi
0.0    4837
1.0     673
Name: count, dtype: int64
       w_proteinposho_bi
count        5510.000000
mean            0.122142
std             0.327479
min             0.000000
25%             0.000000
50%             0.000000
75%             0.000000
max             1.000000


#### 9.92- w_proteinmeat

In [ ]:
extract_information(health_knowledge,'w_proteinmeat')

w_proteinmeat    float64
dtype: object
0    2.0
1    1.0
2    3.0
3    NaN
dtype: float64
2
0.036284470246734396
w_proteinmeat
1.0    3204
2.0    2218
3.0      88
Name: count, dtype: int64
       w_proteinmeat
count    5510.000000
mean        1.434483
std         0.526972
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         3.000000


#### 9.93- w_proteinmeat_bi

In [ ]:
extract_information(health_knowledge,'w_proteinmeat_bi')

w_proteinmeat_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
2
0.036284470246734396
w_proteinmeat_bi
1.0    3204
0.0    2306
Name: count, dtype: int64
       w_proteinmeat_bi
count       5510.000000
mean           0.581488
std            0.493360
min            0.000000
25%            0.000000
50%            1.000000
75%            1.000000
max            1.000000


#### 9.94- w_beanmeat

In [ ]:
extract_information(health_knowledge,'w_beanmeat')

w_beanmeat    float64
dtype: object
0     2.0
1     1.0
2     NaN
3    99.0
dtype: float64
2
0.036284470246734396
w_beanmeat
2.0     5131
1.0      375
99.0       4
Name: count, dtype: int64
        w_beanmeat
count  5510.000000
mean      2.002359
std       2.626746
min       1.000000
25%       2.000000
50%       2.000000
75%       2.000000
max      99.000000


#### 9.95- w_beanmeat_bi

In [ ]:
extract_information(health_knowledge,'w_beanmeat_bi')

w_beanmeat_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_beanmeat_bi
1.0    5131
0.0     379
Name: count, dtype: int64
       w_beanmeat_bi
count    5510.000000
mean        0.931216
std         0.253109
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000


#### 9.96- w_reheat_leftov

In [ ]:
extract_information(health_knowledge,'w_reheat_leftov')

w_reheat_leftov    float64
dtype: object
0     2.0
1     1.0
2    98.0
3     NaN
dtype: float64
2
0.036284470246734396
w_reheat_leftov
2.0     4435
1.0     1067
98.0       8
Name: count, dtype: int64
       w_reheat_leftov
count      5510.000000
mean          1.945735
std           3.684281
min           1.000000
25%           2.000000
50%           2.000000
75%           2.000000
max          98.000000


#### 9.97- w_reheat_leftov_bi

In [ ]:
extract_information(health_knowledge,'w_reheat_leftov_bi')

w_reheat_leftov_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_reheat_leftov_bi
1.0    4435
0.0    1075
Name: count, dtype: int64
       w_reheat_leftov_bi
count         5510.000000
mean             0.804900
std              0.396313
min              0.000000
25%              1.000000
50%              1.000000
75%              1.000000
max              1.000000


#### 9.98- w_wateropen

In [ ]:
extract_information(health_knowledge,'w_wateropen')

w_wateropen    float64
dtype: object
0    2.0
1    1.0
2    NaN
dtype: float64
2
0.036284470246734396
w_wateropen
2.0    5003
1.0     507
Name: count, dtype: int64
       w_wateropen
count  5510.000000
mean      1.907985
std       0.289073
min       1.000000
25%       2.000000
50%       2.000000
75%       2.000000
max       2.000000


#### 9.99- w_wateropen_bi

In [ ]:
extract_information(health_knowledge,'w_wateropen_bi')

w_wateropen_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_wateropen_bi
1.0    5003
0.0     507
Name: count, dtype: int64
       w_wateropen_bi
count     5510.000000
mean         0.907985
std          0.289073
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 9.100- w_boil_1min

In [ ]:
extract_information(health_knowledge,'w_boil_1min')

w_boil_1min    float64
dtype: object
0     1.0
1     2.0
2    98.0
3     NaN
dtype: float64
2
0.036284470246734396
w_boil_1min
2.0     4502
1.0     1003
98.0       5
Name: count, dtype: int64
       w_boil_1min
count  5510.000000
mean      1.905082
std       2.921908
min       1.000000
25%       2.000000
50%       2.000000
75%       2.000000
max      98.000000


#### 9.101- w_boil_1min_bi

In [ ]:
extract_information(health_knowledge,'w_boil_1min_bi')

w_boil_1min_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
2
0.036284470246734396
w_boil_1min_bi
1.0    4502
0.0    1008
Name: count, dtype: int64
       w_boil_1min_bi
count     5510.000000
mean         0.817060
std          0.386653
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 9.102- w_diahfood

In [ ]:
extract_information(health_knowledge,'w_diahfood')

w_diahfood    float64
dtype: object
0     1.0
1     2.0
2     NaN
3    98.0
dtype: float64
2
0.036284470246734396
w_diahfood
1.0     5277
2.0      231
98.0       2
Name: count, dtype: int64
        w_diahfood
count  5510.000000
mean      1.077132
std       1.857914
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max      98.000000


#### 9.103- w_diarrfood_bi

In [ ]:
extract_information(health_knowledge,'w_diarrfood_bi')

w_diarrfood_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2
0.036284470246734396
w_diarrfood_bi
1.0    5277
0.0     233
Name: count, dtype: int64
       w_diarrfood_bi
count     5510.000000
mean         0.957713
std          0.201261
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 9.104- w_oral_salts

In [ ]:
extract_information(health_knowledge,'w_oral_salts')

w_oral_salts    float64
dtype: object
0     1.0
1     2.0
2    99.0
3     NaN
dtype: float64
6
0.10885341074020319
w_oral_salts
1.0     3975
99.0    1161
2.0      370
Name: count, dtype: int64
       w_oral_salts
count   5506.000000
mean      21.731566
std       39.945802
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       99.000000


#### 9.105- w_oral_salts_bi

In [ ]:
extract_information(health_knowledge,'w_oral_salts_bi')

w_oral_salts_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
6
0.10885341074020319
w_oral_salts_bi
1.0    3975
0.0    1531
Name: count, dtype: int64
       w_oral_salts_bi
count      5506.000000
mean          0.721940
std           0.448084
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


#### 9.106- w_porr

In [ ]:
extract_information(health_knowledge,'w_porr')

w_porr    float64
dtype: object
0     1.0
1     2.0
2     3.0
3     5.0
4    99.0
5     4.0
6     NaN
dtype: float64
5
0.09071117561683599
w_porr
1.0     4074
3.0      484
99.0     425
5.0      349
2.0      149
4.0       26
Name: count, dtype: int64
            w_porr
count  5507.000000
mean      9.033594
std      26.042342
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max      99.000000


#### 9.107- w_porr_bi

In [ ]:
extract_information(health_knowledge,'w_porr_bi')

w_porr_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
5
0.09071117561683599
w_porr_bi
1.0    4074
0.0    1433
Name: count, dtype: int64
         w_porr_bi
count  5507.000000
mean      0.739786
std       0.438791
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 9.108- w_weanfat

In [ ]:
extract_information(health_knowledge,'w_weanfat')

w_weanfat    float64
dtype: object
0     1.0
1     2.0
2    98.0
3     NaN
dtype: float64
3
0.054426705370101594
w_weanfat
1.0     4494
2.0      985
98.0      30
Name: count, dtype: int64
         w_weanfat
count  5509.000000
mean      1.707025
std       7.136255
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max      98.000000


#### 9.109- w_weanfat_bi

In [ ]:
extract_information(health_knowledge,'w_weanfat_bi')

w_weanfat_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
3
0.054426705370101594
w_weanfat_bi
0.0    4524
1.0     985
Name: count, dtype: int64
       w_weanfat_bi
count   5509.000000
mean       0.178798
std        0.383218
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


#### 9.110- w_child_protein

In [ ]:
extract_information(health_knowledge,'w_child_protein')

w_child_protein    float64
dtype: object
0     2.0
1     1.0
2    98.0
3     NaN
dtype: float64
3
0.054426705370101594
w_child_protein
2.0     4653
1.0      833
98.0      23
Name: count, dtype: int64
       w_child_protein
count      5509.000000
mean          2.249592
std           6.210683
min           1.000000
25%           2.000000
50%           2.000000
75%           2.000000
max          98.000000


#### 9.111- w_child_protein_bi

In [ ]:
extract_information(health_knowledge,'w_child_protein_bi')

w_child_protein_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_child_protein_bi
1.0    4653
0.0     856
Name: count, dtype: int64
       w_child_protein_bi
count         5509.000000
mean             0.844618
std              0.362301
min              0.000000
25%              1.000000
50%              1.000000
75%              1.000000
max              1.000000


#### 9.112- w_play

In [ ]:
extract_information(health_knowledge,'w_play')

w_play    float64
dtype: object
0     1.0
1     2.0
2     NaN
3    98.0
dtype: float64
3
0.054426705370101594
w_play
1.0     5457
2.0       51
98.0       1
Name: count, dtype: int64
            w_play
count  5509.000000
mean      1.026865
std       1.310259
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max      98.000000


#### 9.113- w_play_bi

In [ ]:
extract_information(health_knowledge,'w_play_bi')

w_play_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_play_bi
1.0    5457
0.0      52
Name: count, dtype: int64
         w_play_bi
count  5509.000000
mean      0.990561
std       0.096704
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 9.114- w_porr_thin

In [ ]:
extract_information(health_knowledge,'w_porr_thin')

w_porr_thin    float64
dtype: object
0     2.0
1     1.0
2    98.0
3     NaN
dtype: float64
3
0.054426705370101594
w_porr_thin
2.0     3671
1.0     1810
98.0      28
Name: count, dtype: int64
       w_porr_thin
count  5509.000000
mean      2.159376
std       6.866796
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max      98.000000


#### 9.115- w_porr_thin_bi

In [ ]:
extract_information(health_knowledge,'w_porr_thin_bi')

w_porr_thin_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
3
0.054426705370101594
w_porr_thin_bi
1.0    3671
0.0    1838
Name: count, dtype: int64
       w_porr_thin_bi
count     5509.000000
mean         0.666364
std          0.471554
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 9.116- w_dietsick

In [ ]:
extract_information(health_knowledge,'w_dietsick')

w_dietsick    float64
dtype: object
0     1.0
1     2.0
2    98.0
3     NaN
dtype: float64
4
0.07256894049346879
w_dietsick
1.0     5126
2.0      380
98.0       2
Name: count, dtype: int64
        w_dietsick
count  5508.000000
mean      1.104212
std       1.864201
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max      98.000000


#### 9.117- w_dietsick_bi

In [ ]:
extract_information(health_knowledge,'w_dietsick_bi')

w_dietsick_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
4
0.07256894049346879
w_dietsick_bi
1.0    5126
0.0     382
Name: count, dtype: int64
       w_dietsick_bi
count    5508.000000
mean        0.930646
std         0.254078
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000


#### 9.118- w_cowmilk

In [ ]:
extract_information(health_knowledge,'w_cowmilk')

w_cowmilk    float64
dtype: object
0     2.0
1     1.0
2    98.0
3     NaN
dtype: float64
4
0.07256894049346879
w_cowmilk
2.0     4700
1.0      783
98.0      25
Name: count, dtype: int64
         w_cowmilk
count  5508.000000
mean      2.293573
std       6.472530
min       1.000000
25%       2.000000
50%       2.000000
75%       2.000000
max      98.000000


#### 9.119- w_cowmilk_bi

In [ ]:
extract_information(health_knowledge,'w_cowmilk_bi')

w_cowmilk_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
4
0.07256894049346879
w_cowmilk_bi
1.0    4700
0.0     808
Name: count, dtype: int64
       w_cowmilk_bi
count   5508.000000
mean       0.853304
std        0.353834
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


#### 9.120- w_menfruit

In [ ]:
extract_information(health_knowledge,'w_menfruit')

w_menfruit    float64
dtype: object
0     1.0
1     2.0
2    98.0
3     NaN
dtype: float64
4
0.07256894049346879
w_menfruit
1.0     3329
2.0     2166
98.0      13
Name: count, dtype: int64
        w_menfruit
count  5508.000000
mean      1.622186
std       4.713525
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max      98.000000


#### 9.121- w_menfruit_bi

In [ ]:
extract_information(health_knowledge,'w_menfruit_bi')

w_menfruit_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
4
0.07256894049346879
w_menfruit_bi
0.0    3342
1.0    2166
Name: count, dtype: int64
       w_menfruit_bi
count    5508.000000
mean        0.393246
std         0.488515
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000


#### 9.122- w_menprot

In [ ]:
extract_information(health_knowledge,'w_menprot')

w_menprot    float64
dtype: object
0     2.0
1     1.0
2    98.0
3     NaN
dtype: float64
4
0.07256894049346879
w_menprot
2.0     4837
1.0      620
98.0      51
Name: count, dtype: int64
         w_menprot
count  5508.000000
mean      2.776325
std       9.211874
min       1.000000
25%       2.000000
50%       2.000000
75%       2.000000
max      98.000000


#### 9.123- w_menprot_bi

In [ ]:
extract_information(health_knowledge,'w_menprot_bi')

w_menprot_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
4
0.07256894049346879
w_menprot_bi
1.0    4837
0.0     671
Name: count, dtype: int64
       w_menprot_bi
count   5508.000000
mean       0.878177
std        0.327111
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


#### 9.124- w_dispose

In [ ]:
extract_information(health_knowledge,'w_dispose')

w_dispose    object
dtype: object
0                                      01 In the garden
1                            05 Put it in a rubbish pit
2                           01 In the garden;04 Bury it
3     01 In the garden;03 Burn it;05 Put it in a rub...
4     01 In the garden;05 Put it in a rubbish pit;06...
                            ...                        
60    01 In the garden;04 Bury it;07 Discard it wher...
61    01 In the garden;02 Put it in a latrine;05 Put...
62     04 Bury it;07 Discard it wherever it is;08 Other
63    03 Burn it;04 Bury it;05 Put it in a rubbish p...
64    01 In the garden;02 Put it in a latrine;03 Bur...
Length: 65, dtype: object
5
0.09071117561683599
w_dispose
05 Put it in a rubbish pit                                                       2191
01 In the garden;05 Put it in a rubbish pit                                      1137
01 In the garden                                                                  822
01 In the garden;04 Bury it       

#### 9.125- w_dispose_bi

In [ ]:
extract_information(health_knowledge,'w_dispose_bi')

w_dispose_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
5
0.09071117561683599
w_dispose_bi
0.0    3316
1.0    2191
Name: count, dtype: int64
       w_dispose_bi
count   5507.000000
mean       0.397857
std        0.489500
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000


#### 9.126- w_timedeworm

In [ ]:
extract_information(health_knowledge,'w_timedeworm')

w_timedeworm    float64
dtype: object
0     4.0
1     5.0
2     1.0
3     3.0
4    99.0
5     7.0
6     2.0
7     6.0
8     NaN
dtype: float64
7
0.1269956458635704
w_timedeworm
4.0     2119
3.0     1935
5.0      736
99.0     283
6.0      195
1.0       93
2.0       87
7.0       57
Name: count, dtype: int64
       w_timedeworm
count   5505.000000
mean       8.685559
std       21.047965
min        1.000000
25%        3.000000
50%        4.000000
75%        4.000000
max       99.000000


#### 9.127- w_timedeworm_1

In [ ]:
extract_information(health_knowledge,'w_timedeworm_1')

w_timedeworm_1    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_timedeworm_1
0.0    5412
1.0      93
Name: count, dtype: int64
       w_timedeworm_1
count     5505.000000
mean         0.016894
std          0.128885
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


#### 9.128- w_timedeworm_2

In [ ]:
extract_information(health_knowledge,'w_timedeworm_2')

w_timedeworm_2    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_timedeworm_2
0.0    5418
1.0      87
Name: count, dtype: int64
       w_timedeworm_2
count     5505.000000
mean         0.015804
std          0.124727
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


#### 9.129- w_timedeworm_3

In [ ]:
extract_information(health_knowledge,'w_timedeworm_3')

w_timedeworm_3    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_timedeworm_3
0.0    3570
1.0    1935
Name: count, dtype: int64
       w_timedeworm_3
count     5505.000000
mean         0.351499
std          0.477482
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


#### 9.130- w_timedeworm_4

In [ ]:
extract_information(health_knowledge,'w_timedeworm_4')

w_timedeworm_4    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
7
0.1269956458635704
w_timedeworm_4
0.0    3386
1.0    2119
Name: count, dtype: int64
       w_timedeworm_4
count     5505.000000
mean         0.384923
std          0.486621
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


#### 9.131- w_timedeworm_5

In [ ]:
extract_information(health_knowledge,'w_timedeworm_5')

w_timedeworm_5    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_timedeworm_5
0.0    4769
1.0     736
Name: count, dtype: int64
       w_timedeworm_5
count     5505.000000
mean         0.133697
std          0.340357
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


#### 9.132- w_timedeworm_6

In [ ]:
extract_information(health_knowledge,'w_timedeworm_6')

w_timedeworm_6    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_timedeworm_6
0.0    5310
1.0     195
Name: count, dtype: int64
       w_timedeworm_6
count     5505.000000
mean         0.035422
std          0.184862
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


#### 9.133- w_timedeworm_7

In [ ]:
extract_information(health_knowledge,'w_timedeworm_7')

w_timedeworm_7    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
7
0.1269956458635704
w_timedeworm_7
0.0    5448
1.0      57
Name: count, dtype: int64
       w_timedeworm_7
count     5505.000000
mean         0.010354
std          0.101237
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


#### 9.134- w_timedeworm_oth

In [ ]:
extract_information(health_knowledge,'w_timedeworm_oth')

w_timedeworm_oth    object
dtype: object
0                       NaN
1                   2 years
2                 Every one
3                Two Months
4                     2year
              ...          
63                  9Months
64                  6 hours
65          every two years
66    when she gets diahrea
67                  4 hours
Length: 68, dtype: object
5427
98.45791001451379
w_timedeworm_oth
daily                     4
2years                    3
5Months                   3
when the child is sick    3
when sick                 3
                         ..
9Months                   1
6 hours                   1
every two years           1
when she gets diahrea     1
4 hours                   1
Name: count, Length: 67, dtype: int64
       w_timedeworm_oth
count                85
unique               67
top               daily
freq                  4


#### 9.135- w_hygint

In [ ]:
extract_information(health_knowledge,'w_hygint')

w_hygint    float64
dtype: object
0     2.0
1     1.0
2    98.0
3     NaN
4     5.0
dtype: float64
5
0.09071117561683599
w_hygint
2.0     4172
1.0     1323
98.0      11
5.0        1
Name: count, dtype: int64
          w_hygint
count  5507.000000
mean      1.952061
std       4.318742
min       1.000000
25%       2.000000
50%       2.000000
75%       2.000000
max      98.000000


#### 9.136- w_hygint_bi

In [ ]:
extract_information(health_knowledge,'w_hygint_bi')

w_hygint_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
5
0.09071117561683599
w_hygint_bi
1.0    4172
0.0    1335
Name: count, dtype: int64
       w_hygint_bi
count  5507.000000
mean      0.757581
std       0.428585
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 9.137- w_childtoilet

In [ ]:
extract_information(health_knowledge,'w_childtoilet')

w_childtoilet    float64
dtype: object
0     5.0
1     3.0
2     4.0
3     2.0
4    99.0
5     1.0
6     NaN
dtype: float64
6
0.10885341074020319
w_childtoilet
4.0     2155
3.0     2037
5.0      972
2.0      332
1.0        8
99.0       2
Name: count, dtype: int64
       w_childtoilet
count    5506.000000
mean        3.716128
std         1.999787
min         1.000000
25%         3.000000
50%         4.000000
75%         4.000000
max        99.000000


#### 9.138- w_childtoilet_bi

In [ ]:
extract_information(health_knowledge,'w_childtoilet_bi')

w_childtoilet_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
6
0.10885341074020319
w_childtoilet_bi
0.0    3137
1.0    2369
Name: count, dtype: int64
       w_childtoilet_bi
count       5506.000000
mean           0.430258
std            0.495157
min            0.000000
25%            0.000000
50%            0.000000
75%            1.000000
max            1.000000


### MENS_HOUSEHOLD Section done

In [ ]:
mens_household = parent_all_sections['MENS_HOUSEHOLD']
mens_household.columns

Index(['Em_numkids', 'Em_hhkids_num', 'Em_hours_kids'], dtype='object')

In [ ]:
extract_information(mens_household,'Em_numkids')

Em_numkids    float64
dtype: object
0      6.0
1      5.0
2      7.0
3      8.0
4      4.0
5      NaN
6      3.0
7      9.0
8      1.0
9      2.0
10    11.0
11    10.0
12     0.0
13    12.0
14    13.0
dtype: float64
337
6.113933236574746
Em_numkids
2.0     1132
3.0      920
4.0      744
1.0      704
5.0      564
6.0      457
7.0      297
8.0      188
9.0       83
10.0      38
0.0       29
11.0       9
12.0       7
13.0       3
Name: count, dtype: int64
        Em_numkids
count  5175.000000
mean      3.709758
std       2.174280
min       0.000000
25%       2.000000
50%       3.000000
75%       5.000000
max      13.000000


In [ ]:
extract_information(mens_household,'Em_hhkids_num')

Em_hhkids_num    float64
dtype: object
0      5.0
1      6.0
2      3.0
3      2.0
4      4.0
5      NaN
6      1.0
7      9.0
8      0.0
9      7.0
10    10.0
11     8.0
12    12.0
dtype: float64
367
6.6582002902757615
Em_hhkids_num
2.0     1162
5.0     1127
3.0      978
4.0      912
1.0      805
0.0       82
6.0       39
7.0       24
8.0       11
9.0        3
10.0       1
12.0       1
Name: count, dtype: int64
       Em_hhkids_num
count    5145.000000
mean        3.087464
std         1.511274
min         0.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        12.000000


In [ ]:
extract_information(mens_household,'Em_hours_kids')

Em_hours_kids    float64
dtype: object
0      5.0
1      NaN
2     12.0
3      1.0
4      2.0
5      4.0
6     11.0
7      3.0
8      6.0
9      0.0
10     8.0
11     9.0
12     7.0
13    20.0
14    10.0
15    24.0
dtype: float64
2859
51.86865021770682
Em_hours_kids
2.0     637
1.0     539
3.0     470
4.0     301
0.0     197
5.0     192
6.0     156
8.0      61
7.0      41
10.0     25
12.0     16
9.0      12
11.0      4
20.0      1
24.0      1
Name: count, dtype: int64
       Em_hours_kids
count    2653.000000
mean        2.915944
std         2.225949
min         0.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        24.000000



* Note: 24 is an outlayer because no one stays 24 hours without sleeping or eating

In [ ]:
mens_household[(mens_household['Em_numkids'] < mens_household['Em_hhkids_num'])].head()

,Em_numkids,Em_hhkids_num,Em_hours_kids


### MENS_EXPENDITURE_EL section done

In [ ]:
mens_expenditure_el = parent_all_sections['MENS_EXPENDITURE_EL']
mens_expenditure_el.columns

Index(['Em_last7d_tobacco', 'Em_last7d_alcohol', 'Em_last1m_snacks',
       'Em_last1m_elec', 'Em_last1m_cloth', 'Em_plan_hhexpend6'],
      dtype='object')

In [ ]:
mens_expenditure_el.describe()

,Em_last7d_tobacco,Em_last7d_alcohol,Em_last1m_snacks,Em_last1m_elec,Em_last1m_cloth,Em_plan_hhexpend6
count,5176.000000,5176.000000,5176.000000,5.176000e+03,5176.000000,5171.000000
mean,480.954598,3448.863794,5878.005216,1.350185e+04,10252.609737,0.751305
std,2195.721451,13851.098440,16023.389785,4.535761e+04,23487.282861,0.432298
min,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000
25%,0.000000,0.000000,0.000000,2.600000e+03,0.000000,1.000000
50%,0.000000,0.000000,0.000000,5.400000e+03,0.000000,1.000000
75%,0.000000,4000.000000,6000.000000,1.300000e+04,15000.000000,1.000000
max,100000.000000,800003.000000,730000.000000,1.900000e+06,700000.000000,1.000000


In [ ]:
extract_information(mens_expenditure_el,'Em_last7d_tobacco')

Em_last7d_tobacco    float64
dtype: object
0          0.0
1       2000.0
2        700.0
3          NaN
4      10000.0
5       5000.0
6        500.0
7       1000.0
8       3000.0
9       8000.0
10      6000.0
11      2500.0
12      3500.0
13       600.0
14      7000.0
15       200.0
16       300.0
17       400.0
18      2800.0
19      1400.0
20      6300.0
21     15000.0
22      1500.0
23       100.0
24      4000.0
25      9000.0
26      1600.0
27     20000.0
28      1300.0
29      1800.0
30      1200.0
31      3600.0
32      2400.0
33     14000.0
34     35000.0
35       800.0
36       900.0
37      5100.0
38     30000.0
39       250.0
40      2100.0
41       150.0
42      4200.0
43     21000.0
44      4800.0
45      8400.0
46      1700.0
47         8.0
48    100000.0
49        55.0
50     10500.0
51      2300.0
52      2600.0
53     12000.0
dtype: float64
336
6.095791001451379
Em_last7d_tobacco
0.0         4185
1000.0       193
500.0        148
2000.0       144
3000.0        81
5000.0 

In [ ]:
extract_information(mens_expenditure_el,'Em_last7d_alcohol')

Em_last7d_alcohol    float64
dtype: object
0          0.0
1      10000.0
2       1000.0
3       4000.0
4       3000.0
        ...   
62      2400.0
63    800003.0
64     14100.0
65      4500.0
66    150000.0
Length: 67, dtype: float64
336
6.095791001451379
Em_last7d_alcohol
0.0         2880
2000.0       387
5000.0       381
10000.0      295
3000.0       256
            ... 
7400.0         1
3800.0         1
800003.0       1
14100.0        1
150000.0       1
Name: count, Length: 66, dtype: int64
       Em_last7d_alcohol
count        5176.000000
mean         3448.863794
std         13851.098440
min             0.000000
25%             0.000000
50%             0.000000
75%          4000.000000
max        800003.000000


In [ ]:
extract_information(mens_expenditure_el,'Em_last1m_snacks')

Em_last1m_snacks    float64
dtype: object
0          0.0
1       2000.0
2       5000.0
3       2500.0
4       8000.0
        ...   
94      3600.0
95     11000.0
96      3700.0
97    420000.0
98      1600.0
Length: 99, dtype: float64
336
6.095791001451379
Em_last1m_snacks
0.0         2601
10000.0      320
5000.0       263
3000.0       256
2000.0       249
            ... 
5400.0         1
11000.0        1
3700.0         1
420000.0       1
1600.0         1
Name: count, Length: 98, dtype: int64
       Em_last1m_snacks
count       5176.000000
mean        5878.005216
std        16023.389785
min            0.000000
25%            0.000000
50%            0.000000
75%         6000.000000
max       730000.000000


In [ ]:
extract_information(mens_expenditure_el,'Em_last1m_elec')

Em_last1m_elec    float64
dtype: object
0       3200.0
1       4000.0
2       7000.0
3          0.0
4      12000.0
        ...   
342     3100.0
343    20800.0
344    43200.0
345    36200.0
346    30600.0
Length: 347, dtype: float64
336
6.095791001451379
Em_last1m_elec
5000.0      482
0.0         459
10000.0     415
3000.0      260
2000.0      211
           ... 
182600.0      1
66200.0       1
43200.0       1
36200.0       1
30600.0       1
Name: count, Length: 346, dtype: int64
       Em_last1m_elec
count    5.176000e+03
mean     1.350185e+04
std      4.535761e+04
min      0.000000e+00
25%      2.600000e+03
50%      5.400000e+03
75%      1.300000e+04
max      1.900000e+06


In [ ]:
extract_information(mens_expenditure_el,'Em_last1m_cloth')

Em_last1m_cloth    float64
dtype: object
0          0.0
1      40000.0
2      30000.0
3          NaN
4      35000.0
        ...   
78     95000.0
79       700.0
80    145000.0
81     74000.0
82     46000.0
Length: 83, dtype: float64
336
6.095791001451379
Em_last1m_cloth
0.0         3481
20000.0      299
30000.0      221
10000.0      149
15000.0      132
            ... 
95000.0        1
700.0          1
145000.0       1
74000.0        1
46000.0        1
Name: count, Length: 82, dtype: int64
       Em_last1m_cloth
count      5176.000000
mean      10252.609737
std       23487.282861
min           0.000000
25%           0.000000
50%           0.000000
75%       15000.000000
max      700000.000000


In [ ]:
extract_information(mens_expenditure_el,'Em_plan_hhexpend6')

Em_plan_hhexpend6    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
341
6.186502177068215
Em_plan_hhexpend6
1.0    3885
0.0    1286
Name: count, dtype: int64
       Em_plan_hhexpend6
count        5171.000000
mean            0.751305
std             0.432298
min             0.000000
25%             1.000000
50%             1.000000
75%             1.000000
max             1.000000


### Mens Sanit Pract section

In [ ]:
mens_sanit_pract = parent_all_sections['MENS_SANIT_PRACT']
mens_sanit_pract.columns

Index(['Em_soap_toilet', 'Em_soap_toilet_bi', 'Em_soap_meal',
       'Em_soap_meal_bi', 'Em_toilet_improv', 'Em_toilet_improv_what'],
      dtype='object')

In [ ]:
mens_sanit_pract.describe()

,Em_soap_toilet,Em_soap_toilet_bi,Em_soap_meal,Em_soap_meal_bi,Em_toilet_improv,Em_toilet_improv_what
count,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,2618.000000
mean,2.361862,0.357226,2.851043,0.605100,0.505796,3.498854
std,1.259340,0.479228,1.349335,0.488876,0.500015,1.187519
min,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000
25%,1.000000,0.000000,2.000000,0.000000,0.000000,3.000000
50%,2.000000,0.000000,3.000000,1.000000,1.000000,3.000000
75%,3.000000,1.000000,4.000000,1.000000,1.000000,5.000000
max,5.000000,1.000000,5.000000,1.000000,1.000000,5.000000


In [ ]:
extract_information(mens_sanit_pract,'Em_soap_toilet')

Em_soap_toilet    float64
dtype: object
0    4.0
1    3.0
2    5.0
3    1.0
4    2.0
5    NaN
dtype: float64
336
6.095791001451379
Em_soap_toilet
1.0    1849
4.0    1078
2.0    1069
3.0     987
5.0     193
Name: count, dtype: int64
       Em_soap_toilet
count     5176.000000
mean         2.361862
std          1.259340
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          5.000000


In [ ]:
grad_map = {
    1.0: 'Every time',
    2.0: 'Most of the time',
    3.0: 'Sometimes',
    4.0: 'Rarely',
    5.0: 'Never'}
parent_df['Em_soap_toilet'] = parent_df['Em_soap_toilet'].map(grad_map)
parent_df['Em_soap_toilet'] = parent_df['Em_soap_toilet'].astype('category')
mens_sanit_pract['Em_soap_toilet'] = mens_sanit_pract['Em_soap_toilet'].map(grad_map)
mens_sanit_pract['Em_soap_toilet'] = mens_sanit_pract['Em_soap_toilet'].astype('category')
extract_information(mens_sanit_pract,'Em_soap_toilet')

Em_soap_toilet    category
dtype: object
0              Rarely
1           Sometimes
2               Never
3          Every time
4    Most of the time
5                 NaN
dtype: category
Categories (5, object): ['Every time', 'Most of the time', 'Never', 'Rarely', 'Sometimes']
336
6.095791001451379
Em_soap_toilet
Every time          1849
Rarely              1078
Most of the time    1069
Sometimes            987
Never                193
Name: count, dtype: int64
       Em_soap_toilet
count            5176
unique              5
top        Every time
freq             1849


In [ ]:
extract_information(mens_sanit_pract,'Em_soap_toilet_bi')

Em_soap_toilet_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
336
6.095791001451379
Em_soap_toilet_bi
0.0    3327
1.0    1849
Name: count, dtype: int64
       Em_soap_toilet_bi
count        5176.000000
mean            0.357226
std             0.479228
min             0.000000
25%             0.000000
50%             0.000000
75%             1.000000
max             1.000000


In [ ]:
extract_information(mens_sanit_pract,'Em_soap_meal')

Em_soap_meal    float64
dtype: object
0    4.0
1    2.0
2    5.0
3    1.0
4    NaN
5    3.0
dtype: float64
336
6.095791001451379
Em_soap_meal
4.0    1510
1.0    1271
3.0    1054
2.0     807
5.0     534
Name: count, dtype: int64
       Em_soap_meal
count   5176.000000
mean       2.851043
std        1.349335
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        5.000000


In [ ]:
parent_df['Em_soap_meal'] = parent_df['Em_soap_meal'].map(grad_map)
parent_df['Em_soap_meal'] = parent_df['Em_soap_meal'].astype('category')

mens_sanit_pract['Em_soap_meal'] = mens_sanit_pract['Em_soap_meal'].map(grad_map)
mens_sanit_pract['Em_soap_meal'] = mens_sanit_pract['Em_soap_meal'].astype('category')
extract_information(mens_sanit_pract,'Em_soap_meal')

Em_soap_meal    category
dtype: object
0              Rarely
1    Most of the time
2               Never
3          Every time
4                 NaN
5           Sometimes
dtype: category
Categories (5, object): ['Every time', 'Most of the time', 'Never', 'Rarely', 'Sometimes']
336
6.095791001451379
Em_soap_meal
Rarely              1510
Every time          1271
Sometimes           1054
Most of the time     807
Never                534
Name: count, dtype: int64
       Em_soap_meal
count          5176
unique            5
top          Rarely
freq           1510


In [ ]:
extract_information(mens_sanit_pract,'Em_soap_meal_bi')

Em_soap_meal_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
336
6.095791001451379
Em_soap_meal_bi
1.0    3132
0.0    2044
Name: count, dtype: int64
       Em_soap_meal_bi
count      5176.000000
mean          0.605100
std           0.488876
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(mens_sanit_pract,'Em_toilet_improv')

Em_toilet_improv    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
336
6.095791001451379
Em_toilet_improv
1.0    2618
0.0    2558
Name: count, dtype: int64
       Em_toilet_improv
count       5176.000000
mean           0.505796
std            0.500015
min            0.000000
25%            0.000000
50%            1.000000
75%            1.000000
max            1.000000


In [ ]:
extract_information(mens_sanit_pract,'Em_toilet_improv_what')

Em_toilet_improv_what    float64
dtype: object
0    3.0
1    2.0
2    5.0
3    NaN
4    4.0
5    1.0
dtype: float64
2894
52.503628447024674
Em_toilet_improv_what
3.0    985
5.0    885
2.0    579
4.0    151
1.0     18
Name: count, dtype: int64
       Em_toilet_improv_what
count            2618.000000
mean                3.498854
std                 1.187519
min                 1.000000
25%                 3.000000
50%                 3.000000
75%                 5.000000
max                 5.000000


In [ ]:
improv_map = {
    1 :'Previously used shared latrine and built own latrine',
    2 :'Added handwashing facilities',
    3 :'Improved structural materials (better wood, etc.)',
    4 :'Added cover',
    5 :'Build new latrine',
}

mens_sanit_pract['Em_toilet_improv_what'] = mens_sanit_pract['Em_toilet_improv_what'].map(improv_map)
mens_sanit_pract['Em_toilet_improv_what'] = mens_sanit_pract['Em_toilet_improv_what'].astype('category')
parent_df['Em_toilet_improv_what'] = parent_df['Em_toilet_improv_what'].map(improv_map)
parent_df['Em_toilet_improv_what'] = parent_df['Em_toilet_improv_what'].astype('category')
extract_information(mens_sanit_pract,'Em_toilet_improv_what')

Em_toilet_improv_what    category
dtype: object
0    Improved structural materials (better wood, etc.)
1                         Added handwashing facilities
2                                    Build new latrine
3                                                  NaN
4                                          Added cover
5    Previously used shared latrine and built own l...
dtype: category
Categories (5, object): ['Added cover', 'Added handwashing facilities', 'Build new latrine',
                         'Improved structural materials (better wood, e..., 'Previously used shared latrine and built own ...]
2894
52.503628447024674
Em_toilet_improv_what
Improved structural materials (better wood, etc.)       985
Build new latrine                                       885
Added handwashing facilities                            579
Added cover                                             151
Previously used shared latrine and built own latrine     18
Name: count, dtype: int64
              

In [ ]:
check_null_ratio(mens_sanit_pract)

                       missing_val  missing_val_ratio
Em_soap_toilet                 336                  6
Em_soap_toilet_bi              336                  6
Em_soap_meal                   336                  6
Em_soap_meal_bi                336                  6
Em_toilet_improv               336                  6
Em_toilet_improv_what         2894                 53


In [ ]:
# columns_to_drop_parnet += ['Em_toilet_improv_what']

### MENS_SANITATION Section

In [ ]:
mens_sanitation = parent_all_sections['MENS_SANITATION']
mens_sanitation.columns

Index(['Em_timedeworm', 'Em_timedeworm_1', 'Em_timedeworm_3',
       'Em_timedeworm_4', 'Em_timedeworm_5', 'Em_timedeworm_6',
       'Em_timedeworm_7', 'Em_timedeworm_8', 'Em_timedeworm_oth', 'Em_hygint',
       'Em_hygint_bi'],
      dtype='object')

In [ ]:
mens_sanitation.describe()

,Em_timedeworm,Em_timedeworm_1,Em_timedeworm_3,Em_timedeworm_4,Em_timedeworm_5,Em_timedeworm_6,Em_timedeworm_7,Em_timedeworm_8,Em_hygint,Em_hygint_bi
count,5176.00000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5172.000000,5172.000000
mean,16.64915,0.002318,0.287674,0.427357,0.153400,0.050618,0.000773,0.059312,0.125483,0.874517
std,104.54454,0.048098,0.452722,0.494743,0.360408,0.219238,0.027791,0.236231,0.331298,0.331298
min,1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,2.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,3.00000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,777.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
extract_information(mens_sanitation,'Em_timedeworm')

Em_timedeworm    float64
dtype: object
0      2.0
1      7.0
2      3.0
3      NaN
4      4.0
5      1.0
6    777.0
7      6.0
8      5.0
dtype: float64
336
6.095791001451379
Em_timedeworm
2.0      2212
1.0      1489
3.0       794
7.0       307
4.0       262
777.0      96
6.0        12
5.0         4
Name: count, dtype: int64
       Em_timedeworm
count     5176.00000
mean        16.64915
std        104.54454
min          1.00000
25%          1.00000
50%          2.00000
75%          3.00000
max        777.00000


In [ ]:
no_training = parent_df[mens_sanitation['Em_timedeworm'].isnull()]
no_training.shape

(336, 1510)

In [ ]:
no_training = mens_sanitation[mens_sanitation['Em_timedeworm'].isnull() & (parent_df['mhn']==1)]
print(no_training.shape)
no_training.head()

(102, 11)


,Em_timedeworm,Em_timedeworm_1,Em_timedeworm_3,Em_timedeworm_4,Em_timedeworm_5,Em_timedeworm_6,Em_timedeworm_7,Em_timedeworm_8,Em_timedeworm_oth,Em_hygint,Em_hygint_bi
227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
406,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
no_training_for_men = mens_sanitation[mens_sanitation['Em_timedeworm'].isnull() & (parent_df['mhn']==0)]
print(no_training_for_men.shape)
no_training_for_men.head()

(234, 11)


,Em_timedeworm,Em_timedeworm_1,Em_timedeworm_3,Em_timedeworm_4,Em_timedeworm_5,Em_timedeworm_6,Em_timedeworm_7,Em_timedeworm_8,Em_timedeworm_oth,Em_hygint,Em_hygint_bi
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
131,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
target_cols = [
    'Em_timedeworm', 'Em_timedeworm_1', 'Em_timedeworm_3', 'Em_timedeworm_4',
    'Em_timedeworm_5', 'Em_timedeworm_6', 'Em_timedeworm_7', 'Em_timedeworm_8',
]
print(no_training_for_men.shape[0])
for col in no_training_for_men.columns:
    print(f"{col} has {no_training_for_men[col].isnull().sum()}")

234
Em_timedeworm has 234
Em_timedeworm_1 has 234
Em_timedeworm_3 has 234
Em_timedeworm_4 has 234
Em_timedeworm_5 has 234
Em_timedeworm_6 has 234
Em_timedeworm_7 has 234
Em_timedeworm_8 has 234
Em_timedeworm_oth has 234
Em_hygint has 234
Em_hygint_bi has 234


did not
<!-- * Here: we changed NaN to -1 in all the fields that are not supposed to be filled (if the man is not in the training set then he will not fill these columns) -->

In [ ]:
timedeworm_map = {
    1.0: 'Every month',
    2.0: 'Every three months',
    3.0: 'Every six months',
    4.0: 'Every year',
    5.0: 'When you see circular patches of hair loss',
    6.0: 'When a child has no appetite',
    7.0: 'When the child looks sick',
    777.0: 'Other',}
mens_sanitation['Em_timedeworm'] = mens_sanitation['Em_timedeworm'].map(timedeworm_map)
mens_sanitation['Em_timedeworm'] = mens_sanitation['Em_timedeworm'].astype('category')

parent_df['Em_timedeworm'] = parent_df['Em_timedeworm'].map(timedeworm_map)
parent_df['Em_timedeworm'] = parent_df['Em_timedeworm'].astype('category')

extract_information(mens_sanitation,'Em_timedeworm')

Em_timedeworm    category
dtype: object
0                            Every three months
1                     When the child looks sick
2                              Every six months
3                                           NaN
4                                    Every year
5                                   Every month
6                                         Other
7                  When a child has no appetite
8    When you see circular patches of hair loss
dtype: category
Categories (8, object): ['Every month', 'Every six months', 'Every three months', 'Every year',
                         'Other', 'When a child has no appetite',
                         'When the child looks sick', 'When you see circular patches of hair loss']
336
6.095791001451379
Em_timedeworm
Every three months                            2212
Every month                                   1489
Every six months                               794
When the child looks sick                      307
Every year

In [ ]:
extract_information(mens_sanitation, 'Em_timedeworm_1')

Em_timedeworm_1    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
336
6.095791001451379
Em_timedeworm_1
0.0    5164
1.0      12
Name: count, dtype: int64
       Em_timedeworm_1
count      5176.000000
mean          0.002318
std           0.048098
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000


In [ ]:
mens_sanitation.drop('Em_timedeworm_1',axis=1,inplace=True)
parent_df.drop('Em_timedeworm_1',axis=1,inplace=True)
mens_sanitation.drop('Em_timedeworm_3',axis=1,inplace=True)
parent_df.drop('Em_timedeworm_3',axis=1,inplace=True)
mens_sanitation.drop('Em_timedeworm_4',axis=1,inplace=True)
parent_df.drop('Em_timedeworm_4',axis=1,inplace=True)
mens_sanitation.drop('Em_timedeworm_5',axis=1,inplace=True)
parent_df.drop('Em_timedeworm_5',axis=1,inplace=True)
mens_sanitation.drop('Em_timedeworm_6',axis=1,inplace=True)
parent_df.drop('Em_timedeworm_6',axis=1,inplace=True)
mens_sanitation.drop('Em_timedeworm_7',axis=1,inplace=True)
parent_df.drop('Em_timedeworm_7',axis=1,inplace=True)
mens_sanitation.drop('Em_timedeworm_8',axis=1,inplace=True)
parent_df.drop('Em_timedeworm_8',axis=1,inplace=True)

In [ ]:
extract_information(mens_sanitation,'Em_timedeworm_oth')

Em_timedeworm_oth    object
dtype: object
0                                                   NaN
1                                             Two years
2                        Follow the doctor's directions
3                                             Dont know
4                                            Two  weeks
5                                           Four months
6                                           Doesnt know
7                                         Every morning
8                      Its only the wife who knows when
9                                            don't know
10                               Every after three days
11                                           Three days
12                                           two months
13                                After every four days
14                                      After two years
15                                        every 2months
16                                               1 week
17    

In [ ]:
extract_information(mens_sanitation,'Em_hygint')

Em_hygint    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
340
6.168359941944847
Em_hygint
0.0    4523
1.0     649
Name: count, dtype: int64
         Em_hygint
count  5172.000000
mean      0.125483
std       0.331298
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(mens_sanitation,'Em_hygint_bi')

Em_hygint_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
340
6.168359941944847
Em_hygint_bi
1.0    4523
0.0     649
Name: count, dtype: int64
       Em_hygint_bi
count   5172.000000
mean       0.874517
std        0.331298
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


we will drop the Em_hygint column since Em_hygint_bi is existed and it's standard

In [ ]:
mens_sanitation = mens_sanitation.drop(columns=['Em_hygint'])
parent_df = parent_df.drop(columns=['Em_hygint'])
columns_to_drop_parent += ['Em_hygint']

### Mens Survey Section

In [ ]:
mens_survey = parent_all_sections['MENS_SURVEY']
mens_survey.columns

Index(['Estatus_m', 'Eincomplete_m', 'Em_moved', 'Em_district', 'Em_subcounty',
       'Em_survey_date', 'Em_still_married', 'Em_still_married_no'],
      dtype='object')

In [ ]:
mens_survey.describe()

,Estatus_m,Eincomplete_m,Em_moved,Em_still_married,Em_still_married_no
count,5512.000000,336.000000,5512.000000,5171.000000,135.000000
mean,0.933237,10.705357,0.001996,0.973893,2.037037
std,0.271902,25.139244,0.044632,0.159469,0.332503
min,-1.000000,1.000000,0.000000,0.000000,1.000000
25%,1.000000,2.000000,0.000000,1.000000,2.000000
50%,1.000000,4.000000,0.000000,1.000000,2.000000
75%,1.000000,5.000000,0.000000,1.000000,2.000000
max,1.000000,99.000000,1.000000,1.000000,3.000000


In [ ]:
extract_information(mens_survey,'Estatus_m')

Estatus_m    int64
dtype: object
0    1
1    0
2   -1
dtype: int64
0
0.0
Estatus_m
 1    5176
 0     304
-1      32
Name: count, dtype: int64
         Estatus_m
count  5512.000000
mean      0.933237
std       0.271902
min      -1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(mens_survey,'Eincomplete_m')

Eincomplete_m    float64
dtype: object
0      NaN
1      4.0
2      2.0
3      3.0
4     99.0
5      6.0
6      5.0
7      1.0
8      8.0
9      9.0
10     7.0
dtype: float64
5176
93.90420899854863
Eincomplete_m
4.0     103
2.0      84
6.0      34
1.0      32
3.0      27
99.0     25
9.0      10
5.0       9
8.0       6
7.0       6
Name: count, dtype: int64
       Eincomplete_m
count     336.000000
mean       10.705357
std        25.139244
min         1.000000
25%         2.000000
50%         4.000000
75%         5.000000
max        99.000000


In [ ]:
extract_information(mens_survey,'Em_still_married_no')

Em_still_married_no    float64
dtype: object
0    NaN
1    2.0
2    3.0
3    1.0
dtype: float64
5377
97.55079825834542
Em_still_married_no
2.0    120
3.0     10
1.0      5
Name: count, dtype: int64
       Em_still_married_no
count           135.000000
mean              2.037037
std               0.332503
min               1.000000
25%               2.000000
50%               2.000000
75%               2.000000
max               3.000000


In [ ]:
mens_survey['Em_still_married_no'] = mens_survey['Em_still_married_no'].astype('category')
mens_survey['Estatus_m'] = mens_survey['Estatus_m'].astype('category')

94% null

In [ ]:
parent_df.drop('Em_still_married_no',axis=1,inplace = True)
mens_survey.drop('Em_still_married_no',axis=1,inplace = True)
parent_df.drop('Eincomplete_m',axis=1,inplace = True)
mens_survey.drop('Eincomplete_m',axis=1,inplace = True)

In [ ]:
extract_information(mens_survey,'Em_still_married')

Em_still_married    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
341
6.186502177068215
Em_still_married
1.0    5036
0.0     135
Name: count, dtype: int64
       Em_still_married
count       5171.000000
mean           0.973893
std            0.159469
min            0.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            1.000000


In [ ]:
extract_information(mens_survey,'Em_survey_date')

Em_survey_date    object
dtype: object
0      2014-07-07
1      2014-07-02
2      2014-07-03
3      2014-07-05
4      2014-07-08
          ...    
109    2014-05-04
110    2014-08-15
111    2014-08-16
112    2014-04-20
113    2014-07-27
Length: 114, dtype: object
320
5.805515239477503
Em_survey_date
2014-07-16    90
2014-05-09    87
2014-06-20    83
2014-04-30    83
2014-06-21    82
              ..
2014-08-17     4
2014-08-07     3
2014-06-22     3
2014-08-16     3
2014-04-20     3
Name: count, Length: 113, dtype: int64
       Em_survey_date
count            5192
unique            113
top        2014-07-16
freq               90


In [ ]:
mens_survey['Em_survey_date'] = pd.to_datetime(mens_survey['Em_survey_date'], errors='coerce')
extract_information(mens_survey,'Em_survey_date')

Em_survey_date    datetime64[ns]
dtype: object
0     2014-07-07
1     2014-07-02
2     2014-07-03
3     2014-07-05
4     2014-07-08
         ...    
109   2014-05-04
110   2014-08-15
111   2014-08-16
112   2014-04-20
113   2014-07-27
Length: 114, dtype: datetime64[ns]
320
5.805515239477503
Em_survey_date
2014-07-16    90
2014-05-09    87
2014-06-20    83
2014-04-30    83
2014-06-21    82
              ..
2014-08-17     4
2014-08-07     3
2014-06-22     3
2014-08-16     3
2014-04-20     3
Name: count, Length: 113, dtype: int64
                      Em_survey_date
count                           5192
mean   2014-05-29 22:54:16.086286592
min              2014-03-21 00:00:00
25%              2014-04-15 00:00:00
50%              2014-06-14 00:00:00
75%              2014-07-07 00:00:00
max              2014-08-20 00:00:00


In [ ]:
extract_information(mens_survey,'Em_subcounty')

Em_subcounty    object
dtype: object
0        Bitereko
1     Kanyabwanga
2     Kashenshero
3         Katenga
4         Kiyanga
5         Mitooma
6          Mutara
7          Rurehe
8       Bwongyera
9          Ihunga
10        Kibatsi
11      Nyabihoko
12          Itojo
13        Kayonza
14       Ntungamo
15        Ruhaama
16    Rukoni East
17     Rweikiniro
18          Ngoma
19        Rubaare
20       Rugarama
21        Buyanja
22       Kebisoni
23    Nyakishenyi
24    Nyarushanje
25        Buhunga
26       Bwambara
27        Ruhinda
28        Bugongi
29        Kasaana
30       Kigarama
31       Kitagata
32     Kyangyenyi
33        Mayanga
34       Nyakyera
35     Nyakagyeme
36      Masheruka
37         Shuuku
38    Rukoni West
39      Bugangari
40         Kabira
41            NaN
42       Rwampara
43          Ndija
dtype: object
6
0.10885341074020319
Em_subcounty
Ruhaama        206
Nyakishenyi    205
Nyarushanje    197
Nyakyera       193
Ngoma          190
Bwongyera      180
Ntungamo

In [ ]:
mens_survey['Em_subcounty'] = mens_survey['Em_subcounty'].astype('category')
extract_information(mens_survey,'Em_subcounty')

Em_subcounty    category
dtype: object
0        Bitereko
1     Kanyabwanga
2     Kashenshero
3         Katenga
4         Kiyanga
5         Mitooma
6          Mutara
7          Rurehe
8       Bwongyera
9          Ihunga
10        Kibatsi
11      Nyabihoko
12          Itojo
13        Kayonza
14       Ntungamo
15        Ruhaama
16    Rukoni East
17     Rweikiniro
18          Ngoma
19        Rubaare
20       Rugarama
21        Buyanja
22       Kebisoni
23    Nyakishenyi
24    Nyarushanje
25        Buhunga
26       Bwambara
27        Ruhinda
28        Bugongi
29        Kasaana
30       Kigarama
31       Kitagata
32     Kyangyenyi
33        Mayanga
34       Nyakyera
35     Nyakagyeme
36      Masheruka
37         Shuuku
38    Rukoni West
39      Bugangari
40         Kabira
41            NaN
42       Rwampara
43          Ndija
dtype: category
Categories (43, object): ['Bitereko', 'Bugangari', 'Bugongi', 'Buhunga', ..., 'Rurehe', 'Rwampara',
                          'Rweikiniro', 'Shuuku']
6
0

In [ ]:
extract_information(mens_survey,'Em_district')

Em_district    object
dtype: object
0      Mitooma
1     Ntungamo
2    Rukungiri
3       Sheema
4          NaN
dtype: object
6
0.10885341074020319
Em_district
Ntungamo     2268
Rukungiri    1359
Mitooma      1076
Sheema        803
Name: count, dtype: int64
       Em_district
count         5506
unique           4
top       Ntungamo
freq          2268


In [ ]:
mens_survey['Em_district'] = mens_survey['Em_district'].astype('category')
extract_information(mens_survey,'Em_district')

Em_district    category
dtype: object
0      Mitooma
1     Ntungamo
2    Rukungiri
3       Sheema
4          NaN
dtype: category
Categories (4, object): ['Mitooma', 'Ntungamo', 'Rukungiri', 'Sheema']
6
0.10885341074020319
Em_district
Ntungamo     2268
Rukungiri    1359
Mitooma      1076
Sheema        803
Name: count, dtype: int64
       Em_district
count         5506
unique           4
top       Ntungamo
freq          2268


In [ ]:
mens_survey.describe(include = 'category')

,Estatus_m,Em_district,Em_subcounty
count,5512,5506,5506
unique,3,4,43
top,1,Ntungamo,Ruhaama
freq,5176,2268,206


In [ ]:
extract_information(mens_survey,'Em_moved')

Em_moved    int64
dtype: object
0    0
1    1
dtype: int64
0
0.0
Em_moved
0    5501
1      11
Name: count, dtype: int64
          Em_moved
count  5512.000000
mean      0.001996
std       0.044632
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
mens_survey.describe()

,Em_moved,Em_survey_date,Em_still_married
count,5512.000000,5192,5171.000000
mean,0.001996,2014-05-29 22:54:16.086286592,0.973893
min,0.000000,2014-03-21 00:00:00,0.000000
25%,0.000000,2014-04-15 00:00:00,1.000000
50%,0.000000,2014-06-14 00:00:00,1.000000
75%,0.000000,2014-07-07 00:00:00,1.000000
max,1.000000,2014-08-20 00:00:00,1.000000
std,0.044632,NaN,0.159469


### Attendance section

In [ ]:
attendance = parent_all_sections['ATTENDANCE']
attendance.columns

Index(['Eattend', 'Eattend_women', 'Eattend_men', 'Eattend_wemp',
       'Eattend_whn', 'sessn_att1', 'sessn_att2', 'sessn_att3', 'sessn_att4',
       'sessn_att5', 'sessn_att6', 'sessn_att7', 'sessn_att8', 'sessn_att9',
       'sessn_att10', 'sessn_att11', 'sessn_att12', 'sessn_att13',
       'sessn_att14', 'sessn_att15', 'sessn_att16', 'sessn_att18',
       'sessn_att19'],
      dtype='object')

In [ ]:
attendance.describe()

,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19
count,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,4083.000000,4083.000000,4115.000000,4114.000000,4130.000000,4131.000000,4131.000000,4130.000000,4131.000000,4131.000000,4131.00000,4131.000000,4131.000000,4131.000000,4131.000000,4130.000000,4130.000000,4131.00000
mean,0.527979,0.385402,0.142578,0.188206,0.197196,0.743571,0.740142,0.709356,0.734079,0.637772,0.745098,0.659647,0.741404,0.675865,0.725732,0.65069,0.743646,0.669572,0.734931,0.654805,0.730024,0.763196,0.80029
std,0.401002,0.425263,0.299790,0.355744,0.358517,0.436715,0.438610,0.454115,0.441876,0.480702,0.435859,0.473885,0.437916,0.468107,0.446199,0.47681,0.436673,0.470424,0.441423,0.475489,0.444001,0.425172,0.39983
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.00000
50%,0.666667,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000
75%,0.888889,0.833333,0.000000,0.000000,0.055556,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


* <b> a maybe true fact: women attend more than men

In [ ]:
session_cols = [
    'sessn_att1', 'sessn_att2', 'sessn_att3', 'sessn_att4', 'sessn_att5',
    'sessn_att6', 'sessn_att7', 'sessn_att8', 'sessn_att9', 'sessn_att10',
    'sessn_att11', 'sessn_att12', 'sessn_att13', 'sessn_att14', 'sessn_att15',
    'sessn_att16', 'sessn_att18', 'sessn_att19'
]

In [ ]:
extract_information(attendance,'Eattend')

Eattend    float64
dtype: object
0     1.000000
1     0.000000
2     0.055556
3     0.666667
4     0.833333
5     0.555556
6     0.777778
7     0.222222
8     0.888889
9     0.944444
10    0.611111
11    0.722222
12    0.111111
13    0.277778
14    0.333333
15    0.388889
16    0.166667
17    0.500000
18    0.444444
dtype: float64
0
0.0
Eattend
0.000000    1594
0.944444     630
1.000000     582
0.888889     519
0.833333     418
0.777778     331
0.722222     244
0.666667     189
0.611111     135
0.555556     131
0.055556     106
0.500000     100
0.444444      93
0.388889      82
0.333333      80
0.111111      76
0.222222      75
0.166667      68
0.277778      59
Name: count, dtype: int64
           Eattend
count  5512.000000
mean      0.527979
std       0.401002
min       0.000000
25%       0.000000
50%       0.666667
75%       0.888889
max       1.000000


In [ ]:
attendance[attendance['Eattend'] == 0.0]

,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19
1,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5501,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5504,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5507,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5508,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


* <b>Eattend: the ration of attendance in general, range 0 -> 1<br>
* <b>Eattend_women: the ration of attendance of womans in general, range 0 -> 1<br>
* <b>Eattend_men: the ration of attendance of men in general,  range 0 -> 1<br>
* <b>Eattend_wemp: the ration of attendance in training and تمكين, range 0 -> 1<br>
* <b>Eattend_whn: the ration of attendance in training only, range 0 -> 1<br>

In [ ]:
attendance[session_cols]

,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5507,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5508,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5509,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
5510,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0


In [ ]:
attendance[(parent_df['mhn']==0) & (parent_df['whn']==0) & (parent_df['wemp']==0)].head()

,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19
1,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
session_cols = [col for col in attendance.columns if col.startswith('sessn_att')]

cat_cols = []
for col in session_cols:
    new_col_name = col + '_cat'
    cat_cols.append(new_col_name)

    attendance[new_col_name] = attendance[col].fillna(-1)

    attendance[new_col_name] = attendance[new_col_name].astype('category')


attendance.head()

,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat
0,1.000000,1.000000,0.0,1.000000,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,0.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
2,0.055556,0.055556,0.0,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
4,0.666667,0.666667,0.0,0.666667,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0



### Mens Food Recall section

In [ ]:
mens_food_recall = parent_all_sections['MENS_FOOD_RECALL']
mens_food_recall.columns

Index(['Em_milk', 'Em_water', 'Em_tea', 'Em_juice', 'Em_otherbev',
       'Em_matooke', 'Em_roots', 'Em_grains', 'Em_beans', 'Em_organ',
       'Em_meat', 'Em_egg', 'Em_fish', 'Em_green', 'Em_pump', 'Em_otherfru',
       'Em_othdairy', 'Em_oil', 'Em_sweet', 'Em_othersolid'],
      dtype='object')

In [ ]:
extract_information(mens_food_recall,'Em_milk')

Em_milk    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
336
6.095791001451379
Em_milk
0.0    3100
1.0    2076
Name: count, dtype: int64
           Em_milk
count  5176.000000
mean      0.401082
std       0.490165
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


In [ ]:
mens_food_recall.describe()

,Em_milk,Em_water,Em_tea,Em_juice,Em_otherbev,Em_matooke,Em_roots,Em_grains,Em_beans,Em_organ,Em_meat,Em_egg,Em_fish,Em_green,Em_pump,Em_otherfru,Em_othdairy,Em_oil,Em_sweet,Em_othersolid
count,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000
mean,0.401082,0.892388,0.170209,0.088292,0.802357,0.837906,0.667117,0.685278,0.940688,0.080178,0.274730,0.048686,0.034583,0.641229,0.268934,0.752898,0.098338,0.268547,0.130410,0.183346
std,0.490165,0.309920,0.375852,0.283747,0.398260,0.368573,0.471290,0.464450,0.236231,0.271594,0.446421,0.215232,0.182738,0.479686,0.443448,0.431368,0.297801,0.443247,0.336786,0.386987
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
mens_food_recall.head()

,Em_milk,Em_water,Em_tea,Em_juice,Em_otherbev,Em_matooke,Em_roots,Em_grains,Em_beans,Em_organ,Em_meat,Em_egg,Em_fish,Em_green,Em_pump,Em_otherfru,Em_othdairy,Em_oil,Em_sweet,Em_othersolid
0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0


In [ ]:
mens_food_recall.nunique()

,0
Em_milk,2
Em_water,2
Em_tea,2
Em_juice,2
Em_otherbev,2
Em_matooke,2
Em_roots,2
Em_grains,2
Em_beans,2
Em_organ,2


### Men Fertility section

In [ ]:
men_fertility = parent_all_sections['MENS_FERTILITY']
men_fertility.columns

Index(['Em_preg_now', 'Em_preg_now_ac', 'Em_preg_past2', 'Em_preg_past2_ac',
       'Em_preg_comp', 'Em_preg_comp_oth', 'Em_labr_present', 'Em_fp_decide',
       'Em_fp_talk', 'Em_hus_ideal_fert_t', 'Em_hus_ideal_fert_b',
       'Em_hus_ideal_fert_g', 'Em_hus_ideal_fert_e', 'Em_nextpreg_when',
       'Em_nextpreg_units', 'Em_op_birthspace', 'Em_op_birthspace_u',
       'Em_wait_nextpreg', 'Em_wait_nextpreg_u'],
      dtype='object')

In [ ]:
men_fertility.describe()

,Em_preg_now,Em_preg_now_ac,Em_preg_past2,Em_preg_past2_ac,Em_labr_present,Em_fp_decide,Em_fp_talk,Em_hus_ideal_fert_t,Em_hus_ideal_fert_b,Em_hus_ideal_fert_g,Em_hus_ideal_fert_e,Em_nextpreg_when,Em_nextpreg_units,Em_op_birthspace,Em_op_birthspace_u,Em_wait_nextpreg,Em_wait_nextpreg_u
count,4978.000000,761.000000,5036.000000,5036.000000,5036.000000,5036.000000,5036.000000,5034.000000,5034.000000,5034.00000,5034.000000,3365.000000,3469.000000,5153.000000,5154.000000,4440.000000,4442.000000
mean,0.152873,0.651774,0.602462,0.981930,0.743646,2.836378,2.625894,4.216925,1.967620,1.66468,0.584625,3.398217,1.394926,2.793130,0.664144,3.813964,1.983341
std,0.359901,0.476721,0.489437,0.133217,0.436663,0.492884,0.621876,1.821845,1.307312,1.14288,1.633669,3.781188,5.072398,1.401909,3.020135,3.470934,0.128005
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,0.000000,3.000000,2.000000,3.000000,2.000000,1.00000,0.000000,2.000000,0.000000,2.000000,0.000000,2.000000,2.000000
50%,0.000000,1.000000,1.000000,1.000000,1.000000,3.000000,3.000000,4.000000,2.000000,2.00000,0.000000,3.000000,0.000000,3.000000,0.000000,3.000000,2.000000
75%,0.000000,1.000000,1.000000,1.000000,1.000000,3.000000,3.000000,5.000000,2.000000,2.00000,0.000000,5.000000,0.000000,3.000000,0.000000,4.000000,2.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,3.000000,3.000000,30.000000,24.000000,20.00000,30.000000,120.000000,72.000000,24.000000,120.000000,60.000000,2.000000


In [ ]:
extract_information(men_fertility,'Em_preg_now_ac')

Em_preg_now_ac    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
4751
86.19375907111755
Em_preg_now_ac
1.0    496
0.0    265
Name: count, dtype: int64
       Em_preg_now_ac
count      761.000000
mean         0.651774
std          0.476721
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


In [ ]:
# men_fertility.drop('Em_preg_now_ac',axis=1,inplace=True)
# parent_df.drop('Em_preg_now_ac',axis=1,inplace=True

In [ ]:
extract_information(men_fertility,'Em_hus_ideal_fert_t')

Em_hus_ideal_fert_t    float64
dtype: object
0      4.0
1      2.0
2     10.0
3      8.0
4      6.0
5      NaN
6      3.0
7      5.0
8      1.0
9      7.0
10    15.0
11    20.0
12     9.0
13    12.0
14     0.0
15    14.0
16    30.0
17    11.0
18    13.0
19    16.0
dtype: float64
478
8.67198838896952
Em_hus_ideal_fert_t
4.0     2433
3.0      810
6.0      543
5.0      483
2.0      477
8.0       88
10.0      68
7.0       57
1.0       34
12.0      15
20.0       8
15.0       6
30.0       4
0.0        2
9.0        2
14.0       1
11.0       1
13.0       1
16.0       1
Name: count, dtype: int64
       Em_hus_ideal_fert_t
count          5034.000000
mean              4.216925
std               1.821845
min               0.000000
25%               3.000000
50%               4.000000
75%               5.000000
max              30.000000


In [ ]:
extract_information(men_fertility,'Em_hus_ideal_fert_b')

Em_hus_ideal_fert_b    float64
dtype: object
0      2.0
1      3.0
2      1.0
3      5.0
4      4.0
5      NaN
6      0.0
7     10.0
8      6.0
9      8.0
10     7.0
11    14.0
12    13.0
13    20.0
14    15.0
15    24.0
16    12.0
dtype: float64
478
8.67198838896952
Em_hus_ideal_fert_b
2.0     2641
3.0      817
0.0      730
1.0      508
4.0      218
5.0       53
6.0       35
8.0       10
10.0       9
7.0        7
14.0       1
13.0       1
20.0       1
15.0       1
24.0       1
12.0       1
Name: count, dtype: int64
       Em_hus_ideal_fert_b
count          5034.000000
mean              1.967620
std               1.307312
min               0.000000
25%               2.000000
50%               2.000000
75%               2.000000
max              24.000000


In [ ]:
extract_information(men_fertility,'Em_hus_ideal_fert_g')

Em_hus_ideal_fert_g    float64
dtype: object
0      2.0
1      1.0
2      5.0
3      4.0
4      3.0
5      NaN
6      0.0
7      8.0
8     10.0
9      6.0
10    17.0
11     7.0
12    20.0
13     9.0
dtype: float64
478
8.67198838896952
Em_hus_ideal_fert_g
2.0     2404
1.0     1091
0.0      830
3.0      510
4.0      135
5.0       29
6.0       20
8.0        4
10.0       4
7.0        4
17.0       1
20.0       1
9.0        1
Name: count, dtype: int64
       Em_hus_ideal_fert_g
count           5034.00000
mean               1.66468
std                1.14288
min                0.00000
25%                1.00000
50%                2.00000
75%                2.00000
max               20.00000


In [ ]:
extract_information(men_fertility,'Em_hus_ideal_fert_e')

Em_hus_ideal_fert_e    float64
dtype: object
0      0.0
1      NaN
2      4.0
3      3.0
4      5.0
5      2.0
6      6.0
7      1.0
8     10.0
9      7.0
10     8.0
11    20.0
12    12.0
13    30.0
dtype: float64
478
8.67198838896952
Em_hus_ideal_fert_e
0.0     4307
4.0      275
3.0      159
2.0       99
6.0       70
5.0       63
1.0       19
10.0      16
8.0       13
7.0        8
20.0       2
12.0       2
30.0       1
Name: count, dtype: int64
       Em_hus_ideal_fert_e
count          5034.000000
mean              0.584625
std               1.633669
min               0.000000
25%               0.000000
50%               0.000000
75%               0.000000
max              30.000000


In [ ]:
extract_information(men_fertility,'Em_preg_comp')

Em_preg_comp    object
dtype: object
0         1
1         3
2         2
3       NaN
4         8
      ...  
72      1 8
73    4 5 8
74    3 7 8
75    2 5 7
76      6 8
Length: 77, dtype: object
476
8.635703918722786
Em_preg_comp
1            3846
-777          309
8             196
4             143
3              96
             ... 
2 3 other       1
2 5 6 7 8       1
3 7 8           1
2 5 7           1
6 8             1
Name: count, Length: 76, dtype: int64
       Em_preg_comp
count          5036
unique           76
top               1
freq           3846


In [ ]:
comp_map = {
    '1': 'None',
    '2': 'Convulsions',
    '3': 'Swelling',
    '4': 'Vaginal Bleeding',
    '5': 'Trouble with vision during day',
    '6': 'Night Blindness',
    '7': 'High Blood Pressure',
    '8': 'Positioning of Baby',
    '9': 'Other',
    'other': 'Other',
    '-777': 'Unknown/Missing'
}

def translate_multiple_choices(val):
    if pd.isna(val):
        return []
    items = str(val).split()
    mapped_items = [comp_map.get(item, item) for item in items]
    return mapped_items

men_fertility['Em_preg_comp'] = men_fertility['Em_preg_comp'].apply(translate_multiple_choices)

In [ ]:
dummies = men_fertility['Em_preg_comp'].str.join('|').str.get_dummies()
men_fertility = pd.concat([men_fertility, dummies], axis=1)
columns_to_drop_parent += ['Em_preg_comp']


In [ ]:
men_fertility.head()

,Em_preg_now,Em_preg_now_ac,Em_preg_past2,Em_preg_past2_ac,Em_preg_comp,Em_preg_comp_oth,Em_labr_present,Em_fp_decide,Em_fp_talk,Em_hus_ideal_fert_t,Em_hus_ideal_fert_b,Em_hus_ideal_fert_g,Em_hus_ideal_fert_e,Em_nextpreg_when,Em_nextpreg_units,Em_op_birthspace,Em_op_birthspace_u,Em_wait_nextpreg,Em_wait_nextpreg_u,Convulsions,High Blood Pressure,Night Blindness,None,Other,Positioning of Baby,Swelling,Trouble with vision during day,Unknown/Missing,Vaginal Bleeding
0,0.0,NaN,1.0,1.0,[None],NaN,1.0,3.0,3.0,4.0,2.0,2.0,0.0,NaN,NaN,4.0,0.0,4.0,2.0,0,0,0,1,0,0,0,0,0,0
1,0.0,NaN,0.0,1.0,[Swelling],NaN,1.0,3.0,3.0,4.0,2.0,2.0,0.0,NaN,NaN,2.0,0.0,2.0,2.0,0,0,0,0,0,0,1,0,0,0
2,0.0,NaN,0.0,1.0,[None],NaN,1.0,3.0,2.0,4.0,2.0,2.0,0.0,NaN,0.0,3.0,0.0,5.0,2.0,0,0,0,1,0,0,0,0,0,0
3,0.0,NaN,0.0,1.0,[None],NaN,1.0,3.0,2.0,4.0,3.0,1.0,0.0,3.0,0.0,3.0,0.0,3.0,2.0,0,0,0,1,0,0,0,0,0,0
4,0.0,NaN,0.0,0.0,[None],NaN,0.0,3.0,2.0,4.0,2.0,2.0,0.0,NaN,NaN,4.0,0.0,4.0,2.0,0,0,0,1,0,0,0,0,0,0


In [ ]:
extract_information(men_fertility,'Em_preg_comp_oth')
#TODO

Em_preg_comp_oth    object
dtype: object
0                                     NaN
1                                 Malalia
2                       Pains in the womb
3                           Malaria fever
4                                Diziness
                      ...                
185                           Chest pains
186              Orukiinda yamara kuzaara
187    Pains/contractions in the abdomen.
188                     Had stomach aches
189             Exceeded gestation period
Length: 190, dtype: object
5161
93.63207547169812
Em_preg_comp_oth
Malaria                               54
Stomach pain                          19
Stomach pains                         17
Abdominal pain                        11
Abdominal Pain                         8
                                      ..
Chest pains                            1
Orukiinda yamara kuzaara               1
Pains/contractions in the abdomen.     1
Had stomach aches                      1
Exceeded gestation p

In [ ]:
extract_information(men_fertility,'Em_fp_decide')

Em_fp_decide    float64
dtype: object
0    3.0
1    1.0
2    NaN
3    2.0
dtype: float64
476
8.635703918722786
Em_fp_decide
3.0    4479
2.0     290
1.0     267
Name: count, dtype: int64
       Em_fp_decide
count   5036.000000
mean       2.836378
std        0.492884
min        1.000000
25%        3.000000
50%        3.000000
75%        3.000000
max        3.000000


In [ ]:
decision_map = {
    1 : 'Mainly my decision',
    2 : 'Mainly my wife’s/partner’s decision',
    3 : 'Joint decision',
}
men_fertility['Em_fp_decide'] = men_fertility['Em_fp_decide'].map(decision_map)
men_fertility['Em_fp_decide'] = men_fertility['Em_fp_decide'].astype('category')
parent_df['Em_fp_decide'] = parent_df['Em_fp_decide'].map(decision_map)
parent_df['Em_fp_decide'] = parent_df['Em_fp_decide'].astype('category')
extract_information(men_fertility,'Em_fp_decide')

Em_fp_decide    category
dtype: object
0                         Joint decision
1                     Mainly my decision
2                                    NaN
3    Mainly my wife’s/partner’s decision
dtype: category
Categories (3, object): ['Joint decision', 'Mainly my decision',
                         'Mainly my wife’s/partner’s decision']
476
8.635703918722786
Em_fp_decide
Joint decision                         4479
Mainly my wife’s/partner’s decision     290
Mainly my decision                      267
Name: count, dtype: int64
          Em_fp_decide
count             5036
unique               3
top     Joint decision
freq              4479


In [ ]:
extract_information(men_fertility,'Em_fp_talk')

Em_fp_talk    float64
dtype: object
0    3.0
1    2.0
2    1.0
3    NaN
dtype: float64
476
8.635703918722786
Em_fp_talk
3.0    3536
2.0    1116
1.0     384
Name: count, dtype: int64
        Em_fp_talk
count  5036.000000
mean      2.625894
std       0.621876
min       1.000000
25%       2.000000
50%       3.000000
75%       3.000000
max       3.000000


In [ ]:
talk_map = {
    1 : 'Never',
    2 : 'Once or twice',
    3 : 'More often',
}
men_fertility['Em_fp_talk'] = men_fertility['Em_fp_talk'].map(decision_map)
men_fertility['Em_fp_talk'] = men_fertility['Em_fp_talk'].astype('category')
extract_information(men_fertility,'Em_fp_talk')

Em_fp_talk    category
dtype: object
0                         Joint decision
1    Mainly my wife’s/partner’s decision
2                     Mainly my decision
3                                    NaN
dtype: category
Categories (3, object): ['Joint decision', 'Mainly my decision',
                         'Mainly my wife’s/partner’s decision']
476
8.635703918722786
Em_fp_talk
Joint decision                         3536
Mainly my wife’s/partner’s decision    1116
Mainly my decision                      384
Name: count, dtype: int64
            Em_fp_talk
count             5036
unique               3
top     Joint decision
freq              3536


In [ ]:
extract_information(men_fertility,'Em_nextpreg_when')

Em_nextpreg_when    float64
dtype: object
0       NaN
1       3.0
2       0.0
3       6.0
4       8.0
5       2.0
6       4.0
7       5.0
8      10.0
9       1.0
10     15.0
11      7.0
12     72.0
13     20.0
14     36.0
15     12.0
16     62.0
17     11.0
18     60.0
19    120.0
20      9.0
21     48.0
dtype: float64
2147
38.951378809869375
Em_nextpreg_when
3.0      779
2.0      694
5.0      537
4.0      425
1.0      311
0.0      302
6.0      113
10.0     111
8.0       36
7.0       33
15.0       8
12.0       3
20.0       2
9.0        2
36.0       2
48.0       2
72.0       1
11.0       1
62.0       1
120.0      1
60.0       1
Name: count, dtype: int64
       Em_nextpreg_when
count       3365.000000
mean           3.398217
std            3.781188
min            0.000000
25%            2.000000
50%            3.000000
75%            5.000000
max          120.000000


In [ ]:
extract_information(men_fertility,'Em_nextpreg_units')

Em_nextpreg_units    float64
dtype: object
0      NaN
1      0.0
2      6.0
3     18.0
4      4.0
5      7.0
6      9.0
7      5.0
8      1.0
9      8.0
10    36.0
11    10.0
12    60.0
13     2.0
14     3.0
15    62.0
16    11.0
17    30.0
18    12.0
19    38.0
20    24.0
21    48.0
22    34.0
23    72.0
24    32.0
25    54.0
26    29.0
27    28.0
28    55.0
29    42.0
30    40.0
31    16.0
32    21.0
33    13.0
34    41.0
dtype: float64
2043
37.06458635703919
Em_nextpreg_units
0.0     2805
6.0      220
1.0       79
5.0       52
2.0       50
3.0       47
8.0       40
4.0       37
7.0       33
9.0       29
10.0      21
11.0      10
36.0       7
12.0       5
72.0       5
18.0       4
24.0       4
62.0       2
28.0       2
54.0       2
32.0       2
60.0       1
30.0       1
34.0       1
38.0       1
48.0       1
29.0       1
55.0       1
42.0       1
40.0       1
16.0       1
21.0       1
13.0       1
41.0       1
Name: count, dtype: int64
       Em_nextpreg_units
count        3469.00000

In [ ]:
extract_information(men_fertility,'Em_op_birthspace')

Em_op_birthspace    float64
dtype: object
0      4.0
1      2.0
2      3.0
3      5.0
4      NaN
5      6.0
6     10.0
7      0.0
8      1.0
9     15.0
10     7.0
11     9.0
12     8.0
13    24.0
14    12.0
15    18.0
dtype: float64
359
6.513062409288825
Em_op_birthspace
2.0     1883
3.0     1526
4.0      592
1.0      493
5.0      451
0.0       83
6.0       70
10.0      19
7.0       17
8.0       12
15.0       3
9.0        1
24.0       1
12.0       1
18.0       1
Name: count, dtype: int64
       Em_op_birthspace
count       5153.000000
mean           2.793130
std            1.401909
min            0.000000
25%            2.000000
50%            3.000000
75%            3.000000
max           24.000000


In [ ]:
extract_information(men_fertility,'Em_op_birthspace_u')

Em_op_birthspace_u    float64
dtype: object
0       0.0
1       NaN
2       5.0
3       6.0
4       8.0
5       3.0
6      18.0
7      20.0
8      24.0
9       9.0
10      4.0
11      2.0
12      7.0
13     17.0
14      1.0
15     10.0
16     12.0
17     48.0
18    120.0
19     30.0
20     84.0
dtype: float64
358
6.494920174165457
Em_op_birthspace_u
0.0      4651
6.0       349
5.0        29
3.0        27
8.0        24
9.0        15
2.0        14
4.0         9
18.0        7
1.0         6
12.0        5
7.0         5
24.0        4
30.0        3
20.0        1
17.0        1
10.0        1
48.0        1
120.0       1
84.0        1
Name: count, dtype: int64
       Em_op_birthspace_u
count         5154.000000
mean             0.664144
std              3.020135
min              0.000000
25%              0.000000
50%              0.000000
75%              0.000000
max            120.000000


In [ ]:
extract_information(men_fertility,'Em_wait_nextpreg')

Em_wait_nextpreg    float64
dtype: object
0      4.0
1      2.0
2      5.0
3      3.0
4      NaN
5      7.0
6      6.0
7     10.0
8     15.0
9      1.0
10    18.0
11     8.0
12    48.0
13     9.0
14    17.0
15    30.0
16    22.0
17    29.0
18     0.0
19    12.0
20    60.0
21    42.0
22    52.0
23    32.0
dtype: float64
1072
19.448476052249635
Em_wait_nextpreg
3.0     1543
2.0     1057
4.0      777
5.0      661
6.0      122
1.0       88
10.0      52
7.0       34
18.0      31
30.0      28
8.0       27
42.0       4
15.0       3
12.0       2
48.0       2
32.0       2
9.0        1
17.0       1
22.0       1
0.0        1
29.0       1
60.0       1
52.0       1
Name: count, dtype: int64
       Em_wait_nextpreg
count       4440.000000
mean           3.813964
std            3.470934
min            0.000000
25%            2.000000
50%            3.000000
75%            4.000000
max           60.000000


In [ ]:
extract_information(men_fertility,'Em_wait_nextpreg_u')

Em_wait_nextpreg_u    float64
dtype: object
0    2.0
1    NaN
2    1.0
dtype: float64
1070
19.4121915820029
Em_wait_nextpreg_u
2.0    4368
1.0      74
Name: count, dtype: int64
       Em_wait_nextpreg_u
count         4442.000000
mean             1.983341
std              0.128005
min              1.000000
25%              2.000000
50%              2.000000
75%              2.000000
max              2.000000


### Mens Health Knowledge section done

In [ ]:
mens_health_knowledge = parent_all_sections['MENS_HEALTH_KNOWLEDGE']
mens_health_knowledge.columns

Index(['Em_cstrum_feed', 'Em_cstrum_no_why', 'Em_cstrum_no_why_oth',
       'Em_cstrum_impt_why', 'Em_cstrum_impt_why_bi', 'Em_grow_lackdiet',
       'Em_grow_lackdiet_sw', 'Em_grow_lackdiet_vm', 'Em_grow_witch',
       'Em_grow_witch_lt', 'Em_grow_witch_vl', 'Em_kwash_cause',
       'Em_kwash_cause_bi', 'Em_kwash_cause_oth', 'Em_kwash_treat',
       'Em_kwash_treat_bi', 'Em_kwash_treat_oth', 'Em_ageliquid',
       'Em_ageliquid_6', 'Em_agefood', 'Em_agefood_6', 'Em_bfeed',
       'Em_bfeed24', 'Em_boil_1min', 'Em_boil_1min_bi', 'Em_weanfat',
       'Em_weanfat_bi', 'Em_porr_thin', 'Em_porr_thin_bi', 'Em_cowmilk',
       'Em_cowmilk_bi', 'Em_menfruit', 'Em_menfruit_bi', 'Em_menprot',
       'Em_menprot_bi', 'Em_hiv_bfeed', 'Em_hiv_bfeed_bi', 'Em_safe_primhc',
       'Em_safe_primhc_bi', 'Em_boys_eatmore', 'Em_boys_eatmore_bi',
       'Em_vita_foods_bi', 'Em_anem_foods_bi', 'Em_food_plate',
       'Em_food_plate_bi', 'Em_child_worms', 'Em_child_worms_bi',
       'Em_condom_times', 'Em_c

In [ ]:
extract_information(mens_health_knowledge,'Em_cstrum_feed')

Em_cstrum_feed    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
336
6.095791001451379
Em_cstrum_feed
1.0    4695
0.0     481
Name: count, dtype: int64
       Em_cstrum_feed
count     5176.000000
mean         0.907071
std          0.290361
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_cstrum_no_why')

Em_cstrum_no_why    float64
dtype: object
0      NaN
1      1.0
2      3.0
3      2.0
4      6.0
5      4.0
6    777.0
7      5.0
dtype: float64
5064
91.8722786647315
Em_cstrum_no_why
1.0      296
4.0       61
2.0       45
777.0     18
3.0       12
5.0       12
6.0        4
Name: count, dtype: int64
       Em_cstrum_no_why
count        448.000000
mean          32.892857
std          152.418364
min            1.000000
25%            1.000000
50%            1.000000
75%            2.000000
max          777.000000


In [ ]:
no_colostrum_map = {
    1 : 'It is unhealthy for the child',
    2 : 'It gives the child diarrhea',
    3 : 'It smells',
    4 : 'It’s useless/ has no nutritional benefits',
    5 :'HIV transmission',
    6 : 'Don’t know',
    777 : 'Other'
}
mens_health_knowledge['Em_cstrum_no_why'] = mens_health_knowledge['Em_cstrum_no_why'].map(no_colostrum_map)
mens_health_knowledge['Em_cstrum_no_why'] = mens_health_knowledge['Em_cstrum_no_why'].astype('category')
parent_df['Em_cstrum_no_why'] = parent_df['Em_cstrum_no_why'].map(no_colostrum_map)
parent_df['Em_cstrum_no_why'] = parent_df['Em_cstrum_no_why'].astype('category')
extract_information(mens_health_knowledge,'Em_cstrum_no_why')


Em_cstrum_no_why    category
dtype: object
0                                          NaN
1                It is unhealthy for the child
2                                    It smells
3                  It gives the child diarrhea
4                                   Don’t know
5    It’s useless/ has no nutritional benefits
6                                        Other
7                             HIV transmission
dtype: category
Categories (7, object): ['Don’t know', 'HIV transmission', 'It gives the child diarrhea',
                         'It is unhealthy for the child', 'It smells',
                         'It’s useless/ has no nutritional benefits', 'Other']
5064
91.8722786647315
Em_cstrum_no_why
It is unhealthy for the child                296
It’s useless/ has no nutritional benefits     61
It gives the child diarrhea                   45
Other                                         18
HIV transmission                              12
It smells                                

In [ ]:
extract_information(mens_health_knowledge,'Em_cstrum_no_why_oth')

Em_cstrum_no_why_oth    object
dtype: object
0                                                  NaN
1                                 Too hot for the baby
2                               Make kids get diarrhea
3                                          Its waterly
4                                  causes constipation
5                                      It brings worms
6                                  Not yet breast milk
7                                     It is infectious
8                                            Not sweet
9                    It infects children with diseases
10                Normal breast milk without colostrum
11                              Makes the child sickly
12    It might cause the child problems in the stomach
13                                It is think not good
14                                           its dirty
15                      The child cannot dificate well
16                                 not yet breast milk
17                  

In [ ]:
extract_information(mens_health_knowledge,'Em_cstrum_impt_why')

Em_cstrum_impt_why    object
dtype: object
0         1 2 4
1           4 5
2             5
3           1 4
4             1
5         1 3 4
6             4
7           NaN
8           1 5
9       1 3 4 5
10          3 4
11          3 5
12        3 4 5
13        1 4 5
14        1 2 5
15          1 3
16          2 3
17      1 2 3 4
18            3
19        2 3 5
20          2 4
21        2 4 5
22            2
23          2 5
24        2 3 4
25          1 2
26        1 3 5
27        1 2 3
28    1 2 3 4 5
29      1 2 3 5
dtype: object
817
14.822206095791001
Em_cstrum_impt_why
5            1205
4             916
1 4           745
1             353
4 5           281
1 3 4         203
3 4           167
1 5           128
1 4 5          92
2              90
3              86
1 3            78
2 4            77
2 5            53
1 2 4          42
1 2            34
3 5            33
1 3 4 5        19
1 3 5          17
3 4 5          16
1 2 3 4        15
2 3            12
2 3 4           9
1 2 3  

In [ ]:
cstrum_impt_why_map = {
    1 :  'Builds immunity//reduces sickness',
    2 : 'It contains iron',
    3 : 'Boosts brain function',
    4 : 'Helps weight gain and growth',
    5 : "Participant says it improves health but can't state how exactly it improves health"
}
#TODO
#Fix the types as an array or more columns

In [ ]:
extract_information(mens_health_knowledge,'Em_cstrum_impt_why_bi')

Em_cstrum_impt_why_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
336
6.095791001451379
Em_cstrum_impt_why_bi
0.0    3162
1.0    2014
Name: count, dtype: int64
       Em_cstrum_impt_why_bi
count            5176.000000
mean                0.389104
std                 0.487594
min                 0.000000
25%                 0.000000
50%                 0.000000
75%                 1.000000
max                 1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_grow_lackdiet')

Em_grow_lackdiet    float64
dtype: object
0    1.0
1    3.0
2    2.0
3    NaN
4    4.0
5    5.0
dtype: float64
344
6.2409288824383164
Em_grow_lackdiet
1.0    4124
2.0     775
3.0     222
4.0      43
5.0       4
Name: count, dtype: int64
       Em_grow_lackdiet
count       5168.000000
mean           1.263932
std            0.582633
min            1.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            5.000000


In [ ]:
mens_health_knowledge['Em_grow_lackdiet'] = mens_health_knowledge['Em_grow_lackdiet'].map(grad_map)
mens_health_knowledge['Em_grow_lackdiet'] = mens_health_knowledge['Em_grow_lackdiet'].astype('category')
parent_df['Em_grow_lackdiet'] = parent_df['Em_grow_lackdiet'].map(grad_map)
parent_df['Em_grow_lackdiet'] = parent_df['Em_grow_lackdiet'].astype('category')
extract_information(mens_health_knowledge,'Em_grow_lackdiet')

Em_grow_lackdiet    category
dtype: object
0          Every time
1           Sometimes
2    Most of the time
3                 NaN
4              Rarely
5               Never
dtype: category
Categories (5, object): ['Every time', 'Most of the time', 'Never', 'Rarely', 'Sometimes']
344
6.2409288824383164
Em_grow_lackdiet
Every time          4124
Most of the time     775
Sometimes            222
Rarely                43
Never                  4
Name: count, dtype: int64
       Em_grow_lackdiet
count              5168
unique                5
top          Every time
freq               4124


In [ ]:
extract_information(mens_health_knowledge,'Em_grow_lackdiet_sw')

Em_grow_lackdiet_sw    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
348
6.313497822931785
Em_grow_lackdiet_sw
1.0    4899
0.0     265
Name: count, dtype: int64
       Em_grow_lackdiet_sw
count          5164.000000
mean              0.948683
std               0.220665
min               0.000000
25%               1.000000
50%               1.000000
75%               1.000000
max               1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_grow_lackdiet_vm')


Em_grow_lackdiet_vm    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
348
6.313497822931785
Em_grow_lackdiet_vm
1.0    4124
0.0    1040
Name: count, dtype: int64
       Em_grow_lackdiet_vm
count          5164.000000
mean              0.798606
std               0.401081
min               0.000000
25%               1.000000
50%               1.000000
75%               1.000000
max               1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_grow_witch')

Em_grow_witch    float64
dtype: object
0    3.0
1    1.0
2    4.0
3    NaN
4    2.0
5    5.0
dtype: float64
412
7.474600870827286
Em_grow_witch
1.0    4122
2.0     497
4.0     272
3.0     177
5.0      32
Name: count, dtype: int64
       Em_grow_witch
count    5100.000000
mean        1.351961
std         0.832422
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         5.000000


In [ ]:
witch_map = {
    1: 'Very much',
    2:'Somewhat',
    3:'Not very much',
    4:'Not at all',
    5:'Don’t know',
}

mens_health_knowledge['Em_grow_witch'] = mens_health_knowledge['Em_grow_witch'].map(witch_map)
mens_health_knowledge['Em_grow_witch'] = mens_health_knowledge['Em_grow_witch'].astype('category')

parent_df['Em_grow_witch'] = parent_df['Em_grow_witch'].map(witch_map)
parent_df['Em_grow_witch'] = parent_df['Em_grow_witch'].astype('category')
extract_information(mens_health_knowledge,'Em_grow_witch')

Em_grow_witch    category
dtype: object
0    Not very much
1        Very much
2       Not at all
3              NaN
4         Somewhat
5       Don’t know
dtype: category
Categories (5, object): ['Don’t know', 'Not at all', 'Not very much', 'Somewhat', 'Very much']
412
7.474600870827286
Em_grow_witch
Very much        4122
Somewhat          497
Not at all        272
Not very much     177
Don’t know         32
Name: count, dtype: int64
       Em_grow_witch
count           5100
unique             5
top        Very much
freq            4122


In [ ]:
extract_information(mens_health_knowledge,'Em_grow_witch_lt')


Em_grow_witch_lt    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
444
8.055152394775037
Em_grow_witch_lt
0.0    4619
1.0     449
Name: count, dtype: int64
       Em_grow_witch_lt
count       5068.000000
mean           0.088595
std            0.284186
min            0.000000
25%            0.000000
50%            0.000000
75%            0.000000
max            1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_grow_witch_vl')


Em_grow_witch_vl    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
444
8.055152394775037
Em_grow_witch_vl
0.0    4796
1.0     272
Name: count, dtype: int64
       Em_grow_witch_vl
count       5068.000000
mean           0.053670
std            0.225388
min            0.000000
25%            0.000000
50%            0.000000
75%            0.000000
max            1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_kwash_cause')


Em_kwash_cause    object
dtype: object
0                    5
1                  3 5
2                  4 5
3                    6
4                  5 6
            ...       
165          1 5 6 7 8
166              2 4 6
167            2 3 4 6
168    1 2 3 5 7 other
169        4 5 6 other
Length: 170, dtype: object
336
6.095791001451379
Em_kwash_cause
5                  1401
5 6                 514
5 7                 483
3 5                 408
1 5                 295
                   ... 
1 5 6 7 8             1
2 4 6                 1
2 3 4 6               1
1 2 3 5 7 other       1
4 5 6 other           1
Name: count, Length: 169, dtype: int64
       Em_kwash_cause
count            5176
unique            169
top                 5
freq             1401


In [ ]:
kwash_cause_map = {
    1: 'Lack of food',
    2: 'Cold food',
    3: 'Lack of breastfeeding',
    4: 'Worms',
    5: 'Lack of certain kind of food/balanced diet',
    6: 'Births too close together',
    7: 'Poor hygiene',
    8: 'Witchcraft',
    9: 'Other',
    98: 'Do not know Q.8',
    99: 'No response Q.8',
}
#TODO fix the multiselection list

In [ ]:
extract_information(mens_health_knowledge,'Em_kwash_cause_bi')


Em_kwash_cause_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
336
6.095791001451379
Em_kwash_cause_bi
0.0    3756
1.0    1420
Name: count, dtype: int64
       Em_kwash_cause_bi
count        5176.000000
mean            0.274343
std             0.446226
min             0.000000
25%             0.000000
50%             0.000000
75%             1.000000
max             1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_kwash_cause_oth')


Em_kwash_cause_oth    object
dtype: object
0                                                    NaN
1                                           Lack of milk
2                                      too much sickness
3             When a child breastfeeds a pregnant mother
4                                    Lack of proper care
                             ...                        
174                   Not giving the child meals on time
175    if child keeps on breastfeeding when the mothe...
176                                    Too much coldness
177                               Lack of  immunisation.
178                                              Anaemia
Length: 179, dtype: object
5283
95.84542815674891
Em_kwash_cause_oth
Lack of milk                                                   11
lack of proper care                                             6
Sickness                                                        6
breastfeeding the child while the mother is pregnant           

In [ ]:
extract_information(mens_health_knowledge,'Em_kwash_treat')


Em_kwash_treat    object
dtype: object
0               1
1             1 2
2         1 2 3 5
3             NaN
4       1 2 other
5               2
6             1 3
7           1 2 3
8             2 5
9             1 4
10            1 5
11              7
12          1 2 5
13            2 3
14              3
15        1 other
16           -777
17            2 4
18        2 other
19    1 2 5 other
20      1 5 other
21          1 3 5
22        3 other
23            1 7
24          1 2 4
25    1 2 3 other
26            1 8
27            3 5
28      2 3 other
29        5 other
30          1 5 8
31      1 3 other
32      1 8 other
33      1 4 other
dtype: object
360
6.531204644412192
Em_kwash_treat
1              2443
1 2            1558
1 other         166
2               165
1 3             156
1 2 other       123
1 2 3           109
1 5             104
1 2 5            89
2 3              55
3                50
7                34
-777             28
1 2 3 5          10
1 5 other         

In [ ]:
kwash_treat_map = {
    1: 'Lack of food',
    2: 'Cold food',
    3: 'Lack of breastfeeding',
    4: 'Worms',
    5: 'Lack of certain kind of food/balanced diet',
    6: 'Births too close together',
    7: 'Poor hygiene',
    8: 'Witchcraft',
    9: 'Other',
    98: 'Do not know Q.8',
    99: 'No response Q.8',
    -777: 'other',
}
#TODO fix the multiselection list

In [ ]:
extract_information(mens_health_knowledge,'Em_kwash_treat_oth')


Em_kwash_treat_oth    object
dtype: object
0                                              NaN
1                      Give them milk and soyabean
2                                 Give them fruits
3                        Give them milk and fruits
4      Keep away from the mom when she is pregnant
                          ...                     
251                           Eating meals on time
252                                    Cleanliness
253                             enough breast milk
254            breasting them up to the right time
255    stop breastfeeding kid if birth's are close
Length: 256, dtype: object
5157
93.55950653120463
Em_kwash_treat_oth
Give them milk                                 15
milk                                           11
Avoiding births too close together             11
Breastfeeding                                   5
Family planning                                 5
                                               ..
Eating meals on time       

In [ ]:
extract_information(mens_health_knowledge,'Em_ageliquid')
#TODO think about the unit

Em_ageliquid    float64
dtype: object
0      6.0
1      4.0
2      3.0
3      1.0
4      2.0
5      NaN
6      7.0
7      8.0
8      5.0
9     24.0
10    12.0
11    39.0
12    78.0
13     0.0
14     9.0
15    60.0
16    72.0
17    18.0
18    10.0
19    89.0
20    76.0
21    22.0
22    16.0
23    48.0
dtype: float64
345
6.2590711175616836
Em_ageliquid
6.0     3040
4.0      565
3.0      488
5.0      292
2.0      209
1.0      129
8.0      121
7.0      107
12.0     107
9.0       43
24.0      20
18.0      19
0.0        7
10.0       5
78.0       5
72.0       2
16.0       2
60.0       1
39.0       1
89.0       1
76.0       1
22.0       1
48.0       1
Name: count, dtype: int64
       Em_ageliquid
count   5167.000000
mean       5.637894
std        3.931752
min        0.000000
25%        4.000000
50%        6.000000
75%        6.000000
max       89.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_ageliquid_6')


Em_ageliquid_6    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
345
6.2590711175616836
Em_ageliquid_6
1.0    3040
0.0    2127
Name: count, dtype: int64
       Em_ageliquid_6
count     5167.000000
mean         0.588349
std          0.492180
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_agefood')


Em_agefood    float64
dtype: object
0       6.0
1      12.0
2       NaN
3       7.0
4       8.0
5       5.0
6      72.0
7       9.0
8      18.0
9      24.0
10     10.0
11      4.0
12      1.0
13     14.0
14      3.0
15      2.0
16    104.0
17     60.0
18     30.0
19     78.0
20     80.0
21     15.0
22     84.0
23      0.0
24     91.0
25     36.0
26     42.0
27     16.0
28     96.0
29     32.0
30     20.0
31     48.0
32     17.0
33     90.0
dtype: float64
341
6.186502177068215
Em_agefood
6.0      3010
8.0       524
12.0      504
7.0       296
9.0       222
5.0       182
4.0       118
18.0       72
3.0        60
24.0       55
10.0       36
2.0        15
14.0       12
72.0       10
1.0         8
78.0        6
30.0        6
0.0         6
15.0        4
36.0        4
16.0        3
60.0        2
104.0       2
96.0        2
84.0        2
48.0        2
20.0        2
91.0        1
80.0        1
32.0        1
42.0        1
17.0        1
90.0        1
Name: count, dtype: int64
        Em_agefood
c

In [ ]:
extract_information(mens_health_knowledge,'Em_agefood_6')


Em_agefood_6    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
341
6.186502177068215
Em_agefood_6
1.0    3010
0.0    2161
Name: count, dtype: int64
       Em_agefood_6
count   5171.000000
mean       0.582092
std        0.493262
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_bfeed')

Em_bfeed    float64
dtype: object
0      36.0
1      24.0
2      12.0
3       NaN
4      18.0
5      30.0
6      29.0
7       6.0
8      15.0
9       9.0
10     13.0
11     20.0
12      2.0
13     60.0
14     84.0
15     48.0
16     14.0
17      0.0
18      8.0
19      7.0
20     17.0
21     78.0
22     72.0
23     28.0
24     19.0
25     16.0
26     10.0
27      5.0
28     26.0
29     61.0
30     42.0
31      4.0
32     90.0
33    144.0
34     73.0
35     21.0
36     27.0
37      3.0
38     74.0
39     32.0
40     25.0
41      1.0
42     96.0
dtype: float64
343
6.222786647314949
Em_bfeed
24.0     3353
36.0      451
18.0      446
12.0      298
30.0      295
48.0       49
6.0        28
14.0       24
20.0       23
9.0        21
17.0       19
8.0        18
60.0       17
2.0        15
29.0       13
72.0       12
15.0       11
26.0        8
7.0         7
16.0        5
32.0        4
78.0        4
10.0        4
19.0        4
5.0         4
0.0         4
3.0         4
21.0        3
28.0        

In [ ]:
extract_information(mens_health_knowledge,'Em_bfeed24')

Em_bfeed24    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
343
6.222786647314949
Em_bfeed24
1.0    3353
0.0    1816
Name: count, dtype: int64
        Em_bfeed24
count  5169.000000
mean      0.648675
std       0.477431
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_boil_1min')


Em_boil_1min    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
347
6.295355587808419
Em_boil_1min
0.0    3089
1.0    2076
Name: count, dtype: int64
       Em_boil_1min
count   5165.000000
mean       0.401936
std        0.490337
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_boil_1min_bi')
#TODO this one and the above one are identical but the inverse

Em_boil_1min_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
347
6.295355587808419
Em_boil_1min_bi
1.0    3089
0.0    2076
Name: count, dtype: int64
       Em_boil_1min_bi
count      5165.000000
mean          0.598064
std           0.490337
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_weanfat')

Em_weanfat    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
363
6.585631349782293
Em_weanfat
1.0    4146
0.0    1003
Name: count, dtype: int64
        Em_weanfat
count  5149.000000
mean      0.805205
std       0.396081
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_weanfat_bi')

Em_weanfat_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
363
6.585631349782293
Em_weanfat_bi
0.0    4146
1.0    1003
Name: count, dtype: int64
       Em_weanfat_bi
count    5149.000000
mean        0.194795
std         0.396081
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_porr_thin')

Em_porr_thin    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
370
6.712626995645864
Em_porr_thin
0.0    3112
1.0    2030
Name: count, dtype: int64
       Em_porr_thin
count   5142.000000
mean       0.394788
std        0.488853
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_porr_thin_bi')


Em_porr_thin_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
370
6.712626995645864
Em_porr_thin_bi
1.0    3112
0.0    2030
Name: count, dtype: int64
       Em_porr_thin_bi
count      5142.000000
mean          0.605212
std           0.488853
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_cowmilk')


Em_cowmilk    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
343
6.222786647314949
Em_cowmilk
0.0    4053
1.0    1116
Name: count, dtype: int64
        Em_cowmilk
count  5169.000000
mean      0.215902
std       0.411487
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_cowmilk_bi')

Em_cowmilk_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
343
6.222786647314949
Em_cowmilk_bi
1.0    4053
0.0    1116
Name: count, dtype: int64
       Em_cowmilk_bi
count    5169.000000
mean        0.784098
std         0.411487
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_menfruit')

Em_menfruit    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
354
6.422351233671988
Em_menfruit
1.0    4365
0.0     793
Name: count, dtype: int64
       Em_menfruit
count  5158.000000
mean      0.846258
std       0.360736
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_menfruit_bi')

Em_menfruit_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
354
6.422351233671988
Em_menfruit_bi
0.0    4365
1.0     793
Name: count, dtype: int64
       Em_menfruit_bi
count     5158.000000
mean         0.153742
std          0.360736
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_hiv_bfeed')

Em_hiv_bfeed    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
505
9.161828737300436
Em_hiv_bfeed
1.0    4029
0.0     978
Name: count, dtype: int64
       Em_hiv_bfeed
count   5007.000000
mean       0.804673
std        0.396491
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_hiv_bfeed_bi')


Em_hiv_bfeed_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
505
9.161828737300436
Em_hiv_bfeed_bi
0.0    4029
1.0     978
Name: count, dtype: int64
       Em_hiv_bfeed_bi
count      5007.000000
mean          0.195327
std           0.396491
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_vita_foods_bi')


Em_vita_foods_bi    float64
dtype: object
0    1.0
1    0.0
2    0.5
3    NaN
dtype: float64
2666
48.367198838896954
Em_vita_foods_bi
1.0    1440
0.5    1181
0.0     225
Name: count, dtype: int64
       Em_vita_foods_bi
count       2846.000000
mean           0.713457
std            0.317379
min            0.000000
25%            0.500000
50%            1.000000
75%            1.000000
max            1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_anem_foods_bi')


Em_anem_foods_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
336
6.095791001451379
Em_anem_foods_bi
1.0    3040
0.0    2136
Name: count, dtype: int64
       Em_anem_foods_bi
count       5176.000000
mean           0.587326
std            0.492363
min            0.000000
25%            0.000000
50%            1.000000
75%            1.000000
max            1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_food_plate')

Em_food_plate    float64
dtype: object
0    2.0
1    1.0
2    NaN
dtype: float64
355
6.4404934687953554
Em_food_plate
1.0    4674
2.0     483
Name: count, dtype: int64
       Em_food_plate
count    5157.000000
mean        1.093659
std         0.291382
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000


In [ ]:
food_plate_map={
    1:"Plate 1",
    2:"Plate 2"
}
mens_health_knowledge['Em_food_plate'] = mens_health_knowledge['Em_food_plate'].map(food_plate_map)
mens_health_knowledge['Em_food_plate'] = mens_health_knowledge['Em_food_plate'].astype('category')
parent_df['Em_food_plate'] = parent_df['Em_food_plate'].map(food_plate_map)
parent_df['Em_food_plate'] = parent_df['Em_food_plate'].astype('category')
extract_information(mens_health_knowledge,'Em_food_plate')

Em_food_plate    category
dtype: object
0    Plate 2
1    Plate 1
2        NaN
dtype: category
Categories (2, object): ['Plate 1', 'Plate 2']
355
6.4404934687953554
Em_food_plate
Plate 1    4674
Plate 2     483
Name: count, dtype: int64
       Em_food_plate
count           5157
unique             2
top          Plate 1
freq            4674


In [ ]:
extract_information(mens_health_knowledge,'Em_food_plate_bi')

Em_food_plate_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
336
6.095791001451379
Em_food_plate_bi
1.0    4674
0.0     502
Name: count, dtype: int64
       Em_food_plate_bi
count       5176.000000
mean           0.903014
std            0.295967
min            0.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_child_worms')

Em_child_worms    float64
dtype: object
0    3.0
1    1.0
2    2.0
3    NaN
4    4.0
dtype: float64
350
6.34978229317852
Em_child_worms
3.0    3316
1.0    1487
2.0     344
4.0      15
Name: count, dtype: int64
       Em_child_worms
count     5162.000000
mean         2.360132
std          0.901410
min          1.000000
25%          1.000000
50%          3.000000
75%          3.000000
max          4.000000


In [ ]:
food_plate_map={
    1:"Anemia",
    2:"Malaria",
    3:"Both",
    4:"Neither",
    998:"Don't know",
}
mens_health_knowledge['Em_child_worms'] = mens_health_knowledge['Em_child_worms'].map(food_plate_map)
mens_health_knowledge['Em_child_worms'] = mens_health_knowledge['Em_child_worms'].astype('category')
parent_df['Em_child_worms'] = parent_df['Em_child_worms'].map(food_plate_map)
parent_df['Em_child_worms'] = parent_df['Em_child_worms'].astype('category')
extract_information(mens_health_knowledge,'Em_child_worms')

Em_child_worms    category
dtype: object
0       Both
1     Anemia
2    Malaria
3        NaN
4    Neither
dtype: category
Categories (4, object): ['Anemia', 'Both', 'Malaria', 'Neither']
350
6.34978229317852
Em_child_worms
Both       3316
Anemia     1487
Malaria     344
Neither      15
Name: count, dtype: int64
       Em_child_worms
count            5162
unique              4
top              Both
freq             3316


In [ ]:
extract_information(mens_health_knowledge,'Em_child_worms_bi')

Em_child_worms_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
336
6.095791001451379
Em_child_worms_bi
1.0    3316
0.0    1860
Name: count, dtype: int64
       Em_child_worms_bi
count        5176.000000
mean            0.640649
std             0.479857
min             0.000000
25%             0.000000
50%             1.000000
75%             1.000000
max             1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_condom_times')

Em_condom_times    float64
dtype: object
0     1.0
1     NaN
2     2.0
3     5.0
4    11.0
5     3.0
6     4.0
dtype: float64
383
6.948476052249637
Em_condom_times
1.0     5090
2.0       32
3.0        3
5.0        2
11.0       1
4.0        1
Name: count, dtype: int64
       Em_condom_times
count      5129.000000
mean          1.011503
std           0.189589
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          11.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_condom_times_bi')

Em_condom_times_bi    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
336
6.095791001451379
Em_condom_times_bi
1.0    5090
0.0      86
Name: count, dtype: int64
       Em_condom_times_bi
count         5176.000000
mean             0.983385
std              0.127837
min              0.000000
25%              1.000000
50%              1.000000
75%              1.000000
max              1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_child_vomit')

Em_child_vomit    object
dtype: object
0               1 3
1               1 2
2                 1
3               2 3
4                 5
5               1 4
6               2 5
7               NaN
8             2 3 5
9             1 2 4
10              1 5
11            1 2 3
12             -777
13                2
14            1 3 4
15                3
16            1 2 5
17          2 other
18          1 2 4 5
19              2 4
20              3 4
21            1 4 5
22          1 3 4 5
23        2 3 other
24          5 other
25          1 other
26              3 5
27          1 2 3 5
28        1 2 other
29          4 other
30            1 3 5
31        1 3 other
32              4 5
33                4
34        1 5 other
35        2 4 other
36          1 2 3 4
37        1 4 other
38          3 other
39            2 3 4
40      1 2 3 other
41        3 4 other
42            3 4 5
43        1 2 3 4 5
44      1 2 5 other
45        2 5 other
46        3 5 other
47        4 5 other
4

In [ ]:
vomit_map={
    1 : 'Medicine from a health center',
    2 : 'Oral rehydration salts',
    3 : 'Clean water',
    4 : 'More food',
    5 : 'Herbs',
    998 : 'other',
} #TODO fix the multiselection list

In [ ]:
extract_information(mens_health_knowledge,'Em_child_vomit_oth')
#TODO normalize this like: an egg and Eggs

Em_child_vomit_oth    object
dtype: object
0                                                 NaN
1                                        Eggs,cassava
2                                                Zinc
3                                   Giving a kid milk
4      lGive the child more water and sugar solution.
                            ...                      
359              Give the child vegetables and fruits
360                                       boiled eggs
361                              Give him solid foods
362                            Give the kid black tea
363                     Give the child abalanced diet
Length: 364, dtype: object
4901
88.91509433962264
Em_child_vomit_oth
Milk                              33
Fruits                            17
Eggs                              15
Porridge                          14
Millet porridge                   12
                                  ..
Give food with ghee                1
Millet flour mixed with sugar      

In [ ]:
extract_information(mens_health_knowledge,'Em_fteeth')

Em_fteeth    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
344
6.2409288824383164
Em_fteeth
1.0    5137
0.0      31
Name: count, dtype: int64
         Em_fteeth
count  5168.000000
mean      0.994002
std       0.077225
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(mens_health_knowledge,'Em_fteeth_cause')

Em_fteeth_cause    float64
dtype: object
0    777.0
1      NaN
2      1.0
3      4.0
4      2.0
5      3.0
dtype: float64
3269
59.30696661828737
Em_fteeth_cause
1.0      1462
4.0       288
777.0     261
2.0       140
3.0        92
Name: count, dtype: int64
       Em_fteeth_cause
count      2243.000000
mean         91.826572
std         248.696408
min           1.000000
25%           1.000000
50%           1.000000
75%           3.000000
max         777.000000


In [ ]:
fteeth_cause_map={
    1:"unknown (non-specific) illness",
    2:"infection (in the teeth or elsewhere)",
    3:"the tooth itself",
    4:"don’t believe false teeth is a real disease ",
    777:"other"
}
mens_health_knowledge['Em_fteeth_cause'] = mens_health_knowledge['Em_fteeth_cause'].map(fteeth_cause_map)
mens_health_knowledge['Em_fteeth_cause'] = mens_health_knowledge['Em_fteeth_cause'].astype('category')
parent_df['Em_fteeth_cause'] = parent_df['Em_fteeth_cause'].map(fteeth_cause_map)
parent_df['Em_fteeth_cause'] = parent_df['Em_fteeth_cause'].astype('category')
extract_information(mens_health_knowledge,'Em_fteeth_cause')

Em_fteeth_cause    category
dtype: object
0                                           other
1                                             NaN
2                  unknown (non-specific) illness
3    don’t believe false teeth is a real disease 
4           infection (in the teeth or elsewhere)
5                                the tooth itself
dtype: category
Categories (5, object): ['don’t believe false teeth is a real disease ', 'infection (in the teeth or elsewhere)', 'other',
                         'the tooth itself', 'unknown (non-specific) illness']
3269
59.30696661828737
Em_fteeth_cause
unknown (non-specific) illness                  1462
don’t believe false teeth is a real disease      288
other                                            261
infection (in the teeth or elsewhere)            140
the tooth itself                                  92
Name: count, dtype: int64
                       Em_fteeth_cause
count                             2243
unique                          

In [ ]:
extract_information(mens_health_knowledge,'Em_fteeth_cause_oth')

Em_fteeth_cause_oth    object
dtype: object
0                                           Poor feeding
1                                                    NaN
2                    When one is suffering from diarrhea
3                                                Malaria
4                                   Child is born wth it
                             ...                        
155         If the child was not immunised against polio
156    the foods that the mother of the child eats wh...
157                                            Allergies
158                                      In born disease
159                          The child is born with them
Length: 160, dtype: object
5251
95.26487663280116
Em_fteeth_cause_oth
Witchcraft                                                          24
Malaria                                                             17
Poor feeding                                                        11
Poor hygiene                                  

In [ ]:
extract_information(mens_health_knowledge,'Em_child_fteeth')


Em_child_fteeth    float64
dtype: object
0      3.0
1      1.0
2      2.0
3      NaN
4    777.0
5      4.0
dtype: float64
568
10.304789550072568
Em_child_fteeth
2.0      3305
1.0      1253
3.0       313
777.0      48
4.0        25
Name: count, dtype: int64
       Em_child_fteeth
count      4944.000000
mean          9.344256
std          76.018935
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max         777.000000


In [ ]:
child_fteeth_map={
    1:"take the to a health center for treatment",
    2:"take them to the local person who removes false teeth",
    3:"take them to another health facility (e.g. hospital)",
    4:"do nothing",
    777:"other",
    998:"don’t know",
}
mens_health_knowledge['Em_child_fteeth'] = mens_health_knowledge['Em_child_fteeth'].map(child_fteeth_map)
mens_health_knowledge['Em_child_fteeth'] = mens_health_knowledge['Em_child_fteeth'].astype('category')
parent_df['Em_child_fteeth'] = parent_df['Em_child_fteeth'].map(child_fteeth_map)
parent_df['Em_child_fteeth'] = parent_df['Em_child_fteeth'].astype('category')
extract_information(mens_health_knowledge,'Em_child_fteeth')

Em_child_fteeth    category
dtype: object
0    take them to another health facility (e.g. hos...
1            take the to a health center for treatment
2    take them to the local person who removes fals...
3                                                  NaN
4                                                other
5                                           do nothing
dtype: category
Categories (5, object): ['do nothing', 'other', 'take the to a health center for treatment',
                         'take them to another health facility (e.g. ho..., 'take them to the local person who removes fal...]
568
10.304789550072568
Em_child_fteeth
take them to the local person who removes false teeth    3305
take the to a health center for treatment                1253
take them to another health facility (e.g. hospital)      313
other                                                      48
do nothing                                                 25
Name: count, dtype: int64
                 

In [ ]:
extract_information(mens_health_knowledge,'Em_child_fteeth_oth')


Em_child_fteeth_oth    object
dtype: object
0                                                   NaN
1         Take them to church for prayers to get healed
2                     Apply honey and sugar to the jaws
3                      Take her or him to be prayed for
4                                                  Pray
5                                     Pray for him /her
6                     take the child to people who pray
7                          Provide enough balanced diet
8                     Clean or brush the child's teeth.
9                                      Give local herbs
10                                   Pray for the child
11         Contact soneome that knows about the disease
12                                           Bamusirike
13                                              Praying
14                                         Good feeding
15                            Give the child some herbs
16                                          Change food
17  

In [ ]:
extract_information(mens_health_knowledge,'Em_discuss_spouse')

Em_discuss_spouse    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    NaN
4    4.0
dtype: float64
336
6.095791001451379
Em_discuss_spouse
1.0    4138
2.0     725
3.0     244
4.0      69
Name: count, dtype: int64
       Em_discuss_spouse
count        5176.000000
mean            1.274343
std             0.611079
min             1.000000
25%             1.000000
50%             1.000000
75%             1.000000
max             4.000000


In [ ]:
discuss_spouse_map={
    1:"Very often",
    2:"Sometimes",
    3:"Rarely",
    4:"Never",
}
mens_health_knowledge['Em_discuss_spouse'] = mens_health_knowledge['Em_discuss_spouse'].map(discuss_spouse_map)
mens_health_knowledge['Em_discuss_spouse'] = mens_health_knowledge['Em_discuss_spouse'].astype('category')
parent_df['Em_discuss_spouse'] = parent_df['Em_discuss_spouse'].map(discuss_spouse_map)
parent_df['Em_discuss_spouse'] = parent_df['Em_discuss_spouse'].astype('category')
extract_information(mens_health_knowledge,'Em_discuss_spouse')

Em_discuss_spouse    category
dtype: object
0    Very often
1     Sometimes
2        Rarely
3           NaN
4         Never
dtype: category
Categories (4, object): ['Never', 'Rarely', 'Sometimes', 'Very often']
336
6.095791001451379
Em_discuss_spouse
Very often    4138
Sometimes      725
Rarely         244
Never           69
Name: count, dtype: int64
       Em_discuss_spouse
count               5176
unique                 4
top           Very often
freq                4138


In [ ]:
extract_information(mens_health_knowledge,'Em_discuss_fam')


Em_discuss_fam    float64
dtype: object
0    1.0
1    3.0
2    2.0
3    NaN
4    4.0
dtype: float64
336
6.095791001451379
Em_discuss_fam
1.0    1477
3.0    1434
2.0    1399
4.0     866
Name: count, dtype: int64
       Em_discuss_fam
count     5176.000000
mean         2.326314
std          1.060841
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          4.000000


In [ ]:
mens_health_knowledge['Em_discuss_fam'] = mens_health_knowledge['Em_discuss_fam'].map(discuss_spouse_map)
mens_health_knowledge['Em_discuss_fam'] = mens_health_knowledge['Em_discuss_fam'].astype('category')
parent_df['Em_discuss_fam'] = parent_df['Em_discuss_fam'].map(discuss_spouse_map)
parent_df['Em_discuss_fam'] = parent_df['Em_discuss_fam'].astype('category')
extract_information(mens_health_knowledge,'Em_discuss_fam')

Em_discuss_fam    category
dtype: object
0    Very often
1        Rarely
2     Sometimes
3           NaN
4         Never
dtype: category
Categories (4, object): ['Never', 'Rarely', 'Sometimes', 'Very often']
336
6.095791001451379
Em_discuss_fam
Very often    1477
Rarely        1434
Sometimes     1399
Never          866
Name: count, dtype: int64
       Em_discuss_fam
count            5176
unique              4
top        Very often
freq             1477


In [ ]:
extract_information(mens_health_knowledge,'Em_husb_sugg_food')

Em_husb_sugg_food    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    4.0
4    NaN
dtype: float64
336
6.095791001451379
Em_husb_sugg_food
1.0    3147
2.0    1259
3.0     698
4.0      72
Name: count, dtype: int64
       Em_husb_sugg_food
count        5176.000000
mean            1.554675
std             0.774787
min             1.000000
25%             1.000000
50%             1.000000
75%             2.000000
max             4.000000


In [ ]:
mens_health_knowledge['Em_husb_sugg_food'] = mens_health_knowledge['Em_husb_sugg_food'].map(discuss_spouse_map)
mens_health_knowledge['Em_husb_sugg_food'] = mens_health_knowledge['Em_husb_sugg_food'].astype('category')
parent_df['Em_husb_sugg_food'] = parent_df['Em_husb_sugg_food'].map(discuss_spouse_map)
parent_df['Em_husb_sugg_food'] = parent_df['Em_husb_sugg_food'].astype('category')
extract_information(mens_health_knowledge,'Em_husb_sugg_food')

Em_husb_sugg_food    category
dtype: object
0    Very often
1     Sometimes
2        Rarely
3         Never
4           NaN
dtype: category
Categories (4, object): ['Never', 'Rarely', 'Sometimes', 'Very often']
336
6.095791001451379
Em_husb_sugg_food
Very often    3147
Sometimes     1259
Rarely         698
Never           72
Name: count, dtype: int64
       Em_husb_sugg_food
count               5176
unique                 4
top           Very often
freq                3147


In [ ]:
extract_information(mens_health_knowledge,'Em_husb_sugg_hc')

Em_husb_sugg_hc    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    NaN
4    4.0
dtype: float64
336
6.095791001451379
Em_husb_sugg_hc
1.0    4040
2.0     781
3.0     311
4.0      44
Name: count, dtype: int64
       Em_husb_sugg_hc
count      5176.000000
mean          1.296561
std           0.616328
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           4.000000


In [ ]:
mens_health_knowledge['Em_husb_sugg_hc'] = mens_health_knowledge['Em_husb_sugg_hc'].map(discuss_spouse_map)
mens_health_knowledge['Em_husb_sugg_hc'] = mens_health_knowledge['Em_husb_sugg_hc'].astype('category')
parent_df['Em_husb_sugg_hc'] = parent_df['Em_husb_sugg_hc'].map(discuss_spouse_map)
parent_df['Em_husb_sugg_hc'] = parent_df['Em_husb_sugg_hc'].astype('category')
extract_information(mens_health_knowledge,'Em_husb_sugg_hc')

Em_husb_sugg_hc    category
dtype: object
0    Very often
1     Sometimes
2        Rarely
3           NaN
4         Never
dtype: category
Categories (4, object): ['Never', 'Rarely', 'Sometimes', 'Very often']
336
6.095791001451379
Em_husb_sugg_hc
Very often    4040
Sometimes      781
Rarely         311
Never           44
Name: count, dtype: int64
       Em_husb_sugg_hc
count             5176
unique               4
top         Very often
freq              4040


In [ ]:
extract_information(mens_health_knowledge,'Em_who_priority_hc')


Em_who_priority_hc    float64
dtype: object
0    2.0
1    1.0
2    NaN
3    3.0
dtype: float64
336
6.095791001451379
Em_who_priority_hc
1.0    3208
2.0    1613
3.0     355
Name: count, dtype: int64
       Em_who_priority_hc
count         5176.000000
mean             1.448802
std              0.620181
min              1.000000
25%              1.000000
50%              1.000000
75%              2.000000
max              3.000000


In [ ]:
priority_map={
    1 : 'I do',
    2 : 'We both put equal priority',
    3 : 'Wife does',
}
mens_health_knowledge['Em_who_priority_hc'] = mens_health_knowledge['Em_who_priority_hc'].map(priority_map)
mens_health_knowledge['Em_who_priority_hc'] = mens_health_knowledge['Em_who_priority_hc'].astype('category')
parent_df['Em_who_priority_hc'] = parent_df['Em_who_priority_hc'].map(priority_map)
parent_df['Em_who_priority_hc'] = parent_df['Em_who_priority_hc'].astype('category')
extract_information(mens_health_knowledge,'Em_who_priority_hc')

Em_who_priority_hc    category
dtype: object
0    We both put equal priority
1                          I do
2                           NaN
3                     Wife does
dtype: category
Categories (3, object): ['I do', 'We both put equal priority', 'Wife does']
336
6.095791001451379
Em_who_priority_hc
I do                          3208
We both put equal priority    1613
Wife does                      355
Name: count, dtype: int64
       Em_who_priority_hc
count                5176
unique                  3
top                  I do
freq                 3208


### Mens Decison Making EL Section

In [ ]:
mens_decision_making_el = parent_all_sections['MENS_DECISION_MAKING_EL']
mens_decision_making_el.columns

Index(['Em_fam_contact', 'Em_fem_friends', 'Em_daily_bi', 'Em_daily_who',
       'Em_daily_diffop', 'Em_daily_express', 'Em_daily_argue',
       'Em_daily_prevail', 'Em_chealth_bi', 'Em_chealth_who',
       'Em_chealth_diffop', 'Em_chealth_express', 'Em_chealth_argue',
       'Em_chealth_prevail', 'Em_cfeed_bi', 'Em_cfeed_who', 'Em_cfeed_diffop',
       'Em_cfeed_express', 'Em_cfeed_argue', 'Em_cfeed_prevail',
       'Em_wmoney_bi', 'Em_wmoney_who', 'Em_wmoney_diffop',
       'Em_wmoney_express', 'Em_wmoney_argue', 'Em_wmoney_prevail',
       'Em_quarrel', 'Em_rights', 'Em_rights_sdi', 'Em_rights_di',
       'Em_equality', 'Em_equality_sdi', 'Em_equality_di', 'Em_obey',
       'Em_obey_sdi', 'Em_obey_di', 'Em_boss', 'Em_boss_sdi', 'Em_boss_di',
       'Em_leaders', 'Em_leaders_sdi', 'Em_leaders_di', 'Em_shrpow',
       'Em_shrpow_sag', 'Em_shrpow_ag', 'Em_opin', 'Em_opin_sag', 'Em_opin_ag',
       'Em_final_say', 'Em_final_say_sag', 'Em_final_say_ag', 'Em_oblig_sex',
       'Em_oblig_s

In [ ]:
extract_information(mens_decision_making_el,'Em_fam_contact')

Em_fam_contact    float64
dtype: object
0    3.0
1    2.0
2    4.0
3    NaN
4    1.0
5    5.0
dtype: float64
476
8.635703918722786
Em_fam_contact
4.0    1736
3.0    1590
2.0    1194
5.0     322
1.0     194
Name: count, dtype: int64
       Em_fam_contact
count     5036.000000
mean         3.158459
std          0.983231
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000


In [ ]:
fam_contact_map= {
    1 : 'More than once a week',
    2 : 'Once or more every month',
    3 : 'Once or more every 6 months',
    4 : 'Once or more every year',
    5 : 'Less than once a year ',
}
mapping_func(mens_decision_making_el,parent_df,'Em_fam_contact',fam_contact_map)

Em_fam_contact    category
dtype: object
0    Once or more every 6 months
1       Once or more every month
2        Once or more every year
3                            NaN
4          More than once a week
5         Less than once a year 
dtype: category
Categories (5, object): ['Less than once a year ', 'More than once a week', 'Once or more every 6 months',
                         'Once or more every month', 'Once or more every year']
476
8.635703918722786
Em_fam_contact
Once or more every year        1736
Once or more every 6 months    1590
Once or more every month       1194
Less than once a year           322
More than once a week           194
Name: count, dtype: int64
                 Em_fam_contact
count                      5036
unique                        5
top     Once or more every year
freq                       1736


In [ ]:
extract_information(mens_decision_making_el,'Em_fem_friends')

Em_fem_friends    float64
dtype: object
0    2.0
1    3.0
2    6.0
3    1.0
4    NaN
5    4.0
6    5.0
dtype: float64
2757
50.01814223512336
Em_fem_friends
2.0    1456
3.0     775
1.0     319
6.0      97
4.0      92
5.0      16
Name: count, dtype: int64
       Em_fem_friends
count     2755.000000
mean         2.390563
std          0.997046
min          1.000000
25%          2.000000
50%          2.000000
75%          3.000000
max          6.000000


In [ ]:
fam_friends_map= {
    1 : 'Less than once a week',
    2 : 'Once per week',
    3 : '2-3 times per week',
    4 : '3-4 times per weeks',
    5 : '5-6 times per week',
    6 : 'Every day',
}
mapping_func(mens_decision_making_el,parent_df,'Em_fem_friends',fam_friends_map)

Em_fem_friends    category
dtype: object
0            Once per week
1       2-3 times per week
2                Every day
3    Less than once a week
4                      NaN
5      3-4 times per weeks
6       5-6 times per week
dtype: category
Categories (6, object): ['2-3 times per week', '3-4 times per weeks', '5-6 times per week', 'Every day',
                         'Less than once a week', 'Once per week']
2757
50.01814223512336
Em_fem_friends
Once per week            1456
2-3 times per week        775
Less than once a week     319
Every day                  97
3-4 times per weeks        92
5-6 times per week         16
Name: count, dtype: int64
       Em_fem_friends
count            2755
unique              6
top     Once per week
freq             1456


In [ ]:
extract_information(mens_decision_making_el,'Em_daily_who')

Em_daily_who    float64
dtype: object
0    3.0
1    1.0
2    2.0
3    NaN
4    6.0
5    4.0
6    5.0
dtype: float64
336
6.095791001451379
Em_daily_who
2.0    3036
3.0    1541
1.0     584
6.0       9
4.0       3
5.0       3
Name: count, dtype: int64
       Em_daily_who
count   5176.000000
mean       2.194745
std        0.638794
min        1.000000
25%        2.000000
50%        2.000000
75%        3.000000
max        6.000000


In [ ]:
daily_who_map= {
    1 : 'Wife',
    2 : 'Husband',
    3 : 'Both together',
    4 : 'Other HH member',
    5 : 'Someone outside the HH',
    6 : 'Not applicable'
}
mapping_func(mens_decision_making_el,parent_df,'Em_daily_who',daily_who_map)

Em_daily_who    category
dtype: object
0             Both together
1                      Wife
2                   Husband
3                       NaN
4            Not applicable
5           Other HH member
6    Someone outside the HH
dtype: category
Categories (6, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member',
                         'Someone outside the HH', 'Wife']
336
6.095791001451379
Em_daily_who
Husband                   3036
Both together             1541
Wife                       584
Not applicable               9
Other HH member              3
Someone outside the HH       3
Name: count, dtype: int64
       Em_daily_who
count          5176
unique            6
top         Husband
freq           3036


In [ ]:
extract_information(mens_decision_making_el,'Em_daily_bi')

Em_daily_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
345
6.2590711175616836
Em_daily_bi
0.0    3042
1.0    2125
Name: count, dtype: int64
       Em_daily_bi
count  5167.000000
mean      0.411264
std       0.492111
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_daily_prevail')

Em_daily_prevail    float64
dtype: object
0    NaN
1    1.0
2    2.0
3    4.0
4    3.0
dtype: float64
4930
89.4412191582003
Em_daily_prevail
1.0    424
2.0     97
4.0     59
3.0      2
Name: count, dtype: int64
       Em_daily_prevail
count        582.000000
mean           1.477663
std            0.930650
min            1.000000
25%            1.000000
50%            1.000000
75%            2.000000
max            4.000000


In [ ]:
daily_prevail_map= {
    1 : 'Yourself',
    2 : 'Your spouse',
    3 : 'A third person',
    4 : 'Respondent finds another way',
}
mapping_func(mens_decision_making_el,parent_df,'Em_daily_prevail',daily_prevail_map)

Em_daily_prevail    category
dtype: object
0                             NaN
1                        Yourself
2                     Your spouse
3    Respondent finds another way
4                  A third person
dtype: category
Categories (4, object): ['A third person', 'Respondent finds another way', 'Your spouse', 'Yourself']
4930
89.4412191582003
Em_daily_prevail
Yourself                        424
Your spouse                      97
Respondent finds another way     59
A third person                    2
Name: count, dtype: int64
       Em_daily_prevail
count               582
unique                4
top            Yourself
freq                424


In [ ]:
extract_information(mens_decision_making_el,'Em_chealth_bi')

Em_chealth_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
354
6.422351233671988
Em_chealth_bi
0.0    3544
1.0    1614
Name: count, dtype: int64
       Em_chealth_bi
count    5158.000000
mean        0.312912
std         0.463724
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_chealth_who')

Em_chealth_who    float64
dtype: object
0    3.0
1    2.0
2    NaN
3    1.0
4    6.0
5    5.0
dtype: float64
336
6.095791001451379
Em_chealth_who
2.0    3540
3.0    1480
1.0     134
6.0      18
5.0       4
Name: count, dtype: int64
       Em_chealth_who
count     5176.000000
mean         2.276275
std          0.546031
min          1.000000
25%          2.000000
50%          2.000000
75%          3.000000
max          6.000000


In [ ]:
chealth_who_map= {
    1 : 'Wife',
    2 : 'Husband',
    3 : 'Both together',
    4 : 'Other HH member',
    5 : "Someone outside the HH",
    6 : "Not applicable"
}
mapping_func(mens_decision_making_el,parent_df,'Em_chealth_who',chealth_who_map)

Em_chealth_who    category
dtype: object
0             Both together
1                   Husband
2                       NaN
3                      Wife
4            Not applicable
5    Someone outside the HH
dtype: category
Categories (5, object): ['Both together', 'Husband', 'Not applicable', 'Someone outside the HH',
                         'Wife']
336
6.095791001451379
Em_chealth_who
Husband                   3540
Both together             1480
Wife                       134
Not applicable              18
Someone outside the HH       4
Name: count, dtype: int64
       Em_chealth_who
count            5176
unique              5
top           Husband
freq             3540


In [ ]:
extract_information(mens_decision_making_el,'Em_chealth_diffop')

Em_chealth_diffop    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
3876
70.31930333817125
Em_chealth_diffop
0.0    1516
1.0     120
Name: count, dtype: int64
       Em_chealth_diffop
count        1636.000000
mean            0.073350
std             0.260789
min             0.000000
25%             0.000000
50%             0.000000
75%             0.000000
max             1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_chealth_express')

Em_chealth_express    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
1852
33.599419448476056
Em_chealth_express
0.0    3336
1.0     324
Name: count, dtype: int64
       Em_chealth_express
count         3660.000000
mean             0.088525
std              0.284095
min              0.000000
25%              0.000000
50%              0.000000
75%              0.000000
max              1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_chealth_argue')

Em_chealth_argue    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
5188
94.12191582002902
Em_chealth_argue
1.0    227
0.0     97
Name: count, dtype: int64
       Em_chealth_argue
count        324.000000
mean           0.700617
std            0.458696
min            0.000000
25%            0.000000
50%            1.000000
75%            1.000000
max            1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_chealth_prevail')

Em_chealth_prevail    float64
dtype: object
0    NaN
1    1.0
2    4.0
3    2.0
dtype: float64
5285
95.88171262699564
Em_chealth_prevail
1.0    169
2.0     45
4.0     13
Name: count, dtype: int64
       Em_chealth_prevail
count          227.000000
mean             1.370044
std              0.761102
min              1.000000
25%              1.000000
50%              1.000000
75%              2.000000
max              4.000000


In [ ]:
mapping_func(mens_decision_making_el,parent_df,'Em_chealth_prevail',daily_prevail_map)

Em_chealth_prevail    category
dtype: object
0                             NaN
1                        Yourself
2    Respondent finds another way
3                     Your spouse
dtype: category
Categories (3, object): ['Respondent finds another way', 'Your spouse', 'Yourself']
5285
95.88171262699564
Em_chealth_prevail
Yourself                        169
Your spouse                      45
Respondent finds another way     13
Name: count, dtype: int64
       Em_chealth_prevail
count                 227
unique                  3
top              Yourself
freq                  169


In [ ]:
extract_information(mens_decision_making_el,'Em_cfeed_who')

Em_cfeed_who    float64
dtype: object
0    1.0
1    3.0
2    NaN
3    2.0
4    6.0
5    4.0
6    5.0
dtype: float64
336
6.095791001451379
Em_cfeed_who
1.0    4258
3.0     612
2.0     258
6.0      24
4.0      15
5.0       9
Name: count, dtype: int64
       Em_cfeed_who
count   5176.000000
mean       1.325155
std        0.766164
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        6.000000


In [ ]:
mapping_func(mens_decision_making_el,parent_df,'Em_cfeed_who',chealth_who_map)

Em_cfeed_who    category
dtype: object
0                      Wife
1             Both together
2                       NaN
3                   Husband
4            Not applicable
5           Other HH member
6    Someone outside the HH
dtype: category
Categories (6, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member',
                         'Someone outside the HH', 'Wife']
336
6.095791001451379
Em_cfeed_who
Wife                      4258
Both together              612
Husband                    258
Not applicable              24
Other HH member             15
Someone outside the HH       9
Name: count, dtype: int64
       Em_cfeed_who
count          5176
unique            6
top            Wife
freq           4258


In [ ]:
extract_information(mens_decision_making_el,'Em_cfeed_bi')

Em_cfeed_bi    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
360
6.531204644412192
Em_cfeed_bi
1.0    4870
0.0     282
Name: count, dtype: int64
       Em_cfeed_bi
count  5152.000000
mean      0.945264
std       0.227486
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_cfeed_prevail')

Em_cfeed_prevail    float64
dtype: object
0    NaN
1    1.0
2    4.0
3    2.0
dtype: float64
5395
97.87735849056604
Em_cfeed_prevail
1.0    66
2.0    45
4.0     6
Name: count, dtype: int64
       Em_cfeed_prevail
count        117.000000
mean           1.538462
std            0.749005
min            1.000000
25%            1.000000
50%            1.000000
75%            2.000000
max            4.000000


In [ ]:
mapping_func(mens_decision_making_el,parent_df,'Em_cfeed_prevail',daily_prevail_map)

Em_cfeed_prevail    category
dtype: object
0                             NaN
1                        Yourself
2    Respondent finds another way
3                     Your spouse
dtype: category
Categories (3, object): ['Respondent finds another way', 'Your spouse', 'Yourself']
5395
97.87735849056604
Em_cfeed_prevail
Yourself                        66
Your spouse                     45
Respondent finds another way     6
Name: count, dtype: int64
       Em_cfeed_prevail
count               117
unique                3
top            Yourself
freq                 66


In [ ]:
extract_information(mens_decision_making_el,'Em_cfeed_bi')

Em_cfeed_bi    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
360
6.531204644412192
Em_cfeed_bi
1.0    4870
0.0     282
Name: count, dtype: int64
       Em_cfeed_bi
count  5152.000000
mean      0.945264
std       0.227486
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_wmoney_who')

Em_wmoney_who    float64
dtype: object
0    3.0
1    1.0
2    NaN
3    2.0
4    6.0
dtype: float64
336
6.095791001451379
Em_wmoney_who
1.0    3103
3.0    1644
2.0     362
6.0      67
Name: count, dtype: int64
       Em_wmoney_who
count    5176.000000
mean        1.769900
std         1.035127
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max         6.000000


In [ ]:
mapping_func(mens_decision_making_el,parent_df,'Em_wmoney_who',chealth_who_map)

Em_wmoney_who    category
dtype: object
0     Both together
1              Wife
2               NaN
3           Husband
4    Not applicable
dtype: category
Categories (4, object): ['Both together', 'Husband', 'Not applicable', 'Wife']
336
6.095791001451379
Em_wmoney_who
Wife              3103
Both together     1644
Husband            362
Not applicable      67
Name: count, dtype: int64
       Em_wmoney_who
count           5176
unique             4
top             Wife
freq            3103


In [ ]:
extract_information(mens_decision_making_el,'Em_wmoney_bi')

Em_wmoney_bi    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
403
7.311320754716981
Em_wmoney_bi
1.0    4747
0.0     362
Name: count, dtype: int64
       Em_wmoney_bi
count   5109.000000
mean       0.929145
std        0.256608
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_wmoney_prevail')

Em_wmoney_prevail    float64
dtype: object
0    NaN
1    4.0
2    1.0
3    2.0
dtype: float64
5137
93.1966618287373
Em_wmoney_prevail
1.0    191
2.0    134
4.0     50
Name: count, dtype: int64
       Em_wmoney_prevail
count         375.000000
mean            1.757333
std             0.993182
min             1.000000
25%             1.000000
50%             1.000000
75%             2.000000
max             4.000000


In [ ]:
mapping_func(mens_decision_making_el,parent_df,'Em_wmoney_prevail',daily_prevail_map)

Em_wmoney_prevail    category
dtype: object
0                             NaN
1    Respondent finds another way
2                        Yourself
3                     Your spouse
dtype: category
Categories (3, object): ['Respondent finds another way', 'Your spouse', 'Yourself']
5137
93.1966618287373
Em_wmoney_prevail
Yourself                        191
Your spouse                     134
Respondent finds another way     50
Name: count, dtype: int64
       Em_wmoney_prevail
count                375
unique                 3
top             Yourself
freq                 191


In [ ]:
extract_information(mens_decision_making_el,'Em_quarrel')

Em_quarrel    float64
dtype: object
0    6.0
1    4.0
2    3.0
3    5.0
4    NaN
5    2.0
6    1.0
dtype: float64
361
6.549346879535559
Em_quarrel
4.0    1515
5.0    1460
3.0     883
6.0     815
2.0     370
1.0     108
Name: count, dtype: int64
        Em_quarrel
count  5151.000000
mean      4.221899
std       1.230784
min       1.000000
25%       3.000000
50%       4.000000
75%       5.000000
max       6.000000


In [ ]:
last_argu = {
    1 : 'in the past day',
    2 : 'in the past week',
    3 : 'in the past month',
    4 : 'in the past year',
    5 : 'more than one year ago',
    6 :  'never',
    998 : 'don’t know'
}
mapping_func(mens_decision_making_el,parent_df,'Em_quarrel',last_argu)

Em_quarrel    category
dtype: object
0                     never
1          in the past year
2         in the past month
3    more than one year ago
4                       NaN
5          in the past week
6           in the past day
dtype: category
Categories (6, object): ['in the past day', 'in the past month', 'in the past week',
                         'in the past year', 'more than one year ago', 'never']
361
6.549346879535559
Em_quarrel
in the past year          1515
more than one year ago    1460
in the past month          883
never                      815
in the past week           370
in the past day            108
Name: count, dtype: int64
              Em_quarrel
count               5151
unique                 6
top     in the past year
freq                1515


In [ ]:
extract_information(mens_decision_making_el,'Em_rights')

Em_rights    float64
dtype: object
0    4.0
1    3.0
2    1.0
3    2.0
4    NaN
5    5.0
dtype: float64
336
6.095791001451379
Em_rights
4.0    2247
1.0    1590
3.0     715
2.0     600
5.0      24
Name: count, dtype: int64
         Em_rights
count  5176.000000
mean      2.713099
std       1.309715
min       1.000000
25%       1.000000
50%       3.000000
75%       4.000000
max       5.000000


In [ ]:
rights_map={
    1:  'Strongly Agree',
    2 :  'Agree',
    3:  'Disagree',
    4: 'Strongly Disagree',
    5: 'N/A'
}
mens_decision_making_el['Em_rights'] = mens_decision_making_el['Em_rights'].map(rights_map)
mens_decision_making_el['Em_rights'] = mens_decision_making_el['Em_rights'].astype('category')
parent_df['Em_rights'] = parent_df['Em_rights'].map(rights_map)
parent_df['Em_rights'] = parent_df['Em_rights'].astype('category')
extract_information(mens_decision_making_el,'Em_rights')

Em_rights    category
dtype: object
0    Strongly Disagree
1             Disagree
2       Strongly Agree
3                Agree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_rights
Strongly Disagree    2247
Strongly Agree       1590
Disagree              715
Agree                 600
N/A                    24
Name: count, dtype: int64
                Em_rights
count                5176
unique                  5
top     Strongly Disagree
freq                 2247


In [ ]:
extract_information(mens_decision_making_el,'Em_rights_sdi')

Em_rights_sdi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
360
6.531204644412192
Em_rights_sdi
0.0    2905
1.0    2247
Name: count, dtype: int64
       Em_rights_sdi
count    5152.000000
mean        0.436141
std         0.495953
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_equality_di')

Em_equality_di    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
354
6.422351233671988
Em_equality_di
0.0    3173
1.0    1985
Name: count, dtype: int64
       Em_equality_di
count     5158.000000
mean         0.384839
std          0.486604
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_equality')

Em_equality    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    4.0
4    NaN
5    5.0
dtype: float64
336
6.095791001451379
Em_equality
1.0    2010
2.0    1163
4.0    1089
3.0     896
5.0      18
Name: count, dtype: int64
       Em_equality
count  5176.000000
mean      2.215997
std       1.178103
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       5.000000


In [ ]:
equility_map={
    1:'Strongly Agree',
    2: 'Agree',
    3: 'Disagree',
    4: 'Strongly Disagree',
    5: 'N/A'
}
mens_decision_making_el['Em_equality'] = mens_decision_making_el['Em_equality'].map(equility_map)
mens_decision_making_el['Em_equality'] = mens_decision_making_el['Em_equality'].astype('category')
parent_df['Em_equality'] = parent_df['Em_equality'].map(equility_map)
parent_df['Em_equality'] = parent_df['Em_equality'].astype('category')
extract_information(mens_decision_making_el,'Em_equality')

Em_equality    category
dtype: object
0       Strongly Agree
1                Agree
2             Disagree
3    Strongly Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_equality
Strongly Agree       2010
Agree                1163
Strongly Disagree    1089
Disagree              896
N/A                    18
Name: count, dtype: int64
           Em_equality
count             5176
unique               5
top     Strongly Agree
freq              2010


In [ ]:
extract_information(mens_decision_making_el,'Em_equality_sdi')

Em_equality_sdi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
354
6.422351233671988
Em_equality_sdi
0.0    4069
1.0    1089
Name: count, dtype: int64
       Em_equality_sdi
count      5158.000000
mean          0.211128
std           0.408149
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_equality_di')


Em_equality_di    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
354
6.422351233671988
Em_equality_di
0.0    3173
1.0    1985
Name: count, dtype: int64
       Em_equality_di
count     5158.000000
mean         0.384839
std          0.486604
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_obey')


Em_obey    float64
dtype: object
0    1.0
1    NaN
2    4.0
3    3.0
4    2.0
5    5.0
dtype: float64
336
6.095791001451379
Em_obey
1.0    4495
2.0     503
3.0      96
4.0      78
5.0       4
Name: count, dtype: int64
           Em_obey
count  5176.000000
mean      1.182573
std       0.534865
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       5.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_boss')


Em_boss    float64
dtype: object
0    1.0
1    4.0
2    2.0
3    NaN
4    3.0
5    5.0
dtype: float64
336
6.095791001451379
Em_boss
1.0    2513
2.0    1221
4.0     940
3.0     495
5.0       7
Name: count, dtype: int64
           Em_boss
count  5176.000000
mean      1.977396
std       1.148690
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       5.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_leaders')

Em_leaders    float64
dtype: object
0    1.0
1    2.0
2    3.0
3    NaN
4    4.0
5    5.0
dtype: float64
336
6.095791001451379
Em_leaders
1.0    2654
2.0    1203
4.0     790
3.0     523
5.0       6
Name: count, dtype: int64
        Em_leaders
count  5176.000000
mean      1.897025
std       1.106512
min       1.000000
25%       1.000000
50%       1.000000
75%       3.000000
max       5.000000


In [ ]:
equality_map={
    1: 'Strongly Agree',
    2:'Agree',
    3:'Disagree',
    4:'Strongly Disagree',
    5:'N/A',
}
mens_decision_making_el['Em_boss'] = mens_decision_making_el['Em_boss'].map(equality_map)
mens_decision_making_el['Em_boss'] = mens_decision_making_el['Em_boss'].astype('category')
parent_df['Em_boss'] = parent_df['Em_boss'].map(equality_map)
parent_df['Em_boss'] = parent_df['Em_boss'].astype('category')
extract_information(mens_decision_making_el,'Em_boss')

Em_boss    category
dtype: object
0       Strongly Agree
1    Strongly Disagree
2                Agree
3                  NaN
4             Disagree
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_boss
Strongly Agree       2513
Agree                1221
Strongly Disagree     940
Disagree              495
N/A                     7
Name: count, dtype: int64
               Em_boss
count             5176
unique               5
top     Strongly Agree
freq              2513


In [ ]:
mens_decision_making_el['Em_obey'] = mens_decision_making_el['Em_obey'].map(equality_map)
mens_decision_making_el['Em_obey'] = mens_decision_making_el['Em_obey'].astype('category')
parent_df['Em_obey'] = parent_df['Em_obey'].map(equality_map)
parent_df['Em_obey'] = parent_df['Em_obey'].astype('category')
extract_information(mens_decision_making_el,'Em_obey')

Em_obey    category
dtype: object
0       Strongly Agree
1                  NaN
2    Strongly Disagree
3             Disagree
4                Agree
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_obey
Strongly Agree       4495
Agree                 503
Disagree               96
Strongly Disagree      78
N/A                     4
Name: count, dtype: int64
               Em_obey
count             5176
unique               5
top     Strongly Agree
freq              4495


In [ ]:
mens_decision_making_el['Em_leaders'] = mens_decision_making_el['Em_leaders'].map(equality_map)
mens_decision_making_el['Em_leaders'] = mens_decision_making_el['Em_leaders'].astype('category')
parent_df['Em_leaders'] = parent_df['Em_leaders'].map(equality_map)
parent_df['Em_leaders'] = parent_df['Em_leaders'].astype('category')
extract_information(mens_decision_making_el,'Em_obey')

Em_obey    category
dtype: object
0       Strongly Agree
1                  NaN
2    Strongly Disagree
3             Disagree
4                Agree
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_obey
Strongly Agree       4495
Agree                 503
Disagree               96
Strongly Disagree      78
N/A                     4
Name: count, dtype: int64
               Em_obey
count             5176
unique               5
top     Strongly Agree
freq              4495


In [ ]:
extract_information(mens_decision_making_el,'Em_shrpow')

Em_shrpow    float64
dtype: object
0    1.0
1    2.0
2    NaN
3    4.0
4    3.0
5    5.0
dtype: float64
336
6.095791001451379
Em_shrpow
1.0    2587
2.0    1403
4.0     775
3.0     406
5.0       5
Name: count, dtype: int64
         Em_shrpow
count  5176.000000
mean      1.880989
std       1.082553
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       5.000000


In [ ]:
mens_decision_making_el['Em_shrpow'] = mens_decision_making_el['Em_shrpow'].map(equality_map)
mens_decision_making_el['Em_shrpow'] = mens_decision_making_el['Em_shrpow'].astype('category')
parent_df['Em_shrpow'] = parent_df['Em_shrpow'].map(equality_map)
parent_df['Em_shrpow'] = parent_df['Em_shrpow'].astype('category')
extract_information(mens_decision_making_el,'Em_shrpow')

Em_shrpow    category
dtype: object
0       Strongly Agree
1                Agree
2                  NaN
3    Strongly Disagree
4             Disagree
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_shrpow
Strongly Agree       2587
Agree                1403
Strongly Disagree     775
Disagree              406
N/A                     5
Name: count, dtype: int64
             Em_shrpow
count             5176
unique               5
top     Strongly Agree
freq              2587


In [ ]:
extract_information(mens_decision_making_el,'Em_shrpow_sag')

Em_shrpow_sag    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
341
6.186502177068215
Em_shrpow_sag
1.0    2587
0.0    2584
Name: count, dtype: int64
       Em_shrpow_sag
count    5171.000000
mean        0.500290
std         0.500048
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_shrpow_ag')

Em_shrpow_ag    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
341
6.186502177068215
Em_shrpow_ag
1.0    3990
0.0    1181
Name: count, dtype: int64
       Em_shrpow_ag
count   5171.000000
mean       0.771611
std        0.419835
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_opin')

Em_opin    float64
dtype: object
0    1.0
1    NaN
2    2.0
3    4.0
4    3.0
5    5.0
dtype: float64
336
6.095791001451379
Em_opin
1.0    4060
2.0     852
3.0     132
4.0     129
5.0       3
Name: count, dtype: int64
           Em_opin
count  5176.000000
mean      1.292697
std       0.643896
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       5.000000


In [ ]:
mens_decision_making_el['Em_opin'] = mens_decision_making_el['Em_opin'].map(rights_map)
mens_decision_making_el['Em_opin'] = mens_decision_making_el['Em_opin'].astype('category')
parent_df['Em_opin'] = parent_df['Em_opin'].map(rights_map)
parent_df['Em_opin'] = parent_df['Em_opin'].astype('category')
extract_information(mens_decision_making_el,'Em_opin')

Em_opin    category
dtype: object
0       Strongly Agree
1                  NaN
2                Agree
3    Strongly Disagree
4             Disagree
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_opin
Strongly Agree       4060
Agree                 852
Disagree              132
Strongly Disagree     129
N/A                     3
Name: count, dtype: int64
               Em_opin
count             5176
unique               5
top     Strongly Agree
freq              4060


In [ ]:
extract_information(mens_decision_making_el,'Em_final_say')

Em_final_say    float64
dtype: object
0    1.0
1    4.0
2    2.0
3    3.0
4    NaN
5    5.0
dtype: float64
336
6.095791001451379
Em_final_say
1.0    2498
4.0    1613
2.0     613
3.0     447
5.0       5
Name: count, dtype: int64
       Em_final_say
count   5176.000000
mean       2.229907
std        1.331042
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max        5.000000


In [ ]:
mens_decision_making_el['Em_final_say'] = mens_decision_making_el['Em_final_say'].map(rights_map)
mens_decision_making_el['Em_final_say'] = mens_decision_making_el['Em_final_say'].astype('category')
parent_df['Em_final_say'] = parent_df['Em_final_say'].map(rights_map)
parent_df['Em_final_say'] = parent_df['Em_final_say'].astype('category')
extract_information(mens_decision_making_el,'Em_final_say')

Em_final_say    category
dtype: object
0       Strongly Agree
1    Strongly Disagree
2                Agree
3             Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_final_say
Strongly Agree       2498
Strongly Disagree    1613
Agree                 613
Disagree              447
N/A                     5
Name: count, dtype: int64
          Em_final_say
count             5176
unique               5
top     Strongly Agree
freq              2498


In [ ]:
mens_decision_making_el['Em_oblig_sex'] = mens_decision_making_el['Em_oblig_sex'].map(rights_map)
mens_decision_making_el['Em_oblig_sex'] = mens_decision_making_el['Em_oblig_sex'].astype('category')
parent_df['Em_oblig_sex'] = parent_df['Em_oblig_sex'].map(rights_map)
parent_df['Em_oblig_sex'] = parent_df['Em_oblig_sex'].astype('category')
extract_information(mens_decision_making_el,'Em_oblig_sex')

Em_oblig_sex    category
dtype: object
0    Strongly Disagree
1       Strongly Agree
2             Disagree
3                Agree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_oblig_sex
Strongly Disagree    1932
Strongly Agree       1860
Disagree              698
Agree                 682
N/A                     4
Name: count, dtype: int64
             Em_oblig_sex
count                5176
unique                  5
top     Strongly Disagree
freq                 1932


In [ ]:
mens_decision_making_el['Em_tol_viol'] = mens_decision_making_el['Em_tol_viol'].map(rights_map)
mens_decision_making_el['Em_tol_viol'] = mens_decision_making_el['Em_tol_viol'].astype('category')
parent_df['Em_tol_viol'] = parent_df['Em_tol_viol'].map(rights_map)
parent_df['Em_tol_viol'] = parent_df['Em_tol_viol'].astype('category')
extract_information(mens_decision_making_el,'Em_tol_viol')

Em_tol_viol    category
dtype: object
0       Strongly Agree
1                Agree
2                  NaN
3    Strongly Disagree
4             Disagree
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_tol_viol
Strongly Agree       4390
Agree                 541
Disagree              125
Strongly Disagree     118
N/A                     2
Name: count, dtype: int64
           Em_tol_viol
count             5176
unique               5
top     Strongly Agree
freq              4390


In [ ]:
mens_decision_making_el['Em_hit_argues'] = mens_decision_making_el['Em_hit_argues'].map(rights_map)
mens_decision_making_el['Em_hit_argues'] = mens_decision_making_el['Em_hit_argues'].astype('category')
parent_df['Em_hit_argues'] = parent_df['Em_hit_argues'].map(rights_map)
parent_df['Em_hit_argues'] = parent_df['Em_hit_argues'].astype('category')
extract_information(mens_decision_making_el,'Em_hit_argues')

Em_hit_argues    category
dtype: object
0    Strongly Disagree
1                Agree
2                  NaN
3             Disagree
4       Strongly Agree
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
336
6.095791001451379
Em_hit_argues
Strongly Disagree    3462
Disagree              668
Strongly Agree        601
Agree                 442
N/A                     3
Name: count, dtype: int64
            Em_hit_argues
count                5176
unique                  5
top     Strongly Disagree
freq                 3462


In [ ]:
talk_hiv_map= {
    1 : 'Yes, we both know each others’ status',
    2 : 'We have only talked about my HIV status',
    3 : 'We have only talked about her HIV status',
    4 : 'Neither of us have ever talked about our HIV status with each other '
}

In [ ]:
mens_decision_making_el['Em_talked_hiv'] = mens_decision_making_el['Em_talked_hiv'].map(talk_hiv_map)
mens_decision_making_el['Em_talked_hiv'] = mens_decision_making_el['Em_talked_hiv'].astype('category')
parent_df['Em_talked_hiv'] = parent_df['Em_talked_hiv'].map(talk_hiv_map)
parent_df['Em_talked_hiv'] = parent_df['Em_talked_hiv'].astype('category')
extract_information(mens_decision_making_el,'Em_talked_hiv')

Em_talked_hiv    category
dtype: object
0                Yes, we both know each others’ status
1             We have only talked about her HIV status
2    Neither of us have ever talked about our HIV s...
3                                                  NaN
4              We have only talked about my HIV status
dtype: category
Categories (4, object): ['Neither of us have ever talked about our HIV ..., 'We have only talked about her HIV status',
                         'We have only talked about my HIV status', 'Yes, we both know each others’ status']
336
6.095791001451379
Em_talked_hiv
Yes, we both know each others’ status                                   3723
Neither of us have ever talked about our HIV status with each other      833
We have only talked about her HIV status                                 528
We have only talked about my HIV status                                   92
Name: count, dtype: int64
                                Em_talked_hiv
count                   

In [ ]:
communication_style_map = {
    1 :'We are not able to discuss an issue without having an argument',
    2 : 'We can talk about most issues, but we argue sometimes ',
    3 : 'We have good communication, but occasionally she gets too emotional',
    4: 'We are always able to discuss issues in a fair and reasonable way each other',
}

In [ ]:
mens_decision_making_el['Em_wife_comm'] = mens_decision_making_el['Em_wife_comm'].map(communication_style_map)
mens_decision_making_el['Em_wife_comm'] = mens_decision_making_el['Em_wife_comm'].astype('category')
parent_df['Em_wife_comm'] = parent_df['Em_wife_comm'].map(communication_style_map)
parent_df['Em_wife_comm'] = parent_df['Em_wife_comm'].astype('category')
extract_information(mens_decision_making_el,'Em_wife_comm')

Em_wife_comm    category
dtype: object
0    We are always able to discuss issues in a fair...
1    We can talk about most issues, but we argue so...
2    We have good communication, but occasionally s...
3                                                  NaN
4    We are not able to discuss an issue without ha...
dtype: category
Categories (4, object): ['We are always able to discuss issues in a fai..., 'We are not able to discuss an issue without h...,
                         'We can talk about most issues, but we argue s..., 'We have good communication, but occasionally ...]
336
6.095791001451379
Em_wife_comm
We are always able to discuss issues in a fair and reasonable way each other    3490
We can talk about most issues, but we argue sometimes                            951
We have good communication, but occasionally she gets too emotional              429
We are not able to discuss an issue without having an argument                   306
Name: count, dtype: int64
               

In [ ]:
extract_information(mens_decision_making_el,'Em_wife_diss')

Em_wife_diss    float64
dtype: object
0    4.0
1    3.0
2    2.0
3    NaN
4    1.0
dtype: float64
336
6.095791001451379
Em_wife_diss
4.0    1986
3.0    1849
2.0     974
1.0     367
Name: count, dtype: int64
       Em_wife_diss
count   5176.000000
mean       3.053709
std        0.923454
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        4.000000


In [ ]:
wife_diss = {
    1:'Very often',
    2:'Somewhat often',
    3:'Rarely',
    4:'Never'
}
mens_decision_making_el['Em_wife_diss'] = mens_decision_making_el['Em_wife_diss'].map(wife_diss)
mens_decision_making_el['Em_wife_diss'] = mens_decision_making_el['Em_wife_diss'].astype('category')
parent_df['Em_wife_diss'] = parent_df['Em_wife_diss'].map(wife_diss)
parent_df['Em_wife_diss'] = parent_df['Em_wife_diss'].astype('category')
extract_information(mens_decision_making_el,'Em_wife_diss')

Em_wife_diss    category
dtype: object
0             Never
1            Rarely
2    Somewhat often
3               NaN
4        Very often
dtype: category
Categories (4, object): ['Never', 'Rarely', 'Somewhat often', 'Very often']
336
6.095791001451379
Em_wife_diss
Never             1986
Rarely            1849
Somewhat often     974
Very often         367
Name: count, dtype: int64
       Em_wife_diss
count          5176
unique            4
top           Never
freq           1986


In [ ]:
extract_information(mens_decision_making_el,'Em_wife_diss_vo')

Em_wife_diss_vo    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
336
6.095791001451379
Em_wife_diss_vo
0.0    4809
1.0     367
Name: count, dtype: int64
       Em_wife_diss_vo
count      5176.000000
mean          0.070904
std           0.256690
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_wife_diss_sw')

Em_wife_diss_sw    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
336
6.095791001451379
Em_wife_diss_sw
0.0    3835
1.0    1341
Name: count, dtype: int64
       Em_wife_diss_sw
count      5176.000000
mean          0.259080
std           0.438172
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_rel_year')

Em_rel_year    float64
dtype: object
0    2.0
1    1.0
2    3.0
3    NaN
dtype: float64
353
6.404208998548622
Em_rel_year
1.0    2662
2.0    2172
3.0     325
Name: count, dtype: int64
       Em_rel_year
count  5159.000000
mean      1.547005
std       0.611438
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       3.000000


In [ ]:
rel_year_map = {
    1 : 'improved',
    2 : 'stayed the same',
    3 : 'worsened',
    888 : 'don’t know',
    999 : 'refuse',
}
mens_decision_making_el['Em_rel_year'] = mens_decision_making_el['Em_rel_year'].map(rel_year_map)
mens_decision_making_el['Em_rel_year'] = mens_decision_making_el['Em_rel_year'].astype('category')
parent_df['Em_rel_year'] = parent_df['Em_rel_year'].map(rel_year_map)
parent_df['Em_rel_year'] = parent_df['Em_rel_year'].astype('category')
extract_information(mens_decision_making_el,'Em_rel_year')

Em_rel_year    category
dtype: object
0    stayed the same
1           improved
2           worsened
3                NaN
dtype: category
Categories (3, object): ['improved', 'stayed the same', 'worsened']
353
6.404208998548622
Em_rel_year
improved           2662
stayed the same    2172
worsened            325
Name: count, dtype: int64
       Em_rel_year
count         5159
unique           3
top       improved
freq          2662


In [ ]:
extract_information(mens_decision_making_el,'Em_rel_year_imp')

Em_rel_year_imp    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
353
6.404208998548622
Em_rel_year_imp
1.0    2662
0.0    2497
Name: count, dtype: int64
       Em_rel_year_imp
count      5159.000000
mean          0.515991
std           0.499793
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_rel_improve')

Em_rel_improve    object
dtype: object
0                  NaN
1            1 2 4 5 6
2                  2 5
3          1 2 3 4 5 6
4                4 5 6
            ...       
64          don't know
65              4 -999
66                 1 5
67    1 2 3 4 5 6 -999
68              6 -999
Length: 69, dtype: object
2850
51.705370101596515
Em_rel_improve
1 2 3 4 5 6         499
5 6                 215
1 2 3 5 6           130
1 2 3 6             122
6                   110
                   ... 
1 4 5 6               1
don't know            1
1 2 3 5 6 -999        1
4 -999                1
1 2 3 4 5 6 -999      1
Name: count, Length: 68, dtype: int64
       Em_rel_improve
count            2662
unique             68
top       1 2 3 4 5 6
freq              499


In [ ]:
extract_information(mens_decision_making_el,'Em_rel_improve1')

Em_rel_improve1    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
353
6.404208998548622
Em_rel_improve1
0.0    3640
1.0    1519
Name: count, dtype: int64
       Em_rel_improve1
count      5159.000000
mean          0.294437
std           0.455833
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(mens_decision_making_el,'Em_rel_worse')

Em_rel_worse    object
dtype: object
0            NaN
1              2
2          2 3 5
3            2 3
4          3 4 5
5        2 3 4 5
6            3 5
7      1 2 3 4 5
8              5
9            4 5
10           2 4
11             3
12           1 5
13         1 2 5
14           2 5
15         1 2 3
16         2 3 4
17           1 3
18             1
19           3 4
20       1 3 4 5
21           1 2
22    don't know
23         1 3 5
24         1 4 5
25             4
26         3 888
27       1 2 3 5
28       1 2 4 5
29       refusal
30         2 4 5
31       1 2 3 4
32           1 4
33         1 3 4
dtype: object
5187
94.10377358490565
Em_rel_worse
2 3 5         42
5             36
3 5           33
2 3           29
3             23
1 2 3 4 5     19
2             17
1 2 3         16
2 3 4 5       14
2 5           10
1 2 5         10
1 3            8
3 4 5          6
4 5            6
4              6
1 2 3 5        6
2 4            5
2 3 4          5
1 5            5
1 3 5       

### Women Anthro section

In [ ]:
women_anthro = parent_all_sections['WOMENS_ANTHRO']
women_anthro.columns

Index(['Ew_elapsed_time_h', 'Ew_elapsed_time_mo', 'Ew_measured', 'Ew_hblevel',
       'Ew_hblevel_flag', 'Ew_anaemia_12', 'Ew_anaemia_11', 'Ew_anaemia_8',
       'Ew_weight', 'Ew_weight_flag', 'Ew_bmi', 'Ew_bmi_18h', 'Ew_bmi_17',
       'Ew_bmi_16', 'Ew_muac', 'Ew_muac_flag', 'Ew_muac_preg23', 'Ew_muac_221',
       'Ew_muac_214'],
      dtype='object')

In [ ]:
extract_information(women_anthro,'Ew_weight')

Ew_weight    float64
dtype: object
0      69.500000
1      45.799999
2      61.200001
3      37.599998
4      52.900002
         ...    
770    61.450001
771    59.350000
772    42.000000
773    74.049999
774    62.250000
Length: 775, dtype: float64
323
5.8599419448476056
Ew_weight
50.000000    51
54.000000    38
57.000000    37
55.599998    36
53.700001    35
             ..
58.150000     1
54.833333     1
77.450001     1
89.800003     1
42.633333     1
Name: count, Length: 774, dtype: int64
         Ew_weight
count  5189.000000
mean     57.114470
std       9.278460
min      32.500000
25%      50.500000
50%      55.799999
75%      62.099998
max      94.400002


In [ ]:
extract_information(women_anthro,'Ew_weight_flag')

Ew_weight_flag    int64
dtype: object
0    0
1    1
dtype: int64
0
0.0
Ew_weight_flag
0    5189
1     323
Name: count, dtype: int64
       Ew_weight_flag
count     5512.000000
mean         0.058599
std          0.234895
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


In [ ]:
extract_information(women_anthro,'Ew_elapsed_time_h')

Ew_elapsed_time_h    float64
dtype: object
0       0.959444
1       6.435000
2       1.019722
3       0.936944
4       1.232222
          ...   
4865    0.016389
4866    2.627778
4867    5.314722
4868    3.194167
4869    2.407778
Length: 4870, dtype: float64
4
0.07256894049346879
Ew_elapsed_time_h
0.524722    5
1.071111    4
1.418889    4
0.589167    4
0.509444    4
           ..
4.125000    1
4.708333    1
2.196944    1
2.137778    1
4.076944    1
Name: count, Length: 4869, dtype: int64
       Ew_elapsed_time_h
count        5508.000000
mean           59.553843
std           339.328893
min             0.000833
25%             1.035208
50%             2.154306
75%             3.837847
max          4130.849167


In [ ]:
extract_information(women_anthro,'Ew_elapsed_time_mo')

Ew_elapsed_time_mo    float64
dtype: object
0       0.001333
1       0.008937
2       0.001416
3       0.001301
4       0.001711
          ...   
4865    0.000023
4866    0.003650
4867    0.007382
4868    0.004436
4869    0.003344
Length: 4870, dtype: float64
4
0.07256894049346879
Ew_elapsed_time_mo
0.000729    5
0.001488    4
0.001971    4
0.000818    4
0.000708    4
           ..
0.005729    1
0.006539    1
0.003051    1
0.002969    1
0.005662    1
Name: count, Length: 4869, dtype: int64
       Ew_elapsed_time_mo
count         5508.000000
mean             0.082714
std              0.471290
min              0.000001
25%              0.001438
50%              0.002992
75%              0.005330
max              5.737291


In [ ]:
extract_information(women_anthro,'Ew_anaemia_12')

Ew_anaemia_12    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
319
5.787373004354136
Ew_anaemia_12
0.0    4368
1.0     825
Name: count, dtype: int64
       Ew_anaemia_12
count    5193.000000
mean        0.158868
std         0.365588
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000


In [ ]:
extract_information(women_anthro,'Ew_anaemia_11')

Ew_anaemia_11    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
319
5.787373004354136
Ew_anaemia_11
0.0    4901
1.0     292
Name: count, dtype: int64
       Ew_anaemia_11
count    5193.000000
mean        0.056230
std         0.230387
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000


In [ ]:
extract_information(women_anthro,'Ew_anaemia_8')

Ew_anaemia_8    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
319
5.787373004354136
Ew_anaemia_8
0.0    5182
1.0      11
Name: count, dtype: int64
       Ew_anaemia_8
count   5193.000000
mean       0.002118
std        0.045980
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


In [ ]:
extract_information(women_anthro,'Ew_hblevel')

Ew_hblevel    float64
dtype: object
0      13.6
1      14.2
2      13.7
3      10.2
4      12.6
       ... 
101     7.7
102     8.0
103     8.2
104     8.3
105    17.7
Length: 106, dtype: float64
312
5.660377358490567
Ew_hblevel
13.4    176
13.2    172
12.9    159
13.3    154
13.5    153
       ... 
17.9      1
8.0       1
8.2       1
8.3       1
17.7      1
Name: count, Length: 105, dtype: int64
        Ew_hblevel
count  5200.000000
mean     13.052346
std       1.431454
min       6.100000
25%      12.200000
50%      13.100000
75%      14.000000
max      17.900000


In [ ]:
extract_information(women_anthro,'Ew_hblevel_flag')

Ew_hblevel_flag    int64
dtype: object
0    0
1    1
dtype: int64
0
0.0
Ew_hblevel_flag
0    5200
1     312
Name: count, dtype: int64
       Ew_hblevel_flag
count      5512.000000
mean          0.056604
std           0.231105
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000


In [ ]:
women_anthro['Ew_measured'] = women_anthro['Ew_measured'].astype('category')
extract_information(women_anthro,'Ew_measured')

Ew_measured    category
dtype: object
0    1.0
1    NaN
2    2.0
dtype: category
Categories (2, float64): [1.0, 2.0]
251
4.553701015965167
Ew_measured
1.0    5208
2.0      53
Name: count, dtype: int64
        Ew_measured
count        5261.0
unique          2.0
top             1.0
freq         5208.0


In [ ]:
extract_information(women_anthro,'Ew_bmi')

Ew_bmi    float64
dtype: object
0       22.693878
1       18.989894
2       23.090993
3       16.823077
4       18.304499
          ...    
4911    22.568634
4912    24.070749
4913    30.026931
4914    24.255642
4915    29.556021
Length: 4916, dtype: float64
373
6.767053701015965
Ew_bmi
21.992800    3
20.515929    3
21.030284    3
19.031605    3
23.631103    3
            ..
19.462499    1
24.046127    1
18.255560    1
20.392889    1
22.695272    1
Name: count, Length: 4915, dtype: int64
            Ew_bmi
count  5139.000000
mean     23.027659
std       3.358080
min      10.943624
25%      20.665712
50%      22.534644
75%      24.772575
max      36.997640


In [ ]:
extract_information(women_anthro,'Ew_muac')

Ew_muac    float64
dtype: object
0      28.200001
1      22.450000
2      28.299999
3      21.000000
4      24.000000
         ...    
378    26.200000
379    28.650001
380    36.450001
381    18.950000
382    25.650000
Length: 383, dtype: float64
312
5.660377358490567
Ew_muac
26.000000    104
25.000000     95
24.000000     95
25.500000     92
26.500000     91
            ... 
26.200000      1
28.650001      1
36.450001      1
18.950000      1
25.650000      1
Name: count, Length: 382, dtype: int64
           Ew_muac
count  5200.000000
mean     26.676054
std       3.128613
min      16.550000
25%      24.500000
50%      26.200001
75%      28.262500
max      45.599998


In [ ]:
extract_information(women_anthro,'Ew_muac_flag')
#TODO need inverse

Ew_muac_flag    int64
dtype: object
0    0
1    1
dtype: int64
0
0.0
Ew_muac_flag
0    5200
1     312
Name: count, dtype: int64
       Ew_muac_flag
count   5512.000000
mean       0.056604
std        0.231105
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


In [ ]:
extract_information(women_anthro,'Ew_muac_preg23')

Ew_muac_preg23    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
4746
86.10304789550072
Ew_muac_preg23
0.0    692
1.0     74
Name: count, dtype: int64
       Ew_muac_preg23
count      766.000000
mean         0.096606
std          0.295613
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


In [ ]:
extract_information(women_anthro,'Ew_muac_221')

Ew_muac_221    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
317
5.751088534107402
Ew_muac_221
0.0    4978
1.0     217
Name: count, dtype: int64
       Ew_muac_221
count  5195.000000
mean      0.041771
std       0.200085
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


### Expenditure EL section

In [ ]:
expanditure = parent_all_sections['EXPENDITURE_EL']
expanditure.columns

Index(['Ew_plan_hhexpend12', 'Ew_last7d_spend', 'Ew_last7d_sugar',
       'Ew_last7d_paraffin', 'Ew_last7d_soap', 'Ew_last7d_tea',
       'Ew_last7d_milk', 'Ew_last7d_meats', 'Ew_last7d_salt',
       'Ew_last7d_transp', 'Ew_last7d_phone', 'Ew_last7d_rice',
       'Ew_last7d_ccloth', 'Ew_last7d_health', 'Ew_last7d_schfee',
       'Ew_last7d_schsupp', 'Ew_last7d_wcloth', 'Ew_change_crops',
       'Ew_change_crops_how', 'Ew_change_crops_how1', 'Ew_change_crops_how2',
       'Ew_change_crops_how3'],
      dtype='object')

In [ ]:
extract_information(expanditure,'Ew_plan_hhexpend12')

Ew_plan_hhexpend12    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
321
5.8236574746008705
Ew_plan_hhexpend12
1.0    3363
0.0    1828
Name: count, dtype: int64
       Ew_plan_hhexpend12
count         5191.000000
mean             0.647852
std              0.477686
min              0.000000
25%              0.000000
50%              1.000000
75%              1.000000
max              1.000000


In [ ]:
extract_information(expanditure,'Ew_last7d_spend')

Ew_last7d_spend    float64
dtype: object
0   NaN
dtype: float64
5512
100.0
Series([], Name: count, dtype: int64)
       Ew_last7d_spend
count              0.0
mean               NaN
std                NaN
min                NaN
25%                NaN
50%                NaN
75%                NaN
max                NaN


In [ ]:
expanditure.drop('Ew_last7d_spend',axis=1,inplace=True)
parent_df.drop('Ew_last7d_spend',axis=1,inplace=True)

In [ ]:
extract_information(expanditure,'Ew_last7d_sugar')

Ew_last7d_sugar    float64
dtype: object
0         0.0
1      2600.0
2         NaN
3      4000.0
4      1300.0
       ...   
68     5800.0
69    24400.0
70       18.0
71     2100.0
72     2250.0
Length: 73, dtype: float64
251
4.553701015965167
Ew_last7d_sugar
0.0        2510
3000.0      767
2000.0      332
1500.0      329
1000.0      146
           ... 
5800.0        1
24400.0       1
18.0          1
2100.0        1
2250.0        1
Name: count, Length: 72, dtype: int64
       Ew_last7d_sugar
count      5261.000000
mean       1402.590002
std        1983.685949
min           0.000000
25%           0.000000
50%        1000.000000
75%        2500.000000
max       50000.000000


In [ ]:
extract_information(expanditure,'Ew_change_crops')

Ew_change_crops    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
285
5.170537010159651
Ew_change_crops
1.0    2735
0.0    2492
Name: count, dtype: int64
       Ew_change_crops
count      5227.000000
mean          0.523245
std           0.499507
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(expanditure,'Ew_change_crops_how')

Ew_change_crops_how    float64
dtype: object
0    1.0
1    NaN
2    3.0
3    2.0
dtype: float64
2777
50.38098693759071
Ew_change_crops_how
3.0    1457
1.0     988
2.0     290
Name: count, dtype: int64
       Ew_change_crops_how
count          2735.000000
mean              2.171481
std               0.929988
min               1.000000
25%               1.000000
50%               3.000000
75%               3.000000
max               3.000000


In [ ]:
extract_information(expanditure,'Ew_change_crops_how1')

Ew_change_crops_how1    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
2777
50.38098693759071
Ew_change_crops_how1
0.0    1747
1.0     988
Name: count, dtype: int64
       Ew_change_crops_how1
count           2735.000000
mean               0.361243
std                0.480449
min                0.000000
25%                0.000000
50%                0.000000
75%                1.000000
max                1.000000


In [ ]:
change_crops_map = {
    1 :'more  fruits / vegetables',
    2 :'less fruit / vegetables',
    3 :'other',
}
mapping_func(expanditure,parent_df,'Ew_change_crops_how',change_crops_map)

Ew_change_crops_how    category
dtype: object
0    more  fruits / vegetables
1                          NaN
2                        other
3      less fruit / vegetables
dtype: category
Categories (3, object): ['less fruit / vegetables', 'more  fruits / vegetables', 'other']
2777
50.38098693759071
Ew_change_crops_how
other                        1457
more  fruits / vegetables     988
less fruit / vegetables       290
Name: count, dtype: int64
       Ew_change_crops_how
count                 2735
unique                   3
top                  other
freq                  1457


In [ ]:
cols = ['Ew_change_crops_how1','Ew_change_crops_how2','Ew_change_crops_how3']
parent_df.drop(columns = cols,axis=1,inplace=True)
expanditure.drop(columns = cols,axis=1,inplace=True)

In [ ]:
expanditure.describe()

,Ew_plan_hhexpend12,Ew_last7d_sugar,Ew_last7d_paraffin,Ew_last7d_soap,Ew_last7d_tea,Ew_last7d_milk,Ew_last7d_meats,Ew_last7d_salt,Ew_last7d_transp,Ew_last7d_phone,Ew_last7d_rice,Ew_last7d_ccloth,Ew_last7d_health,Ew_last7d_schfee,Ew_last7d_schsupp,Ew_last7d_wcloth,Ew_change_crops
count,5191.000000,5261.000000,5253.000000,5270.000000,5276.000000,5279.000000,5268.000000,5279.000000,5248.000000,5122.000000,5280.000000,5.257000e+03,5.259000e+03,5.229000e+03,5.170000e+03,5287.000000,5227.000000
mean,0.647852,1402.590002,1171.248049,2036.043643,260.044731,1486.991476,3957.614085,701.270316,3972.220465,1449.963491,692.789962,2.708807e+04,5.013590e+04,1.562936e+05,3.027322e+04,20487.142236,0.523245
std,0.477686,1983.685949,1650.358422,1923.429426,516.564113,2833.031916,6512.974560,761.284702,9458.004015,2433.983164,2040.069457,4.270912e+04,1.416839e+05,3.900618e+05,6.623081e+04,32727.574095,0.499507
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000
25%,0.000000,0.000000,0.000000,1000.000000,0.000000,0.000000,0.000000,500.000000,0.000000,0.000000,0.000000,0.000000e+00,3.000000e+03,1.000000e+04,1.300000e+03,0.000000,0.000000
50%,1.000000,1000.000000,1000.000000,2000.000000,0.000000,0.000000,0.000000,500.000000,0.000000,1000.000000,0.000000,2.000000e+04,2.000000e+04,4.800000e+04,1.000000e+04,7000.000000,1.000000
75%,1.000000,2500.000000,2000.000000,2400.000000,500.000000,2000.000000,6000.000000,1000.000000,5000.000000,2000.000000,0.000000,4.000000e+04,5.000000e+04,1.400000e+05,3.000000e+04,30000.000000,1.000000
max,1.000000,50000.000000,50000.000000,60000.000000,7000.000000,60000.000000,175000.000000,20000.000000,160000.000000,35000.000000,70000.000000,1.500000e+06,5.000000e+06,7.500000e+06,1.500000e+06,721000.000000,1.000000


### Dcision Making EL section

In [ ]:
decision_making_el = parent_all_sections['DECISION_MAKING_EL']
decision_making_el.columns

Index(['Ew_daily_bi', 'Ew_daily_who', 'Ew_daily_diffop', 'Ew_daily_express',
       'Ew_daily_argue', 'Ew_daily_prevail', 'Ew_purch_bi', 'Ew_purch_who',
       'Ew_purch_diffop', 'Ew_purch_express',
       ...
       'Ew_rel_improve3', 'Ew_rel_improve4', 'Ew_rel_improve5',
       'Ew_rel_improve6', 'Ew_rel_worse', 'Ew_rel_worse1', 'Ew_rel_worse2',
       'Ew_rel_worse3', 'Ew_rel_worse4', 'Ew_rel_worse5'],
      dtype='object', length=155)

In [ ]:
extract_information(decision_making_el,'Ew_daily_bi')

Ew_daily_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
226
4.100145137880987
Ew_daily_bi
1.0    2796
0.0    2490
Name: count, dtype: int64
       Ew_daily_bi
count  5286.000000
mean      0.528944
std       0.499209
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_daily_who',chealth_who_map)

Ew_daily_who    category
dtype: object
0            Husband
1      Both together
2               Wife
3                NaN
4    Other HH member
5     Not applicable
dtype: category
Categories (5, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member', 'Wife']
224
4.063860667634253
Ew_daily_who
Husband            2449
Both together      1786
Wife               1010
Other HH member      41
Not applicable        2
Name: count, dtype: int64
       Ew_daily_who
count          5288
unique            5
top         Husband
freq           2449


In [ ]:
extract_information(decision_making_el,'Ew_daily_prevail')

Ew_daily_prevail    float64
dtype: object
0    NaN
1    2.0
2    1.0
3    4.0
4    3.0
dtype: float64
4447
80.67851959361393
Ew_daily_prevail
2.0    654
1.0    317
4.0     93
3.0      1
Name: count, dtype: int64
       Ew_daily_prevail
count       1065.000000
mean           1.877934
std            0.795979
min            1.000000
25%            1.000000
50%            2.000000
75%            2.000000
max            4.000000


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_daily_prevail',daily_prevail_map)

Ew_daily_prevail    category
dtype: object
0                             NaN
1                     Your spouse
2                        Yourself
3    Respondent finds another way
4                  A third person
dtype: category
Categories (4, object): ['A third person', 'Respondent finds another way', 'Your spouse', 'Yourself']
4447
80.67851959361393
Ew_daily_prevail
Your spouse                     654
Yourself                        317
Respondent finds another way     93
A third person                    1
Name: count, dtype: int64
       Ew_daily_prevail
count              1065
unique                4
top         Your spouse
freq                654


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_purch_who',chealth_who_map)

Ew_purch_who    category
dtype: object
0                   Husband
1             Both together
2                      Wife
3                       NaN
4           Other HH member
5    Someone outside the HH
6            Not applicable
dtype: category
Categories (6, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member',
                         'Someone outside the HH', 'Wife']
224
4.063860667634253
Ew_purch_who
Husband                   2579
Both together             1807
Wife                       857
Other HH member             42
Not applicable               2
Someone outside the HH       1
Name: count, dtype: int64
       Ew_purch_who
count          5288
unique            6
top         Husband
freq           2579


In [ ]:
extract_information(decision_making_el,'Ew_purch_bi')

Ew_purch_bi    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
226
4.100145137880987
Ew_purch_bi
1.0    2664
0.0    2622
Name: count, dtype: int64
       Ew_purch_bi
count  5286.000000
mean      0.503973
std       0.500032
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_purch_prevail',daily_prevail_map)

Ew_purch_prevail    category
dtype: object
0                             NaN
1                     Your spouse
2                        Yourself
3    Respondent finds another way
dtype: category
Categories (3, object): ['Respondent finds another way', 'Your spouse', 'Yourself']
4612
83.67198838896952
Ew_purch_prevail
Your spouse                     529
Yourself                        273
Respondent finds another way     98
Name: count, dtype: int64
       Ew_purch_prevail
count               900
unique                3
top         Your spouse
freq                529


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_savespend_who',chealth_who_map)

Ew_savespend_who    category
dtype: object
0      Both together
1            Husband
2               Wife
3                NaN
4    Other HH member
5     Not applicable
dtype: category
Categories (5, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member', 'Wife']
224
4.063860667634253
Ew_savespend_who
Husband            2568
Both together      2130
Wife                498
Not applicable       64
Other HH member      28
Name: count, dtype: int64
       Ew_savespend_who
count              5288
unique                5
top             Husband
freq               2568


In [ ]:
extract_information( decision_making_el,'Ew_savespend_bi')

Ew_savespend_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
288
5.2249637155297535
Ew_savespend_bi
1.0    2628
0.0    2596
Name: count, dtype: int64
       Ew_savespend_bi
count      5224.000000
mean          0.503063
std           0.500038
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


In [ ]:
decision_making_el['Ew_savespend_prevail'] = decision_making_el['Ew_savespend_prevail'].map(daily_prevail_map)
decision_making_el['Ew_savespend_prevail'] = decision_making_el['Ew_savespend_prevail'].astype('category')
parent_df['Ew_savespend_prevail'] = parent_df['Ew_savespend_prevail'].map(daily_prevail_map)
parent_df['Ew_savespend_prevail'] = parent_df['Ew_savespend_prevail'].astype('category')
extract_information(decision_making_el,'Ew_savespend_prevail')

Ew_savespend_prevail    category
dtype: object
0                             NaN
1                     Your spouse
2                        Yourself
3    Respondent finds another way
4                  A third person
dtype: category
Categories (4, object): ['A third person', 'Respondent finds another way', 'Your spouse', 'Yourself']
4308
78.15674891146588
Ew_savespend_prevail
Your spouse                     782
Yourself                        296
Respondent finds another way    124
A third person                    2
Name: count, dtype: int64
       Ew_savespend_prevail
count                  1204
unique                    4
top             Your spouse
freq                    782


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_wcloth_who',chealth_who_map)

Ew_wcloth_who    category
dtype: object
0             Both together
1                      Wife
2                   Husband
3                       NaN
4           Other HH member
5            Not applicable
6    Someone outside the HH
dtype: category
Categories (6, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member',
                         'Someone outside the HH', 'Wife']
224
4.063860667634253
Ew_wcloth_who
Wife                      2567
Husband                   1412
Both together             1295
Other HH member             10
Someone outside the HH       3
Not applicable               1
Name: count, dtype: int64
       Ew_wcloth_who
count           5288
unique             6
top             Wife
freq            2567


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_wcloth_prevail',daily_prevail_map)

Ew_wcloth_prevail    category
dtype: object
0                             NaN
1                     Your spouse
2                        Yourself
3    Respondent finds another way
dtype: category
Categories (3, object): ['Respondent finds another way', 'Your spouse', 'Yourself']
4922
89.29608127721336
Ew_wcloth_prevail
Yourself                        268
Your spouse                     261
Respondent finds another way     61
Name: count, dtype: int64
       Ew_wcloth_prevail
count                590
unique                 3
top             Yourself
freq                 268


In [ ]:
extract_information(decision_making_el,'Ew_wcloth_who')

Ew_wcloth_who    category
dtype: object
0             Both together
1                      Wife
2                   Husband
3                       NaN
4           Other HH member
5            Not applicable
6    Someone outside the HH
dtype: category
Categories (6, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member',
                         'Someone outside the HH', 'Wife']
224
4.063860667634253
Ew_wcloth_who
Wife                      2567
Husband                   1412
Both together             1295
Other HH member             10
Someone outside the HH       3
Not applicable               1
Name: count, dtype: int64
       Ew_wcloth_who
count           5288
unique             6
top             Wife
freq            2567


In [ ]:
# decision_making_el['Ew_wcloth_who'] = decision_making_el['Ew_wcloth_who'].map(chealth_who_map)
# decision_making_el['Ew_wcloth_who'] = decision_making_el['Ew_wcloth_who'].astype('category')
# extract_information(decision_making_el,'Ew_wcloth_who')
mapping_func(decision_making_el,parent_df,'Ew_wcloth_who',chealth_who_map)

Ew_wcloth_who    category
dtype: object
0   NaN
dtype: category
Categories (0, float64): []
5512
100.0
Series([], Name: count, dtype: int64)
       Ew_wcloth_who
count              0
unique             0
top              NaN
freq             NaN


In [ ]:
# decision_making_el['Ew_wcloth_prevail'] = decision_making_el['Ew_wcloth_prevail'].map(daily_prevail_map)
# decision_making_el['Ew_wcloth_prevail'] = decision_making_el['Ew_wcloth_prevail'].astype('category')
# extract_information(decision_making_el,'Ew_wcloth_prevail')
mapping_func(decision_making_el,parent_df,'Ew_wcloth_prevail',daily_prevail_map)

Ew_wcloth_prevail    category
dtype: object
0   NaN
dtype: category
Categories (0, float64): []
5512
100.0
Series([], Name: count, dtype: int64)
       Ew_wcloth_prevail
count                  0
unique                 0
top                  NaN
freq                 NaN


In [ ]:
extract_information(decision_making_el,'Ew_wcloth_bi')

Ew_wcloth_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
225
4.08200290275762
Ew_wcloth_bi
1.0    3862
0.0    1425
Name: count, dtype: int64
       Ew_wcloth_bi
count   5287.000000
mean       0.730471
std        0.443757
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_chealth_who',chealth_who_map)

Ew_chealth_who    category
dtype: object
0             Both together
1                   Husband
2                      Wife
3                       NaN
4           Other HH member
5            Not applicable
6    Someone outside the HH
dtype: category
Categories (6, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member',
                         'Someone outside the HH', 'Wife']
224
4.063860667634253
Ew_chealth_who
Husband                   3101
Both together             1583
Wife                       558
Other HH member             22
Not applicable              20
Someone outside the HH       4
Name: count, dtype: int64
       Ew_chealth_who
count            5288
unique              6
top           Husband
freq             3101


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_chealth_prevail',daily_prevail_map)

Ew_chealth_prevail    category
dtype: object
0                             NaN
1                     Your spouse
2                        Yourself
3    Respondent finds another way
4                  A third person
dtype: category
Categories (4, object): ['A third person', 'Respondent finds another way', 'Your spouse', 'Yourself']
5129
93.05152394775035
Ew_chealth_prevail
Your spouse                     170
Yourself                        163
Respondent finds another way     49
A third person                    1
Name: count, dtype: int64
       Ew_chealth_prevail
count                 383
unique                  4
top           Your spouse
freq                  170


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_cfeed_who',chealth_who_map)

Ew_cfeed_who    category
dtype: object
0                      Wife
1             Both together
2                   Husband
3                       NaN
4           Other HH member
5            Not applicable
6    Someone outside the HH
dtype: category
Categories (6, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member',
                         'Someone outside the HH', 'Wife']
224
4.063860667634253
Ew_cfeed_who
Wife                      4340
Both together              720
Husband                    181
Other HH member             24
Not applicable              21
Someone outside the HH       2
Name: count, dtype: int64
       Ew_cfeed_who
count          5288
unique            6
top            Wife
freq           4340


In [ ]:
# decision_making_el['Ew_cfeed_prevail'] = decision_making_el['Ew_cfeed_prevail'].map(daily_prevail_map)
# decision_making_el['Ew_cfeed_prevail'] = decision_making_el['Ew_cfeed_prevail'].astype('category')
# extract_information(decision_making_el,'Ew_cfeed_prevail')
mapping_func(decision_making_el,parent_df,'Ew_cfeed_prevail',daily_prevail_map)

Ew_cfeed_prevail    category
dtype: object
0                             NaN
1                     Your spouse
2                        Yourself
3    Respondent finds another way
4                  A third person
dtype: category
Categories (4, object): ['A third person', 'Respondent finds another way', 'Your spouse', 'Yourself']
5306
96.26269956458636
Ew_cfeed_prevail
Yourself                        109
Your spouse                      77
Respondent finds another way     19
A third person                    1
Name: count, dtype: int64
       Ew_cfeed_prevail
count               206
unique                4
top            Yourself
freq                109


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_cschool_who',chealth_who_map)

Ew_cschool_who    category
dtype: object
0             Both together
1                   Husband
2            Not applicable
3                      Wife
4                       NaN
5           Other HH member
6    Someone outside the HH
dtype: category
Categories (6, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member',
                         'Someone outside the HH', 'Wife']
224
4.063860667634253
Ew_cschool_who
Husband                   2857
Both together             1250
Not applicable             819
Wife                       345
Other HH member             13
Someone outside the HH       4
Name: count, dtype: int64
       Ew_cschool_who
count            5288
unique              6
top           Husband
freq             2857


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_cschool_prevail',daily_prevail_map)

Ew_cschool_prevail    category
dtype: object
0                             NaN
1                        Yourself
2                     Your spouse
3    Respondent finds another way
4                  A third person
dtype: category
Categories (4, object): ['A third person', 'Respondent finds another way', 'Your spouse', 'Yourself']
5231
94.90203193033382
Ew_cschool_prevail
Your spouse                     130
Yourself                        108
Respondent finds another way     40
A third person                    3
Name: count, dtype: int64
       Ew_cschool_prevail
count                 281
unique                  4
top           Your spouse
freq                  130


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_ccloth_who',chealth_who_map)

Ew_ccloth_who    category
dtype: object
0             Both together
1                      Wife
2                   Husband
3                       NaN
4           Other HH member
5    Someone outside the HH
6            Not applicable
dtype: category
Categories (6, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member',
                         'Someone outside the HH', 'Wife']
224
4.063860667634253
Ew_ccloth_who
Both together             2032
Wife                      1615
Husband                   1604
Not applicable              22
Other HH member             12
Someone outside the HH       3
Name: count, dtype: int64
        Ew_ccloth_who
count            5288
unique              6
top     Both together
freq             2032


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_ccloth_prevail',daily_prevail_map)

Ew_ccloth_prevail    category
dtype: object
0                             NaN
1                        Yourself
2                     Your spouse
3    Respondent finds another way
dtype: category
Categories (3, object): ['Respondent finds another way', 'Your spouse', 'Yourself']
5113
92.7612481857765
Ew_ccloth_prevail
Yourself                        205
Your spouse                     157
Respondent finds another way     37
Name: count, dtype: int64
       Ew_ccloth_prevail
count                399
unique                 3
top             Yourself
freq                 205


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_wmoney_who',chealth_who_map)

Ew_wmoney_who    category
dtype: object
0      Both together
1               Wife
2            Husband
3     Not applicable
4                NaN
5    Other HH member
dtype: category
Categories (5, object): ['Both together', 'Husband', 'Not applicable', 'Other HH member', 'Wife']
224
4.063860667634253
Ew_wmoney_who
Wife               3589
Both together      1376
Husband             279
Not applicable       43
Other HH member       1
Name: count, dtype: int64
       Ew_wmoney_who
count           5288
unique             5
top             Wife
freq            3589


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_wmoney_prevail',daily_prevail_map)

Ew_wmoney_prevail    category
dtype: object
0                             NaN
1                     Your spouse
2                        Yourself
3    Respondent finds another way
4                  A third person
dtype: category
Categories (4, object): ['A third person', 'Respondent finds another way', 'Your spouse', 'Yourself']
4797
87.02830188679245
Ew_wmoney_prevail
Yourself                        398
Your spouse                     252
Respondent finds another way     63
A third person                    2
Name: count, dtype: int64
       Ew_wmoney_prevail
count                715
unique                 4
top             Yourself
freq                 398


In [ ]:
extract_information(decision_making_el,'Ew_rights')

Ew_rights    float64
dtype: object
0    4.0
1    1.0
2    2.0
3    3.0
4    NaN
5    5.0
dtype: float64
224
4.063860667634253
Ew_rights
4.0    2955
1.0    1036
3.0     697
2.0     570
5.0      30
Name: count, dtype: int64
         Ew_rights
count  5288.000000
mean      3.070537
std       1.211718
min       1.000000
25%       2.000000
50%       4.000000
75%       4.000000
max       5.000000


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_rights',rights_map)

Ew_rights    category
dtype: object
0    Strongly Disagree
1       Strongly Agree
2                Agree
3             Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_rights
Strongly Disagree    2955
Strongly Agree       1036
Disagree              697
Agree                 570
N/A                    30
Name: count, dtype: int64
                Ew_rights
count                5288
unique                  5
top     Strongly Disagree
freq                 2955


In [ ]:
extract_information(decision_making_el,'Ew_rights_di')

Ew_rights_di    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
254
4.6081277213352685
Ew_rights_di
1.0    3652
0.0    1606
Name: count, dtype: int64
       Ew_rights_di
count   5258.000000
mean       0.694561
std        0.460637
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(decision_making_el,'Ew_rights_sdi')

Ew_rights_sdi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
254
4.6081277213352685
Ew_rights_sdi
1.0    2955
0.0    2303
Name: count, dtype: int64
       Ew_rights_sdi
count    5258.000000
mean        0.562001
std         0.496188
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_equality',rights_map)

Ew_equality    category
dtype: object
0    Strongly Disagree
1                Agree
2       Strongly Agree
3                  N/A
4             Disagree
5                  NaN
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_equality
Strongly Disagree    1550
Strongly Agree       1417
Disagree             1245
Agree                1047
N/A                    29
Name: count, dtype: int64
              Ew_equality
count                5288
unique                  5
top     Strongly Disagree
freq                 1550


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_obey',rights_map)

Ew_obey    category
dtype: object
0       Strongly Agree
1                Agree
2    Strongly Disagree
3             Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_obey
Strongly Agree       4399
Agree                 679
Strongly Disagree     126
Disagree               83
N/A                     1
Name: count, dtype: int64
               Ew_obey
count             5288
unique               5
top     Strongly Agree
freq              4399


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_boss',rights_map)

Ew_boss    category
dtype: object
0       Strongly Agree
1                Agree
2             Disagree
3    Strongly Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_boss
Strongly Agree       2498
Agree                1116
Strongly Disagree    1007
Disagree              649
N/A                    18
Name: count, dtype: int64
               Ew_boss
count             5288
unique               5
top     Strongly Agree
freq              2498


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_leaders',rights_map)

Ew_leaders    category
dtype: object
0       Strongly Agree
1    Strongly Disagree
2             Disagree
3                Agree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_leaders
Strongly Disagree    2033
Strongly Agree       1825
Agree                 779
Disagree              632
N/A                    19
Name: count, dtype: int64
               Ew_leaders
count                5288
unique                  5
top     Strongly Disagree
freq                 2033


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_choose',rights_map)

Ew_choose    category
dtype: object
0       Strongly Agree
1                Agree
2    Strongly Disagree
3             Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_choose
Strongly Agree       3306
Agree                 897
Strongly Disagree     659
Disagree              423
N/A                     3
Name: count, dtype: int64
             Ew_choose
count             5288
unique               5
top     Strongly Agree
freq              3306


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_womfre',rights_map)

Ew_womfre    category
dtype: object
0       Strongly Agree
1                Agree
2    Strongly Disagree
3             Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_womfre
Strongly Agree       3609
Agree                 990
Strongly Disagree     385
Disagree              301
N/A                     3
Name: count, dtype: int64
             Ew_womfre
count             5288
unique               5
top     Strongly Agree
freq              3609


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_powhh',rights_map)

Ew_powhh    category
dtype: object
0    Strongly Disagree
1       Strongly Agree
2                  N/A
3                Agree
4             Disagree
5                  NaN
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_powhh
Strongly Disagree    2362
Strongly Agree       1409
Disagree              860
Agree                 639
N/A                    18
Name: count, dtype: int64
                 Ew_powhh
count                5288
unique                  5
top     Strongly Disagree
freq                 2362


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_shrpow',rights_map)

Ew_shrpow    category
dtype: object
0       Strongly Agree
1                Agree
2    Strongly Disagree
3             Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_shrpow
Strongly Agree       2897
Agree                1404
Strongly Disagree     590
Disagree              393
N/A                     4
Name: count, dtype: int64
             Ew_shrpow
count             5288
unique               5
top     Strongly Agree
freq              2897


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_opin',rights_map)

Ew_opin    category
dtype: object
0       Strongly Agree
1                Agree
2    Strongly Disagree
3                  NaN
4             Disagree
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_opin
Strongly Agree       4764
Agree                 440
Disagree               48
Strongly Disagree      32
N/A                     4
Name: count, dtype: int64
               Ew_opin
count             5288
unique               5
top     Strongly Agree
freq              4764


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_avoid_preg',rights_map)

Ew_avoid_preg    category
dtype: object
0       Strongly Agree
1                Agree
2    Strongly Disagree
3             Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_avoid_preg
Strongly Agree       3608
Agree                 860
Strongly Disagree     495
Disagree              324
N/A                     1
Name: count, dtype: int64
         Ew_avoid_preg
count             5288
unique               5
top     Strongly Agree
freq              3608


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_final_say',rights_map)

Ew_final_say    category
dtype: object
0       Strongly Agree
1                Agree
2    Strongly Disagree
3             Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_final_say
Strongly Agree       3568
Strongly Disagree     776
Agree                 599
Disagree              344
N/A                     1
Name: count, dtype: int64
          Ew_final_say
count             5288
unique               5
top     Strongly Agree
freq              3568


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_oblig_sex',rights_map)

Ew_oblig_sex    category
dtype: object
0    Strongly Disagree
1                Agree
2             Disagree
3       Strongly Agree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_oblig_sex
Strongly Agree       2460
Strongly Disagree    1312
Agree                1117
Disagree              395
N/A                     4
Name: count, dtype: int64
          Ew_oblig_sex
count             5288
unique               5
top     Strongly Agree
freq              2460


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_tol_viol',rights_map)

Ew_tol_viol    category
dtype: object
0       Strongly Agree
1                Agree
2    Strongly Disagree
3                  NaN
4             Disagree
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_tol_viol
Strongly Agree       4445
Agree                 610
Strongly Disagree     151
Disagree               78
N/A                     4
Name: count, dtype: int64
           Ew_tol_viol
count             5288
unique               5
top     Strongly Agree
freq              4445


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_hit_goesout',rights_map)

Ew_hit_goesout    category
dtype: object
0    Strongly Disagree
1       Strongly Agree
2                Agree
3             Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_hit_goesout
Strongly Disagree    2292
Strongly Agree       1535
Agree                 905
Disagree              552
N/A                     4
Name: count, dtype: int64
           Ew_hit_goesout
count                5288
unique                  5
top     Strongly Disagree
freq                 2292


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_hit_argues',rights_map)

Ew_hit_argues    category
dtype: object
0    Strongly Disagree
1       Strongly Agree
2                Agree
3             Disagree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_hit_argues
Strongly Disagree    3184
Strongly Agree        725
Agree                 716
Disagree              660
N/A                     3
Name: count, dtype: int64
            Ew_hit_argues
count                5288
unique                  5
top     Strongly Disagree
freq                 3184


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_hit_food',rights_map)

Ew_hit_food    category
dtype: object
0    Strongly Disagree
1       Strongly Agree
2             Disagree
3                Agree
4                  NaN
5                  N/A
dtype: category
Categories (5, object): ['Agree', 'Disagree', 'N/A', 'Strongly Agree', 'Strongly Disagree']
224
4.063860667634253
Ew_hit_food
Strongly Disagree    3070
Strongly Agree        838
Agree                 723
Disagree              651
N/A                     6
Name: count, dtype: int64
              Ew_hit_food
count                5288
unique                  5
top     Strongly Disagree
freq                 3070


In [ ]:
extract_information(decision_making_el,'Ew_husb_listens')

Ew_husb_listens    float64
dtype: object
0    2.0
1    4.0
2    5.0
3    1.0
4    3.0
5    6.0
6    NaN
dtype: float64
325
5.89622641509434
Ew_husb_listens
4.0    1470
1.0    1223
5.0     933
3.0     804
2.0     600
6.0     157
Name: count, dtype: int64
       Ew_husb_listens
count      5187.000000
mean          3.146713
std           1.520863
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           6.000000


In [ ]:
freq_map={
    1: 'All the time',
    2: 'Almost all of the time',
    3: 'Some of the time',
    4: 'Once and awhile',
    5: 'Rarely',
    6: 'Never',
    7: 'Refuse to respond',
}
mapping_func(decision_making_el,parent_df,'Ew_husb_listens',freq_map)

Ew_husb_listens    category
dtype: object
0    Almost all of the time
1           Once and awhile
2                    Rarely
3              All the time
4          Some of the time
5                     Never
6                       NaN
dtype: category
Categories (6, object): ['All the time', 'Almost all of the time', 'Never', 'Once and awhile',
                         'Rarely', 'Some of the time']
325
5.89622641509434
Ew_husb_listens
Once and awhile           1470
All the time              1223
Rarely                     933
Some of the time           804
Almost all of the time     600
Never                      157
Name: count, dtype: int64
        Ew_husb_listens
count              5187
unique                6
top     Once and awhile
freq               1470


In [ ]:
extract_information(decision_making_el,'Ew_husb_listens_bi')

Ew_husb_listens_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
325
5.89622641509434
Ew_husb_listens_bi
0.0    3364
1.0    1823
Name: count, dtype: int64
       Ew_husb_listens_bi
count         5187.000000
mean             0.351456
std              0.477471
min              0.000000
25%              0.000000
50%              0.000000
75%              1.000000
max              1.000000


In [ ]:
extract_information(decision_making_el,'Ew_husb_angry')

Ew_husb_angry    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
321
5.8236574746008705
Ew_husb_angry
1.0    4355
0.0     836
Name: count, dtype: int64
       Ew_husb_angry
count    5191.000000
mean        0.838952
std         0.367611
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000


In [ ]:
extract_information(decision_making_el,'Ew_husb_angryfreq')

Ew_husb_angryfreq    float64
dtype: object
0    NaN
1    5.0
2    3.0
3    4.0
4    1.0
5    2.0
6    6.0
dtype: float64
1157
20.99056603773585
Ew_husb_angryfreq
4.0    1949
5.0    1283
3.0     577
1.0     306
2.0     238
6.0       2
Name: count, dtype: int64
       Ew_husb_angryfreq
count        4355.000000
mean            3.842939
std             1.120504
min             1.000000
25%             3.000000
50%             4.000000
75%             5.000000
max             6.000000


In [ ]:
decision_making_el['Ew_husb_angryfreq'] = decision_making_el['Ew_husb_angryfreq'].map(freq_map)
decision_making_el['Ew_husb_angryfreq'] = decision_making_el['Ew_husb_angryfreq'].astype('category')
extract_information(decision_making_el,'Ew_husb_angryfreq')
mapping_func(decision_making_el,parent_df,'Ew_husb_angryfreq',freq_map)

Ew_husb_angryfreq    category
dtype: object
0                       NaN
1                    Rarely
2          Some of the time
3           Once and awhile
4              All the time
5    Almost all of the time
6                     Never
dtype: category
Categories (6, object): ['All the time', 'Almost all of the time', 'Never', 'Once and awhile',
                         'Rarely', 'Some of the time']
1157
20.99056603773585
Ew_husb_angryfreq
Once and awhile           1949
Rarely                    1283
Some of the time           577
All the time               306
Almost all of the time     238
Never                        2
Name: count, dtype: int64
       Ew_husb_angryfreq
count               4355
unique                 6
top      Once and awhile
freq                1949
Ew_husb_angryfreq    category
dtype: object
0   NaN
dtype: category
Categories (0, float64): []
5512
100.0
Series([], Name: count, dtype: int64)
       Ew_husb_angryfreq
count                  0
unique               

In [ ]:
extract_information(decision_making_el,'Ew_husb_angry_bi')

Ew_husb_angry_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
321
5.8236574746008705
Ew_husb_angry_bi
0.0    3234
1.0    1957
Name: count, dtype: int64
       Ew_husb_angry_bi
count       5191.000000
mean           0.376999
std            0.484681
min            0.000000
25%            0.000000
50%            0.000000
75%            1.000000
max            1.000000


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_quarrel',last_argu)

Ew_quarrel    category
dtype: object
0    more than one year ago
1          in the past week
2           in the past day
3                     never
4         in the past month
5          in the past year
6                       NaN
dtype: category
Categories (6, object): ['in the past day', 'in the past month', 'in the past week',
                         'in the past year', 'more than one year ago', 'never']
342
6.204644412191581
Ew_quarrel
in the past year          1446
in the past month         1267
in the past week           973
more than one year ago     826
never                      413
in the past day            245
Name: count, dtype: int64
              Ew_quarrel
count               5170
unique                 6
top     in the past year
freq                1446


In [ ]:
extract_information(decision_making_el,'Ew_quarrel_recent')

Ew_quarrel_recent    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
342
6.204644412191581
Ew_quarrel_recent
0.0    2685
1.0    2485
Name: count, dtype: int64
       Ew_quarrel_recent
count        5170.000000
mean            0.480658
std             0.499674
min             0.000000
25%             0.000000
50%             0.000000
75%             1.000000
max             1.000000


In [ ]:
extract_information(decision_making_el,'Ew_unfaith')

Ew_unfaith    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
321
5.8236574746008705
Ew_unfaith
0.0    4529
1.0     662
Name: count, dtype: int64
        Ew_unfaith
count  5191.000000
mean      0.127528
std       0.333596
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(decision_making_el,'Ew_notrust_mon')

Ew_notrust_mon    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
321
5.8236574746008705
Ew_notrust_mon
0.0    4244
1.0     947
Name: count, dtype: int64
       Ew_notrust_mon
count     5191.000000
mean         0.182431
std          0.386237
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


In [ ]:
extract_information(decision_making_el,'Ew_notrust_mon_no')

Ew_notrust_mon_no    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
321
5.8236574746008705
Ew_notrust_mon_no
1.0    4244
0.0     947
Name: count, dtype: int64
       Ew_notrust_mon_no
count        5191.000000
mean            0.817569
std             0.386237
min             0.000000
25%             1.000000
50%             1.000000
75%             1.000000
max             1.000000


In [ ]:
extract_information(decision_making_el,'Ew_all_times')

Ew_all_times    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
321
5.8236574746008705
Ew_all_times
0.0    2815
1.0    2376
Name: count, dtype: int64
       Ew_all_times
count   5191.000000
mean       0.457715
std        0.498257
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(decision_making_el,'Ew_all_times_no')

Ew_all_times_no    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
321
5.8236574746008705
Ew_all_times_no
1.0    2815
0.0    2376
Name: count, dtype: int64
       Ew_all_times_no
count      5191.000000
mean          0.542285
std           0.498257
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(decision_making_el,'Ew_humil')

Ew_humil    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
324
5.878084179970973
Ew_humil
0.0    4184
1.0    1004
Name: count, dtype: int64
          Ew_humil
count  5188.000000
mean      0.193524
std       0.395098
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(decision_making_el,'Ew_slapfreq')

Ew_slapfreq    float64
dtype: object
0    NaN
1    3.0
2    2.0
3    1.0
dtype: float64
3246
58.88969521044993
Ew_slapfreq
3.0    1749
2.0     433
1.0      84
Name: count, dtype: int64
       Ew_slapfreq
count  2266.000000
mean      2.734775
std       0.518786
min       1.000000
25%       3.000000
50%       3.000000
75%       3.000000
max       3.000000


In [ ]:
slap_freq={
    1:'Often ',
    2:'Sometimes',
    3:'Not at all'
}
mapping_func(decision_making_el,parent_df,'Ew_slapfreq',slap_freq)

Ew_slapfreq    category
dtype: object
0           NaN
1    Not at all
2     Sometimes
3        Often 
dtype: category
Categories (3, object): ['Not at all', 'Often ', 'Sometimes']
3246
58.88969521044993
Ew_slapfreq
Not at all    1749
Sometimes      433
Often           84
Name: count, dtype: int64
       Ew_slapfreq
count         2266
unique           3
top     Not at all
freq          1749


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_humilfreq',slap_freq)

Ew_humilfreq    category
dtype: object
0           NaN
1    Not at all
2     Sometimes
3        Often 
dtype: category
Categories (3, object): ['Not at all', 'Often ', 'Sometimes']
4508
81.78519593613933
Ew_humilfreq
Not at all    546
Sometimes     334
Often         124
Name: count, dtype: int64
       Ew_humilfreq
count          1004
unique            3
top      Not at all
freq            546


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_threatfreq',slap_freq)

Ew_threatfreq    category
dtype: object
0           NaN
1     Sometimes
2    Not at all
3        Often 
dtype: category
Categories (3, object): ['Not at all', 'Often ', 'Sometimes']
4085
74.11103047895502
Ew_threatfreq
Not at all    749
Sometimes     527
Often         151
Name: count, dtype: int64
       Ew_threatfreq
count           1427
unique             3
top       Not at all
freq             749


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_insultfreq',slap_freq)

Ew_insultfreq    category
dtype: object
0           NaN
1     Sometimes
2    Not at all
3        Often 
dtype: category
Categories (3, object): ['Not at all', 'Often ', 'Sometimes']
3825
69.39404934687954
Ew_insultfreq
Not at all    812
Sometimes     690
Often         185
Name: count, dtype: int64
       Ew_insultfreq
count           1687
unique             3
top       Not at all
freq             812


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_beatfreq',slap_freq)

Ew_beatfreq    category
dtype: object
0           NaN
1    Not at all
2     Sometimes
3        Often 
dtype: category
Categories (3, object): ['Not at all', 'Often ', 'Sometimes']
3221
58.43613933236574
Ew_beatfreq
Not at all    1864
Sometimes      367
Often           60
Name: count, dtype: int64
       Ew_beatfreq
count         2291
unique           3
top     Not at all
freq          1864


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_pushfreq',slap_freq)

Ew_pushfreq    category
dtype: object
0           NaN
1    Not at all
2     Sometimes
3        Often 
dtype: category
Categories (3, object): ['Not at all', 'Often ', 'Sometimes']
4147
75.23584905660378
Ew_pushfreq
Not at all    917
Sometimes     372
Often          76
Name: count, dtype: int64
       Ew_pushfreq
count         1365
unique           3
top     Not at all
freq           917


In [ ]:
extract_information(decision_making_el,'Ew_talked_hiv')

Ew_talked_hiv    float64
dtype: object
0    1.0
1    2.0
2    4.0
3    NaN
4    3.0
dtype: float64
321
5.8236574746008705
Ew_talked_hiv
1.0    3424
2.0    1051
4.0     665
3.0      51
Name: count, dtype: int64
       Ew_talked_hiv
count    5191.000000
mean        1.606434
std         1.013488
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         4.000000


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_talked_hiv',talk_hiv_map)

Ew_talked_hiv    category
dtype: object
0                Yes, we both know each others’ status
1              We have only talked about my HIV status
2    Neither of us have ever talked about our HIV s...
3                                                  NaN
4             We have only talked about her HIV status
dtype: category
Categories (4, object): ['Neither of us have ever talked about our HIV ..., 'We have only talked about her HIV status',
                         'We have only talked about my HIV status', 'Yes, we both know each others’ status']
321
5.8236574746008705
Ew_talked_hiv
Yes, we both know each others’ status                                   3424
We have only talked about my HIV status                                 1051
Neither of us have ever talked about our HIV status with each other      665
We have only talked about her HIV status                                  51
Name: count, dtype: int64
                                Ew_talked_hiv
count                  

In [ ]:
alcohol_num={
    1 : 'daily',
2 : 'several times a week',
3 : 'once a week',
4 : 'once every two weeks',
5 : 'less than once every two weeks',
6 : 'never',
888 : 'don’t know'

}
mapping_func(decision_making_el,parent_df,'Ew_alcohol_num',alcohol_num)

Ew_alcohol_num    category
dtype: object
0                             never
1                       once a week
2                             daily
3    less than once every two weeks
4              once every two weeks
5              several times a week
6                               NaN
dtype: category
Categories (6, object): ['daily', 'less than once every two weeks', 'never', 'once a week',
                         'once every two weeks', 'several times a week']
324
5.878084179970973
Ew_alcohol_num
never                             2314
once a week                        865
daily                              746
several times a week               692
once every two weeks               296
less than once every two weeks     275
Name: count, dtype: int64
       Ew_alcohol_num
count            5188
unique              6
top             never
freq             2314


In [ ]:
extract_information(decision_making_el,'Ew_alcohol_money')

Ew_alcohol_money    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
4319
78.35631349782292
Ew_alcohol_money
1.0    939
0.0    254
Name: count, dtype: int64
       Ew_alcohol_money
count       1193.000000
mean           0.787091
std            0.409535
min            0.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            1.000000


In [ ]:
extract_information(decision_making_el,'Ew_rel_year')

Ew_rel_year    float64
dtype: object
0    2.0
1    1.0
2    3.0
3    NaN
dtype: float64
335
6.077648766328012
Ew_rel_year
2.0    2373
1.0    2080
3.0     724
Name: count, dtype: int64
       Ew_rel_year
count  5177.000000
mean      1.738072
std       0.687831
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       3.000000


In [ ]:
mapping_func(decision_making_el,parent_df,'Ew_rel_year',rel_year_map)

Ew_rel_year    category
dtype: object
0    stayed the same
1           improved
2           worsened
3                NaN
dtype: category
Categories (3, object): ['improved', 'stayed the same', 'worsened']
335
6.077648766328012
Ew_rel_year
stayed the same    2373
improved           2080
worsened            724
Name: count, dtype: int64
            Ew_rel_year
count              5177
unique                3
top     stayed the same
freq               2373


In [ ]:
# 1 = Husband listens more to me
# 2 = I listen more to my husband
# 3 = We have fewer arguments
# 4 = Husband spends more time with and is more invovled with the family
# 5 = Husband shares household finances
# 6 = We share more information with each other
# 888 = Don’t Know
extract_information(decision_making_el,'Ew_rel_improve')

Ew_rel_improve    object
dtype: object
0              NaN
1      1 2 3 4 5 6
2            1 4 6
3            1 2 5
4              3 6
          ...     
61           1 3 4
62    1 3 5 6 -999
63           2 5 6
64      1 2 5 6 98
65             2 4
Length: 66, dtype: object
3432
62.264150943396224
Ew_rel_improve
1 2 3 4 5 6     468
1 2 4 5 6        84
5 6              77
1 2 3 5 6        75
4 5 6            71
               ... 
2                 3
2 5 6             3
1 3 5 6 -999      1
1 2 5 6 98        1
2 4               1
Name: count, Length: 65, dtype: int64
       Ew_rel_improve
count            2080
unique             65
top       1 2 3 4 5 6
freq              468


In [ ]:
# 1 = Husband listens less to me
# 2 = I listen less to my husband
# 3 = We have more arguments
# 4 = Husband is less engaged with the family
# 5 = Husband refuses to share household finances
# 888 = Don’t Know
#TODO
extract_information(decision_making_el,'Ew_rel_worse')

Ew_rel_worse    object
dtype: object
0            NaN
1              1
2        1 2 3 5
3              3
4        1 3 4 5
5            3 4
6      1 2 3 4 5
7            4 5
8          3 4 5
9          1 3 5
10         1 4 5
11           1 2
12         1 3 4
13             .
14           1 3
15       1 2 3 4
16         1 2 3
17       2 3 4 5
18           1 5
19             4
20           1 4
21           3 5
22       1 2 4 5
23             5
24         2 4 5
25         1 2 5
26           2 3
27             2
28      4 5 -999
29         1 2 4
30           2 4
31           2 5
32    don't know
dtype: object
4687
85.03265602322206
Ew_rel_worse
1 3 4 5       139
.             101
1 2 3 4 5      75
1 4 5          61
4 5            44
3              41
3 4 5          41
1 3 5          40
4              33
1 3            27
1 3 4          27
3 5            25
5              25
1              24
1 4            20
1 5            16
3 4            14
1 2 4 5        14
1 2 3          13
1 2 3 5   

### DECISION_MAKING

In [ ]:
print(parent_df.columns.tolist())

['id', 'wemp', 'whn', 'mhn', 'treat', 'control', 'stratum', 'villageid', 'district_name', 'districtid', 'subcountyid', 'sample_women', 'sample_married', 'sample_couple', 'sample_pair', 'sample_anthro', 'BL_WOMENS', 'w_date', 'w_date_code', 'w_enum_id', 'w_time', 'w_eligibility_kids', 'DEMOGRAPHICS_ROSTER', 'w_age', 'w_educ', 'w_inc', 'w_marital', 'w_num_adults', 'w_numkids18', 'w_numkids18_bio', 'w_numkids18_non', 'w_biokids_out', 'w_othkids12', 'w_numkids0to5', 'FOOD__RECALL', 'w_milk', 'w_water', 'w_tea', 'w_juice', 'w_otherbev', 'w_matooke', 'w_roots', 'w_grains', 'w_beans', 'w_organ', 'w_meat', 'w_egg', 'w_fish', 'w_green', 'w_pump', 'w_otherfru', 'w_oil', 'w_sweet', 'w_othersolid', 'w_juiceplus', 'w_meats', 'w_otherfv', 'REPRODUCTIVE_HISTORY', 'w_biokids_hh', 'w_biokids_out2', 'w_biokids_num', 'w_biokids_num_alive', 'w_numkids_husb', 'w_pregnow', 'w_preg_mo', 'w_preg_learn', 'w_preg_want', 'w_preg_ac', 'w_preg_whynoac', 'w_preg_whynoac_oth', 'w_preg_ac_num', 'w_preg_ac_mo', 'w_hus

In [ ]:
sections_adults = [col for col in parent_df.columns if col.isupper()]
print({len(sections_adults)})
print(sections_adults)

{35}
['BL_WOMENS', 'DEMOGRAPHICS_ROSTER', 'FOOD__RECALL', 'REPRODUCTIVE_HISTORY', 'LAST_PREGNANCY', 'INFANT_CARE', 'SANITATION_PRACTICE', 'HEALTH_KNOWLEDGE', 'DECISION_MAKING', 'EXPENDITURE', 'ANTHRO_SURVEY', 'MENS__SURVEY', 'MENS_DECISION_MAKING', 'EXPENDITURE_MENS', 'ENDLINE_WOMENS', 'CURRENT_PREGNANCY', 'PAST_PREGNANCY', 'LAST_PREGNANCY_EL', 'FOOD_RECALL', 'REPRODUCTIVE_HEALTH', 'SANITATION__PRACTICE', 'HEALTH__KNOWLEDGE', 'DECISION_MAKING_EL', 'EXPENDITURE_EL', 'WOMENS_ANTHRO', 'MENS_SURVEY', 'MENS_HOUSEHOLD', 'MENS_FOOD_RECALL', 'MENS_FERTILITY', 'MENS_SANIT_PRACT', 'MENS_HEALTH_KNOWLEDGE', 'MENS_SANITATION', 'MENS_DECISION_MAKING_EL', 'MENS_EXPENDITURE_EL', 'ATTENDANCE']


In [ ]:

start_idx = parent_df.columns.get_loc('DECISION_MAKING')
end_idx = parent_df.columns.get_loc('EXPENDITURE')

decision_making_cols = parent_df.columns[start_idx + 1 : end_idx].tolist()

print({len(decision_making_cols)})

print("coulmns name :")
for col in decision_making_cols:
    print(col)

{184}
coulmns name :
w_husb
w_daily_bi
w_daily_who
w_daily_diffop
w_daily_express
w_daily_argue
w_daily_prevail
w_purch_bi
w_purch_who
w_purch_diffop
w_purch_express
w_purch_argue
w_purch_prevail
w_savespend_bi
w_savespend_who
w_savespend_diffop
w_savespend_express
w_savespend_argue
w_savespend_prevail
w_wcloth_bi
w_wcloth_who
w_wcloth_diffop
w_wcloth_express
w_wcloth_argue
w_wcloth_prevail
w_chealth_bi
w_chealth_who
w_chealth_diffop
w_chealth_express
w_chealth_argue
w_chealth_prevail
w_cfeed_bi
w_cfeed_who
w_cfeed_diffop
w_cfeed_express
w_cfeed_argue
w_cfeed_prevail
w_ceduc_bi
w_ceduc_who
w_ceduc_diffop
w_ceduc_express
w_ceduc_argue
w_ceduc_prevail
w_cschool_bi
w_cschool_who
w_cschool_diffop
w_cschool_express
w_cschool_argue
w_cschool_prevail
w_ccloth_bi
w_ccloth_who
w_ccloth_diffop
w_ccloth_express
w_ccloth_argue
w_ccloth_prevail
w_wmoney_bi
w_wmoney_who
w_wmoney_diffop
w_wmoney_express
w_wmoney_argue
w_wmoney_prevail
w_rights
w_rights_sdi
w_rights_di
w_equality
w_equality_sdi
w_equa

In [ ]:
display(parent_df[['w_all_times_no']].head(10))
min_val = parent_df['w_all_times_no'].min()
max_val = parent_df['w_all_times_no'].max()

print(f"min : {min_val}")
print(f"max : {max_val}")


# w_purch_bi
# w_purch_who

,w_all_times_no
0,1.0
1,0.0
2,0.0
3,1.0
4,0.0
5,0.0
6,0.0
7,0.0
8,1.0
9,0.0


min : 0.0
max : 1.0


In [ ]:
start_idx = parent_df.columns.get_loc('DECISION_MAKING')
end_idx = parent_df.columns.get_loc('EXPENDITURE')

decision_making_cols = parent_df.columns[start_idx + 1 : end_idx].tolist()

print(f"Total columns: {len(decision_making_cols)}")

print("Columns names and data types:")
for col in decision_making_cols:
    print(f"{col}: {parent_df[col].dtype}")

Total columns: 184
Columns names and data types:
w_husb: float64
w_daily_bi: float64
w_daily_who: float64
w_daily_diffop: float64
w_daily_express: float64
w_daily_argue: float64
w_daily_prevail: float64
w_purch_bi: float64
w_purch_who: float64
w_purch_diffop: float64
w_purch_express: float64
w_purch_argue: float64
w_purch_prevail: float64
w_savespend_bi: float64
w_savespend_who: float64
w_savespend_diffop: float64
w_savespend_express: float64
w_savespend_argue: float64
w_savespend_prevail: float64
w_wcloth_bi: float64
w_wcloth_who: float64
w_wcloth_diffop: float64
w_wcloth_express: float64
w_wcloth_argue: float64
w_wcloth_prevail: float64
w_chealth_bi: float64
w_chealth_who: float64
w_chealth_diffop: float64
w_chealth_express: float64
w_chealth_argue: float64
w_chealth_prevail: float64
w_cfeed_bi: float64
w_cfeed_who: float64
w_cfeed_diffop: float64
w_cfeed_express: float64
w_cfeed_argue: float64
w_cfeed_prevail: float64
w_ceduc_bi: float64
w_ceduc_who: float64
w_ceduc_diffop: float64


### EXPENDITURE

In [ ]:

start_idx = parent_df.columns.get_loc('EXPENDITURE')
end_idx = parent_df.columns.get_loc('ANTHRO_SURVEY')

decision_making_cols = parent_df.columns[start_idx + 1 : end_idx].tolist()

print({len(decision_making_cols)})

print("coulmns name :")
for col in decision_making_cols:
    print(col)

{34}
coulmns name :
w_plan_hhexpend6
w_cattle_own
w_goat_own
w_chx_own
w_stove_own
w_wattnk_own
w_bed_own
w_lant_own
w_tv_own
w_radio_own
w_gen_own
w_cell_own
w_bic_own
w_moto_own
w_car_own
w_sug_own
w_par_own
w_soap_own
w_tea_own
w_milk_own
w_mtfs_own
w_salt_own
w_transp_own
w_air_own
w_last6m_rice
w_last6m_ccloth
w_last6m_fun
w_last6m_health
w_last6m_schfee
w_last6m_supp
w_last6m_wedd
w_last6m_wcloth
w_did_womens
w_incomplete_w


In [ ]:

display(parent_df[['men_away_hh']].head(10))
min_val = parent_df['men_away_hh'].min()
max_val = parent_df['men_away_hh'].max()

print(f"min : {min_val}")
print(f"max : {max_val}")


# w_purch_bi
# w_purch_who

,men_away_hh
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0


min : 0
max : 0


### ANTHRO_SURVEY

In [ ]:

start_idx = parent_df.columns.get_loc('ANTHRO_SURVEY')
end_idx = parent_df.columns.get_loc('MENS__SURVEY')

decision_making_cols = parent_df.columns[start_idx + 1 : end_idx].tolist()

# طباعة العدد للتأكد (يجب أن يكون 185)
print({len(decision_making_cols)})

print("coulmns name :")
for col in decision_making_cols:
    print(col)

{38}
coulmns name :
a_enum_id
a_date
anthro_date
a_time
w_weight
w_weight_flag
w_weight_flagwhy
w_height
w_height_flag
w_height_flagwhy
w_anaemia_12
w_anaemia_12_flag
w_anaemia_11
w_anaemia_8
w_bmi
w_bmi_18h
w_bmi_18h_flag
w_bmi_17
w_bmi_16
w_muac
w_muac_flag
w_muac_flagwhy
w_muac_preg23
w_muac_221
w_muac_221_flag
w_muac_214
w_hblevel
w_hblevel_flag
w_hblevel_flagwhy
a_numkids
dum_a
incomplete_a
incomplete_survey
survey_status
survey_date
men_away_hh
ineligible
num_elig_c


In [ ]:

parent_df['m_date'] = pd.to_datetime(parent_df['m_date'], errors='coerce')

min_val = parent_df['m_date'].min()
max_val = parent_df['m_date'].max()

print(f"  (min): {min_val.date}")
print(f"  (max): {max_val.date}")

  (min): <bound method Timestamp.date of Timestamp('2012-10-02 00:00:00')>
  (max): <bound method Timestamp.date of Timestamp('2013-01-15 00:00:00')>


### MENS__SURVEY

In [ ]:
start_idx = parent_df.columns.get_loc('MENS__SURVEY')
end_idx = parent_df.columns.get_loc('MENS_DECISION_MAKING')

decision_making_cols = parent_df.columns[start_idx + 1 : end_idx].tolist()

print({len(decision_making_cols)})

print("coulmns name :")
for col in decision_making_cols:
    print(col)

{118}
coulmns name :
m_village
m_villagepid
m_enum_id
m_enum_notes
m_district
m_date
m_age
m_educ
m_inc
m_milk
m_water
m_tea
m_juice
m_otherbev
m_matooke
m_cas
m_rice
m_beans
m_organ
m_meat
m_egg
m_fish
m_green
m_pump
m_otherfru
m_oil
m_sweet
m_othersolid
m_kidsunder2
m_timekids
m_preg_now
m_preg_now_ac
m_preg_past2_ac
m_preg_comp
m_labr_present
m_assist
m_assist_oth
m_fp_decide
m_fp_talk
m_hus_ideal_fert_t
m_hus_ideal_fert_b
m_hus_ideal_fert_g
m_hus_ideal_fert_e
m_nextpreg_when
m_nextpreg_units
m_op_birthspace
m_op_birthspace_u
m_soap_day
m_wash_bath
m_soap_toilet
m_soap_toilet_bi
m_wash_meal
m_soap_meal
m_soap_meal_bi
m_mal_cause
m_mal_cause_bi
m_mal_cause_bi2
m_mal_cause_oth
m_diar_cause
m_diar_cause_bi
m_diar_cause_bi2
m_cstrum_feed_bi
m_cstrum_no_why
m_bfeed24
m_grow_lackfood
m_grow_lackfood_sw
m_grow_lackfood_vm
m_grow_lackdiet
m_grow_lackdiet_sw
m_grow_lackdiet_vm
m_grow_witch
m_grow_witch_lt
m_grow_witch_vl
m_grow_sick
m_grow_sick_sw
m_grow_sick_vm
m_kwash_cause
m_kwash_cause_b

### MENS_DECISION_MAKING (baseline)

In [ ]:
who_map= {
    1 : 'Wife',
    2 : 'Husband',
    3 : 'Both together',
    4 : 'Other HH member',
    5 : 'Someone outside the HH',
    6 : 'Not applicable'
}

prevail_map= {
    1 : 'Yourself',
    2 : 'Your spouse',
    3 : 'A third person',
    4 : 'Respondent finds another way',
}

agdisag_map={
    1: 'Strongly Agree',
    2: 'Agree',
    3: 'Disagree',
    4: 'Strongly Disagree',
    5: 'N/A'
}

In [ ]:
mens_decision_making = parent_all_sections['MENS_DECISION_MAKING']
mens_decision_making.columns

Index(['m_daily_bi', 'm_daily_who', 'm_daily_diffop', 'm_daily_express',
       'm_daily_argue', 'm_daily_prevail', 'm_chealth_bi', 'm_chealth_who',
       'm_chealth_diffop', 'm_chealth_express', 'm_chealth_argue',
       'm_chealth_prevail', 'm_cfeed_bi', 'm_cfeed_who', 'm_cfeed_diffop',
       'm_cfeed_express', 'm_cfeed_argue', 'm_cfeed_prevail', 'm_wmoney_bi',
       'm_wmoney_who', 'm_wmoney_diffop', 'm_wmoney_express', 'm_wmoney_argue',
       'm_wmoney_prevail', 'm_rights', 'm_rights_sdi', 'm_rights_di',
       'm_equality', 'm_equality_sdi', 'm_equality_di', 'm_obey', 'm_obey_sdi',
       'm_obey_di', 'm_final_say', 'm_final_say_sag', 'm_final_say_ag',
       'm_oblig_sex', 'm_oblig_sex_sdi', 'm_oblig_sex_di', 'm_tol_viol',
       'm_tol_viol_nsag', 'm_tol_viol_sdi', 'm_tol_viol_di', 'm_deserv',
       'm_deserv_sdi', 'm_deserv_di', 'm_hit_argues', 'm_hit_argues_sdi',
       'm_hit_argues_di', 'm_daily_who_wifedec', 'm_daily_who_husbdec',
       'm_daily_who_bothdec', 'm_daily

In [ ]:
extract_information(mens_decision_making,'m_daily_bi')

m_daily_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
140
2.539912917271408
m_daily_bi
0.0    4238
1.0    1134
Name: count, dtype: int64
        m_daily_bi
count  5372.000000
mean      0.211095
std       0.408123
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
mens_decision_making['m_daily_who'] = mens_decision_making['m_daily_who'].map(who_map)
mens_decision_making['m_daily_who'] = mens_decision_making['m_daily_who'].astype('category')

extract_information(mens_decision_making,'m_daily_who')

m_daily_who    category
dtype: object
0             Both together
1                      Wife
2                       NaN
3                   Husband
4           Other HH member
5    Someone outside the HH
dtype: category
Categories (5, object): ['Both together', 'Husband', 'Other HH member', 'Someone outside the HH',
                         'Wife']
140
2.539912917271408
m_daily_who
Wife                      4217
Both together              815
Husband                    319
Other HH member             15
Someone outside the HH       6
Name: count, dtype: int64
       m_daily_who
count         5372
unique           5
top           Wife
freq          4217


In [ ]:
extract_information(mens_decision_making,'m_daily_diffop')

m_daily_diffop    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
4395
79.73512336719884
m_daily_diffop
0.0    766
1.0    351
Name: count, dtype: int64
       m_daily_diffop
count     1117.000000
mean         0.314235
std          0.464418
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


In [ ]:
extract_information(mens_decision_making,'m_daily_argue')

m_daily_argue    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
3988
72.3512336719884
m_daily_argue
1.0    862
0.0    662
Name: count, dtype: int64
       m_daily_argue
count    1524.000000
mean        0.565617
std         0.495838
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000


In [ ]:
mens_decision_making['m_daily_prevail'] = mens_decision_making['m_daily_prevail'].map(prevail_map)
mens_decision_making['m_daily_prevail'] = mens_decision_making['m_daily_prevail'].astype('category')

extract_information(mens_decision_making,'m_daily_prevail')

m_daily_prevail    category
dtype: object
0                             NaN
1                        Yourself
2                     Your spouse
3    Respondent finds another way
4                  A third person
dtype: category
Categories (4, object): ['A third person', 'Respondent finds another way', 'Your spouse', 'Yourself']
4651
84.37953555878084
m_daily_prevail
Yourself                        639
Your spouse                     195
Respondent finds another way     25
A third person                    2
Name: count, dtype: int64
       m_daily_prevail
count              861
unique               4
top           Yourself
freq               639


In [ ]:
extract_information(mens_decision_making,'m_chealth_bi')

m_chealth_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
157
2.8483309143686504
m_chealth_bi
0.0    4319
1.0    1036
Name: count, dtype: int64
       m_chealth_bi
count   5355.000000
mean       0.193464
std        0.395050
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


In [ ]:
mens_decision_making['m_chealth_who'] = mens_decision_making['m_chealth_who'].map(who_map)
mens_decision_making['m_chealth_who'] = mens_decision_making['m_chealth_who'].astype('category')

extract_information(mens_decision_making,'m_chealth_who')

m_chealth_who    category
dtype: object
0             Both together
1                   Husband
2                      Wife
3                       NaN
4    Someone outside the HH
5           Other HH member
dtype: category
Categories (5, object): ['Both together', 'Husband', 'Other HH member', 'Someone outside the HH',
                         'Wife']
157
2.8483309143686504
m_chealth_who
Wife                      4310
Both together              896
Husband                    140
Other HH member              7
Someone outside the HH       2
Name: count, dtype: int64
       m_chealth_who
count           5355
unique             5
top             Wife
freq            4310


In [ ]:
extract_information(mens_decision_making,'m_chealth_diffop')

m_chealth_diffop    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
4497
81.58563134978229
m_chealth_diffop
0.0    868
1.0    147
Name: count, dtype: int64
       m_chealth_diffop
count       1015.000000
mean           0.144828
std            0.352100
min            0.000000
25%            0.000000
50%            0.000000
75%            0.000000
max            1.000000


In [ ]:
extract_information(mens_decision_making,'m_chealth_argue')

m_chealth_argue    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
4847
87.93541364296081
m_chealth_argue
0.0    367
1.0    298
Name: count, dtype: int64
       m_chealth_argue
count       665.000000
mean          0.448120
std           0.497676
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000


In [ ]:
mens_decision_making['m_chealth_prevail'] = mens_decision_making['m_chealth_prevail'].map(prevail_map)
mens_decision_making['m_chealth_prevail'] = mens_decision_making['m_chealth_prevail'].astype('category')

extract_information(mens_decision_making,'m_chealth_prevail')

m_chealth_prevail    category
dtype: object
0                             NaN
1                     Your spouse
2                        Yourself
3    Respondent finds another way
dtype: category
Categories (3, object): ['Respondent finds another way', 'Your spouse', 'Yourself']
5214
94.59361393323658
m_chealth_prevail
Yourself                        205
Your spouse                      86
Respondent finds another way      7
Name: count, dtype: int64
       m_chealth_prevail
count                298
unique                 3
top             Yourself
freq                 205


In [ ]:
extract_information(mens_decision_making,'m_cfeed_bi')

m_cfeed_bi    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
159
2.8846153846153846
m_cfeed_bi
1.0    4997
0.0     356
Name: count, dtype: int64
        m_cfeed_bi
count  5353.000000
mean      0.933495
std       0.249186
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
mens_decision_making['m_cfeed_who'] = mens_decision_making['m_cfeed_who'].map(who_map)
mens_decision_making['m_cfeed_who'] = mens_decision_making['m_cfeed_who'].astype('category')

extract_information(mens_decision_making,'m_cfeed_who')

m_cfeed_who    category
dtype: object
0            Husband
1                NaN
2               Wife
3      Both together
4    Other HH member
dtype: category
Categories (4, object): ['Both together', 'Husband', 'Other HH member', 'Wife']
159
2.8846153846153846
m_cfeed_who
Husband            4657
Wife                344
Both together       340
Other HH member      12
Name: count, dtype: int64
       m_cfeed_who
count         5353
unique           4
top        Husband
freq          4657


In [ ]:
extract_information(mens_decision_making,'m_cfeed_diffop')

m_cfeed_diffop    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
516
9.361393323657474
m_cfeed_diffop
0.0    4368
1.0     628
Name: count, dtype: int64
       m_cfeed_diffop
count     4996.000000
mean         0.125701
std          0.331545
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


In [ ]:
extract_information(mens_decision_making,'m_cfeed_argue')

m_cfeed_argue    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
5057
91.74528301886792
m_cfeed_argue
0.0    285
1.0    170
Name: count, dtype: int64
       m_cfeed_argue
count     455.000000
mean        0.373626
std         0.484299
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000


In [ ]:
mens_decision_making['m_cfeed_prevail'] = mens_decision_making['m_cfeed_prevail'].map(prevail_map)
mens_decision_making['m_cfeed_prevail'] = mens_decision_making['m_cfeed_prevail'].astype('category')

extract_information(mens_decision_making,'m_cfeed_prevail')

m_cfeed_prevail    category
dtype: object
0                             NaN
1                     Your spouse
2                        Yourself
3    Respondent finds another way
dtype: category
Categories (3, object): ['Respondent finds another way', 'Your spouse', 'Yourself']
5342
96.91582002902757
m_cfeed_prevail
Yourself                        102
Your spouse                      65
Respondent finds another way      3
Name: count, dtype: int64
       m_cfeed_prevail
count              170
unique               3
top           Yourself
freq               102


In [ ]:
extract_information(mens_decision_making,'m_wmoney_bi')

m_wmoney_bi    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
251
4.553701015965167
m_wmoney_bi
1.0    4691
0.0     570
Name: count, dtype: int64
       m_wmoney_bi
count  5261.000000
mean      0.891656
std       0.310844
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
mens_decision_making['m_wmoney_who'] = mens_decision_making['m_wmoney_who'].map(who_map)
mens_decision_making['m_wmoney_who'] = mens_decision_making['m_wmoney_who'].astype('category')

extract_information(mens_decision_making,'m_wmoney_who')

m_wmoney_who    category
dtype: object
0            Husband
1      Both together
2               Wife
3                NaN
4    Other HH member
dtype: category
Categories (4, object): ['Both together', 'Husband', 'Other HH member', 'Wife']
251
4.553701015965167
m_wmoney_who
Husband            3513
Both together      1178
Wife                569
Other HH member       1
Name: count, dtype: int64
       m_wmoney_who
count          5261
unique            4
top         Husband
freq           3513


In [ ]:
extract_information(mens_decision_making,'m_wmoney_diffop')

m_wmoney_diffop    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
1338
24.274310595065312
m_wmoney_diffop
0.0    3127
1.0    1047
Name: count, dtype: int64
       m_wmoney_diffop
count      4174.000000
mean          0.250839
std           0.433548
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(mens_decision_making,'m_wmoney_argue')

m_wmoney_argue    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
4529
82.16618287373004
m_wmoney_argue
0.0    541
1.0    442
Name: count, dtype: int64
       m_wmoney_argue
count      983.000000
mean         0.449644
std          0.497711
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


In [ ]:
mens_decision_making['m_wmoney_prevail'] = mens_decision_making['m_wmoney_prevail'].map(prevail_map)
mens_decision_making['m_wmoney_prevail'] = mens_decision_making['m_wmoney_prevail'].astype('category')

extract_information(mens_decision_making,'m_wmoney_prevail')

m_wmoney_prevail    category
dtype: object
0                             NaN
1                        Yourself
2                     Your spouse
3    Respondent finds another way
4                  A third person
dtype: category
Categories (4, object): ['A third person', 'Respondent finds another way', 'Your spouse', 'Yourself']
5070
91.98113207547169
m_wmoney_prevail
Your spouse                     212
Yourself                        210
Respondent finds another way     19
A third person                    1
Name: count, dtype: int64
       m_wmoney_prevail
count               442
unique                4
top         Your spouse
freq                212


In [ ]:
mens_decision_making['m_rights'] = mens_decision_making['m_rights'].map(agdisag_map)
mens_decision_making['m_rights'] = mens_decision_making['m_rights'].astype('category')

extract_information(mens_decision_making,'m_rights')

m_rights    category
dtype: object
0    Strongly Disagree
1             Disagree
2       Strongly Agree
3                Agree
4                  NaN
dtype: category
Categories (4, object): ['Agree', 'Disagree', 'Strongly Agree', 'Strongly Disagree']
249
4.517416545718433
m_rights
Strongly Disagree    2432
Strongly Agree       1526
Disagree              706
Agree                 599
Name: count, dtype: int64
                 m_rights
count                5263
unique                  4
top     Strongly Disagree
freq                 2432


In [ ]:
mens_decision_making['m_equality'] = mens_decision_making['m_equality'].map(agdisag_map)
mens_decision_making['m_equality'] = mens_decision_making['m_equality'].astype('category')

extract_information(mens_decision_making,'m_equality')

m_equality    category
dtype: object
0       Strongly Agree
1             Disagree
2    Strongly Disagree
3                Agree
4                  NaN
dtype: category
Categories (4, object): ['Agree', 'Disagree', 'Strongly Agree', 'Strongly Disagree']
257
4.662554426705371
m_equality
Strongly Agree       2133
Agree                1281
Strongly Disagree    1000
Disagree              841
Name: count, dtype: int64
            m_equality
count             5255
unique               4
top     Strongly Agree
freq              2133


In [ ]:
mens_decision_making['m_obey'] = mens_decision_making['m_obey'].map(agdisag_map)
mens_decision_making['m_obey'] = mens_decision_making['m_obey'].astype('category')

extract_information(mens_decision_making,'m_obey')

m_obey    category
dtype: object
0       Strongly Agree
1    Strongly Disagree
2             Disagree
3                Agree
4                  NaN
dtype: category
Categories (4, object): ['Agree', 'Disagree', 'Strongly Agree', 'Strongly Disagree']
248
4.499274310595065
m_obey
Strongly Agree       3030
Strongly Disagree     802
Disagree              770
Agree                 662
Name: count, dtype: int64
                m_obey
count             5264
unique               4
top     Strongly Agree
freq              3030


In [ ]:
mens_decision_making['m_final_say'] = mens_decision_making['m_final_say'].map(agdisag_map)
mens_decision_making['m_final_say'] = mens_decision_making['m_final_say'].astype('category')

extract_information(mens_decision_making,'m_final_say')

m_final_say    category
dtype: object
0    Strongly Disagree
1       Strongly Agree
2             Disagree
3                Agree
4                  NaN
dtype: category
Categories (4, object): ['Agree', 'Disagree', 'Strongly Agree', 'Strongly Disagree']
250
4.535558780841799
m_final_say
Strongly Agree       2266
Strongly Disagree    1937
Disagree              568
Agree                 491
Name: count, dtype: int64
           m_final_say
count             5262
unique               4
top     Strongly Agree
freq              2266


In [ ]:
mens_decision_making['m_oblig_sex'] = mens_decision_making['m_oblig_sex'].map(agdisag_map)
mens_decision_making['m_oblig_sex'] = mens_decision_making['m_oblig_sex'].astype('category')

extract_information(mens_decision_making,'m_oblig_sex')

m_oblig_sex    category
dtype: object
0    Strongly Disagree
1       Strongly Agree
2             Disagree
3                  NaN
4                Agree
dtype: category
Categories (4, object): ['Agree', 'Disagree', 'Strongly Agree', 'Strongly Disagree']
141
2.558055152394775
m_oblig_sex
Strongly Agree       1913
Strongly Disagree    1788
Disagree             1067
Agree                 603
Name: count, dtype: int64
           m_oblig_sex
count             5371
unique               4
top     Strongly Agree
freq              1913


In [ ]:
mens_decision_making['m_tol_viol'] = mens_decision_making['m_tol_viol'].map(agdisag_map)
mens_decision_making['m_tol_viol'] = mens_decision_making['m_tol_viol'].astype('category')

extract_information(mens_decision_making,'m_tol_viol')

m_tol_viol    category
dtype: object
0       Strongly Agree
1                Agree
2                  NaN
3             Disagree
4    Strongly Disagree
dtype: category
Categories (4, object): ['Agree', 'Disagree', 'Strongly Agree', 'Strongly Disagree']
142
2.576197387518142
m_tol_viol
Strongly Agree       3893
Agree                 922
Strongly Disagree     338
Disagree              217
Name: count, dtype: int64
            m_tol_viol
count             5370
unique               4
top     Strongly Agree
freq              3893


In [ ]:
mens_decision_making['m_deserv'] = mens_decision_making['m_deserv'].map(agdisag_map)
mens_decision_making['m_deserv'] = mens_decision_making['m_deserv'].astype('category')

extract_information(mens_decision_making,'m_deserv')

m_deserv    category
dtype: object
0    Strongly Disagree
1             Disagree
2                Agree
3       Strongly Agree
4                  NaN
dtype: category
Categories (4, object): ['Agree', 'Disagree', 'Strongly Agree', 'Strongly Disagree']
143
2.5943396226415096
m_deserv
Strongly Disagree    3648
Agree                 642
Disagree              592
Strongly Agree        487
Name: count, dtype: int64
                 m_deserv
count                5369
unique                  4
top     Strongly Disagree
freq                 3648


In [ ]:
mens_decision_making['m_hit_argues'] = mens_decision_making['m_hit_argues'].map(agdisag_map)
mens_decision_making['m_hit_argues'] = mens_decision_making['m_hit_argues'].astype('category')

extract_information(mens_decision_making,'m_hit_argues')

m_hit_argues    category
dtype: object
0             Disagree
1    Strongly Disagree
2                Agree
3       Strongly Agree
4                  NaN
dtype: category
Categories (4, object): ['Agree', 'Disagree', 'Strongly Agree', 'Strongly Disagree']
141
2.558055152394775
m_hit_argues
Strongly Disagree    3375
Disagree              912
Strongly Agree        558
Agree                 526
Name: count, dtype: int64
             m_hit_argues
count                5371
unique                  4
top     Strongly Disagree
freq                 3375


### EXPENDITURE_MENS (baseline)

In [ ]:
mens_expenditure = parent_all_sections['EXPENDITURE_MENS']
mens_expenditure.columns

Index(['m_buy_hhsup', 'm_save', 'm_saved_what', 'm_last7d_tobacco',
       'm_last7d_alcohol', 'm_last1m_snacks', 'm_last1m_elec',
       'm_last1m_cloth', 'm_plan_hhexpend6', 'm_did_mens', 'm_incomplete_m'],
      dtype='object')

In [ ]:
extract_information(mens_expenditure,'m_save')

m_save    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
140
2.539912917271408
m_save
1.0    4459
0.0     913
Name: count, dtype: int64
            m_save
count  5372.000000
mean      0.830045
std       0.375629
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:

extract_information(mens_expenditure,'m_saved_what')

m_saved_what    object
dtype: object
0                           4=Buy land;6=Pay school fees
1                             2=Buy Livestock;4=Buy land
2                  1=Use for emergency;6=Pay school fees
3           2=Buy Livestock;4=Buy land;6=Pay school fees
4      1=Use for emergency;2=Buy Livestock;4=Buy land...
                             ...                        
132    3=Send to my family/other;5=Put it into busine...
133    1=Use for emergency;4=Buy land;5=Put it into b...
134                            3=Send to my family/other
135    1=Use for emergency;4=Buy land;7=Buy household...
136    1=Use for emergency;2=Buy Livestock;3=Send to ...
Length: 137, dtype: object
1055
19.140058055152394
m_saved_what
2=Buy Livestock                                                                                                                                 440
2=Buy Livestock;4=Buy land                                                                                                  

In [ ]:
extract_information(mens_expenditure,'m_last7d_tobacco')

m_last7d_tobacco    float64
dtype: object
0          0.0
1        400.0
2       2100.0
3       1500.0
4       1000.0
        ...   
56    130000.0
57     17500.0
58      9000.0
59      3200.0
60     50000.0
Length: 61, dtype: float64
140
2.539912917271408
m_last7d_tobacco
0.0         4347
1000.0       201
500.0        139
2000.0       109
3000.0        56
5000.0        49
300.0         47
1500.0        41
1400.0        37
200.0         37
700.0         31
400.0         29
2100.0        26
10000.0       23
4000.0        21
3500.0        19
600.0         17
7000.0        15
2500.0        13
800.0         10
6000.0         9
20000.0        9
100.0          9
2800.0         7
8000.0         7
14000.0        5
1200.0         4
900.0          4
30000.0        4
1800.0         4
10500.0        3
3600.0         3
2400.0         3
4200.0         3
250.0          2
15000.0        2
5400.0         2
21000.0        2
3150.0         2
1700.0         1
28000.0        1
1100.0         1
5600.0       

In [ ]:
extract_information(mens_expenditure,'m_last1m_elec')

m_last1m_elec    float64
dtype: object
0         4200.0
1         4500.0
2         1200.0
3         5000.0
4         6000.0
         ...    
299      24300.0
300      54400.0
301    1000000.0
302      81000.0
303      31400.0
Length: 304, dtype: float64
140
2.539912917271408
m_last1m_elec
0.0          608
5000.0       468
10000.0      393
3000.0       282
2000.0       253
            ... 
24300.0        1
54400.0        1
1000000.0      1
81000.0        1
31400.0        1
Name: count, Length: 303, dtype: int64
        m_last1m_elec
count     5372.000000
mean     10492.218727
std      26975.808720
min          0.000000
25%       2000.000000
50%       5000.000000
75%      10000.000000
max    1000000.000000


In [ ]:
extract_information(mens_expenditure,'m_plan_hhexpend6')

m_plan_hhexpend6    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
140
2.539912917271408
m_plan_hhexpend6
1.0    4049
0.0    1323
Name: count, dtype: int64
       m_plan_hhexpend6
count       5372.000000
mean           0.753723
std            0.430882
min            0.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            1.000000


### ENDLINE_WOMENS

In [ ]:
endline_womens = parent_all_sections['ENDLINE_WOMENS']
endline_womens.columns

Index(['Estatus_w', 'Estatus_am', 'Eincomplete_w', 'Ew_survey_date',
       'Ew_moved', 'Ew_district', 'Ew_subcounty', 'Ew_age', 'Ew_still_married',
       'Ew_marital', 'Ew_other_wives', 'Ew_num_wives', 'Ew_num_adults'],
      dtype='object')

In [ ]:
extract_information(endline_womens,'Ew_age')

Ew_age    int64
dtype: object
0     40
1     35
2     39
3     28
4     36
5     43
6     32
7     38
8     37
9     27
10    21
11    22
12    31
13    41
14    24
15    30
16    25
17    26
18    29
19    34
20    33
21    42
22    47
23    44
24    23
25    20
26    19
27    49
28    45
29    46
30    18
31    51
32    50
33    48
34    64
35    60
36    14
37    17
dtype: int64
0
0.0
Ew_age
31    419
23    366
29    363
26    363
24    363
21    355
25    343
27    307
22    259
33    249
28    241
36    206
30    201
20    195
32    154
34    144
37    142
35    135
39    132
38     99
19     97
41     94
40     72
43     52
44     45
42     32
45     20
18     19
46     18
47      7
49      6
48      5
51      3
50      2
64      1
60      1
14      1
17      1
Name: count, dtype: int64
            Ew_age
count  5512.000000
mean     28.774129
std       6.334930
min      14.000000
25%      24.000000
50%      28.000000
75%      33.000000
max      64.000000


In [ ]:

extract_information(endline_womens,'Ew_still_married')

Ew_still_married    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_still_married
1.0    5155
0.0     133
Name: count, dtype: int64
       Ew_still_married
count       5288.000000
mean           0.974849
std            0.156599
min            0.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            1.000000


In [ ]:
extract_information(endline_womens,'Ew_marital')

Ew_marital    float64
dtype: object
0    NaN
1    1.0
2    4.0
3    2.0
4    3.0
dtype: float64
5379
97.58708272859216
Ew_marital
1.0    82
4.0    36
2.0    12
3.0     3
Name: count, dtype: int64
       Ew_marital
count  133.000000
mean     1.947368
std      1.316077
min      1.000000
25%      1.000000
50%      1.000000
75%      4.000000
max      4.000000


In [ ]:
extract_information(endline_womens,'Ew_other_wives')

Ew_other_wives    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
5497
99.72786647314949
Ew_other_wives
0.0    11
1.0     4
Name: count, dtype: int64
       Ew_other_wives
count       15.000000
mean         0.266667
std          0.457738
min          0.000000
25%          0.000000
50%          0.000000
75%          0.500000
max          1.000000


In [ ]:
extract_information(endline_womens,'Ew_num_wives')

Ew_num_wives    float64
dtype: object
0    NaN
1    1.0
2    2.0
3    5.0
dtype: float64
5508
99.92743105950653
Ew_num_wives
1.0    2
2.0    1
5.0    1
Name: count, dtype: int64
       Ew_num_wives
count      4.000000
mean       2.250000
std        1.892969
min        1.000000
25%        1.000000
50%        1.500000
75%        2.750000
max        5.000000


In [ ]:
extract_information(endline_womens,'Ew_num_adults')

Ew_num_adults    float64
dtype: object
0      2.0
1      4.0
2      3.0
3      6.0
4      1.0
5      5.0
6      NaN
7      7.0
8     18.0
9      0.0
10     9.0
11     8.0
12    11.0
13    10.0
dtype: float64
224
4.063860667634253
Ew_num_adults
2.0     4054
3.0      551
1.0      330
4.0      239
5.0       72
6.0       25
7.0        9
0.0        2
8.0        2
18.0       1
9.0        1
11.0       1
10.0       1
Name: count, dtype: int64
       Ew_num_adults
count    5288.000000
mean        2.209531
std         0.802851
min         0.000000
25%         2.000000
50%         2.000000
75%         2.000000
max        18.000000


### CURRENT_PREGNANCY

In [ ]:
want_preg_map = {
    1: 'in same time',
    2: 'later',
    3: 'never'
}

In [ ]:
current_pregnancy = parent_all_sections['CURRENT_PREGNANCY']
current_pregnancy.columns

Index(['Ew_pregnow', 'Ew_pregnow_mo', 'Ew_bl_husb_father', 'Ew_preg_learn',
       'Ew_preg_want', 'Ew_preg_ac', 'Ew_preg_ac_num', 'Ew_preg_ac_mo',
       'Ew_hus_go_ac', 'Ew_hus_ac_advice', 'Ew_hus_ac_where', 'Ew_hus_ac_nutr',
       'Ew_hus_ac_deliver', 'Ew_hus_ac_kids', 'Ew_hus_ac_whenkid',
       'Ew_preg_mal', 'Ew_preg_comp_list', 'Ew_hus_aware_comp',
       'Ew_hus_assist_comp', 'Ew_hus_assist_how', 'Ew_plan_birth',
       'Ew_disc_plan_who', 'Ew_preg_foods', 'Ew_preg_foods_list'],
      dtype='object')

In [ ]:
extract_information(current_pregnancy,'Ew_pregnow')

Ew_pregnow    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_pregnow
0.0    4508
1.0     780
Name: count, dtype: int64
        Ew_pregnow
count  5288.000000
mean      0.147504
std       0.354641
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:

extract_information(current_pregnancy,'Ew_pregnow_mo')

Ew_pregnow_mo    float64
dtype: object
0    NaN
1    1.0
2    7.0
3    8.0
4    2.0
5    3.0
6    6.0
7    4.0
8    5.0
9    9.0
dtype: float64
4732
85.84905660377359
Ew_pregnow_mo
4.0    108
6.0    107
3.0    103
5.0     93
8.0     91
7.0     90
2.0     83
9.0     57
1.0     48
Name: count, dtype: int64
       Ew_pregnow_mo
count     780.000000
mean        5.042308
std         2.330686
min         1.000000
25%         3.000000
50%         5.000000
75%         7.000000
max         9.000000


In [ ]:
extract_information(current_pregnancy,'Ew_preg_learn')

Ew_preg_learn    float64
dtype: object
0    NaN
1    1.0
2    2.0
3    3.0
4    4.0
5    5.0
6    0.0
7    6.0
8    7.0
dtype: float64
4732
85.84905660377359
Ew_preg_learn
1.0    455
2.0    208
3.0     76
4.0     28
0.0      6
5.0      5
6.0      1
7.0      1
Name: count, dtype: int64
       Ew_preg_learn
count     780.000000
mean        1.601282
std         0.898579
min         0.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         7.000000


In [ ]:
current_pregnancy['Ew_preg_want'] = current_pregnancy['Ew_preg_want'].map(want_preg_map)
current_pregnancy['Ew_preg_want'] = current_pregnancy['Ew_preg_want'].astype('category')

extract_information(current_pregnancy,'Ew_preg_want')

Ew_preg_want    category
dtype: object
0             NaN
1    in same time
2           later
3           never
dtype: category
Categories (3, object): ['in same time', 'later', 'never']
4732
85.84905660377359
Ew_preg_want
later           382
in same time    284
never           114
Name: count, dtype: int64
       Ew_preg_want
count           780
unique            3
top           later
freq            382


In [ ]:
extract_information(current_pregnancy,'Ew_preg_ac')

Ew_preg_ac    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
4732
85.84905660377359
Ew_preg_ac
1.0    463
0.0    317
Name: count, dtype: int64
       Ew_preg_ac
count  780.000000
mean     0.593590
std      0.491478
min      0.000000
25%      0.000000
50%      1.000000
75%      1.000000
max      1.000000


In [ ]:
extract_information(current_pregnancy,'Ew_preg_ac_num')

Ew_preg_ac_num    float64
dtype: object
0     NaN
1     1.0
2     2.0
3     4.0
4     3.0
5     5.0
6    13.0
7     6.0
dtype: float64
5049
91.60014513788099
Ew_preg_ac_num
1.0     172
2.0     122
3.0      96
4.0      58
5.0      11
6.0       3
13.0      1
Name: count, dtype: int64
       Ew_preg_ac_num
count      463.000000
mean         2.207343
std          1.272499
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         13.000000


In [ ]:
extract_information(current_pregnancy,'Ew_preg_ac_mo')

Ew_preg_ac_mo    float64
dtype: object
0    NaN
1    5.0
2    6.0
3    1.0
4    3.0
5    4.0
6    2.0
7    7.0
8    8.0
dtype: float64
5049
91.60014513788099
Ew_preg_ac_mo
3.0    120
4.0    104
5.0     80
2.0     66
6.0     41
1.0     32
7.0     18
8.0      2
Name: count, dtype: int64
       Ew_preg_ac_mo
count     463.000000
mean        3.732181
std         1.526668
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         8.000000


In [ ]:
extract_information(current_pregnancy,'Ew_preg_foods')

Ew_preg_foods    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
4732
85.84905660377359
Ew_preg_foods
1.0    510
0.0    270
Name: count, dtype: int64
       Ew_preg_foods
count     780.000000
mean        0.653846
std         0.476048
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000


In [ ]:
extract_information(current_pregnancy,'Ew_preg_foods_list')

Ew_preg_foods_list    object
dtype: object
0           NaN
1             2
2         3 5 8
3           777
4      6 11 777
         ...   
151     1 7 8 9
152    9 11 777
153       6 7 9
154         7 9
155    5 10 777
Length: 156, dtype: object
5002
90.74746008708273
Ew_preg_foods_list
6 11        20
777         19
11          18
1 8         18
1           17
            ..
1 7 8 9      1
9 11 777     1
6 7 9        1
7 9          1
5 10 777     1
Name: count, Length: 155, dtype: int64
       Ew_preg_foods_list
count                 510
unique                155
top                  6 11
freq                   20


In [ ]:
extract_information(current_pregnancy,'Ew_preg_ac_num')

Ew_preg_ac_num    float64
dtype: object
0     NaN
1     1.0
2     2.0
3     4.0
4     3.0
5     5.0
6    13.0
7     6.0
dtype: float64
5049
91.60014513788099
Ew_preg_ac_num
1.0     172
2.0     122
3.0      96
4.0      58
5.0      11
6.0       3
13.0      1
Name: count, dtype: int64
       Ew_preg_ac_num
count      463.000000
mean         2.207343
std          1.272499
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         13.000000


### PAST_PREGNANCY

In [ ]:
past_pregnancy = parent_all_sections['PAST_PREGNANCY']
past_pregnancy.columns

Index(['Ew_pregpast2', 'Ew_preg_past2_notcurr', 'Ew_pregpast2_num'], dtype='object')

In [ ]:
extract_information(past_pregnancy,'Ew_pregpast2')

Ew_pregpast2    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
1004
18.21480406386067
Ew_pregpast2
1.0    2991
0.0    1517
Name: count, dtype: int64
       Ew_pregpast2
count   4508.000000
mean       0.663487
std        0.472569
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000


In [ ]:

extract_information(past_pregnancy,'Ew_preg_past2_notcurr')

Ew_preg_past2_notcurr    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
4732
85.84905660377359
Ew_preg_past2_notcurr
0.0    402
1.0    378
Name: count, dtype: int64
       Ew_preg_past2_notcurr
count             780.000000
mean                0.484615
std                 0.500084
min                 0.000000
25%                 0.000000
50%                 0.000000
75%                 1.000000
max                 1.000000


In [ ]:
extract_information(past_pregnancy,'Ew_pregpast2_num')

Ew_pregpast2_num    float64
dtype: object
0    1.0
1    NaN
2    2.0
3    3.0
4    4.0
5    5.0
dtype: float64
2143
38.87880986937591
Ew_pregpast2_num
1.0    3117
2.0     239
3.0      11
4.0       1
5.0       1
Name: count, dtype: int64
       Ew_pregpast2_num
count       3369.000000
mean           1.079549
std            0.291752
min            1.000000
25%            1.000000
50%            1.000000
75%            1.000000
max            5.000000


### LAST_PREGNANCY_EL

In [ ]:
last_pregnancy_el = parent_all_sections['LAST_PREGNANCY_EL']
last_pregnancy_el.columns

Index(['Ew_lastpreg_ac', 'Ew_lastpreg_foods', 'Ew_first_check_weeks',
       'Ew_first_check_weeks_bi', 'Ew_breastfeed', 'Ew_bfeed_atemore',
       'Ew_time_breastfeed_weeks', 'Ew_breastfed_weeks_bi', 'Ew_colostrum',
       'Ew_othliq_1week', 'Ew_othliq_3mo', 'Ew_food_3mo', 'Eiron', 'Evita_6wk',
       'Evita_6mo', 'Etotal_vacc'],
      dtype='object')

In [ ]:
extract_information(last_pregnancy_el,'Ew_lastpreg_ac')

Ew_lastpreg_ac    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
1729
31.367924528301888
Ew_lastpreg_ac
1.0    3446
0.0     337
Name: count, dtype: int64
       Ew_lastpreg_ac
count     3783.000000
mean         0.910917
std          0.284901
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


In [ ]:

extract_information(last_pregnancy_el,'Ew_lastpreg_foods')

Ew_lastpreg_foods    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
1732
31.422351233671986
Ew_lastpreg_foods
1.0    2440
0.0    1340
Name: count, dtype: int64
       Ew_lastpreg_foods
count        3780.000000
mean            0.645503
std             0.478424
min             0.000000
25%             0.000000
50%             1.000000
75%             1.000000
max             1.000000


In [ ]:
extract_information(last_pregnancy_el,'Ew_first_check_weeks')

Ew_first_check_weeks    float64
dtype: object
0      0.005952
1      4.000000
2           NaN
3     12.000000
4      5.000000
        ...    
89     5.428571
90    64.000000
91     0.154762
92    26.000000
93     6.285714
Length: 94, dtype: float64
2446
44.37590711175617
Ew_first_check_weeks
4.000000     692
0.142857     349
8.000000     285
0.005952     238
6.000000     199
            ... 
5.428571       1
64.000000      1
0.154762       1
26.000000      1
6.285714       1
Name: count, Length: 93, dtype: int64
       Ew_first_check_weeks
count           3066.000000
mean               4.380698
std                6.631384
min                0.000000
25%                0.142857
50%                4.000000
75%                6.000000
max              180.000000


In [ ]:
extract_information(last_pregnancy_el,'Ew_breastfeed')

Ew_breastfeed    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
2155
39.09651669085631
Ew_breastfeed
1.0    3327
0.0      30
Name: count, dtype: int64
       Ew_breastfeed
count    3357.000000
mean        0.991063
std         0.094124
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000


In [ ]:
extract_information(last_pregnancy_el,'Ew_bfeed_atemore')

Ew_bfeed_atemore    float64
dtype: object
0    0.0
1    NaN
2    1.0
dtype: float64
2182
39.58635703918723
Ew_bfeed_atemore
0.0    1838
1.0    1492
Name: count, dtype: int64
       Ew_bfeed_atemore
count       3330.000000
mean           0.448048
std            0.497368
min            0.000000
25%            0.000000
50%            0.000000
75%            1.000000
max            1.000000


In [ ]:
extract_information(last_pregnancy_el,'Ew_time_breastfeed_weeks')

Ew_time_breastfeed_weeks    float64
dtype: object
0     0.001488
1     0.002976
2          NaN
3     0.003968
4     0.005952
5     0.017857
6     0.000298
7     0.000496
8     0.000099
9     0.142857
10    0.000198
11    0.011905
12    0.000992
13    0.029762
14    0.023810
15    0.035714
16    0.001984
17    0.000000
18    1.428571
19    0.001786
20    0.004960
21    0.000397
22    0.053571
23    0.008929
24    0.047619
25    0.714286
26    0.071429
27    1.000000
28    0.002480
29    2.000000
30    0.000794
31    0.041667
32    0.000595
33    0.000694
34    0.083333
35    0.065476
36    0.428571
37    0.285714
38    0.178571
39    0.089286
40    0.059524
41    0.001190
42    0.004464
43    4.285714
44    0.003373
45    0.571429
46    0.119048
47    0.095238
48    0.001587
49    0.154762
dtype: float64
2187
39.67706821480406
Ew_time_breastfeed_weeks
0.005952    684
0.002976    529
0.000992    366
0.000496    313
0.001984    209
0.011905    194
0.000198    172
0.000298    152
0.001488 

In [ ]:
extract_information(last_pregnancy_el,'Ew_colostrum')

Ew_colostrum    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
2155
39.09651669085631
Ew_colostrum
1.0    3323
0.0      34
Name: count, dtype: int64
       Ew_colostrum
count   3357.000000
mean       0.989872
std        0.100142
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(last_pregnancy_el,'Ew_othliq_1week')

Ew_othliq_1week    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2156
39.114658925979676
Ew_othliq_1week
1.0    1727
0.0    1629
Name: count, dtype: int64
       Ew_othliq_1week
count      3356.000000
mean          0.514601
std           0.499861
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(last_pregnancy_el,'Ew_othliq_3mo')

Ew_othliq_3mo    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2239
40.620464441219156
Ew_othliq_3mo
1.0    1891
0.0    1382
Name: count, dtype: int64
       Ew_othliq_3mo
count    3273.000000
mean        0.577757
std         0.493992
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000


In [ ]:
extract_information(last_pregnancy_el,'Ew_food_3mo')

Ew_food_3mo    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
2276
41.29172714078375
Ew_food_3mo
1.0    3162
0.0      74
Name: count, dtype: int64
       Ew_food_3mo
count  3236.000000
mean      0.977132
std       0.149505
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(last_pregnancy_el,'Eiron')

Eiron    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2224
40.34833091436865
Eiron
1.0    2814
0.0     474
Name: count, dtype: int64
             Eiron
count  3288.000000
mean      0.855839
std       0.351306
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(last_pregnancy_el,'Evita_6wk')

Evita_6wk    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2237
40.58417997097242
Evita_6wk
1.0    2527
0.0     748
Name: count, dtype: int64
         Evita_6wk
count  3275.000000
mean      0.771603
std       0.419864
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(last_pregnancy_el,'Evita_6mo')

Evita_6mo    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
2231
40.47532656023222
Evita_6mo
1.0    2199
0.0    1082
Name: count, dtype: int64
         Evita_6mo
count  3281.000000
mean      0.670222
std       0.470204
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(last_pregnancy_el,'Etotal_vacc')

Etotal_vacc    float64
dtype: object
0     9.0
1     NaN
2     8.0
3     7.0
4     0.0
5     2.0
6     3.0
7     6.0
8     4.0
9     5.0
10    1.0
dtype: float64
2229
40.43904208998549
Etotal_vacc
9.0    1663
8.0     785
7.0     198
6.0     145
0.0     116
4.0     110
2.0      98
3.0      75
5.0      52
1.0      41
Name: count, dtype: int64
       Etotal_vacc
count  3283.000000
mean      7.512945
std       2.388667
min       0.000000
25%       7.000000
50%       9.000000
75%       9.000000
max       9.000000


### FOOD_RECALL

In [ ]:
food_recall = parent_all_sections['FOOD_RECALL']
food_recall.columns

Index(['Ew_milk', 'Ew_water', 'Ew_tea', 'Ew_juice', 'Ew_otherbev',
       'Ew_matooke', 'Ew_roots', 'Ew_grains', 'Ew_beans', 'Ew_organ',
       'Ew_meat', 'Ew_egg', 'Ew_fish', 'Ew_green', 'Ew_pump', 'Ew_otherfru',
       'Ew_othdairy', 'Ew_oil', 'Ew_sweet', 'Ew_juiceplus', 'Ew_meats',
       'Ew_otherfv'],
      dtype='object')

In [ ]:
extract_information(food_recall,'Ew_milk')

Ew_milk    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_milk
0.0    3395
1.0    1893
Name: count, dtype: int64
           Ew_milk
count  5288.000000
mean      0.357980
std       0.479452
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


In [ ]:

extract_information(food_recall,'Ew_water')

Ew_water    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_water
1.0    4720
0.0     568
Name: count, dtype: int64
          Ew_water
count  5288.000000
mean      0.892587
std       0.309667
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(food_recall,'Ew_tea')

Ew_tea    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_tea
0.0    4387
1.0     901
Name: count, dtype: int64
            Ew_tea
count  5288.000000
mean      0.170386
std       0.376007
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(food_recall,'Ew_juice')

Ew_juice    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_juice
0.0    4985
1.0     303
Name: count, dtype: int64
          Ew_juice
count  5288.000000
mean      0.057300
std       0.232436
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(food_recall,'Ew_otherbev')

Ew_otherbev    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_otherbev
1.0    4506
0.0     782
Name: count, dtype: int64
       Ew_otherbev
count  5288.000000
mean      0.852118
std       0.355017
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(food_recall,'Ew_matooke')

Ew_matooke    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_matooke
1.0    4597
0.0     691
Name: count, dtype: int64
        Ew_matooke
count  5288.000000
mean      0.869327
std       0.337074
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(food_recall,'Ew_roots')

Ew_roots    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_roots
1.0    4096
0.0    1192
Name: count, dtype: int64
          Ew_roots
count  5288.000000
mean      0.774584
std       0.417896
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(food_recall,'Ew_grains')

Ew_grains    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_grains
1.0    3347
0.0    1941
Name: count, dtype: int64
         Ew_grains
count  5288.000000
mean      0.632943
std       0.482048
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(food_recall,'Ew_beans')

Ew_beans    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_beans
1.0    4997
0.0     291
Name: count, dtype: int64
          Ew_beans
count  5288.000000
mean      0.944970
std       0.228061
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


### REPRODUCTIVE_HEALTH

In [ ]:
opinion_map = {
    1: 'A lot',
    2: 'A little',
    3: 'Not much',
    4: 'Nothing',
    5: 'Shouldn not know anything'
}

time_unit_map = {
    1: 'Month',
    2: 'Year'
}

space_op_map = {
  1: 'Best',
  2: 'Worst',
  3: 'Doesn not matter'
}

In [ ]:
reproductive_health = parent_all_sections['REPRODUCTIVE_HEALTH']
reproductive_health.columns

Index(['Ew_hus_ac', 'Ew_hus_feed', 'Ew_hus_preg', 'Ew_hus_hyg',
       'Ew_condom_times', 'Ew_condom_times_bi', 'Ew_wait_nextpreg',
       'Ew_wait_nextpreg_u', 'Ew_fp_decide', 'Ew_fp_talk', 'Ew_ideal_fert_t',
       'Ew_ideal_fert_b', 'Ew_ideal_fert_g', 'Ew_ideal_fert_e',
       'Ew_nextpreg_when', 'Ew_hus_ideal_fert', 'Ew_hus_ideal_fert_t',
       'Ew_hus_ideal_fert_b', 'Ew_hus_ideal_fert_g', 'Ew_hus_ideal_fert_e',
       'Ew_birth_space', 'Ew_birth_space_worse'],
      dtype='object')

In [ ]:
reproductive_health['Ew_hus_ac'] = reproductive_health['Ew_hus_ac'].map(opinion_map)
reproductive_health['Ew_hus_ac'] = reproductive_health['Ew_hus_ac'].astype('category')

extract_information(reproductive_health,'Ew_hus_ac')

Ew_hus_ac    category
dtype: object
0                     A little
1                     Not much
2                        A lot
3                      Nothing
4                          NaN
5    Shouldn not know anything
dtype: category
Categories (5, object): ['A little', 'A lot', 'Not much', 'Nothing', 'Shouldn not know anything']
409
7.420174165457184
Ew_hus_ac
A lot                        2224
A little                     1894
Not much                      502
Nothing                       481
Shouldn not know anything       2
Name: count, dtype: int64
       Ew_hus_ac
count       5103
unique         5
top        A lot
freq        2224


In [ ]:
reproductive_health['Ew_hus_feed'] = reproductive_health['Ew_hus_feed'].map(opinion_map)
reproductive_health['Ew_hus_feed'] = reproductive_health['Ew_hus_feed'].astype('category')

extract_information(reproductive_health,'Ew_hus_feed')

Ew_hus_feed    category
dtype: object
0                        A lot
1                     Not much
2                     A little
3                      Nothing
4                          NaN
5    Shouldn not know anything
dtype: category
Categories (5, object): ['A little', 'A lot', 'Not much', 'Nothing', 'Shouldn not know anything']
342
6.204644412191581
Ew_hus_feed
A lot                        2235
A little                     1988
Not much                      500
Nothing                       445
Shouldn not know anything       2
Name: count, dtype: int64
       Ew_hus_feed
count         5170
unique           5
top          A lot
freq          2235


In [ ]:
reproductive_health['Ew_hus_preg'] = reproductive_health['Ew_hus_preg'].map(opinion_map)
reproductive_health['Ew_hus_preg'] = reproductive_health['Ew_hus_preg'].astype('category')

extract_information(reproductive_health,'Ew_hus_preg')

Ew_hus_preg    category
dtype: object
0                     A little
1                        A lot
2                     Not much
3                      Nothing
4                          NaN
5    Shouldn not know anything
dtype: category
Categories (5, object): ['A little', 'A lot', 'Not much', 'Nothing', 'Shouldn not know anything']
342
6.204644412191581
Ew_hus_preg
A lot                        2117
A little                     1883
Nothing                       595
Not much                      574
Shouldn not know anything       1
Name: count, dtype: int64
       Ew_hus_preg
count         5170
unique           5
top          A lot
freq          2117


In [ ]:
reproductive_health['Ew_hus_hyg'] = reproductive_health['Ew_hus_hyg'].map(opinion_map)
reproductive_health['Ew_hus_hyg'] = reproductive_health['Ew_hus_hyg'].astype('category')

extract_information(reproductive_health,'Ew_hus_hyg')

Ew_hus_hyg    category
dtype: object
0                        A lot
1                     Not much
2                     A little
3                      Nothing
4                          NaN
5    Shouldn not know anything
dtype: category
Categories (5, object): ['A little', 'A lot', 'Not much', 'Nothing', 'Shouldn not know anything']
342
6.204644412191581
Ew_hus_hyg
A lot                        3137
A little                     1628
Not much                      214
Nothing                       190
Shouldn not know anything       1
Name: count, dtype: int64
       Ew_hus_hyg
count        5170
unique          5
top         A lot
freq         3137


In [ ]:
extract_information(reproductive_health,'Ew_condom_times')

Ew_condom_times    float64
dtype: object
0     1.0
1     NaN
2     2.0
3     3.0
4     5.0
5     4.0
6    17.0
dtype: float64
392
7.111756168359943
Ew_condom_times
1.0     5008
2.0       92
3.0       13
5.0        3
4.0        3
17.0       1
Name: count, dtype: int64
       Ew_condom_times
count      5120.000000
mean          1.030273
std           0.303109
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          17.000000


In [ ]:
extract_information(reproductive_health,'Ew_wait_nextpreg')

Ew_wait_nextpreg    float64
dtype: object
0      5.0
1     10.0
2      8.0
3      3.0
4      4.0
5      2.0
6      6.0
7      NaN
8     30.0
9      1.0
10     7.0
11    20.0
12    18.0
13    29.0
14    15.0
15    26.0
16    28.0
17    60.0
18    48.0
19    12.0
20    14.0
21     9.0
22    69.0
23    35.0
24    32.0
dtype: float64
225
4.08200290275762
Ew_wait_nextpreg
3.0     1800
2.0     1374
5.0      984
4.0      731
6.0      120
1.0       68
10.0      60
7.0       46
30.0      33
8.0       22
18.0      22
15.0       4
28.0       4
20.0       3
29.0       3
60.0       2
14.0       2
12.0       2
9.0        2
26.0       1
48.0       1
69.0       1
35.0       1
32.0       1
Name: count, dtype: int64
       Ew_wait_nextpreg
count       5287.000000
mean           3.779648
std            3.344635
min            1.000000
25%            2.000000
50%            3.000000
75%            4.000000
max           69.000000


In [ ]:
reproductive_health['Ew_wait_nextpreg_u'] = reproductive_health['Ew_wait_nextpreg_u'].map(time_unit_map)
reproductive_health['Ew_wait_nextpreg_u'] = reproductive_health['Ew_wait_nextpreg_u'].astype('category')

extract_information(reproductive_health,'Ew_wait_nextpreg_u')

Ew_wait_nextpreg_u    category
dtype: object
0     Year
1      NaN
2    Month
dtype: category
Categories (2, object): ['Month', 'Year']
224
4.063860667634253
Ew_wait_nextpreg_u
Year     5210
Month      78
Name: count, dtype: int64
       Ew_wait_nextpreg_u
count                5288
unique                  2
top                  Year
freq                 5210


In [ ]:
reproductive_health['Ew_birth_space'] = reproductive_health['Ew_birth_space'].map(space_op_map)
reproductive_health['Ew_birth_space'] = reproductive_health['Ew_birth_space'].astype('category')

extract_information(reproductive_health,'Ew_birth_space')

Ew_birth_space    category
dtype: object
0               Worst
1                Best
2    Doesn not matter
3                 NaN
dtype: category
Categories (3, object): ['Best', 'Doesn not matter', 'Worst']
224
4.063860667634253
Ew_birth_space
Worst               4713
Best                 467
Doesn not matter     108
Name: count, dtype: int64
       Ew_birth_space
count            5288
unique              3
top             Worst
freq             4713


In [ ]:
extract_information(reproductive_health,'Ew_birth_space_worse')

Ew_birth_space_worse    object
dtype: object
0         3 4
1           4
2         1 4
3       1 3 4
4           1
5         NaN
6         1 3
7           3
8         2 3
9       2 3 4
10          2
11        1 2
12        2 4
13      1 2 4
14    1 2 3 4
15      1 2 3
dtype: object
802
14.550072568940495
Ew_birth_space_worse
4          1795
3 4         662
1           556
3           522
1 4         500
1 3         297
1 3 4       170
1 2          48
2 4          40
2 3          39
2            29
1 2 3        22
1 2 4        16
2 3 4        11
1 2 3 4       3
Name: count, dtype: int64
       Ew_birth_space_worse
count                  4710
unique                   15
top                       4
freq                   1795


### SANITATION__PRACTICE

In [ ]:
how_many_map = {
  1: 'Always'  ,
  2 :'Most of the time',
  3 :'Sometimes'  ,
  4 :'Rarely'  ,
  5 :'Never'
}



In [ ]:
senitation_practice = parent_all_sections['SANITATION__PRACTICE']
senitation_practice.columns

Index(['Ew_treat_water', 'Ew_treat_water_how', 'Ew_treat_water_boil',
       'Ew_soap_toilet', 'Ew_soap_toilet_bi', 'Ew_soap_meal',
       'Ew_soap_meal_bi', 'Ew_refuse_dispose', 'Ew_refuse_dispose_bi',
       'Ew_toilet_type', 'Ew_toilet_improv', 'Ew_toilet_improv_what',
       'Ew_toilet_sweep', 'Ew_toilet_sweep_bi', 'Ew_handwash_what'],
      dtype='object')

In [ ]:
extract_information(senitation_practice,'Ew_treat_water')

Ew_treat_water    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
224
4.063860667634253
Ew_treat_water
1.0    5015
0.0     273
Name: count, dtype: int64
       Ew_treat_water
count     5288.000000
mean         0.948374
std          0.221292
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


In [ ]:
senitation_practice['Ew_soap_toilet'] = senitation_practice['Ew_soap_toilet'].map(how_many_map)
senitation_practice['Ew_soap_toilet'] = senitation_practice['Ew_soap_toilet'].astype('category')

extract_information(senitation_practice,'Ew_soap_toilet')

Ew_soap_toilet    category
dtype: object
0              Rarely
1              Always
2               Never
3    Most of the time
4           Sometimes
5                 NaN
dtype: category
Categories (5, object): ['Always', 'Most of the time', 'Never', 'Rarely', 'Sometimes']
225
4.08200290275762
Ew_soap_toilet
Always              2195
Sometimes           1068
Most of the time     986
Rarely               857
Never                181
Name: count, dtype: int64
       Ew_soap_toilet
count            5287
unique              5
top            Always
freq             2195


In [ ]:
senitation_practice['Ew_soap_meal'] = senitation_practice['Ew_soap_meal'].map(how_many_map)
senitation_practice['Ew_soap_meal'] = senitation_practice['Ew_soap_meal'].astype('category')

extract_information(senitation_practice,'Ew_soap_meal')

Ew_soap_meal    category
dtype: object
0           Sometimes
1    Most of the time
2              Rarely
3               Never
4              Always
5                 NaN
dtype: category
Categories (5, object): ['Always', 'Most of the time', 'Never', 'Rarely', 'Sometimes']
224
4.063860667634253
Ew_soap_meal
Rarely              1486
Always              1322
Sometimes           1243
Most of the time     730
Never                507
Name: count, dtype: int64
       Ew_soap_meal
count          5288
unique            5
top          Rarely
freq           1486


### HEALTH__KNOWLEDGE

In [ ]:
health_how_much_map = {
  1: 'A lot',
2: 'Somewhat',
3: 'Not much',
4: 'Not at all',
5: 'I do not know'
}
health_how_much_2_map = {
1: 'A lot',
2: 'Somewhat',
3: 'A little',
4: 'Not at all'
}

how_many_map = {
    1: 'Often',
2: 'Sometimes',
3: 'Rarely',
4: 'Never'
}


In [ ]:
health_knowledge = parent_all_sections['HEALTH__KNOWLEDGE']
print(list(health_knowledge.columns))

['Ew_safe_primhc', 'Ew_safe_primhc_bi', 'Ew_cstrum_feed', 'Ew_cstrum_no_why', 'Ew_cstrum_impt_why', 'Ew_cstrum_impt_why_bi', 'Ew_grow_lackdiet', 'Ew_grow_lackdiet_sw', 'Ew_grow_lackdiet_vm', 'Ew_grow_witch', 'Ew_grow_witch_lt', 'Ew_grow_witch_vl', 'Ew_kwash_cause', 'Ew_kwash_cause_bi', 'Ew_kwash_treat', 'Ew_kwash_treat_bi', 'Ew_ugx_spend', 'Ew_ugx_matooke', 'Ew_ugx_matooke_bi', 'Ew_ugx_maize', 'Ew_ugx_maize_bi', 'Ew_ugx_potato', 'Ew_ugx_potato_bi', 'Ew_ugx_rice', 'Ew_ugx_rice_bi', 'Ew_ugx_fish', 'Ew_ugx_fish_bi', 'Ew_ugx_gnuts', 'Ew_ugx_gnuts_bi', 'Ew_ugx_veg', 'Ew_ugx_veg_bi', 'Ew_ugx_fruit', 'Ew_ugx_fruit_bi', 'Ew_ugx_meat', 'Ew_ugx_meat_bi', 'Ew_ugx_bean', 'Ew_ugx_bean_bi', 'Ew_ugx_egg', 'Ew_ugx_egg_bi', 'Ew_ugx_milk', 'Ew_ugx_milk_bi', 'Ew_ugx_sweet', 'Ew_ugx_sweet_bi', 'Ew_ugx_flour', 'Ew_ugx_flour_bi', 'Ew_ugx_bread', 'Ew_ugx_bread_bi', 'Ew_ugx_ghee', 'Ew_ugx_ghee_bi', 'Ew_ugx_soya', 'Ew_ugx_soya_bi', 'Ew_ugx_other', 'Ew_ugx_other_bi', 'Ew_ageliquid', 'Ew_ageliquid_6', 'Ew_agefoo

In [ ]:
extract_information(health_knowledge,'Ew_safe_primhc')

Ew_safe_primhc    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
224
4.063860667634253
Ew_safe_primhc
0.0    3166
1.0    2122
Name: count, dtype: int64
       Ew_safe_primhc
count     5288.000000
mean         0.401286
std          0.490205
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


In [ ]:
extract_information(health_knowledge,'Ew_cstrum_feed')

Ew_cstrum_feed    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
224
4.063860667634253
Ew_cstrum_feed
1.0    5115
0.0     173
Name: count, dtype: int64
       Ew_cstrum_feed
count     5288.000000
mean         0.967284
std          0.177908
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


In [ ]:
health_knowledge['Ew_grow_lackdiet'] = health_knowledge['Ew_grow_lackdiet'].map(health_how_much_map)
health_knowledge['Ew_grow_lackdiet'] = health_knowledge['Ew_grow_lackdiet'].astype('category')

extract_information(health_knowledge,'Ew_grow_lackdiet')

Ew_grow_lackdiet    category
dtype: object
0         A lot
1      Not much
2      Somewhat
3           NaN
4    Not at all
dtype: category
Categories (4, object): ['A lot', 'Not at all', 'Not much', 'Somewhat']
231
4.1908563134978225
Ew_grow_lackdiet
A lot         4782
Somewhat       276
Not much       176
Not at all      47
Name: count, dtype: int64
       Ew_grow_lackdiet
count              5281
unique                4
top               A lot
freq               4782


In [ ]:
health_knowledge['Ew_grow_witch'] = health_knowledge['Ew_grow_witch'].map(health_how_much_map)
health_knowledge['Ew_grow_witch'] = health_knowledge['Ew_grow_witch'].astype('category')

extract_information(health_knowledge,'Ew_grow_witch')

Ew_grow_witch    category
dtype: object
0         A lot
1      Not much
2    Not at all
3      Somewhat
4           NaN
dtype: category
Categories (4, object): ['A lot', 'Not at all', 'Not much', 'Somewhat']
339
6.150217706821481
Ew_grow_witch
A lot         4817
Not at all     148
Somewhat       129
Not much        79
Name: count, dtype: int64
       Ew_grow_witch
count           5173
unique             4
top            A lot
freq            4817


In [ ]:
health_knowledge['Ew_grow_witch'] = health_knowledge['Ew_grow_witch'].map(health_how_much_map)
health_knowledge['Ew_grow_witch'] = health_knowledge['Ew_grow_witch'].astype('category')

extract_information(health_knowledge,'Ew_grow_witch')

Ew_grow_witch    category
dtype: object
0   NaN
dtype: category
Categories (0, float64): []
5512
100.0
Series([], Name: count, dtype: int64)
       Ew_grow_witch
count              0
unique             0
top              NaN
freq             NaN


In [ ]:
health_knowledge['Ew_barrier_time'] = health_knowledge['Ew_barrier_time'].map(health_how_much_2_map)
health_knowledge['Ew_barrier_time'] = health_knowledge['Ew_barrier_time'].astype('category')

extract_information(health_knowledge,'Ew_barrier_time')

Ew_barrier_time    category
dtype: object
0    Not at all
1      A little
2         A lot
3      Somewhat
4           NaN
dtype: category
Categories (4, object): ['A little', 'A lot', 'Not at all', 'Somewhat']
224
4.063860667634253
Ew_barrier_time
Not at all    2500
Somewhat      1122
A lot          878
A little       788
Name: count, dtype: int64
       Ew_barrier_time
count             5288
unique               4
top         Not at all
freq              2500


In [ ]:
health_knowledge['Ew_discuss_friends'] = health_knowledge['Ew_discuss_friends'].map(how_many_map)
health_knowledge['Ew_discuss_friends'] = health_knowledge['Ew_discuss_friends'].astype('category')

extract_information(health_knowledge,'Ew_discuss_friends')

Ew_discuss_friends    category
dtype: object
0        Often
1    Sometimes
2       Rarely
3        Never
4          NaN
dtype: category
Categories (4, object): ['Never', 'Often', 'Rarely', 'Sometimes']
224
4.063860667634253
Ew_discuss_friends
Sometimes    2057
Often        1758
Rarely       1284
Never         189
Name: count, dtype: int64
       Ew_discuss_friends
count                5288
unique                  4
top             Sometimes
freq                 2057


In [ ]:
health_knowledge['Ew_discuss_spouse'] = health_knowledge['Ew_discuss_spouse'].map(how_many_map)
health_knowledge['Ew_discuss_spouse'] = health_knowledge['Ew_discuss_spouse'].astype('category')

extract_information(health_knowledge,'Ew_discuss_spouse')

Ew_discuss_spouse    category
dtype: object
0        Often
1    Sometimes
2       Rarely
3        Never
4          NaN
dtype: category
Categories (4, object): ['Never', 'Often', 'Rarely', 'Sometimes']
321
5.8236574746008705
Ew_discuss_spouse
Often        3506
Sometimes    1074
Rarely        513
Never          98
Name: count, dtype: int64
       Ew_discuss_spouse
count               5191
unique                 4
top                Often
freq                3506


In [ ]:
health_knowledge['Ew_discuss_fam'] = health_knowledge['Ew_discuss_fam'].map(how_many_map)
health_knowledge['Ew_discuss_fam'] = health_knowledge['Ew_discuss_fam'].astype('category')

extract_information(health_knowledge,'Ew_discuss_fam')

Ew_discuss_fam    category
dtype: object
0        Often
1    Sometimes
2       Rarely
3        Never
4          NaN
dtype: category
Categories (4, object): ['Never', 'Often', 'Rarely', 'Sometimes']
224
4.063860667634253
Ew_discuss_fam
Rarely       1526
Sometimes    1486
Often        1405
Never         871
Name: count, dtype: int64
       Ew_discuss_fam
count            5288
unique              4
top            Rarely
freq             1526


In [ ]:
health_knowledge['Ew_husb_sugg_food'] = health_knowledge['Ew_husb_sugg_food'].map(how_many_map)
health_knowledge['Ew_husb_sugg_food'] = health_knowledge['Ew_husb_sugg_food'].astype('category')

extract_information(health_knowledge,'Ew_husb_sugg_food')

Ew_husb_sugg_food    category
dtype: object
0        Often
1       Rarely
2        Never
3    Sometimes
4          NaN
dtype: category
Categories (4, object): ['Never', 'Often', 'Rarely', 'Sometimes']
321
5.8236574746008705
Ew_husb_sugg_food
Often        2320
Sometimes    1549
Rarely        955
Never         367
Name: count, dtype: int64
       Ew_husb_sugg_food
count               5191
unique                 4
top                Often
freq                2320


In [ ]:
health_knowledge['Ew_husb_sugg_hc'] = health_knowledge['Ew_husb_sugg_hc'].map(how_many_map)
health_knowledge['Ew_husb_sugg_hc'] = health_knowledge['Ew_husb_sugg_hc'].astype('category')

extract_information(health_knowledge,'Ew_husb_sugg_hc')

Ew_husb_sugg_hc    category
dtype: object
0        Often
1    Sometimes
2        Never
3       Rarely
4          NaN
dtype: category
Categories (4, object): ['Never', 'Often', 'Rarely', 'Sometimes']
321
5.8236574746008705
Ew_husb_sugg_hc
Often        2930
Sometimes    1221
Rarely        747
Never         293
Name: count, dtype: int64
       Ew_husb_sugg_hc
count             5191
unique               4
top              Often
freq              2930


## Children Section

In [ ]:
child_all_sections = extract_sections(children_df)

,Section Name,Columns Count
0,GENERAL,18
1,BASELINE_SURVEY,8
2,FOOD_RECALL,20
3,OTHER_HEALTH,34
4,BASELINE_ANTHRO,35
5,ENDLINE_SURVEY,16
6,FOOD__RECALL,56
7,ENDLINE_ANTHRO,60


### 1- GENERAL Section

In [ ]:
general_child = child_all_sections['GENERAL']
general_child.columns

Index(['id', 'villageid', 'stratum', 'districtid', 'district_name', 'wemp',
       'whn', 'mhn', 'control', 'treat', 'sample', 'sample_married',
       'sample_moved', 'sample_alive', 'sample_2yr', 'sample_1yr', 'onlyin',
       'survey_status'],
      dtype='object')

#### 1.1- id

In [ ]:
extract_information(general_child,'id')

id    float64
dtype: object
0       4.011010e+09
1       6.110101e+08
2       4.711010e+09
3       7.110101e+08
4       4.511010e+09
            ...     
5611    8.541070e+09
5612    1.841070e+09
5613    6.741070e+09
5614    5.341070e+09
5615    9.141070e+09
Length: 5616, dtype: float64
0
0.0
id
4.931020e+09    12
2.722030e+09    12
1.210204e+08    12
4.121040e+09    11
2.741050e+09    11
                ..
1.051101e+10     1
1.511010e+09     1
8.110103e+08     1
4.441070e+09     1
2.331041e+09     1
Name: count, Length: 5616, dtype: int64
                 id
count  2.230800e+04
mean   5.009174e+09
std    4.435156e+09
min    1.110101e+08
25%    2.032030e+09
50%    4.031031e+09
75%    6.523030e+09
max    2.552301e+10


#### 1.2- villageid

In [ ]:
extract_information(general_child,'villageid')

villageid    int64
dtype: object
0      11010102
1      11010204
2      11010208
3      11010302
4      11010306
         ...   
407    41070212
408    41070304
409    41070308
410    41070405
411    41070407
Length: 412, dtype: int64
0
0.0
villageid
22020204    93
22080101    92
22040101    88
22040506    88
11080502    85
            ..
11100403    18
11100106    17
31010407    15
21020208    15
11020203    14
Name: count, Length: 412, dtype: int64
          villageid
count  2.230800e+04
mean   2.496653e+07
std    9.340230e+06
min    1.101010e+07
25%    2.103020e+07
50%    2.208041e+07
75%    3.201031e+07
max    4.107041e+07


#### 1.3- stratum

In [ ]:
extract_information(general_child,'stratum')

stratum    float64
dtype: object
0    3.0
1    4.0
2    1.0
3    2.0
dtype: float64
0
0.0
stratum
4.0    6080
1.0    5741
3.0    5259
2.0    5228
Name: count, dtype: int64
            stratum
count  22308.000000
mean       2.523489
std        1.144249
min        1.000000
25%        1.000000
50%        3.000000
75%        4.000000
max        4.000000


#### 1.4- districtid

In [ ]:
extract_information(general_child,'districtid')

districtid    float64
dtype: object
0    1.0
1    2.0
2    NaN
3    3.0
4    4.0
dtype: float64
3
0.013448090371167292
districtid
2.0    9311
3.0    5671
1.0    4254
4.0    3069
Name: count, dtype: int64
         districtid
count  22305.000000
mean       2.338713
std        0.938430
min        1.000000
25%        2.000000
50%        2.000000
75%        3.000000
max        4.000000


#### 1.5- district_name

In [ ]:
extract_information(general_child,'district_name')

district_name    object
dtype: object
0      MITOOMA
1     NTUNGAMO
2          NaN
3    RUKUNGIRI
4       SHEEMA
dtype: object
3
0.013448090371167292
district_name
NTUNGAMO     9311
RUKUNGIRI    5671
MITOOMA      4254
SHEEMA       3069
Name: count, dtype: int64
       district_name
count          22305
unique             4
top         NTUNGAMO
freq            9311


#### 1.6- wemp

In [ ]:
extract_information(general_child,'wemp')

wemp    float64
dtype: object
0    1.0
1    0.0
dtype: float64
0
0.0
wemp
0.0    16817
1.0     5491
Name: count, dtype: int64
               wemp
count  22308.000000
mean       0.246145
std        0.430774
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


#### 1.7- whn

In [ ]:
extract_information(general_child,'whn')

whn    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
whn
0.0    16343
1.0     5965
Name: count, dtype: int64
                whn
count  22308.000000
mean       0.267393
std        0.442609
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000


#### 1.8- mhn

In [ ]:
extract_information(general_child,'mhn')

mhn    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
mhn
0.0    16950
1.0     5358
Name: count, dtype: int64
                mhn
count  22308.000000
mean       0.240183
std        0.427204
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


#### 1.9- control

In [ ]:
extract_information(general_child,'control')

control    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
control
0.0    16814
1.0     5494
Name: count, dtype: int64
            control
count  22308.000000
mean       0.246279
std        0.430853
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


#### 1.10- treat

In [ ]:
extract_information(general_child,'treat')

treat    int64
dtype: object
0    2
1    1
2    3
3    4
dtype: int64
0
0.0
treat
3    5965
1    5494
2    5491
4    5358
Name: count, dtype: int64
              treat
count  22308.000000
mean       2.501479
std        1.105883
min        1.000000
25%        2.000000
50%        3.000000
75%        3.000000
max        4.000000


#### 1.11- sample

In [ ]:
extract_information(general_child,'sample')

sample    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
sample
0.0    15058
1.0     7250
Name: count, dtype: int64
             sample
count  22308.000000
mean       0.324996
std        0.468384
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000


#### 1.12- sample_married

In [ ]:
extract_information(general_child,'sample_married')

sample_married    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
sample_married
0.0    15215
1.0     7093
Name: count, dtype: int64
       sample_married
count    22308.000000
mean         0.317958
std          0.465693
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000


#### 1.13- sample_moved

In [ ]:
extract_information(general_child,'sample_moved')

sample_moved    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
sample_moved
0.0    22213
1.0       95
Name: count, dtype: int64
       sample_moved
count  22308.000000
mean       0.004259
std        0.065120
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


#### 1.14- sample_alive

In [ ]:
extract_information(general_child,'sample_alive')

sample_alive    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
1554
6.966110812264659
sample_alive
1.0    20274
0.0      480
Name: count, dtype: int64
       sample_alive
count  20754.000000
mean       0.976872
std        0.150314
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


#### 1.15- sample_2yr

In [ ]:
extract_information(general_child,'sample_2yr')

sample_2yr    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
sample_2yr
0.0    18711
1.0     3597
Name: count, dtype: int64
         sample_2yr
count  22308.000000
mean       0.161243
std        0.367763
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


#### 1.16- sample_1yr

In [ ]:
extract_information(general_child,'sample_1yr')

sample_1yr    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
sample_1yr
0.0    21130
1.0     1178
Name: count, dtype: int64
         sample_1yr
count  22308.000000
mean       0.052806
std        0.223651
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


#### 1.17- onlyin

In [ ]:
extract_information(general_child,'onlyin')

onlyin    int64
dtype: object
0    3
1    2
2    1
dtype: int64
0
0.0
onlyin
3    17064
2     3505
1     1739
Name: count, dtype: int64
             onlyin
count  22308.000000
mean       2.686973
std        0.609070
min        1.000000
25%        3.000000
50%        3.000000
75%        3.000000
max        3.000000


#### 1.18- survey_status

In [ ]:
extract_information(general_child,'survey_status')

survey_status    float64
dtype: object
0    1.0
1    NaN
2    3.0
3    2.0
4    7.0
dtype: float64
3361
15.066343912497759
survey_status
1.0    18399
3.0      392
2.0      145
7.0       11
Name: count, dtype: int64
       survey_status
count   18947.000000
mean        1.052515
std         0.329482
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         7.000000


### 2- BASELINE_SURVEY Section

In [ ]:
baseline_survey_child = child_all_sections['BASELINE_SURVEY']
baseline_survey_child.columns

Index(['age', 'gender', 'survey_date', 'kidsunder18', 'under18bio',
       'under18nonbio', 'biooutside', 'otherkids'],
      dtype='object')

#### 2.1- age

In [ ]:
extract_information(baseline_survey_child,'age')

age    float64
dtype: object
0       69.0
1        NaN
2      124.0
3       79.0
4       22.0
       ...  
236    283.0
237    304.0
238    240.0
239    254.0
240    294.0
Length: 241, dtype: float64
4137
18.5449166218397
age
1.0      288
6.0      278
0.0      277
4.0      270
2.0      265
        ... 
283.0      1
304.0      1
240.0      1
254.0      1
294.0      1
Name: count, Length: 240, dtype: int64
                age
count  18171.000000
mean      70.676462
std       60.954285
min        0.000000
25%       20.000000
50%       56.000000
75%      111.000000
max      982.000000


#### 2.2- gender

In [ ]:
extract_information(baseline_survey_child,'gender')

gender    float64
dtype: object
0    2.0
1    1.0
2    NaN
dtype: float64
350
1.5689438766361843
gender
1.0    11026
2.0    10932
Name: count, dtype: int64
             gender
count  21958.000000
mean       1.497860
std        0.500007
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        2.000000


#### 2.3- survey_date

In [ ]:
extract_information(baseline_survey_child,'survey_date')

survey_date    object
dtype: object
0     2012-10-30
1            NaN
2     2012-11-01
3     2012-10-29
4     2012-10-31
         ...    
87    2012-11-17
88    2012-11-18
89    2012-11-19
90    2013-01-05
91    2012-11-21
Length: 92, dtype: object
3363
15.075309306078537
survey_date
2012-12-14    376
2012-12-08    371
2013-01-07    365
2012-12-13    354
2012-12-18    334
             ... 
2012-11-06     12
2012-10-09      5
2013-01-05      5
2013-12-07      4
2013-01-15      4
Name: count, Length: 91, dtype: int64
       survey_date
count        18945
unique          91
top     2012-12-14
freq           376


#### 2.4- kidsunder18

In [ ]:
extract_information(baseline_survey_child,'kidsunder18')

kidsunder18    float64
dtype: object
0      2.0
1      NaN
2      3.0
3      4.0
4      6.0
5      5.0
6      1.0
7      8.0
8      0.0
9      7.0
10     9.0
11    10.0
12    11.0
13    12.0
dtype: float64
3362
15.070826609288149
kidsunder18
4.0     3619
5.0     3344
3.0     3153
6.0     2593
2.0     2297
7.0     1523
1.0     1166
8.0      746
0.0      202
9.0      202
10.0      71
11.0      22
12.0       8
Name: count, dtype: int64
        kidsunder18
count  18946.000000
mean       4.300011
std        1.961852
min        0.000000
25%        3.000000
50%        4.000000
75%        6.000000
max       12.000000


#### 2.5- under18bio

In [ ]:
extract_information(baseline_survey_child,'under18bio')

under18bio    float64
dtype: object
0      2.0
1      NaN
2      3.0
3      4.0
4      6.0
5      5.0
6      1.0
7      8.0
8      0.0
9      7.0
10     9.0
11    10.0
dtype: float64
3362
15.070826609288149
under18bio
4.0     3566
3.0     3211
5.0     3117
2.0     2655
6.0     2332
1.0     1779
7.0     1431
8.0      452
0.0      268
9.0       98
10.0      37
Name: count, dtype: int64
         under18bio
count  18946.000000
mean       3.982265
std        1.928547
min        0.000000
25%        3.000000
50%        4.000000
75%        5.000000
max       10.000000


#### 2.6- under18nonbio

In [ ]:
extract_information(baseline_survey_child,'under18nonbio')

under18nonbio    float64
dtype: object
0    NaN
1    1.0
2    0.0
3    3.0
4    2.0
5    4.0
6    5.0
7    7.0
8    6.0
9    8.0
dtype: float64
17963
80.52268244575936
under18nonbio
1.0    2439
2.0     726
0.0     568
3.0     274
4.0     159
5.0      95
6.0      44
7.0      24
8.0      16
Name: count, dtype: int64
       under18nonbio
count    4345.000000
mean        1.469275
std         1.281343
min         0.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         8.000000


#### 2.7- biooutside

In [ ]:
extract_information(baseline_survey_child,'biooutside')

biooutside    float64
dtype: object
0    0.0
1    NaN
2    1.0
3    2.0
4    3.0
5    4.0
6    6.0
7    5.0
8    7.0
9    8.0
dtype: float64
3362
15.070826609288149
biooutside
0.0    13440
1.0     3379
2.0     1358
3.0      491
4.0      157
5.0       76
6.0       25
7.0       16
8.0        4
Name: count, dtype: int64
         biooutside
count  18946.000000
mean       0.468173
std        0.902456
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        8.000000


#### 2.8- otherkids

In [ ]:
extract_information(baseline_survey_child,'otherkids')

otherkids    float64
dtype: object
0      0.0
1      NaN
2      1.0
3      6.0
4      4.0
5      2.0
6      3.0
7      7.0
8      9.0
9     10.0
10    14.0
11     5.0
12     8.0
13    11.0
14    21.0
15    13.0
16    12.0
17    20.0
18    17.0
19    15.0
dtype: float64
3561
15.962883270575578
otherkids
0.0     13443
1.0      2250
2.0      1048
3.0       544
4.0       518
5.0       264
6.0       234
7.0       161
8.0        83
10.0       61
9.0        60
11.0       34
20.0       11
13.0       10
14.0        7
12.0        7
21.0        5
15.0        5
17.0        2
Name: count, dtype: int64
          otherkids
count  18747.000000
mean       0.791327
std        1.841409
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max       21.000000


### 3- FOOD_RECALL Section

In [ ]:
food_recall_child = child_all_sections['FOOD_RECALL']
food_recall_child.columns

Index(['milk', 'water', 'tea', 'juice', 'otherbev', 'solidfood', 'matooke',
       'roots', 'grains', 'beans', 'organ', 'meat', 'egg', 'fish', 'green',
       'pump', 'otherfru', 'oil', 'sweet', 'othersolid'],
      dtype='object')

#### 3.1- milk

In [ ]:
extract_information(food_recall_child,'milk')

milk    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
14777
66.2408104715797
milk
0.0    4494
1.0    3037
Name: count, dtype: int64
              milk
count  7531.000000
mean      0.403266
std       0.490586
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 3.2- water

In [ ]:
extract_information(food_recall_child,'water')

water    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
14769
66.20494889725659
water
1.0    5397
0.0    2142
Name: count, dtype: int64
             water
count  7539.000000
mean      0.715877
std       0.451025
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 3.3- tea

In [ ]:
extract_information(food_recall_child,'tea')

tea    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
14771
66.21391429083737
tea
0.0    6622
1.0     915
Name: count, dtype: int64
               tea
count  7537.000000
mean      0.121401
std       0.326614
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 3.4- juice

In [ ]:
extract_information(food_recall_child,'juice')

juice    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
14778
66.2452931683701
juice
0.0    7208
1.0     322
Name: count, dtype: int64
             juice
count  7530.000000
mean      0.042762
std       0.202334
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 3.5- otherbev

In [ ]:
extract_information(food_recall_child,'otherbev')

otherbev    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
14771
66.21391429083737
otherbev
1.0    4521
0.0    3016
Name: count, dtype: int64
          otherbev
count  7537.000000
mean      0.599841
std       0.489963
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 3.6- solidfood

In [ ]:
extract_information(food_recall_child,'solidfood')

solidfood    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
14767
66.19598350367582
solidfood
1.0    6704
0.0     837
Name: count, dtype: int64
         solidfood
count  7541.000000
mean      0.889007
std       0.314145
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 3.7- matooke

In [ ]:
extract_information(food_recall_child,'matooke')

matooke    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15561
69.75524475524476
matooke
1.0    5732
0.0    1015
Name: count, dtype: int64
           matooke
count  6747.000000
mean      0.849563
std       0.357526
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 3.8- roots

In [ ]:
extract_information(food_recall_child,'roots')

roots    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15567
69.78214093598709
roots
1.0    5191
0.0    1550
Name: count, dtype: int64
             roots
count  6741.000000
mean      0.770064
std       0.420823
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 3.9- grains

In [ ]:
extract_information(food_recall_child,'grains')

grains    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
15567
69.78214093598709
grains
0.0    3533
1.0    3208
Name: count, dtype: int64
            grains
count  6741.000000
mean      0.475894
std       0.499456
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 3.10- beans

In [ ]:
extract_information(food_recall_child,'beans')

beans    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15563
69.76421014882554
beans
1.0    6332
0.0     413
Name: count, dtype: int64
             beans
count  6745.000000
mean      0.938769
std       0.239770
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 3.11- organ

In [ ]:
extract_information(food_recall_child,'organ')

organ    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
15569
69.79110632956787
organ
0.0    6622
1.0     117
Name: count, dtype: int64
             organ
count  6739.000000
mean      0.017362
std       0.130624
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 3.12- meat

In [ ]:
extract_information(food_recall_child,'meat')

meat    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
15578
69.83145060068136
meat
0.0    6144
1.0     586
Name: count, dtype: int64
              meat
count  6730.000000
mean      0.087073
std       0.281963
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 3.13- egg

In [ ]:
extract_information(food_recall_child,'egg')

egg    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
15565
69.7731755424063
egg
0.0    6288
1.0     455
Name: count, dtype: int64
               egg
count  6743.000000
mean      0.067477
std       0.250866
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 3.14- fish

In [ ]:
extract_information(food_recall_child,'fish')

fish    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
15576
69.8224852071006
fish
0.0    6572
1.0     160
Name: count, dtype: int64
              fish
count  6732.000000
mean      0.023767
std       0.152334
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 3.15- green

In [ ]:
extract_information(food_recall_child,'green')

green    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
15570
69.79558902635826
green
1.0    3838
0.0    2900
Name: count, dtype: int64
             green
count  6738.000000
mean      0.569605
std       0.495168
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 3.16- pump

In [ ]:
extract_information(food_recall_child,'pump')

pump    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
15579
69.83593329747177
pump
0.0    5441
1.0    1288
Name: count, dtype: int64
             pump
count  6729.00000
mean      0.19141
std       0.39344
min       0.00000
25%       0.00000
50%       0.00000
75%       0.00000
max       1.00000


#### 3.17- otherfru

In [ ]:
extract_information(food_recall_child,'otherfru')

otherfru    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
15568
69.78662363277748
otherfru
1.0    3624
0.0    3116
Name: count, dtype: int64
          otherfru
count  6740.000000
mean      0.537685
std       0.498615
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 3.18- oil

In [ ]:
extract_information(food_recall_child,'oil')

oil    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15571
69.80007172314865
oil
0.0    4969
1.0    1768
Name: count, dtype: int64
               oil
count  6737.000000
mean      0.262431
std       0.439988
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 3.19- sweet

In [ ]:
extract_information(food_recall_child,'sweet')

sweet    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15572
69.80455441993904
sweet
0.0    5251
1.0    1485
Name: count, dtype: int64
             sweet
count  6736.000000
mean      0.220457
std       0.414586
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 3.20- othersolid

In [ ]:
extract_information(food_recall_child,'othersolid')

othersolid    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15570
69.79558902635826
othersolid
0.0    4674
1.0    2064
Name: count, dtype: int64
        othersolid
count  6738.000000
mean      0.306322
std       0.460999
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


### 4- OTHER_HEALTH Section

In [ ]:
other_health_child = child_all_sections['OTHER_HEALTH']
other_health_child.columns

Index(['immcard_w', 'biokid', 'breastfed', 'time_bf', 'bednet', 'diarrhea',
       'diar_days', 'diar_drink', 'diar_medpro', 'malaria', 'mal_days',
       'mal_drink', 'mal_medpro', 'cough', 'cough_days', 'cough_medpro',
       'worm_drugs', 'worm_medpro', 'semi_solidfood', 'num_meal',
       'daysprotein', 'bcg', 'polio0', 'polio1', 'polio2', 'polio3', 'dpt1',
       'dpt2', 'dpt3', 'measles', 'vita_6wk', 'vita_6mo', 'iron',
       'elapsed_time_mo'],
      dtype='object')

#### 4.1- immcard_w

In [ ]:
extract_information(other_health_child,'immcard_w')

immcard_w    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
4234
18.979738210507442
immcard_w
1.0    11136
0.0     6938
Name: count, dtype: int64
          immcard_w
count  18074.000000
mean       0.616134
std        0.486339
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000


#### 4.2- biokid

In [ ]:
extract_information(other_health_child,'biokid')

biokid    float64
dtype: object
0    1.0
1    NaN
2    0.0
dtype: float64
3625
16.249775865160483
biokid
1.0    17360
0.0     1323
Name: count, dtype: int64
             biokid
count  18683.000000
mean       0.929187
std        0.256519
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


#### 4.3- breastfed

In [ ]:
extract_information(other_health_child,'breastfed')

breastfed    float64
dtype: object
0    NaN
1    3.0
2    1.0
3    2.0
dtype: float64
13417
60.14434283665053
breastfed
1.0    5135
3.0    3617
2.0     139
Name: count, dtype: int64
         breastfed
count  8891.000000
mean      1.829266
std       0.977406
min       1.000000
25%       1.000000
50%       1.000000
75%       3.000000
max       3.000000


#### 4.4- time_bf

In [ ]:
extract_information(other_health_child,'time_bf')

time_bf    float64
dtype: object
0      NaN
1     24.0
2     14.0
3     12.0
4     15.0
5     18.0
6     20.0
7     22.0
8     13.0
9      9.0
10     6.0
11     4.0
12    17.0
13    21.0
14    32.0
15     1.0
16    25.0
17    19.0
18    28.0
19     7.0
20    10.0
21    16.0
22     8.0
23    26.0
24     3.0
25    27.0
26    11.0
27    29.0
28    30.0
29     5.0
30     0.0
31    23.0
32     2.0
33    36.0
34    31.0
35    48.0
36    38.0
37    37.0
38    33.0
39    35.0
40    40.0
41    34.0
42    98.0
dtype: float64
17151
76.88273265196342
time_bf
24.0    736
18.0    734
12.0    506
20.0    331
14.0    279
19.0    249
15.0    242
9.0     224
17.0    198
16.0    179
8.0     153
6.0     123
21.0    114
30.0    107
7.0      95
10.0     86
13.0     78
11.0     71
5.0      69
26.0     67
4.0      57
2.0      56
22.0     50
3.0      50
1.0      49
27.0     41
0.0      41
36.0     39
25.0     31
23.0     31
29.0     23
32.0     18
28.0     15
31.0      6
33.0      2
48.0      1
38.0      1
37.

#### 4.5- bednet

In [ ]:
extract_information(other_health_child,'bednet')

bednet    float64
dtype: object
0    NaN
1    2.0
2    3.0
3    1.0
dtype: float64
13414
60.13089474627936
bednet
3.0    3543
2.0    3005
1.0    2346
Name: count, dtype: int64
            bednet
count  8894.000000
mean      2.134585
std       0.802553
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       3.000000


#### 4.7- diarrhea

In [ ]:
extract_information(other_health_child,'diarrhea')

diarrhea    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13411
60.11744665590819
diarrhea
0.0    6662
1.0    2235
Name: count, dtype: int64
          diarrhea
count  8897.000000
mean      0.251208
std       0.433732
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


#### 4.8- diar_days

In [ ]:
extract_information(other_health_child,'diar_days')

diar_days    float64
dtype: object
0      NaN
1      0.0
2      2.0
3      7.0
4      3.0
5      4.0
6     14.0
7      5.0
8      1.0
9      6.0
10     8.0
11    11.0
12    10.0
13    12.0
14     9.0
dtype: float64
13418
60.14882553344092
diar_days
0.0     6662
7.0      564
3.0      518
2.0      387
4.0      338
14.0     145
1.0      109
5.0       92
6.0       38
10.0      14
8.0       12
12.0       6
9.0        4
11.0       1
Name: count, dtype: int64
         diar_days
count  8890.000000
mean      1.215973
std       2.636387
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max      14.000000


#### 4.9- diar_drink

In [ ]:
extract_information(other_health_child,'diar_drink')

diar_drink    float64
dtype: object
0    NaN
1    5.0
2    4.0
3    3.0
4    2.0
5    1.0
dtype: float64
20064
89.94082840236686
diar_drink
3.0    1046
4.0     574
2.0     318
1.0     216
5.0      90
Name: count, dtype: int64
        diar_drink
count  2244.000000
mean      3.001783
std       0.971276
min       1.000000
25%       3.000000
50%       3.000000
75%       4.000000
max       5.000000


#### 4.10- diar_medpro

In [ ]:
extract_information(other_health_child,'diar_medpro')

diar_medpro    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
20067
89.95427649273803
diar_medpro
1.0    1169
0.0    1072
Name: count, dtype: int64
       diar_medpro
count  2241.000000
mean      0.521642
std       0.499643
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.11- malaria

In [ ]:
extract_information(other_health_child,'malaria')

malaria    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13411
60.11744665590819
malaria
0.0    7703
1.0    1194
Name: count, dtype: int64
           malaria
count  8897.000000
mean      0.134203
std       0.340889
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.12- mal_days

In [ ]:
extract_information(other_health_child,'mal_days')

mal_days    float64
dtype: object
0      NaN
1      0.0
2      7.0
3      8.0
4      4.0
5     13.0
6      2.0
7      6.0
8      3.0
9     14.0
10    12.0
11     5.0
12    10.0
13     1.0
14     9.0
15    11.0
dtype: float64
13413
60.12641204948898
mal_days
0.0     7703
7.0      464
3.0      206
4.0      197
14.0      97
2.0       86
5.0       49
6.0       22
1.0       22
10.0      15
8.0       13
12.0      11
13.0       5
9.0        4
11.0       1
Name: count, dtype: int64
          mal_days
count  8895.000000
mean      0.796065
std       2.340146
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max      14.000000


#### 4.13- mal_drink

In [ ]:
extract_information(other_health_child,'mal_drink')

mal_drink    float64
dtype: object
0    NaN
1    4.0
2    3.0
3    2.0
4    5.0
5    1.0
dtype: float64
21613
96.88452573067957
mal_drink
3.0    269
4.0    177
2.0    128
1.0     95
5.0     26
Name: count, dtype: int64
        mal_drink
count  695.000000
mean     2.871942
std      1.058520
min      1.000000
25%      2.000000
50%      3.000000
75%      4.000000
max      5.000000


#### 4.14- mal_medpro

In [ ]:
extract_information(other_health_child,'mal_medpro')

mal_medpro    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
21105
94.60731576116191
mal_medpro
1.0    1057
0.0     146
Name: count, dtype: int64
        mal_medpro
count  1203.000000
mean      0.878637
std       0.326685
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.15- cough

In [ ]:
extract_information(other_health_child,'cough')

cough    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
13410
60.112963959117806
cough
1.0    4620
0.0    4278
Name: count, dtype: int64
             cough
count  8898.000000
mean      0.519218
std       0.499659
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.16- cough_days

In [ ]:
extract_information(other_health_child,'cough_days')

cough_days    float64
dtype: object
0      NaN
1      3.0
2      0.0
3      6.0
4     14.0
5      7.0
6      2.0
7      4.0
8     10.0
9      5.0
10    12.0
11     8.0
12     1.0
13    13.0
14    11.0
15     9.0
dtype: float64
13415
60.13537744306975
cough_days
0.0     4278
7.0     1778
3.0      790
4.0      630
14.0     505
2.0      363
5.0      256
6.0       92
1.0       72
10.0      37
8.0       35
12.0      30
13.0      15
9.0        8
11.0       4
Name: count, dtype: int64
        cough_days
count  8893.000000
mean      3.188688
std       3.939965
min       0.000000
25%       0.000000
50%       2.000000
75%       7.000000
max      14.000000


#### 4.17- cough_medpro

In [ ]:
extract_information(other_health_child,'cough_medpro')

cough_medpro    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
17680
79.25407925407926
cough_medpro
1.0    2725
0.0    1903
Name: count, dtype: int64
       cough_medpro
count   4628.000000
mean       0.588807
std        0.492103
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000


#### 4.18- worm_drugs

In [ ]:
extract_information(other_health_child,'worm_drugs')

worm_drugs    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13416
60.13986013986013
worm_drugs
0.0    7546
1.0    1346
Name: count, dtype: int64
        worm_drugs
count  8892.000000
mean      0.151372
std       0.358431
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


#### 4.19- worm_medpro

In [ ]:
extract_information(other_health_child,'worm_medpro')

worm_medpro    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
20956
93.93939393939394
worm_medpro
1.0    1179
0.0     173
Name: count, dtype: int64
       worm_medpro
count  1352.000000
mean      0.872041
std       0.334167
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.20- semi_solidfood

In [ ]:
extract_information(other_health_child,'semi_solidfood')

semi_solidfood    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
13408
60.10399856553703
semi_solidfood
1.0    7269
0.0    1631
Name: count, dtype: int64
       semi_solidfood
count     8900.000000
mean         0.816742
std          0.386900
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000


#### 4.21- num_meal

In [ ]:
extract_information(other_health_child,'num_meal')

num_meal    float64
dtype: object
0      NaN
1      6.0
2      3.0
3     12.0
4      7.0
5      1.0
6      9.0
7      4.0
8      5.0
9      2.0
10    10.0
11     8.0
12     0.0
13    11.0
dtype: float64
15044
67.43769051461359
num_meal
3.0     2764
4.0     1587
2.0      992
5.0      648
7.0      389
6.0      383
12.0     279
1.0      103
8.0       45
10.0      35
0.0       21
9.0       12
11.0       6
Name: count, dtype: int64
          num_meal
count  7264.000000
mean      4.022577
std       2.166875
min       0.000000
25%       3.000000
50%       3.000000
75%       4.000000
max      12.000000


#### 4.22- daysprotein

In [ ]:
extract_information(other_health_child,'daysprotein')

daysprotein    float64
dtype: object
0      4.0
1      NaN
2      7.0
3      6.0
4      5.0
5      3.0
6      2.0
7      1.0
8      0.0
9     12.0
10    14.0
11     8.0
12    10.0
13     9.0
14    13.0
dtype: float64
5854
26.24170701093778
daysprotein
7.0     7907
3.0     2018
6.0     1568
5.0     1511
2.0     1412
4.0     1389
1.0      455
0.0      133
14.0      36
8.0        8
10.0       8
12.0       5
9.0        2
13.0       2
Name: count, dtype: int64
        daysprotein
count  16454.000000
mean       5.345387
std        2.026989
min        0.000000
25%        4.000000
50%        6.000000
75%        7.000000
max       14.000000


#### 4.23- bcg

In [ ]:
extract_information(other_health_child,'bcg')

bcg    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15685
70.311099157253
bcg
1.0    6119
0.0     504
Name: count, dtype: int64
               bcg
count  6623.000000
mean      0.923902
std       0.265176
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.24- polio0

In [ ]:
extract_information(other_health_child,'polio0')

polio0    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15683
70.30213376367223
polio0
1.0    4855
0.0    1770
Name: count, dtype: int64
            polio0
count  6625.000000
mean      0.732830
std       0.442515
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.25- polio1

In [ ]:
extract_information(other_health_child,'polio1')

polio1    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15686
70.3155818540434
polio1
1.0    5827
0.0     795
Name: count, dtype: int64
            polio1
count  6622.000000
mean      0.879946
std       0.325050
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.26- polio2

In [ ]:
extract_information(other_health_child,'polio2')

polio2    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15688
70.32454724762417
polio2
1.0    5413
0.0    1207
Name: count, dtype: int64
            polio2
count  6620.000000
mean      0.817674
std       0.386142
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.27- polio3

In [ ]:
extract_information(other_health_child,'polio3')

polio3    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15692
70.34247803478573
polio3
1.0    5004
0.0    1612
Name: count, dtype: int64
            polio3
count  6616.000000
mean      0.756348
std       0.429317
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.28- dpt1

In [ ]:
extract_information(other_health_child,'dpt1')

dpt1    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15685
70.311099157253
dpt1
1.0    5733
0.0     890
Name: count, dtype: int64
              dpt1
count  6623.000000
mean      0.865620
std       0.341086
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.29- dpt2

In [ ]:
extract_information(other_health_child,'dpt2')

dpt2    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15687
70.32006455083378
dpt2
1.0    5329
0.0    1292
Name: count, dtype: int64
              dpt2
count  6621.000000
mean      0.804863
std       0.396336
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.30- dpt3

In [ ]:
extract_information(other_health_child,'dpt3')

dpt3    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15690
70.33351264120495
dpt3
1.0    4882
0.0    1736
Name: count, dtype: int64
              dpt3
count  6618.000000
mean      0.737685
std       0.439926
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.31- measles

In [ ]:
extract_information(other_health_child,'measles')

measles    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15695
70.3559261251569
measles
1.0    3622
0.0    2991
Name: count, dtype: int64
           measles
count  6613.000000
mean      0.547709
std       0.497756
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.32- vita_6wk

In [ ]:
extract_information(other_health_child,'vita_6wk')

vita_6wk    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15728
70.50385511923973
vita_6wk
1.0    3947
0.0    2633
Name: count, dtype: int64
          vita_6wk
count  6580.000000
mean      0.599848
std       0.489966
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.33- vita_6mo

In [ ]:
extract_information(other_health_child,'vita_6mo')

vita_6mo    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15718
70.45902815133584
vita_6mo
1.0    3324
0.0    3266
Name: count, dtype: int64
          vita_6mo
count  6590.000000
mean      0.504401
std       0.500019
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.34- iron

In [ ]:
extract_information(other_health_child,'iron')

iron    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
15725
70.49040702886856
iron
1.0    4689
0.0    1894
Name: count, dtype: int64
              iron
count  6583.000000
mean      0.712289
std       0.452730
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


#### 4.35- elapsed_time_mo

In [ ]:
extract_information(other_health_child,'elapsed_time_mo')

elapsed_time_mo    float64
dtype: object
0     0.000000
1          NaN
2     0.133333
3     0.233333
4     0.166667
        ...   
93   -1.633333
94    1.833333
95    1.800000
96    1.666667
97   -2.100000
Length: 98, dtype: float64
3363
15.075309306078537
elapsed_time_mo
 0.000000    17432
-0.033333      249
 0.033333      177
 0.133333       86
 1.633333       73
             ...  
-0.733333        1
 3.366667        1
 3.433333        1
-1.500000        1
 2.833333        1
Name: count, Length: 97, dtype: int64
       elapsed_time_mo
count     18945.000000
mean          0.040070
std           0.383237
min         -11.133333
25%           0.000000
50%           0.000000
75%           0.000000
max           3.466667


### Endline Anthro section

In [ ]:
endline_anthro = children_all_sections['ENDLINE_ANTHRO']
endline_anthro.columns

Index(['Ebirth_weighed', 'Ebirth_weight', 'Ebweight_units', 'Eweight_immcard',
       'Estatus_a', 'Eanthro_status', 'Eanthro_date', 'Eelapsed_time_mo',
       'Ec_hblevel', 'Ec_hblevel_flag', 'Ehb_11_child', 'Ehb_10_child',
       'Ehb_7_child', 'Ec_weight', 'Ec_weight_flag', 'Ewfa', 'Ewfa_neg2',
       'Ewfa_neg1', 'Ewfl', 'Ewfl_neg2', 'Ewfl_neg1', 'Eweigh_behave',
       'Eweigh_how', 'Ec_height', 'Ec_height_flag', 'Elhfa', 'Elhfa_neg2',
       'Elhfa_neg1', 'Eheight_how', 'Ec_muac', 'Ec_muac_flag', 'Emfa',
       'Emfa_neg3', 'Emfa_neg23', 'Emfa_neg2', 'Emfa_neg1', 'Eswelling',
       'Epress_skin', 'Esource', 'Emerge', 'Elhfa2', 'Ewfa2', 'Ewfl2', 'lhfa2',
       'wfa2', 'wfl2', 'female', 'juiceplus', 'meats', 'otherfv', 'Ejuiceplus',
       'Emeats', 'Eotherfv', 'sick_days', 'Esick_days', 'flag_mfa_imp',
       'flag_wfa_imp', 'flag_lhfa_imp', 'flag_c_hblevel_imp',
       'flag_sick_days_imp'],
      dtype='object')

In [ ]:
extract_information(endline_anthro,'Ebirth_weight')

Ebirth_weight    float64
dtype: object
0      NaN
1     3.40
2     4.00
3     3.50
4     3.00
      ... 
83    1.20
84    7.80
85    6.20
86    2.55
87    3.15
Length: 88, dtype: float64
20428
91.5725300340685
Ebirth_weight
3.00    354
3.50    229
4.00    149
3.20    104
3.40     96
       ... 
1.90      1
7.80      1
6.20      1
2.55      1
3.15      1
Name: count, Length: 87, dtype: int64
       Ebirth_weight
count    1880.000000
mean        3.436277
std         0.858602
min         0.000000
25%         3.000000
50%         3.300000
75%         3.800000
max        10.000000


In [ ]:
extract_information(endline_anthro,'Ebweight_units')

Ebweight_units    float64
dtype: object
0    NaN
1    2.0
2    1.0
dtype: float64
20414
91.50977227900304
Ebweight_units
2.0    1892
1.0       2
Name: count, dtype: int64
       Ebweight_units
count     1894.000000
mean         1.998944
std          0.032487
min          1.000000
25%          2.000000
50%          2.000000
75%          2.000000
max          2.000000


In [ ]:
weight_unit = {
    1.0: 'Gram',
    2.0: 'Kilogram',
}
mapping_func(endline_anthro,children_df,'Ebweight_units',weight_unit)

Ebweight_units    category
dtype: object
0         NaN
1    Kilogram
2        Gram
dtype: category
Categories (2, object): ['Gram', 'Kilogram']
20414
91.50977227900304
Ebweight_units
Kilogram    1892
Gram           2
Name: count, dtype: int64
       Ebweight_units
count            1894
unique              2
top          Kilogram
freq             1892


In [ ]:
suspicious_rows = endline_anthro[(endline_anthro['Ebweight_units'] == 'Gram')]
print(suspicious_rows[['Ebirth_weight', 'Ebweight_units']].shape)
suspicious_rows[['Ebirth_weight', 'Ebweight_units']].head()

(2, 2)


,Ebirth_weight,Ebweight_units
10396,NaN,Gram
13165,4.5,Gram


In [ ]:
suspicious_rows = endline_anthro[(endline_anthro['Ebweight_units'] == 'Gram') & (endline_anthro['Ebirth_weight'] <= 100)]
print(suspicious_rows[['Ebirth_weight', 'Ebweight_units']].shape)
suspicious_rows[['Ebirth_weight', 'Ebweight_units']].head()

(1, 2)


,Ebirth_weight,Ebweight_units
13165,4.5,Gram


In [ ]:
fix_condition = (endline_anthro['Ebweight_units'] == 'Grams') & (endline_anthro['Ebirth_weight'] <= 100)
endline_anthro.loc[fix_condition, 'Ebweight_units'] = 'Kilogram'

In [ ]:
endline_anthro.drop('Ebweight_units',axis=1,inplace=True)

In [ ]:
extract_information(endline_anthro,'Eweight_immcard')

Eweight_immcard    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
20414
91.50977227900304
Eweight_immcard
0.0    1020
1.0     874
Name: count, dtype: int64
       Eweight_immcard
count      1894.000000
mean          0.461457
std           0.498644
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(endline_anthro,'Estatus_a')

Estatus_a    float64
dtype: object
0    1.0
1    NaN
2    0.0
3   -1.0
dtype: float64
1739
7.795409718486641
Estatus_a
 1.0    20130
 0.0      433
-1.0        6
Name: count, dtype: int64
          Estatus_a
count  20569.000000
mean       0.978366
std        0.147482
min       -1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


In [ ]:
status_map = {
    1.0: 'Present',
    0.0: 'Not Present',
    -1.0: 'Missing/Exceptional'
}
mapping_func(endline_anthro,children_df,'Estatus_a',status_map)

Estatus_a    category
dtype: object
0                Present
1                    NaN
2            Not Present
3    Missing/Exceptional
dtype: category
Categories (3, object): ['Missing/Exceptional', 'Not Present', 'Present']
1739
7.795409718486641
Estatus_a
Present                20130
Not Present              433
Missing/Exceptional        6
Name: count, dtype: int64
       Estatus_a
count      20569
unique         3
top      Present
freq       20130


In [ ]:
status_map = {
    1.0: 'complete',
    2.0: 'Not complete',
}
mapping_func(endline_anthro,children_df,'Eanthro_status',status_map)

Eanthro_status    category
dtype: object
0             NaN
1        complete
2    Not complete
dtype: category
Categories (2, object): ['Not complete', 'complete']
11583
51.92307692307693
Eanthro_status
complete        10696
Not complete       29
Name: count, dtype: int64
       Eanthro_status
count           10725
unique              2
top          complete
freq            10696


In [ ]:
extract_information(endline_anthro,'Eanthro_date')

Eanthro_date    object
dtype: object
0      2014-07-07
1      2014-07-04
2      2014-07-29
3      2014-07-08
4      2014-07-26
          ...    
109    2014-07-22
110    2014-07-09
111    2014-07-15
112    2014-07-18
113    2014-07-13
Length: 114, dtype: object
312
1.3986013986013985
Eanthro_date
2014-05-06    445
2014-06-12    407
2014-05-09    406
2014-06-18    398
2014-06-11    395
             ... 
2014-09-09      4
2014-09-10      4
2014-04-21      3
2014-04-19      2
2014-04-22      1
Name: count, Length: 113, dtype: int64
       Eanthro_date
count         21996
unique          113
top      2014-05-06
freq            445


In [ ]:
endline_anthro['Eanthro_date'] = pd.to_datetime(endline_anthro['Eanthro_date'], errors='coerce')
extract_information(endline_anthro,'Eanthro_date')

Eanthro_date    datetime64[ns]
dtype: object
0     2014-07-07
1     2014-07-04
2     2014-07-29
3     2014-07-08
4     2014-07-26
         ...    
109   2014-07-22
110   2014-07-09
111   2014-07-15
112   2014-07-18
113   2014-07-13
Length: 114, dtype: datetime64[ns]
312
1.3986013986013985
Eanthro_date
2014-05-06    445
2014-06-12    407
2014-05-09    406
2014-06-18    398
2014-06-11    395
             ... 
2014-09-09      4
2014-09-10      4
2014-04-21      3
2014-04-19      2
2014-04-22      1
Name: count, Length: 113, dtype: int64
                        Eanthro_date
count                          21996
mean   2014-05-24 11:30:28.477905152
min              2014-03-21 00:00:00
25%              2014-04-12 00:00:00
50%              2014-06-11 00:00:00
75%              2014-07-01 00:00:00
max              2014-09-10 00:00:00


In [ ]:
extract_information(endline_anthro,'Ec_hblevel')

Ec_hblevel    float64
dtype: object
0       NaN
1      10.8
2      13.3
3      11.4
4      14.4
       ... 
129    16.0
130    19.4
131    20.6
132    16.8
133    17.3
Length: 134, dtype: float64
11623
52.10238479469249
Ec_hblevel
11.7    371
11.6    368
12.1    349
11.5    349
12.3    337
       ... 
16.0      1
19.4      1
20.6      1
16.8      1
17.3      1
Name: count, Length: 133, dtype: int64
         Ec_hblevel
count  10685.000000
mean      11.700028
std        1.359328
min        0.000000
25%       10.900000
50%       11.800000
75%       12.600000
max       20.600000


In [ ]:
extract_information(endline_anthro,'Ec_hblevel_flag')

Ec_hblevel_flag    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
Ec_hblevel_flag
1    11623
0    10685
Name: count, dtype: int64
       Ec_hblevel_flag
count     22308.000000
mean          0.521024
std           0.499569
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000


In [ ]:
extract_information(endline_anthro,'Ehb_11_child')

Ehb_11_child    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
11623
52.10238479469249
Ehb_11_child
0.0    7935
1.0    2750
Name: count, dtype: int64
       Ehb_11_child
count  10685.000000
mean       0.257370
std        0.437205
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000


In [ ]:
endline_anthro[(endline_anthro['Ehb_11_child']==1) & (endline_anthro['Ec_hblevel']<=11)].shape

(2750, 59)

In [ ]:
extract_information(endline_anthro,'Ehb_10_child')

Ehb_10_child    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
11623
52.10238479469249
Ehb_10_child
0.0    9758
1.0     927
Name: count, dtype: int64
       Ehb_10_child
count  10685.000000
mean       0.086757
std        0.281492
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


In [ ]:
endline_anthro[(endline_anthro['Ehb_10_child']==1) & (endline_anthro['Ec_hblevel']<=10)].shape

(927, 59)

In [ ]:
extract_information(endline_anthro,'Ehb_7_child')

Ehb_7_child    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
11623
52.10238479469249
Ehb_7_child
0.0    10644
1.0       41
Name: count, dtype: int64
        Ehb_7_child
count  10685.000000
mean       0.003837
std        0.061829
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


In [ ]:
endline_anthro[(endline_anthro['Ehb_7_child']==1) & (endline_anthro['Ec_hblevel']<=7)].shape

(41, 59)

In [ ]:
extract_information(endline_anthro,'meats')

meats    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
15564
69.76869284561592
meats
0.0    6131
1.0     613
Name: count, dtype: int64
             meats
count  6744.000000
mean      0.090896
std       0.287482
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(endline_anthro,'Emeats')

Emeats    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
13588
60.91088398780706
Emeats
0.0    7823
1.0     897
Name: count, dtype: int64
            Emeats
count  8720.000000
mean      0.102867
std       0.303802
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(endline_anthro,'flag_sick_days_imp')
#TODO

flag_sick_days_imp    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
flag_sick_days_imp
0.0    21584
1.0      724
Name: count, dtype: int64
       flag_sick_days_imp
count        22308.000000
mean             0.032455
std              0.177208
min              0.000000
25%              0.000000
50%              0.000000
75%              0.000000
max              1.000000


In [ ]:
extract_information(endline_anthro,'sick_days')

sick_days    float64
dtype: object
0           NaN
1      3.000000
2      0.000000
3      6.000000
4      7.000000
         ...   
310    6.461538
311    7.315790
312    5.592593
313    5.916666
314    5.388889
Length: 315, dtype: float64
12703
56.943697328312716
sick_days
0.000000     3218
7.000000     1434
3.000000      733
4.000000      543
14.000000     526
             ... 
5.571429        1
3.777778        1
6.363637        1
5.400000        1
5.388889        1
Name: count, Length: 314, dtype: int64
         sick_days
count  9605.000000
mean      5.200468
std       5.774317
min       0.000000
25%       0.000000
50%       4.000000
75%       7.000000
max      42.000000


In [ ]:
extract_information(endline_anthro,'Esick_days')

Esick_days    float64
dtype: object
0      NaN
1      8.0
2      4.0
3      0.0
4      7.0
5      2.0
6     14.0
7     11.0
8     10.0
9     17.0
10     3.0
11     5.0
12    12.0
13    21.0
14    20.0
15     6.0
16     9.0
17     1.0
18    15.0
19    13.0
20    19.0
21    18.0
22    16.0
dtype: float64
11626
52.115832885063654
Esick_days
0.0     4227
7.0     1838
3.0      931
4.0      741
14.0     579
2.0      549
5.0      293
10.0     276
9.0      196
6.0      180
11.0     172
8.0      115
1.0      106
21.0     104
17.0      95
12.0      84
13.0      56
18.0      44
15.0      38
16.0      37
19.0      12
20.0       9
Name: count, dtype: int64
         Esick_days
count  10682.000000
mean       4.279255
std        4.847800
min        0.000000
25%        0.000000
50%        3.000000
75%        7.000000
max       21.000000


In [ ]:
extract_information(endline_anthro,'juiceplus')

juiceplus    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
14767
66.19598350367582
juiceplus
1.0    4623
0.0    2918
Name: count, dtype: int64
         juiceplus
count  7541.000000
mean      0.613049
std       0.487085
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(endline_anthro,'Ejuiceplus')

Ejuiceplus    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
13433
60.21606598529675
Ejuiceplus
1.0    7457
0.0    1418
Name: count, dtype: int64
        Ejuiceplus
count  8875.000000
mean      0.840225
std       0.366418
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(endline_anthro,'otherfv')

otherfv    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
15564
69.76869284561592
otherfv
1.0    4054
0.0    2690
Name: count, dtype: int64
           otherfv
count  6744.000000
mean      0.601127
std       0.489703
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(endline_anthro,'Eotherfv')

Eotherfv    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
13588
60.91088398780706
Eotherfv
1.0    7172
0.0    1548
Name: count, dtype: int64
          Eotherfv
count  8720.000000
mean      0.822477
std       0.382133
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(endline_anthro,'Eswelling')

Eswelling    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
11534
51.70342478034786
Eswelling
0.0    10635
1.0      139
Name: count, dtype: int64
          Eswelling
count  10774.000000
mean       0.012901
std        0.112855
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000


In [ ]:
extract_information(endline_anthro,'Ec_height')

Ec_height    float64
dtype: object
0              NaN
1        86.199997
2        91.199997
3        80.500000
4        68.800003
           ...    
1308    114.049999
1309     44.500000
1310     49.799999
1311     96.750000
1312     64.850002
Length: 1313, dtype: float64
11678
52.348933118163885
Ec_height
78.000000     102
79.000000      53
77.500000      47
80.000000      44
81.000000      43
             ... 
94.549999       1
88.299999       1
111.250000      1
130.100000      1
83.550000       1
Name: count, Length: 1312, dtype: int64
          Ec_height
count  10630.000000
mean      88.938405
std       14.997230
min       42.599998
25%       78.199997
50%       88.000000
75%      100.800003
max      134.500000


In [ ]:
extract_information(endline_anthro,'Ec_height_flag')

Ec_height_flag    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
Ec_height_flag
1    11678
0    10630
Name: count, dtype: int64
       Ec_height_flag
count    22308.000000
mean         0.523489
std          0.499459
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


In [ ]:
extract_information(endline_anthro,'Ec_weight_flag')

Ec_weight_flag    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
Ec_weight_flag
1    11655
0    10653
Name: count, dtype: int64
       Ec_weight_flag
count    22308.000000
mean         0.522458
std          0.499507
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000


In [ ]:
extract_information(endline_anthro,'Ec_weight')

Ec_weight    float64
dtype: object
0            NaN
1      13.100000
2      13.400000
3      10.500000
4       8.600000
         ...    
480    10.533333
481    16.700000
482    19.750000
483    21.700000
484     8.900000
Length: 485, dtype: float64
11655
52.24583109198494
Ec_weight
11.00    120
10.00    118
12.40    117
11.70    116
12.10    114
        ... 
15.80      1
16.70      1
19.75      1
21.70      1
8.90       1
Name: count, Length: 484, dtype: int64
          Ec_weight
count  10653.000000
mean      13.077715
std        3.886085
min        3.000000
25%       10.200000
50%       12.800000
75%       15.900000
max       24.000000


In [ ]:
endline_anthro[(endline_anthro['Ec_weight_flag']==0) & (endline_anthro['Ec_weight'].notnull())].shape

(10653, 59)

In [ ]:
endline_anthro[(endline_anthro['Ec_weight_flag']==0) & (endline_anthro['Ec_weight'].notnull()) & (children_df['wemp']==1)].shape

(2606, 59)

In [ ]:
endline_anthro[(endline_anthro['Ec_weight_flag']==0) & (endline_anthro['Ec_weight'].isnull())].shape


(0, 59)

In [ ]:
extract_information(endline_anthro,'Eweigh_behave')

Eweigh_behave    float64
dtype: object
0    NaN
1    2.0
2    1.0
3    9.7
dtype: float64
11563
51.83342298726914
Eweigh_behave
2.0    10250
1.0      494
9.7        1
Name: count, dtype: int64
       Eweigh_behave
count   10745.000000
mean        1.954742
std         0.222371
min         1.000000
25%         2.000000
50%         2.000000
75%         2.000000
max         9.700000


In [ ]:
extract_information(endline_anthro,'Eweigh_how')

Eweigh_how    float64
dtype: object
0    NaN
1    2.0
2    1.0
dtype: float64
11562
51.828940290478755
Eweigh_how
1.0    7727
2.0    3019
Name: count, dtype: int64
         Eweigh_how
count  10746.000000
mean       1.280942
std        0.449480
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        2.000000


In [ ]:
extract_information(endline_anthro,'Eheight_how')

Eheight_how    float64
dtype: object
0    NaN
1    1.0
2    2.0
dtype: float64
11563
51.83342298726914
Eheight_how
2.0    8161
1.0    2584
Name: count, dtype: int64
        Eheight_how
count  10745.000000
mean       1.759516
std        0.427397
min        1.000000
25%        2.000000
50%        2.000000
75%        2.000000
max        2.000000


In [ ]:
extract_information(endline_anthro,'Estatus_a')

Estatus_a    category
dtype: object
0                Present
1                    NaN
2            Not Present
3    Missing/Exceptional
dtype: category
Categories (3, object): ['Missing/Exceptional', 'Not Present', 'Present']
1739
7.795409718486641
Estatus_a
Present                20130
Not Present              433
Missing/Exceptional        6
Name: count, dtype: int64
       Estatus_a
count      20569
unique         3
top      Present
freq       20130


In [ ]:
extract_information(endline_anthro,'Eanthro_status')

Eanthro_status    category
dtype: object
0             NaN
1        complete
2    Not complete
dtype: category
Categories (2, object): ['Not complete', 'complete']
11583
51.92307692307693
Eanthro_status
complete        10696
Not complete       29
Name: count, dtype: int64
       Eanthro_status
count           10725
unique              2
top          complete
freq            10696


In [ ]:
extract_information(endline_anthro,'Eanthro_date')

Eanthro_date    datetime64[ns]
dtype: object
0     2014-07-07
1     2014-07-04
2     2014-07-29
3     2014-07-08
4     2014-07-26
         ...    
109   2014-07-22
110   2014-07-09
111   2014-07-15
112   2014-07-18
113   2014-07-13
Length: 114, dtype: datetime64[ns]
312
1.3986013986013985
Eanthro_date
2014-05-06    445
2014-06-12    407
2014-05-09    406
2014-06-18    398
2014-06-11    395
             ... 
2014-09-09      4
2014-09-10      4
2014-04-21      3
2014-04-19      2
2014-04-22      1
Name: count, Length: 113, dtype: int64
                        Eanthro_date
count                          21996
mean   2014-05-24 11:30:28.477905152
min              2014-03-21 00:00:00
25%              2014-04-12 00:00:00
50%              2014-06-11 00:00:00
75%              2014-07-01 00:00:00
max              2014-09-10 00:00:00


In [ ]:
extract_information(endline_anthro,'Eelapsed_time_mo')

Eelapsed_time_mo    float64
dtype: object
0       0.001142
1       0.001571
2       0.001733
3       0.002596
4       0.001511
          ...   
4724    0.003517
4725    0.005809
4726    0.171451
4727    0.002897
4728    0.002532
Length: 4729, dtype: float64
1739
7.795409718486641
Eelapsed_time_mo
0.000729    30
0.003649    21
0.000902    19
0.000106    19
0.000708    18
            ..
0.002897     1
0.003830     1
0.002780     1
1.834620     1
0.005046     1
Name: count, Length: 4728, dtype: int64
       Eelapsed_time_mo
count      20569.000000
mean           0.065369
std            0.413106
min            0.000001
25%            0.001456
50%            0.003003
75%            0.005292
max            5.737291


In [ ]:
extract_information(endline_anthro,'female')

female    float64
dtype: object
0    1.0
1    0.0
dtype: float64
0
0.0
female
0.0    11376
1.0    10932
Name: count, dtype: int64
             female
count  22308.000000
mean       0.490048
std        0.499912
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(endline_anthro,'flag_sick_days_imp')

flag_sick_days_imp    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
flag_sick_days_imp
0.0    21584
1.0      724
Name: count, dtype: int64
       flag_sick_days_imp
count        22308.000000
mean             0.032455
std              0.177208
min              0.000000
25%              0.000000
50%              0.000000
75%              0.000000
max              1.000000


In [ ]:
print(endline_anthro[endline_anthro['flag_sick_days_imp']==1 & (endline_anthro['sick_days'] > endline_anthro['Esick_days'])].shape)
endline_anthro[endline_anthro['flag_sick_days_imp']==1 & (endline_anthro['sick_days'] > endline_anthro['Esick_days'])].head()

(18716, 59)


,Ebirth_weighed,Ebirth_weight,Eweight_immcard,Estatus_a,Eanthro_status,Eanthro_date,Eelapsed_time_mo,Ec_hblevel,Ec_hblevel_flag,Ehb_11_child,Ehb_10_child,Ehb_7_child,Ec_weight,Ec_weight_flag,Ewfa,Ewfa_neg2,Ewfa_neg1,Ewfl,Ewfl_neg2,Ewfl_neg1,Eweigh_behave,Eweigh_how,Ec_height,Ec_height_flag,Elhfa,Elhfa_neg2,Elhfa_neg1,Eheight_how,Ec_muac,Ec_muac_flag,Emfa,Emfa_neg3,Emfa_neg23,Emfa_neg2,Emfa_neg1,Eswelling,Epress_skin,Esource,Emerge,Elhfa2,Ewfa2,Ewfl2,lhfa2,wfa2,wfl2,female,juiceplus,meats,otherfv,Ejuiceplus,Emeats,Eotherfv,sick_days,Esick_days,flag_mfa_imp,flag_wfa_imp,flag_lhfa_imp,flag_c_hblevel_imp,flag_sick_days_imp
0,NaN,NaN,NaN,Present,NaN,2014-07-07,0.001142,NaN,1,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
1,0.0,NaN,NaN,Present,complete,2014-07-07,0.001571,10.8,0,1.0,0.0,0.0,13.1,0,0.806834,0.0,0.0,1.315675,0.0,0.0,2.0,2.0,86.199997,0,-0.247404,0.0,0.0,1.0,15.5,0,0.340733,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,-0.25,0.81,1.27,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0,1.0,1.0,NaN,8.0,0.0,0.0,0.0,0.0,0.0
2,NaN,NaN,NaN,Present,NaN,2014-07-07,0.001733,NaN,1,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
3,NaN,NaN,NaN,Present,NaN,2014-07-07,0.002596,NaN,1,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
4,NaN,NaN,NaN,Present,complete,2014-07-07,0.001511,13.3,0,0.0,0.0,0.0,13.4,0,-0.844948,0.0,0.0,0.578080,0.0,0.0,2.0,1.0,91.199997,0,-1.929466,0.0,1.0,2.0,15.5,0,-0.334844,0.0,0.0,0.0,0.0,0.0,1.0,4.0,1.0,-1.93,-0.84,0.39,-2.38,-0.54,0.88,1.0,1.0,0.0,0.0,1.0,0.0,1.0,3.0,4.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
extract_information(endline_anthro,'flag_lhfa_imp')

flag_lhfa_imp    float64
dtype: object
0    0.0
1    1.0
dtype: float64
0
0.0
flag_lhfa_imp
0.0    21766
1.0      542
Name: count, dtype: int64
       flag_lhfa_imp
count   22308.000000
mean        0.024296
std         0.153971
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000


In [ ]:
extract_information(endline_anthro,'lhfa2')

lhfa2    float64
dtype: object
0       NaN
1     -2.38
2     -2.06
3     -3.28
4     -0.63
       ... 
877   -4.70
878    2.06
879    0.13
880    4.39
881    2.67
Length: 882, dtype: float64
17129
76.78411332257487
lhfa2
-1.32    37
-1.96    28
-1.42    24
-1.55    24
-2.06    23
         ..
-5.60     1
 3.00     1
 3.86     1
-4.09     1
 4.80     1
Name: count, Length: 881, dtype: int64
             lhfa2
count  5179.000000
mean     -1.337864
std       1.837289
min      -7.010000
25%      -2.470000
50%      -1.400000
75%      -0.305000
max       7.170000


In [ ]:
extract_information(endline_anthro,'Elhfa')

Elhfa    float64
dtype: object
0            NaN
1      -0.247404
2      -1.929466
3      -2.150501
4      -2.440515
          ...   
6633   -0.346277
6634   -1.648968
6635   -1.905193
6636   -0.613608
6637   -2.141229
Length: 6638, dtype: float64
13917
62.38569123184507
Elhfa
-1.561157    9
-2.380852    7
-0.932514    7
-0.383294    6
-1.988163    6
            ..
-2.440515    1
-1.929466    1
-0.247404    1
-2.141229    1
-2.093826    1
Name: count, Length: 6637, dtype: int64
             Elhfa
count  8391.000000
mean     -1.542731
std       1.304816
min      -4.993944
25%      -2.402782
50%      -1.602140
75%      -0.749474
max       4.575743


In [ ]:
extract_information(endline_anthro,'Elhfa2')

Elhfa2    float64
dtype: object
0       NaN
1     -0.25
2     -1.93
3     -2.15
4     -2.44
       ... 
735    3.68
736    2.34
737    1.81
738    2.63
739    2.50
Length: 740, dtype: float64
13917
62.38569123184507
Elhfa2
-2.17    44
-1.25    40
-1.42    40
-1.62    38
-1.51    38
         ..
-4.54     1
-4.63     1
 1.81     1
 2.63     1
 2.50     1
Name: count, Length: 739, dtype: int64
            Elhfa2
count  8391.000000
mean     -1.514613
std       1.309768
min      -5.120000
25%      -2.380000
50%      -1.570000
75%      -0.720000
max       4.810000


In [ ]:
extract_information(endline_anthro,'Elhfa_neg1')

Elhfa_neg1    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13917
62.38569123184507
Elhfa_neg1
1.0    5778
0.0    2613
Name: count, dtype: int64
        Elhfa_neg1
count  8391.000000
mean      0.688595
std       0.463096
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(endline_anthro,'Elhfa_neg2')

Elhfa_neg2    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13917
62.38569123184507
Elhfa_neg2
0.0    5314
1.0    3077
Name: count, dtype: int64
        Elhfa_neg2
count  8391.000000
mean      0.366702
std       0.481933
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(endline_anthro,'wfa2')

wfa2    float64
dtype: object
0       NaN
1     -0.54
2     -0.27
3     -0.66
4      1.12
       ... 
673   -1.79
674    3.18
675    2.15
676    2.32
677   -3.60
Length: 678, dtype: float64
17158
76.91411152949614
wfa2
-0.35    30
-0.48    29
 0.40    27
 0.08    27
-1.14    26
         ..
-1.79     1
 3.18     1
 2.15     1
 2.32     1
-3.60     1
Name: count, Length: 677, dtype: int64
              wfa2
count  5150.000000
mean     -0.249588
std       1.332886
min      -4.660000
25%      -1.100000
50%      -0.265000
75%       0.600000
max       5.250000


In [ ]:
temp_df = endline_anthro[((endline_anthro['Ewfa2'].notnull()) & (endline_anthro['Ewfa'].notnull()) &(endline_anthro['Ewfa2'].round(2) != endline_anthro['Ewfa'].round(2)))]

In [ ]:
temp_df[['Ewfa', 'Ewfa2']]

,Ewfa,Ewfa2
28,-0.166586,-0.16
41,0.644267,0.65
94,-1.032747,-1.04
134,0.866233,0.86
135,-3.929522,-3.82
...,...,...
22222,-1.786517,-1.78
22232,1.735020,1.73
22257,-2.005764,-2.00
22273,0.464745,0.47


In [ ]:
diff = temp_df['Ewfa'] - temp_df['Ewfa2']
print(diff.describe())

count    1299.000000
mean       -0.007761
std         0.031554
min        -0.348782
25%        -0.006714
50%        -0.005077
75%         0.005911
max         0.010716
dtype: float64


In [ ]:
extract_information(endline_anthro,'Ewfa2')

Ewfa2    float64
dtype: object
0       NaN
1      0.81
2     -0.84
3     -1.14
4     -0.53
       ... 
620    2.07
621   -4.30
622    2.41
623    3.15
624   -3.18
Length: 625, dtype: float64
13849
62.08086785009862
Ewfa2
-0.35    68
-0.11    66
-0.27    61
-0.19    59
-0.04    57
         ..
-3.77     1
 2.49     1
-3.00     1
-3.25     1
 1.85     1
Name: count, Length: 624, dtype: int64
             Ewfa2
count  8459.000000
mean     -0.579539
std       1.068047
min      -4.670000
25%      -1.255000
50%      -0.580000
75%       0.090000
max       4.970000


In [ ]:
extract_information(endline_anthro,'Ewfa')

Ewfa    float64
dtype: object
0            NaN
1       0.806834
2      -0.844948
3      -1.139438
4      -0.526015
          ...   
4846   -0.562607
4847   -3.192550
4848   -0.150982
4849   -1.937228
4850   -2.107954
Length: 4851, dtype: float64
13849
62.08086785009862
Ewfa
-0.037992    9
-0.745546    8
-0.406636    8
-0.629634    8
 0.206432    8
            ..
-1.066226    1
-1.153745    1
 0.082556    1
 0.670633    1
 0.607874    1
Name: count, Length: 4850, dtype: int64
              Ewfa
count  8459.000000
mean     -0.580592
std       1.070184
min      -4.972263
25%      -1.252447
50%      -0.582956
75%       0.087723
max       4.678680


In [ ]:
endline_anthro[(endline_anthro['Ewfa_neg2']==1)&(endline_anthro['Ewfa']<=-2)].shape

(741, 59)

In [ ]:
extract_information(endline_anthro,'Ewfa_neg1')

Ewfa_neg1    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13849
62.08086785009862
Ewfa_neg1
0.0    5594
1.0    2865
Name: count, dtype: int64
         Ewfa_neg1
count  8459.000000
mean      0.338693
std       0.473293
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


In [ ]:
endline_anthro[(endline_anthro['Ewfa_neg1']==1)&(endline_anthro['Ewfa']<=-1)].shape

(2865, 59)

In [ ]:
extract_information(endline_anthro,'Ewfa_neg2')

Ewfa_neg2    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13849
62.08086785009862
Ewfa_neg2
0.0    7718
1.0     741
Name: count, dtype: int64
         Ewfa_neg2
count  8459.000000
mean      0.087599
std       0.282728
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
endline_anthro.groupby('Ewfa_neg1')['Ewfa2'].agg(['min', 'max', 'count'])

,min,max,count
Ewfa_neg1,,,
0.0,-1.00,4.97,5594
1.0,-4.67,-0.97,2865


In [ ]:
condition_neg2 = endline_anthro['Ewfa2'] < -2

matches = (condition_neg2 == endline_anthro['Ewfa_neg2'])

print(matches.value_counts())

False    13856
True      8452
Name: count, dtype: int64


In [ ]:
diff = children_df['Ewfl'] - children_df['Ewfl2']
print(diff.describe())

count    8394.000000
mean        0.110276
std         0.084262
min        -0.383659
25%         0.042502
50%         0.135402
75%         0.170855
max         0.607896
dtype: float64


In [ ]:
diff = children_df['Elhfa'] - children_df['Elhfa2']
print(diff.describe())

count    8391.000000
mean       -0.028117
std         0.098351
min        -0.393495
25%        -0.004455
50%        -0.000505
75%         0.002962
max         0.230151
dtype: float64


In [ ]:
columns_to_drop_children += ['Ewfa_neg2','Ewfa_neg1','Ewfl_neg2','Ewfl_neg1','Elhfa_neg2','Elhfa_neg1','Emfa_neg3','Emfa_neg23','Emfa_neg2','Emfa_neg1']

In [ ]:
columns_to_drop_children += ['Ewfa2','Elhfa2','Ewfl2']

In [ ]:
columns_to_drop_children += ['Ehb_7_child', 'Ehb_10_child','Ehb_11_child']


In [ ]:
extract_information(endline_anthro,'Ewfl')

Ewfl    float64
dtype: object
0            NaN
1       1.315675
2       0.578080
3      -0.046408
4       0.844789
          ...   
5687    1.031498
5688   -1.327465
5689    0.188563
5690   -1.311021
5691   -1.242690
Length: 5692, dtype: float64
12495
56.01129639591178
Ewfl
 0.011787    10
-0.001725     9
 0.817985     8
 0.255386     8
 0.612141     8
             ..
 2.153540     1
-0.641386     1
 1.418017     1
 2.595426     1
-0.977915     1
Name: count, Length: 5691, dtype: int64
              Ewfl
count  9813.000000
mean      0.446264
std       0.998034
min      -4.804864
25%      -0.166776
50%       0.462491
75%       1.074553
max       4.933459


In [ ]:
extract_information(endline_anthro,'Ewfl2')

Ewfl2    float64
dtype: object
0       NaN
1      1.27
2      0.39
3     -0.05
4      0.90
       ... 
613   -3.30
614   -3.35
615   -3.00
616   -1.71
617   -2.35
Length: 618, dtype: float64
13868
62.16603908911601
Ewfl2
 0.59    50
 0.61    49
 0.42    49
 0.82    49
 0.38    48
         ..
 3.02     1
-3.24     1
-1.69     1
-3.48     1
-2.51     1
Name: count, Length: 617, dtype: int64
             Ewfl2
count  8440.000000
mean      0.373091
std       1.016165
min      -4.610000
25%      -0.250000
50%       0.390000
75%       1.010000
max       5.290000


In [ ]:
extract_information(endline_anthro,'Ec_muac')

Ec_muac    float64
dtype: object
0        NaN
1      15.50
2      14.30
3      14.50
4      15.30
       ...  
230    18.25
231     9.85
232    11.90
233     8.90
234    24.90
Length: 235, dtype: float64
11626
52.115832885063654
Ec_muac
14.50    406
15.00    397
15.20    397
15.50    393
15.40    306
        ... 
18.25      1
9.85       1
11.90      1
8.90       1
24.90      1
Name: count, Length: 234, dtype: int64
            Ec_muac
count  10682.000000
mean      14.963921
std        1.355945
min        8.000000
25%       14.200000
50%       15.000000
75%       15.850000
max       29.600000


In [ ]:
extract_information(endline_anthro,'Ec_muac_flag')

Ec_muac_flag    int64
dtype: object
0    1
1    0
dtype: int64
0
0.0
Ec_muac_flag
1    11626
0    10682
Name: count, dtype: int64
       Ec_muac_flag
count  22308.000000
mean       0.521158
std        0.499563
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000


In [ ]:
extract_information(endline_anthro,'Emfa')

Emfa    float64
dtype: object
0            NaN
1       0.340733
2      -0.334844
3      -0.701370
4       0.214205
          ...   
4394   -2.268045
4395   -0.688003
4396    0.633029
4397    0.080997
4398    1.279937
Length: 4399, dtype: float64
14160
63.47498655190963
Emfa
-0.229281    10
 0.261741    10
-0.196472    10
-0.569998    10
-0.505521     9
             ..
 0.567425     1
-1.414200     1
 0.761325     1
 1.060100     1
-0.065663     1
Name: count, Length: 4398, dtype: int64
              Emfa
count  8148.000000
mean     -0.395721
std       0.959944
min      -4.648570
25%      -1.014405
50%      -0.399268
75%       0.227067
max       4.650838


In [ ]:
endline_anthro['Esource'] = endline_anthro['Esource'].astype('category')
children_df['Esource'] = children_df['Esource'].astype('category')
extract_information(endline_anthro,'Esource')

Esource    category
dtype: object
0      98.0
1       1.0
2       6.0
3       4.0
4       3.0
5       NaN
6       7.0
7       2.0
8      99.0
9       5.0
10      8.0
11      9.0
12    999.0
dtype: category
Categories (12, float64): [1.0, 2.0, 3.0, 4.0, ..., 9.0, 98.0, 99.0, 999.0]
1897
8.503675811368119
Esource
3.0      7308
6.0      7069
1.0      3611
7.0       891
4.0       678
98.0      385
2.0       280
5.0        88
9.0        39
99.0       33
8.0        17
999.0      12
Name: count, dtype: int64
        Esource
count   20411.0
unique     12.0
top         3.0
freq     7308.0


In [ ]:
endline_anthro['Emerge'] = endline_anthro['Emerge'].astype('category')
children_df['Emerge'] = children_df['Emerge'].astype('category')
extract_information(endline_anthro,'Emerge')

Emerge    category
dtype: object
0      3.0
1      1.0
2      NaN
3      4.0
4      6.0
5     10.0
6      7.0
7      9.0
8      2.0
9      5.0
10     8.0
dtype: category
Categories (10, float64): [1.0, 2.0, 3.0, 4.0, ..., 7.0, 8.0, 9.0, 10.0]
10656
47.76761699838623
Emerge
1.0     8395
4.0     1359
2.0      636
6.0      424
10.0     277
7.0      273
3.0      144
9.0       71
8.0       69
5.0        4
Name: count, dtype: int64
         Emerge
count   11652.0
unique     10.0
top         1.0
freq     8395.0


### BASELINE_ANTHRO

In [ ]:
start_idx = children_df.columns.get_loc('BASELINE_ANTHRO')
end_idx = children_df.columns.get_loc('ENDLINE_SURVEY')

decision_making_cols = children_df.columns[start_idx + 1 : end_idx].tolist()

print({len(decision_making_cols)})

print("coulmns name :")
for col in decision_making_cols:
    print(col)

{35}
coulmns name :
anthro_date
c_weight
wfa
wfa_neg2
wfa_neg1
wfl
wfl_neg2
wfl_neg1
c_weight_flag
c_weight_flagwhy
weigh_behave
weigh_how
c_height
lhfa
lhfa_neg2
lhfa_neg1
c_height_flag
c_height_flagwhy
height_how
c_muac
mfa
mfa_neg3
mfa_neg23
mfa_neg2
mfa_neg1
c_muac_flag
c_muac_flagwhy
press_skin
anthro_status
c_hblevel
hb_11_child
hb_10_child
hb_7_child
c_hblevel_flag
c_hblevel_flagwhy


In [ ]:
columns_to_analyze = [
    'press_skin',
    'anthro_status',
    'c_hblevel',
    'hb_11_child',
    'hb_10_child',
    'hb_7_child',
    'c_hblevel_flag',
    'c_hblevel_flagwhy'
]

for col in columns_to_analyze:
    analyze_any_column(children_df, col)

    print("-------------------------------------------------")

Column: press_skin
type :float64
missed values count 17366 from 22308 (77.85%)
 uniqe values count 2
-------------------------------------------------
first 10 rows
0    NaN
1    NaN
2    NaN
3    NaN
4    1.0
5    NaN
6    1.0
7    NaN
8    NaN
9    NaN
-------------------------------------------------
dtype = (Categorical / Flag)

 top 5 frequent values
            Count      %
press_skin              
1.0          3843  77.76
0.0          1099  22.24
-------------------------------------------------
Column: anthro_status
type :float64
missed values count 17021 from 22308 (76.30%)
 uniqe values count 2
-------------------------------------------------
first 10 rows
0    NaN
1    NaN
2    NaN
3    NaN
4    1.0
5    NaN
6    1.0
7    NaN
8    NaN
9    NaN
-------------------------------------------------
dtype = (Categorical / Flag)

 top 5 frequent values
               Count      %
anthro_status              
1.0             5249  99.28
2.0               38   0.72
-------------------

In [ ]:
import pandas as pd

children_df['anthro_date'] = pd.to_datetime(children_df['anthro_date'], errors='coerce')

min_val = children_df['anthro_date'].min()
max_val = children_df['anthro_date'].max()

print(f"  (min): {min_val.date()}")
print(f"  (max): {max_val.date()}")

  (min): 1970-01-01
  (max): 1970-01-01


In [ ]:
children_df['anthro_date'] = pd.to_datetime(children_df['anthro_date'], unit='D', errors='coerce')

min_val = children_df['anthro_date'].min()
max_val = children_df['anthro_date'].max()

print(f"  (min): {min_val.date()}")
print(f"  (max): {max_val.date()}")

  (min): 1970-01-01
  (max): 1970-01-01


In [ ]:

start_idx = children_df.columns.get_loc('ENDLINE_SURVEY')
end_idx = children_df.columns.get_loc('FOOD__RECALL')

decision_making_cols = children_df.columns[start_idx + 1 : end_idx].tolist()

print({len(decision_making_cols)})

print("coulmns name :")
for col in decision_making_cols:
    print(col)

{16}
coulmns name :
Ew_a_complete
Estatus_w
Eage
Eage_flag
Eimmcard_age
Eincomplete_w
Eincomplete_a
Edistrictid
Esubcountyid
Eparishid
Evillageid
Emom_marital
Emoved
Emom_married
Esurvey_date
Ebl_father


In [ ]:
columns_to_analyze = [
    'Ew_a_complete',
    'Estatus_w',
    'Eage',
    'Eage_flag',
    'Eimmcard_age',
    'Eincomplete_w',
    'Eincomplete_a',
    'Edistrictid',
    'Esubcountyid',
    'Eparishid',
    'Evillageid',
    'Emom_marital',
    'Emoved',
    'Emom_married',
    'Esurvey_date',
    'Ebl_father'
]

for col in columns_to_analyze:
    analyze_any_column(children_df, col)

    print("-------------------------------------------------")

Column: Ew_a_complete
type :float64
missed values count 1739 from 22308 (7.80%)
 uniqe values count 4
-------------------------------------------------
first 10 rows
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
5    1.0
6    1.0
7    1.0
8    1.0
9    1.0
-------------------------------------------------
dtype = (Categorical / Flag)

 top 5 frequent values
               Count      %
Ew_a_complete              
1.0            20066  97.55
2.0              421   2.05
3.0               64   0.31
0.0               18   0.09
-------------------------------------------------
Column: Estatus_w
type :float64
missed values count 1739 from 22308 (7.80%)
 uniqe values count 2
-------------------------------------------------
first 10 rows
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
5    1.0
6    1.0
7    1.0
8    1.0
9    1.0
-------------------------------------------------
dtype = (Categorical / Flag)

 top 5 frequent values
           Count     %
Estatus_w             
1.0        20487  99.6

In [ ]:
children_df['Esurvey_date'] = pd.to_datetime(children_df['Esurvey_date'], errors='coerce')

min_val = children_df['Esurvey_date'].min()
max_val = children_df['Esurvey_date'].max()

print(f"  (min): {min_val.date()}")
print(f"  (max): {max_val.date()}")

  (min): 2014-03-21
  (max): 2014-09-10


### FOOD__RECALL

In [ ]:
did_take_map = {
  1 :'Yes',
0: 'No',
3 :'Too small to receive',
888: 'I do not know',
999: 'Refuses to answer'
}


In [ ]:
child_food_recall = children_all_sections['FOOD__RECALL']
print(list(child_food_recall.columns))

['Emilk', 'Ewater', 'Etea', 'Ejuice', 'Eotherbev', 'Esolidfood', 'Ematooke', 'Eroots', 'Egrains', 'Ebeans', 'Eorgan', 'Emeat', 'Eegg', 'Efish', 'Egreen', 'Epump', 'Eotherfru', 'Eothdairy', 'Eoil', 'Esweet', 'Eothersolid', 'Enum_meal', 'Edaysmilk', 'Edaysbeans', 'Edaysmeat', 'Ebednet', 'Ediarrhea', 'Ediar_days', 'Ediar_drink', 'Ediar_medpro', 'Emalaria', 'Emal_days', 'Emal_drink', 'Emal_medpro', 'Ecough', 'Ecough_days', 'Ecough_medpro', 'Eworm_drugs', 'Ebcg', 'Epolio0', 'Epolio1', 'Epolio2', 'Epolio3', 'Edpt1', 'Edpt2', 'Edpt3', 'Emeasles', 'Eimmcard_vacc', 'Evita_6wk', 'Evita_6mo', 'Eiron', 'Ebiokid', 'Echild_alive', 'Edeath_date', 'Ein_hh_now', 'Ewhere_living']


In [ ]:
extract_information(child_food_recall,'Emilk')

Emilk    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13433
60.21606598529675
Emilk
0.0    5522
1.0    3353
Name: count, dtype: int64
             Emilk
count  8875.000000
mean      0.377803
std       0.484865
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(child_food_recall,'Ewater')

Ewater    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
13433
60.21606598529675
Ewater
1.0    7478
0.0    1397
Name: count, dtype: int64
            Ewater
count  8875.000000
mean      0.842592
std       0.364206
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(child_food_recall,'Etea')

Etea    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13433
60.21606598529675
Etea
0.0    7666
1.0    1209
Name: count, dtype: int64
              Etea
count  8875.000000
mean      0.136225
std       0.343047
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(child_food_recall,'Ejuice')

Ejuice    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13433
60.21606598529675
Ejuice
0.0    8357
1.0     518
Name: count, dtype: int64
            Ejuice
count  8875.000000
mean      0.058366
std       0.234448
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(child_food_recall,'Eroots')

Eroots    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
13588
60.91088398780706
Eroots
1.0    6672
0.0    2048
Name: count, dtype: int64
            Eroots
count  8720.000000
mean      0.765138
std       0.423937
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(child_food_recall,'Emeat')

Emeat    float64
dtype: object
0    NaN
1    1.0
2    0.0
dtype: float64
13588
60.91088398780706
Emeat
0.0    7868
1.0     852
Name: count, dtype: int64
             Emeat
count  8720.000000
mean      0.097706
std       0.296934
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(child_food_recall,'Eegg')

Eegg    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13588
60.91088398780706
Eegg
0.0    8239
1.0     481
Name: count, dtype: int64
              Eegg
count  8720.000000
mean      0.055161
std       0.228306
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(child_food_recall,'Efish')

Efish    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13588
60.91088398780706
Efish
0.0    8474
1.0     246
Name: count, dtype: int64
             Efish
count  8720.000000
mean      0.028211
std       0.165585
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
extract_information(child_food_recall,'Esweet')

Esweet    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
13588
60.91088398780706
Esweet
0.0    5844
1.0    2876
Name: count, dtype: int64
            Esweet
count  8720.000000
mean      0.329817
std       0.470173
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


In [ ]:
extract_information(child_food_recall,'Enum_meal')

Enum_meal    float64
dtype: object
0      NaN
1      3.0
2      4.0
3      5.0
4      2.0
5      0.0
6      1.0
7      7.0
8      8.0
9     10.0
10     6.0
11     9.0
dtype: float64
12135
54.39752555137171
Enum_meal
3.0     4774
4.0     2369
5.0     1425
2.0     1409
1.0       72
6.0       60
0.0       32
7.0       18
8.0       11
9.0        2
10.0       1
Name: count, dtype: int64
          Enum_meal
count  10173.000000
mean       3.382975
std        0.980839
min        0.000000
25%        3.000000
50%        3.000000
75%        4.000000
max       10.000000


In [ ]:
extract_information(child_food_recall,'Edaysmilk')

Edaysmilk    float64
dtype: object
0    NaN
1    0.0
2    2.0
3    3.0
4    1.0
5    7.0
6    4.0
7    6.0
8    5.0
dtype: float64
6196
27.774789313250853
Edaysmilk
0.0    7625
7.0    3689
2.0    1765
1.0    1452
3.0     969
4.0     379
5.0     133
6.0     100
Name: count, dtype: int64
          Edaysmilk
count  16112.000000
mean       2.264958
std        2.822275
min        0.000000
25%        0.000000
50%        1.000000
75%        4.000000
max        7.000000


In [ ]:
extract_information(child_food_recall,'Edaysbeans')

Edaysbeans    float64
dtype: object
0    NaN
1    3.0
2    7.0
3    5.0
4    2.0
5    6.0
6    4.0
7    0.0
8    1.0
dtype: float64
6197
27.77927201004124
Edaysbeans
7.0    9658
3.0    1493
4.0    1472
5.0    1291
6.0     873
2.0     719
0.0     317
1.0     288
Name: count, dtype: int64
         Edaysbeans
count  16111.000000
mean       5.672646
std        1.908255
min        0.000000
25%        4.000000
50%        7.000000
75%        7.000000
max        7.000000


In [ ]:
extract_information(child_food_recall,'Edaysmeat')

Edaysmeat    float64
dtype: object
0    NaN
1    0.0
2    2.0
3    4.0
4    1.0
5    3.0
6    7.0
7    6.0
8    5.0
dtype: float64
6198
27.78375470683163
Edaysmeat
0.0    8506
1.0    4677
2.0    2140
3.0     605
4.0     115
7.0      33
5.0      22
6.0      12
Name: count, dtype: int64
          Edaysmeat
count  16110.000000
mean       0.722843
std        0.955380
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        7.000000


In [ ]:
child_food_recall['Epolio0'] = child_food_recall['Epolio0'].map(did_take_map)
child_food_recall['Epolio0'] = child_food_recall['Epolio0'].astype('category')

extract_information(child_food_recall,'Epolio0')

Epolio0    category
dtype: object
0    NaN
1    Yes
2     No
dtype: category
Categories (2, object): ['No', 'Yes']
16566
74.2603550295858
Epolio0
Yes    4554
No     1188
Name: count, dtype: int64
       Epolio0
count     5742
unique       2
top        Yes
freq      4554


In [ ]:
child_food_recall['Epolio1'] = child_food_recall['Epolio1'].map(did_take_map)
child_food_recall['Epolio1'] = child_food_recall['Epolio1'].astype('category')

extract_information(child_food_recall,'Epolio1')

Epolio1    category
dtype: object
0                     NaN
1                     Yes
2    Too small to receive
3                      No
dtype: category
Categories (3, object): ['No', 'Too small to receive', 'Yes']
16567
74.26483772637619
Epolio1
Yes                     5390
No                       239
Too small to receive     112
Name: count, dtype: int64
       Epolio1
count     5741
unique       3
top        Yes
freq      5390


In [ ]:
child_food_recall['Epolio2'] = child_food_recall['Epolio2'].map(did_take_map)
child_food_recall['Epolio2'] = child_food_recall['Epolio2'].astype('category')

extract_information(child_food_recall,'Epolio2')

Epolio2    category
dtype: object
0                     NaN
1                     Yes
2                      No
3    Too small to receive
dtype: category
Categories (3, object): ['No', 'Too small to receive', 'Yes']
16567
74.26483772637619
Epolio2
Yes                     5240
No                       296
Too small to receive     205
Name: count, dtype: int64
       Epolio2
count     5741
unique       3
top        Yes
freq      5240


In [ ]:
child_food_recall['Epolio3'] = child_food_recall['Epolio3'].map(did_take_map)
child_food_recall['Epolio3'] = child_food_recall['Epolio3'].astype('category')

extract_information(child_food_recall,'Epolio3')

Epolio3    category
dtype: object
0                     NaN
1                     Yes
2                      No
3    Too small to receive
dtype: category
Categories (3, object): ['No', 'Too small to receive', 'Yes']
16568
74.26932042316658
Epolio3
Yes                     5091
No                       371
Too small to receive     278
Name: count, dtype: int64
       Epolio3
count     5740
unique       3
top        Yes
freq      5091


In [ ]:
child_food_recall['Edpt1'] = child_food_recall['Edpt1'].map(did_take_map)
child_food_recall['Edpt1'] = child_food_recall['Edpt1'].astype('category')

extract_information(child_food_recall,'Edpt1')

Edpt1    category
dtype: object
0                     NaN
1                     Yes
2                      No
3    Too small to receive
dtype: category
Categories (3, object): ['No', 'Too small to receive', 'Yes']
16568
74.26932042316658
Edpt1
Yes                     5365
No                       222
Too small to receive     153
Name: count, dtype: int64
       Edpt1
count   5740
unique     3
top      Yes
freq    5365


In [ ]:
child_food_recall['Edpt2'] = child_food_recall['Edpt2'].map(did_take_map)
child_food_recall['Edpt2'] = child_food_recall['Edpt2'].astype('category')

extract_information(child_food_recall,'Edpt2')

Edpt2    category
dtype: object
0                     NaN
1                     Yes
2                      No
3    Too small to receive
dtype: category
Categories (3, object): ['No', 'Too small to receive', 'Yes']
16567
74.26483772637619
Edpt2
Yes                     5225
No                       292
Too small to receive     224
Name: count, dtype: int64
       Edpt2
count   5741
unique     3
top      Yes
freq    5225


In [ ]:
child_food_recall['Edpt3'] = child_food_recall['Edpt3'].map(did_take_map)
child_food_recall['Edpt3'] = child_food_recall['Edpt3'].astype('category')

extract_information(child_food_recall,'Edpt3')

Edpt3    category
dtype: object
0                     NaN
1                     Yes
2                      No
3    Too small to receive
dtype: category
Categories (3, object): ['No', 'Too small to receive', 'Yes']
16570
74.27828581674736
Edpt3
Yes                     5053
No                       383
Too small to receive     302
Name: count, dtype: int64
       Edpt3
count   5738
unique     3
top      Yes
freq    5053


In [ ]:
child_food_recall['Emeasles'] = child_food_recall['Emeasles'].map(did_take_map)
child_food_recall['Emeasles'] = child_food_recall['Emeasles'].astype('category')

extract_information(child_food_recall,'Emeasles')

Emeasles    category
dtype: object
0                     NaN
1                     Yes
2                      No
3    Too small to receive
dtype: category
Categories (3, object): ['No', 'Too small to receive', 'Yes']
16570
74.27828581674736
Emeasles
Yes                     4456
No                       668
Too small to receive     614
Name: count, dtype: int64
       Emeasles
count      5738
unique        3
top         Yes
freq       4456


In [ ]:
child_food_recall['Evita_6wk'] = child_food_recall['Evita_6wk'].map(did_take_map)
child_food_recall['Evita_6wk'] = child_food_recall['Evita_6wk'].astype('category')

extract_information(child_food_recall,'Evita_6wk')

Evita_6wk    category
dtype: object
0    NaN
1    Yes
2     No
dtype: category
Categories (2, object): ['No', 'Yes']
16594
74.38587053971669
Evita_6wk
Yes    4629
No     1085
Name: count, dtype: int64
       Evita_6wk
count       5714
unique         2
top          Yes
freq        4629


In [ ]:
child_food_recall['Evita_6mo'] = child_food_recall['Evita_6mo'].map(did_take_map)
child_food_recall['Evita_6mo'] = child_food_recall['Evita_6mo'].astype('category')

extract_information(child_food_recall,'Evita_6mo')

Evita_6mo    category
dtype: object
0    NaN
1    Yes
2     No
dtype: category
Categories (2, object): ['No', 'Yes']
16593
74.3813878429263
Evita_6mo
Yes    3867
No     1848
Name: count, dtype: int64
       Evita_6mo
count       5715
unique         2
top          Yes
freq        3867


In [ ]:
child_food_recall['Eiron'] = child_food_recall['Eiron'].map(did_take_map)
child_food_recall['Eiron'] = child_food_recall['Eiron'].astype('category')

extract_information(child_food_recall,'Eiron')

Eiron    category
dtype: object
0    NaN
1    Yes
dtype: category
Categories (1, object): ['Yes']
17372
77.8734086426394
Eiron
Yes    4936
Name: count, dtype: int64
       Eiron
count   4936
unique     1
top      Yes
freq    4936


# 3- converting into correct dtype

### DECISION_MAKING:  

In [ ]:
categorical_women = [
    'w_husb', 'w_daily_who', 'w_daily_prevail', 'w_purch_who', 'w_purch_prevail',
    'w_savespend_who', 'w_savespend_prevail', 'w_wcloth_who', 'w_wcloth_prevail',
    'w_chealth_who', 'w_chealth_prevail', 'w_cfeed_who', 'w_cfeed_prevail',
    'w_ceduc_who', 'w_ceduc_prevail', 'w_cschool_who', 'w_cschool_prevail',
    'w_ccloth_who', 'w_ccloth_prevail', 'w_wmoney_who', 'w_wmoney_prevail',
    'w_rights', 'w_equality', 'w_obey', 'w_boss', 'w_leaders', 'w_powhh',
    'w_shrpow', 'w_opin', 'w_avoid_preg', 'w_final_say', 'w_oblig_sex',
    'w_tol_viol', 'w_deser', 'w_hit_goesout', 'w_hit_argues', 'w_hit_food',
    'w_husb_listens', 'w_husb_angryfreq', 'w_humilfreq', 'w_threatfreq',
    'w_insultfreq', 'w_beatfreq', 'w_pushfreq', 'w_slapfreq', 'w_violentfreq',
    'w_namesfreq', 'w_vio_verb_freq', 'w_vio_phy_freq'
]

binary_women = [
    'w_daily_bi', 'w_daily_diffop', 'w_daily_express', 'w_daily_argue',
    'w_daily_who_wifedec', 'w_daily_who_husbdec', 'w_daily_who_bothdec',
    'w_daily_who_otherdec', 'w_purch_bi', 'w_purch_diffop', 'w_purch_express',
    'w_purch_argue', 'w_purch_who_wifedec', 'w_purch_who_husbdec',
    'w_purch_who_bothdec', 'w_purch_who_otherdec', 'w_savespend_bi',
    'w_savespend_diffop', 'w_savespend_express', 'w_savespend_argue',
    'w_savespend_who_wifedec', 'w_savespend_who_husbdec', 'w_savespend_who_bothdec',
    'w_savespend_who_otherdec', 'w_wcloth_bi', 'w_wcloth_diffop', 'w_wcloth_express',
    'w_wcloth_argue', 'w_wcloth_who_wifedec', 'w_wcloth_who_husbdec',
    'w_wcloth_who_bothdec', 'w_wcloth_who_otherdec', 'w_chealth_bi',
    'w_chealth_diffop', 'w_chealth_express', 'w_chealth_argue', 'w_chealth_who_wifedec',
    'w_chealth_who_husbdec', 'w_chealth_who_bothdec', 'w_chealth_who_otherdec',
    'w_cfeed_bi', 'w_cfeed_diffop', 'w_cfeed_express', 'w_cfeed_argue',
    'w_cfeed_who_wifedec', 'w_cfeed_who_husbdec', 'w_cfeed_who_bothdec',
    'w_cfeed_who_otherdec', 'w_ceduc_bi', 'w_ceduc_diffop', 'w_ceduc_express',
    'w_ceduc_argue', 'w_ceduc_who_wifedec', 'w_ceduc_who_husbdec', 'w_ceduc_who_bothdec',
    'w_ceduc_who_otherdec', 'w_cschool_bi', 'w_cschool_diffop', 'w_cschool_express',
    'w_cschool_argue', 'w_cschool_who_wifedec', 'w_cschool_who_husbdec',
    'w_cschool_who_bothdec', 'w_cschool_who_otherdec', 'w_ccloth_bi',
    'w_ccloth_diffop', 'w_ccloth_express', 'w_ccloth_argue', 'w_ccloth_who_wifedec',
    'w_ccloth_who_husbdec', 'w_ccloth_who_bothdec', 'w_ccloth_who_otherdec',
    'w_wmoney_bi', 'w_wmoney_diffop', 'w_wmoney_express', 'w_wmoney_argue',
    'w_wmoney_who_wifedec', 'w_wmoney_who_husbdec', 'w_wmoney_who_bothdec',
    'w_wmoney_who_otherdec', 'w_rights_sdi', 'w_rights_di', 'w_equality_sdi',
    'w_equality_di', 'w_obey_sdi', 'w_obey_di', 'w_boss_sdi', 'w_boss_di',
    'w_leaders_sdi', 'w_leaders_di', 'w_powhh_sdi', 'w_powhh_di', 'w_shrpow_sag',
    'w_shrpow_ag', 'w_opin_sag', 'w_opin_ag', 'w_avoid_preg_sdi', 'w_avoid_preg_di',
    'w_final_say_sag', 'w_final_say_ag', 'w_oblig_sex_sdi', 'w_oblig_sex_di',
    'w_tol_viol_nsag', 'w_tol_viol_sdi', 'w_tol_viol_di', 'w_deser_sdi', 'w_deser_di',
    'w_hit_goesout_sdi', 'w_hit_goesout_di', 'w_hit_argues_sdi', 'w_hit_argues_di',
    'w_hit_food_sdi', 'w_hit_food_di', 'w_husb_listens_bi', 'w_husb_angry',
    'w_husb_angry_bi', 'w_unfaith', 'w_fem_friends', 'w_fam_contact', 'w_all_times',
    'w_all_times_no', 'w_notrust_mon', 'w_notrust_mon_no', 'w_humil', 'w_threat',
    'w_insult', 'w_beat', 'w_push', 'w_slap', 'w_violent', 'w_names', 'w_vio_verb',
    'w_vio_verb_freq_no', 'w_vio_phy', 'w_vio_phy_freq_no'
]


parent_df = process_categorical_cols(parent_df, categorical_women)
parent_df = process_binary_cols(parent_df, binary_women)



Processed 49 categorical columns.
Processed 135 binary columns.


In [ ]:
# test
for col in categorical_women:
    print(f"Column: '{col}' - Type: {parent_df[col].dtype}")


Column: 'w_husb' - Type: float64
Column: 'w_daily_who' - Type: float64
Column: 'w_daily_prevail' - Type: float64
Column: 'w_purch_who' - Type: float64
Column: 'w_purch_prevail' - Type: float64
Column: 'w_savespend_who' - Type: float64
Column: 'w_savespend_prevail' - Type: float64
Column: 'w_wcloth_who' - Type: float64
Column: 'w_wcloth_prevail' - Type: float64
Column: 'w_chealth_who' - Type: float64
Column: 'w_chealth_prevail' - Type: float64
Column: 'w_cfeed_who' - Type: float64
Column: 'w_cfeed_prevail' - Type: float64
Column: 'w_ceduc_who' - Type: float64
Column: 'w_ceduc_prevail' - Type: float64
Column: 'w_cschool_who' - Type: float64
Column: 'w_cschool_prevail' - Type: float64
Column: 'w_ccloth_who' - Type: float64
Column: 'w_ccloth_prevail' - Type: float64
Column: 'w_wmoney_who' - Type: float64
Column: 'w_wmoney_prevail' - Type: float64
Column: 'w_rights' - Type: float64
Column: 'w_equality' - Type: float64
Column: 'w_obey' - Type: float64
Column: 'w_boss' - Type: float64
Column:

In [ ]:
# test
for col in binary_women:
    print(f"Column: '{col}' - Type: {parent_df[col].dtype}")


Column: 'w_daily_bi' - Type: Int64
Column: 'w_daily_diffop' - Type: Int64
Column: 'w_daily_express' - Type: Int64
Column: 'w_daily_argue' - Type: Int64
Column: 'w_daily_who_wifedec' - Type: Int64
Column: 'w_daily_who_husbdec' - Type: Int64
Column: 'w_daily_who_bothdec' - Type: Int64
Column: 'w_daily_who_otherdec' - Type: Int64
Column: 'w_purch_bi' - Type: Int64
Column: 'w_purch_diffop' - Type: Int64
Column: 'w_purch_express' - Type: Int64
Column: 'w_purch_argue' - Type: Int64
Column: 'w_purch_who_wifedec' - Type: Int64
Column: 'w_purch_who_husbdec' - Type: Int64
Column: 'w_purch_who_bothdec' - Type: Int64
Column: 'w_purch_who_otherdec' - Type: Int64
Column: 'w_savespend_bi' - Type: Int64
Column: 'w_savespend_diffop' - Type: Int64
Column: 'w_savespend_express' - Type: Int64
Column: 'w_savespend_argue' - Type: Int64
Column: 'w_savespend_who_wifedec' - Type: Int64
Column: 'w_savespend_who_husbdec' - Type: Int64
Column: 'w_savespend_who_bothdec' - Type: Int64
Column: 'w_savespend_who_other

### EXPENDITURE

In [ ]:

numeric_expenditure = [
    'w_cattle_own', 'w_goat_own', 'w_chx_own', 'w_stove_own', 'w_wattnk_own',
    'w_bed_own', 'w_lant_own', 'w_tv_own', 'w_radio_own', 'w_gen_own',
    'w_cell_own', 'w_bic_own', 'w_moto_own', 'w_car_own', 'w_sug_own',
    'w_par_own', 'w_soap_own', 'w_tea_own', 'w_milk_own', 'w_mtfs_own',
    'w_salt_own', 'w_transp_own', 'w_air_own', 'w_last6m_rice',
    'w_last6m_ccloth', 'w_last6m_fun', 'w_last6m_health', 'w_last6m_schfee',
    'w_last6m_supp', 'w_last6m_wedd', 'w_last6m_wcloth'
]

binary_expenditure = [
    'w_plan_hhexpend6', 'w_did_womens'
]

categorical_expenditure = [
    'w_incomplete_w'
]


parent_df = process_numeric_cols(parent_df, numeric_expenditure)
parent_df = process_binary_cols(parent_df, binary_expenditure)
parent_df = process_categorical_cols(parent_df, categorical_expenditure)



Processed 31 numeric columns.
Processed 2 binary columns.
Processed 1 categorical columns.


###  ANTHRO_SURVEY

In [ ]:

date_anthro = ['a_date', 'anthro_date', 'survey_date']

numeric_anthro = [
    'w_weight', 'w_height', 'w_hblevel', 'w_bmi', 'w_muac', 'a_numkids'
]

text_anthro = [
    'a_time', 'w_weight_flagwhy', 'w_height_flagwhy',
    'w_hblevel_flagwhy', 'w_muac_flagwhy'
]


categorical_anthro = [
    'a_enum_id', 'survey_status'
]


binary_anthro = [
    'w_weight_flag', 'w_height_flag', 'w_hblevel_flag', 'w_anaemia_12',
    'w_anaemia_11', 'w_anaemia_8', 'w_bmi_18h', 'w_bmi_17', 'w_bmi_16',
    'w_muac_flag', 'w_muac_preg23', 'w_muac_221', 'w_muac_221_flag',
    'w_muac_214', 'dum_a', 'incomplete_a', 'incomplete_survey', 'men_away_hh'
]


parent_df = process_date_cols(parent_df, date_anthro)
parent_df = process_numeric_cols(parent_df, numeric_anthro)
parent_df = process_text_cols(parent_df, text_anthro)
parent_df = process_categorical_cols(parent_df, categorical_anthro)
parent_df = process_binary_cols(parent_df, binary_anthro)

Processed 3 date columns.
Processed 6 numeric columns.
Processed 5 text columns.
Processed 2 categorical columns.
Processed 18 binary columns.


### MENS__SURVEY

In [ ]:
#
date_men = ['m_date']

numeric_men = [
    'm_age', 'm_timekids', 'm_hus_ideal_fert_t', 'm_hus_ideal_fert_b',
    'm_hus_ideal_fert_g', 'm_hus_ideal_fert_e', 'm_nextpreg_when',
    'm_op_birthspace', 'm_soap_day', 'm_ageliquid', 'm_agefood', 'm_bfeed'
]

text_men = [
    'm_village', 'm_enum_notes', 'm_assist_oth', 'm_mal_cause',
    'm_mal_cause_oth', 'm_diar_cause', 'm_kwash_cause', 'm_kwash_treat',
    'm_timedewor_oth'
]


categorical_men = [
    'm_villagepid', 'm_enum_id', 'm_district', 'm_educ', 'm_assist',
    'm_fp_decide', 'm_fp_talk', 'm_nextpreg_units', 'm_op_birthspace_u',
    'm_wash_bath', 'm_soap_toilet', 'm_wash_meal', 'm_soap_meal',
    'm_cstrum_no_why', 'm_bfeed24', 'm_grow_lackfood', 'm_grow_lackdiet',
    'm_grow_witch', 'm_grow_sick', 'm_timedewor', 'm_childtoilet',
    'm_boil_1min', 'm_diarrfood', 'm_weanfat', 'm_play', 'm_porr_thin',
    'm_dietsick', 'm_cowmilk', 'm_menfruit', 'm_menprot', 'm_hygint'
]


binary_men = [
    'm_inc', 'm_milk', 'm_water', 'm_tea', 'm_juice', 'm_otherbev', 'm_matooke',
    'm_cas', 'm_rice', 'm_beans', 'm_organ', 'm_meat', 'm_egg', 'm_fish', 'm_green',
    'm_pump', 'm_otherfru', 'm_oil', 'm_sweet', 'm_othersolid', 'm_kidsunder2',
    'm_preg_now', 'm_preg_now_ac', 'm_preg_past2_ac', 'm_preg_comp', 'm_labr_present',
    'm_soap_toilet_bi', 'm_soap_meal_bi', 'm_mal_cause_bi', 'm_mal_cause_bi2',
    'm_diar_cause_bi', 'm_diar_cause_bi2', 'm_cstrum_feed_bi',
    'm_grow_lackfood_sw', 'm_grow_lackfood_vm', 'm_grow_lackdiet_sw',
    'm_grow_lackdiet_vm', 'm_grow_witch_lt', 'm_grow_witch_vl',
    'm_grow_sick_sw', 'm_grow_sick_vm', 'm_kwash_cause_bi', 'm_kwash_cause_bi2',
    'm_kwash_treat_bi', 'm_kwash_treat_bi2', 'm_ageliquid_6', 'm_agefood_6',
    'm_boil_1min_bi', 'm_diarrfood_bi', 'm_weanfat_bi', 'm_play_bi', 'm_porr_thin_bi',
    'm_dietsick_bi', 'm_cowmilk_bi', 'm_menfruit_bi', 'm_menprot_bi',
    'm_timedewor_1', 'm_timedewor_2', 'm_timedewor_3', 'm_timedewor_4',
    'm_timedeworm_5', 'm_timedewor_6', 'm_timedewor_7', 'm_hygint_bi', 'm_childtoilet_bi'
]

parent_df = process_date_cols(parent_df, date_men)
parent_df = process_numeric_cols(parent_df, numeric_men)
parent_df = process_text_cols(parent_df, text_men)
parent_df = process_categorical_cols(parent_df, categorical_men)
parent_df = process_binary_cols(parent_df, binary_men)

Processed 1 date columns.
Processed 12 numeric columns.
Processed 9 text columns.
Processed 31 categorical columns.
Processed 65 binary columns.


### child_BASELINE_ANTHRO

In [ ]:
date_child = ['anthro_date']

numeric_child = [
    'c_weight', 'wfa', 'wfl', 'c_height', 'lhfa', 'c_muac', 'mfa', 'c_hblevel'
]

text_child = [] # لا يوجد أعمدة نصية صريحة في هذا الجزء من الجدول، ولكننا نترك القائمة فارغة تحسباً

categorical_child = [
    'c_weight_flagwhy', 'weigh_behave', 'weigh_how', 'c_height_flagwhy',
    'height_how', 'c_muac_flagwhy', 'press_skin', 'anthro_status', 'c_hblevel_flagwhy'
]

binary_child = [
    'wfa_neg2', 'wfa_neg1', 'wfl_neg2', 'wfl_neg1', 'c_weight_flag',
    'lhfa_neg2', 'lhfa_neg1', 'c_height_flag', 'mfa_neg3', 'mfa_neg23',
    'mfa_neg2', 'mfa_neg1', 'c_muac_flag', 'hb_11_child', 'hb_10_child',
    'hb_7_child', 'c_hblevel_flag'
]

df_child = process_date_cols(children_df, date_child)
df_child = process_numeric_cols(children_df, numeric_child)
df_child = process_categorical_cols(children_df, categorical_child)
df_child = process_binary_cols(children_df, binary_child)

# test
print(df_child[numeric_child].dtypes.head())
print(df_child[binary_child].dtypes.head())

Processed 1 date columns.
Processed 8 numeric columns.
Processed 9 categorical columns.
Processed 17 binary columns.
c_weight    float64
wfa         float64
wfl         float64
c_height    float64
lhfa        float64
dtype: object
wfa_neg2         Int64
wfa_neg1         Int64
wfl_neg2         Int64
wfl_neg1         Int64
c_weight_flag    Int64
dtype: object


## Masa's part

In [ ]:
def process_categorical_dummies(section_df, df_type):
    global parent_df, children_df
    global columns_to_drop_parent, columns_to_drop_children
    category_cols = section_df.select_dtypes(include=['category']).columns.tolist()
    for col in category_cols:
        dummies = pd.get_dummies(section_df[col], prefix=col, dummy_na=True, dtype=int)
        section_df = pd.concat([section_df, dummies], axis=1)

        if df_type == 'parent':
            parent_df = pd.concat([parent_df, dummies], axis=1)
            if col not in columns_to_drop_parent:
                columns_to_drop_parent.append(col)

        elif df_type == 'children':
            children_df = pd.concat([children_df, dummies], axis=1)
            if col not in columns_to_drop_children:
                columns_to_drop_children.append(col)
        else:
            raise ValueError("df_type must be either 'parent' or 'children'")
    return section_df

In [ ]:
mens_sanitation = process_categorical_dummies(mens_sanitation,'parent')
mens_household = process_categorical_dummies(mens_household,'parent')
mens_expenditure_el = process_categorical_dummies(mens_expenditure_el,'parent')
mens_survey = process_categorical_dummies(mens_survey,'parent')
men_fertility = process_categorical_dummies(men_fertility,'parent')
mens_health_knowledge = process_categorical_dummies(mens_health_knowledge,'parent')
mens_decision_making_el = process_categorical_dummies(mens_decision_making_el,'parent')
women_anthro = process_categorical_dummies(women_anthro,'parent')
expanditure = process_categorical_dummies(expanditure,'parent')
decision_making_el = process_categorical_dummies(decision_making_el,'parent')
mens_sanit_pract = process_categorical_dummies(mens_sanit_pract,'parent')

In [ ]:
attendance = process_categorical_dummies(attendance,'parent')
attendance.head()

,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan
0,1.000000,1.000000,0.0,1.000000,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0
1,0.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
2,0.055556,0.055556,0.0,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0
3,0.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
4,0.666667,0.666667,0.0,0.666667,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0


### child_ENDLINE_SURVEY

In [ ]:

date_e = ['Esurvey_date']


numeric_e = ['Eage']

text_e = ['Esubcountyid', 'Eparishid', 'Evillageid']

categorical_e = [
    'Ew_a_complete', 'Eage_flag', 'Eincomplete_w', 'Eincomplete_a',
    'Edistrictid', 'Emom_marital', 'Ebl_father'
]


binary_e = [
    'Estatus_w', 'Eimmcard_age', 'Emoved', 'Emom_married'
]

df_child = process_date_cols(children_df, date_e)
df_child = process_numeric_cols(children_df, numeric_e)
df_child = process_text_cols(children_df, text_e)
df_child = process_categorical_cols(children_df, categorical_e)
df_child = process_binary_cols(children_df, binary_e)

# test
print(df_child[categorical_e].dtypes.head())
print(df_child[binary_e].dtypes.head())

Processed 1 date columns.
Processed 1 numeric columns.
Processed 3 text columns.
Processed 7 categorical columns.
Processed 4 binary columns.
Ew_a_complete    float64
Eage_flag        float64
Eincomplete_w    float64
Eincomplete_a    float64
Edistrictid       object
dtype: object
Estatus_w       Int64
Eimmcard_age    Int64
Emoved          Int64
Emom_married    Int64
dtype: object


## Dima's part

### parent's columns correction

In [ ]:
binary_parent = [
    'wemp','whn','mhn','control','sample_women','sample_married','sample_couple','sample_pair',
    'sample_anthro','w_eligibility_kids','w_inc','w_milk','w_water','w_tea','w_juice','w_otherbev',
    'w_matooke','w_roots','w_grains','w_beans','w_organ','w_meat','w_egg','w_fish','w_green',
    'w_pump','w_otherfru','w_oil','w_sweet','w_othersolid','w_juiceplus','w_meats','w_otherfv',
    'w_biokids_num','w_pregnow','w_preg_ac','w_hus_go_ac','w_preg_ac_eat','w_preg_ac_comp',
    'w_hus_ac_advice','w_hus_ac_where','w_hus_ac_nutr','w_hus_ac_deliver','w_hus_ac_kids',
    'w_hus_ac_whenkid','w_preg_mal','w_preg_comp','w_hus_aware_comp','w_hus_assist_comp','w_preg_foods',
    'w_pregpast2','w_preg_ac2','w_hus_go_ac2','w_preg_ac_eat2','w_preg_ac_comp2','w_hus_ac_advice2',
    'w_hus_ac_where2','w_hus_ac_nutr2','w_hus_ac_deliver2','w_hus_ac_kids2','w_hus_ac_whenkid2',
    'w_preg_mal2','w_preg_comp2','w_birth_comp2','w_hus_aware_comp2','w_hus_assist_comp2',
    'w_hus_aware_bcomp2','w_hus_assist_bcomp2','w_preg_foods2','w_hus_assist_birth','w_lastpreg_ac',
    'w_lastpreg_foods','w_bfeed_atemore','w_breastfed_weeks_bi','w_hus_disc_care','w_birth_weighed',
    'w_weight_immcard','w_check2mo','w_first_check_weeks_bi','w_weight_1mo_card','w_breastfeed',
    'w_colostrum','w_othliq_1week','w_othliq_3mo','w_food_3mo','iron','w_vita_6wk','vita_6mo',
    'w_hus_ideal_fert','w_treat_water','w_soap_toilet_bi','w_soap_meal_bi','w_refuse_dispose_bi',
    'w_toilet_sweep_bi','w_birth_bi','w_mal_cause_bi','w_mal_cause_bi2','w_diar_cause_bi','w_diar_cause_bi2',
    'w_cstrum_feed_bi','w_grow_lackfood_sw','w_grow_lackfood_vm','w_grow_lackdiet_sw','w_grow_lackdiet_vm',
    'w_grow_witch_lt','w_grow_witch_vl','w_grow_sick_sw','w_grow_sick_vm','w_kwash_cause_bi','w_kwash_cause_bi2',
    'w_kwash_treat_bi','w_kwash_treat_bi2','w_ugx_matooke_bi','w_ugx_maize_bi','w_ugx_potato_bi','w_ugx_rice_bi',
    'w_ugx_fish_bi','w_ugx_gnuts_bi','w_ugx_veg_bi','w_ugx_fruit_bi','w_ugx_meat_bi','w_ugx_bean_bi','w_ugx_egg_bi',
    'w_ugx_milk_bi','w_ugx_sweet_bi','w_ugx_other_bi','w_ageliquid_6','w_agefood_6','w_bfeed24','w_starchmilk_bi',
    'w_starchmillet_bi','w_starchoil_bi','w_starchgnuts_bi','w_starchrice_bi','w_starchpineapple_bi',
    'w_starchmatooke_bi','w_proteinchicken_bi','w_proteinmatooke_bi','w_proteinfruit_bi','w_proteinbeans_bi',
    'w_proteinposho_bi','w_proteinmeat_bi','w_beanmeat_bi','w_reheat_leftov_bi','w_wateropen_bi','w_boil_1min_bi',
    'w_diarrfood_bi','w_oral_salts_bi','w_porr_bi','w_weanfat_bi','w_child_protein_bi','w_play_bi',
    'w_porr_thin_bi','w_dietsick_bi','w_cowmilk_bi','w_menfruit_bi','w_menprot_bi','w_dispose_bi','w_timedeworm_1',
    'w_timedeworm_2','w_timedeworm_3','w_timedeworm_4','w_timedeworm_5','w_timedeworm_6','w_timedeworm_7',
    'w_hygint_bi','w_childtoilet_bi',
]

parent_df = process_binary_cols(parent_df, binary_parent)


Processed 167 binary columns.


In [ ]:
categorical_parent = [
    'treat','stratum','district_name','districtid','subcountyid','w_marital','w_preg_want',
    'w_preg_comp_list','w_preg_comp_list2','w_where_birth','w_birth_size','w_mom_health2mo',
    'w_mom_healthnow','w_first_check_unit','w_time_bfeed_unit','w_time_milk_units','w_othliq_what',
    'w_bfeed_eat','w_hus_ac','w_hus_feed','w_hus_preg','w_hus_hyg','w_fp_decide','w_fp_talk',
    'w_smallkids','w_earlykids','w_pregcomp','w_weight_3mo_card','w_water_source','w_treat_water_how',
    'w_wash_bath','w_soap_toilet','w_wash_meal','w_soap_meal','w_refuse_dispose','w_toilet_type',
    'w_birth','w_mal_cause','w_diar_cause','w_cstrum_feed','w_cstrum_no_why','w_grow_lackfood',
    'w_grow_lackdiet','w_grow_witch','w_grow_sick','w_kwash_cause','w_kwash_treat','w_ugx_spend',
    'w_starchmilk','w_starchmillet','w_starchoil','w_starchgnuts','w_starchrice','w_starchpineapple',
    'w_starchmatooke','w_proteinchicken','w_proteinmatooke','w_proteinfruit','w_proteinbeans',
    'w_proteinposho','w_proteinmeat','w_beanmeat','w_reheat_leftov','w_wateropen','w_boil_1min',
    'w_diahfood','w_oral_salts','w_porr','w_weanfat','w_child_protein','w_play','w_porr_thin',
    'w_dietsick','w_cowmilk','w_menfruit','w_menprot','w_dispose','w_timedeworm','w_hygint','w_childtoilet',
]

parent_df = process_categorical_cols(parent_df, categorical_parent)


Processed 80 categorical columns.


In [ ]:
numeric_parent = [
    'id','villageid','w_date_code','w_enum_id','w_age','w_educ','w_num_adults','w_numkids18',
    'w_numkids18_bio','w_numkids18_non','w_biokids_out','w_othkids12','w_numkids0to5','w_biokids_hh',
    'w_biokids_out2','w_biokids_num_alive','w_numkids_husb','w_preg_mo','w_preg_learn','w_preg_ac_num',
    'w_preg_ac_mo','w_preg_mal_num','w_preg_ac_num2','w_preg_ac_mo2','w_preg_mal_num2','w_time_breastfeed_weeks',
    'w_birth_weight','w_first_check','w_first_check_weeks','w_weight_1mo','w_weight_3mo','w_time_breastfeed',
    'w_time_milk','total_vacc','w_ideal_fert_t','w_ideal_fert_b','w_ideal_fert_g','w_ideal_fert_e','w_nextpreg_when',
    'w_hus_ideal_fert_t','w_hus_ideal_fert_b','w_hus_ideal_fert_g','w_hus_ideal_fert_e','w_soap_day',
    'w_toilet_sweep','w_ugx_matooke','w_ugx_maize','w_ugx_potato','w_ugx_rice','w_ugx_fish','w_ugx_gnuts',
    'w_ugx_veg','w_ugx_fruit','w_ugx_meat','w_ugx_bean','w_ugx_egg','w_ugx_milk','w_ugx_sweet',
    'w_ugx_other','w_ageliquid','w_agefood','w_bfeed',
]

parent_df = process_numeric_cols(parent_df, numeric_parent)


Processed 62 numeric columns.


In [ ]:
text_parent = [
    'w_preg_whynoac','w_preg_whynoac_oth','w_hus_assist_how','w_hus_assist_how_oth','w_preg_foods_list',
    'w_preg_whynoac2','w_preg_whynoac_oth2','w_birth_comp_list2','w_hus_assist_how2','w_hus_assist_how_oth2',
    'w_hus_assist_bhow2','w_hus_assist_bhow_oth2','w_preg_foods_list2','w_hus_assist_birth_how',
    'w_hus_assist_birth_oth','w_othliq_other','w_water_source_oth','w_treat_water_oth','w_toilet_oth',
    'w_birth_oth','w_mal_cause_oth','w_ugx_oth','w_timedeworm_oth',
]

parent_df = process_text_cols(parent_df, text_parent)


Processed 23 text columns.


In [ ]:
date_parent = [
    'w_date','w_time'
]

parent_df = process_date_cols(parent_df, date_parent)


Processed 2 date columns.


### child's columns correction

In [ ]:
binary_child = [
    'wemp','whn','mhn','control','sample','sample_married','sample_moved','sample_alive','sample_2yr',
    'sample_1yr','gender','milk','water','tea','juice','otherbev','solidfood','matooke','roots',
    'grains','beans','organ','meat','egg','fish','green','pump','otherfru','oil','sweet','othersolid',
    'immcard_w','biokid','diarrhea','diar_medpro','malaria','mal_medpro','cough','cough_medpro',
    'worm_drugs','worm_medpro','semi_solidfood','bcg','polio0','polio1','polio2','polio3',
    'dpt1','dpt2','dpt3','measles','vita_6wk','vita_6mo','iron',
]

children_df = process_binary_cols(children_df, binary_child)


Processed 54 binary columns.


In [ ]:
categorical_child = [
    'stratum','districtid','district_name','treat','onlyin',
    'survey_status','breastfed','bednet','diar_drink','mal_drink',
]

children_df = process_categorical_cols(children_df, categorical_child)


Processed 10 categorical columns.


In [ ]:
numeric_child = [
    'id','villageid','age','kidsunder18','under18bio','under18nonbio','biooutside',
    'otherkids','time_bf','diar_days','mal_days','num_meal','daysprotein','elapsed_time_mo',
]

children_df = process_numeric_cols(children_df, numeric_child)


Processed 14 numeric columns.


# 4 + 5 + 6 + 7 + 8 Preprocessing section

## Parent preprocessing

### women anthro

In [ ]:
extract_information(women_anthro,'Ew_bmi_18h')

Ew_bmi_18h    float64
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64
359
6.513062409288825
Ew_bmi_18h
0.0    4885
1.0     268
Name: count, dtype: int64
        Ew_bmi_18h
count  5153.000000
mean      0.052009
std       0.222066
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [ ]:
women_anthro[(women_anthro['Ew_bmi']<=20 ) & (women_anthro['Ew_bmi_18h']==1 )].shape

(268, 22)

In [ ]:
extract_information(women_anthro,'Ew_muac_preg23')

Ew_muac_preg23    float64
dtype: object
0    NaN
1    0.0
2    1.0
dtype: float64
4746
86.10304789550072
Ew_muac_preg23
0.0    692
1.0     74
Name: count, dtype: int64
       Ew_muac_preg23
count      766.000000
mean         0.096606
std          0.295613
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


In [ ]:
women_anthro[(women_anthro['Ew_muac']<=23 ) & (women_anthro['Ew_muac_preg23']==1 )].shape

(74, 22)

In [ ]:
cols = ['Ew_anaemia_12','Ew_anaemia_11','Ew_anaemia_8','Ew_bmi_18h','Ew_bmi_17','Ew_bmi_16','Ew_muac_preg23','Ew_muac_221','Ew_muac_214']
parent_df.drop(columns=cols,axis=1,inplace=True)
women_anthro.drop(columns=cols,axis=1,inplace=True)

### attendance

In [ ]:
program_attendance_group = parent_df[['wemp','mhn','whn']].copy()
program_attendance_group_df = pd.concat([program_attendance_group, attendance],axis=1)
program_attendance_group_df.head()

,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan
0,1,0,0,1.000000,1.000000,0.0,1.000000,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0
1,0,0,0,0.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
2,1,0,0,0.055556,0.055556,0.0,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0
3,0,0,0,0.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
4,1,0,0,0.666667,0.666667,0.0,0.666667,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0


In [ ]:
program_attendance_group_1_df = program_attendance_group_df[program_attendance_group_df['wemp']==1 ]
print(program_attendance_group_1_df.shape)
program_attendance_group_1_df.head()

(1330, 116)


,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan
0,1,0,0,1.000000,1.000000,0.0,1.000000,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0
2,1,0,0,0.055556,0.055556,0.0,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0
4,1,0,0,0.666667,0.666667,0.0,0.666667,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0
8,1,0,0,0.555556,0.555556,0.0,0.555556,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0
22,1,0,0,1.000000,1.000000,0.0,1.000000,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0


In [ ]:
program_attendance_group_1_nonull_df = program_attendance_group_1_df[~program_attendance_group_1_df['Eattend_wemp'].isnull()]
print(program_attendance_group_1_nonull_df.shape)
program_attendance_group_1_nonull_df.head()

(1330, 116)


,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan
0,1,0,0,1.000000,1.000000,0.0,1.000000,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0
2,1,0,0,0.055556,0.055556,0.0,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0
4,1,0,0,0.666667,0.666667,0.0,0.666667,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0
8,1,0,0,0.555556,0.555556,0.0,0.555556,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0
22,1,0,0,1.000000,1.000000,0.0,1.000000,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0


Note: all wemp = 1 -> Eattend_wemp has a number (no null)

In [ ]:
temp_df = program_attendance_group_1_nonull_df[program_attendance_group_1_nonull_df['Eattend_wemp']==0]
print(temp_df.shape)
temp_df.head()

(39, 116)


,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan
116,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0
157,1,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,NaN,NaN,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0
218,1,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,NaN,NaN,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0
222,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0
380,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0


In [ ]:
Eattend_zero_df = program_attendance_group_df[(program_attendance_group_df['Eattend']==0) & (program_attendance_group_df['Eattend_women']==0) & (program_attendance_group_df['Eattend_men']==0) & (program_attendance_group_df['Eattend_wemp']==0) & (program_attendance_group_df['Eattend_whn']==0)]
print(Eattend_zero_df.shape)
Eattend_zero_df.head()

(1594, 116)


,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan
1,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
3,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
7,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
11,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
14,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0


In [ ]:
w_zero = Eattend_zero_df[(Eattend_zero_df['wemp']== 0) & (Eattend_zero_df['whn']== 0) & (Eattend_zero_df['mhn']== 0)].copy()
print(w_zero.shape)
w_zero.head()

(1379, 116)


,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan
1,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
3,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
7,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
11,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
14,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0


In [ ]:
session_cols = [
    'sessn_att1', 'sessn_att2', 'sessn_att3', 'sessn_att4', 'sessn_att5',
    'sessn_att6', 'sessn_att7', 'sessn_att8', 'sessn_att9', 'sessn_att10',
    'sessn_att11', 'sessn_att12', 'sessn_att13', 'sessn_att14', 'sessn_att15',
    'sessn_att16', 'sessn_att18', 'sessn_att19'
]

all_null_rows = w_zero[w_zero[session_cols].isnull().all(axis=1)].copy()
print(all_null_rows.shape)
all_null_rows.head()


(1379, 116)


,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan
1,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
3,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
7,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
11,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
14,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0


There is 1379 family who is not in wemp,mhn,whn and all of the session columns are Nan TODO

In [ ]:
# general_parent_df = parent_all_sections['GENERAL']
# condition = (
#     (general_parent_df['wemp'] == 0) & (general_parent_df['whn'] == 0) & (general_parent_df['mhn'] == 0) &
#     (attendance['Eattend'] == 0) & (attendance['Eattend_women'] == 0) &
#     (attendance['Eattend_men'] == 0) & (attendance['Eattend_wemp'] == 0) &
#     (attendance['Eattend_whn'] == 0)
# )

# for col in session_cols:
#     attendance.loc[condition, col] = attendance.loc[condition, col].fillna(-1)

In [ ]:
program_group = parent_df[['wemp','mhn','whn']].copy()
program_attendance_group_df = pd.concat([program_group, attendance],axis=1)
program_attendance_group_df.head()

,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan
0,1,0,0,1.000000,1.000000,0.0,1.000000,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0
1,0,0,0,0.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
2,1,0,0,0.055556,0.055556,0.0,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0
3,0,0,0,0.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0
4,1,0,0,0.666667,0.666667,0.0,0.666667,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0


In [ ]:
# attendance['sessions_avg'] = attendance[session_cols].mask(attendance[session_cols] == -1).mean(axis=1)
# attendance.head()


In [ ]:
info_df = pd.DataFrame()
info_df['missing_val'] = attendance.isnull().sum()
info_df['missing_val_ratio'] = (info_df['missing_val'] / attendance.shape[0] * 100).round().astype(int)
info_df

,missing_val,missing_val_ratio
Eattend,0,0
Eattend_women,0,0
Eattend_men,0,0
Eattend_wemp,0,0
Eattend_whn,0,0
...,...,...
sessn_att18_cat_nan,0,0
sessn_att19_cat_-1.0,0,0
sessn_att19_cat_0.0,0,0
sessn_att19_cat_1.0,0,0


In [ ]:
# program_group = parent_df[['wemp','mhn','whn']].copy()
# program_attendance_group_df = pd.concat([program_group, attendance],axis=1)
# sessions_null_df = program_attendance_group_df[program_attendance_group_df['sessions_avg'].isnull()]
# sessions_null_df.head()

null processing: even though mhn = 1 or whn = 1 but all the avg are 0s which means we can set the sessions to zero

In [ ]:
# condition = (program_attendance_group_df['sessions_avg'].isnull())
# for col in session_cols:
#     attendance.loc[condition, col] = attendance.loc[condition, col].fillna(0)

In [ ]:
attendance['sessions_avg'] = attendance[session_cols].mean(axis=1)
parent_df['sessions_avg'] = parent_df[session_cols].mean(axis=1)
attendance.head()

/tmp/ipykernel_1035/353212447.py:2: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan,sessions_avg
0,1.000000,1.000000,0.0,1.000000,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1.000000
1,0.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,NaN
2,0.055556,0.055556,0.0,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0.055556
3,0.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,NaN
4,0.666667,0.666667,0.0,0.666667,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0.666667


In [ ]:
print(attendance[(parent_df['whn'] == 1) & (attendance['sessions_avg'].round(2) != attendance['Eattend_whn'].round(2))].shape)
print(attendance[(parent_df['mhn'] == 1) & (attendance['sessions_avg'].round(2) != attendance['Eattend_men'].round(2))].shape)
print(attendance[(parent_df['wemp'] == 1) & (attendance['sessions_avg'].round(2) != attendance['Eattend_wemp'].round(2))].shape)

(12, 114)
(42, 114)
(6, 114)


In [ ]:
print(attendance[(parent_df['wemp'] == 1) & (attendance['Eattend_women'].round(2) != attendance['Eattend'].round(2))].shape)

(0, 114)


In [ ]:
print(attendance[(parent_df['whn'] == 1) & (attendance['Eattend_women'].round(2) != attendance['Eattend_whn'].round(2))].shape)

(0, 114)


In [ ]:
program_group = parent_df[['wemp','mhn','whn']].copy()
program_attendance_group_df = pd.concat([program_group, attendance],axis=1)
df_rounded = program_attendance_group_df.round(5)
condition = (
    (df_rounded['sessions_avg'] != df_rounded['Eattend']) &
    (df_rounded['sessions_avg'] != df_rounded['Eattend_women']) &
    (df_rounded['sessions_avg'] != df_rounded['Eattend_men']) &
    (df_rounded['sessions_avg'] != df_rounded['Eattend_wemp']) &
    (df_rounded['sessions_avg'] != df_rounded['Eattend_whn']))
mismatched_df = program_attendance_group_df[condition]
print(mismatched_df.shape)
mismatched_df.head()

(1439, 117)


,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan,sessions_avg
1,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,NaN
3,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,NaN
7,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,NaN
11,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,NaN
14,0,0,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,NaN


In [ ]:
null_attendance_df = program_attendance_group_df[program_attendance_group_df[session_cols].isnull().any(axis=1)]
null_attendance_df.shape

(1449, 117)

In [ ]:
anomalies_df = program_attendance_group_df[(program_attendance_group_df['Eattend_men'] == program_attendance_group_df['sessions_avg'])& (program_attendance_group_df['mhn']==1)]
print(anomalies_df.shape)
print(anomalies_df['Eattend_wemp'].value_counts())
print(anomalies_df['Eattend_women'].value_counts())
display(anomalies_df.head())

(1233, 117)
Eattend_wemp
0.0    1233
Name: count, dtype: int64
Eattend_women
0.0    1233
Name: count, dtype: int64


,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan,sessions_avg
5,0,1,0,0.833333,0.0,0.833333,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0.833333
10,0,1,0,0.222222,0.0,0.222222,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0.222222
12,0,1,0,1.000000,0.0,1.000000,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1.000000
26,0,1,0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0.000000
30,0,1,0,0.722222,0.0,0.722222,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0.722222


In [ ]:
import numpy as np

empty_session_cols = [
    col for col in session_cols
    if program_attendance_group_df[col].isin([0, np.nan]).all()
]

print("الجلسات التي لا تحتوي على أي حضور (كلها 0 أو NaN):")
print(empty_session_cols)

الجلسات التي لا تحتوي على أي حضور (كلها 0 أو NaN):
[]


الجلسات ليست مختصة بمجال معين (اي مثلا لا يوجد الجلسة 7 مختصة للرجال)<br>
بالتالي لا يمكن الاستفادة من هذه المعلومة لتعويض ال null

In [ ]:
print(f"wemp {len(program_attendance_group_df[program_attendance_group_df['wemp']==1])}")
print(f"mhn {len(program_attendance_group_df[program_attendance_group_df['mhn']==1])}")
print(f"whn {len(program_attendance_group_df[program_attendance_group_df['whn']==1])}")
print(f"none {len(program_attendance_group_df[(program_attendance_group_df['whn']==0) & (program_attendance_group_df['mhn']==0) & (program_attendance_group_df['wemp']==0)])}")

wemp 1330
mhn 1368
whn 1435
none 1379


In [ ]:
men_lines = program_attendance_group_df[program_attendance_group_df['mhn']==1]
print(men_lines.shape)
for col in session_cols:
    if men_lines[col].isnull().sum() > 0:
        print(f"col {col} has{men_lines[col].isnull().sum()}")
print(f"eattend != men : {len(men_lines[(men_lines['Eattend']!=men_lines['Eattend_men'])])}")
print(f"eattend != avg : {len(men_lines[(men_lines['Eattend']!=men_lines['sessions_avg'])])}")

(1368, 117)
col sessn_att1 has34
col sessn_att2 has34
col sessn_att3 has11
col sessn_att4 has12
col sessn_att5 has2
col sessn_att6 has1
col sessn_att7 has1
col sessn_att8 has1
col sessn_att9 has1
col sessn_att10 has1
col sessn_att11 has1
col sessn_att12 has1
col sessn_att13 has1
col sessn_att14 has1
col sessn_att15 has1
col sessn_att16 has2
col sessn_att18 has2
col sessn_att19 has1
eattend != men : 0
eattend != avg : 135


In [ ]:
null_attendance_men_df = men_lines[(men_lines[session_cols].isnull().any(axis=1) & (men_lines['Eattend_men']!=men_lines['sessions_avg']))]
print(null_attendance_men_df.shape)
null_attendance_men_df.head()

(42, 117)


,wemp,mhn,whn,Eattend,Eattend_women,Eattend_men,Eattend_wemp,Eattend_whn,sessn_att1,sessn_att2,sessn_att3,sessn_att4,sessn_att5,sessn_att6,sessn_att7,sessn_att8,sessn_att9,sessn_att10,sessn_att11,sessn_att12,sessn_att13,sessn_att14,sessn_att15,sessn_att16,sessn_att18,sessn_att19,sessn_att1_cat,sessn_att2_cat,sessn_att3_cat,sessn_att4_cat,sessn_att5_cat,sessn_att6_cat,sessn_att7_cat,sessn_att8_cat,sessn_att9_cat,sessn_att10_cat,sessn_att11_cat,sessn_att12_cat,sessn_att13_cat,sessn_att14_cat,sessn_att15_cat,sessn_att16_cat,sessn_att18_cat,sessn_att19_cat,sessn_att1_cat_-1.0,sessn_att1_cat_0.0,sessn_att1_cat_1.0,sessn_att1_cat_nan,sessn_att2_cat_-1.0,sessn_att2_cat_0.0,sessn_att2_cat_1.0,sessn_att2_cat_nan,sessn_att3_cat_-1.0,sessn_att3_cat_0.0,sessn_att3_cat_1.0,sessn_att3_cat_nan,sessn_att4_cat_-1.0,sessn_att4_cat_0.0,sessn_att4_cat_1.0,sessn_att4_cat_nan,sessn_att5_cat_-1.0,sessn_att5_cat_0.0,sessn_att5_cat_1.0,sessn_att5_cat_nan,sessn_att6_cat_-1.0,sessn_att6_cat_0.0,sessn_att6_cat_1.0,sessn_att6_cat_nan,sessn_att7_cat_-1.0,sessn_att7_cat_0.0,sessn_att7_cat_1.0,sessn_att7_cat_nan,sessn_att8_cat_-1.0,sessn_att8_cat_0.0,sessn_att8_cat_1.0,sessn_att8_cat_nan,sessn_att9_cat_-1.0,sessn_att9_cat_0.0,sessn_att9_cat_1.0,sessn_att9_cat_nan,sessn_att10_cat_-1.0,sessn_att10_cat_0.0,sessn_att10_cat_1.0,sessn_att10_cat_nan,sessn_att11_cat_-1.0,sessn_att11_cat_0.0,sessn_att11_cat_1.0,sessn_att11_cat_nan,sessn_att12_cat_-1.0,sessn_att12_cat_0.0,sessn_att12_cat_1.0,sessn_att12_cat_nan,sessn_att13_cat_-1.0,sessn_att13_cat_0.0,sessn_att13_cat_1.0,sessn_att13_cat_nan,sessn_att14_cat_-1.0,sessn_att14_cat_0.0,sessn_att14_cat_1.0,sessn_att14_cat_nan,sessn_att15_cat_-1.0,sessn_att15_cat_0.0,sessn_att15_cat_1.0,sessn_att15_cat_nan,sessn_att16_cat_-1.0,sessn_att16_cat_0.0,sessn_att16_cat_1.0,sessn_att16_cat_nan,sessn_att18_cat_-1.0,sessn_att18_cat_0.0,sessn_att18_cat_1.0,sessn_att18_cat_nan,sessn_att19_cat_-1.0,sessn_att19_cat_0.0,sessn_att19_cat_1.0,sessn_att19_cat_nan,sessions_avg
156,0,1,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,-1.0,-1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0.5000
276,0,1,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1.0000
284,0,1,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0.0625
443,0,1,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,NaN
484,0,1,0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,-1.0,-1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0.6875


In [ ]:
info_df = pd.DataFrame()
info_df['missing_val'] = attendance.isnull().sum()
info_df['missing_val_ratio'] = (info_df['missing_val'] / attendance.shape[0] * 100).round().astype(int)
info_df

,missing_val,missing_val_ratio
Eattend,0,0
Eattend_women,0,0
Eattend_men,0,0
Eattend_wemp,0,0
Eattend_whn,0,0
...,...,...
sessn_att19_cat_-1.0,0,0
sessn_att19_cat_0.0,0,0
sessn_att19_cat_1.0,0,0
sessn_att19_cat_nan,0,0


In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=attendance['Eattend_women'].values,
                           name="women"))
fig.add_trace(go.Histogram(x=attendance['Eattend_men'].values,
                           name="men"))
fig.update_layout(**layout_options)
fig.show()

نسبة النساء التي التزمت ب 50 بالئة فأكثر اكثر من الرجال

In [ ]:
session_means = program_attendance_group_df[session_cols].mean().reset_index()
session_means.columns = ['Session', 'Average_attendance']

fig2 = px.line(session_means, x='Session', y='Average_attendance', markers=True,
               title="تطور الحضور عبر الجلسات (1-19)")
fig2.update_layout(**layout_options)
fig2.show()

### mens household

In [ ]:
mens_household[(mens_household['Em_hhkids_num']==0 ) & (mens_household['Em_hours_kids'].isnull())]

,Em_numkids,Em_hhkids_num,Em_hours_kids
97,1.0,0.0,NaN
161,2.0,0.0,NaN
197,1.0,0.0,NaN
329,2.0,0.0,NaN
403,2.0,0.0,NaN
...,...,...,...
5240,1.0,0.0,NaN
5257,3.0,0.0,NaN
5340,2.0,0.0,NaN
5348,1.0,0.0,NaN


In [ ]:
mens_household.loc[(mens_household['Em_hhkids_num'] == 0) & (mens_household['Em_hours_kids'].isnull()), 'Em_hours_kids'] = 0
print(mens_household['Em_hours_kids'].isnull().sum())

2777


In [ ]:
fig_box = make_subplots(rows=1, cols=3, subplot_titles=("Total Kids (Em_numkids)", "Kids in HH (Em_hhkids_num)", "Hours with Kids (Em_hours_kids)"))

fig_box.add_trace(go.Box(y=mens_household['Em_numkids'], name="Total Kids", boxpoints='outliers'), row=1, col=1)
fig_box.add_trace(go.Box(y=mens_household['Em_hhkids_num'], name="Kids in HH", boxpoints='outliers'), row=1, col=2)
fig_box.add_trace(go.Box(y=mens_household['Em_hours_kids'], name="Hours with Kids", boxpoints='outliers'), row=1, col=3)

fig_box.update_layout(title_text="توزيع البيانات والقيم الشاذة للأعمدة الثلاثة", **layout_options)
fig_box.show()

24 and 20 for hours with kids are obeviously noise, we will change them with the median of the of the parents that has the same amount of children

In [ ]:
mens_household[(mens_household['Em_hours_kids']==24)|(mens_household['Em_hours_kids']==20)]

,Em_numkids,Em_hhkids_num,Em_hours_kids
257,2.0,1.0,20.0
1172,4.0,3.0,24.0


In [ ]:
print(mens_household[(mens_household['Em_hhkids_num']==2)]['Em_hours_kids'].median())
print(mens_household[(mens_household['Em_hhkids_num']==2)]['Em_hours_kids'].mean())

2.0
2.8794642857142856


In [ ]:
print(mens_household[(mens_household['Em_hhkids_num']==4)]['Em_hours_kids'].median())
print(mens_household[(mens_household['Em_hhkids_num']==4)]['Em_hours_kids'].mean())

2.0
2.7865429234338746


In [ ]:
fig_hist = px.histogram(mens_household, x="Em_hours_kids", nbins=25,
                        title="توزيع عدد الساعات التي يقضيها الآباء مع أطفالهم")
fig_hist.update_layout(**layout_options)
fig_hist.show()

In [ ]:
mens_household.loc[mens_household['Em_hours_kids'].isin([20, 24]), 'Em_hours_kids'] = np.nan
mens_household['Em_hours_kids'] = mens_household['Em_hours_kids'].fillna(
    mens_household.groupby('Em_hhkids_num')['Em_hours_kids'].transform('median'))

In [ ]:
fig_hist = px.box(mens_household, y="Em_hours_kids",
                        title="توزيع عدد الساعات التي يقضيها الآباء مع أطفالهم",
                        labels={'Em_hours_kids': 'عدد الساعات', 'count': 'العدد (التكرار)'},)
fig_hist.update_layout(**layout_options)
fig_hist.show()

In [ ]:
fig_hist = px.histogram(mens_household, x="Em_hours_kids", nbins=14,
                        title="توزيع عدد الساعات التي يقضيها الآباء مع أطفالهم")
fig_hist.update_layout(**layout_options)
fig_hist.show()

In [ ]:
import numpy as np
import plotly.express as px
df = parent_df.copy()
df['Group'] = np.select(
    [df['wemp'] == 1, df['mhn'] == 1, df['whn'] == 1],
    ['WEMP', 'MHN', 'WHN'],
    default='Control'
)


df_grouped = df.groupby(['Em_hhkids_num', 'Group'], as_index=False)['Em_hours_kids'].median()

fig = px.bar(
    df_grouped,
    x='Em_hhkids_num',
    y='Em_hours_kids',
    color='Group',
    barmode='group'
)
fig.update_xaxes(tickvals=list(range(1, 14)))
fig.show()

In [ ]:
parent_df[parent_df['Em_hhkids_num']>=10][['Em_hours_kids','Em_hhkids_num']]

,Em_hours_kids,Em_hhkids_num
455,NaN,10.0
2525,NaN,12.0



### mens expanditure el

In [ ]:
extract_information(mens_expenditure_el,'Em_plan_hhexpend6')

Em_plan_hhexpend6    float64
dtype: object
0    1.0
1    0.0
2    NaN
dtype: float64
341
6.186502177068215
Em_plan_hhexpend6
1.0    3885
0.0    1286
Name: count, dtype: int64
       Em_plan_hhexpend6
count        5171.000000
mean            0.751305
std             0.432298
min             0.000000
25%             1.000000
50%             1.000000
75%             1.000000
max             1.000000


In [ ]:
mens_expenditure_el['Em_monthly_tobacco'] = mens_expenditure_el['Em_last7d_tobacco'] * 4
mens_expenditure_el['Em_monthly_alcohol'] = mens_expenditure_el['Em_last7d_alcohol'] * 4
parent_df['Em_monthly_tobacco'] = parent_df['Em_last7d_tobacco'] * 4
parent_df['Em_monthly_alcohol'] = parent_df['Em_last7d_alcohol'] * 4
mens_expenditure_el.describe()

/tmp/ipykernel_1035/1878335917.py:3: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/tmp/ipykernel_1035/1878335917.py:4: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



,Em_last7d_tobacco,Em_last7d_alcohol,Em_last1m_snacks,Em_last1m_elec,Em_last1m_cloth,Em_plan_hhexpend6,Em_monthly_tobacco,Em_monthly_alcohol
count,5176.000000,5176.000000,5176.000000,5.176000e+03,5176.000000,5171.000000,5176.000000,5.176000e+03
mean,480.954598,3448.863794,5878.005216,1.350185e+04,10252.609737,0.751305,1923.818393,1.379546e+04
std,2195.721451,13851.098440,16023.389785,4.535761e+04,23487.282861,0.432298,8782.885805,5.540439e+04
min,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00
25%,0.000000,0.000000,0.000000,2.600000e+03,0.000000,1.000000,0.000000,0.000000e+00
50%,0.000000,0.000000,0.000000,5.400000e+03,0.000000,1.000000,0.000000,0.000000e+00
75%,0.000000,4000.000000,6000.000000,1.300000e+04,15000.000000,1.000000,0.000000,1.600000e+04
max,100000.000000,800003.000000,730000.000000,1.900000e+06,700000.000000,1.000000,400000.000000,3.200012e+06



note : the median is very different than mean

In [ ]:
for col in mens_expenditure_el.columns:
    tobacco_zeros = parent_df.groupby('mhn')[col].apply(lambda x: (x == 0).sum())
    tobacco_total = parent_df.groupby('mhn')[col].count()

    print(f"{col} has :")
    print(tobacco_zeros / tobacco_total)

Em_last7d_tobacco has :
mhn
0    0.810486
1    0.802528
Name: Em_last7d_tobacco, dtype: float64
Em_last7d_alcohol has :
mhn
0    0.554731
1    0.561611
Name: Em_last7d_alcohol, dtype: float64
Em_last1m_snacks has :
mhn
0    0.497442
1    0.518167
Name: Em_last1m_snacks, dtype: float64
Em_last1m_elec has :
mhn
0    0.086701
1    0.094787
Name: Em_last1m_elec, dtype: float64
Em_last1m_cloth has :
mhn
0    0.671100
1    0.676935
Name: Em_last1m_cloth, dtype: float64
Em_plan_hhexpend6 has :
mhn
0    0.252112
1    0.238133
Name: Em_plan_hhexpend6, dtype: float64
Em_monthly_tobacco has :
mhn
0    0.810486
1    0.802528
Name: Em_monthly_tobacco, dtype: float64
Em_monthly_alcohol has :
mhn
0    0.554731
1    0.561611
Name: Em_monthly_alcohol, dtype: float64


* zeros are real, if the information were missing then the 0's ratio would be the same

In [ ]:
import plotly.graph_objects as go

parent_df['Em_monthly_tobacco'] = mens_expenditure_el['Em_monthly_tobacco']
parent_df['Em_monthly_alcohol'] = mens_expenditure_el['Em_monthly_alcohol']

monthly_expenses = [
    'Em_monthly_tobacco', 'Em_monthly_alcohol',
    'Em_last1m_snacks', 'Em_last1m_elec', 'Em_last1m_cloth'
]

mens_expenditure_el_melted_exp = mens_expenditure_el.melt(id_vars=['Em_plan_hhexpend6'], value_vars=monthly_expenses,
                        var_name='Expense_Type', value_name='Amount')
fig_exp_box = px.box(mens_expenditure_el_melted_exp, x='Expense_Type', y='Amount',
                     color='Expense_Type',
                     title="توزيع النفقات الشهرية للأسرة (بالمقياس اللوغاريتمي)",
                     labels={'Amount': 'المبلغ (لوغاريتم)', 'Expense_Type': 'نوع النفقة'})
fig_exp_box.update_yaxes(type="log")
fig_exp_box.update_layout(template="plotly_white", showlegend=False)
fig_exp_box.show()

In [ ]:
fig = px.box(mens_expenditure_el, y="Em_last7d_tobacco")
fig.update_layout(**layout_options)
fig.show()

تختلف النفقات بشكل كبير جدا  بين : Em_last7d_tobacco,Em_last7d_alcohol,Em_last1m_snacks

In [ ]:
parent_df[(parent_df['mhn']==1)]['Em_plan_hhexpend6'].value_counts()

,count
Em_plan_hhexpend6,
1.0,963
0.0,301


In [ ]:
expense_cols = ['Em_monthly_tobacco', 'Em_monthly_alcohol', 'Em_last1m_snacks', 'Em_last1m_elec', 'Em_last1m_cloth']
grouped_expenses = mens_expenditure_el.groupby('Em_plan_hhexpend6')[expense_cols].median()

plan_labels = {
    0: 'بدون تخطيط (0)',
    1: 'بتخطيط مشترك (1)'
}

fig = go.Figure()

for plan_status, row_values in grouped_expenses.iterrows():
    label = plan_labels.get(plan_status, str(plan_status))

    fig.add_trace(go.Bar(
        name=label,
        x=expense_cols,
        y=row_values.values
    ))
fig.update_layout(
    barmode='group',
    xaxis_title='نوع المصروف',
    yaxis_title='متوسط المبلغ المصروف',
    legend_title='حالة التخطيط المالي'
)

fig.show()

In [ ]:
expense_cols = ['Em_monthly_tobacco','Em_monthly_alcohol',  'Em_last1m_snacks', 'Em_last1m_elec', 'Em_last1m_cloth']
multi_grouped = parent_df.groupby(['mhn', 'Em_plan_hhexpend6'])[expense_cols].mean()
multi_grouped


Em_monthly_tobacco  Em_monthly_alcohol  \
mhn Em_plan_hhexpend6                                           
0   0.0                       2591.065990        16704.194924   
    1.0                       1619.993155        13452.433949   
1   0.0                       3628.571429        12950.272425   
    1.0                       1581.790239        12155.347871   

                       Em_last1m_snacks  Em_last1m_elec  Em_last1m_cloth  
mhn Em_plan_hhexpend6                                                     
0   0.0                     5576.142132    12117.944162      8956.446701  
    1.0                     6145.396988    14223.855236     11002.121834  
1   0.0                     6535.548173    13740.697674      8732.558140  
    1.0                     5190.036345    12626.322949      9832.510903

In [ ]:

fig = go.Figure()

if (0, 0) in multi_grouped.index:
    fig.add_trace(go.Bar(
        x=expense_cols,                    # أسماء النفقات تظهر على المحور X
        y=multi_grouped.loc[(0, 0)],       # جلب سطر القيم المقابل للفئة (0, 0)
        name='مجموعة التحكم (0) - بدون تخطيط (0)',
    ))
if (0, 1) in multi_grouped.index:
    fig.add_trace(go.Bar(
        x=expense_cols,
        y=multi_grouped.loc[(0, 1)],       # جلب سطر القيم المقابل للفئة (0, 1)
        name='مجموعة التحكم (0) - تخطيط مشترك (1)',
    ))
if (1, 0) in multi_grouped.index:
    fig.add_trace(go.Bar(
        x=expense_cols,
        y=multi_grouped.loc[(1, 0)],       # جلب سطر القيم المقابل للفئة (1, 0)
        name='رجال بالتدريب (1) - بدون تخطيط (0)',
    ))

if (1, 1) in multi_grouped.index:
    fig.add_trace(go.Bar(
        x=expense_cols,
        y=multi_grouped.loc[(1, 1)],       # جلب سطر القيم المقابل للفئة (1, 1)
        name='رجال بالتدريب (1) - تخطيط مشترك (1)',
    ))

fig.update_layout(
    barmode='group',
    title=" متوسط نفقات الرجال بحسب حالة المشاركة بالتدريب والتخطيط المالي للأسرة",
    xaxis_title="نوع المصروف المالي",
    yaxis_title="متوسط المبلغ المصروف",
    legend_title_text="حالة التدريب والتخطيط للأسرة",
)

fig.show()

### mans saint pract

In [ ]:
mens_sanit_pract.describe()

,Em_soap_toilet_bi,Em_soap_meal_bi,Em_toilet_improv,Em_soap_toilet_Every time,Em_soap_toilet_Most of the time,Em_soap_toilet_Never,Em_soap_toilet_Rarely,Em_soap_toilet_Sometimes,Em_soap_toilet_nan,Em_soap_meal_Every time,Em_soap_meal_Most of the time,Em_soap_meal_Never,Em_soap_meal_Rarely,Em_soap_meal_Sometimes,Em_soap_meal_nan,Em_toilet_improv_what_Added cover,Em_toilet_improv_what_Added handwashing facilities,Em_toilet_improv_what_Build new latrine,"Em_toilet_improv_what_Improved structural materials (better wood, etc.)",Em_toilet_improv_what_Previously used shared latrine and built own latrine,Em_toilet_improv_what_nan
count,5176.000000,5176.000000,5176.000000,5512.00000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.00000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000
mean,0.357226,0.605100,0.505796,0.33545,0.193940,0.035015,0.195573,0.179064,0.060958,0.230588,0.146408,0.09688,0.273948,0.191219,0.060958,0.027395,0.105044,0.160559,0.178701,0.003266,0.525036
std,0.479228,0.488876,0.500015,0.47219,0.395419,0.183833,0.396677,0.383441,0.239275,0.421247,0.353547,0.29582,0.446023,0.393297,0.239275,0.163246,0.306637,0.367157,0.383137,0.057057,0.499418
min,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,1.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,1.000000,1.000000,1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
mens_sanit_pract.describe(include='category')

,Em_soap_toilet,Em_soap_meal,Em_toilet_improv_what
count,5176,5176,2618
unique,5,5,5
top,Every time,Rarely,"Improved structural materials (better wood, etc.)"
freq,1849,1510,985


In [ ]:
mens_sanit_pract['Em_toilet_improv_what'].isnull().sum()

np.int64(2894)

In [ ]:
col = ['Em_soap_toilet_bi','Em_soap_meal_bi']
mens_sanit_pract.drop(columns=col,axis=1,inplace=True)
parent_df.drop(columns=col,axis=1,inplace=True)

In [ ]:
parent_df['m_soap_toilet'] = parent_df['m_soap_toilet'].map(grad_map)
parent_df['m_soap_toilet'] = parent_df['m_soap_toilet'].astype('category')
parent_df['m_soap_toilet'].describe()

,m_soap_toilet
count,5179
unique,3
top,Every time
freq,3006


In [ ]:
parent_df['m_soap_meal'] = parent_df['m_soap_meal'].map(grad_map)
parent_df['m_soap_meal'] = parent_df['m_soap_meal'].astype('category')
parent_df['m_soap_meal'].describe()

,m_soap_meal
count,5353
unique,3
top,Every time
freq,1888


In [ ]:
base = pd.DataFrame({'m_soap_meal': parent_df['m_soap_meal'], 'Period': 'Baseline'})
end  = pd.DataFrame({'m_soap_meal': mens_sanit_pract['Em_soap_meal'], 'Period': 'Endline'})
soap_df = pd.concat([base, end])

fig_meal = px.histogram(soap_df,
                        x='m_soap_meal',
                        title = 'soap before meal using',
                        color='Period',
                        barmode='group',
                        category_orders={"m_soap_meal": ["Every time","Most of the time","Sometimes","Rarely", "Never"]})

fig_meal.show()

In [ ]:
base = pd.DataFrame({'m_soap_toilet': parent_df['m_soap_toilet'], 'Period': 'Baseline'})
end  = pd.DataFrame({'m_soap_toilet': mens_sanit_pract['Em_soap_toilet'], 'Period': 'Endline'})
soap_df = pd.concat([base, end])

fig_meal = px.histogram(soap_df,
                        x='m_soap_toilet',
                        title = 'soap after toilet using',
                        color='Period',
                        barmode='group',
                        category_orders={"m_soap_meal": ["Every time","Most of the time","Sometimes","Rarely", "Never"]})

fig_meal.show()

In [ ]:
improved_df = parent_df[parent_df['Em_toilet_improv'] == 1]

base_soap = pd.DataFrame({
    'Soap Usage': improved_df['m_soap_toilet'],
    'Period': 'Before (Baseline)'
})

end_soap = pd.DataFrame({
    'Soap Usage': improved_df['Em_soap_toilet'],
    'Period': 'After (Endline)'
})

fig = px.histogram(pd.concat([base_soap, end_soap]),
                   x='Soap Usage',
                   color='Period',
                   barmode='group',
                   title='Change in Handwashing (After Toilet) for Households that Improved Their Toilets',
                   category_orders={"Soap Usage": ["Every time", "Most of the time", "Sometimes", "Rarely", "Never"]},
                   color_discrete_sequence=['#EF553B', '#00CC96']) # ألوان واضحة أحمر وأخضر

fig.show()

Fact: men tend to become worst after the training

In [ ]:
improved_df = parent_df[parent_df['mhn'] == 1]

base_soap = pd.DataFrame({
    'Soap Usage': improved_df['m_soap_toilet'],
    'Period': 'Before (Baseline)'
})

end_soap = pd.DataFrame({
    'Soap Usage': improved_df['Em_soap_toilet'],
    'Period': 'After (Endline)'
})

fig = px.histogram(pd.concat([base_soap, end_soap]),
                   x='Soap Usage',
                   color='Period',
                   barmode='group',
                   title='Change in Handwashing (After Toilet) for Households that Improved Their Toilets',
                   category_orders={"Soap Usage": ["Every time", "Most of the time", "Sometimes", "Rarely", "Never"]},
                   color_discrete_sequence=['#EF553B', '#00CC96']) # ألوان واضحة أحمر وأخضر

fig.show()

### mens sanitation

In [ ]:
plot_data_men = parent_df.groupby(['Em_timedeworm'], as_index=False).count()[['Em_timedeworm', 'id']].rename(columns={'id': 'count'})

fig = px.bar(
    plot_data_men,
    x='Em_timedeworm',
    y='count',
    title="توزيع وتكرار فئات دواء الديدان (Em_timedeworm)",
)

# 4. تحديث التنسيق وعرض الرسمة
fig.update_layout(**layout_options,xaxis_tickangle=-15)
fig.show()

/tmp/ipykernel_1035/3466338182.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_1035/3466338182.py:1: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



### mans food recall

In [ ]:
mens_food_recall.describe()

,Em_milk,Em_water,Em_tea,Em_juice,Em_otherbev,Em_matooke,Em_roots,Em_grains,Em_beans,Em_organ,Em_meat,Em_egg,Em_fish,Em_green,Em_pump,Em_otherfru,Em_othdairy,Em_oil,Em_sweet,Em_othersolid
count,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000,5176.000000
mean,0.401082,0.892388,0.170209,0.088292,0.802357,0.837906,0.667117,0.685278,0.940688,0.080178,0.274730,0.048686,0.034583,0.641229,0.268934,0.752898,0.098338,0.268547,0.130410,0.183346
std,0.490165,0.309920,0.375852,0.283747,0.398260,0.368573,0.471290,0.464450,0.236231,0.271594,0.446421,0.215232,0.182738,0.479686,0.443448,0.431368,0.297801,0.443247,0.336786,0.386987
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
mens_food_recall['Em_milk'].isnull().sum()

np.int64(336)

In [ ]:
print(mens_food_recall[(parent_df['Em_milk'].isnull()) & (parent_df['mhn']!=1)].shape)
mens_food_recall[(parent_df['Em_milk'].isnull()) & (parent_df['mhn']!=1)].head()

(234, 20)


,Em_milk,Em_water,Em_tea,Em_juice,Em_otherbev,Em_matooke,Em_roots,Em_grains,Em_beans,Em_organ,Em_meat,Em_egg,Em_fish,Em_green,Em_pump,Em_otherfru,Em_othdairy,Em_oil,Em_sweet,Em_othersolid
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
131,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(mens_food_recall[(parent_df['Em_milk'].notnull()) & (parent_df['mhn']!=1)].shape)
mens_food_recall[(parent_df['Em_milk'].notnull()) & (parent_df['mhn']!=1)].head()

(3910, 20)


,Em_milk,Em_water,Em_tea,Em_juice,Em_otherbev,Em_matooke,Em_roots,Em_grains,Em_beans,Em_organ,Em_meat,Em_egg,Em_fish,Em_green,Em_pump,Em_otherfru,Em_othdairy,Em_oil,Em_sweet,Em_othersolid
0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0


In [ ]:
info_df = pd.DataFrame()
info_df['missing_val'] = mens_food_recall.isnull().sum()
info_df['missing_val_ratio'] = (info_df['missing_val'] / mens_food_recall.shape[0] * 100).round().astype(int)
info_df

,missing_val,missing_val_ratio
Em_milk,336,6
Em_water,336,6
Em_tea,336,6
Em_juice,336,6
Em_otherbev,336,6
Em_matooke,336,6
Em_roots,336,6
Em_grains,336,6
Em_beans,336,6
Em_organ,336,6


تغير اكل الرجال بحالة تدريب الرجال

In [ ]:
import plotly.graph_objects as go

df_mhn = parent_df[parent_df['mhn'] == 1]
food_items = [
    'milk', 'water', 'tea', 'juice', 'otherbev', 'matooke',
    'roots', 'grains', 'beans', 'organ', 'meat', 'egg',
    'fish', 'green', 'pump', 'otherfru', 'othdairy',
    'oil', 'sweet', 'othersolid'
]

before_percentages = []
after_percentages = []

for item in food_items:
    before_col = f'm_{item}'
    after_col = f'Em_{item}'

    if before_col in df_mhn.columns:
        before_pct = df_mhn[before_col].mean() * 100
    else:
        before_pct = None
    if after_col in df_mhn.columns:
        after_pct = df_mhn[after_col].mean() * 100
    else:
        after_pct = None

    before_percentages.append(before_pct)
    after_percentages.append(after_pct)

fig = go.Figure(data=[
    go.Bar(name='قبل التدريب (Baseline)', x=food_items, y=before_percentages),
    go.Bar(name='بعد التدريب (Endline)', x=food_items, y=after_percentages)
])

fig.update_layout(
    title='مقارنة استهلاك أنواع الطعام للآباء (قبل وبعد تدريب الصحة والتغذية mhn=1)',
    xaxis_title='أنواع الطعام والمشروبات',
    yaxis_title='نسبة الاستهلاك (%)',
    barmode='group',
    xaxis_tickangle=-45,
)
fig.show()

تغير اكل الرجال بحالة تدريب النساء

In [ ]:
import plotly.graph_objects as go

df_mhn = parent_df[parent_df['whn'] == 1]
food_items = [
    'milk', 'water', 'tea', 'juice', 'otherbev', 'matooke',
    'roots', 'grains', 'beans', 'organ', 'meat', 'egg',
    'fish', 'green', 'pump', 'otherfru', 'othdairy',
    'oil', 'sweet', 'othersolid'
]

before_percentages = []
after_percentages = []

for item in food_items:
    before_col = f'm_{item}'
    after_col = f'Em_{item}'

    if before_col in df_mhn.columns:
        before_pct = df_mhn[before_col].mean() * 100
    else:
        before_pct = None
    if after_col in df_mhn.columns:
        after_pct = df_mhn[after_col].mean() * 100
    else:
        after_pct = None
    before_percentages.append(before_pct)
    after_percentages.append(after_pct)

fig = go.Figure(data=[
    go.Bar(name='قبل التدريب (Baseline)', x=food_items, y=before_percentages),
    go.Bar(name='بعد التدريب (Endline)', x=food_items, y=after_percentages)
])

fig.update_layout(
    title='مقارنة استهلاك أنواع الطعام للآباء (قبل وبعد تدريب الصحة والتغذية whn=1)',
    xaxis_title='أنواع الطعام والمشروبات',
    yaxis_title='نسبة الاستهلاك (%)',
    barmode='group',
    xaxis_tickangle=-45,
)
fig.show()

تغير اكل الرجال بحالة تدريب  النساء وتمكينهم

In [ ]:
df_mhn = parent_df[parent_df['wemp'] == 1]
food_items = [
    'milk', 'water', 'tea', 'juice', 'otherbev', 'matooke',
    'roots', 'grains', 'beans', 'organ', 'meat', 'egg',
    'fish', 'green', 'pump', 'otherfru', 'othdairy',
    'oil', 'sweet', 'othersolid'
]

before_percentages = []
after_percentages = []

for item in food_items:
    before_col = f'm_{item}'
    after_col = f'Em_{item}'

    if before_col in df_mhn.columns:
        before_pct = df_mhn[before_col].mean() * 100
    else:
        before_pct = None
    if after_col in df_mhn.columns:
        after_pct = df_mhn[after_col].mean() * 100
    else:
        after_pct = None

    before_percentages.append(before_pct)
    after_percentages.append(after_pct)

fig = go.Figure(data=[
    go.Bar(name='قبل التدريب (Baseline)', x=food_items, y=before_percentages),
    go.Bar(name='بعد التدريب (Endline)', x=food_items, y=after_percentages)
])

fig.update_layout(
    title='مقارنة استهلاك أنواع الطعام للآباء (قبل وبعد تدريب الصحة والتغذية wemp=1)',
    xaxis_title='أنواع الطعام والمشروبات',
    yaxis_title='نسبة الاستهلاك (%)',
    barmode='group',
    xaxis_tickangle=-45,
)
fig.show()

بدون تدريب

In [ ]:
import plotly.graph_objects as go

df_mhn = parent_df[(parent_df['mhn'] == 0) & (parent_df['whn'] == 0) & (parent_df['wemp'] == 0)]
food_items = [
    'milk', 'water', 'tea', 'juice', 'otherbev', 'matooke',
    'roots', 'grains', 'beans', 'organ', 'meat', 'egg',
    'fish', 'green', 'pump', 'otherfru', 'othdairy',
    'oil', 'sweet', 'othersolid'
]

before_percentages = []
after_percentages = []

for item in food_items:
    before_col = f'm_{item}'
    after_col = f'Em_{item}'

    if before_col in df_mhn.columns:
        before_pct = df_mhn[before_col].mean() * 100
    else:
        before_pct = None
    if after_col in df_mhn.columns:
        after_pct = df_mhn[after_col].mean() * 100
    else:
        after_pct = None
    before_percentages.append(before_pct)
    after_percentages.append(after_pct)

fig = go.Figure(data=[
    go.Bar(name='قبل التدريب (Baseline)', x=food_items, y=before_percentages),
    go.Bar(name='بعد التدريب (Endline)', x=food_items, y=after_percentages)
])

fig.update_layout(
    title='مقارنة استهلاك أنواع الطعام للآباء بدون تدريب',
    xaxis_title='أنواع الطعام والمشروبات',
    yaxis_title='نسبة الاستهلاك (%)',
    barmode='group',
    xaxis_tickangle=-45,
)
fig.show()

In [ ]:
group_conditions = {
    'تدريب الرجال (Men HN)': children_df['mhn'] == 1,
    'تدريب النساء (Women HN)': children_df['whn'] == 1,
    'تمكين النساء (Women Empowerment)': children_df['wemp'] == 1,
    'مجموعة التحكم (Control - لا تدريب)': (children_df['mhn'] != 1) & (children_df['whn'] != 1) & (children_df['wemp'] != 1)
}
food_items = [
    'milk', 'water', 'tea', 'juice', 'otherbev', 'matooke',
    'beans', 'organ', 'meat', 'egg',
    'fish', 'green', 'pump', 'otherfru',
    'oil', 'sweet', 'othersolid'
]
group_differences = {group: [] for group in group_conditions.keys()}
for item in food_items:
    before_col = f'{item}'
    after_col = f'E{item}'

    for group_name, condition_mask in group_conditions.items():
        df_sub = children_df[condition_mask]
        if before_col in df_sub.columns and after_col in df_sub.columns:
            diff = (df_sub[after_col].mean() - df_sub[before_col].mean()) * 100
        else:
            diff = None
        if diff != None :
          group_differences[group_name].append(diff)
fig = go.Figure()
for (group_name, diff_values) in group_differences.items():
    fig.add_trace(go.Bar(
        name=group_name,
        x=food_items,
        y=diff_values,
    ))
fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=2)
fig.update_layout(
    title='قياس أثر التدريب: مقدار التغير في نسبة استهلاك الأطعمة (بعد التدريب % - قبل التدريب %)',
    xaxis_title='أنواع الطعام والمشروبات',
    yaxis_title='مقدار التغير في النسبة المئوية (%)',
    barmode='group',
    xaxis_tickangle=-45,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

النساء غالبا ذات تاثير اعلى على اطعام الطفل بعد التدريب

بأغلب الاطعمة، الرجال كان

### Mens health knowledge

In [ ]:
col=['Em_cstrum_no_why_oth','Em_grow_witch_lt','Em_grow_witch_vl','Em_kwash_cause_oth','Em_kwash_treat_oth','Em_boil_1min','Em_weanfat','Em_porr_thin',
     'Em_cowmilk','Em_menfruit','Em_menprot','Em_hiv_bfeed','Em_safe_primhc','Em_boys_eatmore','Em_fteeth_cause_oth','Em_child_fteeth_oth',
     'Em_child_vomit_oth'

]
mens_health_knowledge.drop(columns = col,axis=1,inplace=True)
parent_df.drop(columns = col,axis=1,inplace=True)

### mens survey

In [ ]:
mens_survey.describe()

,Em_moved,Em_survey_date,Em_still_married,Estatus_m_-1,Estatus_m_0,Estatus_m_1,Estatus_m_nan,Em_district_Mitooma,Em_district_Ntungamo,Em_district_Rukungiri,Em_district_Sheema,Em_district_nan,Em_subcounty_Bitereko,Em_subcounty_Bugangari,Em_subcounty_Bugongi,Em_subcounty_Buhunga,Em_subcounty_Buyanja,Em_subcounty_Bwambara,Em_subcounty_Bwongyera,Em_subcounty_Ihunga,Em_subcounty_Itojo,Em_subcounty_Kabira,Em_subcounty_Kanyabwanga,Em_subcounty_Kasaana,Em_subcounty_Kashenshero,Em_subcounty_Katenga,Em_subcounty_Kayonza,Em_subcounty_Kebisoni,Em_subcounty_Kibatsi,Em_subcounty_Kigarama,Em_subcounty_Kitagata,Em_subcounty_Kiyanga,Em_subcounty_Kyangyenyi,Em_subcounty_Masheruka,Em_subcounty_Mayanga,Em_subcounty_Mitooma,Em_subcounty_Mutara,Em_subcounty_Ndija,Em_subcounty_Ngoma,Em_subcounty_Ntungamo,Em_subcounty_Nyabihoko,Em_subcounty_Nyakagyeme,Em_subcounty_Nyakishenyi,Em_subcounty_Nyakyera,Em_subcounty_Nyarushanje,Em_subcounty_Rubaare,Em_subcounty_Rugarama,Em_subcounty_Ruhaama,Em_subcounty_Ruhinda,Em_subcounty_Rukoni East,Em_subcounty_Rukoni West,Em_subcounty_Rurehe,Em_subcounty_Rwampara,Em_subcounty_Rweikiniro,Em_subcounty_Shuuku,Em_subcounty_nan
count,5512.000000,5192,5171.000000,5512.000000,5512.000000,5512.000000,5512.0,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.00000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000
mean,0.001996,2014-05-29 22:54:16.086286592,0.973893,0.005806,0.055152,0.939042,0.0,0.195210,0.411466,0.246553,0.145682,0.001089,0.030116,0.029935,0.019956,0.018687,0.028483,0.024311,0.032656,0.023041,0.018505,0.008345,0.026306,0.023766,0.016509,0.015058,0.030298,0.023403,0.013425,0.022859,0.018505,0.025762,0.026125,0.015965,0.013607,0.018505,0.031386,0.000181,0.03447,0.032293,0.028302,0.027032,0.037192,0.035015,0.035740,0.030660,0.030842,0.037373,0.021771,0.026125,0.011430,0.009615,0.000181,0.026669,0.018505,0.001089
min,0.000000,2014-03-21 00:00:00,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,2014-04-15 00:00:00,1.000000,0.000000,0.000000,1.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,2014-06-14 00:00:00,1.000000,0.000000,0.000000,1.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,2014-07-07 00:00:00,1.000000,0.000000,0.000000,1.000000,0.0,0.000000,1.000000,0.000000,0.000000,0.000000,0.00

In [ ]:
mens_survey.describe(include = 'category')

,Estatus_m,Em_district,Em_subcounty
count,5512,5506,5506
unique,3,4,43
top,1,Ntungamo,Ruhaama
freq,5176,2268,206


دراسة العلاقة بين المقاطعة ومتوسط الاموال المدفوعة

In [ ]:
expense_columns = [
    'Em_last7d_tobacco',
    'Em_last7d_alcohol',
    'Em_last1m_snacks',
    'Em_last1m_elec',
    'Em_last1m_cloth',
]
district_expenses = parent_df.groupby('Em_district')[expense_columns].mean().reset_index()
district_expenses

,Em_district,Em_last7d_tobacco,Em_last7d_alcohol,Em_last1m_snacks,Em_last1m_elec,Em_last1m_cloth
0,Mitooma,511.472081,3542.842640,5755.126904,13900.355330,9802.538071
1,Ntungamo,528.724409,3423.858731,6479.529875,12964.000000,10594.719778
2,Rukungiri,418.160561,3322.453624,5147.622759,13873.774747,8429.078722
3,Sheema,411.236631,3618.048128,5561.564171,13902.941176,12999.331551


In [ ]:
subcounrty_expenses = parent_df.groupby(['Em_district', 'Em_subcounty'])[expense_columns].mean()
subcounrty_expenses

Em_last7d_tobacco  Em_last7d_alcohol  \
Em_district Em_subcounty                                         
Mitooma     Bitereko             532.278481        4043.670886   
            Kabira               416.666667        2666.666667   
            Kanyabwanga          202.173913        4244.927536   
            Kashenshero          334.246575        4684.931507   
            Katenga              447.368421        1401.315789   
            Kiyanga              243.609023        2439.849624   
            Mayanga             1100.000000        5284.722222   
            Mitooma              290.425532        2978.723404   
            Mutara               969.798658        3469.798658   
            Rurehe               650.000000        4050.000000   
Ntungamo    Bwongyera            616.071429        1628.571429   
            Ihunga               412.500000        3580.833333   
            Itojo                483.168317        3212.871287   
            Kayonza              569.670886        4651.898734   
            Kibatsi              989.062500        3859.375000   
            Ndija                  0.000000        7000.000000   
            Ngoma                441.304348        3438.043478   
            Ntungamo             917.105882        3067.058824   
            Nyabihoko            695.918367        2418.367347   
            Nyakyera             348.097826        3419.021739   
            Rubaare              557.763975        3519.925466   
            Rugarama             499.386503        3072.392638   
            Ruhaama              428.535354        6517.186869   
            Rukoni East          459.124088        2763.503650   
            Rukoni West          249.180328        2803.278689   
            Rwampara               0.000000           0.000000   
            Rweikiniro           346.099291        2334.751773   
Rukungiri   Bugangari            451.265823        2796.835443   
            Buhunga              987.755102        4105.102041   
            Buyanja              547.619048        4246.312925   
            Bwambara             198.437500        1575.000000   
            Kebisoni             533.884298        4332.231405   
            Nyakagyeme           316.312057        3237.588652   
            Nyakishenyi          360.937500        3727.604167   
            Nyarushanje          229.842932        3406.282723   
            Ruhinda              374.766355        2296.261682   
Sheema      Bugongi              428.571429        2933.333333   
            Kasaana              269.870690        3014.655172   
            Kigarama             582.051282        6329.059829   
            Kitagata             360.638298        2158.510638   
            Kyangyenyi           327.857143        4575.714286   
            Masheruka            564.102564        2764.102564   
            Shuuku               402.040816        2540.816327   

                          Em_last1m_snacks  Em_last1m_elec  Em_last1m_cloth  
Em_district Em_subcounty                                                     
Mitooma     Bitereko           4692.405063     9920.886076      7670.886076  
            Kabira             6142.857143    11942.857143     11547.619048  
            Kanyabwanga        4710.869565     9728.260870     11793.478261  
            Kashenshero        8641.095890    15186.301370      7698.630137  
            Katenga            3592.105263     9884.210526      6539.473684  
            Kiyanga            5672.932331    19374.812030     11804.511278  
            Mayanga            6951.388889    11784.722222      9666.666667  
            Mitooma            7755.319149    23453.191489     12287.234043  
            Mutara             5973.154362    12432.885906     10006.711409  
            Rurehe             4830.000000    18760.000000      7200.000000  
Ntungamo    Bwongyera          7356.547619    11566.375000      8080.357143  
            Ihunga             7475.00000

### mens fertility

In [ ]:
men_fertility.describe()

,Em_preg_now,Em_preg_now_ac,Em_preg_past2,Em_preg_past2_ac,Em_labr_present,Em_hus_ideal_fert_t,Em_hus_ideal_fert_b,Em_hus_ideal_fert_g,Em_hus_ideal_fert_e,Em_nextpreg_when,Em_nextpreg_units,Em_op_birthspace,Em_op_birthspace_u,Em_wait_nextpreg,Em_wait_nextpreg_u,Convulsions,High Blood Pressure,Night Blindness,None,Other,Positioning of Baby,Swelling,Trouble with vision during day,Unknown/Missing,Vaginal Bleeding,Em_fp_decide_Joint decision,Em_fp_decide_Mainly my decision,Em_fp_decide_Mainly my wife’s/partner’s decision,Em_fp_decide_nan,Em_fp_talk_Joint decision,Em_fp_talk_Mainly my decision,Em_fp_talk_Mainly my wife’s/partner’s decision,Em_fp_talk_nan
count,4978.000000,761.000000,5036.000000,5036.000000,5036.000000,5034.000000,5034.000000,5034.00000,5034.000000,3365.000000,3469.000000,5153.000000,5154.000000,4440.000000,4442.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000
mean,0.152873,0.651774,0.602462,0.981930,0.743646,4.216925,1.967620,1.66468,0.584625,3.398217,1.394926,2.793130,0.664144,3.813964,1.983341,0.029390,0.012155,0.006350,0.699565,0.007620,0.054971,0.038462,0.033563,0.056060,0.044086,0.812591,0.048440,0.052612,0.086357,0.641509,0.069666,0.202467,0.086357
std,0.359901,0.476721,0.489437,0.133217,0.436663,1.821845,1.307312,1.14288,1.633669,3.781188,5.072398,1.401909,3.020135,3.470934,0.128005,0.168914,0.109589,0.079439,0.458489,0.086966,0.227944,0.192325,0.180118,0.230057,0.205304,0.390275,0.214713,0.223279,0.280916,0.479601,0.254607,0.401875,0.280916
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,1.000000,0.000000,3.000000,2.000000,1.00000,0.000000,2.000000,0.000000,2.000000,0.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,1.000000,1.000000,1.000000,4.000000,2.000000,2.00000,0.000000,3.000000,0.000000,3.000000,0.000000,3.000000,2.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
75%,0.000000,1.000000,1.000000,1.000000,1.000000,5.000000,2.000000,2.00000,0.000000,5.000000,0.000000,3.000000,0.000000,4.000000,2.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,30.000000,24.000000,20.00000,30.000000,120.000000,72.000000,24.000000,120.000000,60.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
men_fertility.drop('Em_preg_now_ac',axis=1,inplace=True)

In [ ]:
fig = px.box(men_fertility, y="Em_hus_ideal_fert_t")
fig.update_layout(**layout_options)
fig.show()

In [ ]:
avg_boys = men_fertility['Em_hus_ideal_fert_b'].mean()
avg_girls = men_fertility['Em_hus_ideal_fert_g'].mean()
avg_either = men_fertility['Em_hus_ideal_fert_e'].mean()

plot_data = pd.DataFrame({
    'الجنس': ['boyes', 'Girls','either'],
    'متوسط العدد المفضل': [avg_boys, avg_girls,avg_either]
})
fig = px.bar(
    plot_data,
    x='الجنس',
    y='متوسط العدد المفضل',
    color='الجنس',
    color_discrete_sequence=['#636EFA', '#EF553B'],
)
fig.show()

### Men decision making el

In [ ]:
mens_decision_making_el.drop(columns=['Em_rights_sdi', 'Em_rights_di'],axis=1, inplace=True)
mens_decision_making_el.drop(columns=['Em_equality_sdi', 'Em_equality_di'], axis=1,inplace=True)
mens_decision_making_el.drop(columns=['Em_obey_sdi', 'Em_obey_di'], axis=1,inplace=True)
mens_decision_making_el.drop(columns=['Em_oblig_sex_sdi', 'Em_oblig_sex_di'],axis=1, inplace=True)
mens_decision_making_el.drop(columns=['Em_tol_viol_sdi', 'Em_tol_viol_di'],axis=1, inplace=True)
mens_decision_making_el.drop(columns=['Em_hit_argues_sdi', 'Em_hit_argues_di'],axis=1, inplace=True)
mens_decision_making_el.drop(columns=['Em_boss_sdi', 'Em_boss_di'],axis=1, inplace=True)
mens_decision_making_el.drop(columns=['Em_leaders_sdi', 'Em_leaders_di'],axis=1, inplace=True)
mens_decision_making_el.drop(columns=['Em_shrpow_sag', 'Em_shrpow_ag'], axis=1,inplace=True)
mens_decision_making_el.drop(columns=['Em_opin_sag', 'Em_opin_ag'], axis=1,inplace=True)
mens_decision_making_el.drop(columns=['Em_final_say_sag', 'Em_final_say_ag'], axis=1,inplace=True)

parent_df.drop(columns=['Em_rights_sdi', 'Em_rights_di'],axis=1, inplace=True)
parent_df.drop(columns=['Em_equality_sdi', 'Em_equality_di'], axis=1,inplace=True)
parent_df.drop(columns=['Em_obey_sdi', 'Em_obey_di'], axis=1,inplace=True)
parent_df.drop(columns=['Em_oblig_sex_sdi', 'Em_oblig_sex_di'],axis=1, inplace=True)
parent_df.drop(columns=['Em_tol_viol_sdi', 'Em_tol_viol_di'],axis=1, inplace=True)
parent_df.drop(columns=['Em_hit_argues_sdi', 'Em_hit_argues_di'],axis=1, inplace=True)
parent_df.drop(columns=['Em_boss_sdi', 'Em_boss_di'],axis=1, inplace=True)
parent_df.drop(columns=['Em_leaders_sdi', 'Em_leaders_di'],axis=1, inplace=True)
parent_df.drop(columns=['Em_shrpow_sag', 'Em_shrpow_ag'], axis=1,inplace=True)
parent_df.drop(columns=['Em_opin_sag', 'Em_opin_ag'], axis=1,inplace=True)
parent_df.drop(columns=['Em_final_say_sag', 'Em_final_say_ag'], axis=1,inplace=True)

In [ ]:
mens_decision_making_el.drop(columns=['Em_wmoney_bi','Em_chealth_bi','Em_cfeed_bi','Em_daily_bi'], axis=1,inplace=True)
parent_df.drop(columns=['Em_wmoney_bi','Em_chealth_bi','Em_cfeed_bi','Em_daily_bi'],axis=1, inplace=True)

In [ ]:
df = mens_decision_making_el.copy()
df['Em_rel_improve'] = df['Em_rel_improve'].astype(str)

for i in range(1, 7):
    col_name = f'Em_rel_improve{i}'
    expected_values = df['Em_rel_improve'].str.contains(rf'\b{i}\b', regex=True, na=False).astype(int)
    actual_values = df[col_name].fillna(0).astype(int)
    mismatches = df[expected_values != actual_values]
    num_mismatches = len(mismatches)

    if num_mismatches == 0:
        print(f" col  {col_name}: is 100% the same")
    else:
        print(f" col {col_name}: has {num_mismatches} unequal rows")

 col  Em_rel_improve1: is 100% the same
 col  Em_rel_improve2: is 100% the same
 col  Em_rel_improve3: is 100% the same
 col  Em_rel_improve4: is 100% the same
 col  Em_rel_improve5: is 100% the same
 col  Em_rel_improve6: is 100% the same


In [ ]:
df = mens_decision_making_el.copy()
df['Em_rel_worse'] = df['Em_rel_worse'].astype(str)

for i in range(1, 6):
    col_name = f'Em_rel_worse{i}'

    expected_values = df['Em_rel_worse'].str.contains(rf'\b{i}\b', regex=True, na=False).astype(int)
    actual_values = df[col_name].fillna(0).astype(int)
    mismatches = df[expected_values != actual_values]
    num_mismatches = len(mismatches)

    if num_mismatches == 0:
        print(f" col  {col_name}: is 100% the same")
    else:
        print(f" col {col_name}: has {num_mismatches} unequal rows")

 col  Em_rel_worse1: is 100% the same
 col  Em_rel_worse2: is 100% the same
 col  Em_rel_worse3: is 100% the same
 col  Em_rel_worse4: is 100% the same
 col  Em_rel_worse5: is 100% the same


In [ ]:
mens_decision_making_el.drop(columns=['Em_rel_worse','Em_rel_improve',], axis=1,inplace=True)
parent_df.drop(columns=['Em_rel_worse','Em_rel_improve',],axis=1, inplace=True)

In [ ]:
mens_decision_making_el.describe()

,Em_daily_diffop,Em_daily_express,Em_daily_argue,Em_chealth_diffop,Em_chealth_express,Em_chealth_argue,Em_cfeed_diffop,Em_cfeed_express,Em_cfeed_argue,Em_wmoney_diffop,Em_wmoney_express,Em_wmoney_argue,Em_wife_diss_sw,Em_wife_diss_vo,Em_rel_year_imp,Em_rel_improve1,Em_rel_improve2,Em_rel_improve3,Em_rel_improve4,Em_rel_improve5,Em_rel_improve6,Em_rel_worse1,Em_rel_worse2,Em_rel_worse3,Em_rel_worse4,Em_rel_worse5,Em_fam_contact_Less than once a year,Em_fam_contact_More than once a week,Em_fam_contact_Once or more every 6 months,Em_fam_contact_Once or more every month,Em_fam_contact_Once or more every year,Em_fam_contact_nan,Em_fem_friends_2-3 times per week,Em_fem_friends_3-4 times per weeks,Em_fem_friends_5-6 times per week,Em_fem_friends_Every day,Em_fem_friends_Less than once a week,Em_fem_friends_Once per week,Em_fem_friends_nan,Em_daily_who_Both together,Em_daily_who_Husband,Em_daily_who_Not applicable,Em_daily_who_Other HH member,Em_daily_who_Someone outside the HH,Em_daily_who_Wife,Em_daily_who_nan,Em_daily_prevail_A third person,Em_daily_prevail_Respondent finds another way,Em_daily_prevail_Your spouse,Em_daily_prevail_Yourself,Em_daily_prevail_nan,Em_chealth_who_Both together,Em_chealth_who_Husband,Em_chealth_who_Not applicable,Em_chealth_who_Someone outside the HH,Em_chealth_who_Wife,Em_chealth_who_nan,Em_chealth_prevail_Respondent finds another way,Em_chealth_prevail_Your spouse,Em_chealth_prevail_Yourself,Em_chealth_prevail_nan,Em_cfeed_who_Both together,Em_cfeed_who_Husband,Em_cfeed_who_Not applicable,Em_cfeed_who_Other HH member,Em_cfeed_who_Someone outside the HH,Em_cfeed_who_Wife,Em_cfeed_who_nan,Em_cfeed_prevail_Respondent finds another way,Em_cfeed_prevail_Your spouse,Em_cfeed_prevail_Yourself,Em_cfeed_prevail_nan,Em_wmoney_who_Both together,Em_wmoney_who_Husband,Em_wmoney_who_Not applicable,Em_wmoney_who_Wife,Em_wmoney_who_nan,Em_wmoney_prevail_Respondent finds another way,Em_wmoney_prevail_Your spouse,Em_wmoney_prevail_Yourself,Em_wmoney_prevail_nan,Em_quarrel_in the past day,Em_quarrel_in the past month,Em_quarrel_in the past week,Em_quarrel_in the past year,Em_quarrel_more than one year ago,Em_quarrel_never,Em_quarrel_nan,Em_rights_Agree,Em_rights_Disagree,Em_rights_N/A,Em_rights_Strongly Agree,Em_rights_Strongly Disagree,Em_rights_nan,Em_equality_Agree,Em_equality_Disagree,Em_equality_N/A,Em_equality_Strongly Agree,Em_equality_Strongly Disagree,Em_equality_nan,Em_obey_Agree,Em_obey_Disagree,Em_obey_N/A,Em_obey_Strongly Agree,Em_obey_Strongly Disagree,Em_obey_nan,Em_boss_Agree,Em_boss_Disagree,Em_boss_N/A,Em_boss_Strongly Agree,Em_boss_Strongly Disagree,Em_boss_nan,Em_leaders_Agree,Em_leaders_Disagree,Em_leaders_N/A,Em_leaders_Strongly Agree,Em_leaders_Strongly Disagree,Em_leaders_nan,Em_shrpow_Agree,Em_shrpow_Disagree,Em_shrpow_N/A,Em_shrpow_Strongly Agree,Em_shrpow_Strongly Disagree,Em_shrpow_nan,Em_opin_Agree,Em_opin_Disagree,Em_opin_N/A,Em_opin_Strongly Agree,Em_opin_Strongly Disagree,Em_opin_nan,Em_final_say_Agree,Em_final_say_Disagree,Em_final_say_N/A,Em_final_say_Strongly Agree,Em_final_say_Strongly Disagree,Em_final_say_nan,Em_oblig_sex_Agree,Em_oblig_sex_Disagree,Em_oblig_sex_N/A,Em_oblig_sex_Strongly Agree,Em_oblig_sex_Strongly Disagree,Em_oblig_sex_nan,Em_tol_viol_Agree,Em_tol_viol_Disagree,Em_tol_viol_N/A,Em_tol_viol_Strongly Agree,Em_tol_viol_Strongly Disagree,Em_tol_viol_nan,Em_hit_argues_Agree,Em_hit_argues_Disagree,Em_hit_argues_N/A,Em_hit_argues_Strongly Agree,Em_hit_argues_Strongly Disagree,Em_hit_argues_nan,Em_talked_hiv_Neither of us have ever talked about our HIV status with each other,Em_talked_hiv_We have only talked about her HIV status,Em_talked_hiv_We have only talked about my HIV status,"Em_talked_hiv_Yes, we both know each others’ status",Em_talked_hiv_nan,Em_wife_comm_We are always able to discuss issues in a fair and reasonable way each other,Em_wife_comm_We are not able to discuss an issue without having an argument,"Em_wife_comm_We can talk about most issues, but we argue so

In [ ]:
mens_decision_making_el.to_csv('out.csv',index=False)

In [ ]:
prefixes = ['Em_chealth', 'Em_daily', 'Em_cfeed', 'Em_wmoney']
for cat in prefixes:
  diffop_col = f'{cat}_diffop'
  express_col = f'{cat}_express'
  argue_col = f'{cat}_argue'
  prevail_col = f'{cat}_prevail'
  print(f"col {argue_col} has {mens_decision_making_el[argue_col].isnull().sum()}")

col Em_chealth_argue has 5188
col Em_daily_argue has 4652
col Em_cfeed_argue has 5326
col Em_wmoney_argue has 4862


In [ ]:
df = mens_decision_making_el[(mens_decision_making_el['Em_chealth_prevail'].notnull()) & (mens_decision_making_el['Em_chealth_argue'].isnull()) ]
df.head()

,Em_fam_contact,Em_fem_friends,Em_daily_who,Em_daily_diffop,Em_daily_express,Em_daily_argue,Em_daily_prevail,Em_chealth_who,Em_chealth_diffop,Em_chealth_express,Em_chealth_argue,Em_chealth_prevail,Em_cfeed_who,Em_cfeed_diffop,Em_cfeed_express,Em_cfeed_argue,Em_cfeed_prevail,Em_wmoney_who,Em_wmoney_diffop,Em_wmoney_express,Em_wmoney_argue,Em_wmoney_prevail,Em_quarrel,Em_rights,Em_equality,Em_obey,Em_boss,Em_leaders,Em_shrpow,Em_opin,Em_final_say,Em_oblig_sex,Em_tol_viol,Em_hit_argues,Em_talked_hiv,Em_wife_comm,Em_wife_diss,Em_wife_diss_sw,Em_wife_diss_vo,Em_rel_year,Em_rel_year_imp,Em_rel_improve1,Em_rel_improve2,Em_rel_improve3,Em_rel_improve4,Em_rel_improve5,Em_rel_improve6,Em_rel_worse1,Em_rel_worse2,Em_rel_worse3,Em_rel_worse4,Em_rel_worse5,Em_fam_contact_Less than once a year,Em_fam_contact_More than once a week,Em_fam_contact_Once or more every 6 months,Em_fam_contact_Once or more every month,Em_fam_contact_Once or more every year,Em_fam_contact_nan,Em_fem_friends_2-3 times per week,Em_fem_friends_3-4 times per weeks,Em_fem_friends_5-6 times per week,Em_fem_friends_Every day,Em_fem_friends_Less than once a week,Em_fem_friends_Once per week,Em_fem_friends_nan,Em_daily_who_Both together,Em_daily_who_Husband,Em_daily_who_Not applicable,Em_daily_who_Other HH member,Em_daily_who_Someone outside the HH,Em_daily_who_Wife,Em_daily_who_nan,Em_daily_prevail_A third person,Em_daily_prevail_Respondent finds another way,Em_daily_prevail_Your spouse,Em_daily_prevail_Yourself,Em_daily_prevail_nan,Em_chealth_who_Both together,Em_chealth_who_Husband,Em_chealth_who_Not applicable,Em_chealth_who_Someone outside the HH,Em_chealth_who_Wife,Em_chealth_who_nan,Em_chealth_prevail_Respondent finds another way,Em_chealth_prevail_Your spouse,Em_chealth_prevail_Yourself,Em_chealth_prevail_nan,Em_cfeed_who_Both together,Em_cfeed_who_Husband,Em_cfeed_who_Not applicable,Em_cfeed_who_Other HH member,Em_cfeed_who_Someone outside the HH,Em_cfeed_who_Wife,Em_cfeed_who_nan,Em_cfeed_prevail_Respondent finds another way,Em_cfeed_prevail_Your spouse,Em_cfeed_prevail_Yourself,Em_cfeed_prevail_nan,Em_wmoney_who_Both together,Em_wmoney_who_Husband,Em_wmoney_who_Not applicable,Em_wmoney_who_Wife,Em_wmoney_who_nan,Em_wmoney_prevail_Respondent finds another way,Em_wmoney_prevail_Your spouse,Em_wmoney_prevail_Yourself,Em_wmoney_prevail_nan,Em_quarrel_in the past day,Em_quarrel_in the past month,Em_quarrel_in the past week,Em_quarrel_in the past year,Em_quarrel_more than one year ago,Em_quarrel_never,Em_quarrel_nan,Em_rights_Agree,Em_rights_Disagree,Em_rights_N/A,Em_rights_Strongly Agree,Em_rights_Strongly Disagree,Em_rights_nan,Em_equality_Agree,Em_equality_Disagree,Em_equality_N/A,Em_equality_Strongly Agree,Em_equality_Strongly Disagree,Em_equality_nan,Em_obey_Agree,Em_obey_Disagree,Em_obey_N/A,Em_obey_Strongly Agree,Em_obey_Strongly Disagree,Em_obey_nan,Em_boss_Agree,Em_boss_Disagree,Em_boss_N/A,Em_boss_Strongly Agree,Em_boss_Strongly Disagree,Em_boss_nan,Em_leaders_Agree,Em_leaders_Disagree,Em_leaders_N/A,Em_leaders_Strongly Agree,Em_leaders_Strongly Disagree,Em_leaders_nan,Em_shrpow_Agree,Em_shrpow_Disagree,Em_shrpow_N/A,Em_shrpow_Strongly Agree,Em_shrpow_Strongly Disagree,Em_shrpow_nan,Em_opin_Agree,Em_opin_Disagree,Em_opin_N/A,Em_opin_Strongly Agree,Em_opin_Strongly Disagree,Em_opin_nan,Em_final_say_Agree,Em_final_say_Disagree,Em_final_say_N/A,Em_final_say_Strongly Agree,Em_final_say_Strongly Disagree,Em_final_say_nan,Em_oblig_sex_Agree,Em_oblig_sex_Disagree,Em_oblig_sex_N/A,Em_oblig_sex_Strongly Agree,Em_oblig_sex_Strongly Disagree,Em_oblig_sex_nan,Em_tol_viol_Agree,Em_tol_viol_Disagree,Em_tol_viol_N/A,Em_tol_viol_Strongly Agree,Em_tol_viol_Strongly Disagree,Em_tol_viol_nan,Em_hit_argues_Agree,Em_hit_argues_Disagree,Em_hit_argues_N/A,Em_hit_argues_Strongly Agree,Em_hit_argues_Strongly Disagree,Em_hit_argues_nan,Em_talked_hiv_Neither of us have ever talked about our HIV status with each other,Em_talked_hiv_We have only talked about her HIV status,Em_talked_hiv_

In [ ]:
df = mens_decision_making_el[(mens_decision_making_el['Em_daily_express'].notnull()) & (mens_decision_making_el['Em_daily_diffop'].isnull()) ]
df.head()

,Em_fam_contact,Em_fem_friends,Em_daily_who,Em_daily_diffop,Em_daily_express,Em_daily_argue,Em_daily_prevail,Em_chealth_who,Em_chealth_diffop,Em_chealth_express,Em_chealth_argue,Em_chealth_prevail,Em_cfeed_who,Em_cfeed_diffop,Em_cfeed_express,Em_cfeed_argue,Em_cfeed_prevail,Em_wmoney_who,Em_wmoney_diffop,Em_wmoney_express,Em_wmoney_argue,Em_wmoney_prevail,Em_quarrel,Em_rights,Em_equality,Em_obey,Em_boss,Em_leaders,Em_shrpow,Em_opin,Em_final_say,Em_oblig_sex,Em_tol_viol,Em_hit_argues,Em_talked_hiv,Em_wife_comm,Em_wife_diss,Em_wife_diss_sw,Em_wife_diss_vo,Em_rel_year,Em_rel_year_imp,Em_rel_improve1,Em_rel_improve2,Em_rel_improve3,Em_rel_improve4,Em_rel_improve5,Em_rel_improve6,Em_rel_worse1,Em_rel_worse2,Em_rel_worse3,Em_rel_worse4,Em_rel_worse5,Em_fam_contact_Less than once a year,Em_fam_contact_More than once a week,Em_fam_contact_Once or more every 6 months,Em_fam_contact_Once or more every month,Em_fam_contact_Once or more every year,Em_fam_contact_nan,Em_fem_friends_2-3 times per week,Em_fem_friends_3-4 times per weeks,Em_fem_friends_5-6 times per week,Em_fem_friends_Every day,Em_fem_friends_Less than once a week,Em_fem_friends_Once per week,Em_fem_friends_nan,Em_daily_who_Both together,Em_daily_who_Husband,Em_daily_who_Not applicable,Em_daily_who_Other HH member,Em_daily_who_Someone outside the HH,Em_daily_who_Wife,Em_daily_who_nan,Em_daily_prevail_A third person,Em_daily_prevail_Respondent finds another way,Em_daily_prevail_Your spouse,Em_daily_prevail_Yourself,Em_daily_prevail_nan,Em_chealth_who_Both together,Em_chealth_who_Husband,Em_chealth_who_Not applicable,Em_chealth_who_Someone outside the HH,Em_chealth_who_Wife,Em_chealth_who_nan,Em_chealth_prevail_Respondent finds another way,Em_chealth_prevail_Your spouse,Em_chealth_prevail_Yourself,Em_chealth_prevail_nan,Em_cfeed_who_Both together,Em_cfeed_who_Husband,Em_cfeed_who_Not applicable,Em_cfeed_who_Other HH member,Em_cfeed_who_Someone outside the HH,Em_cfeed_who_Wife,Em_cfeed_who_nan,Em_cfeed_prevail_Respondent finds another way,Em_cfeed_prevail_Your spouse,Em_cfeed_prevail_Yourself,Em_cfeed_prevail_nan,Em_wmoney_who_Both together,Em_wmoney_who_Husband,Em_wmoney_who_Not applicable,Em_wmoney_who_Wife,Em_wmoney_who_nan,Em_wmoney_prevail_Respondent finds another way,Em_wmoney_prevail_Your spouse,Em_wmoney_prevail_Yourself,Em_wmoney_prevail_nan,Em_quarrel_in the past day,Em_quarrel_in the past month,Em_quarrel_in the past week,Em_quarrel_in the past year,Em_quarrel_more than one year ago,Em_quarrel_never,Em_quarrel_nan,Em_rights_Agree,Em_rights_Disagree,Em_rights_N/A,Em_rights_Strongly Agree,Em_rights_Strongly Disagree,Em_rights_nan,Em_equality_Agree,Em_equality_Disagree,Em_equality_N/A,Em_equality_Strongly Agree,Em_equality_Strongly Disagree,Em_equality_nan,Em_obey_Agree,Em_obey_Disagree,Em_obey_N/A,Em_obey_Strongly Agree,Em_obey_Strongly Disagree,Em_obey_nan,Em_boss_Agree,Em_boss_Disagree,Em_boss_N/A,Em_boss_Strongly Agree,Em_boss_Strongly Disagree,Em_boss_nan,Em_leaders_Agree,Em_leaders_Disagree,Em_leaders_N/A,Em_leaders_Strongly Agree,Em_leaders_Strongly Disagree,Em_leaders_nan,Em_shrpow_Agree,Em_shrpow_Disagree,Em_shrpow_N/A,Em_shrpow_Strongly Agree,Em_shrpow_Strongly Disagree,Em_shrpow_nan,Em_opin_Agree,Em_opin_Disagree,Em_opin_N/A,Em_opin_Strongly Agree,Em_opin_Strongly Disagree,Em_opin_nan,Em_final_say_Agree,Em_final_say_Disagree,Em_final_say_N/A,Em_final_say_Strongly Agree,Em_final_say_Strongly Disagree,Em_final_say_nan,Em_oblig_sex_Agree,Em_oblig_sex_Disagree,Em_oblig_sex_N/A,Em_oblig_sex_Strongly Agree,Em_oblig_sex_Strongly Disagree,Em_oblig_sex_nan,Em_tol_viol_Agree,Em_tol_viol_Disagree,Em_tol_viol_N/A,Em_tol_viol_Strongly Agree,Em_tol_viol_Strongly Disagree,Em_tol_viol_nan,Em_hit_argues_Agree,Em_hit_argues_Disagree,Em_hit_argues_N/A,Em_hit_argues_Strongly Agree,Em_hit_argues_Strongly Disagree,Em_hit_argues_nan,Em_talked_hiv_Neither of us have ever talked about our HIV status with each other,Em_talked_hiv_We have only talked about her HIV status,Em_talked_hiv_

In [ ]:
condition = (mens_decision_making_el['Em_daily_express'].notnull()) & (mens_decision_making_el['Em_daily_diffop'].isnull())
mens_decision_making_el.loc[condition, 'Em_daily_diffop'] = 1
parent_df.loc[condition, 'Em_daily_diffop'] = 1
df_check = mens_decision_making_el[(mens_decision_making_el['Em_daily_express'].notnull()) & (mens_decision_making_el['Em_daily_diffop'].isnull())]


In [ ]:
df = mens_decision_making_el[(mens_decision_making_el['Em_chealth_express'].notnull()) & (mens_decision_making_el['Em_chealth_diffop'].isnull()) ]
df.head()

,Em_fam_contact,Em_fem_friends,Em_daily_who,Em_daily_diffop,Em_daily_express,Em_daily_argue,Em_daily_prevail,Em_chealth_who,Em_chealth_diffop,Em_chealth_express,Em_chealth_argue,Em_chealth_prevail,Em_cfeed_who,Em_cfeed_diffop,Em_cfeed_express,Em_cfeed_argue,Em_cfeed_prevail,Em_wmoney_who,Em_wmoney_diffop,Em_wmoney_express,Em_wmoney_argue,Em_wmoney_prevail,Em_quarrel,Em_rights,Em_equality,Em_obey,Em_boss,Em_leaders,Em_shrpow,Em_opin,Em_final_say,Em_oblig_sex,Em_tol_viol,Em_hit_argues,Em_talked_hiv,Em_wife_comm,Em_wife_diss,Em_wife_diss_sw,Em_wife_diss_vo,Em_rel_year,Em_rel_year_imp,Em_rel_improve1,Em_rel_improve2,Em_rel_improve3,Em_rel_improve4,Em_rel_improve5,Em_rel_improve6,Em_rel_worse1,Em_rel_worse2,Em_rel_worse3,Em_rel_worse4,Em_rel_worse5,Em_fam_contact_Less than once a year,Em_fam_contact_More than once a week,Em_fam_contact_Once or more every 6 months,Em_fam_contact_Once or more every month,Em_fam_contact_Once or more every year,Em_fam_contact_nan,Em_fem_friends_2-3 times per week,Em_fem_friends_3-4 times per weeks,Em_fem_friends_5-6 times per week,Em_fem_friends_Every day,Em_fem_friends_Less than once a week,Em_fem_friends_Once per week,Em_fem_friends_nan,Em_daily_who_Both together,Em_daily_who_Husband,Em_daily_who_Not applicable,Em_daily_who_Other HH member,Em_daily_who_Someone outside the HH,Em_daily_who_Wife,Em_daily_who_nan,Em_daily_prevail_A third person,Em_daily_prevail_Respondent finds another way,Em_daily_prevail_Your spouse,Em_daily_prevail_Yourself,Em_daily_prevail_nan,Em_chealth_who_Both together,Em_chealth_who_Husband,Em_chealth_who_Not applicable,Em_chealth_who_Someone outside the HH,Em_chealth_who_Wife,Em_chealth_who_nan,Em_chealth_prevail_Respondent finds another way,Em_chealth_prevail_Your spouse,Em_chealth_prevail_Yourself,Em_chealth_prevail_nan,Em_cfeed_who_Both together,Em_cfeed_who_Husband,Em_cfeed_who_Not applicable,Em_cfeed_who_Other HH member,Em_cfeed_who_Someone outside the HH,Em_cfeed_who_Wife,Em_cfeed_who_nan,Em_cfeed_prevail_Respondent finds another way,Em_cfeed_prevail_Your spouse,Em_cfeed_prevail_Yourself,Em_cfeed_prevail_nan,Em_wmoney_who_Both together,Em_wmoney_who_Husband,Em_wmoney_who_Not applicable,Em_wmoney_who_Wife,Em_wmoney_who_nan,Em_wmoney_prevail_Respondent finds another way,Em_wmoney_prevail_Your spouse,Em_wmoney_prevail_Yourself,Em_wmoney_prevail_nan,Em_quarrel_in the past day,Em_quarrel_in the past month,Em_quarrel_in the past week,Em_quarrel_in the past year,Em_quarrel_more than one year ago,Em_quarrel_never,Em_quarrel_nan,Em_rights_Agree,Em_rights_Disagree,Em_rights_N/A,Em_rights_Strongly Agree,Em_rights_Strongly Disagree,Em_rights_nan,Em_equality_Agree,Em_equality_Disagree,Em_equality_N/A,Em_equality_Strongly Agree,Em_equality_Strongly Disagree,Em_equality_nan,Em_obey_Agree,Em_obey_Disagree,Em_obey_N/A,Em_obey_Strongly Agree,Em_obey_Strongly Disagree,Em_obey_nan,Em_boss_Agree,Em_boss_Disagree,Em_boss_N/A,Em_boss_Strongly Agree,Em_boss_Strongly Disagree,Em_boss_nan,Em_leaders_Agree,Em_leaders_Disagree,Em_leaders_N/A,Em_leaders_Strongly Agree,Em_leaders_Strongly Disagree,Em_leaders_nan,Em_shrpow_Agree,Em_shrpow_Disagree,Em_shrpow_N/A,Em_shrpow_Strongly Agree,Em_shrpow_Strongly Disagree,Em_shrpow_nan,Em_opin_Agree,Em_opin_Disagree,Em_opin_N/A,Em_opin_Strongly Agree,Em_opin_Strongly Disagree,Em_opin_nan,Em_final_say_Agree,Em_final_say_Disagree,Em_final_say_N/A,Em_final_say_Strongly Agree,Em_final_say_Strongly Disagree,Em_final_say_nan,Em_oblig_sex_Agree,Em_oblig_sex_Disagree,Em_oblig_sex_N/A,Em_oblig_sex_Strongly Agree,Em_oblig_sex_Strongly Disagree,Em_oblig_sex_nan,Em_tol_viol_Agree,Em_tol_viol_Disagree,Em_tol_viol_N/A,Em_tol_viol_Strongly Agree,Em_tol_viol_Strongly Disagree,Em_tol_viol_nan,Em_hit_argues_Agree,Em_hit_argues_Disagree,Em_hit_argues_N/A,Em_hit_argues_Strongly Agree,Em_hit_argues_Strongly Disagree,Em_hit_argues_nan,Em_talked_hiv_Neither of us have ever talked about our HIV status with each other,Em_talked_hiv_We have only talked about her HIV status,Em_talked_hiv_

In [ ]:
condition = (mens_decision_making_el['Em_chealth_express'].notnull()) & (mens_decision_making_el['Em_chealth_diffop'].isnull())
mens_decision_making_el.loc[condition, 'Em_chealth_diffop'] = 1
parent_df.loc[condition, 'Em_chealth_diffop'] = 1
df_check = mens_decision_making_el[(mens_decision_making_el['Em_chealth_express'].notnull()) & (mens_decision_making_el['Em_chealth_diffop'].isnull())]

In [ ]:
df = mens_decision_making_el[(mens_decision_making_el['Em_cfeed_express'].notnull()) & (mens_decision_making_el['Em_cfeed_diffop'].isnull()) ]
df.head()

,Em_fam_contact,Em_fem_friends,Em_daily_who,Em_daily_diffop,Em_daily_express,Em_daily_argue,Em_daily_prevail,Em_chealth_who,Em_chealth_diffop,Em_chealth_express,Em_chealth_argue,Em_chealth_prevail,Em_cfeed_who,Em_cfeed_diffop,Em_cfeed_express,Em_cfeed_argue,Em_cfeed_prevail,Em_wmoney_who,Em_wmoney_diffop,Em_wmoney_express,Em_wmoney_argue,Em_wmoney_prevail,Em_quarrel,Em_rights,Em_equality,Em_obey,Em_boss,Em_leaders,Em_shrpow,Em_opin,Em_final_say,Em_oblig_sex,Em_tol_viol,Em_hit_argues,Em_talked_hiv,Em_wife_comm,Em_wife_diss,Em_wife_diss_sw,Em_wife_diss_vo,Em_rel_year,Em_rel_year_imp,Em_rel_improve1,Em_rel_improve2,Em_rel_improve3,Em_rel_improve4,Em_rel_improve5,Em_rel_improve6,Em_rel_worse1,Em_rel_worse2,Em_rel_worse3,Em_rel_worse4,Em_rel_worse5,Em_fam_contact_Less than once a year,Em_fam_contact_More than once a week,Em_fam_contact_Once or more every 6 months,Em_fam_contact_Once or more every month,Em_fam_contact_Once or more every year,Em_fam_contact_nan,Em_fem_friends_2-3 times per week,Em_fem_friends_3-4 times per weeks,Em_fem_friends_5-6 times per week,Em_fem_friends_Every day,Em_fem_friends_Less than once a week,Em_fem_friends_Once per week,Em_fem_friends_nan,Em_daily_who_Both together,Em_daily_who_Husband,Em_daily_who_Not applicable,Em_daily_who_Other HH member,Em_daily_who_Someone outside the HH,Em_daily_who_Wife,Em_daily_who_nan,Em_daily_prevail_A third person,Em_daily_prevail_Respondent finds another way,Em_daily_prevail_Your spouse,Em_daily_prevail_Yourself,Em_daily_prevail_nan,Em_chealth_who_Both together,Em_chealth_who_Husband,Em_chealth_who_Not applicable,Em_chealth_who_Someone outside the HH,Em_chealth_who_Wife,Em_chealth_who_nan,Em_chealth_prevail_Respondent finds another way,Em_chealth_prevail_Your spouse,Em_chealth_prevail_Yourself,Em_chealth_prevail_nan,Em_cfeed_who_Both together,Em_cfeed_who_Husband,Em_cfeed_who_Not applicable,Em_cfeed_who_Other HH member,Em_cfeed_who_Someone outside the HH,Em_cfeed_who_Wife,Em_cfeed_who_nan,Em_cfeed_prevail_Respondent finds another way,Em_cfeed_prevail_Your spouse,Em_cfeed_prevail_Yourself,Em_cfeed_prevail_nan,Em_wmoney_who_Both together,Em_wmoney_who_Husband,Em_wmoney_who_Not applicable,Em_wmoney_who_Wife,Em_wmoney_who_nan,Em_wmoney_prevail_Respondent finds another way,Em_wmoney_prevail_Your spouse,Em_wmoney_prevail_Yourself,Em_wmoney_prevail_nan,Em_quarrel_in the past day,Em_quarrel_in the past month,Em_quarrel_in the past week,Em_quarrel_in the past year,Em_quarrel_more than one year ago,Em_quarrel_never,Em_quarrel_nan,Em_rights_Agree,Em_rights_Disagree,Em_rights_N/A,Em_rights_Strongly Agree,Em_rights_Strongly Disagree,Em_rights_nan,Em_equality_Agree,Em_equality_Disagree,Em_equality_N/A,Em_equality_Strongly Agree,Em_equality_Strongly Disagree,Em_equality_nan,Em_obey_Agree,Em_obey_Disagree,Em_obey_N/A,Em_obey_Strongly Agree,Em_obey_Strongly Disagree,Em_obey_nan,Em_boss_Agree,Em_boss_Disagree,Em_boss_N/A,Em_boss_Strongly Agree,Em_boss_Strongly Disagree,Em_boss_nan,Em_leaders_Agree,Em_leaders_Disagree,Em_leaders_N/A,Em_leaders_Strongly Agree,Em_leaders_Strongly Disagree,Em_leaders_nan,Em_shrpow_Agree,Em_shrpow_Disagree,Em_shrpow_N/A,Em_shrpow_Strongly Agree,Em_shrpow_Strongly Disagree,Em_shrpow_nan,Em_opin_Agree,Em_opin_Disagree,Em_opin_N/A,Em_opin_Strongly Agree,Em_opin_Strongly Disagree,Em_opin_nan,Em_final_say_Agree,Em_final_say_Disagree,Em_final_say_N/A,Em_final_say_Strongly Agree,Em_final_say_Strongly Disagree,Em_final_say_nan,Em_oblig_sex_Agree,Em_oblig_sex_Disagree,Em_oblig_sex_N/A,Em_oblig_sex_Strongly Agree,Em_oblig_sex_Strongly Disagree,Em_oblig_sex_nan,Em_tol_viol_Agree,Em_tol_viol_Disagree,Em_tol_viol_N/A,Em_tol_viol_Strongly Agree,Em_tol_viol_Strongly Disagree,Em_tol_viol_nan,Em_hit_argues_Agree,Em_hit_argues_Disagree,Em_hit_argues_N/A,Em_hit_argues_Strongly Agree,Em_hit_argues_Strongly Disagree,Em_hit_argues_nan,Em_talked_hiv_Neither of us have ever talked about our HIV status with each other,Em_talked_hiv_We have only talked about her HIV status,Em_talked_hiv_

In [ ]:
condition = (mens_decision_making_el['Em_cfeed_express'].notnull()) & (mens_decision_making_el['Em_cfeed_diffop'].isnull())
mens_decision_making_el.loc[condition, 'Em_cfeed_diffop'] = 1
parent_df.loc[condition, 'Em_cfeed_diffop'] = 1
df_check = mens_decision_making_el[(mens_decision_making_el['Em_cfeed_express'].notnull()) & (mens_decision_making_el['Em_cfeed_diffop'].isnull())]

In [ ]:
condition = (mens_decision_making_el['Em_wmoney_express'].notnull()) & (mens_decision_making_el['Em_wmoney_diffop'].isnull())
mens_decision_making_el.loc[condition, 'Em_wmoney_diffop'] = 1
parent_df.loc[condition, 'Em_wmoney_diffop'] = 1
df_check = mens_decision_making_el[(mens_decision_making_el['Em_wmoney_express'].notnull()) & (mens_decision_making_el['Em_wmoney_diffop'].isnull())]

In [ ]:
prefixes = ['Em_chealth', 'Em_daily', 'Em_cfeed', 'Em_wmoney']
for cat in prefixes:
  diffop_col = f'{cat}_diffop'
  express_col = f'{cat}_express'
  argue_col = f'{cat}_argue'
  prevail_col = f'{cat}_prevail'
  print(f"col {diffop_col} has {mens_decision_making_el[diffop_col].isnull().sum()}")

col Em_chealth_diffop has 336
col Em_daily_diffop has 336
col Em_cfeed_diffop has 336
col Em_wmoney_diffop has 336


In [ ]:
men_improve_rename = {
    'Em_rel_improve1': 'Em_improve_husband_listens',
    'Em_rel_improve2': 'Em_improve_wife_listens',
    'Em_rel_improve3': 'Em_improve_fewer_args',
    'Em_rel_improve4': 'Em_improve_husb_involved',
    'Em_rel_improve5': 'Em_improve_share_finances',
    'Em_rel_improve6': 'Em_improve_share_info'
}

men_worse_rename = {
    'Em_rel_worse1': 'Em_worse_husband_listens_less',
    'Em_rel_worse2': 'Em_worse_wife_listens_less',
    'Em_rel_worse3': 'Em_worse_more_args',
    'Em_rel_worse4': 'Em_worse_husb_less_time',
    'Em_rel_worse5': 'Em_worse_argue_finances'
}

parent_df.rename(columns=men_improve_rename, inplace=True)
parent_df.rename(columns=men_worse_rename, inplace=True)
mens_decision_making_el.rename(columns=men_improve_rename, inplace=True)
mens_decision_making_el.rename(columns=men_worse_rename, inplace=True)

In [ ]:
prefixes = ['Em_chealth', 'Em_daily', 'Em_cfeed', 'Em_wmoney']
for cat in prefixes:
  diffop_col = f'{cat}_diffop'
  express_col = f'{cat}_express'
  argue_col = f'{cat}_argue'
  prevail_col = f'{cat}_prevail'
  print(f"col {express_col} has {mens_decision_making_el[express_col].isnull().sum()}")
  print(f"col {argue_col} has {mens_decision_making_el[argue_col].isnull().sum()}")
  print(f"col {prevail_col} has {mens_decision_making_el[prevail_col].isnull().sum()}")

col Em_chealth_express has 1852
col Em_chealth_argue has 5188
col Em_chealth_prevail has 5285
col Em_daily_express has 1977
col Em_daily_argue has 4652
col Em_daily_prevail has 4930
col Em_cfeed_express has 4946
col Em_cfeed_argue has 5326
col Em_cfeed_prevail has 5395
col Em_wmoney_express has 4281
col Em_wmoney_argue has 4862
col Em_wmoney_prevail has 5137


In [ ]:
df = parent_df.copy()
group_conditions = {
    'مجموعة التحكم (لا تدريب)': (df['mhn'] != 1) & (df['whn'] != 1) & (df['wemp'] != 1),
    'تدريب الرجال (MHN)': df['mhn'] == 1,
    'تدريب النساء (WHN)': df['whn'] == 1,
    'تمكين النساء (WEMP)': df['wemp'] == 1
}
prefixes = ['Em_chealth', 'Em_daily', 'Em_cfeed', 'Em_wmoney']
x_labels = ['صحة الأطفال', 'القرارات اليومية', 'إطعام الأطفال', 'أموال الزوجة']
metrics = ['diffop', 'express', 'argue']

for metric in metrics:
    fig = go.Figure()
    for group_name, condition in group_conditions.items():
        df_sub = df[condition]
        y_values = []
        for prefix in prefixes:
            col_name = f'{prefix}_{metric}'
            if col_name in df_sub.columns:
                val = df_sub[col_name].mean() * 100
            else:
                val = 0
            y_values.append(val)

        fig.add_trace(go.Bar(
            name=group_name,
            x=x_labels,
            y=y_values
        ))

    fig.update_layout(
        barmode='group',
        title=f"تحليل مرحلة: {metric}",
        xaxis_title="مجال اتخاذ القرار",
        yaxis_title="النسبة (%)",
    )
    fig.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd
df = parent_df.copy()
group_conditions = {
    'مجموعة التحكم (لا تدريب)': (df['mhn'] != 1) & (df['whn'] != 1) & (df['wemp'] != 1),
    'تدريب الرجال (MHN)': df['mhn'] == 1,
    'تدريب النساء (WHN)': df['whn'] == 1,
    'تمكين النساء (WEMP)': df['wemp'] == 1
}

prefixes = ['Em_chealth', 'Em_daily', 'Em_cfeed', 'Em_wmoney']
x_labels = ['صحة الأطفال', 'القرارات اليومية', 'إطعام الأطفال', 'أموال الزوجة']
col_suffix = '_prevail'

all_categories = set()
for prefix in prefixes:
    col_name = f'{prefix}{col_suffix}'
    if col_name in df.columns:
        all_categories.update(df[col_name].dropna().unique())

categories = list(all_categories)
fig = go.Figure()

for group_name, condition in group_conditions.items():
    df_sub = df[condition]
    for cat in categories:
        y_values = []
        for prefix in prefixes:
            col_name = f'{prefix}{col_suffix}'
            if col_name in df_sub.columns:
                # حساب النسبة المئوية لكل فئة
                val = (df_sub[col_name] == cat).mean() * 100
                y_values.append(val)
            else:
                y_values.append(0)

        if sum(y_values) > 0:
            fig.add_trace(go.Bar(
                name=f"{group_name} ({cat})",
                x=x_labels,
                y=y_values
            ))
fig.update_layout(
    barmode='group',
    title="توزيع 'من يفرض رأيه' (Prevail) بحسب التدريب ومجال القرار بالنسبة للرجال",
    xaxis_title="مجال اتخاذ القرار",
    yaxis_title="النسبة (%)",
    legend_title="المجموعة - الطرف المقرر",

    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)

fig.show()


1. السيطرة الذكورية المطلقة على القرارات (Male Dominance)

الاستنتاج الأبرز هو أن الزوج هو من يفرض رأيه (الفئة 1) في الأغلبية الساحقة من حالات الخلاف، وفي جميع المجالات تقريباً (سواء صحة الأطفال، أو إطعامهم، أو القرارات اليومية).

هذا يعكس بوضوح الطبيعة الأبوية  للمجتمعات الريفية المدروسة، حيث يعتبر الرجل نفسه صاحب "الكلمة الأخيرة" والسلطة العليا في المنزل عندما يحدث أي تضارب في الآراء.

2. التناقض في "أموال الزوجة"

حتى في المجال الذي يخص "أموال الزوجة" (التي تجنيها بنفسها)، إذا كانت الفئة 1 (الزوج) لا تزال مرتفعة،:

 "النساء يفتقرن للقدرة على التفاوض حتى فيما يخص أموالهن الخاصة، حيث يفرض الرجل رأيه في كيفية إنفاق دخل زوجته."

3. تقييم أثر التدريب (WEMP و MHN)
  إن التدريب نجح بالفعل في تخفيف حدة السيطرة الذكورية المطلقة، وجعل الرجال أكثر تقبلاً للتنازل أو إشراك الزوجة في القرار النهائي عند الخلاف.


In [ ]:
df = parent_df.copy()
group_conditions = {
    'مجموعة التحكم (لا تدريب)': (df['mhn'] != 1) & (df['whn'] != 1) & (df['wemp'] != 1),
    'تدريب الرجال (MHN)': df['mhn'] == 1,
    'تدريب النساء (WHN)': df['whn'] == 1,
    'تمكين النساء (WEMP)': df['wemp'] == 1
}
domains = ['daily', 'chealth', 'cfeed', 'wmoney']
metrics = ['diffop', 'express', 'argue', 'prevail']
respondents = [
    ('w', 'Ew', 'النساء'),
    ('m', 'Em', 'الرجال')
]
for base_pref, end_pref, resp_name in respondents:
    for domain in domains:
        for metric in metrics:
            base_col = f"{base_pref}_{domain}_{metric}"
            end_col = f"{end_pref}_{domain}_{metric}"
            if base_col not in df.columns or end_col not in df.columns:
                continue
            fig = go.Figure()
            for group_name, condition in group_conditions.items():
                df_sub = df[condition]
                base_val = df_sub[base_col].mean() * 100 if len(df_sub) > 0 else 0
                end_val = df_sub[end_col].mean() * 100 if len(df_sub) > 0 else 0
                fig.add_trace(go.Bar(
                    name=group_name,
                    x=['قبل التدريب (Baseline)', 'بعد التدريب (Endline)'],
                    y=[base_val, end_val]
                ))
            fig.update_layout(
                barmode='group',
                title=f"استبيان {resp_name}: تطور مرحلة ({metric}) في ({domain})",
                xaxis_title="مرحلة التجربة",
                yaxis_title="النسبة (%)"
            )
            fig.show()

TypeError: 'Categorical' with dtype category does not support reduction 'mean'

### Decision making el

In [ ]:
decision_making_el.drop(columns=['Ew_rights_sdi', 'Ew_rights_di'],axis=1, inplace=True)
decision_making_el.drop(columns=['Ew_equality_sdi', 'Ew_equality_di'], axis=1,inplace=True)
decision_making_el.drop(columns=['Ew_obey_sdi', 'Ew_obey_di'], axis=1,inplace=True)
decision_making_el.drop(columns=['Ew_powhh_sdi', 'Ew_powhh_di'], axis=1,inplace=True)
decision_making_el.drop(columns=['Ew_avoid_preg_sdi', 'Ew_avoid_preg_di'],axis=1, inplace=True)
decision_making_el.drop(columns=['Ew_oblig_sex_sdi', 'Ew_oblig_sex_di'],axis=1, inplace=True)
decision_making_el.drop(columns=['Ew_tol_viol_sdi', 'Ew_tol_viol_di'],axis=1, inplace=True)
decision_making_el.drop(columns=['Ew_hit_goesout_sdi', 'Ew_hit_goesout_di'],axis=1, inplace=True)
decision_making_el.drop(columns=['Ew_hit_argues_sdi', 'Ew_hit_argues_di'],axis=1, inplace=True)
decision_making_el.drop(columns=['Ew_hit_food_sdi', 'Ew_hit_food_di'],axis=1, inplace=True)
decision_making_el.drop(columns=['Ew_boss_sdi', 'Ew_boss_di'],axis=1, inplace=True)
decision_making_el.drop(columns=['Ew_leaders_sdi', 'Ew_leaders_di'],axis=1, inplace=True)
decision_making_el.drop(columns=['Ew_choose_sag', 'Ew_choose_ag'], axis=1,inplace=True)
decision_making_el.drop(columns=['Ew_shrpow_sag', 'Ew_shrpow_ag'], axis=1,inplace=True)
decision_making_el.drop(columns=['Ew_womfre_sag', 'Ew_womfre_ag'], axis=1,inplace=True)
decision_making_el.drop(columns=['Ew_opin_sag', 'Ew_opin_ag'], axis=1,inplace=True)
decision_making_el.drop(columns=['Ew_final_say_sag', 'Ew_final_say_ag'], axis=1,inplace=True)

parent_df.drop(columns=['Ew_rights_sdi', 'Ew_rights_di'],axis=1, inplace=True)
parent_df.drop(columns=['Ew_equality_sdi', 'Ew_equality_di'], axis=1,inplace=True)
parent_df.drop(columns=['Ew_obey_sdi', 'Ew_obey_di'], axis=1,inplace=True)
parent_df.drop(columns=['Ew_powhh_sdi', 'Ew_powhh_di'], axis=1,inplace=True)
parent_df.drop(columns=['Ew_avoid_preg_sdi', 'Ew_avoid_preg_di'],axis=1, inplace=True)
parent_df.drop(columns=['Ew_oblig_sex_sdi', 'Ew_oblig_sex_di'],axis=1, inplace=True)
parent_df.drop(columns=['Ew_tol_viol_sdi', 'Ew_tol_viol_di'],axis=1, inplace=True)
parent_df.drop(columns=['Ew_hit_goesout_sdi', 'Ew_hit_goesout_di'],axis=1, inplace=True)
parent_df.drop(columns=['Ew_hit_argues_sdi', 'Ew_hit_argues_di'],axis=1, inplace=True)
parent_df.drop(columns=['Ew_hit_food_sdi', 'Ew_hit_food_di'],axis=1, inplace=True)
parent_df.drop(columns=['Ew_boss_sdi', 'Ew_boss_di'],axis=1, inplace=True)
parent_df.drop(columns=['Ew_leaders_sdi', 'Ew_leaders_di'],axis=1, inplace=True)
parent_df.drop(columns=['Ew_choose_sag', 'Ew_choose_ag'], axis=1,inplace=True)
parent_df.drop(columns=['Ew_shrpow_sag', 'Ew_shrpow_ag'], axis=1,inplace=True)
parent_df.drop(columns=['Ew_womfre_sag', 'Ew_womfre_ag'], axis=1,inplace=True)
parent_df.drop(columns=['Ew_opin_sag', 'Ew_opin_ag'], axis=1,inplace=True)
parent_df.drop(columns=['Ew_final_say_sag', 'Ew_final_say_ag'], axis=1,inplace=True)

In [ ]:
decision_making_el.drop(columns=['Ew_daily_bi','Ew_purch_bi','Ew_savespend_bi','Ew_wcloth_bi'], axis=1,inplace=True)
parent_df.drop(columns=['Ew_daily_bi','Ew_purch_bi','Ew_savespend_bi','Ew_wcloth_bi'],axis=1, inplace=True)

decision_making_el.drop(columns=['Ew_chealth_bi','Ew_cfeed_bi','Ew_cschool_bi','Ew_ccloth_bi','Ew_wmoney_bi'], axis=1,inplace=True)
parent_df.drop(columns=['Ew_chealth_bi','Ew_cfeed_bi','Ew_cschool_bi','Ew_ccloth_bi','Ew_wmoney_bi'],axis=1, inplace=True)

In [ ]:
decision_making_el.drop(columns=['Ew_all_times_no','Ew_notrust_mon_no'],axis=1,inplace=True)
parent_df.drop(columns=['Ew_all_times_no','Ew_notrust_mon_no'],axis=1,inplace=True)

In [ ]:
df = decision_making_el.copy()
df['Ew_rel_improve'] = df['Ew_rel_improve'].astype(str)

for i in range(1, 7):
    col_name = f'Ew_rel_improve{i}'
    expected_values = df['Ew_rel_improve'].str.contains(rf'\b{i}\b', regex=True, na=False).astype(int)
    actual_values = df[col_name].fillna(0).astype(int)
    mismatches = df[expected_values != actual_values]
    num_mismatches = len(mismatches)

    if num_mismatches == 0:
        print(f" col  {col_name}: is 100% the same")
    else:
        print(f" col {col_name}: has {num_mismatches} unequal rows")

In [ ]:
df = decision_making_el.copy()
df['Ew_rel_worse'] = df['Ew_rel_worse'].astype(str)

for i in range(1, 6):
    col_name = f'Ew_rel_worse{i}'
    expected_values = df['Ew_rel_worse'].str.contains(rf'\b{i}\b', regex=True, na=False).astype(int)
    actual_values = df[col_name].fillna(0).astype(int)
    mismatches = df[expected_values != actual_values]
    num_mismatches = len(mismatches)

    if num_mismatches == 0:
        print(f" col  {col_name}: is 100% the same")
    else:
        print(f" col {col_name}: has {num_mismatches} unequal rows")

In [ ]:
decision_making_el.drop(columns=['Ew_rel_worse','Ew_rel_improve'],axis=1,inplace=True)
parent_df.drop(columns=['Ew_rel_worse','Ew_rel_improve'],axis=1,inplace=True)

In [ ]:
women_improve_rename = {
    'Ew_rel_improve1': 'Ew_improve_husb_listens',
    'Ew_rel_improve2': 'Ew_improve_wife_listens',
    'Ew_rel_improve3': 'Ew_improve_fewer_args',
    'Ew_rel_improve4': 'Ew_improve_husb_involved',
    'Ew_rel_improve5': 'Ew_improve_share_finances',
    'Ew_rel_improve6': 'Ew_improve_share_info'
}

women_worse_rename = {
    'Ew_rel_worse1': 'Ew_worse_husb_listens_less',
    'Ew_rel_worse2': 'Ew_worse_wife_listens_less',
    'Ew_rel_worse3': 'Ew_worse_more_args',
    'Ew_rel_worse4': 'Ew_worse_husb_less_engaged',
    'Ew_rel_worse5': 'Ew_worse_husb_refuses_fin'
}

parent_df.rename(columns=women_improve_rename, inplace=True)
parent_df.rename(columns=women_worse_rename, inplace=True)
decision_making_el.rename(columns=women_improve_rename, inplace=True)
decision_making_el.rename(columns=women_worse_rename, inplace=True)

In [ ]:
decision_making_el.describe()

In [ ]:
import plotly.graph_objects as go

improve_cols = [
    'Ew_improve_husb_listens',
    'Ew_improve_wife_listens',
    'Ew_improve_fewer_args',
    'Ew_improve_husb_involved',
    'Ew_improve_share_finances',
    'Ew_improve_share_info'
]

sum_wemp = parent_df[parent_df['wemp'] == 1][improve_cols].sum()
sum_whn = parent_df[parent_df['whn'] == 1][improve_cols].sum()
sum_mhn = parent_df[parent_df['mhn'] == 1][improve_cols].sum()
sum_control = parent_df[(parent_df['wemp'] == 0) & (parent_df['whn'] == 0) & (parent_df['mhn'] == 0)][improve_cols].sum()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=improve_cols,
    y=sum_wemp,
    name='WEMP=1',
    marker_color='rgb(66, 133, 244)'
))

fig.add_trace(go.Bar(
    x=improve_cols,
    y=sum_whn,
    name='WHN=1',
    marker_color='rgb(244, 180, 26)'
))

fig.add_trace(go.Bar(
    x=improve_cols,
    y=sum_mhn,
    name='MHN=1',
    marker_color='rgb(15, 157, 88)'
))

fig.add_trace(go.Bar(
    x=improve_cols,
    y=sum_control,
    name='Control (0,0,0)',
    marker_color='rgb(219, 68, 85)'
))

fig.update_layout(
    barmode='group',
    title="تكرار نواحي التحسن في العلاقة الزوجية حسب المجموعات التدريبية",
    xaxis_title="مجالات التحسن",
    yaxis_title="عدد الحالات",
    template='plotly_white'
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd
df = parent_df.copy()
group_conditions = {
    'مجموعة التحكم (لا تدريب)': (df['mhn'] != 1) & (df['whn'] != 1) & (df['wemp'] != 1),
    'تدريب الرجال (MHN)': df['mhn'] == 1,
    'تدريب النساء (WHN)': df['whn'] == 1,
    'تمكين النساء (WEMP)': df['wemp'] == 1
}

prefixes = ['Ew_chealth', 'Ew_daily', 'Ew_cfeed', 'Ew_wmoney']
x_labels = ['صحة الأطفال', 'القرارات اليومية', 'إطعام الأطفال', 'أموال الزوجة']
col_suffix = '_prevail'

all_categories = set()
for prefix in prefixes:
    col_name = f'{prefix}{col_suffix}'
    if col_name in df.columns:
        all_categories.update(df[col_name].dropna().unique())

categories = list(all_categories)
fig = go.Figure()

for group_name, condition in group_conditions.items():
    df_sub = df[condition]
    for cat in categories:
        y_values = []
        for prefix in prefixes:
            col_name = f'{prefix}{col_suffix}'
            if col_name in df_sub.columns:
                # حساب النسبة المئوية لكل فئة
                val = (df_sub[col_name] == cat).mean() * 100
                y_values.append(val)
            else:
                y_values.append(0)

        if sum(y_values) > 0:
            fig.add_trace(go.Bar(
                name=f"{group_name} ({cat})",
                x=x_labels,
                y=y_values
            ))
fig.update_layout(
    barmode='group',
    title="توزيع 'من يفرض رأيه' (Prevail) بحسب التدريب ومجال القرار بالنسبة للنساء",
    xaxis_title="مجال اتخاذ القرار",
    yaxis_title="النسبة (%)",
    legend_title="المجموعة - الطرف المقرر",

    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)

fig.show()


 1. الاعتراف المشترك بالسيطرة الذكورية

الاستنتاج الأهم هنا هو أن النساء يتفقن مع الرجال على أن الزوج هو الطرف المهيمن في اتخاذ القرارات الأساسية، وتحديداً في القرارات اليومية وفي كيفية إنفاق أموال الزوجة.

 الفرضية الأساسية للمشروع: البيئة المدروسة هي بيئة أبوية تفتقر فيها النساء إلى السلطة التفاوضية، حتى فيما يخص الأموال التي يجنينها بأنفسهن.

2. فجوة حجم السيطرة

على الرغم من اتفاق الطرفين على أن الرجل هو المهيمن، إلا أن هناك اختلافاً جوهرياً في تقدير حجم هذه الهيمنة:

* الرجال يبالغون في تقدير تفردهم بالقرار: في القرارات اليومية مثلاً، يدعي الرجال سيطرة مطلقة وشبه كاسحة
* النساء يرين واقعاً أكثر تقارباً: النساء يعترفن بسيطرة الزوج ، لكن بنسب أقل حدة ، مع إعطاء مساحة أكبر بكثير لوجود تسويات مشتركة أو لقراراتهن الفردية مقارنة بما يصرح به الرجال. هذا يدل على أن النساء يحاولن ممارسة أو إدراك استقلالية لا يعترف بها الرجال.


3. الأطفال هم المساحة الآمنة الوحيدة

المكان الوحيد الذي تتراجع فيه سيطرة الزوج بشكل واضح ليحل محلها القرار المشترك  لدى النساء هو مجالا صحة الأطفال و إطعام الأطفال.


### women decision makeing (hiba)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


#### رأي الزوج في الفئة الأكثر استفادة من الجلسات التدريبية


In [ ]:
# w_husb


#### تحليل قرار المرأة في الاحتياجات اليومية

هل تشارك الزوجة؟ هل تعبر عن رأيها؟ وإذا اختلفت مع زوجها، من ينتصر؟


In [ ]:
daily_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}

prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}

parent_df['w_daily_who_label'] = parent_df['w_daily_who'].map(daily_who_map).astype('category')
parent_df['w_daily_prevail_label'] = parent_df['w_daily_prevail'].map(prevail_map).astype('category')

In [ ]:

print(" نسبة مشاركة الزوجة في قرار الاحتياجات اليومية")
print(parent_df['w_daily_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟")
print(parent_df['w_daily_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_daily_diffop', 'w_daily_express', 'w_daily_argue']].mean() * 100
print("\n عملية الاختلاف  ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_daily_prevail_label'].value_counts(normalize=True) * 100)


In [ ]:
fig = px.pie(parent_df, names='w_daily_who_label',
             title='توزيع صاحب القرار  بالاحتياجات اليومية',
             hole=0.4,
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

In [ ]:
prevail_counts = parent_df['w_daily_prevail_label'].value_counts().reset_index()

fig = px.bar(prevail_counts, x='w_daily_prevail_label', y='count',
             title='من ينتصر في الجدال من أجل الاحتياجات اليومية؟',
             labels={'w_daily_prevail_label': ' who', 'count': 'couont '},
             color='w_daily_prevail_label')
fig.show()


In [ ]:
data = {
    'action': ['الاختلاف بالرأي', 'التعبير عن الرأي', 'حدوث جدال', 'الانتصار بالجدال'],
    'percent': [39.1, 66.1, 64.4, 28.5]
}
df_funnel = pd.DataFrame(data)

fig = px.funnel(df_funnel, x='percent', y='action', title='كيف تتفاعل المرأة  ؟ ')
fig.show()


#### تحليل قرار المرأة بشأن المشتريات الكبيرة

هل تشارك الزوجة؟ هل تعبر عن رأيها؟ وإذا اختلفت مع زوجها، من ينتصر؟


In [ ]:
purch_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}

prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}

parent_df['w_purch_who_label'] = parent_df['w_purch_who'].map(purch_who_map).astype('category')
parent_df['w_purch_prevail_label'] = parent_df['w_purch_prevail'].map(prevail_map).astype('category')



In [ ]:

print("نسبة مشاركة الزوجة في قرار المشتريات الكبيرة ")
print(parent_df['w_purch_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟ ")
print(parent_df['w_purch_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_purch_diffop', 'w_purch_express', 'w_purch_argue']].mean() * 100
print("\n عملية الاختلاف  ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_purch_prevail_label'].value_counts(normalize=True) * 100)



In [ ]:

fig = px.pie(parent_df, names='w_purch_who_label',
             title='توزيع صاحب القرار في المشتريات الكبيرة',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()


In [ ]:

prevail_counts = parent_df['w_purch_prevail_label'].value_counts().reset_index()
fig = px.bar(prevail_counts, x='w_purch_prevail_label', y='count',
             title='من ينتصر في الجدال بشأن المشتريات الكبيرة؟',
             labels={'w_daily_prevail_label': ' who', 'count': 'couont '},
             color='w_purch_prevail_label')
fig.show()

In [ ]:
data = {
    'action': ['الاختلاف بالرأي', 'التعبير عن الرأي', 'حدوث جدال', 'الانتصار بالجدال'],
    'percent': [35.5, 66.1, 57.0, 27.2]
}
df_funnel = pd.DataFrame(data)

fig = px.funnel(df_funnel, x='percent', y='action',
                title='كيف تتفاعل المرأة  ؟ ')
fig.show()

#### تحليل قرار المرأة بقرار الادخار أو الإنفاق

هل تشارك الزوجة؟ هل تعبر عن رأيها؟ وإذا اختلفت مع زوجها، من ينتصر؟


In [ ]:
savespend_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}

prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}
parent_df['w_savespend_who_label'] = parent_df['w_savespend_who'].map(savespend_who_map).astype('category')
parent_df['w_savespend_prevail_label'] = parent_df['w_savespend_prevail'].map(prevail_map).astype('category')


In [ ]:

print(" نسبة مشاركة الزوجة في قرار الادخار أو الإنفاق")
print(parent_df['w_savespend_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟ ")
print(parent_df['w_savespend_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_savespend_diffop', 'w_savespend_express', 'w_savespend_argue']].mean() * 100
print("\n عملية الاختلاف  ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_savespend_prevail_label'].value_counts(normalize=True) * 100)


In [ ]:

fig = px.pie(parent_df, names='w_savespend_who_label',
             title='توزيع صاحب القرار بشأن الادخار والإنفاق',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

In [ ]:

prevail_counts = parent_df['w_savespend_prevail_label'].value_counts().reset_index()
fig = px.bar(prevail_counts, x='w_savespend_prevail_label', y='count',
             title='من ينتصر في الجدال بشأن الادخار والإنفاق؟',
             labels={'w_savespend_prevail_label': 'who ', 'count': 'count'},
             color='w_savespend_prevail_label')
fig.show()


In [ ]:
data = {
    'action': ['الاختلاف بالرأي', 'التعبير عن الرأي', 'حدوث جدال', 'الانتصار بالجدال'],
    'percent': [46.8, 71.0, 61.8, 21.9]
}
df_funnel = pd.DataFrame(data)

fig = px.funnel(df_funnel, x='percent', y='action',
                title='كيف تتفاعل المرأة في قرار الادخار والإنفاق؟')
fig.update_layout(template='plotly_white')
fig.show()


#### تحليل قرار المرأة في شراء ملابسها

هل تشارك الزوجة؟ هل تعبر عن رأيها؟ وإذا اختلفت مع زوجها، من ينتصر؟


In [ ]:
wcloth_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}
prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}

parent_df['w_wcloth_who_label'] = parent_df['w_wcloth_who'].map(wcloth_who_map).astype('category')
parent_df['w_wcloth_prevail_label'] = parent_df['w_wcloth_prevail'].map(prevail_map).astype('category')


In [ ]:
print(" نسبة مشاركة الزوجة في قرار شراء ملابسها ")
print(parent_df['w_wcloth_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟ ")
print(parent_df['w_wcloth_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_wcloth_diffop', 'w_wcloth_express', 'w_wcloth_argue']].mean() * 100
print("\n عملية الاختلاف ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_wcloth_prevail_label'].value_counts(normalize=True) * 100)


In [ ]:
fig = px.pie(parent_df, names='w_wcloth_who_label',
             title='توزيع صاحب القرار بشأن شراء ملابس الزوجة',
             hole=0.4,
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()


In [ ]:
prevail_counts = parent_df['w_wcloth_prevail_label'].value_counts().reset_index()
fig = px.bar(prevail_counts, x='w_wcloth_prevail_label', y='count',
             title='من ينتصر في الجدال بشأن شراء ملابس الزوجة؟',
             labels={'w_wcloth_prevail_label': 'who ', 'count': 'count'},
             color='w_wcloth_prevail_label')
fig.show()

In [ ]:
data = {
    'action': ['الاختلاف بالرأي', 'التعبير عن الرأي', 'حدوث جدال', 'الانتصار بالجدال'],
    'percent': [28.7, 33.1, 55.8, 40.9]
}
df_funnel = pd.DataFrame(data)

fig = px.funnel(df_funnel, x='percent', y='action',
                title='كيف تتفاعل المرأة في قرار شراء ملابسها؟')
fig.update_layout(template='plotly_white')
fig.show()


#### تحليل قرار المرأة في تكاليف صحة الأطفال

هل تشارك الزوجة؟ هل تعبر عن رأيها؟ وإذا اختلفت مع زوجها، من ينتصر؟


In [ ]:
chealth_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}

prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}

parent_df['w_chealth_who_label'] = parent_df['w_chealth_who'].map(chealth_who_map).astype('category')
parent_df['w_chealth_prevail_label'] = parent_df['w_chealth_prevail'].map(prevail_map).astype('category')

In [ ]:
print(" نسبة مشاركة الزوجة بقرار تكاليف صحة الأطفال")
print(parent_df['w_chealth_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟ ")
print(parent_df['w_chealth_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_chealth_diffop', 'w_chealth_express', 'w_chealth_argue']].mean() * 100
print("\n عملية الاختلاف  ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_chealth_prevail_label'].value_counts(normalize=True) * 100)

In [ ]:
fig = px.pie(parent_df, names='w_chealth_who_label',
             title='توزيع صاحب القرار بشأن تكاليف صحة الأطفال',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

In [ ]:
prevail_counts = parent_df['w_chealth_prevail_label'].value_counts().reset_index()
fig = px.bar(prevail_counts, x='w_chealth_prevail_label', y='count',
             title='من ينتصر في الجدال بشأن تكاليف صحة الأطفال؟',
             labels={'w_chealth_prevail_label': 'who ', 'count': 'count'},
             color='w_chealth_prevail_label')
fig.show()

In [ ]:
data = {
    'action': ['الاختلاف بالرأي', 'التعبير عن الرأي', 'حدوث جدال', 'الانتصار بالجدال'],
    'percent': [13.8, 49.3, 68.9, 41.4]
}
df_funnel = pd.DataFrame(data)

fig = px.funnel(df_funnel, x='percent', y='action',
                title='كيف تتفاعل المرأة في قرار تكاليف صحة الأطفال؟')
fig.update_layout(template='plotly_white')
fig.show()


#### تحليل قرار المرأة  بقرار تغذية الأطفال

هل تشارك الزوجة؟ هل تعبر عن رأيها؟ وإذا اختلفت مع زوجها، من ينتصر؟


In [ ]:
cfeed_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}

prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}

parent_df['w_cfeed_who_label'] = parent_df['w_cfeed_who'].map(cfeed_who_map).astype('category')
parent_df['w_cfeed_prevail_label'] = parent_df['w_cfeed_prevail'].map(prevail_map).astype('category')

In [ ]:
print(" نسبة مشاركة الزوجة بقرار تغذية الأطفال")
print(parent_df['w_cfeed_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟ ")
print(parent_df['w_cfeed_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_cfeed_diffop', 'w_cfeed_express', 'w_cfeed_argue']].mean() * 100
print("\n عملية الاختلاف  ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_cfeed_prevail_label'].value_counts(normalize=True) * 100)

In [ ]:
fig = px.pie(parent_df, names='w_cfeed_who_label',
             title='توزيع صاحب القرار بشأن تغذية الأطفال',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

In [ ]:
prevail_counts = parent_df['w_cfeed_prevail_label'].value_counts().reset_index()
fig = px.bar(prevail_counts, x='w_cfeed_prevail_label', y='count',
             title='من ينتصر في الجدال بشأن تغذية الأطفال؟',
             labels={'w_cfeed_prevail_label': 'who ', 'count': 'count'},
             color='w_cfeed_prevail_label')
fig.show()

In [ ]:
data = {
    'action': ['الاختلاف بالرأي', 'التعبير عن الرأي', 'حدوث جدال', 'الانتصار بالجدال'],
    'percent': [16.8, 11.8, 45.4, 56.1]
}
df_funnel = pd.DataFrame(data)

fig = px.funnel(df_funnel, x='percent', y='action',
                title='كيف تتفاعل المرأة في قرار تغذية الأطفال؟')
fig.update_layout(template='plotly_white')
fig.show()


#### تحليل قرار المرأة  بقرار تعليم الأطفال

هل تشارك الزوجة؟ هل تعبر عن رأيها؟ وإذا اختلفت مع زوجها، من ينتصر؟


In [ ]:
ceduc_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}

prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}

parent_df['w_ceduc_who_label'] = parent_df['w_ceduc_who'].map(ceduc_who_map).astype('category')
parent_df['w_ceduc_prevail_label'] = parent_df['w_ceduc_prevail'].map(prevail_map).astype('category')

In [ ]:
print(" نسبة مشاركة الزوجة بقرار تعليم الأطفال")
print(parent_df['w_ceduc_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟ ")
print(parent_df['w_ceduc_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_ceduc_diffop', 'w_ceduc_express', 'w_ceduc_argue']].mean() * 100
print("\n عملية الاختلاف  ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_ceduc_prevail_label'].value_counts(normalize=True) * 100)

In [ ]:
fig = px.pie(parent_df, names='w_ceduc_who_label',
             title='توزيع صاحب القرار بشأن تعليم الأطفال',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

In [ ]:
prevail_counts = parent_df['w_ceduc_prevail_label'].value_counts().reset_index()
fig = px.bar(prevail_counts, x='w_ceduc_prevail_label', y='count',
             title='من ينتصر في الجدال بشأن تعليم الأطفال؟',
             labels={'w_ceduc_prevail_label': 'who ', 'count': 'count'},
             color='w_ceduc_prevail_label')
fig.show()

In [ ]:
data = {
    'action': ['الاختلاف بالرأي', 'التعبير عن الرأي', 'حدوث جدال', 'الانتصار بالجدال'],
    'percent': [19.4, 57.2, 58.8, 36.3]
}
df_funnel = pd.DataFrame(data)

fig = px.funnel(df_funnel, x='percent', y='action',
                title='كيف تتفاعل المرأة في قرار تعليم الأطفال؟')
fig.update_layout(template='plotly_white')
fig.show()


#### تحليل قرار المرأة  بقرار نفقات مدارس الأطفال

هل تشارك الزوجة؟ هل تعبر عن رأيها؟ وإذا اختلفت مع زوجها، من ينتصر؟


In [ ]:
cschool_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}

prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}

parent_df['w_cschool_who_label'] = parent_df['w_cschool_who'].map(cschool_who_map).astype('category')
parent_df['w_cschool_prevail_label'] = parent_df['w_cschool_prevail'].map(prevail_map).astype('category')

In [ ]:
print(" نسبة مشاركة الزوجة بقرار نفقات المدارس")
print(parent_df['w_cschool_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟ ")
print(parent_df['w_cschool_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_cschool_diffop', 'w_cschool_express', 'w_cschool_argue']].mean() * 100
print("\n عملية الاختلاف  ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_cschool_prevail_label'].value_counts(normalize=True) * 100)

In [ ]:
fig = px.pie(parent_df, names='w_cschool_who_label',
             title='توزيع صاحب القرار بشأن نفقات المدارس',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

In [ ]:
prevail_counts = parent_df['w_cschool_prevail_label'].value_counts().reset_index()
fig = px.bar(prevail_counts, x='w_cschool_prevail_label', y='count',
             title='من ينتصر في الجدال بشأن نفقات المدارس؟',
             labels={'w_cschool_prevail_label': 'who ', 'count': 'count'},
             color='w_cschool_prevail_label')
fig.show()

In [ ]:
data = {
    'action': ['الاختلاف بالرأي', 'التعبير عن الرأي', 'حدوث جدال', 'الانتصار بالجدال'],
    'percent': [9.9, 43.0, 70.1, 44.3]
}
df_funnel = pd.DataFrame(data)

fig = px.funnel(df_funnel, x='percent', y='action',
                title='كيف تتفاعل المرأة في قرار نفقات المدارس؟')
fig.update_layout(template='plotly_white')
fig.show()


#### تحليل قرار المرأة  بقرار ملابس الأطفال

هل تشارك الزوجة؟ هل تعبر عن رأيها؟ وإذا اختلفت مع زوجها، من ينتصر؟


In [ ]:
ccloth_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}

prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}

parent_df['w_ccloth_who_label'] = parent_df['w_ccloth_who'].map(ccloth_who_map).astype('category')
parent_df['w_ccloth_prevail_label'] = parent_df['w_ccloth_prevail'].map(prevail_map).astype('category')

In [ ]:
print(" نسبة مشاركة الزوجة بقرار ملابس الأطفال")
print(parent_df['w_ccloth_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟ ")
print(parent_df['w_ccloth_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_ccloth_diffop', 'w_ccloth_express', 'w_ccloth_argue']].mean() * 100
print("\n عملية الاختلاف  ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_ccloth_prevail_label'].value_counts(normalize=True) * 100)

In [ ]:
fig = px.pie(parent_df, names='w_ccloth_who_label',
             title='توزيع صاحب القرار بشأن ملابس الأطفال',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

In [ ]:
prevail_counts = parent_df['w_ccloth_prevail_label'].value_counts().reset_index()
fig = px.bar(prevail_counts, x='w_ccloth_prevail_label', y='count',
             title='من ينتصر في الجدال بشأن ملابس الأطفال؟',
             labels={'w_ccloth_prevail_label': 'who ', 'count': 'count'},
             color='w_ccloth_prevail_label')
fig.show()

In [ ]:
data = {
    'action': ['الاختلاف بالرأي', 'التعبير عن الرأي', 'حدوث جدال', 'الانتصار بالجدال'],
    'percent': [17.3, 33.5, 58.3, 44.6] }
df_funnel = pd.DataFrame(data)

fig = px.funnel(df_funnel, x='percent', y='action',
                title='كيف تتفاعل المرأة في قرار ملابس الأطفال؟')
fig.update_layout(template='plotly_white')
fig.show()


#### تحليل قرار المرأة  بقرار إنفاق أموالها

هل تشارك الزوجة؟ هل تعبر عن رأيها؟ وإذا اختلفت مع زوجها، من ينتصر؟


In [ ]:
wmoney_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}

prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}

parent_df['w_wmoney_who_label'] = parent_df['w_wmoney_who'].map(wmoney_who_map).astype('category')
parent_df['w_wmoney_prevail_label'] = parent_df['w_wmoney_prevail'].map(prevail_map).astype('category')

In [ ]:
print(" نسبة مشاركة الزوجة بقرار إنفاق أموالها")
print(parent_df['w_wmoney_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟ ")
print(parent_df['w_wmoney_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_wmoney_diffop', 'w_wmoney_express', 'w_wmoney_argue']].mean() * 100
print("\n عملية الاختلاف  ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_wmoney_prevail_label'].value_counts(normalize=True) * 100)

In [ ]:
fig = px.pie(parent_df, names='w_wmoney_who_label',
             title='توزيع صاحب القرار بشأن إنفاق أموال الزوجة',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

In [ ]:
prevail_counts = parent_df['w_wmoney_prevail_label'].value_counts().reset_index()
fig = px.bar(prevail_counts, x='w_wmoney_prevail_label', y='count',
             title='من ينتصر في الجدال بشأن إنفاق أموال الزوجة؟',
             labels={'w_wmoney_prevail_label': 'who ', 'count': 'count'},
             color='w_wmoney_prevail_label')
fig.show()

In [ ]:
data = {
    'action': ['الاختلاف بالرأي', 'التعبير عن الرأي', 'حدوث جدال', 'الانتصار بالجدال'],
    'percent': [36.9, 32.5, 54.4, 57.6] }
df_funnel = pd.DataFrame(data)

fig = px.funnel(df_funnel, x='percent', y='action',
                title='كيف تتفاعل المرأة في قرار إنفاق أموالها؟')
fig.update_layout(template='plotly_white')
fig.show()

#### تحليل نسبة مشاركة المرأة باتخاذ القرار بشكل عام

In [ ]:
decision_cols = {
    'الاحتياجات اليومية': 'w_daily_who',
    'المشتريات الكبيرة': 'w_purch_who',
    'الادخار والإنفاق': 'w_savespend_who',
    'ملابس الزوجة': 'w_wcloth_who',
    'صحة الأطفال': 'w_chealth_who',
    'تغذية الأطفال': 'w_cfeed_who',
    'تعليم الأطفال': 'w_ceduc_who',
    'نفقات المدارس': 'w_cschool_who',
    'ملابس الأطفال': 'w_ccloth_who',
    'أموال الزوجة': 'w_wmoney_who'}

results = []
for arabic_name, col in decision_cols.items():
    if col in parent_df.columns:
        counts = parent_df[col].dropna().astype(int).value_counts(normalize=True) * 100
        results.append({'القرار': arabic_name, 'صاحب القرار': 'الزوجة', 'النسبة': counts.get(1, 0)})
        results.append({'القرار': arabic_name, 'صاحب القرار': 'الزوج', 'النسبة': counts.get(2, 0)})
        results.append({'القرار': arabic_name, 'صاحب القرار': 'مشترك', 'النسبة': counts.get(3, 0)})

df_summary = pd.DataFrame(results)

wife_order = df_summary[df_summary['صاحب القرار'] == 'الزوجة'].sort_values(by='النسبة')
sorted_decisions = wife_order['القرار'].tolist()

fig2 = px.bar(df_summary, x='القرار', y='النسبة', color='صاحب القرار', barmode='group',
              title='مقارنة صناعة القرار: الزوج مقابل الزوجة في جميع المجالات',
              labels={'النسبة': 'النسبة المئوية (%)', 'القرار': 'نوع القرار'},
              color_discrete_map={'الزوجة': '#FF9999', 'الزوج': '#99CCFF', 'مشترك': '#99FF99'})

fig2.update_xaxes(categoryorder='array', categoryarray=sorted_decisions)
fig2.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig2.show()


### تحليل مفهوم "السلطة و القيادة"  بين الزوجة و الزوج

In [ ]:
print("ترفض ان حقوقها تسلب حقوق الرجل (غير موافق):", parent_df['w_rights_di'].mean() * 100)
print("ترفض طاعة الزوج العمياء (غير موافق):", parent_df['w_obey_di'].mean() * 100)
print("ترفض ان الرجل هو الرئيس (غير موافق):", parent_df['w_boss_di'].mean() * 100)
print("ترفض ان القيادة للرجال فقط (غير موافق):", parent_df['w_leaders_di'].mean() * 100)
print("تؤيد تقاسم السلطة (موافق):", parent_df['w_shrpow_ag'].mean() * 100)
print("تؤيد ان رأيها قيّم (موافق):", parent_df['w_opin_ag'].mean() * 100)

In [ ]:
attitude_data = {
    'الحالة': [
        'حقوق المرأة تسلب الرجل',
        'الرجل هو الرئيس',
        'القيادة للرجال فقط',
        'طاعة الزوج العمياء',
        'تقاسم السلطة ممكن',
        'رأي المرأة قيم'],

    'النسبة (%)': [
        parent_df['w_rights_di'].mean() * 100,
        parent_df['w_boss_di'].mean() * 100,
        parent_df['w_leaders_di'].mean() * 100,
        parent_df['w_obey_di'].mean() * 100,
        parent_df['w_shrpow_ag'].mean() * 100,
        parent_df['w_opin_ag'].mean() * 100         ],
    'نوع الموقف': [
       'رفض الافكار السلبية','رفض الافكار السلبية','رفض الافكار السلبية','رفض الافكار السلبية',
        'تأييد الافكار التشاركية', 'تأييد الافكار التشاركية' ]
}

df_attitudes = pd.DataFrame(attitude_data)

fig = px.bar(df_attitudes, x='الحالة', y='النسبة (%)', color='نوع الموقف',
             title='  : نسبة الوعي لدى النساء بحقوقها وسلطتها',
             text_auto='.1f',
             color_discrete_map={'رفض الأفكار السلبية': '#FF9999', 'تأييد الأفكار التشاركية': '#99CCFF'})

fig.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig.show()



<div dir="rtl">


 هنا يوجد نوع من التناقض عند النساء اذ انهن يوافقن بنسبة كبيرة على المشاركة مع الرجل الا ان معظمهن يؤيدن فكرة ان الرجل هو الرئيس و عليهن بالطاعة العمياء !!



</div>

### تحليل مدى الاستقلالية الجسدية والإنجابية لدى الزوجة

In [ ]:
print(f"ترفض ان تجنب الحمل مسؤوليتها وحدها (غير موافق): {(parent_df['w_avoid_preg_di'].mean() * 100).round(2)}")
print(f"تؤيد ان القول الفصل لها بتنظيم الأسرة (موافق): {(parent_df['w_final_say_ag'].mean() * 100).round(2)}")
print(f"ترفض اجبارها على العلاقة الزوجية (غير موافق): {(parent_df['w_oblig_sex_di'].mean() * 100).round(2)}")

In [ ]:
autonomy_data = {
    'الحالة': [
        'رفض مسؤولية منع الحمل وحدها',
        'الرأي لها بتنظيم الاسرة',
        'رفض الاجبار على العلاقة الزوجية'
    ],
    'النسبة (%)': [
        parent_df['w_avoid_preg_di'].mean() * 100,
        parent_df['w_final_say_ag'].mean() * 100,
        parent_df['w_oblig_sex_di'].mean() * 100
    ]
}
df_autonomy = pd.DataFrame(autonomy_data)

fig3 = px.bar(df_autonomy, x='الحالة', y='النسبة (%)',
             title='مستوى الاستقلالية الجسدية والإنجابية لدى النساء',
             text_auto='.1f',
             color_discrete_sequence=['#FF9999'])

fig3.update_layout(xaxis_tickangle=-15, template='plotly_white')
fig3.show()



<div dir="rtl">


معظم النساء يرفضن الاجبار على العلاقة ويؤيدن التدخل و ابداء الرأي بشأن تنظيم الاسرة الا ان معظمهن يعتبرن انفسهن المسؤولات عن منع الحمل

### تحليل مفهوم "تبرير العنف" لدى الزوجة

In [ ]:
print(f"ترفض تحمل العنف للحفاظ على الاسرة (غير موافق): {(parent_df['w_tol_viol_di'].mean() * 100).round(2)}")
print(f"ترفض فكرة ان المرأة تستحق الضرب احيانا (غير موافق): {(parent_df['w_deser_di'].mean() * 100).round(2)}")
print(f"ترفض تبرير الضرب إذا خرجت دون اذن (غير موافق): {(parent_df['w_hit_goesout_di'].mean() * 100).round(2)}")
print(f"ترفض تبرير الضرب اذا جادلته (غير موافق): {(parent_df['w_hit_argues_di'].mean() * 100).round(2)}")
print(f"ترفض تبرير الضرب بسبب التأخر بالطعام (غير موافق): {(parent_df['w_hit_food_di'].mean() * 100).round(2)}")

In [ ]:
justification_data = {
    'الحالة': [
        'تحمل العنف لأجل الاسؤة',
        'المرأة تستحق الضرب احيانا',
        'الضرب بسبب الخروج دون اذن',
        'الضرب بسبب الجدال',
        'الضرب بسبب التأخر بالطعام'
    ],
    'نسبة الرفض (%)': [
        parent_df['w_tol_viol_di'].mean() * 100,
        parent_df['w_deser_di'].mean() * 100,
        parent_df['w_hit_goesout_di'].mean() * 100,
        parent_df['w_hit_argues_di'].mean() * 100,
        parent_df['w_hit_food_di'].mean() * 100
    ]
}
df_justification = pd.DataFrame(justification_data)

df_justification = df_justification.sort_values(by='نسبة الرفض (%)', ascending=False)

fig4 = px.bar(df_justification, x='الحالة', y='نسبة الرفض (%)',
             title='نسبة النساء اللواتي يرفضن تبرير العنف الموجه ضدهن',
             text_auto='.1f',
             color_discrete_sequence=['#FF9999'])

fig4.update_layout(xaxis_tickangle=-25, template='plotly_white')
fig4.show()



<div dir="rtl">


تحليل غريب بشكل خاص اذا ان النساء يرفضن تحمل العنف من اجل الاسرة او الواجبات المنزلية (  يعرفن حقوقهن( ولكن يقبلن التعنيف بسبب التمرد على طاعة الرجل !!

اي ان المرأة تعرف حقوقها لكن لا تسطيع الخروج عن سلطة الزوج

</div>

### تحليل مدى تحكم الزوج بالزوجة


In [ ]:
print(f"تجاهل رأيها (لا يستمع): {((1 - parent_df['w_husb_listens_bi']).mean() * 100).round(2)}")
print(f"يغضب عند الاختلاف معه: {(parent_df['w_husb_angry_bi'].mean() * 100).round(2)}")
print(f"يصر على معرفة مكانها دائما: {(parent_df['w_all_times'].mean() * 100).round(2)}")
print(f"لا يثق بها في الاموال: {(parent_df['w_notrust_mon'].mean() * 100).round(2)}")
print(f"يتهمها بالخيانة: {(parent_df['w_unfaith'].mean() * 100).round(2)}")
print(f"يقيد اتصالها باسرتها: {(parent_df['w_fam_contact'].mean() * 100).round(2)}")
print(f"يمنعها من مقابلة صديقاتها: {(parent_df['w_fem_friends'].mean() * 100).round(2)}")


In [ ]:

control_data = {
    'السلوك': [
        'تجاهل رأيها (لا يستمع)',
        'يغضب عند الاختلاف',
        'المراقبة الدائمة (مكانها)',
        'انعدام الثقة المالية',
        'الاتهام بالخيانة',
        'تقييد الاتصال بالاسرة',
        'المنع من الصديقات'
    ],
    'النسبة (%)': [
        (1 - parent_df['w_husb_listens_bi']).mean() * 100,
        parent_df['w_husb_angry_bi'].mean() * 100,
        parent_df['w_all_times'].mean() * 100,
        parent_df['w_notrust_mon'].mean() * 100,
        parent_df['w_unfaith'].mean() * 100,
        parent_df['w_fam_contact'].mean() * 100,
        parent_df['w_fem_friends'].mean() * 100
    ]
}
df_control = pd.DataFrame(control_data)
df_control = df_control.sort_values(by='النسبة (%)', ascending=False)

fig5 = px.bar(df_control, x='السلوك', y='النسبة (%)',
             title='نسبة تعرض النساء للسلوك التحكمي من قبل الزوج',
             text_auto='.1f',
             color_discrete_sequence=['#99CCFF'])

fig5.update_layout(xaxis_tickangle=-25, template='plotly_white')
fig5.show()



<div dir="rtl">


تحليل صادم لقدرة الزوج على التحكم بزوجته بأبسط الامور (حرمانها ابسط حقوقها بالتواصل البشري!)

 الا انه لا يشعر بالغضب عند الخلاف معها (لانه اصلا لا يستمع لها و لا يأخد برأيها)

### تحليل مدى التعرض الفعلي للعنف الجسدي واللفظي عند الزوجة

In [ ]:
print(f"التعرض للعنف اللفظي : {(parent_df['w_vio_verb'].mean() * 100).round(2)}")
print(f"التعرض للعنف الجسدي : {(parent_df['w_vio_phy'].mean() * 100).round(2)}")
print(f"تعرضت للإهانة: {(parent_df['w_insult'].mean() * 100).round(2)}")
print(f"تعرضت للإذلال أمام الآخرين: {(parent_df['w_humil'].mean() * 100).round(2)}")
print(f"تعرضت للصفع: {(parent_df['w_slap'].mean() * 100).round(2)}")
print(f"تعرضت للضرب: {(parent_df['w_beat'].mean() * 100).round(2)}")


In [ ]:
violence_data = {
    'نوع العنف': [
        'عنف لفظي',
        'عنف جسدي ',
        'الإهانة',
        'الإذلال أمام الآخرين',
        'الصفع',
        'الضرب'
    ],
    'النسبة (%)': [
        parent_df['w_vio_verb'].mean() * 100,
        parent_df['w_vio_phy'].mean() * 100,
        parent_df['w_insult'].mean() * 100,
        parent_df['w_humil'].mean() * 100,
        parent_df['w_slap'].mean() * 100,
        parent_df['w_beat'].mean() * 100
    ]
}

df_violence = pd.DataFrame(violence_data)
df_violence = df_violence.sort_values(by='النسبة (%)', ascending=False)

fig6 = px.bar(df_violence, x='نوع العنف', y='النسبة (%)',
             title='نسبة النساء اللواتي تعرضن للعنف اللفظي والجسدي',
             text_auto='.1f',
             color_discrete_sequence=['#E63946'])

fig6.update_layout(xaxis_tickangle=-25, template='plotly_white')
fig6.show()



<div dir="rtl">

يوضح المخطط مدى سهولة التعنيف اللفظي و الاهانة من القبل الرجل للمرأة بدون استثناء النسبة الجيدة من الاذاء الجسدي ايضا


<div dir="rtl">


## ملخص شامل لوضع المرأة في مجتمع الدراسة

- يوجد حجب كبير عن دور المرأة في اتخاذا القرارات سواء كانت متعلقة بطفلها او حتى بأمورها الشخصية (ملابس المرأة (

- معظم النساء على دراية كبيرة بحقوقهن لكن لا يطالبن بها اذا كانت على حساب السطلة الذكورية الزوجية

- يوجد وعي نسبي لدى النساء تجاه الاستقلالية الجسدية و لكن ليس الانجابية !

- معظم النساء رافضة للتبرير العنف الا في حال مخالفة سلطة الزوج

- المرأة في مجتمع الدراسة تفقتر لابسط حقوقها اليومية و البشرية كالتواصل مع الاهل و لصديقات بالاضافة لرزوحها التامة تحت المراقبة وعدم الاستماع لها

- الارقام الصادمة هي نسبة تعرض للمرأة لكافة انواع الاذى الجسدي

## عزلة اجتماعية + سلب القدرة على ابداء الرأي و اتخاذ القرات + تعنيف جسدي




<div dir="rtl">





</div>

### women EXPENDITURE


#### مشاركة الزوجة بالتخطيط المادي

In [ ]:
partnership_rate = parent_df['w_plan_hhexpend6'].mean() * 100

print(f"نسبة النساء اللتي تناقش زوجها حول نفقات المنزل: {partnership_rate.round(2)}")

#### تحليل ممتلكات الاسرة المادية (اثاث , مواشي , رفاهية)

In [ ]:
owns = [
    'w_cattle_own', 'w_goat_own', 'w_chx_own', 'w_stove_own',
    'w_wattnk_own', 'w_bed_own', 'w_lant_own', 'w_tv_own',
    'w_radio_own', 'w_gen_own', 'w_cell_own', 'w_bic_own',
    'w_moto_own', 'w_car_own'
]
stats = parent_df[owns].describe()
print(stats)

<div dir="rtl">

مجتمع الدراسة يعيش في فقر شديد يفتقر لاقل الحاجات الاساسية

- بالنسبة للمواشي فهي موجودة ولكن بنسبة بسيطة (ذاتية فقط(
- بالنسبة للكهربائيات فيوجد تفاوت طبقي كبير جدا حيث ان اغلب 50% قيمها 0 بينما الmax  ارقام جيدة و المتوسط قليل جدا
- اي يوجد تفاوت طبقي واضح ومحتكر على عدد قليل

In [ ]:
import plotly.express as px

mean_assets = parent_df[owns].mean().reset_index()
mean_assets.columns = ['الملكية', 'متوسط الامتلاك']

mean_assets = mean_assets.sort_values(by='متوسط الامتلاك', ascending=False)

fig = px.bar(mean_assets, x='الملكية', y='متوسط الامتلاك',
             title='متوسط الممتلكات لدى العائلات',
             text_auto='.2f',
             color_discrete_sequence=['#4682B4'])

fig.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig.show()

####  تحليل الاحتياجات اليومية من الغذاء
السؤال كان كالتالي : في اخر 7 ايام كم انفق زوجك على ما يلي

In [ ]:
import numpy as np
import plotly.express as px

daily_needs = [
    'w_sug_own', 'w_par_own', 'w_soap_own', 'w_tea_own',
    'w_milk_own', 'w_mtfs_own', 'w_salt_own', 'w_transp_own', 'w_air_own'
]

df_clean_needs = parent_df[daily_needs].replace([-88, -99], np.nan)

stats_needs = df_clean_needs.describe()
print(stats_needs)


In [ ]:

mean_needs = df_clean_needs.mean().reset_index()
mean_needs.columns = ['الاحتياج اليومي', 'متوسط الإنفاق (آخر 7 أيام)']

mean_needs = mean_needs.sort_values(by='متوسط الإنفاق (آخر 7 أيام)', ascending=False)

fig_needs = px.bar(mean_needs, x='الاحتياج اليومي', y='متوسط الإنفاق (آخر 7 أيام)',
             title='متوسط إنفاق الأسرة على الاحتياجات اليومية والغذائية (في آخر 7 أيام)',
             text_auto='.2f',
             color_discrete_sequence=['#2CA02C'])

fig_needs.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig_needs.show()

<div dir="rtl">

ملاحظة : هذه الارقام هي المبالغ المادية التي تدفعها الاسرة بالاسبوع مقاسة بالعملة الاوغندية

- الأسرة تصرف في المتوسط حوالي 3782  أسبوعياً على المواصلات

- وتصرف 2190 على اللحوم والأسماك

- وتصرف 1987 على الصابون

In [ ]:
major_expenses = [
    'w_last6m_rice', 'w_last6m_ccloth', 'w_last6m_fun',
    'w_last6m_health', 'w_last6m_schfee', 'w_last6m_supp',
    'w_last6m_wedd', 'w_last6m_wcloth'
]

df_clean_major = parent_df[major_expenses].replace([-88, -99], np.nan)

mean_major = df_clean_major.mean().reset_index()
mean_major.columns = ['نوع النفقة', 'متوسط الإنفاق (آخر 6 أشهر)']
mean_major = mean_major.sort_values(by='متوسط الإنفاق (آخر 6 أشهر)', ascending=False)

fig_major = px.bar(mean_major, x='نوع النفقة', y='متوسط الإنفاق (آخر 6 أشهر)',
             title='متوسط النفقات الكبرى للأسرة (خلال آخر 6 أشهر)',
             text_auto='.2f',
             color_discrete_sequence=['#6EADBC'])

fig_major.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig_major.show()

<div dir="rtl">

من الواضح من المخطط ان العائلة ملتزمة بالتزامات عديدة اكبرها الرسوم الدراسية تليها الاحتياجات الصحية
بالاضافة لبعض الاعباء المتعلقة بالزواج وغيرها


### women another servey


#### تحليل النحافة ومؤشر كتلة الجسم



In [ ]:

bmi_cols = ['w_bmi_18h', 'w_bmi_17', 'w_bmi_16']

bmi_percentages = parent_df[bmi_cols].mean() * 100

df_bmi = pd.DataFrame({
    'مستوى النحافة': [
        'نقص وزن عام (BMI < 18.5)',
        'نقص وزن متوسط (BMI < 17)',
        'نقص وزن حاد (BMI < 16)'
    ],
    'النسبة المئوية %': bmi_percentages.values
})

df_bmi = df_bmi.sort_values(by='النسبة المئوية %', ascending=False)

print(" نسبة النحافة ونقص الوزن عند النساء ")
for index, row in df_bmi.iterrows():
    print(f"{row['مستوى النحافة']}: {row['النسبة المئوية %']:.2f}%")


In [ ]:
fig_bmi = px.bar(df_bmi, x='مستوى النحافة', y='النسبة المئوية %',
             title='نسبة النساء اللواتي يعانين من نقص الوزن وسوء التغذية',
             text_auto='.2f',
             color_discrete_sequence=['#F4A261'])

fig_bmi.update_layout(template='plotly_white')
fig_bmi.show()

<div dir="rtl">

5.7 % من النساء ققط تعاني من نقص وزن عام ونسبة قليلة جدا تعاني من النقص الحاد

اذ برغم المستوى المعيشي المتدني اوزان النساء ليست صغيرة

In [ ]:
import pandas as pd
import plotly.express as px

# 1. تحويل الوزن إلى الكيلوغرام الطبيعي (قسمة القيم على 10)
parent_df['weight_kg'] = parent_df['w_weight'] / 10

fig_weight = px.histogram(parent_df, x='weight_kg',
                         nbins=30,
                         title='توزيع أوزان النساء في مجتمع الدراسة (بالنسبة المئوية)',
                         labels={'weight_kg': 'الوزن (كغ)'},
                         color_discrete_sequence=['#9C27B0'])

fig_weight.update_traces(histnorm='percent', texttemplate='%{y:.1f}%', textposition='outside')
fig_weight.update_layout(yaxis_title='النسبة المئوية (%)', template='plotly_white')

fig_weight.show()

#### تحليل  وفقر الدم

In [ ]:
anemia_cols = ['w_anaemia_12', 'w_anaemia_11', 'w_anaemia_8']
anemia_percentages = parent_df[anemia_cols].mean() * 100
healthy_percentage = 100 - anemia_percentages['w_anaemia_12']
df_anemia = pd.DataFrame({
    'مستوى فقر الدم': [
        'طبيعي (لا يوجد فقر دم)',
        'فقر دم عام (Hb < 12)',
        'فقر دم متوسط (Hb < 11)',
        'فقر دم حاد (Hb < 8)'
    ],
    'النسبة المئوية (%)': [
        healthy_percentage,
        anemia_percentages['w_anaemia_12'],
        anemia_percentages['w_anaemia_11'],
        anemia_percentages['w_anaemia_8']
    ]
})

for index, row in df_anemia.iterrows():
    print(f"{row['مستوى فقر الدم']}: {row['النسبة المئوية (%)']:.2f}")

In [ ]:
fig_anemia = px.bar(df_anemia,
             x='مستوى فقر الدم',
             y='النسبة المئوية (%)',
             title='نسبة النساء اللواتي يعانين من فقر الدم مقارنة بالسليمات',
             text_auto='.2f',
             color='مستوى فقر الدم',
             color_discrete_sequence=['#2CA02C', '#D51C39', '#B71C1C', '#7F0000'])

fig_anemia.update_layout(template='plotly_white', showlegend=False)
fig_anemia.show()

#### MUAC تحليل  سوء التغذية الحاد

In [ ]:
muac_cols = ['w_muac_preg23', 'w_muac_221', 'w_muac_214']
muac_percentages = parent_df[muac_cols].mean() * 100
healthy_percentage = 100 - muac_percentages.max()

df_muac = pd.DataFrame({
    'مستوى سوء التغذية (MUAC)': [
        'طبيعي (لا يوجد سوء تغذية)',
        'سوء تغذية للأم الحامل (< 23 سم)',
        'سوء تغذية متوسط (< 22.1 سم)',
        'سوء تغذية حاد (< 21.4 سم)'
    ],
    'النسبة المئوية (%)': [
        healthy_percentage,
        muac_percentages['w_muac_preg23'],
        muac_percentages['w_muac_221'],
        muac_percentages['w_muac_214']
    ]
})

for index, row in df_muac.iterrows():
    print(f"{row['مستوى سوء التغذية (MUAC)']}: {row['النسبة المئوية (%)']:.2f}")


In [ ]:

fig_muac = px.bar(df_muac,
             x='مستوى سوء التغذية (MUAC)',
             y='النسبة المئوية (%)',
             title='نسبة النساء اللواتي يعانين من سوء التغذية (حسب محيط العضد MUAC)',
             text_auto='.2f',
             color='مستوى سوء التغذية (MUAC)',
             color_discrete_sequence=['#2CA02C', '#E9C46A', '#F4A261', '#D51C39'])

fig_muac.update_layout(template='plotly_white', showlegend=False)
fig_muac.show()

### men another servey


#### تحليل نوع التغذية العام عند الزوج

السؤال بالضبط :

 I am now going to ask you about foods and drinks that have been consumed by you  in the last 24 hours

In [ ]:
mens_food_cols = [
    'm_water', 'm_tea', 'm_otherbev', 'm_juice', 'm_milk',
    'm_matooke', 'm_cas', 'm_rice', 'm_beans', 'm_green',
    'm_pump', 'm_otherfru', 'm_oil', 'm_sweet', 'm_othersolid',
    'm_meat', 'm_organ', 'm_fish', 'm_egg'
]

food_arabic_names = {
    'm_water': 'ماء',
    'm_tea': 'شاي/قهوة',
    'm_otherbev': 'مشروبات أخرى',
    'm_juice': 'عصير الفواكه',
    'm_milk': 'حليب/لبن',
    'm_matooke': 'ماتوك ( اكلة اوغندية)',
    'm_cas': 'كاسافا',
    'm_rice': 'رز/خبز/معكرونة',
    'm_beans': 'فاصوليا/بقوليات',
    'm_green': 'الخضراوات',
    'm_pump': 'يقطين/جزر',
    'm_otherfru': 'فواكه أخرى',
    'm_oil': 'زيوتلدهون',
    'm_sweet': 'حلوبات/سكريات',
    'm_othersolid': 'أطعمة صلبة أخرى',
    'm_meat': 'اللحوم',
    'm_organ': 'أعضاء (كبد، قلب)',
    'm_fish': 'سمك',
    'm_egg': 'بيض'
}

food_percentages = parent_df[mens_food_cols].mean() * 100


In [ ]:
df_mens_food = pd.DataFrame({
    'رمز الصنف': food_percentages.index,
    'النسبة المئوية (%)': food_percentages.values
})

df_mens_food['الصنف الغذائي'] = df_mens_food['رمز الصنف'].map(food_arabic_names)
df_mens_food = df_mens_food.sort_values(by='النسبة المئوية (%)', ascending=False)

fig_food = px.bar(
    df_mens_food,
    x='الصنف الغذائي',
    y='النسبة المئوية (%)',
    title='التنوع الغذائي للرجال: نسبة استهلاك الأطعمة خلال 24 ساعة',
    text_auto='.1f',
    color='النسبة المئوية (%)',
    color_continuous_scale='Viridis'
)

fig_food.update_layout(
    xaxis_tickangle=-45,
    template='plotly_white',
)

fig_food.show()

<div dir="rtl">
لا يعتبر نوع طعام الزوج جيد اذا ان نسبة اللحوم"البروتين" منخفضة  بالاضافة الى ندرة البيض و السمك

لكن يجب الملاجظة ان هذه الاسئلة كانت لاخر 24 ساعة



#### تحليل عدد الاولاد ومتوسط قضاء الزوج الوقت معهم


In [ ]:
kids_under_2_percent = parent_df['m_kidsunder2'].mean() * 100
print(f"نسبة الآباء الذين لديهم طفل تحت السنتين في المنزل: {kids_under_2_percent:.2f}%")

mean_hours = parent_df['m_timekids'].mean()
print(f"متوسط عدد ساعات قضاء الآباء وقتهم مع أطفالهم: {mean_hours:.2f} ساعة")

#### تحليل تعامل الرجل مع زوجته خلال الحمل و الولادة


In [ ]:
preg_now_pct = parent_df['m_preg_now'].mean() * 100
comp_pct = parent_df['m_preg_comp'].mean() * 100
labr_present_pct = parent_df['m_labr_present'].mean() * 100
print(f"نسبة الزوجات الحوامل حالياً في العينة: {preg_now_pct:.2f}")
print(f"نسبة النساء اللواتي واجهن مضاعفات في الولادة الأخيرة: {comp_pct:.2f}")
print(f"نسبة الرجال الذين تواجدوا/ساعدوا زوجاتهم أثناء المخاض: {labr_present_pct:.2f}")


In [ ]:
#  تحليل كيف قام الزوج بمساعدة زوجته خلال الحمل والولادة
non_null_count = parent_df['m_assist'].notna().sum()
print(non_null_count)

if non_null_count > 0:
    print(parent_df['m_assist'].value_counts().head(10))


In [ ]:

all_assists_text = []
for row in parent_df['m_assist'].dropna():
    items = str(row).split(';')
    all_assists_text.extend([item.strip() for item in items])

assist_counts = pd.Series(all_assists_text).value_counts().reset_index()
assist_counts.columns = ['نوع المساعدة', 'التكرار']

translate_map = {
    'Home chores': 'القيام بالأعمال المنزلية',
    'Took you to clinic': 'أخذها للمركز الصحي',
    'Was present at clicinc': 'التواجد معها في المركز',
    'Arranged transport to clinic': 'ترتيب وسيلة نقل للمركز',
    'Had TBA help at home': 'إحضار قابلة (داية) للمنزل',
    'Gave $ to go to health clini': 'إعطاء مال للمركز الصحي',
    'Other': 'أخرى'
}

assist_counts['نوع المساعدة'] = assist_counts['نوع المساعدة'].map(lambda x: translate_map.get(x, x))
top_assists = assist_counts.head(6)

fig_assist = px.bar(
    top_assists,
    x='نوع المساعدة',
    y='التكرار',
    title='أنواع الدعم الفعلي الذي قدمه الرجل لزوجته أثناء الولادة',
    text_auto=True,
    color='التكرار',
    color_continuous_scale='Teal'
)

fig_assist.update_layout(
    xaxis_tickangle=-45,
    template='plotly_white',
    coloraxis_showscale=False,
    yaxis_title="عدد مرات التكرار"
)
fig_assist.show()

<div dir="rtl">

من المخطط نلاحظ نوع من دعم الزوج اثناء الحمل و الولادة

المخطط نوعا ما كسر نمطية ان معظم الرجال تقدم دعم مادي فقط


 , ملاحظة : كان الخيار متعدد

#### family planning تحليل من المسؤول عن تنظيم عدد الاسرة

In [ ]:
fp_decision_map = {
    1: 'قرار الزوج غالباً',
    2: 'قرار الزوجة غالباً',
    3: 'قرار مشترك'
}

df_fp = parent_df['m_fp_decide'].dropna().map(fp_decision_map).value_counts().reset_index()
df_fp.columns = ['صاحب القرار', 'عدد الحالات']

fig_fp = px.pie(
    df_fp,
    values='عدد الحالات',
    names='صاحب القرار',
    title='fmaily planning ؟ من يتخذ قرار التحكم بعدد الأطفال ',
    color='صاحب القرار',
    color_discrete_map={
        'قرار مشترك': '#2A9D8F',
        'قرار الزوج غالباً': '#E63946',
        'قرار الزوجة غالباً': '#F4A261'
    }
)
fig_fp.update_traces(textinfo='percent+label')
fig_fp.update_layout(template='plotly_white')
fig_fp.show()


<div dir="rtl">
تحليل ايجابي !

نسبة وعي جيدة لدى الوزج في حق المرأة بتقرير طريقة تنظيم الاسرة يدل على نسبة وعي لحقوق المرأة الانجابية







#### نسبة وعي الرجل بعدد الاولاد المناسب و تفضيله للذكور/الاناث

In [ ]:
mean_total_kids = parent_df['m_hus_ideal_fert_t'].mean()
print(f"المتوسط العام لعدد الأطفال  الذي يفضله الرجل : {mean_total_kids:.2f}")


In [ ]:

def get_preference_percentages(col_name):
    counts = parent_df[col_name].value_counts(normalize=True) * 100

    grouped = {
        'لا يريد (0)': counts.get(0, 0),
        'طفل واحد (1)': counts.get(1, 0),
        'طفلان (2)': counts.get(2, 0),
        '3 أطفال': counts.get(3, 0),
        '4 أطفال فما فوق': counts[counts.index >= 4].sum()
    }
    return grouped

boys_pref = get_preference_percentages('m_hus_ideal_fert_b')
girls_pref = get_preference_percentages('m_hus_ideal_fert_g')

categories = list(boys_pref.keys())
fig_gender = go.Figure(data=[
    go.Bar(name='ذكور', x=categories, y=list(boys_pref.values()), marker_color='#89D4FF', text=[f"{v:.1f}%" for v in boys_pref.values()], textposition='auto'),
    go.Bar(name='إناث', x=categories, y=list(girls_pref.values()), marker_color='#F9B2D7', text=[f"{v:.1f}%" for v in girls_pref.values()], textposition='auto')
])

fig_gender.update_layout(
    title='تفضيل جنس المولود: كم طفلاً يريد الرجل من كل جنس؟',
    xaxis_title='العدد المطلوب من الأطفال',
    yaxis_title='النسبة المئوية من الرجال (%)',
    barmode='group',
    template='plotly_white',
    legend_title='الجنس'
)

fig_gender.show()

<div dir="rtl">

يكشف المخطط امرين : الاول : لا يوجد تفرقة او تحيز تجاه جنس الطفل
و الثاني : معظم الازواج يفضلون بالمتوسط 4 اطفال بالاسرة

فيبقى السؤال للتحليل هل ها العدد يتناسب مع الوضع المادي للاسرة و الدخل الشهري؟

#### تحليل وعي الزوج بالنظافة الشخصية

In [ ]:

mean_soap_day = parent_df['m_soap_day'].mean()
print(f"متوسط غسل اليدين بالصابون يومياً : {mean_soap_day:.1f} مرات\n")


In [ ]:

soap_toilet_pct = parent_df['m_soap_toilet_bi'].mean() * 100
soap_meal_pct = parent_df['m_soap_meal_bi'].mean() * 100

df_hygiene = pd.DataFrame({
    'الحالة': ['استخدام الصابون بعد المرحاض', 'استخدام الصابون قبل الأكل'],
    'النسبة المئوية (%)': [soap_toilet_pct, soap_meal_pct]
})

fig_hygiene = px.bar(
    df_hygiene,
    x='الحالة',
    y='النسبة المئوية (%)',
    title='مقارنة الوعي الصحي: استخدام الصابون بعد المرحاض مقابل قبل تناول الطعام',
    text_auto='.1f',
    color='الحالة',
    color_discrete_map={
        'استخدام الصابون بعد المرحاض': '#2A9D8F',
        'استخدام الصابون قبل الأكل': '#F4A261'
    }
)

fig_hygiene.update_layout(
    template='plotly_white',
    yaxis_title='النسبة المئوية من الرجال (%)',
    xaxis_title='',
    showlegend=False
)

fig_hygiene.show()

<div dir="rtl">
لما نقول إنو 58% فقط بيستخدمو الصابون بعد المرحاض هاد معناه انو عننا 42% من الرجال (نسبة صغيرة ) بيطلعوا من المرحاض وما بيعقمو ايديهن!

وهاد ممكن يكون واحد من اسباب انتشار الامراض

وفكرة انه غسل اليدين قبل الطعام اكبر من غسله بعد المرحاض دليل انو غسل اليدين هون مستخرج بناء على عادات مجتمعية و ليس وعي عام بأهمية النضافة !


#### تحليل مدى وعي الرجل بالامراض و الخرافات المرتبطة بخا

In [ ]:
disease_awareness = {
    'المرض يساهم بشكل كبير في تقزم الطفل': parent_df['m_grow_sick_vm'].dropna().mean() * 100,
    'إنكار أن السحر يسبب توقف نمو الطفل': parent_df['m_grow_witch_lt'].dropna().mean() * 100,
    'سوء النظافة يؤثر سلباً على ذكاء الطفل': parent_df['m_hygint_bi'].dropna().mean() * 100,
    'العمر الصحيح لتدريب الطفل على المرحاض': parent_df['m_childtoilet_bi'].dropna().mean() * 100,
    'غلي الماء لدقيقة واحدة يكفي لتعقيمه': parent_df['m_boil_1min_bi'].dropna().mean() * 100
}

df_disease = pd.DataFrame(list(disease_awareness.items()), columns=['المؤشر الطبي', 'نسبة الوعي الصحيح (%)'])

df_disease = df_disease.sort_values(by='نسبة الوعي الصحيح (%)', ascending=True)

fig_disease = px.bar(
    df_disease,
    x='نسبة الوعي الصحيح (%)',
    y='المؤشر الطبي',
    orientation='h',
    title='مؤشرات الوعي الطبي للآباء: بين الطب والخرافة',
    text_auto='.1f',
    color='نسبة الوعي الصحيح (%)',
    color_continuous_scale='Reds'
)

fig_disease.update_layout(
    template='plotly_white',
    xaxis_title='نسبة الرجال الذين أجابوا إجابة طبية صحيحة (%)',
    yaxis_title='',
    coloraxis_showscale=False
)

fig_disease.show()

#### تحليل وعي الرجل بغذاء المرأة و الطفل

In [ ]:
nutrition_awareness = {
    'إعطاء اللبأ (الحليب الام) للمولود': parent_df['m_cstrum_feed_bi'].dropna().mean() * 100,
    'حليب الأم أفضل للطفل من حليب البقر': parent_df['m_cowmilk_bi'].dropna().mean() * 100,
    'النساء يحتجن لأكل البروتين الحيواني كالرجال': parent_df['m_menprot_bi'].dropna().mean() * 100,
    'البدء بإدخال الطعام الصلب في عمر 6 أشهر تماماً': parent_df['m_agefood_6'].dropna().mean() * 100,
    'عصيدة الفطام يجب أن تكون كثيفة ومغذية': parent_df['m_porr_thin_bi'].dropna().mean() * 100,
    'الطفل يحتاج للدهون والزيوت أثناء الفطام': parent_df['m_weanfat_bi'].dropna().mean() * 100,
    'الطفل المصاب بالإسهال يحتاج طعاماً أكثر (وليس أقل)': parent_df['m_diarrfood_bi'].dropna().mean() * 100
}

df_nutrition = pd.DataFrame(list(nutrition_awareness.items()), columns=['المؤشر الغذائي', 'نسبة الوعي الصحيح (%)'])
df_nutrition = df_nutrition.sort_values(by='نسبة الوعي الصحيح (%)', ascending=True)

fig_nutrition = px.bar(
    df_nutrition,
    x='نسبة الوعي الصحيح (%)',
    y='المؤشر الغذائي',
    orientation='h',
    title='مؤشرات الوعي الغذائي للآباء: بين المعرفة الطبية والعادات الخاطئة',
    text_auto='.1f',
    color='نسبة الوعي الصحيح (%)',
    color_continuous_scale='Oranges'
)

fig_nutrition.update_layout(
    template='plotly_white',
    xaxis_title='نسبة الرجال الذين يمتلكون وعياً غذائياً صحيحاً (%)',
    yaxis_title='',
    coloraxis_showscale=False
)

fig_nutrition.show()

#### تحليل بعض العوامل الاخرى

In [ ]:
remaining_indicators = {
    'نقص الغذاء المتوازن يسبب التقزم': parent_df['m_grow_lackdiet_sw'].dropna().mean() * 100,
    'تغيير النظام الغذائي هو علاج الكواشيوركور': parent_df['m_kwash_treat_bi'].dropna().mean() * 100,
    'الديدان يجب علاجها دورياً (كل 6 أشهر إلى سنة)': (parent_df['m_timedewor_3'].dropna().mean() + parent_df['m_timedewor_4'].dropna().mean()) * 100,
    'الفواكه مهمة للرجال البالغين (رفض خرافة أنها للأطفال فقط)': parent_df['m_menfruit_bi'].dropna().mean() * 100
}

df_remaining = pd.DataFrame(list(remaining_indicators.items()), columns=['المؤشر', 'نسبة الوعي الصحيح (%)'])
df_remaining = df_remaining.sort_values(by='نسبة الوعي الصحيح (%)', ascending=True)

fig_rem = px.bar(
    df_remaining,
    x='نسبة الوعي الصحيح (%)',
    y='المؤشر',
    orientation='h',
    title='مؤشرات الوعي المتقدم: الكواشيوركور، الطفيليات، والخرافات الغذائية',
    text_auto='.1f',
    color='نسبة الوعي الصحيح (%)',
    color_continuous_scale='Purples'
)

fig_rem.update_layout(
    template='plotly_white',
    xaxis_title='النسبة المئوية للإجابات الصحيحة طبياً (%)',
    yaxis_title='',
    coloraxis_showscale=False
)

fig_rem.show()

## Children section

### children BASELINE ANTHRO  

<div dir="rtl">

في  قسم الطفل يوجد فيم كبيرة مفقودة اغلبه 75 % يجب معرفة السبب قبل تجاهل هذه القيم او تعويضها

- هل تم فحص اطفال معينين :

اي تم الاخذ بحالة مرضية

تم الاخذ بفئة عمرية محدة

الخ

In [ ]:
measured_kids = children_df[children_df['c_weight'].notna()]
unmeasured_kids = children_df[children_df['c_weight'].isna()]

print(f"أكبر طفل تم أخذ وزنه الفعلي يبلغ من العمر: {measured_kids['age'].max()} شهراً")
print(f"أصغر طفل تم تجاهل وزنه (بين الفئة الأكبر سناً) يبلغ: {unmeasured_kids[unmeasured_kids['age'] > 60]['age'].min()} شهراً")

<div dir="rtl">

اذا القياسات في ها القسم كانت مخخصة للاطفال بين عمر 2ونصف تقريبا لعمر 5 سنوات تقريبت

وبالتالي يمكن تجاهل القيم المفقودة بدون معالجة

#### تحليل تحليل التقزم لدى الاطفال

In [ ]:

stunting_rate = children_df['lhfa_neg2'].dropna().mean() * 100
normal_rate = 100 - stunting_rate

df_stunting = pd.DataFrame({
    'حالة النمو الإجمالي': ['نمو طبيعي', 'تقزم (متوسط إلى حاد)'],
    'النسبة المئوية (%)': [normal_rate, stunting_rate]
})

fig_stunting = px.pie(
    df_stunting,
    names='حالة النمو الإجمالي',
    values='النسبة المئوية (%)',
    title='مؤشر النمو المزمن: نسبة التقزم لدى الأطفال دون سن الخامسة (lhfa < -2)',
    color='حالة النمو الإجمالي',
    color_discrete_map={
        'نمو طبيعي': '#2A9D8F',
        'تقزم (متوسط إلى حاد)': '#E63946'
    },

)

fig_stunting.update_traces(textposition='inside', textinfo='percent+label', pull=[0, 0.1])
fig_stunting.update_layout(template='plotly_white')

fig_stunting.show()

### تحليل نقص الوزن عند الاطفال

In [ ]:
underweight_rate = children_df['wfa_neg2'].dropna().mean() * 100
wasting_rate = children_df['wfl_neg2'].dropna().mean() * 100

df_acute = pd.DataFrame({
    'نوع النقص (Z-score < -2)': ['نقص الوزن العام (Underweight)', 'الهزال الشديد (Wasting)'],
    'النسبة المئوية (%)': [underweight_rate, wasting_rate]
})

fig_acute = px.bar(
    df_acute,
    x='نوع النقص (Z-score < -2)',
    y='النسبة المئوية (%)',
    title='ركيزة النقص الحاد: مقارنة بين نقص الوزن والهزال لدى الأطفال',
    text_auto='.1f',
    color='نوع النقص (Z-score < -2)',
    color_discrete_map={
        'نقص الوزن العام (Underweight)': '#F4A261',
        'الهزال الشديد (Wasting)': '#E76F51'
    }
)

fig_acute.update_layout(
    template='plotly_white',
    yaxis_title='النسبة المئوية للأطفال (%)',
    xaxis_title='',
    showlegend=False
)

fig_acute.show()

#### SAM و MUAC  تحليل

In [ ]:

print("\n فحص التقزم:")
print(children_df.groupby('lhfa_neg2')['lhfa'].mean())
print("\n فحص نقص الوزن:")
print(children_df.groupby('wfa_neg2')['wfa'].mean())
print("\n فحص الهزال:")
print(children_df.groupby('wfl_neg2')['wfl'].mean())

In [ ]:
mam_rate = children_df['mfa_neg23'].dropna().mean() * 100
sam_rate = children_df['mfa_neg3'].dropna().mean() * 100

df_lethal = pd.DataFrame({
    'مؤشر الخطر الطبي': [
        'سوء تغذية متوسط (MAM) - MUAC',
        'سوء تغذية حاد شديد (SAM) - MUAC',
    ],
    'النسبة المئوية (%)': [mam_rate, sam_rate]
})

df_lethal = df_lethal.sort_values(by='النسبة المئوية (%)', ascending=True)

fig_lethal = px.bar(
    df_lethal,
    x='النسبة المئوية (%)',
    y='مؤشر الخطر الطبي',
    orientation='h',
    text_auto='.2f',
    color='مؤشر الخطر الطبي',
    color_discrete_map={
        'سوء تغذية متوسط (MAM) - MUAC': '#E9C46A',
        'سوء تغذية حاد شديد (SAM) - MUAC': '#E76F51',
    }
)

fig_lethal.update_layout(
    template='plotly_white',
    xaxis_title='النسبة المئوية من الأطفال المفحوصين (%)',
    yaxis_title='',
    showlegend=False
)

fig_lethal.show()

#### press skin تحليل

In [ ]:
print("عدد الأطفال في كل فئة:")
print(children_df['press_skin'].value_counts())
print("\nمتوسط القياسات الجسدية (Z-scores) لكل فئة:")
print(children_df.groupby('press_skin')[['mfa', 'wfa', 'wfl']].mean())

### Endline Anthro

In [ ]:

fig_weight = px.box(children_df, y="Ec_weight", points="all",
                    title="توزيع أوزان الأطفال بعد التدريب (Ec_weight)",
                    labels={"Ec_weight": "الوزن (كغ)"})
fig_weight.show()

fig_height = px.box(children_df, y="Ec_height", points="all",
                    title="توزيع أطوال الأطفال بعد التدريب (Ec_height)",
                    labels={"Ec_height": "الطول (سم)"})
fig_height.show()

In [ ]:
import pandas as pd
import plotly.express as px
df_plot = children_df[['wfa', 'Ewfa', 'female']].copy()
df_melted = df_plot.melt(id_vars=['female'],
                         value_vars=['wfa', 'Ewfa'],
                         var_name='Period',
                         value_name='Z_Score')

df_melted['Period'] = df_melted['Period'].map({'wfa': 'قبل التدريب (Baseline)',
                                               'Ewfa': 'بعد التدريب (Endline)'})
df_melted['Gender'] = df_melted['female'].map({0: 'ذكور (Boys)', 1: 'إناث (Girls)'})
fig1 = px.box(df_melted,
              x='Period',
              y='Z_Score',
              color='Gender',
              title='مقارنة مؤشر الوزن بالنسبة للعمر (WFA) قبل وبعد التدريب حسب الجنس',
              labels={'Z_Score': 'مؤشر الوزن للعمر (Z-Score)', 'Period': 'فترة القياس'},
              color_discrete_sequence=['#1f77b4', '#e377c2']) # أزرق للذكور، زهري للإناث

fig1.show()

In [ ]:
conditions = [
    children_df['wemp'] == 1,
    children_df['whn'] == 1,
    children_df['mhn'] == 1
]
choices = ['WEMP (تمكين)', 'WHN (صحة نساء)', 'MHN (صحة رجال)']
children_df['Treatment_Group'] = np.select(conditions, choices, default='Control (بدون تدريب)')

df_plot = children_df[['wfa', 'Ewfa', 'female', 'Treatment_Group']].copy()
df_melted = df_plot.melt(id_vars=['female', 'Treatment_Group'],
                         value_vars=['wfa', 'Ewfa'],
                         var_name='Period',
                         value_name='Z_Score')

df_melted['Period'] = df_melted['Period'].map({'wfa': 'قبل التدريب (Baseline)',
                                               'Ewfa': 'بعد التدريب (Endline)'})
df_boys = df_melted[df_melted['female'] == 0]
df_girls = df_melted[df_melted['female'] == 1]
fig_boys = px.box(df_boys,
                  x='Treatment_Group',
                  y='Z_Score',
                  color='Period',
                  title='تأثير أنواع التدريب على وزن [الذكور] (قبل وبعد التدريب)',
                  labels={'Z_Score': 'مؤشر الوزن للعمر (Z-Score)', 'Treatment_Group': 'مجموعة التجربة'},
                  category_orders={'Treatment_Group': ['Control (بدون تدريب)', 'WHN (صحة نساء)', 'MHN (صحة رجال)', 'WEMP (تمكين)']},
                  color_discrete_sequence=['#A6CEE3', '#1F78B4']) # ألوان متدرجة للذكور

fig_boys.show()

fig_girls = px.box(df_girls,
                   x='Treatment_Group',
                   y='Z_Score',
                   color='Period',
                   title='تأثير أنواع التدريب على وزن [الإناث] (قبل وبعد التدريب)',
                   labels={'Z_Score': 'مؤشر الوزن للعمر (Z-Score)', 'Treatment_Group': 'مجموعة التجربة'},
                   category_orders={'Treatment_Group': ['Control (بدون تدريب)', 'WHN (صحة نساء)', 'MHN (صحة رجال)', 'WEMP (تمكين)']},
                   color_discrete_sequence=['#FBB4AE', '#E31A1C'])

fig_girls.show()

In [ ]:
conditions = [
    children_df['wemp'] == 1,
    children_df['whn'] == 1,
    children_df['mhn'] == 1
]
choices = ['WEMP (تمكين)', 'WHN (صحة نساء)', 'MHN (صحة رجال)']
children_df['Treatment_Group'] = np.select(conditions, choices, default='Control (بدون تدريب)')

fig_sick = go.Figure()

fig_sick.add_trace(go.Box(
    x=children_df['Treatment_Group'],
    y=children_df['sick_days'],
    name='قبل التدريب (Baseline)',
    marker_color='#ff9999'
))

fig_sick.add_trace(go.Box(
    x=children_df['Treatment_Group'],
    y=children_df['Esick_days'],
    name='بعد التدريب (Endline)',
    marker_color='#66b3ff'
))

fig_sick.update_layout(
    title='تطور عدد أيام مرض الأطفال (قبل وبعد التدريب) حسب مجموعة التجربة',
    yaxis_title='عدد أيام المرض',
    xaxis_title='مجموعة التجربة',
    boxmode='group'
)

fig_sick.show()

In [ ]:
import plotly.graph_objects as go

# 1. تعريف الحالات الأربعة بشروط صارمة جداً (كما طلبتِ)
groups_info = [
    {
        'name': 'Control (بدون تدريب)',
        'condition': (children_df['mhn'] == 0) & (children_df['whn'] == 0) & (children_df['wemp'] == 0)
    },
    {
        'name': 'WEMP (تمكين)',
        'condition': (children_df['mhn'] == 0) & (children_df['whn'] == 0) & (children_df['wemp'] == 1)
    },
    {
        'name': 'WHN (صحة نساء)',
        'condition': (children_df['mhn'] == 0) & (children_df['whn'] == 1) & (children_df['wemp'] == 0)
    },
    {
        'name': 'MHN (صحة رجال)',
        'condition': (children_df['mhn'] == 1) & (children_df['whn'] == 0) & (children_df['wemp'] == 0)
    }
]

# 2. حلقة للمرور على كل حالة ورسمها بشكل مستقل
for group in groups_info:
    # فلترة البيانات بناءً على الشرط الصارم
    df_filtered = children_df[group['condition']]

    # التأكد من وجود بيانات لهذه المجموعة (لتجنب الأخطاء إذا كانت إحدى المجموعات فارغة)
    if df_filtered.empty:
        continue

    fig_sick = go.Figure()

    # صندوق "قبل التدريب"
    fig_sick.add_trace(go.Box(
        y=df_filtered['sick_days'],
        name='قبل التدريب (Baseline)',
        marker_color='#ff9999'
    ))

    # صندوق "بعد التدريب"
    fig_sick.add_trace(go.Box(
        y=df_filtered['Esick_days'],
        name='بعد التدريب (Endline)',
        marker_color='#66b3ff'
    ))

    # تنسيق الرسمة الحالية
    fig_sick.update_layout(
        title=f"تطور عدد أيام مرض الأطفال - مجموعة: {group['name']}",
        yaxis_title='عدد أيام المرض',
        boxmode='group'
    )

    # عرض الرسمة لهذه المجموعة
    fig_sick.show()

In [ ]:
children_df[children_df['Esick_days'].notnull() & children_df['sick_days'].notnull() &children_df['Ec_weight'].notnull() & children_df['c_weight'].notnull()]

# Question section

## 2- عدد الأزواج المشاركين في الدراسة وعدد اطفال كل عائلة

In [ ]:
num_couples = (parent_df['sample_couple'] == 1).sum()
#num_couples /= 2

print("Number of participating couples:", num_couples)

In [ ]:
children_per_family = (
    children_df.groupby('id')
               .size()
               .reset_index(name='num_children')
)

print(children_per_family.head())

In [ ]:
print("Number of families:", len(children_per_family))
print("Average children per family:",
      children_per_family['num_children'].mean())
print("Maximum children in a family:",
      children_per_family['num_children'].max())
print("Minimum children in a family:",
      children_per_family['num_children'].min())

## 9- هل فعلا كانت التجربة عشوائية ؟





الاعداد

In [ ]:
control_group = parent_df[(parent_df['mhn'] != 1) & (parent_df['whn'] != 1) & (parent_df['wemp'] != 1)]
mhn_group     = parent_df[parent_df['mhn'] == 1]
whn_group     = parent_df[parent_df['whn'] == 1]
wemp_group    = parent_df[parent_df['wemp'] == 1]

control_count = control_group.shape[0]
mhn_count     = mhn_group.shape[0]
whn_count     = whn_group.shape[0]
wemp_count    = wemp_group.shape[0]

print(f"مجموعة التحكم (لا تدريب): {control_count} عائلة")
print(f"تدريب الرجال (MHN): {mhn_count} عائلة")
print(f"تدريب النساء (WHN): {whn_count} عائلة")
print(f"تمكين النساء (WEMP): {wemp_count} عائلة")


🔍 تم العثور تلقائياً على عمود المجموعات الصحيح وهو: [wemp]
--- فحص المتوسطات لجميع مجموعات الدراسة (Balance Check) ---
🔹 المجموعة [0]:
   العدد: 4176 عائلة
   متوسط العمر: 27.8 سنة
   متوسط الإنفاق (الدخل): 10002.5 شلن
----------------------------------------
🔹 المجموعة [1]:
   العدد: 1329 عائلة
   متوسط العمر: 27.7 سنة
   متوسط الإنفاق (الدخل): 9988.9 شلن
----------------------------------------


متوسط الاعمار

In [ ]:
control_age_mean = control_group['w_age'].mean()
mhn_age_mean     = mhn_group['w_age'].mean()
whn_age_mean     = whn_group['w_age'].mean()
wemp_age_mean    = wemp_group['w_age'].mean()

print(f"متوسط العمر في مجموعة تدريب الرجال (MHN): {mhn_age_mean:.1f} ")
print(f"متوسط العمر في مجموعة تدريب النساء (WHN): {whn_age_mean:.1f} ")
print(f"متوسط العمر في مجموعة تمكين النساء (WEMP): {wemp_age_mean:.1f} ")


متوسط العمر في مجموعة تدريب الرجال (MHN): 27.7 
متوسط العمر في مجموعة تدريب النساء (WHN): 28.0 
متوسط العمر في مجموعة تمكين النساء (WEMP): 27.7 


متوسط مستوى التعليم

In [ ]:
control_edu_mean = control_group['w_educ'].mean()
mhn_edu_mean     = mhn_group['w_educ'].mean()
whn_edu_mean     = whn_group['w_educ'].mean()
wemp_edu_mean    = wemp_group['w_educ'].mean()

print(f"متوسط التعليم في مجموعة التحكم: {control_edu_mean:.1f} ")
print(f"متوسط التعليم في مجموعة تدريب الرجال (MHN): {mhn_edu_mean:.1f} ")
print(f"متوسط التعليم في مجموعة تدريب النساء (WHN): {whn_edu_mean:.1f} ")
print(f"متوسط التعليم في مجموعة تمكين النساء (WEMP): {wemp_edu_mean:.1f} ")

متوسط التعليم في مجموعة التحكم: 5.2 
متوسط التعليم في مجموعة تدريب الرجال (MHN): 5.2 
متوسط التعليم في مجموعة تدريب النساء (WHN): 5.2 
متوسط التعليم في مجموعة تمكين النساء (WEMP): 5.3 


In [ ]:
baseline_spend_cols = [col for col in parent_df.columns if col.startswith('w_ugx_')]
for col in baseline_spend_cols:
    parent_df[col] = pd.to_numeric(parent_df[col], errors='coerce')

parent_df['baseline_spend_proxy'] = parent_df[baseline_spend_cols].sum(axis=1)
print(f"مجموعة التحكم: {control_group['baseline_spend_proxy'].mean():.1f} ")
print(f"تدريب الرجال: {mhn_group['baseline_spend_proxy'].mean():.1f} ")
print(f"تدريب النساء: {whn_group['baseline_spend_proxy'].mean():.1f} ")
print(f"تمكين النساء: {wemp_group['baseline_spend_proxy'].mean():.1f} ")


مجموعة التحكم: 10006.8 
تدريب الرجال: 10004.0 
تدريب النساء: 9997.0 
تمكين النساء: 9989.0 


In [ ]:
fig_balance = make_subplots(
    rows=1, cols=4,
    subplot_titles=("التحكم", "تدريب رجال", "تدريب نساء", "تمكين نساء"),
    shared_yaxes=True
)

fig_balance.add_trace(go.Box(y=control_group['baseline_spend_proxy'], name="التحكم", fillcolor='#FF6B6B', line=dict(color='#C92A2A', width=2)), row=1, col=1)
fig_balance.add_trace(go.Box(y=mhn_group['baseline_spend_proxy'], name="تدريب رجال", fillcolor='#20C997', line=dict(color='#087F5B', width=2)), row=1, col=2)
fig_balance.add_trace(go.Box(y=whn_group['baseline_spend_proxy'], name="تدريب نساء", fillcolor='#4DABF7', line=dict(color='#1864AB', width=2)), row=1, col=3)
fig_balance.add_trace(go.Box(y=wemp_group['baseline_spend_proxy'], name="تمكين نساء", fillcolor='#FCC419', line=dict(color='#E67700', width=2)), row=1, col=4)

fig_balance.update_layout(
    title_text="فحص توازن الدخل (الإنفاق) لإثبات العشوائية",
    showlegend=False,
    font=dict(color="white"),
    hoverlabel=dict(font_color="white", bgcolor="#333333"),
    height=450,
    **layout_options
)

fig_balance.update_yaxes(range=[0, 30000], color="white", tickfont=dict(color="white"))
fig_balance.update_xaxes(color="white", tickfont=dict(color="white"))
fig_balance.update_annotations(font_color="white")

fig_balance.show()

الارقام شبه متطابقة و بالتالي نعم التجربة كانت عشوائية

## 10- مقارنة أوزان الأطفال لكل عائلة واستهلاك المنتجات الحيوانية قبل وبعد الدراسة

In [ ]:
animal_before = ['milk','meat','egg','fish','organ']
animal_after  = ['Emilk','Emeat','Eegg','Efish','Eorgan']

children_df['nutrition_before'] = children_df[animal_before].sum(axis=1)
children_df['nutrition_after']  = children_df[animal_after].sum(axis=1)

children_df['weight_before'] = children_df['c_weight']
children_df['weight_after']  = children_df['Ec_weight']

In [ ]:
analysis1 = children_df[[
    'nutrition_before','nutrition_after',
    'weight_before','weight_after'
]].dropna()

analysis1['improvement'] = (
    (analysis1['nutrition_after'] - analysis1['nutrition_before']) +
    (analysis1['weight_after'] - analysis1['weight_before'])
)

In [ ]:
from scipy.stats import ttest_rel

t_stat, p_value = ttest_rel(
    analysis1['nutrition_after'] + analysis1['weight_after'],
    analysis1['nutrition_before'] + analysis1['weight_before']
)

print(t_stat, p_value)

In [ ]:
alpha = 0.05

if p_value < alpha:
    print("Reject H0")
    print("Children weights and meat consumption improved after the training.")
else:
    print("Fail to Reject H0")
    print("No significant difference between weights and consumption before and afoer the training.")

In [ ]:
analysis2 = children_df[[
    'treat',
    'nutrition_before','nutrition_after',
    'weight_before','weight_after'
]].dropna()

analysis2['improvement'] = analysis1['improvement']

In [ ]:
from scipy.stats import f_oneway

g1 = analysis2[analysis2['treat']==1]['improvement']
g2 = analysis2[analysis2['treat']==2]['improvement']
g3 = analysis2[analysis2['treat']==3]['improvement']
g4 = analysis2[analysis2['treat']==4]['improvement']

alpha = 0.05

f_stat, p_value = f_oneway(g2, g1)
if p_value < alpha:
    print("Reject H0")
    print("Training method 2(WEMP) is better than training method 1(No Training).")
else:
  print("Fail to Reject H0")
  print("No significant difference between training methods.")
print()

f_stat, p_value = f_oneway(g3, g1)
if p_value < alpha:
    print("Reject H0")
    print("Training method 3(WHN) is better than training method 1(No Training).")
else:
  print("Fail to Reject H0")
  print("No significant difference between training methods.")
print()

f_stat, p_value = f_oneway(g4, g1)
if p_value < alpha:
    print("Reject H0")
    print("Training method 4(MHN) is better than training method 1(No Training).")
else:
  print("Fail to Reject H0")
  print("No significant difference between training methods.")
print()


In [ ]:
analysis2.groupby('treat')['improvement'].mean().sort_values(ascending=False)

## 11- توسط حضور الجلسات لمجموعة wemp

In [ ]:
wemp_course_df = attendance[parent_df['wemp']==1]
wemp_course_df.columns

In [ ]:
wemp_course_df[wemp_course_df['Eattend_men']!=0].shape

In [ ]:
wemp_course_df[wemp_course_df['Eattend_whn']!=0].shape

we are going to drop "Eattend_men" and "Eattend_whn" because it's all zero

In [ ]:
wemp_course_df.drop("Eattend_men",axis=1,inplace=True)
wemp_course_df.drop("Eattend_whn",axis=1,inplace=True)
wemp_course_df.describe()

In [ ]:
info_df = pd.DataFrame()
info_df['missing_val'] = wemp_course_df.isnull().sum()
info_df['missing_val_ratio'] = (info_df['missing_val'] / wemp_course_df.shape[0] * 100).round().astype(int)
info_df

In [ ]:
not_equal_wemp = wemp_course_df[
    wemp_course_df['Eattend_wemp'].round(2) != wemp_course_df['sessions_avg'].round(2)
]

print(not_equal_wemp.shape)
not_equal_wemp.head()

In [ ]:
wemp_course_df['sessions_avg'].describe()

In [ ]:
fig_box = px.box(
    wemp_course_df,
    y='sessions_avg',
    title="توزيع المتوسط الحقيقي للجلسات في الاستبيان",
)
fig_box.update_layout(**layout_options)
fig_box.show()

In [ ]:
wemp_course_df = pd.concat([wemp_course_df, parent_df['id']],axis=1)
plot_data_wemp = wemp_course_df.groupby(['sessions_avg'], as_index=False).count()[['sessions_avg', 'id']].rename(columns={'id': 'count'})
plot_data_wemp['sessions_avg'] = plot_data_wemp['sessions_avg'].round(2).astype(str)

In [ ]:
fig = px.histogram(
    wemp_course_df,
    x='sessions_avg',
    title="توزيع تكرار متوسط الجلسات المحسوب",
    opacity=0.85
)
fig.update_layout(**layout_options)
fig.show()

## 12- متوسط اوزان الاطفال قبل وبعد التدريب

In [ ]:
clean_baseline = children_df[
    (children_df['wemp'] == 1) &
    (children_df['c_weight_flag'] == 0) &
    (children_df['c_weight'].notnull())
]

clean_endline = children_df[
    (children_df['wemp'] == 1) &
    (children_df['Ec_weight_flag'] == 0) &
    (children_df['Ec_weight'].notnull())
]


print("Baseline")
print(f"عدد الأطفال: {clean_baseline.shape[0]} طفل")
print(f"أقل وزن: {clean_baseline['c_weight'].min():.2f} كغ")
print(f"أعلى وزن: {clean_baseline['c_weight'].max():.2f} كغ")
print(f"متوسط الأوزان: {clean_baseline['c_weight'].mean():.2f} كغ")
print("\n")

print("Endline")
print(f"عدد الأطفال: {clean_endline.shape[0]} طفل")
print(f"أقل وزن: {clean_endline['Ec_weight'].min():.2f} كغ")
print(f"أعلى وزن: {clean_endline['Ec_weight'].max():.2f} كغ")
print(f"متوسط الأوزان: {clean_endline['Ec_weight'].mean():.2f} كغ")
print("\n")


# Box Plots

fig_box = make_subplots(rows=1, cols=2, subplot_titles=("قبل التدريب (Baseline)", "بعد التدريب (Endline)"))

fig_box.add_trace(
    go.Box(
        y=clean_baseline['c_weight'],
        name="قبل التدريب",
        fillcolor='#FF6B6B',
        line=dict(color='#C92A2A', width=2)
    ),
    row=1, col=1
)

fig_box.add_trace(
    go.Box(
        y=clean_endline['Ec_weight'],
        name="بعد التدريب",
        fillcolor='#20C997',
        line=dict(color='#087F5B', width=2)
    ),
    row=1, col=2
)

fig_box.update_layout(
    title_text="مقارنة توزع أوزان الأطفال (قبل وبعد التدريب)",
    showlegend=False,
    **layout_options
)

fig_box.update_layout(
    font=dict(color="white"),
    hoverlabel=dict(font_color="white", bgcolor="#333333")
)
fig_box.update_annotations(font_color="white")
fig_box.update_xaxes(color="white", tickfont=dict(color="white"))
fig_box.update_yaxes(color="white", tickfont=dict(color="white"))

fig_box.show()


# Histogram

fig_hist = make_subplots(rows=1, cols=2, subplot_titles=("قبل التدريب (Baseline)", "بعد التدريب (Endline)"))

fig_hist.add_trace(
    go.Histogram(
        x=clean_baseline['c_weight'],
        name="قبل التدريب",
        marker=dict(color='#FF6B6B', line=dict(color='#C92A2A', width=1))
    ),
    row=1, col=1
)

fig_hist.add_trace(
    go.Histogram(
        x=clean_endline['Ec_weight'],
        name="بعد التدريب",
        marker=dict(color='#20C997', line=dict(color='#087F5B', width=1))
    ),
    row=1, col=2
)

fig_hist.update_layout(
    title_text="مقارنة المدرج التكراري لأوزان الأطفال",
    showlegend=False,
    **layout_options
)

fig_hist.update_layout(
    font=dict(color="white"),
    hoverlabel=dict(font_color="white", bgcolor="#333333")
)
fig_hist.update_annotations(font_color="white")
fig_hist.update_xaxes(color="white", tickfont=dict(color="white"))
fig_hist.update_yaxes(color="white", tickfont=dict(color="white"))

fig_hist.show()

In [ ]:
children_wieght_wemp = endline_anthro[children_df['wemp'] == 1][['Ec_weight', 'Ec_weight_flag']]
print(children_wieght_wemp.shape)
children_wieght_wemp.head()

In [ ]:
children_wieght_wemp[(children_wieght_wemp['Ec_weight_flag'] == 1) & (~children_wieght_wemp['Ec_weight'].isnull())].head()

In [ ]:
children_wieght_wemp[(children_wieght_wemp['Ec_weight_flag'] == 0) & (children_wieght_wemp['Ec_weight'].isnull())].head()

In [ ]:
children_wieght_wemp['Ec_weight'].isnull().sum()

In [ ]:
fig = go.Figure()
fig_box = px.box(
    children_wieght_wemp,
    y='Ec_weight',
    title="توزع اوزان الاطفال",
)
fig_box.update_layout(**layout_options)
fig_box.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=children_wieght_wemp['Ec_weight'].values))
fig.update_layout(**layout_options)
fig.show()

## 13- إيجاد العلاقة بين وزن الطفل ومدة البقاء مريضاً

In [ ]:
child_wight = children_df['Ec_weight']

In [ ]:

df_clean = children_df.dropna(subset=['c_weight', 'sick_days', 'Ec_weight', 'Esick_days']).copy()

groups_info = [
    {
        'name': 'Control (بدون تدريب)',
        'condition': (df_clean['mhn'] == 0) & (df_clean['whn'] == 0) & (df_clean['wemp'] == 0)
    },
    {
        'name': 'WEMP (تمكين)',
        'condition': (df_clean['mhn'] == 0) & (df_clean['whn'] == 0) & (df_clean['wemp'] == 1)
    },
    {
        'name': 'WHN (صحة نساء)',
        'condition': (df_clean['mhn'] == 0) & (df_clean['whn'] == 1) & (df_clean['wemp'] == 0)
    },
    {
        'name': 'MHN (صحة رجال)',
        'condition': (df_clean['mhn'] == 1) & (df_clean['whn'] == 0) & (df_clean['wemp'] == 0)
    }
]

for group in groups_info:
    df_filtered = df_clean[group['condition']]

    if df_filtered.empty:
        continue

    corr_baseline = df_filtered['c_weight'].corr(df_filtered['sick_days'])
    corr_endline = df_filtered['Ec_weight'].corr(df_filtered['Esick_days'])

    print(f"مجموعة: {group['name']} ")
    print(f"عدد الأطفال (بيانات كاملة): {df_filtered.shape[0]}")
    print(f"قوة العلاقة قبل التدريب: {corr_baseline:.2f}")
    print(f"قوة العلاقة بعد التدريب: {corr_endline:.2f}\n")



 1. مجموعة التحكم Control (بدون تدريب)

* قبل التدريب (0.03): علاقة شبه معدومة.
* بعد التدريب (-0.04): تغيير بسيط جداً نحو السالب، ولكنه لا يزال قريباً من الصفر.
* العائلات التي لم تتلقَّ أي تدريب لم يتغير لديها شيء تقريباً. علاقة وزن الطفل بمرضه بقيت ضعيفة جداً وعشوائية. هذا يثبت أن أي تحسن سنراه في المجموعات الأخرى هو بسبب التدريب حصراً وليس بسبب مرور الزمن أو عوامل خارجية.

 2. مجموعة التمكين WEMP (تمكين النساء)

* قبل التدريب (-0.00): لا توجد أي علاقة إطلاقاً.
* بعد التدريب (-0.10): ظهرت علاقة عكسية واضحة ومهمة.
* تمكين المرأة جعلها قادرة على استثمار موارد الأسرة في تغذية الطفل بشكل صحيح. هذا الاستثمار انعكس فوراً على صحة الطفل، فأصبحنا نرى بوضوح أنه كلما زاد وزن الطفل، انخفضت أيام مرضه بشكل ملحوظ.

 3. مجموعة صحة النساء WHN (تدريب صحي للنساء فقط)

* قبل التدريب (0.09): العلاقة كانت موجبة قليلاً (عشوائية أو ربما الأطفال الأثقل كانوا يمرضون لأسباب أخرى).
* بعد التدريب (-0.12): انقلاب جذري وقوي نحو العلاقة العكسية السالبة!
* : عندما تلقت الأمهات معلومات صحية وغذائية دقيقة، عرفن بالضبط ماذا يطعمن أطفالهن (وليس مجرد زيادة وزن بالنشويات مثلاً). النتيجة كانت قفزة ممتازة في قوة العلاقة، حيث أصبح الوزن الجيد للطفل درعاً حقيقياً يقلل من أيام مرضه بشكل كبير جداً (-0.12).

 4. مجموعة صحة الرجال MHN (تدريب صحي للرجال فقط)

* قبل التدريب (0.06): علاقة موجبة ضعيفة.
* بعد التدريب (-0.13): أقوى علاقة عكسية بين كل المجموعات!
*  هذه النتيجة مفاجأة قوية جداً وممتازة للبحث! عندما تم تثقيف الآباء صحياً وغذائياً، حقق الأطفال أكبر استفادة (-0.13). لماذا؟ لأن الرجل في هذا المجتمع هو من يملك المال (كما رأينا في تحليلات اتخاذ القرار). عندما اقتنع الرجل بأهمية الغذاء والصحة، قام غالباً بتوفير ميزانية أفضل لشراء اللحوم والفواكه، مما جعل وزن الطفل يرتبط بشدة بمناعة قوية وانخفاض حاد في أيام المرض.


In [ ]:
df_clean = children_df.dropna(subset=['c_weight', 'sick_days', 'Ec_weight', 'Esick_days']).copy()

df_clean['Weight_Bin_Before'] = df_clean['c_weight'].round()
df_clean['Weight_Bin_After'] = df_clean['Ec_weight'].round()

groups_info = [
    {'name': 'Control (بدون تدريب)', 'condition': (df_clean['mhn'] == 0) & (df_clean['whn'] == 0) & (df_clean['wemp'] == 0)},
    {'name': 'WEMP (تمكين)', 'condition': (df_clean['mhn'] == 0) & (df_clean['whn'] == 0) & (df_clean['wemp'] == 1)},
    {'name': 'WHN (صحة نساء)', 'condition': (df_clean['mhn'] == 0) & (df_clean['whn'] == 1) & (df_clean['wemp'] == 0)},
    {'name': 'MHN (صحة رجال)', 'condition': (df_clean['mhn'] == 1) & (df_clean['whn'] == 0) & (df_clean['wemp'] == 0)}
]

for group in groups_info:
    df_filtered = df_clean[group['condition']]

    if df_filtered.empty:
        continue

    median_before = df_filtered.groupby('Weight_Bin_Before')['sick_days'].median().reset_index()
    median_after = df_filtered.groupby('Weight_Bin_After')['Esick_days'].median().reset_index()

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=median_before['Weight_Bin_Before'],
        y=median_before['sick_days'],
        name='قبل التدريب (Baseline)',
        marker_color='#ff9999'
    ))

    fig.add_trace(go.Bar(
        x=median_after['Weight_Bin_After'],
        y=median_after['Esick_days'],
        name='بعد التدريب (Endline)',
        marker_color='#66b3ff'
    ))

    fig.update_layout(
        title=f"وسيط أيام المرض مقارنة بوزن الطفل - مجموعة: {group['name']}",
        xaxis_title='وزن الطفل التقريبي (كغ)',
        yaxis_title='وسيط عدد أيام المرض (Median Sick Days)',
        barmode='group',
        xaxis=dict(tickmode='linear', dtick=1)
    )

    fig.show()

تُظهر المخططات البيانية بوضوح وجود علاقة عكسية بين زيادة وزن الطفل وانخفاض أيام مرضه في المجموعات المتدربة. وتبرز مجموعة (تدريب صحة النساء WHN) كأكثر المجموعات نجاحاً في تقليص أيام مرض الأطفال بشكل جذري، مما يؤكد أن تثقيف الأمهات صحياً هو الاستثمار الأسرع والأعلى عائداً على صحة الطفل، متفوقاً على تدريب الآباء (MHN). كما يثبت تطابق معدلات المرض في مجموعة التحكم (Control) أن هذا التحسن هو نتيجة مباشرة وحصرية لبرامج التدريب والتمكين.

## 14- استكشاف الفروقات العمرية بين الأزواج  ومن يتخذ القرار بشأن صحة الطفل

In [ ]:
# men mean age from (men_another_servey)

print(f"متوسط عمر النساء {parent_df['w_age'].mean().round()} ")
print(f"\n متوسط عمر الرجال {parent_df['m_age'].mean().round()} ")

# decision making
chealth_who_map = {
    1: 'Wife', 2: 'Husband', 3: 'Both together',
    4: 'Other HH member', 5: 'Someone outside the HH', 6: 'Not applicable'
}

prevail_map = {
    1: 'Yourself', 2: 'Your spouse',
    3: 'A third person', 4: 'Respondent finds another way'
}

parent_df['w_chealth_who_label'] = parent_df['w_chealth_who'].map(chealth_who_map).astype('category')
parent_df['w_chealth_prevail_label'] = parent_df['w_chealth_prevail'].map(prevail_map).astype('category')

print("\n  نسبة مشاركة الزوجة بقرار تكاليف صحة الأطفال")
print(parent_df['w_chealth_bi'].value_counts(normalize=True) * 100)

print("\n من يتخذ القرار؟ ")
print(parent_df['w_chealth_who_label'].value_counts(normalize=True) * 100)

argue_flow = parent_df[['w_chealth_diffop', 'w_chealth_express', 'w_chealth_argue']].mean() * 100
print("\n عملية الاختلاف  ")
print(argue_flow)

print("\n من ينتصر في الجدال؟ ")
print(parent_df['w_chealth_prevail_label'].value_counts(normalize=True) * 100)

fig = px.pie(parent_df, names='w_chealth_who_label',
             title='توزيع صاحب القرار بشأن تكاليف صحة الأطفال',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

In [ ]:
# فرق العمر
parent_df['age_gap'] = parent_df['m_age'] - parent_df['w_age']

def categorize_gap(gap):
    if gap <= 3: return 'Small (<=3 years)'
    elif gap <= 7: return 'Medium (4-7 years)'
    else: return 'Large (>7 years)'

parent_df['age_gap_category'] = parent_df['age_gap'].apply(categorize_gap)

# تقاطع
gap_decision = pd.crosstab(parent_df['age_gap_category'], parent_df['w_chealth_who_label'], normalize='index') * 100
print("تأثير الفجوة العمرية على من يتخذ القرار (%):")
print(gap_decision)

fig_gap = px.bar(gap_decision.reset_index(), x='age_gap_category', y=['Husband', 'Both together', 'Wife'],
                 title='صاحب القرار بناءً على الفجوة العمرية بين الزوجين',
                 barmode='group',
                 color_discrete_sequence=px.colors.qualitative.Set2)
fig_gap.show()

<div dir="rtl">

نعم يوجد فروقات عمرية بين الرجال و النساء

تظهر البيانات أن متوسط أعمار النساء في العينة هو 27 عام بينما متوسط أعمار الأزواج هو 33  هذا يعني أن هناك فرق  بمعدل 6 سنوات تقريباً  (الزوج اكبر طبعا)



اما بالنسبة للعلاقة بين العمر و اتخاذ القرار
فمن الواضح من المخطط ان الزوج هو المسيطر على القرارات بنسبة جدا عالية
وبالتدقيق نلاحظ ان اكبر نسبة لمشاركة الزوجة الرأي هو عند الفارق العمري المقدر ب7 سنوات !

## 15- دراسة العلاقة بين مستوى تعليم المرأة  ودخل الأسرة



In [ ]:
exp_vars = [
    'Ew_last7d_sugar',
    'Ew_last7d_paraffin',
    'Ew_last7d_soap',
    'Ew_last7d_tea',
    'Ew_last7d_milk',
    'Ew_last7d_meats',
    'Ew_last7d_salt',
    'Ew_last7d_transp',
    'Ew_last7d_phone',
    'Ew_last7d_rice',
    'Ew_last7d_ccloth',
    'Ew_last7d_health',
    'Ew_last7d_schfee',
    'Ew_last7d_schsupp',
    'Ew_last7d_wcloth'
]

parent_df['w_exp'] = parent_df[exp_vars].fillna(0).sum(axis=1)


In [ ]:
tmp = parent_df[['w_educ', 'w_exp']].copy()

tmp['w_educ'] = pd.to_numeric(tmp['w_educ'], errors='coerce')
tmp['w_exp'] = pd.to_numeric(tmp['w_exp'], errors='coerce')

tmp = tmp.dropna()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = tmp[['w_educ', 'w_exp']].corr(method='spearman')

plt.figure(figsize=(5,4))

sns.heatmap(
    corr,
    annot=True,
    cmap='YlGnBu',
    vmin=-1,
    vmax=1
)

plt.title('Spearman Correlation')
plt.show()

In [ ]:
from scipy.stats import spearmanr

rho, p = spearmanr(
    parent_df['w_educ'],
    parent_df['w_exp'],
    nan_policy='omit'
)

print("rho =", rho)
print("p-value =", p)

على الرغم من وجود دلالة إحصائية، فإن حجم الأثر ضعيف جداً، وبالتالي لا يمكن اعتبار تعليم المرأة مؤشراً قوياً لمستوى دخل أو إنفاق الأسرة في هذه العينة

## 16- دراسة العلاقة بين مقدار الإنفاق على صحة الطفل ووزنه

In [ ]:
child_wight = children_df['Ec_weight']

## 17.a - حساب المتغير الجديد total_asset_score للممتلكات

In [2310]:
asset_cols = [
    'w_cattle_own', 'w_goat_own', 'w_chx_own', 'w_stove_own', 'w_wattnk_own',
    'w_bed_own', 'w_lant_own', 'w_tv_own', 'w_radio_own', 'w_gen_own',
    'w_cell_own', 'w_bic_own', 'w_moto_own', 'w_car_own'
]

for col in asset_cols:
    parent_df[col] = pd.to_numeric(parent_df[col], errors='coerce')
parent_df['total_asset_score'] = parent_df[asset_cols].fillna(0).sum(axis=1)
print(parent_df['total_asset_score'].describe())

count    5512.000000
mean        9.229776
std         6.617852
min         0.000000
25%         5.000000
50%         8.000000
75%        12.000000
max        56.000000
Name: total_asset_score, dtype: float64


In [2313]:
print(parent_df[['id', 'total_asset_score']].head(15))

             id  total_asset_score
0   111010102.0               10.0
1   111010408.0               12.0
2   111030106.0                7.0
3   111030207.0                2.0
4   111040310.0               17.0
5   111050205.0               11.0
6   111060407.0               11.0
7   111080101.0               11.0
8   111080205.0               20.0
9   111080402.0               18.0
10  111080502.0               21.0
11  111090602.0                9.0
12  111090701.0               15.0
13  111090806.0               20.0
14  111100110.0                6.0


In [2315]:
parent_df = parent_df.copy()
def categorize_wealth(score):
    if score <= 5.0: return '1. طبقة فقيرة جداً (<= 5)'
    elif score <= 12.0: return '2. طبقة متوسطة (6 - 12)'
    else: return '3. طبقة ميسورة (> 12)'

parent_df['wealth_class'] = parent_df['total_asset_score'].apply(categorize_wealth)


fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("توزع الممتلكات (التفاوت الطبقي)", "الطبقات الاجتماعية", "القيم الشاذة "),
    specs=[[{"type": "histogram"}, {"type": "domain"}, {"type": "box"}]]
)

fig.add_trace(
    go.Histogram(x=parent_df['total_asset_score'], name="عدد العائلات", marker_color='#4DABF7'),
    row=1, col=1
)

wealth_counts = parent_df['wealth_class'].value_counts().sort_index()
fig.add_trace(
    go.Pie(labels=wealth_counts.index, values=wealth_counts.values, marker=dict(colors=['#FF6B6B', '#FCC419', '#20C997']), hole=0.4),
    row=1, col=2
)

fig.add_trace(
    go.Box(y=parent_df['total_asset_score'], name="الممتلكات", marker_color='#9C27B0'),
    row=1, col=3
)

fig.update_layout(
    title_text="  الممتلكات ",
    showlegend=False,
    height=500,
    font=dict(color="white"),
    hoverlabel=dict(font_color="white", bgcolor="#333333"),
    **layout_options
)
fig.update_annotations(font_color="white")
fig.update_xaxes(color="white", tickfont=dict(color="white"))
fig.update_yaxes(color="white", tickfont=dict(color="white"))

fig.show()

## 17.b - حساب المتغير الجديد dietary_diversity للتنويع الغذائي للطفل

In [ ]:
import numpy as np

children_df['g_grains'] = children_df[
    ['matooke', 'roots', 'grains']
].max(axis=1)

children_df['g_legumes'] = children_df['beans']

children_df['g_dairy'] = children_df['milk']

children_df['g_flesh'] = children_df[
    ['meat', 'fish', 'organ']
].max(axis=1)

children_df['g_eggs'] = children_df['egg']

children_df['g_vitA'] = children_df[
    ['pump', 'green']
].max(axis=1)

children_df['g_otherfv'] = children_df['otherfru']

dd_cols = [
    'g_grains',
    'g_legumes',
    'g_dairy',
    'g_flesh',
    'g_eggs',
    'g_vitA',
    'g_otherfv'
]

children_df['dietary_diversity'] = children_df[dd_cols].sum(axis=1)

children_df['minimum_dietary_diversity'] = (
    children_df['dietary_diversity'] >= 5
).astype(int)

In [ ]:
children_df['Eg_grains'] = children_df[
    ['Ematooke', 'Eroots', 'Egrains']
].max(axis=1)

children_df['Eg_legumes'] = children_df['Ebeans']

children_df['Eg_dairy'] = children_df['Emilk']

children_df['Eg_flesh'] = children_df[
    ['Emeat', 'Efish', 'Eorgan']
].max(axis=1)

children_df['Eg_eggs'] = children_df['Eegg']

children_df['Eg_vitA'] = children_df[
    ['Epump', 'Egreen']
].max(axis=1)

children_df['Eg_otherfv'] = children_df['Eotherfru']

edd_cols = [
    'Eg_grains',
    'Eg_legumes',
    'Eg_dairy',
    'Eg_flesh',
    'Eg_eggs',
    'Eg_vitA',
    'Eg_otherfv'
]

children_df['Edietary_diversity'] = children_df[edd_cols].sum(axis=1)

children_df['Eminimum_dietary_diversity'] = (
    children_df['Edietary_diversity'] >= 5
).astype(int)

In [ ]:
children_df['DD_change'] = (
    children_df['Edietary_diversity']
    - children_df['dietary_diversity']
)

children_df['MDD_change'] = (
    children_df['Eminimum_dietary_diversity']
    - children_df['minimum_dietary_diversity']
)